# State Profile 주 모델 훈련

모든 필요 project Python source를 raw code로 포함. Base64 ZIP 및 custom import hook 없음.


In [ ]:
API_BASE_URL = 'https://161.33.212.6'
DRIVE_CACHE_PATH = '/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3'
import shutil, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'joblib>=1.4,<2', 'numpy>=2.0,<3', 'pandas>=2.2,<4', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5', 'httpx>=0.27,<1'], check=True)
from google.colab import drive, userdata
drive.mount('/content/drive')
API_KEY = userdata.get('GBIS_API_KEY')
if not API_KEY: raise RuntimeError('Colab Secret GBIS_API_KEY 필요')
LOCAL_CACHE = Path('/content/gbis_api_cache.sqlite3')
DRIVE_CACHE = Path(DRIVE_CACHE_PATH)
DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_CACHE.is_file(): shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)


In [ ]:
# 노트북에 직접 포함된 Python source. Base64/ZIP/import hook 없음.
# 표준 import가 가능하도록 Colab runtime에 raw .py 파일로 복원한다.
import sys
from pathlib import Path
MODULE_SOURCES = {'gbis_client/cache.py': 'from __future__ import annotations\n\nimport os\nimport sqlite3\nimport time\nfrom contextlib import contextmanager\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterator, Mapping\n\nimport httpx\nimport pandas as pd\n\n\nDEFAULT_CACHE_PATH = Path("data/gbis_api_cache.sqlite3")\n\n\nSCHEMA = """\nPRAGMA journal_mode = WAL;\nPRAGMA foreign_keys = ON;\n\nCREATE TABLE IF NOT EXISTS cache_metadata (\n    resource TEXT PRIMARY KEY,\n    refreshed_at_utc TEXT NOT NULL\n);\n\nCREATE TABLE IF NOT EXISTS routes (\n    route_id TEXT PRIMARY KEY,\n    station_count INTEGER NOT NULL,\n    observation_count INTEGER NOT NULL,\n    first_collected_at TEXT,\n    last_collected_at TEXT,\n    cached_at_utc TEXT NOT NULL\n);\n\nCREATE TABLE IF NOT EXISTS route_stations (\n    route_id TEXT NOT NULL,\n    station_id TEXT NOT NULL,\n    station_seq INTEGER NOT NULL,\n    station_name TEXT,\n    mobile_no TEXT,\n    region_name TEXT,\n    x REAL,\n    y REAL,\n    center_yn TEXT,\n    synced_at_kst TEXT,\n    cached_at_utc TEXT NOT NULL,\n    PRIMARY KEY (route_id, station_seq)\n);\n\nCREATE INDEX IF NOT EXISTS idx_api_cache_stations_id\n    ON route_stations(station_id);\n\nCREATE TABLE IF NOT EXISTS latest_locations (\n    route_id TEXT NOT NULL,\n    vehicle_id TEXT NOT NULL,\n    observed_at TEXT NOT NULL,\n    query_time TEXT,\n    plate_no TEXT,\n    route_type_code INTEGER,\n    station_id TEXT,\n    station_seq INTEGER,\n    station_name TEXT,\n    remaining_seats INTEGER,\n    crowded INTEGER,\n    low_plate INTEGER,\n    state_code INTEGER,\n    tagless_code INTEGER,\n    cached_at_utc TEXT NOT NULL,\n    PRIMARY KEY (route_id, vehicle_id)\n);\n\nCREATE INDEX IF NOT EXISTS idx_api_cache_latest_station\n    ON latest_locations(route_id, station_seq);\n\nCREATE TABLE IF NOT EXISTS location_history (\n    route_id TEXT NOT NULL,\n    vehicle_id TEXT NOT NULL,\n    observed_at TEXT NOT NULL,\n    query_time TEXT,\n    plate_no TEXT,\n    route_type_code INTEGER,\n    station_id TEXT,\n    station_seq INTEGER,\n    station_name TEXT,\n    remaining_seats INTEGER,\n    crowded INTEGER,\n    low_plate INTEGER,\n    state_code INTEGER,\n    tagless_code INTEGER,\n    cached_at_utc TEXT NOT NULL,\n    PRIMARY KEY (route_id, observed_at, vehicle_id)\n);\n\nCREATE INDEX IF NOT EXISTS idx_api_cache_history_time\n    ON location_history(observed_at);\nCREATE INDEX IF NOT EXISTS idx_api_cache_history_route_time\n    ON location_history(route_id, observed_at);\n\nCREATE TABLE IF NOT EXISTS history_sync_state (\n    route_id TEXT PRIMARY KEY,\n    source_max_observed_at TEXT,\n    refreshed_at_utc TEXT NOT NULL\n);\n\nPRAGMA user_version = 2;\n"""\n\n\nLOCATION_COLUMNS = (\n    "observed_at",\n    "query_time",\n    "route_id",\n    "vehicle_id",\n    "plate_no",\n    "route_type_code",\n    "station_id",\n    "station_seq",\n    "station_name",\n    "remaining_seats",\n    "crowded",\n    "low_plate",\n    "state_code",\n    "tagless_code",\n)\n\n\nclass GBISClientError(RuntimeError):\n    """API 호출이나 응답 처리에 실패했을 때 발생합니다."""\n\n\ndef _utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat(timespec="seconds")\n\n\ndef _route_id(value: str) -> str:\n    normalized = str(value).strip()\n    if not normalized.isdigit():\n        raise ValueError("route_id는 숫자 문자열이어야 합니다.")\n    return normalized\n\n\ndef _datetime_param(value: datetime | str | None) -> str | None:\n    if value is None:\n        return None\n    if isinstance(value, datetime):\n        return value.isoformat(timespec="seconds")\n    normalized = value.strip()\n    if not normalized:\n        raise ValueError("날짜/시간 값은 비어 있을 수 없습니다.")\n    return normalized\n\n\ndef _parse_datetime(value: str, *, default_timezone: Any = None) -> datetime:\n    try:\n        parsed = datetime.fromisoformat(value)\n    except ValueError as exc:\n        raise ValueError(f"날짜/시간 형식이 올바르지 않습니다: {value}") from exc\n    if parsed.tzinfo is None and default_timezone is not None:\n        parsed = parsed.replace(tzinfo=default_timezone)\n    return parsed\n\n\ndef _read_env_file(path: Path) -> dict[str, str]:\n    if not path.is_file():\n        return {}\n\n    values: dict[str, str] = {}\n    lines = path.read_text(encoding="utf-8").splitlines()\n    for line_number, raw_line in enumerate(lines, 1):\n        line = raw_line.strip()\n        if not line or line.startswith("#"):\n            continue\n        if line.startswith("export "):\n            line = line[7:].lstrip()\n        if "=" not in line:\n            raise ValueError(f"{path}:{line_number}: KEY=VALUE 형식이 아닙니다.")\n\n        key, value = line.split("=", 1)\n        key = key.strip()\n        value = value.strip()\n        if not key:\n            raise ValueError(f"{path}:{line_number}: 환경변수 이름이 비어 있습니다.")\n        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"\'", \'"\'}:\n            value = value[1:-1]\n        values[key] = value\n    return values\n\n\ndef _setting(name: str, file_values: Mapping[str, str]) -> str:\n    if name in os.environ:\n        return os.environ[name]\n    return file_values.get(name, "")\n\n\nclass GBISApiCache:\n    """GBIS API 응답을 로컬 SQLite에 저장하고 DataFrame으로 읽습니다."""\n\n    def __init__(\n        self,\n        *,\n        base_url: str,\n        api_key: str,\n        cache_path: Path | str = DEFAULT_CACHE_PATH,\n        timeout_seconds: float = 20.0,\n        max_rate_limit_retries: int = 6,\n        rate_limit_wait_seconds: float = 1.1,\n        transport: httpx.BaseTransport | None = None,\n        sleep: Callable[[float], None] = time.sleep,\n    ) -> None:\n        normalized_url = base_url.strip().rstrip("/")\n        if not normalized_url:\n            raise ValueError("base_url은 비어 있을 수 없습니다.")\n        if not api_key.strip():\n            raise ValueError("api_key는 비어 있을 수 없습니다.")\n        if timeout_seconds <= 0:\n            raise ValueError("timeout_seconds는 0보다 커야 합니다.")\n        if max_rate_limit_retries < 0:\n            raise ValueError("max_rate_limit_retries는 0 이상이어야 합니다.")\n        if rate_limit_wait_seconds <= 0:\n            raise ValueError("rate_limit_wait_seconds는 0보다 커야 합니다.")\n\n        self.base_url = normalized_url\n        self.cache_path = Path(cache_path).expanduser()\n        self.max_rate_limit_retries = max_rate_limit_retries\n        self.rate_limit_wait_seconds = rate_limit_wait_seconds\n        self._sleep = sleep\n        self._http = httpx.Client(\n            base_url=self.base_url,\n            headers={"Authorization": f"Bearer {api_key.strip()}"},\n            timeout=timeout_seconds,\n            transport=transport,\n        )\n        self._initialize()\n\n    @classmethod\n    def from_env(\n        cls,\n        *,\n        env_file: Path | str = Path(".env"),\n        cache_path: Path | str | None = None,\n        timeout_seconds: float = 20.0,\n        transport: httpx.BaseTransport | None = None,\n    ) -> "GBISApiCache":\n        env_path = Path(env_file).expanduser()\n        file_values = _read_env_file(env_path)\n        base_url = _setting("GBIS_API_BASE_URL", file_values).strip()\n        api_key = _setting("GBIS_API_KEY", file_values).strip()\n        if not base_url:\n            raise ValueError(\n                f"GBIS_API_BASE_URL이 환경변수 또는 {env_path}에 없습니다."\n            )\n        if not api_key:\n            raise ValueError(f"GBIS_API_KEY가 환경변수 또는 {env_path}에 없습니다.")\n        configured_cache_path = _setting("GBIS_API_CACHE_PATH", file_values).strip()\n        resolved_cache_path = cache_path or configured_cache_path or DEFAULT_CACHE_PATH\n        return cls(\n            base_url=base_url,\n            api_key=api_key,\n            cache_path=resolved_cache_path,\n            timeout_seconds=timeout_seconds,\n            transport=transport,\n        )\n\n    def close(self) -> None:\n        self.checkpoint()\n        self._http.close()\n\n    def __enter__(self) -> "GBISApiCache":\n        return self\n\n    def __exit__(self, *_: object) -> None:\n        self.close()\n\n    @contextmanager\n    def _connect(self) -> Iterator[sqlite3.Connection]:\n        self.cache_path.parent.mkdir(parents=True, exist_ok=True)\n        connection = sqlite3.connect(self.cache_path, timeout=30)\n        connection.row_factory = sqlite3.Row\n        connection.execute("PRAGMA busy_timeout = 30000")\n        connection.execute("PRAGMA foreign_keys = ON")\n        try:\n            yield connection\n            connection.commit()\n        except Exception:\n            connection.rollback()\n            raise\n        finally:\n            connection.close()\n\n    def _initialize(self) -> None:\n        with self._connect() as connection:\n            connection.executescript(SCHEMA)\n            route_columns = {\n                row["name"]\n                for row in connection.execute("PRAGMA table_info(routes)").fetchall()\n            }\n            if "first_collected_at" not in route_columns:\n                connection.execute(\n                    "ALTER TABLE routes ADD COLUMN first_collected_at TEXT"\n                )\n\n    def checkpoint(self) -> None:\n        """Drive 등으로 복사하기 전에 WAL 내용을 기본 DB 파일에 반영합니다."""\n        if not self.cache_path.is_file():\n            return\n        with sqlite3.connect(self.cache_path, timeout=30) as connection:\n            connection.execute("PRAGMA busy_timeout = 30000")\n            connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")\n\n    def _get(\n        self,\n        path: str,\n        *,\n        params: Mapping[str, Any] | None = None,\n    ) -> dict[str, Any]:\n        response: httpx.Response | None = None\n        try:\n            for attempt in range(self.max_rate_limit_retries + 1):\n                response = self._http.get(path, params=params)\n                if response.status_code != 429 or attempt == self.max_rate_limit_retries:\n                    break\n                retry_after = response.headers.get("Retry-After", "").strip()\n                try:\n                    wait_seconds = float(retry_after)\n                except ValueError:\n                    wait_seconds = min(\n                        self.rate_limit_wait_seconds * (2**attempt),\n                        30.0,\n                    )\n                self._sleep(max(wait_seconds, self.rate_limit_wait_seconds))\n            assert response is not None\n            response.raise_for_status()\n        except httpx.HTTPStatusError as exc:\n            detail: Any = None\n            try:\n                body = exc.response.json()\n                if isinstance(body, dict):\n                    detail = body.get("detail")\n            except ValueError:\n                pass\n            message = f"GBIS API가 HTTP {exc.response.status_code}를 반환했습니다."\n            if detail:\n                message = f"{message} {detail}"\n            raise GBISClientError(message) from exc\n        except httpx.RequestError as exc:\n            raise GBISClientError(f"GBIS API에 연결할 수 없습니다: {exc}") from exc\n\n        try:\n            payload = response.json()\n        except ValueError as exc:\n            raise GBISClientError("GBIS API 응답이 JSON 형식이 아닙니다.") from exc\n        if not isinstance(payload, dict):\n            raise GBISClientError("GBIS API 응답의 최상위 값이 객체가 아닙니다.")\n        return payload\n\n    @staticmethod\n    def _items(payload: Mapping[str, Any]) -> list[dict[str, Any]]:\n        items = payload.get("items")\n        if not isinstance(items, list) or any(\n            not isinstance(item, dict) for item in items\n        ):\n            raise GBISClientError("GBIS API 응답의 items 형식이 올바르지 않습니다.")\n        return items\n\n    @staticmethod\n    def _mark_refreshed(\n        connection: sqlite3.Connection,\n        resource: str,\n        refreshed_at: str,\n    ) -> None:\n        connection.execute(\n            """\n            INSERT INTO cache_metadata (resource, refreshed_at_utc)\n            VALUES (?, ?)\n            ON CONFLICT(resource) DO UPDATE SET\n                refreshed_at_utc = excluded.refreshed_at_utc\n            """,\n            (resource, refreshed_at),\n        )\n\n    def refresh_routes(self) -> int:\n        items = self._items(self._get("/v1/routes"))\n        cached_at = _utc_now()\n        rows = [\n            (\n                str(item["route_id"]),\n                int(item.get("station_count") or 0),\n                int(item.get("observation_count") or 0),\n                item.get("first_collected_at"),\n                item.get("last_collected_at"),\n                cached_at,\n            )\n            for item in items\n        ]\n        with self._connect() as connection:\n            connection.execute("DELETE FROM routes")\n            connection.executemany(\n                """\n                INSERT INTO routes (\n                    route_id, station_count, observation_count,\n                    first_collected_at, last_collected_at, cached_at_utc\n                ) VALUES (?, ?, ?, ?, ?, ?)\n                """,\n                rows,\n            )\n            self._mark_refreshed(connection, "routes", cached_at)\n        return len(rows)\n\n    def refresh_stations(self, route_id: str) -> int:\n        normalized_route_id = _route_id(route_id)\n        payload = self._get(f"/v1/routes/{normalized_route_id}/stations")\n        items = self._items(payload)\n        cached_at = _utc_now()\n        rows = [\n            (\n                normalized_route_id,\n                str(item["station_id"]),\n                int(item["station_seq"]),\n                item.get("station_name"),\n                item.get("mobile_no"),\n                item.get("region_name"),\n                item.get("x"),\n                item.get("y"),\n                item.get("center_yn"),\n                item.get("synced_at_kst"),\n                cached_at,\n            )\n            for item in items\n        ]\n        with self._connect() as connection:\n            connection.execute(\n                "DELETE FROM route_stations WHERE route_id = ?",\n                (normalized_route_id,),\n            )\n            connection.executemany(\n                """\n                INSERT INTO route_stations (\n                    route_id, station_id, station_seq, station_name, mobile_no,\n                    region_name, x, y, center_yn, synced_at_kst, cached_at_utc\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n                """,\n                rows,\n            )\n            self._mark_refreshed(\n                connection, f"stations:{normalized_route_id}", cached_at\n            )\n        return len(rows)\n\n    @staticmethod\n    def _location_rows(\n        items: list[dict[str, Any]],\n        cached_at: str,\n    ) -> list[tuple[Any, ...]]:\n        return [\n            tuple(item.get(column) for column in LOCATION_COLUMNS) + (cached_at,)\n            for item in items\n        ]\n\n    def refresh_latest(self, route_id: str | None = None) -> int:\n        normalized_route_id = _route_id(route_id) if route_id is not None else None\n        params = {"route_id": normalized_route_id} if normalized_route_id else None\n        items = self._items(self._get("/v1/locations/latest", params=params))\n        cached_at = _utc_now()\n        rows = self._location_rows(items, cached_at)\n\n        with self._connect() as connection:\n            if normalized_route_id:\n                connection.execute(\n                    "DELETE FROM latest_locations WHERE route_id = ?",\n                    (normalized_route_id,),\n                )\n            else:\n                connection.execute("DELETE FROM latest_locations")\n            connection.executemany(\n                """\n                INSERT INTO latest_locations (\n                    observed_at, query_time, route_id, vehicle_id, plate_no,\n                    route_type_code, station_id, station_seq, station_name,\n                    remaining_seats, crowded, low_plate, state_code, tagless_code,\n                    cached_at_utc\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n                """,\n                rows,\n            )\n            resource = f"latest:{normalized_route_id or \'all\'}"\n            self._mark_refreshed(connection, resource, cached_at)\n        return len(rows)\n\n    def _last_history_timestamp(self, route_id: str) -> str | None:\n        with self._connect() as connection:\n            row = connection.execute(\n                """\n                SELECT source_max_observed_at\n                FROM history_sync_state\n                WHERE route_id = ?\n                """,\n                (route_id,),\n            ).fetchone()\n        return None if row is None else row["source_max_observed_at"]\n\n    def refresh_history(\n        self,\n        route_id: str,\n        *,\n        from_at: datetime | str | None = None,\n        to_at: datetime | str | None = None,\n        page_size: int = 500,\n    ) -> int:\n        normalized_route_id = _route_id(route_id)\n        if not 1 <= page_size <= 500:\n            raise ValueError("page_size는 1~500 범위여야 합니다.")\n\n        previous_checkpoint = self._last_history_timestamp(normalized_route_id)\n        start = _datetime_param(from_at)\n        if start is None:\n            start = previous_checkpoint\n        end = _datetime_param(to_at)\n        params: dict[str, Any] = {\n            "route_id": normalized_route_id,\n            "limit": page_size,\n        }\n        if start is not None:\n            params["from"] = start\n        if end is not None:\n            params["to"] = end\n\n        total = 0\n        max_observed_at = previous_checkpoint\n        cursor: str | None = None\n        seen_cursors: set[str] = set()\n        while True:\n            if cursor is not None:\n                params["cursor"] = cursor\n            payload = self._get("/v1/locations", params=params)\n            items = self._items(payload)\n            cached_at = _utc_now()\n            rows = self._location_rows(items, cached_at)\n            with self._connect() as connection:\n                connection.executemany(\n                    """\n                    INSERT INTO location_history (\n                        observed_at, query_time, route_id, vehicle_id, plate_no,\n                        route_type_code, station_id, station_seq, station_name,\n                        remaining_seats, crowded, low_plate, state_code, tagless_code,\n                        cached_at_utc\n                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n                    ON CONFLICT(route_id, observed_at, vehicle_id) DO UPDATE SET\n                        query_time = excluded.query_time,\n                        plate_no = excluded.plate_no,\n                        route_type_code = excluded.route_type_code,\n                        station_id = excluded.station_id,\n                        station_seq = excluded.station_seq,\n                        station_name = excluded.station_name,\n                        remaining_seats = excluded.remaining_seats,\n                        crowded = excluded.crowded,\n                        low_plate = excluded.low_plate,\n                        state_code = excluded.state_code,\n                        tagless_code = excluded.tagless_code,\n                        cached_at_utc = excluded.cached_at_utc\n                    """,\n                    rows,\n                )\n            total += len(rows)\n            observed_values = [\n                str(item["observed_at"])\n                for item in items\n                if item.get("observed_at") is not None\n            ]\n            if observed_values:\n                page_max = max(observed_values)\n                max_observed_at = max(\n                    value for value in (max_observed_at, page_max) if value is not None\n                )\n\n            next_cursor = payload.get("next_cursor")\n            if next_cursor is None:\n                break\n            if not isinstance(next_cursor, str) or not next_cursor:\n                raise GBISClientError("GBIS API의 next_cursor 형식이 올바르지 않습니다.")\n            if next_cursor in seen_cursors:\n                raise GBISClientError("GBIS API가 동일한 cursor를 반복해서 반환했습니다.")\n            seen_cursors.add(next_cursor)\n            cursor = next_cursor\n\n        with self._connect() as connection:\n            refreshed_at = _utc_now()\n            connection.execute(\n                """\n                INSERT INTO history_sync_state (\n                    route_id, source_max_observed_at, refreshed_at_utc\n                ) VALUES (?, ?, ?)\n                ON CONFLICT(route_id) DO UPDATE SET\n                    source_max_observed_at = excluded.source_max_observed_at,\n                    refreshed_at_utc = excluded.refreshed_at_utc\n                """,\n                (normalized_route_id, max_observed_at, refreshed_at),\n            )\n            self._mark_refreshed(\n                connection,\n                f"history:{normalized_route_id}",\n                refreshed_at,\n            )\n        return total\n\n    def _history_start(self, route_id: str) -> str:\n        checkpoint = self._last_history_timestamp(route_id)\n        if checkpoint is not None:\n            return checkpoint\n\n        with self._connect() as connection:\n            row = connection.execute(\n                "SELECT first_collected_at FROM routes WHERE route_id = ?",\n                (route_id,),\n            ).fetchone()\n        if row is None or row["first_collected_at"] is None:\n            self.refresh_routes()\n            with self._connect() as connection:\n                row = connection.execute(\n                    "SELECT first_collected_at FROM routes WHERE route_id = ?",\n                    (route_id,),\n                ).fetchone()\n        if row is None:\n            raise GBISClientError(f"노선 {route_id}을 서버에서 찾을 수 없습니다.")\n        if row["first_collected_at"] is None:\n            raise GBISClientError(\n                "서버가 first_collected_at을 제공하지 않습니다. "\n                "Oracle 서버의 API 코드를 먼저 업데이트하세요."\n            )\n        return str(row["first_collected_at"])\n\n    def refresh_full_history(\n        self,\n        route_id: str,\n        *,\n        to_at: datetime | str | None = None,\n        window_days: int = 30,\n        page_size: int = 500,\n    ) -> int:\n        """최초에는 전체 이력을, 이후에는 마지막 성공 시각부터 동기화합니다."""\n        normalized_route_id = _route_id(route_id)\n        if not 1 <= window_days <= 30:\n            raise ValueError("window_days는 1~30 범위여야 합니다.")\n\n        start = _parse_datetime(self._history_start(normalized_route_id))\n        end_value = _datetime_param(to_at)\n        if end_value is None:\n            end = datetime.now(start.tzinfo or timezone.utc)\n        else:\n            end = _parse_datetime(end_value, default_timezone=start.tzinfo)\n        if start.tzinfo is None and end.tzinfo is not None:\n            start = start.replace(tzinfo=end.tzinfo)\n        if start > end:\n            return 0\n\n        total = 0\n        window_start = start\n        window_size = timedelta(days=window_days)\n        while window_start <= end:\n            window_end = min(window_start + window_size, end)\n            total += self.refresh_history(\n                normalized_route_id,\n                from_at=window_start,\n                to_at=window_end,\n                page_size=page_size,\n            )\n            if window_end >= end:\n                break\n            window_start = window_end\n        return total\n\n    def refresh_all(\n        self,\n        route_ids: list[str] | tuple[str, ...] | None = None,\n    ) -> dict[str, int]:\n        counts: dict[str, int] = {"routes": self.refresh_routes()}\n        if route_ids is None:\n            routes = self.routes_df()\n            normalized_route_ids = [\n                str(row.route_id)\n                for row in routes.itertuples()\n                if int(row.station_count) > 0\n            ]\n        else:\n            normalized_route_ids = [_route_id(value) for value in route_ids]\n\n        counts["stations"] = sum(\n            self.refresh_stations(route_id) for route_id in normalized_route_ids\n        )\n        counts["latest_locations"] = self.refresh_latest()\n        return counts\n\n    def _read_dataframe(\n        self,\n        query: str,\n        params: tuple[Any, ...] = (),\n    ) -> pd.DataFrame:\n        with self._connect() as connection:\n            return pd.read_sql_query(query, connection, params=params)\n\n    def routes_df(self) -> pd.DataFrame:\n        return self._read_dataframe("SELECT * FROM routes ORDER BY route_id")\n\n    def stations_df(self, route_id: str | None = None) -> pd.DataFrame:\n        if route_id is None:\n            return self._read_dataframe(\n                "SELECT * FROM route_stations ORDER BY route_id, station_seq"\n            )\n        normalized_route_id = _route_id(route_id)\n        return self._read_dataframe(\n            """\n            SELECT * FROM route_stations\n            WHERE route_id = ?\n            ORDER BY station_seq\n            """,\n            (normalized_route_id,),\n        )\n\n    def latest_locations_df(self, route_id: str | None = None) -> pd.DataFrame:\n        if route_id is None:\n            return self._read_dataframe(\n                """\n                SELECT * FROM latest_locations\n                ORDER BY route_id, station_seq, vehicle_id\n                """\n            )\n        normalized_route_id = _route_id(route_id)\n        return self._read_dataframe(\n            """\n            SELECT * FROM latest_locations\n            WHERE route_id = ?\n            ORDER BY station_seq, vehicle_id\n            """,\n            (normalized_route_id,),\n        )\n\n    def history_df(\n        self,\n        route_id: str | None = None,\n        *,\n        from_at: datetime | str | None = None,\n        to_at: datetime | str | None = None,\n    ) -> pd.DataFrame:\n        conditions: list[str] = []\n        params: list[Any] = []\n        if route_id is not None:\n            conditions.append("route_id = ?")\n            params.append(_route_id(route_id))\n        start = _datetime_param(from_at)\n        if start is not None:\n            conditions.append("observed_at >= ?")\n            params.append(start)\n        end = _datetime_param(to_at)\n        if end is not None:\n            conditions.append("observed_at <= ?")\n            params.append(end)\n        where = f"WHERE {\' AND \'.join(conditions)}" if conditions else ""\n        return self._read_dataframe(\n            f"""\n            SELECT * FROM location_history\n            {where}\n            ORDER BY observed_at, route_id, vehicle_id\n            """,\n            tuple(params),\n        )\n\n    def cache_status_df(self) -> pd.DataFrame:\n        return self._read_dataframe(\n            "SELECT * FROM cache_metadata ORDER BY resource"\n        )\n', 'gbis_client/__init__.py': '"""GBIS 읽기 전용 API의 로컬 SQLite 캐시 클라이언트."""\n\nfrom .cache import GBISApiCache, GBISClientError\n\n__all__ = ["GBISApiCache", "GBISClientError"]\n', 'analysis/latest_main_model_feature_recheck.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\nfrom hypothesis_model_search import add_observed_capacity_features\nfrom linear_feature_experiment import json_ready\nfrom main_model_feature_augmentation import (\n    feature_sets,\n    paired_trip_bootstrap,\n    prepare_augmented,\n    required_metrics,\n    run_feature_set,\n)\n\n\nROLLING_DATES = (\n    "2026-08-04",\n    "2026-08-05",\n    "2026-08-06",\n    "2026-08-07",\n    "2026-08-10",\n    "2026-08-11",\n    "2026-08-12",\n    "2026-08-13",\n)\nLATEST_COMPLETE_DATES = ("2026-08-11", "2026-08-12")\nLATEST_PARTIAL_DATE = "2026-08-13"\nDEFAULT_FEATURE_SETS = (\n    "baseline_official",\n    "target_low_10_only",\n    "path_low_10_sum_only",\n    "low_rate_without_count",\n    "correlation_pruned",\n    "augmented_unique",\n)\n\n\ndef rolling_folds(data: pd.DataFrame) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []\n    for position, validation_date in enumerate(ROLLING_DATES[1:], start=1):\n        train = data.loc[data["date"].isin(ROLLING_DATES[:position])].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        if train.empty or validation.empty:\n            raise ValueError(f"rolling fold is empty: {validation_date}")\n        folds.append((validation_date, train, validation))\n    return folds\n\n\ndef partition_metrics(predictions: pd.DataFrame) -> pd.DataFrame:\n    partitions = {\n        "latest_complete_08_11_12": predictions["date"].isin(LATEST_COMPLETE_DATES),\n        "latest_partial_08_13": predictions["date"].eq(LATEST_PARTIAL_DATE),\n        "latest_all_08_11_13": predictions["date"].isin(\n            (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)\n        ),\n    }\n    rows: list[dict[str, Any]] = []\n    for partition, mask in partitions.items():\n        for candidate, frame in predictions.loc[mask].groupby("candidate", sort=False):\n            rows.append(\n                {"partition": partition, "feature_set": candidate, **required_metrics(frame)}\n            )\n    return pd.DataFrame(rows)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="최신 완전일과 부분일에서 주 모델 신규 피처를 rolling-origin 재검증"\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features/219000013_snapshots.pkl"),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features/219000013_flows.pkl"),\n    )\n    parser.add_argument(\n        "--augmented-cache",\n        type=Path,\n        default=Path("data/analysis_cache/main_model_augmented_features_latest.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/latest_main_model_feature_results"),\n    )\n    parser.add_argument("--feature-sets", nargs="*", default=list(DEFAULT_FEATURE_SETS))\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    available = feature_sets()\n    unknown = sorted(set(args.feature_sets) - set(available))\n    if unknown:\n        raise ValueError(f"unknown feature sets: {unknown}")\n    data = prepare_augmented(\n        args.snapshot_cache, args.flow_cache, args.augmented_cache\n    )\n    data = add_observed_capacity_features(data)\n    missing_dates = sorted(set(ROLLING_DATES) - set(data["date"]))\n    if missing_dates:\n        raise ValueError(f"missing rolling dates: {missing_dates}")\n    folds = rolling_folds(data)\n\n    predictions: list[pd.DataFrame] = []\n    daily: list[pd.DataFrame] = []\n    for name in args.feature_sets:\n        output, _, candidate_daily, _ = run_feature_set(\n            name, available[name], folds, seed=args.seed\n        )\n        predictions.append(output)\n        daily.append(candidate_daily)\n    prediction_table = pd.concat(predictions, ignore_index=True)\n    daily_table = pd.concat(daily, ignore_index=True)\n    metrics = partition_metrics(prediction_table)\n\n    bootstrap_rows: list[dict[str, Any]] = []\n    for partition, dates in {\n        "latest_complete_08_11_12": LATEST_COMPLETE_DATES,\n        "latest_partial_08_13": (LATEST_PARTIAL_DATE,),\n        "latest_all_08_11_13": (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE),\n    }.items():\n        selected = prediction_table.loc[prediction_table["date"].isin(dates)]\n        for candidate in args.feature_sets:\n            if candidate == "baseline_official":\n                continue\n            for low_only in (False, True):\n                row = paired_trip_bootstrap(\n                    selected,\n                    candidate,\n                    low_only=low_only,\n                    seed=args.seed,\n                )\n                bootstrap_rows.append({"partition": partition, **row})\n    bootstrap = pd.DataFrame(bootstrap_rows)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    metrics.to_csv(args.output_dir / "metrics_by_partition.csv", index=False)\n    daily_table.to_csv(args.output_dir / "daily_metrics.csv", index=False)\n    bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)\n    prediction_table.to_pickle(args.output_dir / "oof_predictions.pkl")\n    summary = {\n        "protocol": {\n            "rolling_dates": ROLLING_DATES,\n            "latest_complete_dates": LATEST_COMPLETE_DATES,\n            "latest_partial_date": LATEST_PARTIAL_DATE,\n            "training_rule": "each fold uses only prior listed weekdays",\n            "feature_history_rule": "same-route strictly earlier calendar dates",\n            "selection_caveat": "all dates have been inspected in prior experiments; comparison is retrospective, not pristine prospective validation",\n        },\n        "source": {\n            "snapshot_cache": str(args.snapshot_cache),\n            "rows": int(len(data)),\n            "events": int(data["event_id"].nunique()),\n            "min_date": str(data["date"].min()),\n            "max_date": str(data["date"].max()),\n            "max_snapshot_time": pd.Timestamp(data["snapshot_time"].max()).isoformat(),\n            "preceding_bus_segment_audit": data.attrs.get("preceding_bus_segment_audit"),\n        },\n        "feature_sets": args.feature_sets,\n        "metrics": metrics.to_dict(orient="records"),\n        "bootstrap": bootstrap.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(metrics.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/export_exact_primary_colab_model.py': '"""Export the exact pooled main ensemble as one portable Colab artifact.\n\nThe script deliberately trains the full three-component production ensemble.\nIt must not be replaced with a sampled or reduced-model alternative.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport tempfile\nfrom pathlib import Path\n\nimport joblib\nimport pandas as pd\n\nfrom emerging_low_correction_experiment import experiment_components, pooled_features\nfrom hypothesis_model_search import candidate_weights, encode_target\nfrom main_model_feature_augmentation import make_model\nfrom pooled_main_model_registry import build_pooled_feature_profile\nfrom route_specific_feature_experiment import DEFAULT_ROUTES, cache_paths\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="정확한 통합 주 모델 Colab artifact export")\n    parser.add_argument(\n        "--featured-cache",\n        type=Path,\n        default=Path("data/analysis_cache/pooled_main_model_features_colab_release.pkl"),\n    )\n    parser.add_argument(\n        "--cache-dir",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features"),\n    )\n    parser.add_argument(\n        "--output",\n        type=Path,\n        default=Path("colab/model_artifacts/arrival_seat_primary_v1.pkl"),\n    )\n    parser.add_argument("--training-cutoff-date", default="2026-08-12")\n    parser.add_argument("--source-cutoff", required=True)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.featured_cache)\n    if data.attrs.get("source_cutoff") != args.source_cutoff:\n        raise ValueError(\n            "featured cache source cutoff 불일치: "\n            f"{data.attrs.get(\'source_cutoff\')} != {args.source_cutoff}"\n        )\n    metadata = data.attrs.get("route_metadata")\n    if not isinstance(metadata, dict):\n        raise ValueError("featured cache route metadata가 없습니다.")\n    for route_id in DEFAULT_ROUTES:\n        if metadata.get(route_id, {}).get("source_cutoff") != args.source_cutoff:\n            raise ValueError(f"{route_id} feature cache cutoff 불일치")\n\n    training = data.loc[\n        data["date"].le(args.training_cutoff_date)\n        & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)\n    ].copy()\n    if training.empty:\n        raise ValueError("학습 행이 없습니다.")\n    if set(training["route_code"].astype(str)) != set(DEFAULT_ROUTES):\n        raise ValueError("활성 6개 노선 학습 데이터가 완전하지 않습니다.")\n\n    features = pooled_features()\n    models = {}\n    target_kinds = {}\n    for index, candidate in enumerate(experiment_components("l1")):\n        print(f"[exact primary export] fitting {candidate.name}", flush=True)\n        model = make_model(candidate, features, seed=args.seed + index)\n        weights = candidate_weights(\n            training,\n            candidate.low_weight,\n            weighting_kind=candidate.weighting_kind,\n            gap_weight_power=candidate.gap_weight_power,\n        )\n        model.fit(\n            training[list(features.columns)],\n            encode_target(training, candidate.target_kind),\n            regressor__sample_weight=weights,\n        )\n        models[candidate.name] = model\n        target_kinds[candidate.name] = candidate.target_kind\n\n    route_flows = {\n        route_id: pd.read_pickle(cache_paths(args.cache_dir, route_id)[1])\n        for route_id in DEFAULT_ROUTES\n    }\n    payload = {\n        "format_version": 1,\n        "model_variant": "pooled_main",\n        "model_id": "arrival-seat-pooled/v1.0.0-exact",\n        "training_cutoff_date": args.training_cutoff_date,\n        "source_cutoff": args.source_cutoff,\n        "routes": dict(DEFAULT_ROUTES),\n        "feature_columns": list(features.columns),\n        "component_target_kinds": target_kinds,\n        "ensemble_weights": {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2},\n        "feature_profile": build_pooled_feature_profile(\n            data, route_flows, training_cutoff_date=args.training_cutoff_date\n        ),\n        "models": models,\n        "training_policy": {\n            "fit_population": "all pooled training rows",\n            "sampling": "none",\n        },\n        "training_rows": int(len(training)),\n        "training_events": int(training["event_id"].nunique()),\n        "training_dates": sorted(training["date"].unique().tolist()),\n        "route_cache_metadata": metadata,\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(\n        dir=args.output.parent, prefix=".primary-model-", suffix=".pkl", delete=False\n    ) as temporary:\n        temporary_path = Path(temporary.name)\n    try:\n        joblib.dump(payload, temporary_path, compress=3)\n        os.replace(temporary_path, args.output)\n    finally:\n        temporary_path.unlink(missing_ok=True)\n    print(args.output, flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/emerging_low_correction_experiment.py': '"""Strict rolling comparison for current-11-to-25-seat emerging-low cases.\n\nAll features and risk labels are formed from each outer fold\'s earlier dates.\nThe held-out completed dates retain every pre-arrival snapshot and are scored\nwith the project\'s event-balanced metrics.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sqlite3\nfrom dataclasses import replace\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.pipeline import Pipeline\n\nfrom hypothesis_model_search import (\n    candidate_weights,\n    combine_weighted_oof,\n    decode_target,\n    encode_target,\n    scored_frame,\n    strict_forward_bias_predictions,\n)\nfrom latest_main_model_feature_recheck import LATEST_COMPLETE_DATES\nfrom linear_feature_experiment import json_ready\nfrom main_model_feature_augmentation import (\n    component_candidates,\n    feature_sets,\n    make_model,\n    required_metrics,\n)\nfrom model_feasibility import make_preprocessor\nfrom route_specific_feature_experiment import DEFAULT_ROUTES\n\n\nCURRENT_SEAT_MIN = 11\nCURRENT_SEAT_MAX = 25\nGATE_STOP_GAP_MIN = 3\nGATE_STOP_GAP_MAX = 20\nTRANSITION_WEIGHT = 4.0\nGATE_THRESHOLD = 0.50\nSPECIALIST_BLEND = 0.50\nWEIGHTS = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}\n\n\ndef experiment_components(loss_variant: str, *, fast_screen: bool = False):\n    """Return the deployed components, optionally swapping only LGBM\'s loss.\n\n    HistGradientBoostingRegressor and ExtraTreesRegressor do not offer a Huber\n    objective in the installed sklearn version.  Keeping them fixed makes this\n    a controlled ablation of the ensemble\'s LightGBM component instead of\n    confounding loss choice with a model-family change.\n    """\n    components = component_candidates()\n    if loss_variant not in {"l1", "lightgbm_huber"}:\n        raise ValueError(f"unknown loss variant: {loss_variant}")\n    output = []\n    for candidate in components:\n        params = dict(candidate.params)\n        if candidate.name == "lightgbm" and loss_variant == "lightgbm_huber":\n            params.update({"objective": "huber", "alpha": 0.9})\n        if candidate.name == "hgb" and fast_screen:\n            params["max_iter"] = min(int(params["max_iter"]), 150)\n        if candidate.name == "lightgbm" and fast_screen:\n            params["n_estimators"] = min(int(params["n_estimators"]), 150)\n        output.append(replace(candidate, params=params))\n    return tuple(output)\n\n\ndef complete_folds(data: pd.DataFrame) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []\n    for validation_date in LATEST_COMPLETE_DATES:\n        train = data.loc[\n            data["date"].lt(validation_date)\n            & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)\n        ].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        if train.empty or validation.empty:\n            raise ValueError(f"empty fold: {validation_date}")\n        folds.append((validation_date, train, validation))\n    return folds\n\n\ndef transition_label(data: pd.DataFrame) -> pd.Series:\n    return (\n        data["snapshot_remaining_seats"].between(CURRENT_SEAT_MIN, CURRENT_SEAT_MAX)\n        & data["label_seats"].le(10)\n    )\n\n\ndef gate_eligible(data: pd.DataFrame) -> pd.Series:\n    return (\n        data["snapshot_remaining_seats"].between(CURRENT_SEAT_MIN, CURRENT_SEAT_MAX)\n        & data["target_stop_gap"].between(GATE_STOP_GAP_MIN, GATE_STOP_GAP_MAX)\n    )\n\n\ndef stratified_row_sample(data: pd.DataFrame, maximum: int, *, seed: int) -> pd.DataFrame:\n    """Sample rows proportionally by route and <=10-seat status."""\n    if len(data) <= maximum:\n        return data\n    if maximum < 100:\n        raise ValueError("sample maximum must be at least 100")\n    strata = pd.DataFrame(\n        {\n            "route": data["route_code"].astype(str),\n            "low": data["label_seats"].le(10).astype(str),\n        },\n        index=data.index,\n    ).agg("::".join, axis=1)\n    counts = strata.value_counts()\n    quotas = (counts / counts.sum() * maximum).round().astype(int).clip(lower=1)\n    quotas = quotas.clip(upper=counts)\n    return pd.concat(\n        [\n            data.loc[strata.eq(stratum)].sample(n=int(quota), random_state=seed)\n            for stratum, quota in quotas.items()\n        ],\n        ignore_index=True,\n    )\n\n\ndef pooled_features() -> Any:\n    base = feature_sets()["importance_pruned"]\n    return type(base)(\n        "pooled_40_emerging_low",\n        base.numeric,\n        (*base.categorical, "route_code"),\n    )\n\n\ndef fit_probability_gate(\n    train: pd.DataFrame,\n    validation: pd.DataFrame,\n    features: Any,\n    *,\n    seed: int,\n) -> np.ndarray:\n    classifier = Pipeline(\n        [\n            ("features", make_preprocessor(features)),\n            (\n                "classifier",\n                HistGradientBoostingClassifier(\n                    loss="log_loss",\n                    learning_rate=0.04,\n                    max_iter=350,\n                    max_leaf_nodes=31,\n                    min_samples_leaf=20,\n                    l2_regularization=1.0,\n                    random_state=seed,\n                ),\n            ),\n        ]\n    )\n    target = train["label_seats"].le(10).astype(int)\n    weights = candidate_weights(train, 2.0, weighting_kind="event")\n    classifier.fit(\n        train[list(features.columns)], target, classifier__sample_weight=weights\n    )\n    return classifier.predict_proba(validation[list(features.columns)])[:, 1]\n\n\ndef fit_components(\n    train: pd.DataFrame,\n    validation: pd.DataFrame,\n    features: Any,\n    *,\n    policy: str,\n    fold_number: int,\n    seed: int,\n    transition_weight: float = TRANSITION_WEIGHT,\n    loss_variant: str = "l1",\n    component_names: tuple[str, ...] = tuple(WEIGHTS),\n    max_train_rows: int | None = None,\n    fast_screen: bool = False,\n) -> dict[str, pd.DataFrame]:\n    if policy == "transition_specialist":\n        fit_train = train.loc[transition_label(train)].copy()\n    elif policy == "low25_transition_weighted":\n        fit_train = train.loc[train["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX)].copy()\n    else:\n        fit_train = train\n    if fit_train.empty:\n        raise ValueError(f"{policy}: no fitting rows")\n    if max_train_rows is not None and len(fit_train) > max_train_rows:\n        fit_train = stratified_row_sample(fit_train, max_train_rows, seed=seed)\n    frames: dict[str, pd.DataFrame] = {}\n    selected_components = [\n        candidate\n        for candidate in experiment_components(loss_variant, fast_screen=fast_screen)\n        if candidate.name in component_names\n    ]\n    if not selected_components:\n        raise ValueError("no model components selected")\n    for candidate_index, candidate in enumerate(selected_components):\n        print(f"[{policy}] {validation[\'date\'].iloc[0]} {candidate.name}", flush=True)\n        model = make_model(candidate, features, seed + fold_number * 10 + candidate_index)\n        weights = candidate_weights(\n            fit_train,\n            candidate.low_weight,\n            weighting_kind=candidate.weighting_kind,\n            gap_weight_power=candidate.gap_weight_power,\n        )\n        if policy in {"transition_weighted", "low25_transition_weighted"}:\n            weights = weights * np.where(transition_label(fit_train), transition_weight, 1.0)\n        model.fit(\n            fit_train[list(features.columns)],\n            encode_target(fit_train, candidate.target_kind),\n            regressor__sample_weight=weights,\n        )\n        prediction = decode_target(\n            model.predict(validation[list(features.columns)]), validation, candidate.target_kind\n        )\n        frame = scored_frame(validation, prediction, candidate=candidate.name)\n        frame["is_transition_case"] = transition_label(validation).to_numpy()\n        frame["is_gate_eligible"] = gate_eligible(validation).to_numpy()\n        frames[candidate.name] = frame\n    return frames\n\n\ndef blend_by_fold(\n    policy: str,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    features: Any,\n    *,\n    seed: int,\n    transition_weight: float = TRANSITION_WEIGHT,\n    loss_variant: str = "l1",\n    component_names: tuple[str, ...] = tuple(WEIGHTS),\n    max_train_rows: int | None = None,\n    fast_screen: bool = False,\n) -> tuple[pd.DataFrame, pd.DataFrame | None, pd.DataFrame]:\n    blend_weights = {name: WEIGHTS[name] for name in component_names}\n    base_components: dict[str, list[pd.DataFrame]] = {key: [] for key in blend_weights}\n    specialist_components: dict[str, list[pd.DataFrame]] = {key: [] for key in blend_weights}\n    gate_rows: list[pd.DataFrame] = []\n    audits: list[dict[str, Any]] = []\n    for fold_number, (validation_date, train, validation) in enumerate(folds):\n        base = fit_components(\n            train,\n            validation,\n            features,\n            policy=policy,\n            fold_number=fold_number,\n            seed=seed,\n            transition_weight=transition_weight,\n            loss_variant=loss_variant,\n            component_names=component_names,\n            max_train_rows=max_train_rows,\n            fast_screen=fast_screen,\n        )\n        for name, frame in base.items():\n            base_components[name].append(frame)\n        if policy != "specialist_gate":\n            continue\n        specialist = fit_components(\n            train, validation, features, policy="transition_specialist", fold_number=fold_number, seed=seed + 1000,\n            loss_variant=loss_variant,\n            component_names=component_names,\n            max_train_rows=max_train_rows,\n            fast_screen=fast_screen,\n        )\n        for name, frame in specialist.items():\n            specialist_components[name].append(frame)\n        probability = fit_probability_gate(train, validation, features, seed=seed + 2000 + fold_number)\n        gate = pd.DataFrame(\n            {\n                "event_id": validation["event_id"].to_numpy(),\n                "snapshot_time": validation["snapshot_time"].to_numpy(),\n                "low_probability": probability,\n                "gate_applied": gate_eligible(validation).to_numpy() & (probability >= GATE_THRESHOLD),\n            }\n        )\n        gate_rows.append(gate)\n        audits.append(\n            {\n                "validation_date": validation_date,\n                "train_rows": int(len(train)),\n                "transition_training_rows": int(transition_label(train).sum()),\n                "transition_training_events": int(train.loc[transition_label(train), "event_id"].nunique()),\n                "validation_transition_rows": int(transition_label(validation).sum()),\n                "validation_transition_events": int(validation.loc[transition_label(validation), "event_id"].nunique()),\n                "gate_rows": int(gate["gate_applied"].sum()),\n                "gate_transition_rows": int(\n                    gate.loc[transition_label(validation).to_numpy(), "gate_applied"].sum()\n                ),\n            }\n        )\n    base = combine_weighted_oof(\n        policy, {name: pd.concat(rows, ignore_index=True) for name, rows in base_components.items()}, blend_weights\n    )\n    if policy != "specialist_gate":\n        return base, None, pd.DataFrame(audits)\n    specialist = combine_weighted_oof(\n        "transition_specialist",\n        {name: pd.concat(rows, ignore_index=True) for name, rows in specialist_components.items()}, blend_weights\n    )\n    gate = pd.concat(gate_rows, ignore_index=True)\n    merged = base.merge(gate, on=["event_id", "snapshot_time"], how="left", validate="one_to_one")\n    if merged["gate_applied"].isna().any():\n        raise RuntimeError("gate predictions missing after OOF merge")\n    merged["prediction"] = np.where(\n        merged["gate_applied"],\n        merged["prediction"] + SPECIALIST_BLEND * (specialist["prediction"].to_numpy() - merged["prediction"]),\n        merged["prediction"],\n    )\n    return merged.drop(columns=["low_probability", "gate_applied"]), specialist, pd.DataFrame(audits)\n\n\ndef calibrate(predictions: pd.DataFrame, name: str) -> pd.DataFrame:\n    output = predictions.copy()\n    strict, _ = strict_forward_bias_predictions(\n        output, output["prediction"].to_numpy(float), validation_dates=list(LATEST_COMPLETE_DATES)\n    )\n    output["prediction"] = strict\n    output["variant"] = name\n    output["is_transition_case"] = transition_label(output).to_numpy()\n    output["is_gate_eligible"] = gate_eligible(output).to_numpy()\n    return output\n\n\ndef metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pooled_rows: list[dict[str, Any]] = []\n    route_rows: list[dict[str, Any]] = []\n    scopes = {\n        "all_complete": lambda frame: pd.Series(True, index=frame.index),\n        "emerging_low_11_25_to_low10": transition_label,\n        "current_seats_le25": lambda frame: frame["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX),\n        "gate_eligible_11_25_gap3_20": gate_eligible,\n    }\n    expanded = predictions.copy()\n    expanded["route_id"] = expanded["event_id"].str.split("::", n=1).str[0]\n    for variant, frame in expanded.groupby("variant", sort=False):\n        for scope, rule in scopes.items():\n            selected = frame.loc[rule(frame)]\n            if selected.empty:\n                continue\n            pooled_rows.append({"variant": variant, "scope": scope, **required_metrics(selected)})\n            for route_id, route in selected.groupby("route_id", sort=True):\n                route_rows.append({"variant": variant, "scope": scope, "route_id": route_id, "route_name": DEFAULT_ROUTES[route_id], **required_metrics(route)})\n    pooled = pd.DataFrame(pooled_rows)\n    by_route = pd.DataFrame(route_rows)\n    metric_columns = ["event_balanced_mae", "low_0_10_mae", "full_accuracy", "full_recall", "full_precision", "full_f1"]\n    macro = by_route.groupby(["variant", "scope"], sort=False)[metric_columns].mean().reset_index()\n    macro.insert(2, "routes", macro.apply(lambda _: len(DEFAULT_ROUTES), axis=1))\n    return pooled, by_route, macro\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="11~25석 전환 저잔여 보정 실험")\n    parser.add_argument("--featured-cache", type=Path, default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"))\n    parser.add_argument("--database", type=Path, default=Path("data/gbis_api_cache.sqlite3"))\n    parser.add_argument("--output-dir", type=Path, default=Path("analysis/emerging_low_correction_results"))\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument(\n        "--include-lightgbm-huber",\n        action="store_true",\n        help="add a controlled Huber ablation that replaces only LightGBM\'s L1 objective",\n    )\n    parser.add_argument(\n        "--fast-screen",\n        action="store_true",\n        help="screen candidates with HGB/LightGBM capped at 150 trees; not a final comparison",\n    )\n    parser.add_argument(\n        "--only-transition-weighted",\n        action="store_true",\n        help="compare only the transition-weighted L1 and Huber variants",\n    )\n    parser.add_argument(\n        "--max-train-rows",\n        type=int,\n        help="deterministic route × low-seat stratified training-row cap for a screening run",\n    )\n    parser.add_argument(\n        "--skip-specialist",\n        action="store_true",\n        help="skip the probability-gated specialist variants",\n    )\n    parser.add_argument(\n        "--components",\n        nargs="+",\n        choices=tuple(WEIGHTS),\n        default=list(WEIGHTS),\n        help="components to train; use lightgbm alone for a low-resource loss ablation",\n    )\n    args = parser.parse_args()\n    component_names = tuple(args.components)\n    data = pd.read_pickle(args.featured_cache)\n    source_cutoff = data.attrs.get("source_cutoff")\n    if not source_cutoff:\n        raise ValueError("featured cache lacks source_cutoff metadata")\n    with sqlite3.connect(f"file:{args.database.resolve()}?mode=ro", uri=True) as connection:\n        source_newest = connection.execute("SELECT max(observed_at) FROM location_history").fetchone()[0]\n    if args.max_train_rows is not None:\n        # Keep both complete held-out dates intact.  Only their shared history\n        # is downsampled, which prevents the screening run from changing the\n        # evaluation population while fitting within a constrained machine.\n        first_validation = min(LATEST_COMPLETE_DATES)\n        history = data.loc[data["date"].lt(first_validation)]\n        held_out = data.loc[data["date"].isin(LATEST_COMPLETE_DATES)]\n        data = pd.concat(\n            [stratified_row_sample(history, args.max_train_rows, seed=args.seed), held_out],\n            ignore_index=True,\n        )\n    features = pooled_features()\n    folds = complete_folds(data)\n    outputs: list[pd.DataFrame] = []\n    audits: list[pd.DataFrame] = []\n    variants = [\n        ("whole_data", "baseline"),\n        ("transition_weighted", "transition_weighted"),\n    ]\n    if args.only_transition_weighted:\n        variants = [("transition_weighted", "transition_weighted")]\n    if not args.skip_specialist:\n        variants.append(("specialist_gate", "specialist_probability_gate"))\n    if args.include_lightgbm_huber:\n        huber_variants = [\n            ("whole_data", "baseline_lightgbm_huber"),\n            ("transition_weighted", "transition_weighted_lightgbm_huber"),\n        ]\n        if args.only_transition_weighted:\n            huber_variants = [huber_variants[1]]\n        variants.extend(huber_variants)\n    for policy, variant in variants:\n        loss_variant = "lightgbm_huber" if variant.endswith("_lightgbm_huber") else "l1"\n        prediction, _, audit = blend_by_fold(\n            policy,\n            folds,\n            features,\n            seed=args.seed,\n            loss_variant=loss_variant,\n            component_names=component_names,\n            max_train_rows=None,\n            fast_screen=args.fast_screen,\n        )\n        outputs.append(calibrate(prediction, variant))\n        if not audit.empty:\n            audit["variant"] = variant\n            audits.append(audit)\n    predictions = pd.concat(outputs, ignore_index=True)\n    pooled, by_route, macro = metric_tables(predictions)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")\n    pooled.to_csv(args.output_dir / "metrics_pooled.csv", index=False)\n    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)\n    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)\n    pd.concat(audits, ignore_index=True).to_csv(args.output_dir / "gate_audit.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": source_cutoff,\n            "source_newest_observation_at_refresh": source_newest,\n            "included_routes": DEFAULT_ROUTES,\n            "excluded_routes": "historical-only routes not actively collected at source cutoff",\n            "selection_dates": list(LATEST_COMPLETE_DATES),\n            "partial_dates_excluded": ["2026-08-13", "2026-08-14"],\n            "evaluation": "all pre-arrival snapshots, event-balanced metrics",\n            "transition_case": f"current seats {CURRENT_SEAT_MIN}-{CURRENT_SEAT_MAX} and arrival seats <=10",\n            "gate": f"current seats {CURRENT_SEAT_MIN}-{CURRENT_SEAT_MAX}, stop gap {GATE_STOP_GAP_MIN}-{GATE_STOP_GAP_MAX}, low probability >= {GATE_THRESHOLD}, specialist blend={SPECIALIST_BLEND}",\n            "policies": [variant for _, variant in variants],\n            "loss_variants": {\n                "l1": "HGB absolute_error, ExtraTrees squared-error default, LightGBM regression_l1",\n                "lightgbm_huber": "same ensemble; only LightGBM objective=huber, alpha=0.9",\n            },\n            "components": list(component_names),\n            "max_train_rows": args.max_train_rows,\n            "fast_screen": args.fast_screen,\n            "only_transition_weighted": args.only_transition_weighted,\n        },\n        "pooled_metrics": pooled.to_dict(orient="records"),\n        "route_macro_metrics": macro.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8")\n    print("\\nPooled metrics\\n", pooled.to_string(index=False))\n    print("\\nRoute-macro metrics\\n", macro.to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'analysis/previous_bus_feature_audit.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\n\ndef weighted_mean(values: pd.Series, weights: pd.Series) -> float:\n    return float(np.average(values.to_numpy(dtype=float), weights=weights))\n\n\ndef event_weights(data: pd.DataFrame) -> pd.Series:\n    counts = data.groupby("event_id")["event_id"].transform("size")\n    return 1.0 / counts\n\n\ndef previous_bus_profile(data: pd.DataFrame) -> dict[str, Any]:\n    available = data["previous_bus_departure_seats"].notna()\n    age = data.loc[available, "previous_bus_departure_age_minutes"]\n    low = data["label_seats"].le(5)\n    weights = event_weights(data)\n    reference_counts = data.groupby("event_id")["previous_bus_trip_id"].nunique()\n    event_count = int(data["event_id"].nunique())\n    return {\n        "rows": int(len(data)),\n        "events": event_count,\n        "row_coverage": float(available.mean()),\n        "event_balanced_coverage": weighted_mean(available.astype(float), weights),\n        "low_0_5_row_coverage": float(available.loc[low].mean()),\n        "departure_seats_median": float(\n            data.loc[available, "previous_bus_departure_seats"].median()\n        ),\n        "age_minutes": {\n            "p10": float(age.quantile(0.1)),\n            "median": float(age.median()),\n            "p90": float(age.quantile(0.9)),\n            "p95": float(age.quantile(0.95)),\n            "share_over_30": float(age.gt(30).mean()),\n            "share_over_60": float(age.gt(60).mean()),\n        },\n        "events_with_reference_change_share": float(reference_counts.gt(1).sum())\n        / event_count,\n        "events_with_at_least_3_references_share": float(\n            reference_counts.ge(3).sum()\n        )\n        / event_count,\n        "same_trip_references": int(\n            data["previous_bus_trip_id"].astype("string").eq(\n                data["trip_id"].astype("string")\n            ).sum()\n        ),\n        "negative_age_rows": int(age.lt(0).sum()),\n        "age_over_180_rows": int(age.gt(180).sum()),\n    }\n\n\ndef grouped_profile(data: pd.DataFrame, group: pd.Series) -> pd.DataFrame:\n    work = data.assign(_group=group)\n    rows: list[dict[str, Any]] = []\n    for name, frame in work.groupby("_group", observed=True, sort=True):\n        available = frame["previous_bus_departure_seats"].notna()\n        age = frame.loc[available, "previous_bus_departure_age_minutes"]\n        rows.append(\n            {\n                "group": str(name),\n                "rows": int(len(frame)),\n                "events": int(frame["event_id"].nunique()),\n                "row_coverage": float(available.mean()),\n                "age_median_minutes": float(age.median()),\n                "age_p90_minutes": float(age.quantile(0.9)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="직전 목표정류장 출발 피처 감사")\n    parser.add_argument(\n        "--cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    data = pd.read_pickle(args.cache)\n\n    summary = previous_bus_profile(data)\n    by_date = grouped_profile(data, data["date"])\n    stop_band = pd.cut(\n        data["target_stop_gap"],\n        bins=[0, 2, 5, 10, 20, np.inf],\n        labels=["1-2", "3-5", "6-10", "11-20", "21+"],\n    )\n    by_stop = grouped_profile(data, stop_band)\n    by_date.to_csv(args.output_dir / "previous_bus_profile_by_date.csv", index=False)\n    by_stop.to_csv(\n        args.output_dir / "previous_bus_profile_by_stop_band.csv", index=False\n    )\n    (args.output_dir / "previous_bus_feature_profile.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/build_flat_standalone_colab_notebooks.py': '"""Generate genuinely single-file Colab notebooks.\n\nAll project implementation is flattened into a normal notebook code cell.\nThe generated notebooks contain no project-module imports, source bundles, or\nruntime source-file extraction.\n"""\n\nfrom __future__ import annotations\n\nimport ast\nimport json\nfrom pathlib import Path\n\n\nROOT = Path(__file__).resolve().parents[1]\nANALYSIS = ROOT / "analysis"\nOUTPUT = ROOT / "colab"\nENTRY_MODULES = (\n    "route_specific_feature_experiment",\n    "pooled_main_model_overfit_ablation",\n    "pooled_main_model_registry",\n    "export_exact_primary_colab_model",\n    "export_exact_weighted_colab_model",\n)\n\n\ndef module_path(name: str) -> Path | None:\n    path = ANALYSIS / f"{name}.py"\n    return path if path.is_file() else None\n\n\ndef dependencies(name: str) -> set[str]:\n    path = module_path(name)\n    if path is None:\n        return set()\n    tree = ast.parse(path.read_text(encoding="utf-8"))\n    output: set[str] = set()\n    for node in ast.walk(tree):\n        if isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:\n            if module_path(node.module):\n                output.add(node.module)\n    return output\n\n\ndef dependency_order() -> list[str]:\n    visited: set[str] = set()\n    visiting: set[str] = set()\n    ordered: list[str] = []\n\n    def visit(name: str) -> None:\n        if name in visited:\n            return\n        if name in visiting:\n            # All project imports occur inside functions except harmless shared\n            # constants. The remaining cyclic edge is resolved by the global\n            # namespace once definitions are evaluated.\n            return\n        visiting.add(name)\n        for dependency in sorted(dependencies(name)):\n            visit(dependency)\n        visiting.remove(name)\n        visited.add(name)\n        ordered.append(name)\n\n    for entry in ENTRY_MODULES:\n        visit(entry)\n    return ordered\n\n\nclass FlattenProjectImports(ast.NodeTransformer):\n    def __init__(self, module_name: str) -> None:\n        self.module_name = module_name\n\n    def visit_ImportFrom(self, node: ast.ImportFrom):  # noqa: N802\n        if node.module == "__future__":\n            return None\n        if node.level == 0 and node.module and module_path(node.module):\n            aliases = []\n            for item in node.names:\n                source = ast.Name(id=item.name, ctx=ast.Load())\n                target = ast.Name(id=item.asname or item.name, ctx=ast.Store())\n                aliases.append(ast.Assign(targets=[target], value=source))\n            return aliases or None\n        return node\n\n    def visit_If(self, node: ast.If):  # noqa: N802\n        # Do not run any CLI entry point while evaluating notebook definitions.\n        if (\n            isinstance(node.test, ast.Compare)\n            and isinstance(node.test.left, ast.Name)\n            and node.test.left.id == "__name__"\n        ):\n            return None\n        return self.generic_visit(node)\n\n    def visit_FunctionDef(self, node: ast.FunctionDef):  # noqa: N802\n        if node.name == "main" and self.module_name.startswith("export_exact_"):\n            suffix = "primary" if "primary" in self.module_name else "weighted"\n            node.name = f"export_{suffix}_main"\n        return self.generic_visit(node)\n\n\ndef flat_source() -> str:\n    parts = [\n        "# Project implementation flattened into this notebook.\\n"\n        "# It intentionally has no project-module imports.\\n"\n    ]\n    source_paths = [("gbis_client_cache", ROOT / "gbis_client" / "cache.py")]\n    source_paths.extend((name, module_path(name)) for name in dependency_order())\n    for name, path in source_paths:\n        tree = ast.parse(path.read_text(encoding="utf-8"))  # type: ignore[union-attr]\n        tree = FlattenProjectImports(name).visit(tree)\n        ast.fix_missing_locations(tree)\n        parts.append("\\n# ---- embedded implementation section ----\\n")\n        parts.append(ast.unparse(tree))\n        parts.append("\\n")\n    return "\\n".join(parts)\n\n\ndef cell(kind: str, source: str) -> dict:\n    base = {"cell_type": kind, "metadata": {}, "source": source.splitlines(keepends=True)}\n    if kind == "code":\n        base.update({"execution_count": None, "outputs": []})\n    return base\n\n\ndef setup() -> str:\n    return \'\'\'# Standalone runtime setup: only public Python packages and GBIS credentials are used.\nAPI_BASE_URL = "https://161.33.212.6"\nDRIVE_CACHE_PATH = "/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3"\n\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nsubprocess.run([\n    sys.executable, "-m", "pip", "install", "-q",\n    "joblib>=1.4,<2", "numpy>=2.0,<3", "pandas>=2.2,<4",\n    "scikit-learn>=1.5,<2", "lightgbm>=4.5,<5", "httpx>=0.27,<1",\n], check=True)\nfrom google.colab import drive, userdata\nAPI_KEY = userdata.get("GBIS_API_KEY")\nif not API_KEY:\n    raise RuntimeError("Colab Secrets에 GBIS_API_KEY를 등록하고 Notebook access를 켜주세요.")\ndrive.mount("/content/drive")\nLOCAL_CACHE = Path("/content/gbis_api_cache.sqlite3")\nDRIVE_CACHE = Path(DRIVE_CACHE_PATH)\nDRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)\nCACHE_RESTORED = DRIVE_CACHE.is_file()\nif CACHE_RESTORED:\n    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)\n\'\'\'\n\n\nRAW = \'\'\'# Raw GBIS collection (all six actively collected routes)\ncache = GBISApiCache(base_url=API_BASE_URL, api_key=API_KEY, cache_path=LOCAL_CACHE)\ntry:\n    cache.refresh_routes()\n    for index, route_id in enumerate(DEFAULT_ROUTES, 1):\n        print(f"[sync {index}/6] {DEFAULT_ROUTES[route_id]} ({route_id})", flush=True)\n        cache.refresh_stations(route_id)\n        cache.refresh_full_history(route_id)\n    cache.refresh_latest()\n    routes_df = cache.routes_df()\n    stations_df = cache.stations_df()\n    latest_df = cache.latest_locations_df()\n    history_df = cache.history_df()\nfinally:\n    cache.close()\n    if LOCAL_CACHE.is_file():\n        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)\n\nsource_cutoff = str(history_df["observed_at"].max())\nprint(f"raw history rows: {len(history_df):,}; source cutoff: {source_cutoff}")\n\'\'\'\n\n\nFEATURES = \'\'\'# Raw SQLite history -> pooled strict-prior model features\nimport json\nimport pandas as pd\n\nfeature_dir = Path("/content/arrival_seat_feature_cache")\nfor route_id in DEFAULT_ROUTES:\n    print(f"[features] {DEFAULT_ROUTES[route_id]} ({route_id})", flush=True)\n    snapshots, flows, metadata = build_route_snapshots(LOCAL_CACHE, route_id, source_cutoff=source_cutoff)\n    snapshot_path, flow_path, metadata_path = cache_paths(feature_dir, route_id)\n    feature_dir.mkdir(parents=True, exist_ok=True)\n    snapshots.to_pickle(snapshot_path)\n    flows.to_pickle(flow_path)\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False), encoding="utf-8")\npooled_features, route_metadata = prepare_pooled_data(feature_dir, source_cutoff=source_cutoff)\npooled_features.attrs["source_cutoff"] = source_cutoff\npooled_features.attrs["route_metadata"] = route_metadata\nfeatured_cache = Path("/content/pooled_features.pkl")\npooled_features.to_pickle(featured_cache)\nprint(f"prepared rows: {len(pooled_features):,}; events: {pooled_features.event_id.nunique():,}")\n\'\'\'\n\n\ndef notebook(cells: list[dict], name: str) -> dict:\n    return {\n        "cells": cells,\n        "metadata": {"colab": {"name": name}, "kernelspec": {"display_name": "Python 3", "name": "python3"}, "language_info": {"name": "python"}},\n        "nbformat": 4,\n        "nbformat_minor": 5,\n    }\n\n\ndef build() -> None:\n    flattened = flat_source()\n    release = notebook([\n        cell("markdown", "# 도착 전 잔여좌석 예측 — 단일 Standalone Colab\\n\\n이 ipynb 파일 자체에 GBIS 수집, 전처리, 피처 생성, 모델 추론 구현을 모두 포함합니다. 프로젝트 파일·GitHub·별도 Python 모듈은 사용하지 않습니다.\\n"),\n        cell("code", setup()),\n        cell("code", flattened),\n        cell("code", RAW),\n        cell("code", FEATURES),\n        cell("code", \'\'\'from google.colab import files\nimport joblib\nimport numpy as np\n\nfiles.upload()\nprimary = joblib.load("arrival_seat_primary_v1.pkl")\nweighted = joblib.load("arrival_seat_low25_weighted_v1.pkl")\nif weighted.get("training_policy", {}).get("sampling") != "none":\n    raise ValueError("샘플링된 weighted artifact는 사용할 수 없습니다.")\n\'\'\'),\n        cell("code", \'\'\'def add_weighted_features(frame):\n    out = frame.copy()\n    ceiling = np.where(out["snapshot_low_plate_cat"].astype(str).eq("2"), 70.0, 44.0)\n    out["observed_ceiling_capacity"] = ceiling\n    out["observed_ceiling_load_ratio"] = (1 - out["snapshot_remaining_seats"].to_numpy(float) / ceiling).clip(0, 1)\n    out["observed_ceiling_load_gap"] = out["observed_ceiling_load_ratio"] * out["target_stop_gap"].to_numpy(float)\n    return out\n\ndef predict_artifact(payload, frame):\n    prepared = apply_pooled_feature_profile(frame, payload["feature_profile"])\n    if payload.get("model_variant") == "low25_transition_weighted_specialist":\n        prepared = add_weighted_features(prepared)\n    prediction = np.zeros(len(prepared), dtype=float)\n    for name, model in payload["models"].items():\n        prediction += payload["ensemble_weights"][name] * decode_target(\n            model.predict(prepared[payload["feature_columns"]]), prepared, payload["component_target_kinds"][name]\n        )\n    return np.clip(prediction, 0, prepared["capacity"].to_numpy(float))\n\ninference_rows = pooled_features.loc[pooled_features["date"].gt(primary["training_cutoff_date"])].copy()\nif inference_rows.empty:\n    raise RuntimeError("학습 cutoff 이후 원시 이력이 없습니다.")\ninference_rows["predicted_arrival_seats"] = predict_artifact(primary, inference_rows)\nlow25 = inference_rows["snapshot_remaining_seats"].le(25)\ninference_rows.loc[low25, "predicted_arrival_seats"] = predict_artifact(weighted, inference_rows.loc[low25])\ninference_rows["predicted_arrival_seats_rounded"] = inference_rows["predicted_arrival_seats"].round().astype(int)\ninference_rows.to_csv("/content/arrival_seat_predictions.csv", index=False)\ndisplay(inference_rows[["route_name", "vehicle_id", "snapshot_time", "snapshot_remaining_seats", "predicted_arrival_seats_rounded"]].head(20))\nfiles.download("/content/arrival_seat_predictions.csv")\n\'\'\'),\n    ], "arrival_seat_model_release.ipynb")\n    training = notebook([\n        cell("markdown", "# 도착 전 잔여좌석 모델 학습 — 단일 Standalone Colab\\n\\n이 ipynb 하나가 raw GBIS 수집부터 정확한 전체 학습 artifact 두 개 생성까지 수행합니다. 샘플링·경량 모델은 사용하지 않습니다.\\n"),\n        cell("code", setup() + \'\\nTRAINING_CUTOFF_DATE = "2026-08-12"\\n\'),\n        cell("code", flattened),\n        cell("code", RAW),\n        cell("code", FEATURES),\n        cell("code", \'\'\'import sys\nfrom google.colab import files\nimport joblib\n\nprimary_output = Path("/content/arrival_seat_primary_v1.pkl")\nsys.argv = ["primary", "--featured-cache", str(featured_cache), "--cache-dir", str(feature_dir), "--output", str(primary_output), "--training-cutoff-date", TRAINING_CUTOFF_DATE, "--source-cutoff", source_cutoff]\nif export_primary_main() != 0:\n    raise RuntimeError("주 모델 학습 실패")\nweighted_output = Path("/content/arrival_seat_low25_weighted_v1.pkl")\nsys.argv = ["weighted", "--featured-cache", str(featured_cache), "--cache-dir", str(feature_dir), "--output", str(weighted_output), "--training-cutoff-date", TRAINING_CUTOFF_DATE, "--source-cutoff", source_cutoff]\nif export_weighted_main() != 0:\n    raise RuntimeError("가중 모델 학습 실패")\nfor output in (primary_output, weighted_output):\n    artifact = joblib.load(output)\n    if artifact.get("training_policy", {}).get("sampling") != "none":\n        raise ValueError(f"{output.name}: 샘플링 artifact는 허용하지 않습니다.")\n    print(output.name, artifact["model_id"], artifact["training_rows"])\n    files.download(str(output))\n\'\'\'),\n    ], "arrival_seat_model_training.ipynb")\n    (OUTPUT / "arrival_seat_model_release.ipynb").write_text(json.dumps(release, ensure_ascii=False, indent=1), encoding="utf-8")\n    (OUTPUT / "arrival_seat_model_training.ipynb").write_text(json.dumps(training, ensure_ascii=False, indent=1), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    build()\n', 'analysis/low25_transition_weight_experiment.py': '"""Compare the full baseline with a <=25-current-seat transition-weighted model."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sqlite3\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom emerging_low_correction_experiment import (\n    CURRENT_SEAT_MAX,\n    TRANSITION_WEIGHT,\n    calibrate,\n    complete_folds,\n    metric_tables,\n    pooled_features,\n    transition_label,\n    blend_by_fold,\n)\nfrom latest_main_model_feature_recheck import LATEST_COMPLETE_DATES\nfrom linear_feature_experiment import json_ready\nfrom route_specific_feature_experiment import DEFAULT_ROUTES\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="전체 baseline 대 현재 25석 이하 전환 가중 모델 비교"\n    )\n    parser.add_argument(\n        "--featured-cache",\n        type=Path,\n        default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"),\n    )\n    parser.add_argument("--database", type=Path, default=Path("data/gbis_api_cache.sqlite3"))\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/low25_transition_weight_results"),\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.featured_cache)\n    source_cutoff = data.attrs.get("source_cutoff")\n    if not source_cutoff:\n        raise ValueError("featured cache lacks source_cutoff metadata")\n    with sqlite3.connect(f"file:{args.database.resolve()}?mode=ro", uri=True) as connection:\n        source_newest = connection.execute("SELECT max(observed_at) FROM location_history").fetchone()[0]\n    folds = complete_folds(data)\n    features = pooled_features()\n\n    baseline_raw, _, _ = blend_by_fold(\n        "whole_data", folds, features, seed=args.seed\n    )\n    low25_raw, _, _ = blend_by_fold(\n        "low25_transition_weighted", folds, features, seed=args.seed\n    )\n    keys = ["event_id", "snapshot_time"]\n    baseline_index = pd.MultiIndex.from_frame(baseline_raw[keys])\n    low25_index = pd.MultiIndex.from_frame(low25_raw[keys])\n    if not baseline_index.equals(low25_index):\n        raise RuntimeError("baseline and low25 OOF rows are not aligned")\n\n    mixed_raw = low25_raw.copy()\n    use_low25 = mixed_raw["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX)\n    mixed_raw["prediction"] = np.where(\n        use_low25,\n        low25_raw["prediction"].to_numpy(float),\n        baseline_raw["prediction"].to_numpy(float),\n    )\n    predictions = pd.concat(\n        [\n            calibrate(baseline_raw, "baseline"),\n            calibrate(mixed_raw, "low25_transition_weighted_mixture"),\n        ],\n        ignore_index=True,\n    )\n    pooled, by_route, macro = metric_tables(predictions)\n    audit_rows = []\n    for validation_date, train, validation in folds:\n        low25_train = train["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX)\n        low25_validation = validation["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX)\n        transition_train = transition_label(train)\n        audit_rows.append(\n            {\n                "validation_date": validation_date,\n                "train_rows": int(len(train)),\n                "low25_train_rows": int(low25_train.sum()),\n                "low25_train_events": int(train.loc[low25_train, "event_id"].nunique()),\n                "weighted_transition_rows": int((low25_train & transition_train).sum()),\n                "validation_rows": int(len(validation)),\n                "low25_validation_rows": int(low25_validation.sum()),\n                "low25_validation_events": int(validation.loc[low25_validation, "event_id"].nunique()),\n                "validation_transition_rows": int(transition_label(validation).sum()),\n            }\n        )\n    audit = pd.DataFrame(audit_rows)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")\n    pooled.to_csv(args.output_dir / "metrics_pooled.csv", index=False)\n    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)\n    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)\n    audit.to_csv(args.output_dir / "low25_training_audit.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": source_cutoff,\n            "source_newest_observation_at_refresh": source_newest,\n            "included_routes": DEFAULT_ROUTES,\n            "excluded_routes": "historical-only routes not actively collected at source cutoff",\n            "selection_dates": list(LATEST_COMPLETE_DATES),\n            "partial_dates_excluded": ["2026-08-13", "2026-08-14"],\n            "evaluation": "all pre-arrival snapshots; event-balanced metrics",\n            "baseline": "all training rows; deployed-family weighted ensemble",\n            "candidate_training": f"only current seats <= {CURRENT_SEAT_MAX}; transition rows (current 11-25 and arrival <=10) have {TRANSITION_WEIGHT:g}x extra sample weight",\n            "candidate_serving": f"candidate prediction when current seats <= {CURRENT_SEAT_MAX}; baseline prediction otherwise",\n        },\n        "pooled_metrics": pooled.to_dict(orient="records"),\n        "route_macro_metrics": macro.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print("\\nPooled metrics\\n", pooled.to_string(index=False))\n    print("\\nRoute macro metrics\\n", macro.to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'analysis/seat_service_model.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom collections.abc import Mapping\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport all_prearrival_seat_regression as prearrival_module\nimport hypothesis_model_search as hypothesis_module\nimport route_distribution as route_distribution_module\n\nfrom all_prearrival_seat_regression import (\n    DYNAMIC_ALL_PREARRIVAL,\n    ENGINEERED_ALL_PREARRIVAL,\n    clip_seats,\n)\nfrom hypothesis_model_search import (\n    CAPACITY_44_ALL_PREARRIVAL,\n    PAIR_ALL_PREARRIVAL,\n    PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,\n    NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,\n    PREVIOUS_BUS_SEAT_ALL_PREARRIVAL,\n    PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL,\n    ROUTE_PROFILE_ALL_PREARRIVAL,\n    Candidate,\n    EnsembleSpec,\n    add_observed_capacity_features,\n    add_source_target_pair,\n    decode_target,\n    ensemble_prediction,\n    formula_prediction,\n    route_profile_values,\n)\nfrom route_distribution import (\n    ROUTE_DISTRIBUTION_FEATURES,\n    apply_interval_policy,\n    distribution_flow_fingerprint,\n    file_sha256,\n    point_model_input_provenance_sha256,\n    route_distribution_values,\n    validate_interval_policy,\n)\n\n\nFEATURE_SETS = {\n    "dynamic": DYNAMIC_ALL_PREARRIVAL,\n    "previous_bus": ENGINEERED_ALL_PREARRIVAL,\n    "previous_bus_seat": PREVIOUS_BUS_SEAT_ALL_PREARRIVAL,\n    "previous_bus_seat_capacity_44_70": PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL,\n    "previous_bus_normalized_capacity_44_70": (\n        NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL\n    ),\n    "previous_bus_capacity_44_70": PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,\n    "source_target_pair": PAIR_ALL_PREARRIVAL,\n    "capacity_44_70": CAPACITY_44_ALL_PREARRIVAL,\n    "route_profile": ROUTE_PROFILE_ALL_PREARRIVAL,\n}\n\n\ndef read_model_artifact_bundle(\n    results_dir: Path,\n    summary: Mapping[str, Any],\n    *,\n    required: bool = False,\n) -> dict[Path, bytes]:\n    """Read the exact point-model bytes bound by the summary manifest once."""\n\n    deployment = summary.get("deployment_constraint")\n    raw_manifest = (\n        deployment.get("saved_artifact_sha256")\n        if isinstance(deployment, Mapping)\n        else None\n    )\n    if raw_manifest is None:\n        if required:\n            raise ValueError("model summary에 saved artifact SHA manifest가 없습니다.")\n        return {}\n    if not isinstance(raw_manifest, Mapping) or not raw_manifest:\n        raise ValueError("model saved artifact SHA manifest가 비어 있거나 객체가 아닙니다.")\n    declared: dict[Path, str] = {}\n    for raw_name, raw_sha256 in raw_manifest.items():\n        name = str(raw_name)\n        digest = str(raw_sha256)\n        if Path(name).name != name or not name.endswith((".joblib", ".metadata.json")):\n            raise ValueError(f"model artifact manifest 파일명이 올바르지 않습니다: {name}")\n        if len(digest) != 64 or any(\n            character not in "0123456789abcdef" for character in digest\n        ):\n            raise ValueError(f"model artifact manifest SHA가 올바르지 않습니다: {name}")\n        declared[Path(results_dir) / name] = digest\n    actual_names = {\n        path.name\n        for pattern in ("*.joblib", "*.metadata.json")\n        for path in Path(results_dir).glob(pattern)\n    }\n    declared_names = {path.name for path in declared}\n    if actual_names != declared_names:\n        raise ValueError(\n            "model artifact manifest와 results directory 파일 집합이 다릅니다: "\n            f"missing={sorted(declared_names - actual_names)}, "\n            f"unexpected={sorted(actual_names - declared_names)}"\n        )\n    payloads: dict[Path, bytes] = {}\n    for path, expected_sha256 in declared.items():\n        if not path.is_file():\n            raise ValueError(f"model artifact가 없습니다: {path}")\n        payload = path.read_bytes()\n        if hashlib.sha256(payload).hexdigest() != expected_sha256:\n            raise ValueError(f"model artifact가 summary SHA와 다릅니다: {path}")\n        payloads[path] = payload\n    return payloads\n\n\ndef validate_model_artifact_manifest(\n    results_dir: Path,\n    summary: Mapping[str, Any],\n    *,\n    required: bool = False,\n) -> dict[Path, str]:\n    """Verify and return hashes for the exact saved point-model bundle."""\n\n    return {\n        path: hashlib.sha256(payload).hexdigest()\n        for path, payload in read_model_artifact_bundle(\n            results_dir, summary, required=required\n        ).items()\n    }\n\n\ndef validate_model_source_manifest(\n    summary: Mapping[str, Any],\n    *,\n    required: bool = False,\n) -> dict[str, str]:\n    """Reject mixing frozen model artifacts with changed inference semantics."""\n\n    protocol = summary.get("protocol")\n    protocol = protocol if isinstance(protocol, Mapping) else {}\n    expected_hypothesis = protocol.get("experiment_source_sha256")\n    expected_inference = protocol.get("inference_source_sha256")\n    if expected_hypothesis is None and expected_inference is None:\n        if required:\n            raise ValueError("model summary에 inference source SHA manifest가 없습니다.")\n        return {}\n    if not isinstance(expected_inference, Mapping):\n        if required:\n            raise ValueError("model summary의 inference_source_sha256가 없습니다.")\n        # Legacy summaries recorded only the experiment script.  That digest was\n        # never an inference-runtime contract, so changing today\'s loader must\n        # not make those otherwise supported bundles unloadable.  Prospective\n        # evaluation passes required=True and therefore accepts only the new,\n        # complete source manifest.\n        return {}\n    expected_inference_names = {\n        "seat_service_model.py",\n        "all_prearrival_seat_regression.py",\n    }\n    actual_inference_names = {str(key) for key in expected_inference}\n    if actual_inference_names != expected_inference_names:\n        raise ValueError(\n            "model summary의 inference source 파일 집합이 고정 계약과 다릅니다: "\n            f"expected={sorted(expected_inference_names)}, "\n            f"actual={sorted(actual_inference_names)}"\n        )\n    current = {\n        "hypothesis_model_search.py": file_sha256(Path(hypothesis_module.__file__)),\n        "seat_service_model.py": file_sha256(Path(__file__)),\n        "all_prearrival_seat_regression.py": file_sha256(\n            Path(prearrival_module.__file__)\n        ),\n    }\n    expected = {\n        "hypothesis_model_search.py": expected_hypothesis,\n        **{str(key): str(value) for key, value in expected_inference.items()},\n    }\n    for filename, digest in expected.items():\n        normalized_digest = str(digest)\n        if (\n            len(normalized_digest) != 64\n            or any(\n                character not in "0123456789abcdef"\n                for character in normalized_digest\n            )\n            or current.get(filename) != normalized_digest\n        ):\n            raise ValueError(\n                "model artifact와 현재 inference source가 다릅니다: "\n                f"{filename}"\n            )\n    return current\n\n\n@dataclass(frozen=True)\nclass LoadedComponent:\n    name: str\n    model: Any | None\n    metadata: dict[str, Any]\n\n\ndef normalize_service_input(data: pd.DataFrame) -> pd.DataFrame:\n    """공식/학습 모델이 공유하는 CSV 입력 dtype을 학습 스키마로 맞춘다."""\n    normalized = data.copy()\n    if "snapshot_time" in normalized.columns and not pd.api.types.is_datetime64_any_dtype(\n        normalized["snapshot_time"]\n    ):\n        normalized["snapshot_time"] = pd.to_datetime(\n            normalized["snapshot_time"], errors="raise"\n        )\n    # Normalize raw categorical inputs before any derived transform.  In\n    # particular, capacity engineering compares snapshot_low_plate_cat with\n    # string values, so casting only after that transform misclassifies CSV\n    # inferred integer values.\n    for column in DYNAMIC_ALL_PREARRIVAL.categorical:\n        if column in normalized.columns:\n            normalized[column] = normalized[column].astype("string")\n    return normalized\n\n\ndef prepare_service_features(\n    data: pd.DataFrame,\n    variant: str,\n    *,\n    route_flows: pd.DataFrame | None = None,\n) -> tuple[pd.DataFrame, list[str]]:\n    """학습 때와 같은 결정적 피처 변환만 서비스 입력에 적용한다."""\n    normalized = normalize_service_input(data)\n    if variant in {"dynamic", "previous_bus", "previous_bus_seat"}:\n        prepared = normalized\n    elif variant == "source_target_pair":\n        prepared = add_source_target_pair(normalized)\n    elif variant in {\n        "capacity_44_70",\n        "previous_bus_seat_capacity_44_70",\n        "previous_bus_normalized_capacity_44_70",\n        "previous_bus_capacity_44_70",\n    }:\n        prepared = add_observed_capacity_features(normalized)\n    elif variant == "route_profile":\n        if route_flows is None:\n            raise ValueError("route_profile 서비스 피처에 과거 stop flow가 필요합니다.")\n        prepared = normalized\n        values = route_profile_values(route_flows, normalized)\n        for column in ROUTE_PROFILE_ALL_PREARRIVAL.numeric:\n            if column in values.columns:\n                prepared[column] = values[column].to_numpy()\n    else:\n        raise ValueError(\n            f"서비스 변환이 저장되지 않은 feature variant입니다: {variant}"\n        )\n    columns = FEATURE_SETS[variant].columns\n    missing = sorted(set(columns) - set(prepared.columns))\n    if missing:\n        raise ValueError(f"서비스 입력 피처가 누락되었습니다: {missing}")\n    # CSV readers infer digit-only categorical values (station sequence, weekday,\n    # time bin, ...) as integers.  The fitted OneHotEncoder saw their string\n    # representation, so leaving the inferred dtype untouched silently maps every\n    # value to the unknown category.  Normalize at the model boundary so DataFrame\n    # and CSV inputs follow the exact same schema.\n    for column in FEATURE_SETS[variant].categorical:\n        prepared[column] = prepared[column].astype("string")\n    return prepared, columns\n\n\nclass SeatServiceModel:\n    """저장된 선택 앙상블로 도착 시 잔여좌석을 배치 예측한다."""\n\n    def __init__(\n        self,\n        *,\n        selected_name: str,\n        ensemble: EnsembleSpec | None,\n        components: dict[str, LoadedComponent],\n        route_flows: pd.DataFrame | None = None,\n        point_model_summary_sha256: str | None = None,\n        point_model_source_sha256: str | None = None,\n        point_model_input_sha256: str | None = None,\n        distribution_flows: pd.DataFrame | None = None,\n        interval_policy: Mapping[str, Any] | Path | None = None,\n        full_threshold_seats: float = 0.0,\n    ) -> None:\n        self.selected_name = selected_name\n        self.ensemble = ensemble\n        self.components = components\n        self.route_flows = route_flows\n        self.point_model_summary_sha256 = point_model_summary_sha256\n        self.point_model_source_sha256 = point_model_source_sha256\n        self.point_model_input_sha256 = point_model_input_sha256\n        self.distribution_flows: pd.DataFrame | None = None\n        self.interval_policy: dict[str, Any] | None = None\n        self.full_threshold_seats = float(full_threshold_seats)\n        if distribution_flows is not None or interval_policy is not None:\n            if distribution_flows is None or interval_policy is None:\n                raise ValueError(\n                    "분포 추론에는 distribution_flows와 interval_policy가 "\n                    "모두 필요합니다."\n                )\n            self.attach_distribution(\n                distribution_flows,\n                interval_policy,\n                full_threshold_seats=full_threshold_seats,\n            )\n\n    @classmethod\n    def load(\n        cls,\n        results_dir: Path,\n        *,\n        distribution_flows: pd.DataFrame | None = None,\n        interval_policy: Mapping[str, Any] | Path | None = None,\n        full_threshold_seats: float = 0.0,\n    ) -> "SeatServiceModel":\n        summary_path = results_dir / "summary.json"\n        summary_payload = summary_path.read_bytes()\n        summary = json.loads(summary_payload.decode("utf-8"))\n        summary_sha256 = hashlib.sha256(summary_payload).hexdigest()\n        validate_model_source_manifest(summary, required=False)\n        artifact_payloads = read_model_artifact_bundle(\n            results_dir, summary, required=False\n        )\n        point_model_source_sha256 = (\n            summary.get("protocol", {}).get("experiment_source_sha256")\n        )\n        try:\n            point_model_input_sha256 = point_model_input_provenance_sha256(\n                summary\n            )\n        except ValueError:\n            # Old model summaries predate provenance. They remain usable with\n            # old interval policies, which intentionally have no provenance.\n            point_model_input_sha256 = None\n        selected_name = str(summary["stage_winners"]["selected_final"])\n        raw_ensemble = summary.get("selected_ensemble")\n        ensemble = (\n            EnsembleSpec(\n                name=str(raw_ensemble["name"]),\n                kind=str(raw_ensemble["kind"]),\n                components=tuple(raw_ensemble["components"]),\n                params={\n                    str(key): float(value)\n                    for key, value in raw_ensemble["params"].items()\n                },\n                bias=float(raw_ensemble.get("bias", 0.0)),\n            )\n            if raw_ensemble is not None\n            else None\n        )\n        names = ensemble.components if ensemble is not None else (selected_name,)\n        components: dict[str, LoadedComponent] = {}\n        for name in names:\n            metadata_path = results_dir / f"{name}.metadata.json"\n            metadata_payload = artifact_payloads.get(metadata_path)\n            metadata = json.loads(\n                (\n                    metadata_payload\n                    if metadata_payload is not None\n                    else metadata_path.read_bytes()\n                ).decode("utf-8")\n            )\n            candidate = metadata["candidate"]\n            model = None\n            if candidate["model_kind"] not in {"formula", "route_formula"}:\n                model_path = results_dir / f"{name}.joblib"\n                model_payload = artifact_payloads.get(model_path)\n                model = (\n                    joblib.load(io.BytesIO(model_payload))\n                    if model_payload is not None\n                    else joblib.load(model_path)\n                )\n            components[name] = LoadedComponent(name, model, metadata)\n        route_flows = None\n        if any(\n            component.metadata["candidate"].get("model_kind") == "route_formula"\n            or component.metadata["candidate"].get("feature_variant")\n            == "route_profile"\n            for component in components.values()\n        ):\n            flow_artifact = results_dir / "route_profile_flows.joblib"\n            if not flow_artifact.is_file():\n                raise ValueError(\n                    "route 공식/피처 모델에 필요한 artifact가 없습니다: "\n                    f"{flow_artifact}"\n                )\n            flow_payload = artifact_payloads.get(flow_artifact)\n            route_flows = (\n                joblib.load(io.BytesIO(flow_payload))\n                if flow_payload is not None\n                else joblib.load(flow_artifact)\n            )\n        service = cls(\n            selected_name=selected_name,\n            ensemble=ensemble,\n            components=components,\n            route_flows=route_flows,\n            point_model_summary_sha256=summary_sha256,\n            point_model_source_sha256=point_model_source_sha256,\n            point_model_input_sha256=point_model_input_sha256,\n        )\n        if distribution_flows is not None or interval_policy is not None:\n            if distribution_flows is None or interval_policy is None:\n                raise ValueError(\n                    "분포 추론에는 distribution_flows와 interval_policy가 "\n                    "모두 필요합니다."\n                )\n            service.attach_distribution(\n                distribution_flows,\n                interval_policy,\n                full_threshold_seats=full_threshold_seats,\n            )\n        return service\n\n    @property\n    def distribution_attached(self) -> bool:\n        return (\n            self.distribution_flows is not None\n            and self.interval_policy is not None\n        )\n\n    def attach_distribution(\n        self,\n        flows: pd.DataFrame,\n        policy: Mapping[str, Any] | Path,\n        *,\n        full_threshold_seats: float = 0.0,\n    ) -> "SeatServiceModel":\n        """Attach past stop flows and a calibrated 90% interval policy.\n\n        Point predictions remain unchanged.  Once attached, ``predict_frame``\n        appends intervals, risk probabilities, and route uncertainty diagnostics.\n        """\n        if not isinstance(flows, pd.DataFrame):\n            raise TypeError("distribution flows는 pandas DataFrame이어야 합니다.")\n        flow_columns = {\n            "date",\n            "station_seq",\n            "direction",\n            "time_bin_2h",\n            "stop_net",\n        }\n        missing = sorted(flow_columns - set(flows.columns))\n        if missing:\n            raise ValueError(f"distribution flow 열이 누락되었습니다: {missing}")\n        if isinstance(policy, Path):\n            loaded_policy = json.loads(\n                policy.read_text(encoding="utf-8")\n            )\n            if not isinstance(loaded_policy, dict):\n                raise ValueError("interval policy JSON은 객체여야 합니다.")\n            raw_policy: Mapping[str, Any] = loaded_policy\n        elif isinstance(policy, Mapping):\n            raw_policy = policy\n        else:\n            raise TypeError("interval policy는 mapping 또는 JSON Path여야 합니다.")\n        threshold = float(full_threshold_seats)\n        if not np.isfinite(threshold):\n            raise ValueError("full_threshold_seats는 유한한 값이어야 합니다.")\n\n        normalized_policy = validate_interval_policy(dict(raw_policy))\n        provenance = normalized_policy.get("provenance")\n        if provenance is not None:\n            point_model = provenance["point_model"]\n            if point_model["selected_name"] != self.selected_name:\n                raise ValueError(\n                    "interval policy의 point-model 이름이 현재 모델과 "\n                    "다릅니다: "\n                    f"policy={point_model[\'selected_name\']}, "\n                    f"service={self.selected_name}"\n                )\n            expected_model_hashes = {\n                "summary_sha256": self.point_model_summary_sha256,\n                "source_sha256": self.point_model_source_sha256,\n                "input_provenance_sha256": self.point_model_input_sha256,\n            }\n            for field, expected in expected_model_hashes.items():\n                if expected is None:\n                    raise ValueError(\n                        "현재 point-model summary가 provenance 검증에 필요한 "\n                        f"{field}를 제공하지 않습니다."\n                    )\n                if point_model[field] != expected:\n                    raise ValueError(\n                        "interval policy가 다른 point-model summary에서 "\n                        f"생성되었습니다: {field} 불일치"\n                    )\n\n            distribution = provenance["distribution"]\n            expected_distribution_source = distribution["source_hashes"].get(\n                "route_distribution.py"\n            )\n            if expected_distribution_source is None:\n                raise ValueError(\n                    "interval policy provenance에 inference source "\n                    "route_distribution.py hash가 없습니다."\n                )\n            distribution_source_path = Path(route_distribution_module.__file__)\n            actual_distribution_source = file_sha256(distribution_source_path)\n            if expected_distribution_source != actual_distribution_source:\n                raise ValueError(\n                    "interval policy를 생성한 route_distribution.py와 현재 "\n                    "inference source가 다릅니다: source hash 불일치"\n                )\n            if distribution["flow_rows"] != len(flows):\n                raise ValueError(\n                    "interval policy와 distribution flow 행 수가 다릅니다: "\n                    f"policy={distribution[\'flow_rows\']}, service={len(flows)}"\n                )\n            actual_flow_sha256 = distribution_flow_fingerprint(flows)\n            if distribution["flow_data_sha256"] != actual_flow_sha256:\n                raise ValueError(\n                    "interval policy와 distribution flow fingerprint가 "\n                    "일치하지 않습니다."\n                )\n        self.distribution_flows = flows\n        self.interval_policy = normalized_policy\n        self.full_threshold_seats = threshold\n        return self\n\n    def detach_distribution(self) -> "SeatServiceModel":\n        """Return to the original point-only prediction output."""\n        self.distribution_flows = None\n        self.interval_policy = None\n        self.full_threshold_seats = 0.0\n        return self\n\n    @staticmethod\n    def _validate_prediction_columns(data: pd.DataFrame) -> None:\n        required = {"snapshot_remaining_seats", "target_stop_gap", "capacity"}\n        missing = sorted(required - set(data.columns))\n        if missing:\n            raise ValueError(f"예측에 필요한 열이 없습니다: {missing}")\n        if data.empty:\n            raise ValueError("예측할 행이 없습니다.")\n\n    def _predict_component(\n        self,\n        component: LoadedComponent,\n        data: pd.DataFrame,\n    ) -> np.ndarray:\n        candidate = component.metadata["candidate"]\n        model_kind = str(candidate.get("model_kind", ""))\n        bias = float(component.metadata.get("bias_correction", 0.0))\n        if model_kind == "formula" or candidate["name"] == "persistence":\n            normalized = normalize_service_input(data)\n            formula_candidate = Candidate(\n                name=str(candidate["name"]),\n                stage=str(candidate.get("stage", "service")),\n                why=str(candidate.get("why", "")),\n                if_works=str(candidate.get("if_works", "")),\n                if_fails=str(candidate.get("if_fails", "")),\n                model_kind="formula",\n                target_kind=str(candidate.get("target_kind", "delta")),\n                feature_variant=str(candidate.get("feature_variant", "dynamic")),\n                low_weight=float(candidate.get("low_weight", 1.0)),\n                weighting_kind=str(candidate.get("weighting_kind", "event")),\n                far_weight=float(candidate.get("far_weight", 1.0)),\n                far_threshold=int(candidate.get("far_threshold", 6)),\n                gap_weight_power=float(candidate.get("gap_weight_power", 0.0)),\n                params=dict(candidate.get("params", {})),\n            )\n            return clip_seats(\n                formula_prediction(formula_candidate, normalized) + bias,\n                normalized["capacity"],\n            )\n        if model_kind == "route_formula":\n            if self.route_flows is None:\n                raise ValueError(\n                    f"{component.name} route 공식에 과거 stop flow가 필요합니다."\n                )\n            normalized = normalize_service_input(data)\n            raw = route_profile_values(self.route_flows, normalized)[\n                "route_profile_seats"\n            ].to_numpy(dtype=float)\n            return clip_seats(raw + bias, normalized["capacity"])\n        if component.model is None:\n            raise ValueError(\n                f"{component.name} 공식 모델은 별도 상태 없이 서비스할 수 없습니다."\n            )\n        variant = str(candidate["feature_variant"])\n        prepared, default_columns = prepare_service_features(\n            data,\n            variant,\n            route_flows=self.route_flows,\n        )\n        columns = component.metadata.get("feature_columns", default_columns)\n        raw = component.model.predict(prepared[columns])\n        decoded = decode_target(raw, prepared, str(candidate["target_kind"]))\n        return clip_seats(decoded + bias, prepared["capacity"])\n\n    def predict(self, data: pd.DataFrame) -> np.ndarray:\n        self._validate_prediction_columns(data)\n        component_predictions = {\n            name: self._predict_component(component, data)\n            for name, component in self.components.items()\n        }\n        if self.ensemble is None:\n            return component_predictions[self.selected_name]\n        return ensemble_prediction(self.ensemble, component_predictions, data)\n\n    def predict_frame(self, data: pd.DataFrame) -> pd.DataFrame:\n        output = data.copy()\n        output["predicted_arrival_seats"] = self.predict(data)\n        output["predicted_arrival_seats_rounded"] = np.rint(\n            output["predicted_arrival_seats"]\n        ).astype(int)\n        output["prediction_model"] = self.selected_name\n        if self.distribution_attached:\n            assert self.distribution_flows is not None\n            assert self.interval_policy is not None\n            route_values = route_distribution_values(\n                self.distribution_flows, data\n            )\n            interval_input = data.copy()\n            for column in ROUTE_DISTRIBUTION_FEATURES:\n                interval_input[column] = route_values[column].to_numpy(dtype=float)\n            interval = apply_interval_policy(\n                interval_input,\n                output["predicted_arrival_seats"].to_numpy(dtype=float),\n                self.interval_policy,\n                full_threshold_seats=self.full_threshold_seats,\n            )\n            output["predicted_arrival_seats_lower_90"] = interval[\n                "interval_lower"\n            ].to_numpy(dtype=float)\n            output["predicted_arrival_seats_lower_90_unrefined"] = interval[\n                "interval_lower_unrefined"\n            ].to_numpy(dtype=float)\n            output["predicted_arrival_seats_upper_90"] = interval[\n                "interval_upper"\n            ].to_numpy(dtype=float)\n            output["prediction_interval_half_width"] = interval[\n                "interval_half_width"\n            ].to_numpy(dtype=float)\n            output["prediction_base_interval_nominal_half_width"] = interval[\n                "interval_half_width"\n            ].to_numpy(dtype=float)\n            output["prediction_interval_width_90"] = (\n                output["predicted_arrival_seats_upper_90"]\n                - output["predicted_arrival_seats_lower_90"]\n            )\n            output["prediction_interval_lower_distance_90"] = (\n                output["predicted_arrival_seats"]\n                - output["predicted_arrival_seats_lower_90"]\n            )\n            output["prediction_interval_upper_distance_90"] = (\n                output["predicted_arrival_seats_upper_90"]\n                - output["predicted_arrival_seats"]\n            )\n            output["prediction_lower_tail_refinement_candidate"] = interval[\n                "lower_tail_refinement_candidate"\n            ].to_numpy(dtype=float)\n            output["prediction_lower_tail_refinement_factor"] = interval[\n                "lower_tail_refinement_factor"\n            ].to_numpy(dtype=float)\n            output["prediction_lower_tail_refinement_applied"] = interval[\n                "lower_tail_refinement_applied"\n            ].to_numpy(dtype=bool)\n            output["probability_arrival_seats_le_5"] = interval[\n                "p_low_5"\n            ].to_numpy(dtype=float)\n            output["probability_arrival_seats_le_5_raw"] = interval[\n                "p_low_5_raw"\n            ].to_numpy(dtype=float)\n            output["probability_arrival_seats_le_5_calibrated"] = interval[\n                "p_low_5_calibrated"\n            ].to_numpy(dtype=float)\n            output["probability_arrival_full"] = interval["p_full"].to_numpy(\n                dtype=float\n            )\n            output["arrival_full_threshold_seats"] = self.full_threshold_seats\n            probability_calibration = self.interval_policy.get(\n                "probability_calibration"\n            )\n            output["probability_calibration_status"] = (\n                str(probability_calibration["status"])\n                if probability_calibration is not None\n                else "none"\n            )\n            output["probability_calibration_slope"] = (\n                float(probability_calibration["slope"])\n                if probability_calibration is not None\n                else 1.0\n            )\n            lower_tail_refinement = self.interval_policy.get(\n                "lower_tail_refinement"\n            )\n            output["prediction_lower_tail_refinement_status"] = (\n                str(lower_tail_refinement["status"])\n                if lower_tail_refinement is not None\n                else "none"\n            )\n            output["prediction_lower_tail_refinement_kind"] = (\n                str(lower_tail_refinement["kind"])\n                if lower_tail_refinement is not None\n                else "none"\n            )\n            if lower_tail_refinement is not None:\n                lower_tail_quantile = int(\n                    float(lower_tail_refinement["calibration_quantile"]) * 1000\n                )\n                lower_tail_shrinkage = int(\n                    float(lower_tail_refinement["shrinkage_event_count"])\n                )\n                output["prediction_lower_tail_refinement_policy"] = (\n                    f"{lower_tail_refinement[\'kind\']}_q"\n                    f"{lower_tail_quantile:03d}_k{lower_tail_shrinkage}"\n                )\n            else:\n                output["prediction_lower_tail_refinement_policy"] = "none"\n            for column in (\n                "route_dist_scale",\n                "route_dist_cell_prior_share",\n                "route_dist_cross_bin_share",\n                "route_dist_nonpass_stops",\n                "route_dist_global_fallback_share",\n            ):\n                output[column] = route_values[column].to_numpy(dtype=float)\n            calibration_quantile = float(\n                self.interval_policy.get("calibration_quantile", 0.9)\n            )\n            output["prediction_interval_policy"] = (\n                f"{self.interval_policy[\'kind\']}_q"\n                f"{int(calibration_quantile * 1000):03d}"\n            )\n        return output\n\n\ndef read_service_input(path_or_buffer: Any) -> pd.DataFrame:\n    """CSV 숫자를 tree threshold까지 손실 없이 왕복해 읽는다."""\n    return pd.read_csv(path_or_buffer, float_precision="round_trip")\n\n\ndef verify_saved_predictions(\n    results_dir: Path,\n    snapshot_cache: Path,\n    *,\n    test_date: str,\n) -> dict[str, float | int | str]:\n    snapshots = pd.read_pickle(snapshot_cache)\n    test = snapshots.loc[snapshots["date"].eq(test_date)].copy()\n    expected = pd.read_csv(results_dir / "final_test_predictions.csv")\n    if len(test) != len(expected):\n        raise ValueError(\n            f"검증 행 수가 다릅니다: cache={len(test)}, saved={len(expected)}"\n        )\n    model = SeatServiceModel.load(results_dir)\n    actual = model.predict(test)\n    target = expected["selected_prediction"].to_numpy(dtype=float)\n    difference = np.abs(actual - target)\n    return {\n        "model": model.selected_name,\n        "rows": int(len(test)),\n        "max_absolute_difference": float(difference.max()),\n        "mean_absolute_difference": float(difference.mean()),\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="도착 잔여좌석 서비스 모델 추론")\n    parser.add_argument(\n        "--results-dir",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    parser.add_argument("--input-csv", type=Path)\n    parser.add_argument("--output-csv", type=Path)\n    parser.add_argument("--verify-cache", type=Path)\n    parser.add_argument("--test-date", default="2026-08-11")\n    parser.add_argument(\n        "--distribution-flow-cache",\n        type=Path,\n        help="과거 stop flow pickle; --interval-policy와 함께 지정",\n    )\n    parser.add_argument(\n        "--interval-policy",\n        type=Path,\n        help="검증 데이터로 보정한 90%% interval policy JSON",\n    )\n    parser.add_argument(\n        "--full-threshold-seats",\n        type=float,\n        default=0.0,\n        help="P(full)에 사용할 도착 잔여좌석 임계값 (기본 0)",\n    )\n    args = parser.parse_args()\n\n    if args.verify_cache is not None:\n        result = verify_saved_predictions(\n            args.results_dir,\n            args.verify_cache,\n            test_date=args.test_date,\n        )\n        print(json.dumps(result, ensure_ascii=False, indent=2))\n        return 0\n    if args.input_csv is None or args.output_csv is None:\n        parser.error("추론에는 --input-csv와 --output-csv가 모두 필요합니다.")\n    # Preserve binary float round-trips so tree threshold decisions are identical\n    # to predictions made from the cached DataFrame.\n    data = read_service_input(args.input_csv)\n    if (args.distribution_flow_cache is None) != (args.interval_policy is None):\n        parser.error(\n            "--distribution-flow-cache와 --interval-policy는 함께 지정해야 합니다."\n        )\n    distribution_flows = (\n        pd.read_pickle(args.distribution_flow_cache)\n        if args.distribution_flow_cache is not None\n        else None\n    )\n    output = SeatServiceModel.load(\n        args.results_dir,\n        distribution_flows=distribution_flows,\n        interval_policy=args.interval_policy,\n        full_threshold_seats=args.full_threshold_seats,\n    ).predict_frame(data)\n    output.to_csv(args.output_csv, index=False)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/low_seat_scope_experiment.py': '"""Compare whole-data, risk-weighted, and specialist low-seat training.\n\nThe risk bucket is deliberately computed within each training fold only.  A\nbucket is (route, destination stop, destination-arrival 2-hour band), and its\nrate is calculated after collapsing repeated pre-arrival snapshots to events.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sqlite3\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom hypothesis_model_search import (\n    candidate_weights,\n    combine_weighted_oof,\n    decode_target,\n    encode_target,\n    scored_frame,\n    strict_forward_bias_predictions,\n)\nfrom latest_main_model_feature_recheck import LATEST_COMPLETE_DATES\nfrom linear_feature_experiment import json_ready\nfrom main_model_feature_augmentation import (\n    component_candidates,\n    feature_sets,\n    make_model,\n    required_metrics,\n)\n\n\nRISK_MIN_EVENTS = 12\nRISK_RATE_QUANTILE = 0.75\nRISK_WEIGHT = 4.0\n\n\ndef complete_folds(data: pd.DataFrame) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:\n    """Use only completed holdout dates; never select on the partial day."""\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []\n    for validation_date in LATEST_COMPLETE_DATES:\n        train = data.loc[\n            data["date"].lt(validation_date)\n            & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)\n        ].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        if train.empty or validation.empty:\n            raise ValueError(f"empty complete-day fold: {validation_date}")\n        folds.append((validation_date, train, validation))\n    return folds\n\n\ndef with_risk_bucket(data: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    output["risk_arrival_2h"] = (\n        pd.to_datetime(output["event_time"]).dt.hour // 2 * 2\n    ).astype(int)\n    return output\n\n\ndef risk_catalog(train: pd.DataFrame) -> pd.DataFrame:\n    """Return sufficiently supported high-low-rate buckets from train only."""\n    event_level = (\n        train.sort_values("snapshot_time")\n        .groupby("event_id", observed=True, as_index=False)\n        .first()[["route_id", "station_seq_cat", "risk_arrival_2h", "label_seats"]]\n    )\n    catalog = (\n        event_level.assign(low=lambda frame: frame["label_seats"].le(10))\n        .groupby(["route_id", "station_seq_cat", "risk_arrival_2h"], observed=True)\n        .agg(events=("low", "size"), low_events=("low", "sum"))\n        .reset_index()\n    )\n    catalog["low_rate"] = catalog["low_events"] / catalog["events"]\n    supported = catalog.loc[catalog["events"].ge(RISK_MIN_EVENTS)].copy()\n    if supported.empty:\n        raise ValueError("no supported risk buckets in training fold")\n    threshold = float(supported["low_rate"].quantile(RISK_RATE_QUANTILE))\n    catalog["high_risk"] = catalog["events"].ge(RISK_MIN_EVENTS) & catalog[\n        "low_rate"\n    ].ge(threshold) & catalog["low_events"].gt(0)\n    catalog.attrs["threshold"] = threshold\n    return catalog\n\n\ndef attach_risk(data: pd.DataFrame, catalog: pd.DataFrame) -> pd.DataFrame:\n    keys = ["route_id", "station_seq_cat", "risk_arrival_2h"]\n    selected = catalog.loc[catalog["high_risk"], [*keys, "low_rate"]]\n    output = data.merge(selected, on=keys, how="left", validate="many_to_one")\n    output["is_high_risk_bucket"] = output["low_rate"].notna()\n    return output\n\n\ndef experiment_candidates() -> tuple[Any, ...]:\n    """Use exactly the deployed candidate families and hyperparameters."""\n    return component_candidates()\n\n\ndef run_variant(\n    name: str,\n    features: Any,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    *,\n    seed: int,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Fit the fixed three-model ensemble under one training-policy variant."""\n    candidates = experiment_candidates()\n    component_frames: dict[str, list[pd.DataFrame]] = {\n        candidate.name: [] for candidate in candidates\n    }\n    risk_rows: list[dict[str, Any]] = []\n    for fold_number, (validation_date, raw_train, raw_validation) in enumerate(folds):\n        catalog = risk_catalog(raw_train)\n        train = attach_risk(raw_train, catalog)\n        validation = attach_risk(raw_validation, catalog)\n        specialist = name == "high_risk_specialist"\n        if specialist:\n            fit_train = train.loc[train["is_high_risk_bucket"]].copy()\n        else:\n            fit_train = train\n        if fit_train.empty:\n            raise ValueError(f"{name} has no training rows on {validation_date}")\n        risk_rows.append(\n            {\n                "variant": name,\n                "validation_date": validation_date,\n                "risk_rate_threshold": catalog.attrs["threshold"],\n                "risk_buckets": int(catalog["high_risk"].sum()),\n                "train_rows_before_policy": int(len(train)),\n                "train_rows_used": int(len(fit_train)),\n                "train_events_used": int(fit_train["event_id"].nunique()),\n                "validation_rows_high_risk": int(validation["is_high_risk_bucket"].sum()),\n                "validation_events_high_risk": int(\n                    validation.loc[validation["is_high_risk_bucket"], "event_id"].nunique()\n                ),\n            }\n        )\n        for candidate_index, candidate in enumerate(candidates):\n            print(f"[{name}] {validation_date} {candidate.name}", flush=True)\n            model = make_model(candidate, features, seed + fold_number * 10 + candidate_index)\n            weights = candidate_weights(\n                fit_train,\n                candidate.low_weight,\n                weighting_kind=candidate.weighting_kind,\n                gap_weight_power=candidate.gap_weight_power,\n            )\n            if name == "high_risk_weighted":\n                weights = weights * np.where(\n                    fit_train["is_high_risk_bucket"].to_numpy(), RISK_WEIGHT, 1.0\n                )\n            model.fit(\n                fit_train[list(features.columns)],\n                encode_target(fit_train, candidate.target_kind),\n                regressor__sample_weight=weights,\n            )\n            prediction = decode_target(\n                model.predict(validation[list(features.columns)]), validation, candidate.target_kind\n            )\n            frame = scored_frame(validation, prediction, candidate=candidate.name)\n            frame["is_high_risk_bucket"] = validation["is_high_risk_bucket"].to_numpy()\n            component_frames[candidate.name].append(frame)\n\n    components = {\n        candidate: pd.concat(frames, ignore_index=True)\n        for candidate, frames in component_frames.items()\n    }\n    blended = combine_weighted_oof(\n        name, components, {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}\n    )\n    strict, _ = strict_forward_bias_predictions(\n        blended, blended["prediction"].to_numpy(float), validation_dates=list(LATEST_COMPLETE_DATES)\n    )\n    blended["prediction"] = strict\n    blended["variant"] = name\n    return blended, pd.DataFrame(risk_rows)\n\n\ndef scoped_metrics(predictions: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for variant, frame in predictions.groupby("variant", sort=False):\n        for scope, scoped in {\n            "all_complete": frame,\n            "high_risk_bucket": frame.loc[frame["is_high_risk_bucket"]],\n        }.items():\n            if scoped.empty:\n                continue\n            rows.append({"variant": variant, "evaluation_scope": scope, **required_metrics(scoped)})\n    return pd.DataFrame(rows)\n\n\nSTOP_BINS = (0, 1, 2, 5, 10, 20, np.inf)\nSTOP_LABELS = ("1", "2", "3-5", "6-10", "11-20", "21+")\n\n\ndef stop_gap_metrics(predictions: pd.DataFrame) -> pd.DataFrame:\n    """Report the same required metrics at operational stop-gap bands."""\n    output = predictions.copy()\n    output["stop_gap_band"] = pd.cut(\n        output["target_stop_gap"],\n        bins=STOP_BINS,\n        labels=STOP_LABELS,\n        include_lowest=False,\n    )\n    rows: list[dict[str, Any]] = []\n    for (variant, band), frame in output.groupby(\n        ["variant", "stop_gap_band"], observed=True, sort=False\n    ):\n        if frame.empty or not frame["label_seats"].le(10).any():\n            continue\n        rows.append(\n            {\n                "variant": variant,\n                "stop_gap_band": str(band),\n                "stop_gap_order": STOP_LABELS.index(str(band)),\n                **required_metrics(frame),\n            }\n        )\n    return pd.DataFrame(rows).sort_values(["stop_gap_order", "variant"])\n\n\ndef plot_stop_gap_metrics(metrics: pd.DataFrame, path: Path) -> None:\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)\n    for variant, frame in metrics.groupby("variant", sort=False):\n        ordered = frame.sort_values("stop_gap_order")\n        axes[0].plot(\n            ordered["stop_gap_band"], ordered["event_balanced_mae"], marker="o", label=variant\n        )\n        axes[1].plot(\n            ordered["stop_gap_band"], ordered["low_0_10_mae"], marker="o", label=variant\n        )\n    axes[0].set(title="Overall MAE by stop gap", xlabel="Stops before arrival", ylabel="Event-balanced MAE")\n    axes[1].set(title="Low-seat MAE by stop gap", xlabel="Stops before arrival", ylabel="Low (≤10) MAE")\n    for axis in axes:\n        axis.grid(alpha=0.25)\n        axis.legend(fontsize=8)\n    fig.tight_layout()\n    fig.savefig(path, dpi=180)\n    plt.close(fig)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="저잔여 고위험 구간 학습 정책 비교")\n    parser.add_argument("--featured-cache", type=Path, default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"))\n    parser.add_argument("--database", type=Path, default=Path("data/gbis_api_cache.sqlite3"))\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/low_seat_scope_full_horizon_results"),\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    data = pd.read_pickle(args.featured_cache)\n    source_cutoff = data.attrs.get("source_cutoff")\n    if not source_cutoff:\n        raise ValueError("featured cache에 source_cutoff 메타데이터가 없습니다.")\n    data = with_risk_bucket(data)\n    with sqlite3.connect(f"file:{args.database.resolve()}?mode=ro", uri=True) as connection:\n        source_newest_observation = connection.execute(\n            "SELECT max(observed_at) FROM location_history"\n        ).fetchone()[0]\n    base_features = feature_sets()["importance_pruned"]\n    features = type(base_features)(\n        "pooled_40_risk_scope",\n        base_features.numeric,\n        (*base_features.categorical, "route_code"),\n    )\n    folds = complete_folds(data)\n\n    baseline, audit = run_variant("whole_data", features, folds, seed=args.seed)\n    weighted, weighted_audit = run_variant("high_risk_weighted", features, folds, seed=args.seed)\n    specialist, specialist_audit = run_variant("high_risk_specialist", features, folds, seed=args.seed)\n    mixed = baseline.copy()\n    mixed["prediction"] = np.where(\n        mixed["is_high_risk_bucket"], specialist["prediction"].to_numpy(), mixed["prediction"].to_numpy()\n    )\n    mixed["variant"] = "specialist_mixture"\n    predictions = pd.concat([baseline, weighted, specialist, mixed], ignore_index=True)\n    metrics = scoped_metrics(predictions)\n    by_stop_gap = stop_gap_metrics(predictions)\n    risk_audit = pd.concat([audit, weighted_audit, specialist_audit], ignore_index=True)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")\n    metrics.to_csv(args.output_dir / "metrics.csv", index=False)\n    by_stop_gap.to_csv(args.output_dir / "metrics_by_stop_gap.csv", index=False)\n    plot_stop_gap_metrics(by_stop_gap, args.output_dir / "mae_by_stop_gap.png")\n    risk_audit.to_csv(args.output_dir / "risk_bucket_audit.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": source_cutoff,\n            "source_data_newest_observation_at_refresh": source_newest_observation,\n            "included_routes": "all six actively collected routes",\n            "excluded_routes": "historical-only routes; not active at cutoff",\n            "selection_dates": list(LATEST_COMPLETE_DATES),\n            "partial_date_excluded": "2026-08-13",\n            "risk_bucket": "route_id × target station × arrival 2-hour band",\n            "risk_definition": f"at least {RISK_MIN_EVENTS} earlier training events; low<=10 rate at or above fold-supported {RISK_RATE_QUANTILE:.0%} quantile; positive low-event count",\n            "model": "deployed-family weighted HGB/ExtraTrees/LightGBM ensemble with 40 core features plus route_code",\n            "policies": {"whole_data": "all training rows", "high_risk_weighted": f"all rows; high-risk rows have {RISK_WEIGHT:g}x additional sample weight", "high_risk_specialist": "fit only to high-risk rows", "specialist_mixture": "whole-data prediction except specialist prediction in a high-risk bucket"},\n            "snapshot_policy": "all pre-arrival snapshots in both training and validation; event-balanced aggregation",\n        },\n        "metrics": metrics.to_dict(orient="records"),\n        "metrics_by_stop_gap": by_stop_gap.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8")\n    print(metrics.to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'analysis/tminus_feasibility.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom high_risk_feasibility import high_risk_mask\nfrom model_feasibility import (\n    PLANNING,\n    FeatureSet,\n    _as_records,\n    build_model_table,\n    build_visits,\n    evaluate_final_split,\n    infer_turnaround_seq,\n    json_ready,\n    load_data,\n    nominal_capacity,\n    prepare_subset,\n)\n\n\nROUTE_ID = "219000013"\nROUTE_NAME = "1000"\nDEFAULT_HORIZONS = (5, 10, 15, 20, 30)\nTARGET_FRESHNESS_MINUTES = 5.0\nROUTE_FRESHNESS_MINUTES = 3.0\nRECENT_LOOKBACK_MINUTES = 5\n\n\nFIXED_3STOP = FeatureSet(\n    name="fixed_3stop_target",\n    numeric=(\n        *PLANNING.numeric,\n        "capacity",\n        "upstream_load_ratio_3",\n        "upstream_stop_distance_3",\n        "upstream_age_minutes_3",\n    ),\n    categorical=(*PLANNING.categorical, "low_plate_cat"),\n)\n\n\ndef tminus_feature_sets(horizon: int) -> tuple[FeatureSet, FeatureSet]:\n    target = FeatureSet(\n        name=f"tminus_{horizon}_target",\n        numeric=(\n            *PLANNING.numeric,\n            "snapshot_capacity",\n            "target_load_ratio",\n            "target_stop_gap",\n            "target_observation_age",\n            "target_load_change_5m",\n            "target_seq_change_5m",\n        ),\n        categorical=(\n            *PLANNING.categorical,\n            "snapshot_low_plate_cat",\n            "target_state_cat",\n        ),\n    )\n    route = FeatureSet(\n        name=f"tminus_{horizon}_route_context",\n        numeric=(\n            *target.numeric,\n            "lead_load_ratio",\n            "lead_gap_stops",\n            "lead_relative_to_target_stop",\n            "lead_observation_age",\n            "lead_passed_target_stop",\n            "following_load_ratio",\n            "following_gap_stops",\n            "following_observation_age",\n            "route_vehicle_count",\n            "route_full_vehicle_count",\n            "route_mean_load_ratio",\n            "route_load_std",\n        ),\n        categorical=target.categorical,\n    )\n    return target, route\n\n\ndef prepare_raw_locations(\n    locations: pd.DataFrame,\n    turnaround_seq: int,\n) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:\n    raw = locations.loc[\n        locations["station_seq"].notna() & locations["remaining_seats"].ge(0)\n    ].copy()\n    raw["station_seq"] = raw["station_seq"].astype(int)\n    raw["direction"] = np.where(\n        raw["station_seq"].le(turnaround_seq), "to_city", "return"\n    )\n    raw["capacity"] = nominal_capacity(raw["low_plate"])\n    raw["load_ratio"] = (1 - raw["remaining_seats"] / raw["capacity"]).clip(0, 1)\n    raw = raw.sort_values(["vehicle_id", "observed_at", "run_id"]).reset_index(drop=True)\n    by_vehicle = {\n        str(vehicle_id): group.reset_index(drop=True)\n        for vehicle_id, group in raw.groupby("vehicle_id", sort=False)\n    }\n    return raw, by_vehicle\n\n\ndef latest_row_before(\n    rows: pd.DataFrame,\n    when: pd.Timestamp,\n    *,\n    not_before: pd.Timestamp | None = None,\n) -> pd.Series | None:\n    # pandas 3 may preserve the source timestamp at microsecond resolution, while\n    # Timestamp.value is nanoseconds. Series.searchsorted keeps the units aligned.\n    position = int(rows["observed_at"].searchsorted(when, side="right") - 1)\n    if position < 0:\n        return None\n    row = rows.iloc[position]\n    if not_before is not None and row["observed_at"] < not_before:\n        return None\n    return row\n\n\ndef vehicle_snapshot(\n    by_vehicle: dict[str, pd.DataFrame],\n    when: pd.Timestamp,\n    freshness_minutes: float,\n) -> pd.DataFrame:\n    rows: list[pd.Series] = []\n    for vehicle_rows in by_vehicle.values():\n        row = latest_row_before(vehicle_rows, when)\n        if row is None:\n            continue\n        age = (when - row["observed_at"]).total_seconds() / 60\n        if 0 <= age <= freshness_minutes:\n            copied = row.copy()\n            copied["observation_age"] = age\n            rows.append(copied)\n    if not rows:\n        return pd.DataFrame()\n    return pd.DataFrame(rows)\n\n\ndef nearest_vehicle_features(\n    candidates: pd.DataFrame,\n    target_current_seq: int,\n    target_stop_seq: int,\n) -> dict[str, float]:\n    result = {\n        "lead_load_ratio": np.nan,\n        "lead_gap_stops": np.nan,\n        "lead_relative_to_target_stop": np.nan,\n        "lead_observation_age": np.nan,\n        "lead_passed_target_stop": np.nan,\n        "following_load_ratio": np.nan,\n        "following_gap_stops": np.nan,\n        "following_observation_age": np.nan,\n    }\n    lead = candidates.loc[candidates["station_seq"].gt(target_current_seq)].sort_values(\n        ["station_seq", "observed_at"], ascending=[True, False]\n    )\n    if not lead.empty:\n        row = lead.iloc[0]\n        result.update(\n            {\n                "lead_load_ratio": float(row["load_ratio"]),\n                "lead_gap_stops": float(row["station_seq"] - target_current_seq),\n                "lead_relative_to_target_stop": float(\n                    row["station_seq"] - target_stop_seq\n                ),\n                "lead_observation_age": float(row["observation_age"]),\n                "lead_passed_target_stop": float(\n                    row["station_seq"] >= target_stop_seq\n                ),\n            }\n        )\n\n    following = candidates.loc[candidates["station_seq"].lt(target_current_seq)].sort_values(\n        ["station_seq", "observed_at"], ascending=[False, False]\n    )\n    if not following.empty:\n        row = following.iloc[0]\n        result.update(\n            {\n                "following_load_ratio": float(row["load_ratio"]),\n                "following_gap_stops": float(target_current_seq - row["station_seq"]),\n                "following_observation_age": float(row["observation_age"]),\n            }\n        )\n    return result\n\n\ndef build_tminus_table(\n    events: pd.DataFrame,\n    visits: pd.DataFrame,\n    by_vehicle: dict[str, pd.DataFrame],\n    horizon: int,\n) -> pd.DataFrame:\n    trip_starts = visits.groupby("trip_id", sort=False)["first_seen"].min()\n    output_rows: list[dict[str, Any]] = []\n\n    for event in events.itertuples(index=False):\n        snapshot_time = event.event_time - pd.Timedelta(minutes=horizon)\n        trip_start = trip_starts.get(event.trip_id)\n        target_rows = by_vehicle.get(str(event.vehicle_id))\n        if target_rows is None or pd.isna(trip_start):\n            continue\n\n        target = latest_row_before(target_rows, snapshot_time, not_before=trip_start)\n        if target is None:\n            continue\n        target_age = (snapshot_time - target["observed_at"]).total_seconds() / 60\n        if not 0 <= target_age <= TARGET_FRESHNESS_MINUTES:\n            continue\n        if int(target["station_seq"]) > int(event.station_seq):\n            continue\n\n        previous_time = snapshot_time - pd.Timedelta(minutes=RECENT_LOOKBACK_MINUTES)\n        previous = latest_row_before(target_rows, previous_time, not_before=trip_start)\n        previous_age = (\n            (previous_time - previous["observed_at"]).total_seconds() / 60\n            if previous is not None\n            else np.nan\n        )\n        previous_is_fresh = previous is not None and 0 <= previous_age <= TARGET_FRESHNESS_MINUTES\n\n        snapshot = vehicle_snapshot(\n            by_vehicle, snapshot_time, freshness_minutes=ROUTE_FRESHNESS_MINUTES\n        )\n        if snapshot.empty:\n            same_direction = pd.DataFrame(\n                columns=[\n                    "station_seq", "observed_at", "remaining_seats",\n                    "load_ratio", "observation_age",\n                ]\n            )\n        else:\n            same_direction = snapshot.loc[\n                snapshot["direction"].eq(event.direction)\n                & snapshot["vehicle_id"].ne(event.vehicle_id)\n            ].copy()\n\n        record = event._asdict()\n        record.update(\n            {\n                "horizon_minutes": horizon,\n                "snapshot_time": snapshot_time,\n                "snapshot_capacity": float(target["capacity"]),\n                "snapshot_low_plate_cat": str(int(target["low_plate"])),\n                "target_state_cat": str(int(target["state_code"])),\n                "target_load_ratio": float(target["load_ratio"]),\n                "target_stop_gap": float(event.station_seq - target["station_seq"]),\n                "target_observation_age": float(target_age),\n                "target_load_change_5m": (\n                    float(target["load_ratio"] - previous["load_ratio"])\n                    if previous_is_fresh\n                    else np.nan\n                ),\n                "target_seq_change_5m": (\n                    float(target["station_seq"] - previous["station_seq"])\n                    if previous_is_fresh\n                    else np.nan\n                ),\n                "route_vehicle_count": float(len(same_direction) + 1),\n                "route_full_vehicle_count": float(\n                    same_direction["remaining_seats"].eq(0).sum()\n                    + int(target["remaining_seats"] == 0)\n                ),\n                "route_mean_load_ratio": float(\n                    pd.concat(\n                        [\n                            same_direction["load_ratio"],\n                            pd.Series([float(target["load_ratio"])])\n                        ],\n                        ignore_index=True,\n                    ).mean()\n                ),\n                "route_load_std": float(\n                    pd.concat(\n                        [\n                            same_direction["load_ratio"],\n                            pd.Series([float(target["load_ratio"])])\n                        ],\n                        ignore_index=True,\n                    ).std(ddof=0)\n                ),\n            }\n        )\n        record.update(\n            nearest_vehicle_features(\n                same_direction,\n                int(target["station_seq"]),\n                int(event.station_seq),\n            )\n        )\n        output_rows.append(record)\n\n    if not output_rows:\n        return pd.DataFrame(columns=[*events.columns, "horizon_minutes", "snapshot_time"])\n    return pd.DataFrame(output_rows).sort_values("event_time").reset_index(drop=True)\n\n\ndef coverage_table(source: pd.DataFrame, snapshots: dict[int, pd.DataFrame]) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for horizon, table in snapshots.items():\n        for date in ["2026-08-04", "2026-08-05", "2026-08-06", "2026-08-07"]:\n            source_day = source.loc[source["date"].eq(date)]\n            snapshot_day = table.loc[table["date"].eq(date)]\n            source_positive = int(source_day["is_full"].sum())\n            snapshot_positive = int(snapshot_day["is_full"].sum())\n            rows.append(\n                {\n                    "horizon_minutes": horizon,\n                    "date": date,\n                    "source_rows": int(len(source_day)),\n                    "snapshot_rows": int(len(snapshot_day)),\n                    "row_coverage": (\n                        float(len(snapshot_day) / len(source_day)) if len(source_day) else np.nan\n                    ),\n                    "source_positives": source_positive,\n                    "snapshot_positives": snapshot_positive,\n                    "positive_coverage": (\n                        float(snapshot_positive / source_positive)\n                        if source_positive\n                        else np.nan\n                    ),\n                    "median_target_observation_age": (\n                        float(snapshot_day["target_observation_age"].median())\n                        if len(snapshot_day)\n                        else np.nan\n                    ),\n                    "median_target_stop_gap": (\n                        float(snapshot_day["target_stop_gap"].median())\n                        if len(snapshot_day)\n                        else np.nan\n                    ),\n                    "recent_change_coverage": (\n                        float(snapshot_day["target_load_change_5m"].notna().mean())\n                        if len(snapshot_day)\n                        else np.nan\n                    ),\n                    "lead_vehicle_coverage": (\n                        float(snapshot_day["lead_load_ratio"].notna().mean())\n                        if len(snapshot_day)\n                        else np.nan\n                    ),\n                    "following_vehicle_coverage": (\n                        float(snapshot_day["following_load_ratio"].notna().mean())\n                        if len(snapshot_day)\n                        else np.nan\n                    ),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef selected_metric(metrics: pd.DataFrame, feature_set: str) -> pd.Series:\n    return (\n        metrics.loc[metrics["feature_set"].eq(feature_set)]\n        .sort_values(["selection_average_precision", "brier"], ascending=[False, True])\n        .iloc[0]\n    )\n\n\ndef make_plot(selected: pd.DataFrame, output: Path) -> None:\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))\n    colors = {\n        "fixed_3stop_target": "#94a3b8",\n        "target": "#2563eb",\n        "route_context": "#f97316",\n    }\n    for family, group in selected.groupby("family", sort=False):\n        group = group.sort_values("horizon_minutes")\n        label = family.replace("_", " ")\n        color = colors.get(family, "#334155")\n        axes[0].plot(\n            group["horizon_minutes"], group["average_precision"] * 100,\n            marker="o", label=label, color=color,\n        )\n        axes[1].plot(\n            group["horizon_minutes"], group["precision_at_threshold"] * 100,\n            marker="o", label=label, color=color,\n        )\n        axes[2].plot(\n            group["horizon_minutes"], group["recall_at_threshold"] * 100,\n            marker="o", label=label, color=color,\n        )\n    axes[0].set_title("Average precision")\n    axes[1].set_title("Precision at selected threshold")\n    axes[2].set_title("Recall at selected threshold")\n    axes[1].axhline(30, color="#dc2626", linestyle="--", linewidth=1)\n    axes[2].axhline(50, color="#dc2626", linestyle="--", linewidth=1)\n    for axis in axes:\n        axis.set_xlabel("Minutes before arrival")\n        axis.set_ylabel("Percent")\n        axis.set_xticks(sorted(selected["horizon_minutes"].unique()))\n        axis.grid(alpha=0.2)\n    axes[0].legend()\n    fig.tight_layout()\n    fig.savefig(output, dpi=160)\n    plt.close(fig)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="1000번 버스 T-minus 만차 예측 검증")\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument(\n        "--output-dir", type=Path, default=Path("analysis/tminus_results")\n    )\n    parser.add_argument(\n        "--horizons", type=int, nargs="+", default=list(DEFAULT_HORIZONS)\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    locations, stations = load_data(args.db, ROUTE_ID)\n    visits = build_visits(locations, stations)\n    model_table, turnaround_seq = build_model_table(visits, stations)\n    source = prepare_subset(model_table)\n    source = source.loc[high_risk_mask(source)].copy().reset_index(drop=True)\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n\n    snapshots: dict[int, pd.DataFrame] = {}\n    all_metrics: list[pd.DataFrame] = []\n    selected_rows: list[dict[str, Any]] = []\n    for horizon in sorted(set(args.horizons)):\n        snapshot = build_tminus_table(source, visits, by_vehicle, horizon)\n        snapshots[horizon] = snapshot\n        target_features, route_features = tminus_feature_sets(horizon)\n        metrics, _, _ = evaluate_final_split(\n            snapshot,\n            [FIXED_3STOP, target_features, route_features],\n            args.seed,\n            threshold_target_recall=0.50,\n        )\n        metrics.insert(0, "horizon_minutes", horizon)\n        all_metrics.append(metrics)\n\n        for feature_set, family in [\n            (FIXED_3STOP.name, "fixed_3stop_target"),\n            (target_features.name, "target"),\n            (route_features.name, "route_context"),\n        ]:\n            selected = selected_metric(metrics, feature_set).to_dict()\n            selected["family"] = family\n            selected_rows.append(selected)\n\n    metrics_table = pd.concat(all_metrics, ignore_index=True)\n    selected = pd.DataFrame(selected_rows)\n    coverage = coverage_table(source, snapshots)\n    route_candidates = selected.loc[selected["family"].eq("route_context")]\n    best = route_candidates.sort_values(\n        ["selection_average_precision", "brier"], ascending=[False, True]\n    ).iloc[0]\n    dynamic_candidates = selected.loc[selected["family"].isin(["target", "route_context"])]\n    best_dynamic = dynamic_candidates.sort_values(\n        ["selection_average_precision", "brier"], ascending=[False, True]\n    ).iloc[0]\n\n    metrics_table.to_csv(args.output_dir / "all_metrics.csv", index=False)\n    selected.to_csv(args.output_dir / "selected_by_horizon.csv", index=False)\n    coverage.to_csv(args.output_dir / "snapshot_coverage.csv", index=False)\n    make_plot(selected, args.output_dir / "horizon_comparison.png")\n\n    result = {\n        "route_id": ROUTE_ID,\n        "route_name": ROUTE_NAME,\n        "scope": "high_risk_gate",\n        "horizons_minutes": sorted(snapshots),\n        "snapshot_definition": {\n            "anchor": "actual target-stop first-seen time minus horizon",\n            "target_max_observation_age_minutes": TARGET_FRESHNESS_MINUTES,\n            "route_vehicle_max_observation_age_minutes": ROUTE_FRESHNESS_MINUTES,\n            "leakage_rule": "all feature observations are at or before snapshot_time",\n        },\n        "split": {\n            "train": ["2026-08-04", "2026-08-05"],\n            "calibration": ["2026-08-06"],\n            "untouched_test": ["2026-08-07"],\n        },\n        "coverage_test": json_ready(\n            _as_records(coverage.loc[coverage["date"].eq("2026-08-07")])\n        ),\n        "selected_by_horizon": json_ready(_as_records(selected)),\n        "best_dynamic_selected_on_calibration": json_ready(best_dynamic.to_dict()),\n        "best_route_context_selected_on_calibration": json_ready(best.to_dict()),\n        "product_kpi": {"minimum_precision": 0.30, "minimum_recall": 0.50},\n        "threshold_selection": (\n            "calibration date에서 recall >= 0.50인 임계값 중 precision 최대"\n        ),\n        "best_route_context_passes_test_kpi": bool(\n            best["precision_at_threshold"] >= 0.30\n            and best["recall_at_threshold"] >= 0.50\n        ),\n        "best_dynamic_passes_test_kpi": bool(\n            best_dynamic["precision_at_threshold"] >= 0.30\n            and best_dynamic["recall_at_threshold"] >= 0.50\n        ),\n        "limitations": [\n            "실제 도착시각으로 T-minus 기준점을 만들었으므로 운영 시 ETA 오차가 추가된다.",\n            "수집 기간이 짧고 최종 테스트가 하루라 수치는 가능성 검증용이다.",\n            "앞·뒤 차량은 정류장 순번 기준이며 실제 도로 거리와 차량 간 시간 간격은 아니다.",\n            "3정거장 전 기준선은 예측 기준시각이 달라 T-minus 모델과 완전히 동일한 제품 조건은 아니다.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/plot_station_seat_heatmap.py': 'from __future__ import annotations\n\nimport argparse\nimport html\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom gbis_client.cache import GBISApiCache\nfrom route_specific_feature_experiment import DEFAULT_ROUTES\n\n\nSOURCE_CUTOFF = "2026-08-14 13:45:03+09:00"\n\n\ndef color_for(value: float, minimum: float, maximum: float) -> str:\n    """Low seats are warm; high seats are cool."""\n    position = np.clip((value - minimum) / max(maximum - minimum, 1e-9), 0, 1)\n    anchors = np.asarray([[215, 48, 39], [253, 231, 37], [69, 117, 180]], dtype=float)\n    scaled = position * 2\n    lower = min(int(scaled), 1)\n    fraction = scaled - lower\n    rgb = anchors[lower] * (1 - fraction) + anchors[lower + 1] * fraction\n    return "#" + "".join(f"{int(round(item)):02x}" for item in rgb)\n\n\ndef station_trend_table(cache: GBISApiCache) -> pd.DataFrame:\n    stations = cache.stations_df()\n    history = cache.history_df()\n    stations = stations.loc[stations["route_id"].isin(DEFAULT_ROUTES), [\n        "route_id", "station_id", "station_seq", "station_name", "x", "y",\n    ]].copy()\n    history = history.loc[\n        history["route_id"].isin(DEFAULT_ROUTES)\n        & history["remaining_seats"].ge(0),\n        ["route_id", "station_id", "remaining_seats"],\n    ].copy()\n    merged = history.merge(stations, on=["route_id", "station_id"], how="inner")\n    trends = merged.groupby(["x", "y"], as_index=False).agg(\n        observations=("remaining_seats", "size"),\n        route_count=("route_id", "nunique"),\n        station_count=("station_id", "nunique"),\n        mean_remaining_seats=("remaining_seats", "mean"),\n        median_remaining_seats=("remaining_seats", "median"),\n        low_10_rate=("remaining_seats", lambda values: float((values <= 10).mean())),\n    )\n    return trends.sort_values(["mean_remaining_seats", "observations"]).reset_index(drop=True)\n\n\ndef render_fragment(trends: pd.DataFrame, output: Path) -> None:\n    width, height = 900, 650\n    left, right, top, bottom = 76, 28, 60, 72\n    x_min, x_max = trends["x"].min(), trends["x"].max()\n    y_min, y_max = trends["y"].min(), trends["y"].max()\n    x_pad, y_pad = (x_max - x_min) * 0.03, (y_max - y_min) * 0.03\n    x_min, x_max = x_min - x_pad, x_max + x_pad\n    y_min, y_max = y_min - y_pad, y_max + y_pad\n    mean_min, mean_max = 0.0, 45.0\n    count_scale = np.log1p(trends["observations"])\n    count_95 = float(np.quantile(count_scale, 0.95))\n\n    def px(value: float) -> float:\n        return left + (value - x_min) / (x_max - x_min) * (width - left - right)\n\n    def py(value: float) -> float:\n        return height - bottom - (value - y_min) / (y_max - y_min) * (height - top - bottom)\n\n    circles: list[str] = []\n    for row in trends.itertuples(index=False):\n        radius = 2.2 + 7.0 * min(np.log1p(row.observations) / count_95, 1.0)\n        tooltip = (\n            f"평균 잔여좌석 {row.mean_remaining_seats:.1f}석 | "\n            f"≤10석 비율 {row.low_10_rate:.1%} | 관측 {row.observations:,}건"\n        )\n        circles.append(\n            f\'<circle cx="{px(row.x):.1f}" cy="{py(row.y):.1f}" r="{radius:.1f}" \'\n            f\'fill="{color_for(row.mean_remaining_seats, mean_min, mean_max)}" \'\n            f\'fill-opacity="0.68"><title>{html.escape(tooltip)}</title></circle>\'\n        )\n\n    legend = "".join(\n        f\'<rect x="{688 + step * 22}" y="24" width="22" height="12" fill="{color_for(step * 5, mean_min, mean_max)}" />\'\n        for step in range(10)\n    )\n    x_ticks = "".join(\n        f\'<text x="{px(value):.1f}" y="{height - 42}" text-anchor="middle">{value:.2f}</text>\'\n        for value in np.linspace(x_min, x_max, 5)\n    )\n    y_ticks = "".join(\n        f\'<text x="{left - 10}" y="{py(value) + 4:.1f}" text-anchor="end">{value:.2f}</text>\'\n        for value in np.linspace(y_min, y_max, 5)\n    )\n    fragment = f\'\'\'<div id="station-seat-heatmap" style="max-width:920px;margin:0 auto">\n  <svg viewBox="0 0 {width} {height}" role="img" aria-label="전체 활성 노선 정류장별 평균 잔여좌석 히트맵" style="width:100%;height:auto;font-family:system-ui,sans-serif">\n    <text x="{left}" y="25" font-size="18" font-weight="700">정류장 좌표별 평균 잔여좌석 히트맵</text>\n    <text x="{left}" y="44" font-size="12" fill="#555">활성 6개 노선 · 유효 관측 {int(trends.observations.sum()):,}건 · 색: 평균 잔여좌석(낮을수록 빨강) · 원 크기: 관측 수</text>\n    <rect x="{left}" y="{top}" width="{width-left-right}" height="{height-top-bottom}" fill="none" stroke="#777" />\n    {\'\'.join(circles)}\n    <g font-size="11" fill="#333">{x_ticks}{y_ticks}</g>\n    <text x="{width / 2}" y="{height - 12}" text-anchor="middle" font-size="12">경도 (x)</text>\n    <text x="18" y="{height / 2}" transform="rotate(-90 18 {height / 2})" text-anchor="middle" font-size="12">위도 (y)</text>\n    <g>{legend}</g>\n    <text x="688" y="51" font-size="11">0석</text><text x="885" y="51" text-anchor="end" font-size="11">45석+</text>\n  </svg>\n</div>\'\'\'\n    output.write_text(fragment, encoding="utf-8")\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="정류장 좌표별 잔여좌석 히트맵 생성")\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--summary", type=Path)\n    args = parser.parse_args()\n    with GBISApiCache.from_env() as cache:\n        trends = station_trend_table(cache)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    render_fragment(trends, args.output)\n    if args.summary:\n        args.summary.parent.mkdir(parents=True, exist_ok=True)\n        args.summary.write_text(json.dumps({\n            "source_cutoff": SOURCE_CUTOFF,\n            "routes": DEFAULT_ROUTES,\n            "coordinate_points": int(len(trends)),\n            "valid_observations": int(trends["observations"].sum()),\n            "mean_remaining_seats": float(\n                np.average(trends["mean_remaining_seats"], weights=trends["observations"])\n            ),\n        }, ensure_ascii=False, indent=2), encoding="utf-8")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/remaining_seats_eda.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport sqlite3\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\nfrom model_feasibility import (\n    build_model_table,\n    build_visits,\n    json_ready,\n    load_data,\n)\n\n\nDEFAULT_ROUTE_ID = "219000013"\nDEFAULT_ROUTE_NAME = "1000"\n\n\ndef global_route_profile(db_path: Path) -> pd.DataFrame:\n    uri = f"file:{db_path.resolve()}?mode=ro"\n    with sqlite3.connect(uri, uri=True) as connection:\n        return pd.read_sql_query(\n            """\n            SELECT route_id,\n                   COUNT(*) AS rows,\n                   COUNT(DISTINCT vehicle_id) AS vehicles,\n                   MIN(observed_at_kst) AS first_observed_at,\n                   MAX(observed_at_kst) AS last_observed_at,\n                   SUM(CASE WHEN remaining_seats < 0 THEN 1 ELSE 0 END) AS invalid_seat_rows,\n                   SUM(CASE WHEN remaining_seats = 0 THEN 1 ELSE 0 END) AS raw_zero_rows\n            FROM bus_locations\n            GROUP BY route_id\n            ORDER BY rows DESC\n            """,\n            connection,\n        )\n\n\ndef quantiles(series: pd.Series) -> dict[str, float | None]:\n    clean = series.dropna().astype(float)\n    if clean.empty:\n        return {key: None for key in ["min", "p10", "p25", "median", "p75", "p90", "max"]}\n    values = clean.quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1])\n    return {\n        "min": float(values.loc[0.0]),\n        "p10": float(values.loc[0.1]),\n        "p25": float(values.loc[0.25]),\n        "median": float(values.loc[0.5]),\n        "p75": float(values.loc[0.75]),\n        "p90": float(values.loc[0.9]),\n        "max": float(values.loc[1.0]),\n    }\n\n\ndef observation_dynamics(locations: pd.DataFrame) -> dict[str, Any]:\n    valid = locations.loc[\n        locations["remaining_seats"].ge(0) & locations["station_seq"].notna()\n    ].copy()\n    valid = valid.sort_values(["vehicle_id", "observed_at", "run_id"])\n    grouped = valid.groupby("vehicle_id", sort=False)\n    valid["previous_time"] = grouped["observed_at"].shift()\n    valid["previous_seq"] = grouped["station_seq"].shift()\n    valid["previous_seats"] = grouped["remaining_seats"].shift()\n    valid["gap_minutes"] = (\n        valid["observed_at"] - valid["previous_time"]\n    ).dt.total_seconds() / 60\n    valid["seat_delta"] = valid["remaining_seats"] - valid["previous_seats"]\n\n    consecutive = valid.loc[valid["gap_minutes"].between(0, 5, inclusive="right")]\n    same_stop = consecutive.loc[consecutive["station_seq"].eq(consecutive["previous_seq"])]\n    next_stop = consecutive.loc[\n        consecutive["station_seq"].eq(consecutive["previous_seq"] + 1)\n    ]\n    cadence = consecutive["gap_minutes"]\n    return {\n        "valid_raw_rows": int(len(valid)),\n        "median_cadence_minutes": float(cadence.median()),\n        "p90_cadence_minutes": float(cadence.quantile(0.9)),\n        "same_stop_pairs": int(len(same_stop)),\n        "same_stop_unchanged_rate": float(same_stop["seat_delta"].eq(0).mean()),\n        "next_stop_pairs": int(len(next_stop)),\n        "next_stop_unchanged_rate": float(next_stop["seat_delta"].eq(0).mean()),\n        "next_stop_absolute_change": quantiles(next_stop["seat_delta"].abs()),\n        "next_stop_decrease_rate": float(next_stop["seat_delta"].lt(0).mean()),\n        "next_stop_increase_rate": float(next_stop["seat_delta"].gt(0).mean()),\n    }\n\n\ndef arrival_label_profile(table: pd.DataFrame) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:\n    exact = table.loc[table["label_quality"].isin(["A", "B"])].copy()\n    exact["is_weekend"] = exact["event_time"].dt.dayofweek.ge(5)\n    exact["is_low_0_5"] = exact["label_seats"].le(5)\n    exact["is_zero"] = exact["label_seats"].eq(0)\n\n    by_date = (\n        exact.groupby("date", sort=True)\n        .agg(\n            rows=("label_seats", "size"),\n            trips=("trip_id", "nunique"),\n            mean_seats=("label_seats", "mean"),\n            median_seats=("label_seats", "median"),\n            low_0_5_rate=("is_low_0_5", "mean"),\n            zero_rate=("is_zero", "mean"),\n        )\n        .reset_index()\n    )\n    by_hour = (\n        exact.groupby("hour", sort=True)\n        .agg(\n            rows=("label_seats", "size"),\n            mean_seats=("label_seats", "mean"),\n            median_seats=("label_seats", "median"),\n            low_0_5_rate=("is_low_0_5", "mean"),\n        )\n        .reset_index()\n    )\n    quality_counts = exact["label_quality"].value_counts().sort_index()\n    weekday = exact.loc[~exact["is_weekend"]]\n    weekend = exact.loc[exact["is_weekend"]]\n    profile = {\n        "rows": int(len(exact)),\n        "trips": int(exact["trip_id"].nunique()),\n        "vehicles": int(exact["vehicle_id"].nunique()),\n        "date_range": [str(exact["date"].min()), str(exact["date"].max())],\n        "quality_rows": {str(key): int(value) for key, value in quality_counts.items()},\n        "remaining_seats": quantiles(exact["label_seats"]),\n        "mean_remaining_seats": float(exact["label_seats"].mean()),\n        "low_0_5_rows": int(exact["is_low_0_5"].sum()),\n        "low_0_5_rate": float(exact["is_low_0_5"].mean()),\n        "zero_rows": int(exact["is_zero"].sum()),\n        "zero_rate": float(exact["is_zero"].mean()),\n        "weekday_low_0_5_rate": float(weekday["is_low_0_5"].mean()),\n        "weekend_low_0_5_rate": float(weekend["is_low_0_5"].mean()),\n        "capacity_rows": {\n            str(int(key)): int(value)\n            for key, value in exact["capacity"].value_counts().sort_index().items()\n        },\n        "upstream_3_coverage": float(exact["upstream_seats_3"].notna().mean()),\n        "previous_bus_coverage": float(exact["previous_bus_seats"].notna().mean()),\n    }\n    return profile, by_date, by_hour\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="잔여좌석 회귀용 데이터 특성 분석")\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument("--route-id", default=DEFAULT_ROUTE_ID)\n    parser.add_argument("--route-name", default=DEFAULT_ROUTE_NAME)\n    parser.add_argument(\n        "--output-dir", type=Path, default=Path("analysis/remaining_seats_eda_results")\n    )\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    routes = global_route_profile(args.db)\n    locations, stations = load_data(args.db, args.route_id)\n    visits = build_visits(locations, stations)\n    table, turnaround_seq = build_model_table(\n        visits, stations, label_target="arrival"\n    )\n    labels, by_date, by_hour = arrival_label_profile(table)\n    target_route_rows = int(routes.loc[routes["route_id"].eq(args.route_id), "rows"].iloc[0])\n    total_rows = int(routes["rows"].sum())\n\n    result = {\n        "source": {\n            "db": str(args.db),\n            "routes": int(len(routes)),\n            "rows": total_rows,\n            "target_route_id": args.route_id,\n            "target_route_name": args.route_name,\n            "target_route_rows": target_route_rows,\n            "target_route_share": float(target_route_rows / total_rows),\n            "target_route_raw_invalid_seat_rows": int(\n                routes.loc[\n                    routes["route_id"].eq(args.route_id), "invalid_seat_rows"\n                ].iloc[0]\n            ),\n        },\n        "route_geometry": {\n            "stations": int(len(stations)),\n            "turnaround_station_seq_estimate": turnaround_seq,\n        },\n        "observation_dynamics": observation_dynamics(locations),\n        "arrival_labels": labels,\n        "arrival_label_definition": {\n            "A": "target stop stateCd=1 first valid remaining-seat observation",\n            "B": "previous sequential stop stateCd=2 departure seats within the same trip",\n        },\n        "modeling_implications": [\n            "행 단위 무작위 분할은 같은 차량·운행의 인접 관측을 양쪽에 섞으므로 날짜 순서 분할이 필요하다.",\n            "현재 잔여좌석이 강한 기준선이므로 절대 좌석보다 도착까지의 좌석 변화량을 학습한다.",\n            "0석과 0~5석은 드물어 전체 MAE와 저잔여 구간 MAE를 함께 최적화·보고한다.",\n            "잔여좌석은 0과 차량 정원 사이의 제한값이므로 예측을 범위 내로 자르고 구간 또는 분위수로 제공한다.",\n            "노선별 표본량 차이가 커서 현 단계에서는 1000번 단일 노선 모델로 검증하고 충분한 이력이 쌓인 뒤 계층형 노선 모델을 비교한다.",\n        ],\n    }\n\n    routes.to_csv(args.output_dir / "route_coverage.csv", index=False)\n    by_date.to_csv(args.output_dir / "labels_by_date.csv", index=False)\n    by_hour.to_csv(args.output_dir / "labels_by_hour.csv", index=False)\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/prospective_model_evaluation.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport math\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable, Sequence\nfrom zoneinfo import ZoneInfo\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nfrom all_prearrival_seat_regression import event_weights\nfrom hypothesis_model_search import (\n    cluster_bootstrap_mae_delta,\n    event_summary,\n    stop_band_metrics,\n)\nfrom route_distribution import (\n    apply_interval_policy,\n    file_sha256,\n    validate_interval_policy,\n)\nfrom seat_service_model import (\n    SeatServiceModel,\n    validate_model_artifact_manifest,\n    validate_model_source_manifest,\n)\nfrom strict_meta_stack_experiment import (\n    COMPONENTS as META_COMPONENTS,\n    META_VARIANTS,\n    MetaVariant,\n    build_target_meta_table,\n    deployment_meta_predict,\n    meta_feature_columns,\n)\n\n\nREQUIRED_EVALUATION_COLUMNS = frozenset(\n    {\n        "date",\n        "event_id",\n        "trip_id",\n        "label_seats",\n        "snapshot_remaining_seats",\n        "target_stop_gap",\n        "capacity",\n    }\n)\nPOINT_PREDICTION_COLUMN = "predicted_arrival_seats"\nMETA_SHADOW_PREDICTION_COLUMN = "predicted_arrival_seats_meta_shadow"\nINTERVAL_COLUMNS = (\n    "predicted_arrival_seats_lower_90",\n    "predicted_arrival_seats_upper_90",\n)\nUNREFINED_LOWER_COLUMN = "predicted_arrival_seats_lower_90_unrefined"\nPROBABILITY_COLUMNS = {\n    "raw": "probability_arrival_seats_le_5_raw",\n    "shadow": "probability_arrival_seats_le_5_calibrated",\n}\nEXPECTED_SERVICE_EVENT_HOURS = frozenset(\n    {6, 7, 8, 9, 10, 16, 17, 18, 19, 20, 21}\n)\nSTOP_BAND_ORDER = ("1-2", "3-5", "6-10", "11+")\nPROSPECTIVE_CLAIM_REGISTRY_ROOT = (\n    Path(__file__).resolve().parent / "prospective_claim_registry"\n)\nRUNTIME_SOURCE_FILENAMES = (\n    "prospective_model_evaluation.py",\n    "seat_service_model.py",\n    "all_prearrival_seat_regression.py",\n    "hypothesis_model_search.py",\n    "route_distribution.py",\n    "strict_meta_stack_experiment.py",\n)\n\n\n@dataclass(frozen=True)\nclass ProspectiveEvaluation:\n    """In-memory result of one frozen, prospective evaluation run."""\n\n    summary: dict[str, Any]\n    predictions: pd.DataFrame\n    point_metrics: pd.DataFrame\n    stop_band_metrics: pd.DataFrame\n    bootstrap_vs_persistence: pd.DataFrame\n    interval_metrics: pd.DataFrame\n    interval_bootstrap: pd.DataFrame\n    probability_metrics: pd.DataFrame\n\n\ndef read_stable_bytes(path: Path, *, field: str) -> tuple[bytes, str]:\n    """Read exact artifact bytes and reject replacement during the read."""\n\n    path = Path(path)\n    before = file_sha256(path)\n    payload = path.read_bytes()\n    payload_sha256 = hashlib.sha256(payload).hexdigest()\n    after = file_sha256(path)\n    if before != payload_sha256 or after != before:\n        raise RuntimeError(f"{field} 파일이 읽는 동안 변경되었습니다: {path}")\n    return payload, before\n\n\ndef pin_selected_model_artifacts(\n    results_dir: Path,\n    model_summary: dict[str, Any],\n) -> dict[Path, str]:\n    """Pin every metadata/model file SeatServiceModel.load is expected to read."""\n\n    validate_model_source_manifest(model_summary, required=True)\n    pinned = validate_model_artifact_manifest(\n        results_dir, model_summary, required=True\n    )\n    selected = str(model_summary.get("stage_winners", {}).get("selected_final", ""))\n    if not selected:\n        raise ValueError("model summary에 selected_final이 없습니다.")\n    raw_ensemble = model_summary.get("selected_ensemble")\n    names = (\n        [str(name) for name in raw_ensemble.get("components", [])]\n        if isinstance(raw_ensemble, dict)\n        else [selected]\n    )\n    if not names:\n        raise ValueError("model summary의 selected component가 비어 있습니다.")\n    needs_route_flows = False\n    for name in names:\n        if Path(name).name != name:\n            raise ValueError(f"올바르지 않은 model component 이름입니다: {name}")\n        metadata_path = Path(results_dir) / f"{name}.metadata.json"\n        metadata_payload, metadata_sha256 = read_stable_bytes(\n            metadata_path, field=f"model component metadata {name}"\n        )\n        if pinned.get(metadata_path) != metadata_sha256:\n            raise ValueError(f"model metadata가 summary manifest와 다릅니다: {name}")\n        metadata = json.loads(metadata_payload.decode("utf-8"))\n        candidate = metadata.get("candidate", {})\n        model_kind = str(candidate.get("model_kind", ""))\n        if model_kind not in {"formula", "route_formula"}:\n            model_path = Path(results_dir) / f"{name}.joblib"\n            _, model_sha256 = read_stable_bytes(\n                model_path, field=f"model component artifact {name}"\n            )\n            if pinned.get(model_path) != model_sha256:\n                raise ValueError(f"model artifact가 summary manifest와 다릅니다: {name}")\n        needs_route_flows = needs_route_flows or (\n            model_kind == "route_formula"\n            or candidate.get("feature_variant") == "route_profile"\n        )\n    if needs_route_flows:\n        route_path = Path(results_dir) / "route_profile_flows.joblib"\n        _, route_sha256 = read_stable_bytes(\n            route_path, field="route profile flow artifact"\n        )\n        if pinned.get(route_path) != route_sha256:\n            raise ValueError("route profile flow가 summary manifest와 다릅니다.")\n    return pinned\n\n\ndef pin_meta_shadow_artifacts(\n    meta_shadow_dir: Path,\n    results_dir: Path,\n    model_summary: dict[str, Any],\n    model_summary_sha256: str,\n) -> tuple[dict[str, Any], dict[Path, str]]:\n    """Pin and validate the optional disagreement-meta shadow bundle."""\n\n    meta_shadow_dir = Path(meta_shadow_dir)\n    metadata_path = meta_shadow_dir / "selected_meta_deployment.metadata.json"\n    model_path = meta_shadow_dir / "selected_meta_deployment.joblib"\n    experiment_summary_path = meta_shadow_dir / "summary.json"\n    metadata_payload, metadata_sha256 = read_stable_bytes(\n        metadata_path, field="meta shadow metadata"\n    )\n    model_payload, model_sha256 = read_stable_bytes(\n        model_path, field="meta shadow artifact"\n    )\n    del model_payload\n    metadata = json.loads(metadata_payload.decode("utf-8"))\n    if not isinstance(metadata, dict):\n        raise ValueError("meta shadow metadata JSON은 객체여야 합니다.")\n    if metadata.get("point_model_summary_sha256") != model_summary_sha256:\n        raise ValueError(\n            "meta shadow가 현재 frozen point-model summary에 결합되어 있지 않습니다."\n        )\n    if metadata.get("model_artifact_sha256") != model_sha256:\n        raise ValueError("meta shadow artifact SHA가 metadata와 다릅니다.")\n    source_path = Path(__file__).resolve().parent / "strict_meta_stack_experiment.py"\n    source_sha256 = file_sha256(source_path)\n    if metadata.get("source_script_sha256") != source_sha256:\n        raise ValueError("meta shadow source가 deployment metadata와 다릅니다.")\n    cache_payload_hashes = model_summary.get("data_cache", {}).get("payload_sha256", {})\n    if metadata.get("snapshot_cache_sha256") != cache_payload_hashes.get("snapshots"):\n        raise ValueError("meta shadow snapshot cache가 point-model cache와 다릅니다.")\n\n    # Validate the cheap, direct deployment bindings before reading the larger\n    # experiment bundle.  Besides clearer errors, this prevents a missing sidecar\n    # from masking a point-model mismatch.\n    experiment_payload, experiment_sha256 = read_stable_bytes(\n        experiment_summary_path, field="meta shadow experiment summary"\n    )\n\n    candidate = str(metadata.get("candidate", ""))\n    expected_variant = next(\n        variant\n        for variant in META_VARIANTS\n        if variant.name == "meta_hgb_regularized_disagreement_residual"\n    )\n    raw_variant = metadata.get("variant")\n    if (\n        candidate != expected_variant.name\n        or not isinstance(raw_variant, dict)\n        or raw_variant.get("feature_kind") != expected_variant.feature_kind\n        or raw_variant.get("params") != expected_variant.params\n    ):\n        raise ValueError("meta shadow candidate/variant가 사전 고정 정의와 다릅니다.")\n\n    experiment = json.loads(experiment_payload.decode("utf-8"))\n    selection = experiment.get("selection", {})\n    provenance = experiment.get("provenance", {})\n    if (\n        selection.get("best_meta_candidate") != candidate\n        or selection.get("development_gate_passed") is not True\n        or selection.get("decision")\n        != "keep_fixed_primary_promote_meta_to_pristine_shadow"\n        or selection.get("service_adoption_recommended") is not False\n    ):\n        raise ValueError("meta experiment summary가 frozen shadow 사용을 승인하지 않습니다.")\n    if (\n        provenance.get("script_sha256") != source_sha256\n        or provenance.get("point_model_summary_sha256") != model_summary_sha256\n        or provenance.get("deployment_meta_artifact_sha256") != model_sha256\n        or provenance.get("deployment_meta_metadata_sha256") != metadata_sha256\n    ):\n        raise ValueError("meta experiment summary provenance가 deployment bundle과 다릅니다.")\n\n    raw_ensemble = model_summary.get("selected_ensemble")\n    frozen_ensemble = metadata.get("fixed_point_ensemble")\n    if not isinstance(raw_ensemble, dict) or not isinstance(frozen_ensemble, dict):\n        raise ValueError("meta shadow의 fixed point ensemble provenance가 없습니다.")\n    for key in ("name", "kind", "components", "params", "bias"):\n        if frozen_ensemble.get(key) != raw_ensemble.get(key):\n            raise ValueError(\n                f"meta shadow fixed ensemble의 {key}가 point model과 다릅니다."\n            )\n    if tuple(frozen_ensemble["components"]) != META_COMPONENTS:\n        raise ValueError("meta shadow component 순서가 고정 정의와 다릅니다.")\n\n    expected_component_hashes = metadata.get("component_metadata_sha256")\n    if not isinstance(expected_component_hashes, dict):\n        raise ValueError("meta shadow component metadata SHA가 없습니다.")\n    for component in META_COMPONENTS:\n        component_path = Path(results_dir) / f"{component}.metadata.json"\n        if expected_component_hashes.get(component) != file_sha256(component_path):\n            raise ValueError(\n                f"meta shadow component metadata가 현재 point model과 다릅니다: {component}"\n            )\n\n    features = metadata.get("feature_columns")\n    if not isinstance(features, list):\n        raise ValueError("meta shadow variant/feature provenance가 없습니다.")\n    feature_kind = str(raw_variant.get("feature_kind", ""))\n    expected_features = meta_feature_columns(feature_kind)\n    if [str(value) for value in features] != expected_features:\n        raise ValueError("meta shadow feature columns가 frozen 정의와 다릅니다.")\n    forbidden = {"label_seats", "minutes_to_arrival", "actual_minutes_to_arrival"}\n    if forbidden.intersection(expected_features):\n        raise ValueError("meta shadow inference feature에 평가 전용 열이 있습니다.")\n    return metadata, {\n        metadata_path: metadata_sha256,\n        model_path: model_sha256,\n        experiment_summary_path: experiment_sha256,\n    }\n\n\ndef pin_runtime_source_artifacts() -> dict[Path, str]:\n    """Pin source modules involved in loading and running frozen inference."""\n\n    source_dir = Path(__file__).resolve().parent\n    pinned: dict[Path, str] = {}\n    for filename in RUNTIME_SOURCE_FILENAMES:\n        path = source_dir / filename\n        _, sha256 = read_stable_bytes(path, field=f"inference runtime source {filename}")\n        pinned[path] = sha256\n    return pinned\n\n\ndef canonical_claim_registry() -> Path:\n    """Return the project-global one-shot date registry.\n\n    Model and policy hashes are stored inside each claim, not in the path, so\n    changing either artifact or ``--output-root`` cannot reopen a date.\n    """\n\n    return PROSPECTIVE_CLAIM_REGISTRY_ROOT\n\n\ndef validate_snapshot_cache_commit(\n    snapshot_cache: Path,\n    snapshot_sha256: str,\n    *,\n    required: bool,\n) -> tuple[dict[str, Any] | None, dict[Path, str]]:\n    """Validate the metadata commit marker for the mutable cache bundle."""\n\n    snapshot_cache = Path(snapshot_cache)\n    metadata_path = snapshot_cache.with_suffix(snapshot_cache.suffix + ".json")\n    flow_path = snapshot_cache.with_name(\n        snapshot_cache.stem + "_stop_flows" + snapshot_cache.suffix\n    )\n    if not metadata_path.is_file():\n        if required:\n            raise ValueError(\n                f"snapshot cache commit metadata가 없습니다: {metadata_path}"\n            )\n        return None, {}\n    metadata_payload, metadata_sha256 = read_stable_bytes(\n        metadata_path, field="snapshot cache commit metadata"\n    )\n    metadata = json.loads(metadata_payload.decode("utf-8"))\n    payload_hashes = metadata.get("payload_sha256")\n    if not isinstance(payload_hashes, dict):\n        raise ValueError("snapshot cache metadata에 payload_sha256이 없습니다.")\n    if payload_hashes.get("snapshots") != snapshot_sha256:\n        raise ValueError(\n            "snapshot cache SHA가 metadata commit marker와 일치하지 않습니다."\n        )\n    if not flow_path.is_file():\n        raise ValueError(f"snapshot cache bundle stop-flow가 없습니다: {flow_path}")\n    flow_sha256 = file_sha256(flow_path)\n    if payload_hashes.get("stop_flows") != flow_sha256:\n        raise ValueError(\n            "stop-flow cache SHA가 metadata commit marker와 일치하지 않습니다."\n        )\n    return metadata, {\n        metadata_path: metadata_sha256,\n        flow_path: flow_sha256,\n    }\n\n\ndef _iso_date(value: Any, *, field: str) -> str:\n    """Normalize a date-like value without accepting missing/ambiguous values."""\n\n    try:\n        timestamp = pd.Timestamp(value)\n    except (TypeError, ValueError) as error:\n        raise ValueError(f"{field} 날짜를 해석할 수 없습니다: {value!r}") from error\n    if pd.isna(timestamp):\n        raise ValueError(f"{field} 날짜가 비어 있습니다.")\n    return timestamp.date().isoformat()\n\n\ndef locked_confirmation_date(model_summary: dict[str, Any]) -> str:\n    """Read the last opened/inspected date from a frozen model summary."""\n\n    protocol = model_summary.get("protocol")\n    if not isinstance(protocol, dict):\n        raise ValueError("model summary에 protocol 객체가 없습니다.")\n    locked = protocol.get("locked_confirmation_test")\n    if isinstance(locked, dict):\n        raw_date = locked.get("date")\n    else:\n        raw_date = locked\n    if raw_date is None:\n        raise ValueError(\n            "model summary에 protocol.locked_confirmation_test.date가 없습니다."\n        )\n    return _iso_date(raw_date, field="locked confirmation")\n\n\ndef forbidden_protocol_dates(model_summary: dict[str, Any]) -> dict[str, list[str]]:\n    """Collect dates already touched by development, stress, or confirmation."""\n\n    protocol = model_summary.get("protocol", {})\n    selection = protocol.get("hypothesis_and_parameter_selection", {})\n    raw_groups: dict[str, Iterable[Any]] = {\n        "development": selection.get("rolling_origin_validation_dates", []) or [],\n        "stress": protocol.get("weekend_distribution_shift_stress", []) or [],\n        "confirmation": [locked_confirmation_date(model_summary)],\n    }\n    normalized: dict[str, list[str]] = {}\n    for name, values in raw_groups.items():\n        if isinstance(values, (str, bytes)):\n            values = [values]\n        normalized[name] = sorted(\n            {\n                _iso_date(value, field=f"protocol {name}")\n                for value in values\n            }\n        )\n    return normalized\n\n\ndef policy_calibration_dates(policy: dict[str, Any]) -> list[str]:\n    """Conservatively collect every ISO date declared anywhere in a policy."""\n\n    dates: set[str] = set()\n\n    def visit(value: Any, *, date_context: bool = False) -> None:\n        if isinstance(value, dict):\n            for key, item in value.items():\n                visit(item, date_context=(date_context or "date" in str(key).lower()))\n        elif isinstance(value, (list, tuple)):\n            for item in value:\n                visit(item, date_context=date_context)\n        elif date_context and isinstance(value, str):\n            try:\n                normalized = _iso_date(value, field="interval policy provenance")\n            except ValueError:\n                return\n            # Reject only canonical calendar strings, not arbitrary timestamps or\n            # prose that pd.Timestamp happens to understand.\n            if value == normalized:\n                dates.add(normalized)\n\n    visit(policy)\n    return sorted(dates)\n\n\ndef validate_closed_evaluation_dates(\n    dates: Sequence[str],\n    *,\n    today_kst: str | None = None,\n) -> None:\n    """CLI gate: never open an in-progress or future KST service day."""\n\n    today = (\n        _iso_date(today_kst, field="today_kst")\n        if today_kst is not None\n        else datetime.now(ZoneInfo("Asia/Seoul")).date().isoformat()\n    )\n    not_closed = sorted(\n        date\n        for date in {_iso_date(value, field="prospective date") for value in dates}\n        if date >= today\n    )\n    if not_closed:\n        raise ValueError(\n            "진행 중이거나 미래인 KST 날짜는 prospective 평가할 수 없습니다"\n            f"(today={today}): {not_closed}"\n        )\n\n\ndef validate_complete_evaluation_days(\n    snapshots: pd.DataFrame,\n    dates: Sequence[str],\n    *,\n    expected_event_hours: frozenset[int] = EXPECTED_SERVICE_EVENT_HOURS,\n) -> dict[str, dict[str, Any]]:\n    """Fail closed when a prospective day lacks a full service-hour footprint.\n\n    A closed calendar date can still be a partial collection day.  The model is\n    defined on morning 06--10 and evening 16--21 KST peaks, so every claimed\n    date must contain at least one labeled arrival event in each of those hours.\n    This catches truncated morning/evening collections without inspecting any\n    target values or choosing dates by their eventual metric.\n    """\n\n    required_columns = {"date", "event_time", "event_id"}\n    missing_columns = sorted(required_columns - set(snapshots.columns))\n    if missing_columns:\n        raise ValueError(\n            "prospective 일별 수집 완결성을 검증할 열이 없습니다: "\n            f"{missing_columns}"\n        )\n    if not expected_event_hours or any(\n        not isinstance(hour, int) or not 0 <= hour <= 23\n        for hour in expected_event_hours\n    ):\n        raise ValueError("expected_event_hours는 0~23 정수의 비어 있지 않은 집합이어야 합니다.")\n\n    normalized_dates = snapshots["date"].map(\n        lambda value: _iso_date(value, field="snapshot completeness")\n    )\n    event_times = pd.to_datetime(snapshots["event_time"], errors="coerce")\n    if event_times.isna().any():\n        raise ValueError("prospective snapshot의 event_time을 해석할 수 없습니다.")\n    if event_times.dt.tz is None:\n        event_times_kst = event_times.dt.tz_localize("Asia/Seoul")\n    else:\n        event_times_kst = event_times.dt.tz_convert("Asia/Seoul")\n\n    audit: dict[str, dict[str, Any]] = {}\n    incomplete: dict[str, list[int]] = {}\n    for raw_date in dates:\n        date = _iso_date(raw_date, field="prospective completeness date")\n        mask = normalized_dates.eq(date)\n        day_hours = sorted(\n            set(event_times_kst.loc[mask].dt.hour.astype(int).tolist())\n        )\n        covered_expected = sorted(set(day_hours) & set(expected_event_hours))\n        missing_hours = sorted(set(expected_event_hours) - set(day_hours))\n        audit[date] = {\n            "expected_event_hours_kst": sorted(expected_event_hours),\n            "covered_event_hours_kst": covered_expected,\n            "missing_event_hours_kst": missing_hours,\n            "events": int(snapshots.loc[mask, "event_id"].nunique()),\n            "status": "complete" if not missing_hours else "incomplete",\n        }\n        if missing_hours:\n            incomplete[date] = missing_hours\n    if incomplete:\n        raise ValueError(\n            "종료된 날짜라도 피크 시간대 수집이 불완전하면 prospective "\n            f"평가할 수 없습니다: {incomplete}"\n        )\n    return audit\n\n\ndef resolve_pristine_dates(\n    snapshots: pd.DataFrame,\n    model_summary: dict[str, Any],\n    requested_dates: Sequence[str] | None = None,\n) -> tuple[list[str], str, dict[str, list[str]]]:\n    """Select only dates that remain unopened after the locked confirmation.\n\n    When no dates are supplied, every cache date strictly after the cutoff is\n    evaluated.  Dates declared as development, stress, or confirmation in the\n    frozen summary are rejected explicitly even if a malformed protocol places\n    one after the cutoff.\n    """\n\n    if "date" not in snapshots.columns:\n        raise ValueError("snapshot cache에 date 열이 없습니다.")\n    if snapshots.empty:\n        raise ValueError("snapshot cache가 비어 있습니다.")\n\n    available = sorted(\n        {\n            _iso_date(value, field="snapshot cache")\n            for value in snapshots["date"].drop_duplicates().tolist()\n        }\n    )\n    cutoff = locked_confirmation_date(model_summary)\n    forbidden = forbidden_protocol_dates(model_summary)\n    forbidden_lookup = {\n        date: group\n        for group, dates in forbidden.items()\n        for date in dates\n    }\n    all_protocol_dates = sorted(\n        {date for dates in forbidden.values() for date in dates}\n    )\n    effective_protocol_cutoff = max(all_protocol_dates)\n\n    if requested_dates is None or len(requested_dates) == 0:\n        selected = [\n            date for date in available if date > effective_protocol_cutoff\n        ]\n        if not selected:\n            raise ValueError(\n                "모든 개발/스트레스/확인 날짜 이후의 pristine 날짜가 "\n                "snapshot cache에 없습니다"\n                f"(effective_cutoff={effective_protocol_cutoff})."\n            )\n    else:\n        selected = sorted(\n            {\n                _iso_date(value, field="requested evaluation")\n                for value in requested_dates\n            }\n        )\n\n    rejected_protocol = {\n        date: forbidden_lookup[date]\n        for date in selected\n        if date in forbidden_lookup\n    }\n    if rejected_protocol:\n        details = ", ".join(\n            f"{date}({group})"\n            for date, group in sorted(rejected_protocol.items())\n        )\n        raise ValueError(\n            "개발/스트레스/확인에 이미 사용된 날짜는 pristine 평가에 "\n            f"사용할 수 없습니다: {details}"\n        )\n\n    before_protocol_closed = [\n        date for date in selected if date <= effective_protocol_cutoff\n    ]\n    if before_protocol_closed:\n        raise ValueError(\n            "prospective 날짜는 개발/스트레스/확인에 사용된 모든 날짜보다 "\n            "뒤여야 합니다"\n            f"(effective_cutoff={effective_protocol_cutoff}): "\n            f"{before_protocol_closed}"\n        )\n\n    not_after_cutoff = [date for date in selected if date <= cutoff]\n    if not_after_cutoff:\n        raise ValueError(\n            "모든 prospective 날짜는 locked confirmation cutoff보다 엄격히 "\n            f"이후여야 합니다(cutoff={cutoff}): {not_after_cutoff}"\n        )\n\n    missing = sorted(set(selected) - set(available))\n    if missing:\n        raise ValueError(f"요청한 날짜가 snapshot cache에 없습니다: {missing}")\n    if not selected:\n        raise ValueError("평가할 pristine 날짜가 없습니다.")\n    return selected, cutoff, forbidden\n\n\ndef resolve_distribution_flow_artifact(\n    distribution_flow_cache: Path | None,\n    interval_policy: Path | None,\n) -> Path | None:\n    """Resolve and byte-verify the flow artifact bound to a frozen policy.\n\n    Supplying only ``interval_policy`` is the safe path: the policy-relative\n    immutable artifact is selected.  An explicit flow path remains supported for\n    legacy policies, while new policies require its exact file SHA to match.\n    """\n\n    if interval_policy is None:\n        if distribution_flow_cache is not None:\n            raise ValueError(\n                "distribution_flow_cache는 interval_policy와 함께 지정해야 합니다."\n            )\n        return None\n    interval_policy = Path(interval_policy)\n    if not interval_policy.is_file():\n        raise ValueError(f"frozen interval policy가 없습니다: {interval_policy}")\n    raw_policy = json.loads(interval_policy.read_text(encoding="utf-8"))\n    if not isinstance(raw_policy, dict):\n        raise ValueError("frozen interval policy JSON은 객체여야 합니다.")\n    policy = validate_interval_policy(raw_policy)\n    provenance = policy.get("provenance")\n    distribution_provenance = (\n        provenance.get("distribution") if isinstance(provenance, dict) else None\n    )\n    artifact_declared = isinstance(distribution_provenance, dict) and (\n        "flow_artifact" in distribution_provenance\n    )\n    artifact = (\n        distribution_provenance.get("flow_artifact")\n        if isinstance(distribution_provenance, dict)\n        else None\n    )\n    if artifact_declared and not isinstance(artifact, dict):\n        raise ValueError(\n            "policy provenance의 flow_artifact는 null이 아닌 객체여야 합니다."\n        )\n    if isinstance(artifact, dict):\n        required_artifact_fields = {"path", "file_sha256", "format"}\n        missing_artifact_fields = sorted(required_artifact_fields - set(artifact))\n        if missing_artifact_fields:\n            raise ValueError(\n                "policy provenance의 flow_artifact 필드가 누락되었습니다: "\n                f"{missing_artifact_fields}"\n            )\n        expected_sha256 = str(artifact["file_sha256"])\n        if len(expected_sha256) != 64 or any(\n            character not in "0123456789abcdef" for character in expected_sha256\n        ):\n            raise ValueError("policy flow_artifact file_sha256 형식이 올바르지 않습니다.")\n        if artifact["format"] != "pandas_pickle":\n            raise ValueError("policy flow_artifact format은 pandas_pickle이어야 합니다.")\n\n    if distribution_flow_cache is None:\n        if artifact is None:\n            raise ValueError(\n                "interval policy에 frozen flow_artifact가 없습니다. legacy policy는 "\n                "--distribution-flow-cache를 명시해야 합니다."\n            )\n        relative_path = Path(str(artifact["path"]))\n        if relative_path.is_absolute():\n            raise ValueError("policy의 frozen flow_artifact path는 상대 경로여야 합니다.")\n        policy_dir = interval_policy.parent.resolve()\n        resolved = (policy_dir / relative_path).resolve()\n        if not resolved.is_relative_to(policy_dir):\n            raise ValueError(\n                "policy의 frozen flow_artifact가 policy 디렉터리 밖을 가리킵니다."\n            )\n    else:\n        resolved = Path(distribution_flow_cache).resolve()\n\n    if not resolved.is_file():\n        raise ValueError(f"frozen distribution flow artifact가 없습니다: {resolved}")\n    if isinstance(artifact, dict):\n        expected_sha256 = str(artifact["file_sha256"])\n        actual_sha256 = file_sha256(resolved)\n        if actual_sha256 != expected_sha256:\n            raise ValueError(\n                "frozen distribution flow artifact file SHA가 policy provenance와 "\n                "일치하지 않습니다."\n            )\n    return resolved\n\n\ndef _validate_evaluation_rows(data: pd.DataFrame) -> None:\n    missing = sorted(REQUIRED_EVALUATION_COLUMNS - set(data.columns))\n    if missing:\n        raise ValueError(f"prospective 평가 열이 누락되었습니다: {missing}")\n    if data.empty:\n        raise ValueError("prospective 평가 행이 없습니다.")\n    numeric = (\n        "label_seats",\n        "snapshot_remaining_seats",\n        "target_stop_gap",\n        "capacity",\n    )\n    for column in numeric:\n        values = pd.to_numeric(data[column], errors="coerce").to_numpy(dtype=float)\n        if not np.isfinite(values).all():\n            raise ValueError(f"prospective 평가의 {column}에 비유한 값이 있습니다.")\n    if data["event_id"].isna().any() or data["trip_id"].isna().any():\n        raise ValueError("prospective 평가의 event_id/trip_id에 결측값이 있습니다.")\n    label_counts = data.groupby("event_id", sort=False)["label_seats"].nunique()\n    inconsistent = label_counts.loc[label_counts.gt(1)]\n    if not inconsistent.empty:\n        raise ValueError(\n            "하나의 event_id에 서로 다른 label_seats가 있습니다: "\n            f"{inconsistent.index.astype(str).tolist()[:5]}"\n        )\n\n\ndef _scopes(data: pd.DataFrame) -> list[tuple[str, str | None, pd.DataFrame]]:\n    result: list[tuple[str, str | None, pd.DataFrame]] = [("all", None, data)]\n    for date in sorted(data["date"].unique()):\n        result.append(("date", str(date), data.loc[data["date"].eq(date)]))\n    return result\n\n\ndef point_metric_tables(\n    data: pd.DataFrame,\n    selected_prediction: np.ndarray,\n    persistence_prediction: np.ndarray,\n    *,\n    selected_name: str,\n    bootstrap_seed: int,\n    bootstrap_repeats: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Score frozen point predictions and their persistence comparison."""\n\n    if bootstrap_repeats <= 0:\n        raise ValueError("bootstrap_repeats는 1 이상이어야 합니다.")\n    selected_prediction = np.asarray(selected_prediction, dtype=float)\n    persistence_prediction = np.asarray(persistence_prediction, dtype=float)\n    if len(selected_prediction) != len(data) or len(persistence_prediction) != len(data):\n        raise ValueError("prediction 행 수가 prospective 평가 행 수와 다릅니다.")\n    if not np.isfinite(selected_prediction).all():\n        raise ValueError("point prediction에 비유한 값이 있습니다.")\n\n    metric_rows: list[dict[str, Any]] = []\n    band_frames: list[pd.DataFrame] = []\n    bootstrap_rows: list[dict[str, Any]] = []\n    for position, (scope, date, scoped) in enumerate(_scopes(data)):\n        indices = data.index.get_indexer(scoped.index)\n        scoped_selected = selected_prediction[indices]\n        scoped_persistence = persistence_prediction[indices]\n        split = "all_pristine_dates" if date is None else date\n        for model, prediction in (\n            (selected_name, scoped_selected),\n            ("persistence", scoped_persistence),\n        ):\n            metrics = event_summary(scoped, prediction)\n            metrics.update(\n                {"scope": scope, "date": date, "split": split, "model": model}\n            )\n            metric_rows.append(metrics)\n            bands = stop_band_metrics(scoped, prediction, model)\n            if not bands.empty:\n                bands.insert(0, "date", date)\n                bands.insert(0, "scope", scope)\n                bands.insert(2, "split", split)\n                band_frames.append(bands)\n\n        bootstrap = cluster_bootstrap_mae_delta(\n            scoped,\n            scoped_selected,\n            scoped_persistence,\n            seed=bootstrap_seed + position,\n            repeats=bootstrap_repeats,\n        )\n        bootstrap.update(\n            {\n                "scope": scope,\n                "date": date,\n                "split": split,\n                "selected_model": selected_name,\n                "baseline_model": "persistence",\n                "seed": bootstrap_seed + position,\n                "repeats": bootstrap_repeats,\n            }\n        )\n        bootstrap_rows.append(bootstrap)\n\n    point = pd.DataFrame(metric_rows)\n    bands = (\n        pd.concat(band_frames, ignore_index=True)\n        if band_frames\n        else pd.DataFrame()\n    )\n    if not bands.empty:\n        bands["stop_band"] = pd.Categorical(\n            bands["stop_band"], categories=STOP_BAND_ORDER, ordered=True\n        )\n        bands = bands.sort_values(\n            ["scope", "date", "model", "stop_band"], na_position="first"\n        ).reset_index(drop=True)\n        bands["stop_band"] = bands["stop_band"].astype("string")\n    return point, bands, pd.DataFrame(bootstrap_rows)\n\n\ndef meta_shadow_metric_tables(\n    data: pd.DataFrame,\n    shadow_prediction: np.ndarray,\n    primary_prediction: np.ndarray,\n    *,\n    shadow_name: str,\n    primary_name: str,\n    bootstrap_seed: int,\n    bootstrap_repeats: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Score a frozen shadow against the primary without selecting on the result."""\n\n    metric_rows: list[dict[str, Any]] = []\n    band_frames: list[pd.DataFrame] = []\n    bootstrap_rows: list[dict[str, Any]] = []\n    shadow_prediction = np.asarray(shadow_prediction, dtype=float)\n    primary_prediction = np.asarray(primary_prediction, dtype=float)\n    if len(shadow_prediction) != len(data) or not np.isfinite(shadow_prediction).all():\n        raise ValueError("meta shadow prediction이 evaluation 행과 일치하지 않습니다.")\n    for position, (scope, date, scoped) in enumerate(_scopes(data)):\n        indices = data.index.get_indexer(scoped.index)\n        scoped_shadow = shadow_prediction[indices]\n        scoped_primary = primary_prediction[indices]\n        split = "all_pristine_dates" if date is None else date\n        metrics = event_summary(scoped, scoped_shadow)\n        metrics.update(\n            {"scope": scope, "date": date, "split": split, "model": shadow_name}\n        )\n        metric_rows.append(metrics)\n        bands = stop_band_metrics(scoped, scoped_shadow, shadow_name)\n        if not bands.empty:\n            bands.insert(0, "date", date)\n            bands.insert(0, "scope", scope)\n            bands.insert(2, "split", split)\n            band_frames.append(bands)\n        bootstrap = cluster_bootstrap_mae_delta(\n            scoped,\n            scoped_shadow,\n            scoped_primary,\n            seed=bootstrap_seed + position,\n            repeats=bootstrap_repeats,\n        )\n        bootstrap.update(\n            {\n                "scope": scope,\n                "date": date,\n                "split": split,\n                "selected_model": shadow_name,\n                "baseline_model": primary_name,\n                "seed": bootstrap_seed + position,\n                "repeats": bootstrap_repeats,\n                "evaluation_role": "frozen_shadow_no_reselection",\n            }\n        )\n        bootstrap_rows.append(bootstrap)\n    bands = (\n        pd.concat(band_frames, ignore_index=True)\n        if band_frames\n        else pd.DataFrame()\n    )\n    if not bands.empty:\n        bands["stop_band"] = pd.Categorical(\n            bands["stop_band"], categories=STOP_BAND_ORDER, ordered=True\n        )\n        bands = bands.sort_values(\n            ["scope", "date", "model", "stop_band"], na_position="first"\n        ).reset_index(drop=True)\n        bands["stop_band"] = bands["stop_band"].astype("string")\n    return pd.DataFrame(metric_rows), bands, pd.DataFrame(bootstrap_rows)\n\n\ndef predict_frozen_meta_shadow(\n    meta_shadow_dir: Path,\n    metadata: dict[str, Any],\n    service: SeatServiceModel,\n    target: pd.DataFrame,\n    primary_prediction: np.ndarray,\n) -> tuple[np.ndarray, dict[str, Any]]:\n    """Run the disagreement meta model from label-free inference inputs only."""\n\n    if service.ensemble is None:\n        raise ValueError("meta shadow에는 point ensemble이 필요합니다.")\n    forbidden = ["label_seats", "minutes_to_arrival", "actual_minutes_to_arrival"]\n    inference_target = target.drop(columns=forbidden, errors="ignore").copy()\n    reconstructed_primary = service.predict(inference_target)\n    primary_prediction = np.asarray(primary_prediction, dtype=float)\n    reconstruction_max_abs_diff = float(\n        np.max(np.abs(reconstructed_primary - primary_prediction))\n    )\n    if reconstruction_max_abs_diff > 1e-10:\n        raise ValueError("label-free point prediction이 primary prediction과 다릅니다.")\n    component_predictions = {\n        name: service._predict_component(service.components[name], inference_target)\n        for name in META_COMPONENTS\n    }\n    table = build_target_meta_table(\n        inference_target, component_predictions, reconstructed_primary\n    )\n    raw_variant = metadata["variant"]\n    variant = MetaVariant(\n        name=str(metadata["candidate"]),\n        why="frozen prospective shadow",\n        if_works="pristine overall and protected metrics improve",\n        if_fails="development disagreement correction does not generalize",\n        feature_kind=str(raw_variant["feature_kind"]),\n        params=dict(raw_variant.get("params", {})),\n    )\n    model_path = Path(meta_shadow_dir) / "selected_meta_deployment.joblib"\n    model_payload, model_sha256 = read_stable_bytes(\n        model_path, field="meta shadow inference artifact"\n    )\n    if model_sha256 != metadata["model_artifact_sha256"]:\n        raise ValueError("meta shadow inference artifact SHA가 metadata와 다릅니다.")\n    model = joblib.load(io.BytesIO(model_payload))\n    prediction, fit = deployment_meta_predict(\n        model, table, variant, service.ensemble\n    )\n    return prediction, {\n        "candidate": variant.name,\n        "label_or_actual_minutes_passed_to_meta": False,\n        "point_reconstruction_max_abs_diff": reconstruction_max_abs_diff,\n        "model_artifact_sha256": model_sha256,\n        "inference": fit,\n    }\n\n\ndef interval_metric_row(\n    data: pd.DataFrame,\n    lower: np.ndarray,\n    upper: np.ndarray,\n    *,\n    interval_variant: str,\n    target_population: str,\n    refinement_status: str,\n    scope: str,\n    date: str | None,\n) -> dict[str, Any]:\n    """Event-balanced 90% coverage, width, and Winkler interval score."""\n\n    lower = np.asarray(lower, dtype=float)\n    upper = np.asarray(upper, dtype=float)\n    if len(lower) != len(data) or len(upper) != len(data):\n        raise ValueError("interval 행 수가 prospective 평가 행 수와 다릅니다.")\n    if not np.isfinite(lower).all() or not np.isfinite(upper).all():\n        raise ValueError("prediction interval에 비유한 값이 있습니다.")\n    if np.any(lower > upper):\n        raise ValueError("prediction interval의 lower가 upper보다 큽니다.")\n\n    base = {\n        "scope": scope,\n        "date": date,\n        "split": "all_pristine_dates" if date is None else date,\n        "interval_variant": interval_variant,\n        "target_population": target_population,\n        "lower_tail_refinement_status": refinement_status,\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n    }\n    if data.empty:\n        return {\n            **base,\n            "coverage_90": None,\n            "mean_width_90": None,\n            "weighted_interval_score_90": None,\n        }\n\n    target = data["label_seats"].to_numpy(dtype=float)\n    weights = event_weights(data)\n    covered = (target >= lower) & (target <= upper)\n    width = upper - lower\n    alpha = 0.10\n    interval_score = width.copy()\n    below = target < lower\n    above = target > upper\n    interval_score[below] += 2 / alpha * (lower[below] - target[below])\n    interval_score[above] += 2 / alpha * (target[above] - upper[above])\n    return {\n        **base,\n        "coverage_90": float(np.average(covered, weights=weights)),\n        "mean_width_90": float(np.average(width, weights=weights)),\n        "weighted_interval_score_90": float(\n            np.average(interval_score, weights=weights)\n        ),\n    }\n\n\ndef trip_cluster_interval_score_improvement(\n    data: pd.DataFrame,\n    refined_lower: np.ndarray,\n    base_lower: np.ndarray,\n    upper: np.ndarray,\n    *,\n    target_population: str,\n    refinement_status: str,\n    scope: str,\n    date: str | None,\n    seed: int,\n    repeats: int,\n) -> dict[str, Any]:\n    """Paired trip bootstrap of base-minus-refined Winkler interval score."""\n\n    if repeats <= 0:\n        raise ValueError("interval bootstrap repeats는 1 이상이어야 합니다.")\n    refined_lower = np.asarray(refined_lower, dtype=float)\n    base_lower = np.asarray(base_lower, dtype=float)\n    upper = np.asarray(upper, dtype=float)\n    if any(len(values) != len(data) for values in (refined_lower, base_lower, upper)):\n        raise ValueError("interval bootstrap 행 수가 prospective 평가 행 수와 다릅니다.")\n    result: dict[str, Any] = {\n        "scope": scope,\n        "date": date,\n        "split": "all_pristine_dates" if date is None else date,\n        "target_population": target_population,\n        "lower_tail_refinement_status": refinement_status,\n        "metric": (\n            "audit base interval score minus primary refined interval score; "\n            "positive favors refinement"\n        ),\n        "unit": "trip_id",\n        "seed": int(seed),\n        "repeats": int(repeats),\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n        "trips": int(data["trip_id"].nunique()),\n    }\n    if data.empty:\n        return {\n            **result,\n            "bootstrap_status": "no_events_in_population",\n            "observed_delta": None,\n            "lower_95": None,\n            "median": None,\n            "upper_95": None,\n            "probability_refinement_better": None,\n        }\n\n    target = data["label_seats"].to_numpy(dtype=float)\n\n    def score(lower: np.ndarray) -> np.ndarray:\n        interval_score = upper - lower\n        below = target < lower\n        above = target > upper\n        interval_score[below] += 20.0 * (lower[below] - target[below])\n        interval_score[above] += 20.0 * (target[above] - upper[above])\n        return interval_score\n\n    scored = data[["event_id", "trip_id"]].copy()\n    scored["delta"] = score(base_lower) - score(refined_lower)\n    per_event = scored.groupby("event_id", sort=False).agg(\n        trip_id=("trip_id", "first"),\n        delta=("delta", "mean"),\n    )\n    trip_groups = [\n        group["delta"].to_numpy(dtype=float)\n        for _, group in per_event.groupby("trip_id", sort=False)\n    ]\n    rng = np.random.default_rng(seed)\n    boot = np.empty(repeats, dtype=float)\n    for position in range(repeats):\n        sampled = rng.integers(0, len(trip_groups), size=len(trip_groups))\n        boot[position] = np.concatenate(\n            [trip_groups[index] for index in sampled]\n        ).mean()\n    lower, median, upper_bound = np.quantile(boot, [0.025, 0.5, 0.975])\n    return {\n        **result,\n        "bootstrap_status": "ok",\n        "observed_delta": float(per_event["delta"].mean()),\n        "lower_95": float(lower),\n        "median": float(median),\n        "upper_95": float(upper_bound),\n        "probability_refinement_better": float((boot > 0).mean()),\n    }\n\n\ndef safe_probability_metric_row(\n    data: pd.DataFrame,\n    probability: np.ndarray,\n    *,\n    probability_model: str,\n    calibration_status: str,\n    scope: str,\n    date: str | None,\n) -> dict[str, Any]:\n    """Event-balanced P(arrival seats <= 5) metrics, safe for one class."""\n\n    probability = np.asarray(probability, dtype=float)\n    if len(probability) != len(data):\n        raise ValueError("probability 행 수가 prospective 평가 행 수와 다릅니다.")\n    if not np.isfinite(probability).all():\n        raise ValueError("P(arrival seats <= 5)에 비유한 값이 있습니다.")\n    if np.any((probability < 0) | (probability > 1)):\n        raise ValueError("P(arrival seats <= 5)가 [0, 1] 범위를 벗어났습니다.")\n\n    target = data["label_seats"].le(5).astype(int).to_numpy()\n    weights = event_weights(data)\n    clipped = np.clip(probability, 1e-6, 1 - 1e-6)\n    classes = np.unique(target)\n    single_class = len(classes) < 2\n    event_labels = data.groupby("event_id", sort=False)["label_seats"].first().le(5)\n    average_precision: float | None\n    roc_auc: float | None\n    if single_class:\n        average_precision = None\n        roc_auc = None\n    else:\n        average_precision = float(\n            average_precision_score(target, probability, sample_weight=weights)\n        )\n        roc_auc = float(roc_auc_score(target, probability, sample_weight=weights))\n    log_loss_value = -(\n        target * np.log(clipped) + (1 - target) * np.log1p(-clipped)\n    )\n    return {\n        "scope": scope,\n        "date": date,\n        "split": "all_pristine_dates" if date is None else date,\n        "probability_model": probability_model,\n        "calibration_status": calibration_status,\n        "classification_status": "single_class" if single_class else "two_class",\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n        "positive_events": int(event_labels.sum()),\n        "negative_events": int((~event_labels).sum()),\n        "weighted_prevalence": float(np.average(target, weights=weights)),\n        "mean_probability": float(np.average(probability, weights=weights)),\n        "brier": float(np.average((target - probability) ** 2, weights=weights)),\n        "log_loss": float(np.average(log_loss_value, weights=weights)),\n        "average_precision": average_precision,\n        "roc_auc": roc_auc,\n    }\n\n\ndef distribution_metric_tables(\n    data: pd.DataFrame,\n    prediction_frame: pd.DataFrame,\n    *,\n    bootstrap_seed: int,\n    bootstrap_repeats: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Score an already attached frozen interval policy without fitting it."""\n\n    required = (\n        set(INTERVAL_COLUMNS)\n        | {UNREFINED_LOWER_COLUMN}\n        | set(PROBABILITY_COLUMNS.values())\n    )\n    missing = sorted(required - set(prediction_frame.columns))\n    if missing:\n        raise ValueError(f"distribution prediction 열이 누락되었습니다: {missing}")\n    status_values = (\n        prediction_frame.get(\n            "probability_calibration_status",\n            pd.Series("unknown", index=prediction_frame.index),\n        )\n        .astype(str)\n        .unique()\n        .tolist()\n    )\n    if len(status_values) != 1:\n        raise ValueError(\n            "한 평가 run 안에서 probability calibration status가 다릅니다: "\n            f"{status_values}"\n        )\n    calibration_status = status_values[0]\n    refinement_status_values = (\n        prediction_frame.get(\n            "prediction_lower_tail_refinement_status",\n            pd.Series("none", index=prediction_frame.index),\n        )\n        .astype(str)\n        .unique()\n        .tolist()\n    )\n    if len(refinement_status_values) != 1:\n        raise ValueError(\n            "한 평가 run 안에서 lower-tail refinement status가 다릅니다: "\n            f"{refinement_status_values}"\n        )\n    refinement_status = refinement_status_values[0]\n\n    interval_rows: list[dict[str, Any]] = []\n    interval_bootstrap_rows: list[dict[str, Any]] = []\n    probability_rows: list[dict[str, Any]] = []\n    for scope_position, (scope, date, scoped) in enumerate(_scopes(data)):\n        indices = data.index.get_indexer(scoped.index)\n        refined_lower = prediction_frame[INTERVAL_COLUMNS[0]].to_numpy(\n            dtype=float\n        )[indices]\n        base_lower = prediction_frame[UNREFINED_LOWER_COLUMN].to_numpy(\n            dtype=float\n        )[indices]\n        upper = prediction_frame[INTERVAL_COLUMNS[1]].to_numpy(dtype=float)[indices]\n        populations = (\n            ("all", np.ones(len(scoped), dtype=bool)),\n            (\n                "truth_low_0_5",\n                scoped["label_seats"].le(5).to_numpy(dtype=bool),\n            ),\n        )\n        for population_position, (target_population, population_mask) in enumerate(\n            populations\n        ):\n            population = scoped.loc[population_mask]\n            for interval_variant, lower in (\n                ("primary_refined", refined_lower[population_mask]),\n                ("audit_base_unrefined", base_lower[population_mask]),\n            ):\n                interval_rows.append(\n                    interval_metric_row(\n                        population,\n                        lower,\n                        upper[population_mask],\n                        interval_variant=interval_variant,\n                        target_population=target_population,\n                        refinement_status=refinement_status,\n                        scope=scope,\n                        date=date,\n                    )\n                )\n            interval_bootstrap_rows.append(\n                trip_cluster_interval_score_improvement(\n                    population,\n                    refined_lower[population_mask],\n                    base_lower[population_mask],\n                    upper[population_mask],\n                    target_population=target_population,\n                    refinement_status=refinement_status,\n                    scope=scope,\n                    date=date,\n                    seed=(\n                        bootstrap_seed\n                        + 10_000\n                        + 10 * scope_position\n                        + population_position\n                    ),\n                    repeats=bootstrap_repeats,\n                )\n            )\n        for probability_model, column in PROBABILITY_COLUMNS.items():\n            probability_rows.append(\n                safe_probability_metric_row(\n                    scoped,\n                    prediction_frame[column].to_numpy(dtype=float)[indices],\n                    probability_model=probability_model,\n                    calibration_status=(\n                        "none" if probability_model == "raw" else calibration_status\n                    ),\n                    scope=scope,\n                    date=date,\n                )\n            )\n    return (\n        pd.DataFrame(interval_rows),\n        pd.DataFrame(interval_bootstrap_rows),\n        pd.DataFrame(probability_rows),\n    )\n\n\ndef attach_lower_tail_invariance_audit(\n    data: pd.DataFrame,\n    prediction_frame: pd.DataFrame,\n    interval_policy: dict[str, Any],\n    *,\n    full_threshold_seats: float,\n    tolerance: float = 1e-12,\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    """Reapply the base policy and audit outputs protected by lower refinement."""\n\n    policy = validate_interval_policy(interval_policy)\n    base_policy = dict(policy)\n    refinement = base_policy.pop("lower_tail_refinement", None)\n    required_prediction_columns = {\n        POINT_PREDICTION_COLUMN,\n        UNREFINED_LOWER_COLUMN,\n        INTERVAL_COLUMNS[1],\n        "route_dist_scale",\n        "probability_arrival_seats_le_5",\n        "probability_arrival_seats_le_5_raw",\n        "probability_arrival_seats_le_5_calibrated",\n        "probability_arrival_full",\n    }\n    missing = sorted(required_prediction_columns - set(prediction_frame.columns))\n    if missing:\n        raise ValueError(f"lower-tail invariance audit 열이 누락되었습니다: {missing}")\n\n    audit_input = data.copy()\n    audit_input["route_dist_scale"] = prediction_frame[\n        "route_dist_scale"\n    ].to_numpy(dtype=float)\n    point = prediction_frame[POINT_PREDICTION_COLUMN].to_numpy(dtype=float)\n    base = apply_interval_policy(\n        audit_input,\n        point,\n        base_policy,\n        full_threshold_seats=full_threshold_seats,\n    )\n    output = prediction_frame.copy()\n    output["lower_tail_audit_base_point_prediction"] = point\n    output["lower_tail_audit_base_lower_90"] = base[\n        "interval_lower"\n    ].to_numpy(dtype=float)\n    output["lower_tail_audit_base_upper_90"] = base[\n        "interval_upper"\n    ].to_numpy(dtype=float)\n    output["lower_tail_audit_base_probability_low_5"] = base[\n        "p_low_5"\n    ].to_numpy(dtype=float)\n    output["lower_tail_audit_base_probability_low_5_raw"] = base[\n        "p_low_5_raw"\n    ].to_numpy(dtype=float)\n    output["lower_tail_audit_base_probability_low_5_calibrated"] = base[\n        "p_low_5_calibrated"\n    ].to_numpy(dtype=float)\n    output["lower_tail_audit_base_probability_full"] = base[\n        "p_full"\n    ].to_numpy(dtype=float)\n\n    comparisons = {\n        "point_prediction": (\n            output[POINT_PREDICTION_COLUMN].to_numpy(dtype=float),\n            output["lower_tail_audit_base_point_prediction"].to_numpy(dtype=float),\n        ),\n        "upper_90": (\n            output[INTERVAL_COLUMNS[1]].to_numpy(dtype=float),\n            output["lower_tail_audit_base_upper_90"].to_numpy(dtype=float),\n        ),\n        "probability_low_5": (\n            output["probability_arrival_seats_le_5"].to_numpy(dtype=float),\n            output["lower_tail_audit_base_probability_low_5"].to_numpy(dtype=float),\n        ),\n        "probability_low_5_raw": (\n            output["probability_arrival_seats_le_5_raw"].to_numpy(dtype=float),\n            output["lower_tail_audit_base_probability_low_5_raw"].to_numpy(\n                dtype=float\n            ),\n        ),\n        "probability_low_5_calibrated": (\n            output["probability_arrival_seats_le_5_calibrated"].to_numpy(\n                dtype=float\n            ),\n            output[\n                "lower_tail_audit_base_probability_low_5_calibrated"\n            ].to_numpy(dtype=float),\n        ),\n        "probability_full": (\n            output["probability_arrival_full"].to_numpy(dtype=float),\n            output["lower_tail_audit_base_probability_full"].to_numpy(dtype=float),\n        ),\n        "service_unrefined_lower_matches_recomputed_base": (\n            output[UNREFINED_LOWER_COLUMN].to_numpy(dtype=float),\n            output["lower_tail_audit_base_lower_90"].to_numpy(dtype=float),\n        ),\n    }\n    maxima: dict[str, float] = {}\n    for name, (actual, expected) in comparisons.items():\n        difference = np.abs(actual - expected)\n        output[f"lower_tail_audit_{name}_absolute_difference"] = difference\n        maxima[name] = float(difference.max())\n    passed = all(value <= tolerance for value in maxima.values())\n    return output, {\n        "policy_has_lower_tail_refinement": refinement is not None,\n        "refinement_status": (\n            str(refinement["status"]) if refinement is not None else "none"\n        ),\n        "tolerance": float(tolerance),\n        "max_absolute_differences": maxima,\n        "protected_outputs_unchanged": passed,\n    }\n\n\ndef _prediction_audit_frame(\n    data: pd.DataFrame,\n    prediction_frame: pd.DataFrame,\n    persistence: np.ndarray,\n) -> pd.DataFrame:\n    base_columns = [\n        column\n        for column in (\n            "event_id",\n            "date",\n            "trip_id",\n            "snapshot_time",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "minutes_to_arrival",\n            "capacity",\n        )\n        if column in data.columns\n    ]\n    output = data[base_columns].copy().reset_index(drop=True)\n    service_columns = [\n        column\n        for column in (\n            POINT_PREDICTION_COLUMN,\n            META_SHADOW_PREDICTION_COLUMN,\n            "predicted_arrival_seats_rounded",\n            "prediction_model",\n            *INTERVAL_COLUMNS,\n            UNREFINED_LOWER_COLUMN,\n            "prediction_interval_half_width",\n            "prediction_base_interval_nominal_half_width",\n            "prediction_interval_width_90",\n            "prediction_interval_lower_distance_90",\n            "prediction_interval_upper_distance_90",\n            "prediction_lower_tail_refinement_candidate",\n            "prediction_lower_tail_refinement_factor",\n            "prediction_lower_tail_refinement_applied",\n            "prediction_lower_tail_refinement_status",\n            "prediction_lower_tail_refinement_kind",\n            "prediction_lower_tail_refinement_policy",\n            "probability_arrival_seats_le_5",\n            *PROBABILITY_COLUMNS.values(),\n            "probability_arrival_full",\n            "probability_calibration_status",\n            "probability_calibration_slope",\n            "prediction_interval_policy",\n            "route_dist_scale",\n        )\n        if column in prediction_frame.columns\n    ]\n    service_columns.extend(\n        column\n        for column in prediction_frame.columns\n        if column.startswith("lower_tail_audit_") and column not in service_columns\n    )\n    for column in service_columns:\n        output[column] = prediction_frame[column].to_numpy()\n    output["persistence_prediction"] = persistence\n    output["selected_absolute_error"] = np.abs(\n        output["label_seats"].to_numpy(dtype=float)\n        - output[POINT_PREDICTION_COLUMN].to_numpy(dtype=float)\n    )\n    output["persistence_absolute_error"] = np.abs(\n        output["label_seats"].to_numpy(dtype=float) - persistence\n    )\n    if META_SHADOW_PREDICTION_COLUMN in output.columns:\n        output["meta_shadow_absolute_error"] = np.abs(\n            output["label_seats"].to_numpy(dtype=float)\n            - output[META_SHADOW_PREDICTION_COLUMN].to_numpy(dtype=float)\n        )\n    return output\n\n\ndef _frame_records(frame: pd.DataFrame) -> list[dict[str, Any]]:\n    records: list[dict[str, Any]] = []\n    for raw in frame.to_dict(orient="records"):\n        record: dict[str, Any] = {}\n        for key, value in raw.items():\n            missing = False\n            if value is None:\n                missing = True\n            elif not isinstance(value, (list, tuple, dict)):\n                missing_value = pd.isna(value)\n                if isinstance(missing_value, (bool, np.bool_)):\n                    missing = bool(missing_value)\n            if missing:\n                record[str(key)] = None\n            elif isinstance(value, (np.integer,)):\n                record[str(key)] = int(value)\n            elif isinstance(value, (np.floating, float)):\n                number = float(value)\n                record[str(key)] = number if math.isfinite(number) else None\n            else:\n                record[str(key)] = value\n        records.append(record)\n    return records\n\n\ndef _json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): _json_ready(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [_json_ready(item) for item in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating, float)):\n        number = float(value)\n        return number if math.isfinite(number) else None\n    if isinstance(value, (pd.Timestamp, datetime)):\n        return value.isoformat()\n    return value\n\n\ndef _evaluate_claimed_prospective(\n    results_dir: Path,\n    snapshot_cache: Path,\n    *,\n    dates: Sequence[str] | None = None,\n    distribution_flow_cache: Path | None = None,\n    interval_policy: Path | None = None,\n    meta_shadow_dir: Path | None = None,\n    full_threshold_seats: float = 0.0,\n    bootstrap_seed: int = 20260812,\n    bootstrap_repeats: int = 5_000,\n) -> ProspectiveEvaluation:\n    """Internal evaluator used only after the safe runner claims every date.\n\n    Keeping the metric engine private prevents callers from accidentally\n    bypassing the project-global one-shot claim and closed-day gates.  Tests may\n    exercise it directly with synthetic data; production callers must use\n    :func:`run_prospective_evaluation`.\n    """\n\n    results_dir = Path(results_dir)\n    snapshot_cache = Path(snapshot_cache)\n    summary_path = results_dir / "summary.json"\n    if not summary_path.is_file():\n        raise ValueError(f"frozen model summary가 없습니다: {summary_path}")\n    if not snapshot_cache.is_file():\n        raise ValueError(f"snapshot cache가 없습니다: {snapshot_cache}")\n    summary_payload, model_summary_sha256 = read_stable_bytes(\n        summary_path, field="frozen model summary"\n    )\n    model_summary = json.loads(summary_payload.decode("utf-8"))\n    if not isinstance(model_summary, dict):\n        raise ValueError("frozen model summary JSON은 객체여야 합니다.")\n    model_artifact_hashes = pin_selected_model_artifacts(\n        results_dir, model_summary\n    )\n    meta_shadow_metadata: dict[str, Any] | None = None\n    if meta_shadow_dir is not None:\n        meta_shadow_dir = Path(meta_shadow_dir)\n        meta_shadow_metadata, meta_shadow_hashes = pin_meta_shadow_artifacts(\n            meta_shadow_dir,\n            results_dir,\n            model_summary,\n            model_summary_sha256,\n        )\n        overlap = set(model_artifact_hashes).intersection(meta_shadow_hashes)\n        if overlap:\n            raise ValueError(f"meta shadow artifact 경로가 point model과 겹칩니다: {overlap}")\n        model_artifact_hashes.update(meta_shadow_hashes)\n    runtime_source_hashes = pin_runtime_source_artifacts()\n\n    loaded_interval_policy: dict[str, Any] | None = None\n    interval_policy_sha256: str | None = None\n    if interval_policy is not None:\n        interval_policy = Path(interval_policy)\n        policy_payload, interval_policy_sha256 = read_stable_bytes(\n            interval_policy, field="frozen interval policy"\n        )\n        raw_interval_policy = json.loads(policy_payload.decode("utf-8"))\n        if not isinstance(raw_interval_policy, dict):\n            raise ValueError("frozen interval policy JSON은 객체여야 합니다.")\n        loaded_interval_policy = validate_interval_policy(raw_interval_policy)\n    resolved_flow_cache = resolve_distribution_flow_artifact(\n        distribution_flow_cache, interval_policy\n    )\n    snapshot_payload, snapshot_cache_sha256 = read_stable_bytes(\n        snapshot_cache, field="snapshot cache"\n    )\n    snapshot_cache_size_bytes = len(snapshot_payload)\n    snapshots = pd.read_pickle(io.BytesIO(snapshot_payload))\n    del snapshot_payload\n    if not isinstance(snapshots, pd.DataFrame):\n        raise ValueError("snapshot cache는 pandas DataFrame이어야 합니다.")\n    selected_dates, cutoff, forbidden = resolve_pristine_dates(\n        snapshots, model_summary, dates\n    )\n    cache_commit_metadata, cache_bundle_hashes = validate_snapshot_cache_commit(\n        snapshot_cache, snapshot_cache_sha256, required=False\n    )\n    declared_policy_dates = (\n        policy_calibration_dates(loaded_interval_policy)\n        if loaded_interval_policy is not None\n        else []\n    )\n    if loaded_interval_policy is not None and not declared_policy_dates:\n        raise ValueError(\n            "interval policy에 calibration/selection 날짜 provenance가 없어 "\n            "prospective chronology를 검증할 수 없습니다."\n        )\n    policy_not_strictly_prior = (\n        bool(declared_policy_dates)\n        and max(declared_policy_dates) >= min(selected_dates)\n    )\n    if policy_not_strictly_prior:\n        raise ValueError(\n            "모든 interval policy calibration/selection 날짜는 prospective "\n            "평가일보다 엄격히 이전이어야 합니다"\n            f"(policy_max={max(declared_policy_dates)}, "\n            f"evaluation_min={min(selected_dates)})"\n        )\n\n    normalized_dates = snapshots["date"].map(\n        lambda value: _iso_date(value, field="snapshot cache")\n    )\n    evaluation = snapshots.loc[normalized_dates.isin(selected_dates)].copy()\n    evaluation["date"] = normalized_dates.loc[evaluation.index].to_numpy()\n    evaluation = evaluation.reset_index(drop=True)\n    _validate_evaluation_rows(evaluation)\n\n    flow_artifact_sha256: str | None = None\n    if resolved_flow_cache is not None:\n        flow_payload, flow_artifact_sha256 = read_stable_bytes(\n            resolved_flow_cache, field="frozen distribution flow"\n        )\n        flows = pd.read_pickle(io.BytesIO(flow_payload))\n        del flow_payload\n        assert loaded_interval_policy is not None\n        artifact = loaded_interval_policy.get("provenance", {}).get(\n            "distribution", {}\n        ).get("flow_artifact")\n        if artifact is not None and artifact["file_sha256"] != flow_artifact_sha256:\n            raise ValueError(\n                "고정해 읽은 frozen flow SHA가 policy provenance와 다릅니다."\n            )\n    else:\n        flows = None\n    service = SeatServiceModel.load(\n        results_dir,\n        distribution_flows=flows,\n        interval_policy=loaded_interval_policy,\n        full_threshold_seats=full_threshold_seats,\n    )\n    if file_sha256(summary_path) != model_summary_sha256:\n        raise RuntimeError("frozen model summary가 service load 중 변경되었습니다.")\n    for artifact_path, expected_sha256 in model_artifact_hashes.items():\n        if file_sha256(artifact_path) != expected_sha256:\n            raise RuntimeError(\n                "frozen model component가 service load 중 변경되었습니다: "\n                f"{artifact_path}"\n            )\n    stage_winners = model_summary.get("stage_winners")\n    if not isinstance(stage_winners, dict) or not stage_winners.get(\n        "selected_final"\n    ):\n        raise ValueError("frozen model summary에 stage_winners.selected_final이 없습니다.")\n    expected_name = str(stage_winners["selected_final"])\n    if service.selected_name != expected_name:\n        raise ValueError(\n            "loaded service model과 frozen summary의 selected_final이 다릅니다: "\n            f"service={service.selected_name}, summary={expected_name}"\n        )\n    distribution_requested = interval_policy is not None\n    if distribution_requested and not service.distribution_attached:\n        raise ValueError("interval policy를 요청했지만 service model에 부착되지 않았습니다.")\n\n    prediction_frame = service.predict_frame(evaluation).reset_index(drop=True)\n    if len(prediction_frame) != len(evaluation):\n        raise ValueError("service prediction 행 수가 prospective 평가 행 수와 다릅니다.")\n    if POINT_PREDICTION_COLUMN not in prediction_frame.columns:\n        raise ValueError(\n            f"service prediction에 {POINT_PREDICTION_COLUMN} 열이 없습니다."\n        )\n    selected_prediction = prediction_frame[POINT_PREDICTION_COLUMN].to_numpy(\n        dtype=float\n    )\n    persistence = np.clip(\n        evaluation["snapshot_remaining_seats"].to_numpy(dtype=float),\n        0,\n        evaluation["capacity"].to_numpy(dtype=float),\n    )\n    point, bands, bootstrap = point_metric_tables(\n        evaluation,\n        selected_prediction,\n        persistence,\n        selected_name=service.selected_name,\n        bootstrap_seed=bootstrap_seed,\n        bootstrap_repeats=bootstrap_repeats,\n    )\n    meta_shadow_provenance: dict[str, Any] | None = None\n    if meta_shadow_metadata is not None:\n        assert meta_shadow_dir is not None\n        shadow_prediction, shadow_diagnostic = predict_frozen_meta_shadow(\n            meta_shadow_dir,\n            meta_shadow_metadata,\n            service,\n            evaluation,\n            selected_prediction,\n        )\n        shadow_name = str(meta_shadow_metadata["candidate"])\n        shadow_point, shadow_bands, shadow_bootstrap = meta_shadow_metric_tables(\n            evaluation,\n            shadow_prediction,\n            selected_prediction,\n            shadow_name=shadow_name,\n            primary_name=service.selected_name,\n            bootstrap_seed=bootstrap_seed + 100_000,\n            bootstrap_repeats=bootstrap_repeats,\n        )\n        point = pd.concat([point, shadow_point], ignore_index=True)\n        if not shadow_bands.empty:\n            bands = pd.concat([bands, shadow_bands], ignore_index=True)\n        bootstrap = pd.concat([bootstrap, shadow_bootstrap], ignore_index=True)\n        prediction_frame[META_SHADOW_PREDICTION_COLUMN] = shadow_prediction\n        meta_shadow_provenance = {\n            "status": "frozen_shadow_no_retraining_or_reselection",\n            "directory": str(meta_shadow_dir.resolve()),\n            "candidate": shadow_name,\n            "metadata_sha256": model_artifact_hashes[\n                meta_shadow_dir / "selected_meta_deployment.metadata.json"\n            ],\n            "artifact_sha256": model_artifact_hashes[\n                meta_shadow_dir / "selected_meta_deployment.joblib"\n            ],\n            "diagnostic": shadow_diagnostic,\n        }\n    lower_tail_invariance_audit: dict[str, Any] | None = None\n    if distribution_requested:\n        assert interval_policy is not None\n        prediction_frame, lower_tail_invariance_audit = (\n            attach_lower_tail_invariance_audit(\n                evaluation,\n                prediction_frame,\n                loaded_interval_policy,\n                full_threshold_seats=full_threshold_seats,\n            )\n        )\n        intervals, interval_bootstrap, probabilities = distribution_metric_tables(\n            evaluation,\n            prediction_frame,\n            bootstrap_seed=bootstrap_seed,\n            bootstrap_repeats=bootstrap_repeats,\n        )\n    else:\n        intervals = pd.DataFrame(\n            columns=(\n                "scope",\n                "date",\n                "split",\n                "interval_variant",\n                "target_population",\n                "lower_tail_refinement_status",\n                "rows",\n                "events",\n                "coverage_90",\n                "mean_width_90",\n                "weighted_interval_score_90",\n            )\n        )\n        interval_bootstrap = pd.DataFrame(\n            columns=(\n                "scope",\n                "date",\n                "split",\n                "target_population",\n                "lower_tail_refinement_status",\n                "metric",\n                "unit",\n                "seed",\n                "repeats",\n                "rows",\n                "events",\n                "trips",\n                "bootstrap_status",\n                "observed_delta",\n                "lower_95",\n                "median",\n                "upper_95",\n                "probability_refinement_better",\n            )\n        )\n        probabilities = pd.DataFrame(\n            columns=(\n                "scope",\n                "date",\n                "split",\n                "probability_model",\n                "calibration_status",\n                "classification_status",\n                "rows",\n                "events",\n                "positive_events",\n                "negative_events",\n                "weighted_prevalence",\n                "mean_probability",\n                "brier",\n                "log_loss",\n                "average_precision",\n                "roc_auc",\n            )\n        )\n\n    predictions = _prediction_audit_frame(evaluation, prediction_frame, persistence)\n    available_dates = sorted(normalized_dates.unique().tolist())\n    by_date = []\n    for date in selected_dates:\n        day = evaluation.loc[evaluation["date"].eq(date)]\n        by_date.append(\n            {\n                "date": date,\n                "rows": int(len(day)),\n                "events": int(day["event_id"].nunique()),\n                "trips": int(day["trip_id"].nunique()),\n                "low_0_5_events": int(\n                    day.loc[day["label_seats"].le(5), "event_id"].nunique()\n                ),\n            }\n        )\n    distribution_provenance: dict[str, Any] | None = None\n    if distribution_requested:\n        assert resolved_flow_cache is not None\n        assert interval_policy is not None\n        assert interval_policy_sha256 is not None\n        assert flow_artifact_sha256 is not None\n        distribution_provenance = {\n            "attached": True,\n            "flow_artifact": str(resolved_flow_cache),\n            "flow_artifact_sha256": flow_artifact_sha256,\n            "interval_policy": str(Path(interval_policy).resolve()),\n            "interval_policy_sha256": interval_policy_sha256,\n            "full_threshold_seats": float(full_threshold_seats),\n            "lower_tail_invariance_audit": lower_tail_invariance_audit,\n        }\n\n    pinned_artifacts: list[tuple[str, Path, str]] = [\n        ("frozen model summary", summary_path, model_summary_sha256),\n        ("snapshot cache", snapshot_cache, snapshot_cache_sha256),\n    ]\n    pinned_artifacts.extend(\n        ("frozen model component", path, sha256)\n        for path, sha256 in model_artifact_hashes.items()\n    )\n    pinned_artifacts.extend(\n        ("inference runtime source", path, sha256)\n        for path, sha256 in runtime_source_hashes.items()\n    )\n    pinned_artifacts.extend(\n        ("snapshot cache bundle", path, sha256)\n        for path, sha256 in cache_bundle_hashes.items()\n    )\n    if distribution_requested:\n        assert interval_policy is not None\n        assert interval_policy_sha256 is not None\n        assert resolved_flow_cache is not None\n        assert flow_artifact_sha256 is not None\n        pinned_artifacts.extend(\n            [\n                ("frozen interval policy", interval_policy, interval_policy_sha256),\n                (\n                    "frozen distribution flow",\n                    resolved_flow_cache,\n                    flow_artifact_sha256,\n                ),\n            ]\n        )\n    for field, path, expected_sha256 in pinned_artifacts:\n        if not path.is_file() or file_sha256(path) != expected_sha256:\n            raise RuntimeError(\n                f"{field}가 prospective 평가 도중 변경되어 결과를 폐기합니다: {path}"\n            )\n\n    summary: dict[str, Any] = {\n        "schema_version": 2,\n        "evaluation_kind": "frozen_prospective_no_retraining_or_reselection",\n        "generated_at_utc": datetime.now(timezone.utc).isoformat(),\n        "protocol": {\n            "eligibility_rule": (\n                "date > locked confirmation date and date not declared as "\n                "development, stress, or confirmation"\n            ),\n            "locked_confirmation_cutoff": cutoff,\n            "forbidden_dates": forbidden,\n            "policy_declared_dates": declared_policy_dates,\n            "training_performed": False,\n            "parameter_selection_performed": False,\n            "model_selection_performed": False,\n            "bootstrap": {\n                "unit": "trip_id",\n                "seed": int(bootstrap_seed),\n                "repeats": int(bootstrap_repeats),\n            },\n        },\n        "model": {\n            "selected_name": service.selected_name,\n            "results_dir": str(results_dir.resolve()),\n            "summary_path": str(summary_path.resolve()),\n            "summary_sha256": model_summary_sha256,\n            "component_artifact_sha256": {\n                str(path.resolve()): sha256\n                for path, sha256 in model_artifact_hashes.items()\n            },\n            "runtime_source_sha256": {\n                str(path.resolve()): sha256\n                for path, sha256 in runtime_source_hashes.items()\n            },\n            "meta_shadow": meta_shadow_provenance,\n        },\n        "input": {\n            "snapshot_cache": str(snapshot_cache.resolve()),\n            "snapshot_cache_sha256": snapshot_cache_sha256,\n            "snapshot_cache_size_bytes": snapshot_cache_size_bytes,\n            "cache_available_dates": available_dates,\n            "cache_commit_metadata": cache_commit_metadata,\n            "cache_bundle_artifact_sha256": {\n                str(path.resolve()): sha256\n                for path, sha256 in cache_bundle_hashes.items()\n            },\n            "cache_rows": int(len(snapshots)),\n            "cache_events": int(snapshots["event_id"].nunique()),\n            "cache_trips": int(snapshots["trip_id"].nunique()),\n            "evaluated_dates": selected_dates,\n            "rows": int(len(evaluation)),\n            "events": int(evaluation["event_id"].nunique()),\n            "trips": int(evaluation["trip_id"].nunique()),\n            "evaluated_rows": int(len(evaluation)),\n            "evaluated_events": int(evaluation["event_id"].nunique()),\n            "evaluated_trips": int(evaluation["trip_id"].nunique()),\n            "by_date": by_date,\n        },\n        "distribution": distribution_provenance or {"attached": False},\n        "metrics": {\n            "point": _frame_records(point),\n            "stop_band": _frame_records(bands),\n            "point_bootstrap_comparisons": _frame_records(bootstrap),\n            "interval": _frame_records(intervals),\n            "interval_bootstrap_base_vs_refined": _frame_records(\n                interval_bootstrap\n            ),\n            "probability_arrival_seats_le_5": _frame_records(probabilities),\n        },\n    }\n    return ProspectiveEvaluation(\n        summary=_json_ready(summary),\n        predictions=predictions,\n        point_metrics=point,\n        stop_band_metrics=bands,\n        bootstrap_vs_persistence=bootstrap,\n        interval_metrics=intervals,\n        interval_bootstrap=interval_bootstrap,\n        probability_metrics=probabilities,\n    )\n\n\ndef _safe_run_id(run_id: str) -> str:\n    if not run_id or run_id in {".", ".."} or Path(run_id).name != run_id:\n        raise ValueError("run_id는 경로 구분자 없는 단일 디렉터리 이름이어야 합니다.")\n    return run_id\n\n\ndef evaluated_dates_in_output_root(output_root: Path) -> set[str]:\n    """Read append-only claims plus legacy run summaries conservatively."""\n\n    output_root = Path(output_root)\n    evaluated: set[str] = set()\n    claim_dir = output_root / ".date_claims"\n    if claim_dir.is_dir():\n        for claim in claim_dir.glob("*.json"):\n            evaluated.add(_iso_date(claim.stem, field="prospective date claim"))\n    if output_root.is_dir():\n        for summary_path in output_root.glob("*/summary.json"):\n            try:\n                summary = json.loads(summary_path.read_text(encoding="utf-8"))\n                dates = summary["input"]["evaluated_dates"]\n            except (KeyError, TypeError, json.JSONDecodeError) as error:\n                raise ValueError(\n                    "기존 prospective summary를 확인할 수 없어 재평가를 "\n                    f"중단합니다: {summary_path}"\n                ) from error\n            evaluated.update(\n                _iso_date(date, field="existing prospective result")\n                for date in dates\n            )\n    return evaluated\n\n\ndef claim_prospective_dates(\n    output_root: Path,\n    dates: Sequence[str],\n    *,\n    claim_id: str,\n    artifact_sha256: dict[str, str],\n    evaluation_scope: dict[str, Any],\n) -> list[Path]:\n    """Atomically claim each date once before any metrics are revealed."""\n\n    output_root = Path(output_root)\n    normalized_dates = sorted(\n        {_iso_date(date, field="prospective claim") for date in dates}\n    )\n    if not normalized_dates:\n        raise ValueError("claim할 prospective 날짜가 없습니다.")\n    if not artifact_sha256:\n        raise ValueError("prospective claim artifact manifest가 비어 있습니다.")\n    normalized_artifacts = {\n        str(name): str(digest) for name, digest in sorted(artifact_sha256.items())\n    }\n    invalid_hashes = [\n        name\n        for name, digest in normalized_artifacts.items()\n        if len(digest) != 64\n        or any(character not in "0123456789abcdef" for character in digest)\n    ]\n    if invalid_hashes:\n        raise ValueError(\n            f"prospective claim artifact SHA가 올바르지 않습니다: {invalid_hashes}"\n        )\n    already_evaluated = evaluated_dates_in_output_root(output_root)\n    duplicate = sorted(set(normalized_dates) & already_evaluated)\n    if duplicate:\n        raise ValueError(\n            "canonical registry에서 이미 평가/claim한 날짜는 다시 열 수 "\n            f"없습니다: {duplicate}"\n        )\n    output_root.mkdir(parents=True, exist_ok=True)\n    claim_dir = output_root / ".date_claims"\n    claim_dir.mkdir(exist_ok=True)\n    created: list[Path] = []\n    for date in normalized_dates:\n        claim_path = claim_dir / f"{date}.json"\n        claim = {\n            "schema_version": 1,\n            "date": date,\n            "claim_id": claim_id,\n            "claimed_at_utc": datetime.now(timezone.utc).isoformat(),\n            "artifact_sha256": normalized_artifacts,\n            "evaluation_scope": _json_ready(evaluation_scope),\n            "status": "claimed_before_evaluation",\n        }\n        try:\n            with claim_path.open("x", encoding="utf-8") as output:\n                json.dump(claim, output, ensure_ascii=False, indent=2)\n        except FileExistsError as error:\n            # Keep any earlier claims from this multi-date run.  Conservatively\n            # blocking a date is safer than allowing partial cherry-picking.\n            raise ValueError(\n                "prospective 날짜가 concurrent run에서 이미 claim되었습니다: "\n                f"{date}"\n            ) from error\n        created.append(claim_path)\n    return created\n\n\ndef append_claim_event(\n    output_root: Path,\n    claim_paths: Sequence[Path],\n    *,\n    claim_id: str,\n    status: str,\n    detail: str,\n) -> None:\n    if status not in {"completed", "failed"}:\n        raise ValueError(f"알 수 없는 prospective claim event status: {status}")\n    event_dir = Path(output_root) / ".date_claim_events"\n    event_dir.mkdir(exist_ok=True)\n    for claim_path in claim_paths:\n        date = _iso_date(claim_path.stem, field="prospective claim event")\n        event_path = event_dir / f"{date}__{claim_id}__{status}.json"\n        event = {\n            "schema_version": 1,\n            "date": date,\n            "claim_id": claim_id,\n            "status": status,\n            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),\n            "detail": detail,\n        }\n        with event_path.open("x", encoding="utf-8") as output:\n            json.dump(event, output, ensure_ascii=False, indent=2)\n\n\ndef prospective_artifact_manifest(\n    *,\n    model_summary_sha256: str,\n    snapshot_cache_sha256: str,\n    interval_policy_sha256: str,\n    flow_artifact_sha256: str,\n    model_artifact_hashes: dict[Path, str],\n    runtime_source_hashes: dict[Path, str],\n    cache_bundle_hashes: dict[Path, str],\n) -> dict[str, str]:\n    """Flatten all frozen inputs into a claim-comparable SHA manifest."""\n\n    manifest = {\n        "model_summary": model_summary_sha256,\n        "snapshot_cache": snapshot_cache_sha256,\n        "interval_policy": interval_policy_sha256,\n        "distribution_flow": flow_artifact_sha256,\n    }\n    for prefix, hashes in (\n        ("model_component", model_artifact_hashes),\n        ("runtime_source", runtime_source_hashes),\n        ("cache_bundle", cache_bundle_hashes),\n    ):\n        manifest.update(\n            {\n                f"{prefix}:{path.resolve()}": sha256\n                for path, sha256 in hashes.items()\n            }\n        )\n    return dict(sorted(manifest.items()))\n\n\ndef evaluation_artifact_manifest(summary: dict[str, Any]) -> dict[str, str]:\n    """Recover the exact frozen-input manifest recorded by an evaluation."""\n\n    model = summary["model"]\n    cache = summary["input"]\n    distribution = summary["distribution"]\n    manifest = {\n        "model_summary": str(model["summary_sha256"]),\n        "snapshot_cache": str(cache["snapshot_cache_sha256"]),\n        "interval_policy": str(distribution["interval_policy_sha256"]),\n        "distribution_flow": str(distribution["flow_artifact_sha256"]),\n    }\n    for prefix, hashes in (\n        ("model_component", model["component_artifact_sha256"]),\n        ("runtime_source", model["runtime_source_sha256"]),\n        ("cache_bundle", cache["cache_bundle_artifact_sha256"]),\n    ):\n        manifest.update(\n            {f"{prefix}:{Path(path).resolve()}": str(sha256) for path, sha256 in hashes.items()}\n        )\n    return dict(sorted(manifest.items()))\n\n\ndef write_evaluation_outputs(\n    evaluation: ProspectiveEvaluation,\n    output_root: Path,\n    *,\n    results_dir: Path,\n    run_id: str | None = None,\n) -> Path:\n    """Atomically reserve a new run directory and never overwrite artifacts."""\n\n    output_root = Path(output_root)\n    results_resolved = Path(results_dir).resolve()\n    root_resolved = output_root.resolve()\n    if root_resolved == results_resolved or root_resolved.is_relative_to(results_resolved):\n        raise ValueError(\n            "prospective output은 frozen model results_dir와 분리해야 합니다."\n        )\n    output_root.mkdir(parents=True, exist_ok=True)\n    if run_id is None:\n        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n        dates = evaluation.summary["input"]["evaluated_dates"]\n        run_id = f"prospective_{dates[0]}_{dates[-1]}_{timestamp}"\n    run_id = _safe_run_id(run_id)\n    run_dir = output_root / run_id\n    try:\n        run_dir.mkdir(parents=False, exist_ok=False)\n    except FileExistsError as error:\n        raise FileExistsError(\n            f"기존 prospective 결과를 덮어쓰지 않습니다: {run_dir}"\n        ) from error\n\n    output_frames = {\n        "predictions.csv": evaluation.predictions,\n        "point_metrics.csv": evaluation.point_metrics,\n        "stop_band_metrics.csv": evaluation.stop_band_metrics,\n        "point_bootstrap_comparisons.csv": evaluation.bootstrap_vs_persistence,\n        "interval_metrics.csv": evaluation.interval_metrics,\n        "interval_bootstrap_base_vs_refined.csv": evaluation.interval_bootstrap,\n        "low_5_probability_metrics.csv": evaluation.probability_metrics,\n    }\n    for filename, frame in output_frames.items():\n        frame.to_csv(run_dir / filename, index=False)\n    summary = dict(evaluation.summary)\n    summary["output"] = {\n        "run_dir": str(run_dir.resolve()),\n        "files": ["summary.json", *output_frames.keys()],\n    }\n    (run_dir / "summary.json").write_text(\n        json.dumps(_json_ready(summary), ensure_ascii=False, indent=2, allow_nan=False),\n        encoding="utf-8",\n    )\n    return run_dir\n\n\ndef run_prospective_evaluation(\n    results_dir: Path,\n    snapshot_cache: Path,\n    output_root: Path,\n    **kwargs: Any,\n) -> Path:\n    """One-shot safe runner: claim pristine dates, evaluate, then append result."""\n\n    results_dir = Path(results_dir)\n    snapshot_cache = Path(snapshot_cache)\n    output_root = Path(output_root)\n    if output_root.resolve() == results_dir.resolve() or output_root.resolve().is_relative_to(\n        results_dir.resolve()\n    ):\n        raise ValueError(\n            "prospective output은 frozen model results_dir와 분리해야 합니다."\n        )\n    snapshot_resolved = snapshot_cache.resolve()\n    output_resolved = output_root.resolve()\n    if (\n        output_resolved in {snapshot_resolved, snapshot_resolved.parent}\n        or output_resolved.is_relative_to(snapshot_resolved.parent)\n    ):\n        raise ValueError(\n            "prospective output-root는 snapshot cache 파일/디렉터리와 "\n            "분리해야 합니다."\n        )\n    interval_policy = kwargs.get("interval_policy")\n    if interval_policy is None:\n        raise ValueError(\n            "safe prospective runner에는 frozen interval policy가 필수입니다."\n        )\n    interval_policy = Path(interval_policy)\n    kwargs["interval_policy"] = interval_policy\n    meta_shadow_dir = kwargs.get("meta_shadow_dir")\n    if meta_shadow_dir is not None:\n        meta_shadow_dir = Path(meta_shadow_dir)\n        kwargs["meta_shadow_dir"] = meta_shadow_dir\n        meta_root = meta_shadow_dir.resolve()\n        if output_resolved == meta_root or output_resolved.is_relative_to(meta_root):\n            raise ValueError(\n                "prospective output-root는 frozen meta shadow bundle과 분리해야 합니다."\n            )\n    policy_root = interval_policy.resolve().parent\n    if output_resolved == policy_root or output_resolved.is_relative_to(policy_root):\n        raise ValueError(\n            "prospective output-root는 frozen interval policy bundle과 "\n            "분리해야 합니다."\n        )\n    requested_dates = kwargs.pop("dates", None)\n    summary_payload, summary_sha256 = read_stable_bytes(\n        results_dir / "summary.json", field="prospective claim model summary"\n    )\n    model_summary = json.loads(summary_payload.decode("utf-8"))\n    snapshot_payload, snapshot_sha256 = read_stable_bytes(\n        snapshot_cache, field="prospective claim snapshot cache"\n    )\n    snapshots = pd.read_pickle(io.BytesIO(snapshot_payload))\n    eligible_dates, _, _ = resolve_pristine_dates(snapshots, model_summary, None)\n    today_kst = datetime.now(ZoneInfo("Asia/Seoul")).date().isoformat()\n    closed_dates = [date for date in eligible_dates if date < today_kst]\n    policy_payload, policy_sha256 = read_stable_bytes(\n        interval_policy, field="prospective claim interval policy"\n    )\n    claim_policy = validate_interval_policy(json.loads(policy_payload.decode("utf-8")))\n    provenance = claim_policy.get("provenance")\n    distribution_provenance = (\n        provenance.get("distribution") if isinstance(provenance, dict) else None\n    )\n    point_model_provenance = (\n        provenance.get("point_model") if isinstance(provenance, dict) else None\n    )\n    flow_artifact = (\n        distribution_provenance.get("flow_artifact")\n        if isinstance(distribution_provenance, dict)\n        else None\n    )\n    if (\n        not isinstance(point_model_provenance, dict)\n        or not isinstance(flow_artifact, dict)\n    ):\n        raise ValueError(\n            "safe prospective runner는 point-model provenance와 byte-locked "\n            "non-null flow_artifact 객체가 있는 policy만 허용합니다."\n        )\n    if point_model_provenance.get("summary_sha256") != summary_sha256:\n        raise ValueError(\n            "interval policy가 현재 frozen point-model summary에 결합되어 "\n            "있지 않습니다."\n        )\n    claim_registry = canonical_claim_registry()\n    registry_root = PROSPECTIVE_CLAIM_REGISTRY_ROOT.resolve()\n    if output_resolved == registry_root or output_resolved.is_relative_to(\n        registry_root\n    ):\n        raise ValueError(\n            "prospective output-root는 canonical date-claim registry와 "\n            "분리해야 합니다."\n        )\n    consumed_dates = evaluated_dates_in_output_root(claim_registry)\n    unconsumed_closed_dates = sorted(set(closed_dates) - consumed_dates)\n    if not unconsumed_closed_dates:\n        raise ValueError(\n            "평가하지 않은 종료된 pristine 날짜가 없습니다"\n            f"(today={today_kst}, eligible={eligible_dates}, "\n            f"consumed={sorted(consumed_dates)})."\n        )\n    if requested_dates is None or len(requested_dates) == 0:\n        selected_dates = unconsumed_closed_dates\n    else:\n        normalized_requested = sorted(\n            {_iso_date(date, field="requested evaluation") for date in requested_dates}\n        )\n        if normalized_requested != unconsumed_closed_dates:\n            raise ValueError(\n                "--dates cherry-pick은 허용되지 않습니다. 요청 날짜는 아직 "\n                "평가하지 않은 종료된 pristine 날짜 전체와 정확히 같아야 합니다: "\n                f"required={unconsumed_closed_dates}, requested={normalized_requested}"\n            )\n        selected_dates = normalized_requested\n    validate_closed_evaluation_dates(selected_dates, today_kst=today_kst)\n    day_completeness_audit = validate_complete_evaluation_days(\n        snapshots, selected_dates\n    )\n\n    declared_dates = policy_calibration_dates(claim_policy)\n    if not declared_dates or max(declared_dates) >= min(selected_dates):\n        raise ValueError(\n            "interval policy chronology가 prospective claim보다 엄격히 "\n            "이전임을 증명할 수 없습니다."\n        )\n\n    # Mutable cache commit validation intentionally happens only after the\n    # no-pristine/closed-date gates, but before a date is irreversibly claimed.\n    _, cache_bundle_hashes = validate_snapshot_cache_commit(\n        snapshot_cache, snapshot_sha256, required=True\n    )\n    model_artifact_hashes = pin_selected_model_artifacts(results_dir, model_summary)\n    if meta_shadow_dir is not None:\n        _, meta_shadow_hashes = pin_meta_shadow_artifacts(\n            meta_shadow_dir,\n            results_dir,\n            model_summary,\n            summary_sha256,\n        )\n        model_artifact_hashes.update(meta_shadow_hashes)\n    runtime_source_hashes = pin_runtime_source_artifacts()\n    resolved_flow_cache = resolve_distribution_flow_artifact(\n        kwargs.get("distribution_flow_cache"), interval_policy\n    )\n    if resolved_flow_cache is None:\n        raise ValueError("frozen distribution flow artifact를 확인할 수 없습니다.")\n    _, flow_sha256 = read_stable_bytes(\n        resolved_flow_cache, field="prospective claim frozen distribution flow"\n    )\n    claim_manifest = prospective_artifact_manifest(\n        model_summary_sha256=summary_sha256,\n        snapshot_cache_sha256=snapshot_sha256,\n        interval_policy_sha256=policy_sha256,\n        flow_artifact_sha256=flow_sha256,\n        model_artifact_hashes=model_artifact_hashes,\n        runtime_source_hashes=runtime_source_hashes,\n        cache_bundle_hashes=cache_bundle_hashes,\n    )\n    run_id = kwargs.pop("run_id", None)\n    if run_id is None:\n        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n        run_id = (\n            f"prospective_{selected_dates[0]}_{selected_dates[-1]}_{timestamp}"\n        )\n    run_id = _safe_run_id(run_id)\n    if (output_root / run_id).exists():\n        raise FileExistsError(\n            f"기존 prospective 결과를 덮어쓰지 않습니다: {output_root / run_id}"\n        )\n    claim_paths = claim_prospective_dates(\n        claim_registry,\n        selected_dates,\n        claim_id=run_id,\n        artifact_sha256=claim_manifest,\n        evaluation_scope={\n            "kind": (\n                "point_interval_probability_meta_shadow"\n                if meta_shadow_dir is not None\n                else "point_interval_probability"\n            ),\n            "day_completeness_gate": day_completeness_audit,\n            "full_threshold_seats": float(kwargs.get("full_threshold_seats", 0.0)),\n            "bootstrap_seed": int(kwargs.get("bootstrap_seed", 20260812)),\n            "bootstrap_repeats": int(kwargs.get("bootstrap_repeats", 5_000)),\n        },\n    )\n    try:\n        evaluation = _evaluate_claimed_prospective(\n            results_dir,\n            snapshot_cache,\n            dates=selected_dates,\n            **kwargs,\n        )\n        evaluation.summary["input"]["day_completeness_gate"] = (\n            day_completeness_audit\n        )\n        if evaluation_artifact_manifest(evaluation.summary) != claim_manifest:\n            raise RuntimeError(\n                "prospective date claim provenance와 실제 평가 입력 SHA가 "\n                "일치하지 않습니다."\n            )\n        run_dir = write_evaluation_outputs(\n            evaluation,\n            output_root,\n            results_dir=results_dir,\n            run_id=run_id,\n        )\n    except Exception as error:\n        append_claim_event(\n            claim_registry,\n            claim_paths,\n            claim_id=run_id,\n            status="failed",\n            detail=f"{type(error).__name__}: {error}"[:2000],\n        )\n        raise\n    append_claim_event(\n        claim_registry,\n        claim_paths,\n        claim_id=run_id,\n        status="completed",\n        detail=str(run_dir.resolve()),\n    )\n    return run_dir\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description=(\n            "Frozen 잔여좌석 모델을 locked confirmation 이후 pristine cache "\n            "날짜에 재학습 없이 평가"\n        )\n    )\n    parser.add_argument(\n        "--results-dir",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--dates",\n        nargs="*",\n        help="평가할 YYYY-MM-DD 목록; 생략하면 cutoff 이후 cache 날짜 전부",\n    )\n    parser.add_argument(\n        "--output-root",\n        type=Path,\n        default=Path("analysis/prospective_evaluation_results"),\n    )\n    parser.add_argument("--run-id", help="생략하면 UTC timestamp 기반 이름 사용")\n    parser.add_argument(\n        "--distribution-flow-cache",\n        type=Path,\n        help=(\n            "선택 사항: legacy policy용 stop-flow pickle; 새 policy는 "\n            "policy 옆 frozen artifact를 자동 사용"\n        ),\n    )\n    parser.add_argument(\n        "--interval-policy",\n        type=Path,\n        required=True,\n        help=(\n            "필수: provenance 및 byte-locked flow를 포함한 frozen 90%% "\n            "interval/probability policy JSON"\n        ),\n    )\n    parser.add_argument(\n        "--meta-shadow-dir",\n        type=Path,\n        default=None,\n        help=(\n            "선택 사항: 고정 disagreement meta shadow bundle; 평가 결과로 "\n            "재학습·승격하지 않음"\n        ),\n    )\n    parser.add_argument("--full-threshold-seats", type=float, default=0.0)\n    parser.add_argument("--bootstrap-seed", type=int, default=20260812)\n    parser.add_argument("--bootstrap-repeats", type=int, default=5_000)\n    args = parser.parse_args()\n\n    run_dir = run_prospective_evaluation(\n        args.results_dir,\n        args.snapshot_cache,\n        args.output_root,\n        dates=args.dates,\n        distribution_flow_cache=args.distribution_flow_cache,\n        interval_policy=args.interval_policy,\n        meta_shadow_dir=args.meta_shadow_dir,\n        full_threshold_seats=args.full_threshold_seats,\n        bootstrap_seed=args.bootstrap_seed,\n        bootstrap_repeats=args.bootstrap_repeats,\n        run_id=args.run_id,\n    )\n    result = {\n        "status": "completed",\n        "output_dir": str(run_dir.resolve()),\n        "summary": str((run_dir / "summary.json").resolve()),\n    }\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/hypothesis_model_search.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sqlite3\nimport time\nfrom dataclasses import asdict, dataclass, field, replace\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\nimport joblib\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.ensemble import (\n    ExtraTreesRegressor,\n    HistGradientBoostingRegressor,\n    RandomForestRegressor,\n)\nfrom sklearn.pipeline import Pipeline\n\nfrom all_prearrival_seat_regression import (\n    DYNAMIC_ALL_PREARRIVAL,\n    ENGINEERED_ALL_PREARRIVAL,\n    build_all_prearrival_table,\n    clip_seats,\n    event_bias,\n    event_weights,\n)\nfrom model_feasibility import (\n    FeatureSet,\n    _as_records,\n    build_model_table,\n    build_visits,\n    json_ready,\n    make_preprocessor,\n)\nfrom tminus_feasibility import ROUTE_ID, ROUTE_NAME, prepare_raw_locations\n\n\nCACHE_VERSION = 6\nDEFAULT_DEVELOPMENT_DATES = (\n    "2026-08-04",\n    "2026-08-05",\n    "2026-08-06",\n    "2026-08-07",\n    "2026-08-10",\n)\nDEFAULT_STRESS_DATES = ("2026-08-08", "2026-08-09")\nDEFAULT_TEST_DATE = "2026-08-11"\nHISTORICAL_FEATURES = (\n    "historical_pair_delta",\n    "historical_pair_log_count",\n    "historical_pair_time_delta",\n    "historical_pair_time_log_count",\n)\nROUTE_PROFILE_FEATURES = (\n    "route_profile_delta",\n    "route_profile_seats",\n    "route_profile_uncertainty",\n    "route_profile_fallback_share",\n)\nCAPACITY_44_FEATURES = (\n    "observed_ceiling_capacity",\n    "observed_ceiling_load_ratio",\n    "observed_ceiling_load_gap",\n)\nOBSERVED_SEAT_CEILING_PARAM = "observed_seat_ceiling_clip"\nOBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF = "2026-08-04"\nPREVIOUS_BUS_SEAT_FEATURES = (\n    "previous_bus_departure_seats",\n    "previous_bus_departure_age_minutes",\n    "previous_bus_departure_missing",\n)\nNORMALIZED_PREVIOUS_BUS_FEATURES = (\n    "previous_bus_departure_capacity",\n    "previous_bus_departure_load_ratio",\n    "previous_bus_same_capacity",\n    "previous_bus_departure_age_minutes",\n    "previous_bus_projected_headway_minutes",\n    "previous_bus_freshness_exp",\n    "previous_bus_departure_missing",\n    "previous_3_bus_departure_load_ratio_mean",\n    "previous_3_bus_departure_load_ratio_std",\n    "previous_3_bus_departure_load_ratio_trend",\n    "previous_bus_was_full_normalized",\n    "previous_bus_was_low_10pct",\n)\nPAIR_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_source_target_pair",\n    numeric=DYNAMIC_ALL_PREARRIVAL.numeric,\n    categorical=(\n        *DYNAMIC_ALL_PREARRIVAL.categorical,\n        "snapshot_target_pair_cat",\n    ),\n)\nHISTORICAL_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_historical_profiles",\n    numeric=(*PAIR_ALL_PREARRIVAL.numeric, *HISTORICAL_FEATURES),\n    categorical=PAIR_ALL_PREARRIVAL.categorical,\n)\nROUTE_PROFILE_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_route_profile",\n    numeric=(*DYNAMIC_ALL_PREARRIVAL.numeric, *ROUTE_PROFILE_FEATURES),\n    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,\n)\nCAPACITY_44_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_observed_44_70_capacity",\n    numeric=(*DYNAMIC_ALL_PREARRIVAL.numeric, *CAPACITY_44_FEATURES),\n    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,\n)\nPREVIOUS_BUS_SEAT_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_previous_bus_seat",\n    numeric=(*DYNAMIC_ALL_PREARRIVAL.numeric, *PREVIOUS_BUS_SEAT_FEATURES),\n    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,\n)\nPREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_previous_bus_seat_observed_44_70_capacity",\n    numeric=(\n        *DYNAMIC_ALL_PREARRIVAL.numeric,\n        *PREVIOUS_BUS_SEAT_FEATURES,\n        *CAPACITY_44_FEATURES,\n    ),\n    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,\n)\nNORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_normalized_previous_bus_observed_44_70_capacity",\n    numeric=(\n        *DYNAMIC_ALL_PREARRIVAL.numeric,\n        *NORMALIZED_PREVIOUS_BUS_FEATURES,\n        *CAPACITY_44_FEATURES,\n    ),\n    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,\n)\nPREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_previous_bus_observed_44_70_capacity",\n    numeric=(*ENGINEERED_ALL_PREARRIVAL.numeric, *CAPACITY_44_FEATURES),\n    categorical=ENGINEERED_ALL_PREARRIVAL.categorical,\n)\n\n\n@dataclass\nclass Candidate:\n    name: str\n    stage: str\n    why: str\n    if_works: str\n    if_fails: str\n    model_kind: str\n    target_kind: str = "delta"\n    feature_variant: str = "dynamic"\n    low_weight: float = 1.0\n    weighting_kind: str = "event"\n    far_weight: float = 1.0\n    far_threshold: int = 6\n    gap_weight_power: float = 0.0\n    params: dict[str, Any] = field(default_factory=dict)\n\n\n@dataclass\nclass EnsembleSpec:\n    """앙상블 구성과 미래 추론에 쓰는 배포용 bias를 보관한다.\n\n    ``bias``는 모든 개발 OOF residual로 추정한 배포용 보정값이다. 모델\n    선택용 OOF prediction에는 이 값을 적용하지 않고, 각 날짜보다 앞선 OOF\n    fold에서만 추정한 ``oof_biases``를 적용한다.\n    """\n\n    name: str\n    kind: str\n    components: tuple[str, ...]\n    params: dict[str, float]\n    bias: float = 0.0\n    oof_biases: dict[str, float] = field(default_factory=dict)\n\n\ndef load_analysis_data(\n    db_path: Path,\n    route_id: str,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """수집기 DB와 API 로컬 캐시를 같은 분석 스키마로 읽는다."""\n    uri = f"file:{db_path.resolve()}?mode=ro"\n    with sqlite3.connect(uri, uri=True) as connection:\n        tables = {\n            str(row[0])\n            for row in connection.execute(\n                "SELECT name FROM sqlite_master WHERE type = \'table\'"\n            )\n        }\n        if "bus_locations" in tables:\n            location_query = """\n                SELECT run_id, observed_at_kst, query_time, route_id, vehicle_id,\n                       plate_no, station_id, station_seq, remaining_seats,\n                       low_plate, state_code\n                FROM bus_locations\n                WHERE route_id = ?\n                ORDER BY vehicle_id, observed_at_kst, run_id\n            """\n        elif "location_history" in tables:\n            location_query = """\n                SELECT rowid AS run_id, observed_at AS observed_at_kst,\n                       query_time, route_id, vehicle_id, plate_no, station_id,\n                       station_seq, remaining_seats, low_plate, state_code\n                FROM location_history\n                WHERE route_id = ?\n                ORDER BY vehicle_id, observed_at, rowid\n            """\n        else:\n            raise ValueError(\n                f"{db_path}에 bus_locations 또는 location_history가 없습니다."\n            )\n        locations = pd.read_sql_query(\n            location_query, connection, params=(route_id,)\n        )\n        stations = pd.read_sql_query(\n            """\n            SELECT route_id, station_id, station_seq, station_name, mobile_no,\n                   region_name, x, y, center_yn\n            FROM route_stations\n            WHERE route_id = ?\n            ORDER BY station_seq\n            """,\n            connection,\n            params=(route_id,),\n        )\n    if locations.empty or stations.empty:\n        raise ValueError(f"노선 {route_id}의 위치 또는 정류장 데이터가 없습니다.")\n    locations["observed_at"] = pd.to_datetime(locations["observed_at_kst"])\n    return locations, stations\n\n\ndef cache_fingerprint(db_path: Path) -> dict[str, Any]:\n    stat = db_path.stat()\n    return {\n        "cache_version": CACHE_VERSION,\n        "source": str(db_path.resolve()),\n        "source_size": int(stat.st_size),\n        "source_mtime_ns": int(stat.st_mtime_ns),\n        "route_id": ROUTE_ID,\n        "label_quality": "A",\n    }\n\n\ndef cache_payload_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as payload:\n        for chunk in iter(lambda: payload.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef cache_payload_hashes_match(\n    metadata: dict[str, Any],\n    snapshot_path: Path,\n    flow_path: Path,\n) -> bool:\n    """Accept a cache commit marker only when both payload bytes match it."""\n\n    payload_hashes = metadata.get("payload_sha256")\n    if not isinstance(payload_hashes, dict):\n        return False\n    expected_snapshot = str(payload_hashes.get("snapshots", ""))\n    expected_flow = str(payload_hashes.get("stop_flows", ""))\n    if len(expected_snapshot) != 64 or len(expected_flow) != 64:\n        return False\n    try:\n        return (\n            cache_payload_sha256(snapshot_path) == expected_snapshot\n            and cache_payload_sha256(flow_path) == expected_flow\n        )\n    except OSError:\n        return False\n\n\ndef audit_route_output_support(\n    db_path: Path,\n    route_id: str,\n    *,\n    evidence_cutoff: str = OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF,\n) -> dict[str, Any]:\n    """Audit the raw route feed before enabling the empirical point projection.\n\n    Candidate eligibility uses only rows on or before the frozen evidence\n    cutoff. Later rows are a deployment-safety monitor and must never influence\n    selection; runtime inference falls back per row when it observes a changed\n    support.\n    """\n    uri = f"file:{db_path.resolve()}?mode=ro"\n    with sqlite3.connect(uri, uri=True) as connection:\n        tables = {\n            str(row[0])\n            for row in connection.execute(\n                "SELECT name FROM sqlite_master WHERE type = \'table\'"\n            )\n        }\n        if "bus_locations" in tables:\n            table = "bus_locations"\n            observed_column = "observed_at_kst"\n        elif "location_history" in tables:\n            table = "location_history"\n            observed_column = "observed_at"\n        else:\n            raise ValueError(\n                f"{db_path}에 bus_locations 또는 location_history가 없습니다."\n            )\n        raw = pd.read_sql_query(\n            f"""\n            SELECT vehicle_id, low_plate, remaining_seats,\n                   {observed_column} AS observed_at\n            FROM {table}\n            WHERE route_id = ? AND remaining_seats >= 0\n            """,\n            connection,\n            params=(route_id,),\n        )\n    if raw.empty:\n        raise ValueError("Route 1000 output-support audit에 유효 좌석 관측이 없습니다.")\n\n    category = pd.to_numeric(raw["low_plate"], errors="coerce")\n    integral = category.notna() & np.isclose(category % 1, 0.0)\n    supported = integral & category.isin([0.0, 2.0])\n    unknown_rows = int((~supported).sum())\n    seats = raw["remaining_seats"].to_numpy(dtype=float)\n    ordinary = supported.to_numpy() & category.eq(0).to_numpy()\n    double_decker = supported.to_numpy() & category.eq(2).to_numpy()\n    violation = (ordinary & (seats > 44.0)) | (double_decker & (seats > 70.0))\n    dates = raw["observed_at"].astype(str).str.slice(0, 10)\n    evidence = dates.le(evidence_cutoff).to_numpy()\n    unsupported = (~supported).to_numpy()\n\n    def category_profile(code: int, ceiling: float, mask: np.ndarray) -> dict[str, Any]:\n        scoped = raw.loc[mask]\n        evidence_mask = mask & evidence\n        evidence_seats = seats[evidence_mask]\n        return {\n            "category": code,\n            "ceiling": ceiling,\n            "rows": int(mask.sum()),\n            "vehicles": int(scoped["vehicle_id"].nunique()),\n            "maximum": float(seats[mask].max()) if mask.any() else None,\n            "ceiling_hits": int(np.sum(seats[mask] == ceiling)),\n            "evidence_rows": int(evidence_mask.sum()),\n            "evidence_maximum": (\n                float(evidence_seats.max()) if evidence_mask.any() else None\n            ),\n            "evidence_ceiling_hits": int(np.sum(evidence_seats == ceiling)),\n        }\n\n    profiles = {\n        "0": category_profile(0, 44.0, ordinary),\n        "2": category_profile(2, 70.0, double_decker),\n    }\n    selection_eligible = bool(\n        not (unsupported & evidence).any()\n        and not (violation & evidence).any()\n        and all(\n            profile["evidence_ceiling_hits"] > 0\n            for profile in profiles.values()\n        )\n    )\n    return {\n        "scope": f"route_{route_id}_current_fleet_only",\n        "source": str(db_path.resolve()),\n        "evidence_cutoff": evidence_cutoff,\n        "valid_seat_rows": int(len(raw)),\n        "observed_at_min": str(raw["observed_at"].min()),\n        "observed_at_max": str(raw["observed_at"].max()),\n        "unknown_or_unsupported_category_rows": unknown_rows,\n        "above_ceiling_rows": int(violation.sum()),\n        "selection_evidence_unknown_or_unsupported_rows": int(\n            (unsupported & evidence).sum()\n        ),\n        "selection_evidence_above_ceiling_rows": int(\n            (violation & evidence).sum()\n        ),\n        "post_cutoff_unknown_or_unsupported_rows": int(\n            (unsupported & ~evidence).sum()\n        ),\n        "post_cutoff_above_ceiling_rows": int((violation & ~evidence).sum()),\n        "by_category": profiles,\n        "selection_candidate_eligible": selection_eligible,\n        "current_deployment_support_safe": bool(\n            unknown_rows == 0 and not violation.any()\n        ),\n        "selection_rule": "use_only_rows_on_or_before_evidence_cutoff",\n        "post_cutoff_rule": "monitor_only_never_select; runtime row fallback or fail",\n    }\n\n\ndef load_or_build_snapshots(\n    db_path: Path,\n    cache_path: Path,\n    *,\n    rebuild: bool,\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\n    metadata_path = cache_path.with_suffix(cache_path.suffix + ".json")\n    flow_cache_path = cache_path.with_name(\n        cache_path.stem + "_stop_flows" + cache_path.suffix\n    )\n    fingerprint = cache_fingerprint(db_path)\n    if (\n        not rebuild\n        and cache_path.exists()\n        and flow_cache_path.exists()\n        and metadata_path.exists()\n    ):\n        existing = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if (\n            existing.get("fingerprint") == fingerprint\n            and cache_payload_hashes_match(\n                existing, cache_path, flow_cache_path\n            )\n        ):\n            return pd.read_pickle(cache_path), pd.read_pickle(flow_cache_path), {\n                "cache_hit": True,\n                "cache_path": str(cache_path),\n                **existing,\n            }\n\n    locations, stations = load_analysis_data(db_path, ROUTE_ID)\n    visits = build_visits(locations, stations)\n    table, turnaround_seq = build_model_table(\n        visits, stations, label_target="arrival"\n    )\n    source = table.loc[\n        table["is_peak"]\n        & table["label_quality"].eq("A")\n        & table["label_seats"].notna()\n    ].copy()\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n    snapshots = build_all_prearrival_table(\n        source,\n        visits,\n        by_vehicle,\n        turnaround_seq=turnaround_seq,\n        max_seq=int(stations["station_seq"].max()),\n    )\n    stop_flows = visits.loc[\n        (~visits["is_pass_node"])\n        & visits["observed_arrival_seats"].ge(0)\n        & visits["departure_seats"].ge(0)\n        & visits["arrival_seen"].notna()\n        & visits["departure_seen"].notna()\n    ].copy()\n    stop_flows["direction"] = np.where(\n        stop_flows["station_seq"].le(turnaround_seq), "to_city", "return"\n    )\n    stop_flows["date"] = stop_flows["arrival_seen"].dt.date.astype(str)\n    stop_flows["time_bin_2h"] = (\n        stop_flows["arrival_seen"].dt.hour // 2 * 2\n    ).astype(int)\n    stop_flows["stop_net"] = (\n        stop_flows["departure_seats"]\n        - stop_flows["observed_arrival_seats"]\n    ).astype(float)\n    stop_flows = stop_flows[\n        [\n            "vehicle_id",\n            "trip_id",\n            "station_seq",\n            "direction",\n            "date",\n            "arrival_seen",\n            "departure_seen",\n            "time_bin_2h",\n            "observed_arrival_seats",\n            "departure_seats",\n            "stop_net",\n        ]\n    ].sort_values(["arrival_seen", "station_seq", "vehicle_id"])\n    stop_flows.attrs["pass_node_sequences"] = sorted(\n        visits.loc[visits["is_pass_node"], "station_seq"]\n        .dropna()\n        .astype(int)\n        .unique()\n        .tolist()\n    )\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\n    snapshots.to_pickle(cache_path)\n    stop_flows.to_pickle(flow_cache_path)\n    metadata = {\n        "fingerprint": fingerprint,\n        "payload_sha256": {\n            "snapshots": cache_payload_sha256(cache_path),\n            "stop_flows": cache_payload_sha256(flow_cache_path),\n        },\n        "source_route_rows": int(len(locations)),\n        "source_observed_at_min": locations["observed_at"].min().isoformat(),\n        "source_observed_at_max": locations["observed_at"].max().isoformat(),\n        "rows": int(len(snapshots)),\n        "events": int(snapshots["event_id"].nunique()),\n        "stop_flow_rows": int(len(stop_flows)),\n        "dates": sorted(snapshots["date"].unique().tolist()),\n    }\n    metadata_path.write_text(\n        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    return snapshots, stop_flows, {\n        "cache_hit": False,\n        "cache_path": str(cache_path),\n        **metadata,\n    }\n\n\ndef smoothed_history_feature(\n    train: pd.DataFrame,\n    target: pd.DataFrame,\n    keys: tuple[str, ...],\n    *,\n    alpha: float,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\n    delta = (\n        train["label_seats"].to_numpy(dtype=float)\n        - train["snapshot_remaining_seats"].to_numpy(dtype=float)\n    )\n    total_sum = float(delta.sum())\n    total_count = len(delta)\n    global_mean = total_sum / max(total_count, 1)\n    working = train.loc[:, list(keys)].copy()\n    working["_delta"] = delta\n    stats = working.groupby(list(keys), dropna=False)["_delta"].agg(["sum", "count"])\n\n    train_index = pd.MultiIndex.from_frame(train.loc[:, list(keys)])\n    group_sum = stats["sum"].reindex(train_index).to_numpy(dtype=float)\n    group_count = stats["count"].reindex(train_index).to_numpy(dtype=float)\n    global_loo = np.where(\n        total_count > 1,\n        (total_sum - delta) / (total_count - 1),\n        global_mean,\n    )\n    train_value = (\n        group_sum - delta + alpha * global_loo\n    ) / np.maximum(group_count - 1 + alpha, 1)\n    train_count = np.maximum(group_count - 1, 0)\n\n    target_index = pd.MultiIndex.from_frame(target.loc[:, list(keys)])\n    target_sum = stats["sum"].reindex(target_index).to_numpy(dtype=float)\n    target_count = stats["count"].reindex(target_index).to_numpy(dtype=float)\n    seen = np.isfinite(target_sum) & np.isfinite(target_count)\n    target_value = np.full(len(target), global_mean, dtype=float)\n    target_value[seen] = (\n        target_sum[seen] + alpha * global_mean\n    ) / (target_count[seen] + alpha)\n    target_count = np.where(seen, target_count, 0.0)\n    return train_value, np.log1p(train_count), target_value, np.log1p(target_count)\n\n\ndef add_historical_profiles(\n    train: pd.DataFrame,\n    target: pd.DataFrame,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """학습 행은 leave-one-row-out, 미래 행은 train-only 통계로 만든다."""\n    train_output = add_source_target_pair(train)\n    target_output = add_source_target_pair(target)\n    specs = [\n        (\n            "historical_pair",\n            ("station_seq_cat", "snapshot_station_seq_cat", "direction"),\n            20.0,\n        ),\n        (\n            "historical_pair_time",\n            (\n                "station_seq_cat",\n                "snapshot_station_seq_cat",\n                "direction",\n                "snapshot_time_bin_30",\n            ),\n            30.0,\n        ),\n    ]\n    for prefix, keys, alpha in specs:\n        train_value, train_count, target_value, target_count = (\n            smoothed_history_feature(train, target, keys, alpha=alpha)\n        )\n        train_output[f"{prefix}_delta"] = train_value\n        train_output[f"{prefix}_log_count"] = train_count\n        target_output[f"{prefix}_delta"] = target_value\n        target_output[f"{prefix}_log_count"] = target_count\n    return train_output, target_output\n\n\ndef add_source_target_pair(data: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    output["snapshot_target_pair_cat"] = (\n        output["snapshot_station_seq_cat"].astype(str)\n        + "->"\n        + output["station_seq_cat"].astype(str)\n        + "|"\n        + output["direction"].astype(str)\n    )\n    return output\n\n\ndef add_observed_capacity_features(data: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    capacity = np.where(\n        output["snapshot_low_plate_cat"].astype(str).eq("2"), 70.0, 44.0\n    )\n    output["observed_ceiling_capacity"] = capacity\n    output["observed_ceiling_load_ratio"] = (\n        1\n        - output["snapshot_remaining_seats"].to_numpy(dtype=float) / capacity\n    ).clip(0, 1)\n    output["observed_ceiling_load_gap"] = (\n        output["observed_ceiling_load_ratio"]\n        * output["target_stop_gap"].to_numpy(dtype=float)\n    )\n    return output\n\n\ndef observed_seat_ceiling(data: pd.DataFrame) -> np.ndarray:\n    """Return Route 1000\'s repeatedly observed seat-count support.\n\n    Route 1000\'s ordinary vehicles expose at most 44 passenger seats even though\n    the legacy nominal ``capacity`` column is 45.  Category 2 vehicles expose 70.\n    This is a route-scoped output constraint, not a GBIS-wide physical invariant:\n    other locally cached routes contain ordinary vehicles with 45+ seats.  Fail\n    closed if Route 1000 ever exposes a new vehicle category instead of silently\n    applying the current fleet mapping to it.\n    """\n    if "snapshot_low_plate_cat" not in data.columns:\n        raise ValueError(\n            "observed seat-ceiling clipping requires snapshot_low_plate_cat"\n        )\n    category = data["snapshot_low_plate_cat"].astype("string")\n    unsupported = sorted(category.dropna().loc[~category.isin(["0", "2"])].unique())\n    if category.isna().any() or unsupported:\n        raise ValueError(\n            "Route 1000 observed seat-ceiling clipping supports only vehicle "\n            f"categories 0 and 2; unsupported={unsupported}, "\n            f"missing={int(category.isna().sum())}"\n        )\n    return np.where(category.eq("2"), 70.0, 44.0)\n\n\ndef postprocess_ensemble_prediction(\n    spec: EnsembleSpec,\n    prediction: np.ndarray,\n    data: pd.DataFrame,\n) -> np.ndarray:\n    """Apply the exact deployment-time support constraints for an ensemble."""\n    output = clip_seats(np.asarray(prediction, dtype=float), data["capacity"])\n    if float(spec.params.get(OBSERVED_SEAT_CEILING_PARAM, 0.0)) != 0.0:\n        if "snapshot_remaining_seats" not in data.columns:\n            raise ValueError(\n                "observed seat-ceiling clipping requires snapshot_remaining_seats"\n            )\n        empirical_ceiling = observed_seat_ceiling(data)\n        current = data["snapshot_remaining_seats"].to_numpy(dtype=float)\n        nominal = data["capacity"].to_numpy(dtype=float)\n        # A newly observed counterexample means the current-fleet assumption has\n        # changed. Disable the projection for that row rather than forcing a\n        # stale 44/70 ceiling. Later observations remain monitor-only and never\n        # retroactively change the pre-cutoff H8 candidate definition.\n        effective_ceiling = np.where(\n            current > empirical_ceiling, nominal, empirical_ceiling\n        )\n        output = np.minimum(output, effective_ceiling)\n    return output\n\n\ndef route_profile_values(\n    flows: pd.DataFrame,\n    target: pd.DataFrame,\n    *,\n    alpha: float = 10.0,\n) -> pd.DataFrame:\n    """과거 직접 도착→출발 순좌석변화를 목표 직전까지 누적한다."""\n    output = pd.DataFrame(index=target.index)\n    for column in ROUTE_PROFILE_FEATURES:\n        output[column] = 0.0\n    if target.empty or flows.empty:\n        output["route_profile_seats"] = target[\n            "snapshot_remaining_seats"\n        ].to_numpy(dtype=float)\n        output["route_profile_fallback_share"] = 1.0\n        return output\n\n    weekday_flows = flows.loc[\n        pd.to_datetime(flows["date"]).dt.dayofweek.lt(5)\n    ].copy()\n    if weekday_flows.empty:\n        output["route_profile_seats"] = target[\n            "snapshot_remaining_seats"\n        ].to_numpy(dtype=float)\n        output["route_profile_fallback_share"] = 1.0\n        return output\n\n    global_mean = float(weekday_flows["stop_net"].mean())\n    global_variance = float(weekday_flows["stop_net"].var(ddof=0))\n    station_stats = weekday_flows.groupby(\n        ["station_seq", "direction"], sort=False\n    )["stop_net"].agg(["mean", "var", "count"])\n    cell_stats = weekday_flows.groupby(\n        ["station_seq", "direction", "time_bin_2h"], sort=False\n    )["stop_net"].agg(["mean", "var", "count"])\n\n    station_mean = station_stats["mean"].to_dict()\n    station_variance = station_stats["var"].fillna(global_variance).to_dict()\n    cell_mean: dict[tuple[int, str, int], float] = {}\n    cell_variance: dict[tuple[int, str, int], float] = {}\n    pass_node_sequences = {\n        int(value) for value in flows.attrs.get("pass_node_sequences", [])\n    }\n    for key, row in cell_stats.iterrows():\n        station_key = (int(key[0]), str(key[1]))\n        prior_mean = float(station_mean.get(station_key, global_mean))\n        prior_variance = float(\n            station_variance.get(station_key, global_variance)\n        )\n        count = float(row["count"])\n        cell_mean[(int(key[0]), str(key[1]), int(key[2]))] = float(\n            (count * float(row["mean"]) + alpha * prior_mean)\n            / (count + alpha)\n        )\n        raw_variance = (\n            float(row["var"]) if pd.notna(row["var"]) else prior_variance\n        )\n        cell_variance[(int(key[0]), str(key[1]), int(key[2]))] = float(\n            (count * raw_variance + alpha * prior_variance)\n            / (count + alpha)\n        )\n\n    deltas = np.zeros(len(target), dtype=float)\n    uncertainties = np.zeros(len(target), dtype=float)\n    fallback_shares = np.zeros(len(target), dtype=float)\n    for position, row in enumerate(target.itertuples(index=False)):\n        day_of_week = int(row.snapshot_day_of_week)\n        if day_of_week >= 5:\n            fallback_shares[position] = 1.0\n            continue\n        start = int(row.snapshot_station_seq)\n        if str(row.target_state_cat) != "1":\n            start += 1\n        stop = int(row.station_seq_cat)\n        if start >= stop:\n            continue\n        time_bin = int(row.snapshot_time.hour // 2 * 2)\n        means: list[float] = []\n        variances: list[float] = []\n        fallback_count = 0\n        for station_seq in range(start, stop):\n            if station_seq in pass_node_sequences:\n                means.append(0.0)\n                variances.append(0.0)\n                continue\n            cell_key = (station_seq, str(row.direction), time_bin)\n            station_key = (station_seq, str(row.direction))\n            if cell_key in cell_mean:\n                means.append(cell_mean[cell_key])\n                variances.append(cell_variance[cell_key])\n            elif station_key in station_mean:\n                means.append(float(station_mean[station_key]))\n                variances.append(\n                    float(station_variance.get(station_key, global_variance))\n                )\n                fallback_count += 1\n            else:\n                means.append(global_mean)\n                variances.append(global_variance)\n                fallback_count += 1\n        deltas[position] = float(np.sum(means))\n        uncertainties[position] = float(np.sqrt(np.maximum(np.sum(variances), 0)))\n        fallback_shares[position] = fallback_count / max(len(means), 1)\n\n    current = target["snapshot_remaining_seats"].to_numpy(dtype=float)\n    capacity = target["capacity"].to_numpy(dtype=float)\n    output["route_profile_delta"] = deltas\n    output["route_profile_seats"] = np.minimum(\n        np.maximum(current + deltas, 0), capacity\n    )\n    output["route_profile_uncertainty"] = uncertainties\n    output["route_profile_fallback_share"] = fallback_shares\n    return output\n\n\ndef add_route_profile_frames(\n    train: pd.DataFrame,\n    target: pd.DataFrame,\n    flows: pd.DataFrame,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    train_output = train.copy()\n    for column in ROUTE_PROFILE_FEATURES:\n        train_output[column] = 0.0\n    for date, indices in train.groupby("date", sort=False).groups.items():\n        # Service-time simulation: a training row may only use flows observed on\n        # strictly earlier dates.  Leave-one-date-out would let older rows see\n        # later training dates and creates a train/serve distribution mismatch.\n        crossfit_flows = flows.loc[flows["date"].lt(str(date))]\n        values = route_profile_values(crossfit_flows, train.loc[indices])\n        train_output.loc[indices, list(ROUTE_PROFILE_FEATURES)] = values[\n            list(ROUTE_PROFILE_FEATURES)\n        ].to_numpy()\n    target_output = target.copy()\n    for date, indices in target.groupby("date", sort=False).groups.items():\n        historical_flows = flows.loc[flows["date"].lt(str(date))]\n        values = route_profile_values(historical_flows, target.loc[indices])\n        target_output.loc[indices, list(ROUTE_PROFILE_FEATURES)] = values[\n            list(ROUTE_PROFILE_FEATURES)\n        ].to_numpy()\n    return train_output, target_output\n\n\ndef feature_frames(\n    train: pd.DataFrame,\n    target: pd.DataFrame,\n    variant: str,\n    flows: pd.DataFrame | None = None,\n) -> tuple[pd.DataFrame, pd.DataFrame, FeatureSet]:\n    if variant == "dynamic":\n        return train, target, DYNAMIC_ALL_PREARRIVAL\n    if variant == "source_target_pair":\n        return (\n            add_source_target_pair(train),\n            add_source_target_pair(target),\n            PAIR_ALL_PREARRIVAL,\n        )\n    if variant == "capacity_44_70":\n        return (\n            add_observed_capacity_features(train),\n            add_observed_capacity_features(target),\n            CAPACITY_44_ALL_PREARRIVAL,\n        )\n    if variant == "previous_bus_seat":\n        return train, target, PREVIOUS_BUS_SEAT_ALL_PREARRIVAL\n    if variant == "previous_bus_seat_capacity_44_70":\n        return (\n            add_observed_capacity_features(train),\n            add_observed_capacity_features(target),\n            PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL,\n        )\n    if variant == "previous_bus_normalized_capacity_44_70":\n        return (\n            add_observed_capacity_features(train),\n            add_observed_capacity_features(target),\n            NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,\n        )\n    if variant == "previous_bus_capacity_44_70":\n        return (\n            add_observed_capacity_features(train),\n            add_observed_capacity_features(target),\n            PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,\n        )\n    if variant == "previous_bus":\n        return train, target, ENGINEERED_ALL_PREARRIVAL\n    if variant == "historical":\n        historical_train, historical_target = add_historical_profiles(train, target)\n        return historical_train, historical_target, HISTORICAL_ALL_PREARRIVAL\n    if variant == "route_profile":\n        if flows is None:\n            raise ValueError("route_profile feature에는 과거 stop flow가 필요합니다.")\n        profile_train, profile_target = add_route_profile_frames(\n            train, target, flows\n        )\n        return profile_train, profile_target, ROUTE_PROFILE_ALL_PREARRIVAL\n    raise ValueError(f"알 수 없는 feature variant: {variant}")\n\n\ndef make_regressor(candidate: Candidate, seed: int) -> Pipeline:\n    _, _, feature_set = feature_frames(\n        pd.DataFrame(columns=DYNAMIC_ALL_PREARRIVAL.columns),\n        pd.DataFrame(columns=DYNAMIC_ALL_PREARRIVAL.columns),\n        "dynamic",\n    )\n    if candidate.feature_variant == "previous_bus":\n        feature_set = ENGINEERED_ALL_PREARRIVAL\n    elif candidate.feature_variant == "previous_bus_seat":\n        feature_set = PREVIOUS_BUS_SEAT_ALL_PREARRIVAL\n    elif candidate.feature_variant == "previous_bus_seat_capacity_44_70":\n        feature_set = PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL\n    elif candidate.feature_variant == "previous_bus_normalized_capacity_44_70":\n        feature_set = NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL\n    elif candidate.feature_variant == "previous_bus_capacity_44_70":\n        feature_set = PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL\n    elif candidate.feature_variant == "source_target_pair":\n        feature_set = PAIR_ALL_PREARRIVAL\n    elif candidate.feature_variant == "capacity_44_70":\n        feature_set = CAPACITY_44_ALL_PREARRIVAL\n    elif candidate.feature_variant == "historical":\n        feature_set = HISTORICAL_ALL_PREARRIVAL\n    elif candidate.feature_variant == "route_profile":\n        feature_set = ROUTE_PROFILE_ALL_PREARRIVAL\n    preprocessor = clone(make_preprocessor(feature_set))\n    if candidate.model_kind == "hgb":\n        model = HistGradientBoostingRegressor(\n            random_state=seed,\n            **candidate.params,\n        )\n    elif candidate.model_kind == "extra_trees":\n        model = ExtraTreesRegressor(\n            n_jobs=-1,\n            random_state=seed,\n            **candidate.params,\n        )\n    elif candidate.model_kind == "random_forest":\n        model = RandomForestRegressor(\n            n_jobs=-1,\n            random_state=seed,\n            **candidate.params,\n        )\n    elif candidate.model_kind == "lightgbm":\n        model = lgb.LGBMRegressor(\n            n_jobs=-1,\n            random_state=seed,\n            verbosity=-1,\n            **candidate.params,\n        )\n    else:\n        raise ValueError(f"학습 모델이 아닌 candidate입니다: {candidate.model_kind}")\n    return Pipeline([("features", preprocessor), ("regressor", model)])\n\n\ndef tree_node_count(model: Any | None) -> int:\n    """Count nodes in a fitted supported tree model or pipeline.\n\n    The deployment bundle is fitted on more rows than any rolling-origin fold,\n    so its node count must be measured from that final fitted object.  This\n    helper also handles HistGradientBoosting\'s private predictor layout, which\n    does not expose ``estimators_``.\n    """\n    if model is None:\n        return 0\n    regressor = (\n        model.named_steps.get("regressor", model)\n        if hasattr(model, "named_steps")\n        else model\n    )\n    if hasattr(regressor, "estimators_"):\n        estimators = np.asarray(regressor.estimators_, dtype=object).ravel()\n        return int(\n            sum(\n                estimator.tree_.node_count\n                for estimator in estimators\n                if hasattr(estimator, "tree_")\n            )\n        )\n    if hasattr(regressor, "booster_"):\n        return int(\n            sum(\n                2 * int(tree["num_leaves"]) - 1\n                for tree in regressor.booster_.dump_model()["tree_info"]\n            )\n        )\n    if hasattr(regressor, "_predictors"):\n        return int(\n            sum(\n                len(predictor.nodes)\n                for stage in regressor._predictors\n                for predictor in stage\n            )\n        )\n    return 0\n\n\ndef encode_target(data: pd.DataFrame, kind: str) -> np.ndarray:\n    label = data["label_seats"].to_numpy(dtype=float)\n    current = data["snapshot_remaining_seats"].to_numpy(dtype=float)\n    if kind == "direct":\n        return label\n    if kind == "delta":\n        return label - current\n    if kind == "delta_per_stop":\n        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)\n        return (label - current) / gap\n    if kind == "delta_per_sqrt_stop":\n        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)\n        return (label - current) / np.sqrt(gap)\n    if kind == "route_residual":\n        return label - data["route_profile_seats"].to_numpy(dtype=float)\n    if kind == "route_residual_per_stop":\n        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)\n        baseline = data["route_profile_seats"].to_numpy(dtype=float)\n        return (label - baseline) / gap\n    raise ValueError(f"알 수 없는 target kind: {kind}")\n\n\ndef decode_target(raw: np.ndarray, data: pd.DataFrame, kind: str) -> np.ndarray:\n    current = data["snapshot_remaining_seats"].to_numpy(dtype=float)\n    if kind == "direct":\n        prediction = raw\n    elif kind == "delta":\n        prediction = current + raw\n    elif kind == "delta_per_stop":\n        prediction = current + raw * data["target_stop_gap"].to_numpy(dtype=float)\n    elif kind == "delta_per_sqrt_stop":\n        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)\n        prediction = current + raw * np.sqrt(gap)\n    elif kind == "route_residual":\n        prediction = data["route_profile_seats"].to_numpy(dtype=float) + raw\n    elif kind == "route_residual_per_stop":\n        prediction = data["route_profile_seats"].to_numpy(dtype=float) + (\n            raw * data["target_stop_gap"].to_numpy(dtype=float)\n        )\n    else:\n        raise ValueError(f"알 수 없는 target kind: {kind}")\n    return clip_seats(np.asarray(prediction, dtype=float), data["capacity"])\n\n\ndef formula_prediction(candidate: Candidate, data: pd.DataFrame) -> np.ndarray:\n    current = data["snapshot_remaining_seats"].to_numpy(dtype=float)\n    if candidate.name == "persistence":\n        return current\n    shrinkage = float(candidate.params["shrinkage"])\n    projected = data["projected_arrival_seats"].fillna(\n        data["snapshot_remaining_seats"]\n    ).to_numpy(dtype=float)\n    return clip_seats(\n        current + shrinkage * (projected - current), data["capacity"]\n    )\n\n\ndef event_summary(data: pd.DataFrame, predictions: np.ndarray) -> dict[str, Any]:\n    scored = data[\n        [\n            "event_id",\n            "date",\n            "trip_id",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "capacity",\n        ]\n    ].copy()\n    scored["prediction"] = clip_seats(predictions, data["capacity"])\n    scored["absolute_error"] = (\n        scored["label_seats"] - scored["prediction"]\n    ).abs()\n    scored["within_3"] = scored["absolute_error"].le(3)\n    per_event = scored.groupby("event_id", sort=False).agg(\n        trip_id=("trip_id", "first"),\n        label_seats=("label_seats", "first"),\n        mae=("absolute_error", "mean"),\n        within_3=("within_3", "mean"),\n        bias=("prediction", "mean"),\n    )\n    per_event["bias"] -= per_event["label_seats"]\n    per_trip = per_event.groupby("trip_id", sort=False)["mae"].mean()\n    low = per_event["label_seats"].le(5)\n    full = per_event["label_seats"].eq(0)\n\n    emerging_rows = scored.loc[\n        scored["label_seats"].le(5)\n        & scored["snapshot_remaining_seats"].gt(5)\n    ]\n    emerging = emerging_rows.groupby("event_id", sort=False)["absolute_error"].mean()\n    far_rows = scored.loc[scored["target_stop_gap"].ge(6)]\n    far = far_rows.groupby("event_id", sort=False)["absolute_error"].mean()\n    near_rows = scored.loc[scored["target_stop_gap"].le(2)]\n    near = near_rows.groupby("event_id", sort=False)["absolute_error"].mean()\n    return {\n        "rows": int(len(scored)),\n        "events": int(len(per_event)),\n        "low_0_5_events": int(low.sum()),\n        "full_events": int(full.sum()),\n        "event_balanced_mae": float(per_event["mae"].mean()),\n        "trip_balanced_mae": float(per_trip.mean()),\n        "event_balanced_mae_std": float(per_event["mae"].std(ddof=0)),\n        "event_balanced_mae_p90": float(per_event["mae"].quantile(0.9)),\n        "event_balanced_within_3": float(per_event["within_3"].mean()),\n        "event_balanced_bias": float(per_event["bias"].mean()),\n        "low_0_5_mae": float(per_event.loc[low, "mae"].mean()) if low.any() else np.nan,\n        "full_mae": float(per_event.loc[full, "mae"].mean()) if full.any() else np.nan,\n        "emerging_low_events": int(emerging.index.nunique()),\n        "emerging_low_mae": float(emerging.mean()) if len(emerging) else np.nan,\n        "near_1_2_stop_mae": float(near.mean()) if len(near) else np.nan,\n        "far_6_plus_stop_mae": float(far.mean()) if len(far) else np.nan,\n    }\n\n\ndef scored_frame(\n    data: pd.DataFrame,\n    predictions: np.ndarray,\n    *,\n    candidate: str,\n) -> pd.DataFrame:\n    columns = [\n        "event_id",\n        "date",\n        "trip_id",\n        "snapshot_time",\n        "label_seats",\n        "snapshot_remaining_seats",\n        "target_stop_gap",\n        "minutes_to_arrival",\n        "capacity",\n    ]\n    if "snapshot_low_plate_cat" in data.columns:\n        columns.append("snapshot_low_plate_cat")\n    output = data[columns].copy()\n    output["candidate"] = candidate\n    output["prediction"] = clip_seats(predictions, data["capacity"])\n    return output\n\n\ndef candidate_weights(\n    data: pd.DataFrame,\n    low_weight: float,\n    *,\n    weighting_kind: str = "event",\n    far_weight: float = 1.0,\n    far_threshold: int = 6,\n    gap_weight_power: float = 0.0,\n) -> np.ndarray:\n    if not np.isfinite(gap_weight_power) or gap_weight_power < 0:\n        raise ValueError("gap_weight_power는 유한한 0 이상이어야 합니다.")\n    event_balanced = event_weights(data)\n    rows_per_event = data.groupby("event_id")["event_id"].transform(\n        "size"\n    ).to_numpy(dtype=float)\n    events_per_trip = data.groupby("trip_id")["event_id"].transform(\n        "nunique"\n    ).to_numpy(dtype=float)\n    trip_balanced = 1.0 / (rows_per_event * events_per_trip)\n    if weighting_kind == "event":\n        weights = event_balanced\n    elif weighting_kind == "trip":\n        weights = trip_balanced\n    elif weighting_kind == "event_trip_hybrid":\n        weights = np.sqrt(event_balanced * trip_balanced)\n    else:\n        raise ValueError(f"알 수 없는 weighting kind: {weighting_kind}")\n    if low_weight != 1:\n        weights = weights * np.where(data["label_seats"].le(5), low_weight, 1.0)\n    if far_weight != 1:\n        weights = weights * np.where(\n            data["target_stop_gap"].ge(far_threshold), far_weight, 1.0\n        )\n    if gap_weight_power != 0:\n        gap = np.maximum(\n            data["target_stop_gap"].to_numpy(dtype=float), 1.0\n        )\n        weights = weights * np.power(gap, gap_weight_power)\n    if weighting_kind == "trip":\n        trip_totals = pd.Series(weights, index=data.index).groupby(\n            data["trip_id"]\n        ).transform("sum").to_numpy(dtype=float)\n        weights = weights / trip_totals\n    return weights / weights.mean()\n\n\ndef development_folds(\n    data: pd.DataFrame,\n    dates: tuple[str, ...],\n) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []\n    for position, validation_date in enumerate(dates[1:], start=1):\n        train_dates = dates[:position]\n        train = data.loc[data["date"].isin(train_dates)].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        if train.empty or validation.empty:\n            raise ValueError(f"개발 fold {validation_date}가 비어 있습니다.")\n        folds.append((validation_date, train, validation))\n    return folds\n\n\nSTRICT_FORWARD_BIAS_RULE = (\n    "strict_forward_prior_oof_folds_first_fold_zero"\n)\nDEPLOYMENT_BIAS_SOURCE = (\n    "all_development_oof_residuals_for_future_predictions_only"\n)\n\n\ndef validate_test_date(\n    test_date: str,\n    *,\n    development_dates: tuple[str, ...] = DEFAULT_DEVELOPMENT_DATES,\n    stress_dates: tuple[str, ...] = DEFAULT_STRESS_DATES,\n) -> None:\n    """잠금 테스트가 개발 또는 스트레스 관측 기간을 침범하지 않게 한다."""\n    try:\n        parsed_test = pd.Timestamp(test_date).date()\n        parsed_development = tuple(\n            pd.Timestamp(value).date() for value in development_dates\n        )\n        parsed_stress = tuple(pd.Timestamp(value).date() for value in stress_dates)\n    except (TypeError, ValueError) as error:\n        raise ValueError(f"유효하지 않은 최종 테스트 날짜입니다: {test_date}") from error\n\n    if parsed_test in parsed_development:\n        raise ValueError(\n            f"최종 테스트 날짜 {test_date}가 개발 날짜와 겹칩니다."\n        )\n    if parsed_test in parsed_stress:\n        raise ValueError(\n            f"최종 테스트 날짜 {test_date}가 스트레스 날짜와 겹칩니다."\n        )\n    if parsed_test <= max(parsed_development):\n        raise ValueError(\n            "최종 테스트 날짜는 마지막 개발 날짜 "\n            f"{max(parsed_development).isoformat()}보다 뒤여야 합니다: {test_date}"\n        )\n\n\ndef strict_forward_bias_predictions(\n    data: pd.DataFrame,\n    predictions: np.ndarray,\n    *,\n    validation_dates: Iterable[str] | None = None,\n) -> tuple[np.ndarray, dict[str, float]]:\n    """각 OOF 날짜를 오직 이전 OOF residual로 bias 보정한다.\n\n    첫 fold는 이전 residual이 없으므로 0을 사용한다. 반환 prediction은 원래\n    행 순서를 유지하며 좌석 범위로 clip된다. 현재 또는 미래 fold의 label은\n    해당 fold의 보정값 계산에 절대 사용되지 않는다.\n    """\n    raw = np.asarray(predictions, dtype=float)\n    if len(raw) != len(data):\n        raise ValueError("prediction과 OOF data의 길이가 다릅니다.")\n    if "date" not in data or data["date"].isna().any():\n        raise ValueError("strict-forward bias에는 결측 없는 date가 필요합니다.")\n\n    row_dates = data["date"].astype(str).to_numpy()\n    ordered_dates = (\n        tuple(str(value) for value in validation_dates)\n        if validation_dates is not None\n        else tuple(sorted(set(row_dates)))\n    )\n    if len(set(ordered_dates)) != len(ordered_dates):\n        raise ValueError("validation_dates에 중복 날짜가 있습니다.")\n\n    adjusted = raw.copy()\n    covered = np.zeros(len(data), dtype=bool)\n    history_positions: list[int] = []\n    biases: dict[str, float] = {}\n    for validation_date in ordered_dates:\n        current = row_dates == validation_date\n        if not current.any():\n            raise ValueError(f"OOF validation 날짜 {validation_date}가 비어 있습니다.")\n        if history_positions:\n            history = np.asarray(history_positions, dtype=int)\n            bias = event_bias(data.iloc[history], raw[history])\n        else:\n            bias = 0.0\n        biases[validation_date] = float(bias)\n        adjusted[current] = clip_seats(\n            raw[current] + bias,\n            data.loc[current, "capacity"],\n        )\n        positions = np.flatnonzero(current)\n        history_positions.extend(positions.tolist())\n        covered[current] = True\n\n    if not covered.all():\n        missing = sorted(set(row_dates[~covered]))\n        raise ValueError(f"validation_dates에 포함되지 않은 OOF 날짜가 있습니다: {missing}")\n    return adjusted, biases\n\n\ndef run_candidate(\n    candidate: Candidate,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    flows: pd.DataFrame,\n    *,\n    seed: int,\n) -> tuple[dict[str, Any], pd.DataFrame, list[dict[str, Any]]]:\n    raw_frames: list[pd.DataFrame] = []\n    train_maes: list[float] = []\n    fit_seconds: list[float] = []\n    prediction_ms_per_1000: list[float] = []\n    fold_tree_nodes: list[int] = []\n    for fold_number, (validation_date, train, validation) in enumerate(folds):\n        fold_flows = flows.loc[flows["date"].lt(validation_date)]\n        if candidate.model_kind == "formula":\n            raw_validation = formula_prediction(candidate, validation)\n        elif candidate.model_kind == "route_formula":\n            raw_validation = route_profile_values(\n                fold_flows, validation\n            )["route_profile_seats"].to_numpy(dtype=float)\n        else:\n            prepared_train, prepared_validation, feature_set = feature_frames(\n                train,\n                validation,\n                candidate.feature_variant,\n                flows=fold_flows,\n            )\n            model = make_regressor(candidate, seed + fold_number)\n            target = encode_target(prepared_train, candidate.target_kind)\n            fit_started = time.perf_counter()\n            model.fit(\n                prepared_train[feature_set.columns],\n                target,\n                regressor__sample_weight=candidate_weights(\n                    prepared_train,\n                    candidate.low_weight,\n                    weighting_kind=candidate.weighting_kind,\n                    far_weight=candidate.far_weight,\n                    far_threshold=candidate.far_threshold,\n                    gap_weight_power=candidate.gap_weight_power,\n                ),\n            )\n            fit_seconds.append(time.perf_counter() - fit_started)\n            predict_started = time.perf_counter()\n            validation_raw_model = model.predict(\n                prepared_validation[feature_set.columns]\n            )\n            prediction_seconds = time.perf_counter() - predict_started\n            prediction_ms_per_1000.append(\n                prediction_seconds * 1_000_000 / max(len(prepared_validation), 1)\n            )\n            raw_validation = decode_target(\n                validation_raw_model,\n                prepared_validation,\n                candidate.target_kind,\n            )\n            train_prediction = decode_target(\n                model.predict(prepared_train[feature_set.columns]),\n                prepared_train,\n                candidate.target_kind,\n            )\n            train_maes.append(\n                event_summary(prepared_train, train_prediction)["event_balanced_mae"]\n            )\n            fold_tree_nodes.append(tree_node_count(model))\n        raw_frames.append(\n            scored_frame(\n                validation,\n                raw_validation,\n                candidate=candidate.name,\n            )\n        )\n\n    oof = pd.concat(raw_frames, ignore_index=True)\n    raw_oof_prediction = oof["prediction"].to_numpy(dtype=float)\n    deployment_bias = event_bias(oof, raw_oof_prediction)\n    strict_prediction, oof_biases = strict_forward_bias_predictions(\n        oof,\n        raw_oof_prediction,\n        validation_dates=[fold[0] for fold in folds],\n    )\n    oof["raw_prediction"] = raw_oof_prediction\n    oof["prediction"] = strict_prediction\n    oof["deployment_prediction"] = clip_seats(\n        raw_oof_prediction + deployment_bias, oof["capacity"]\n    )\n    fold_rows: list[dict[str, Any]] = []\n    for validation_date in [fold[0] for fold in folds]:\n        fold_data = oof.loc[oof["date"].eq(validation_date)]\n        metrics = event_summary(\n            fold_data, fold_data["prediction"].to_numpy(dtype=float)\n        )\n        metrics.update(\n            {\n                "candidate": candidate.name,\n                "stage": candidate.stage,\n                "validation_date": validation_date,\n                "oof_bias_correction": oof_biases[validation_date],\n                "oof_bias_correction_rule": STRICT_FORWARD_BIAS_RULE,\n            }\n        )\n        fold_rows.append(metrics)\n\n    aggregate = event_summary(oof, oof["prediction"].to_numpy(dtype=float))\n    daily_mae = np.asarray(\n        [row["event_balanced_mae"] for row in fold_rows], dtype=float\n    )\n    aggregate.update(\n        {\n            "candidate": candidate.name,\n            "stage": candidate.stage,\n            "model_kind": candidate.model_kind,\n            "target_kind": candidate.target_kind,\n            "feature_variant": candidate.feature_variant,\n            "low_weight": candidate.low_weight,\n            "weighting_kind": candidate.weighting_kind,\n            "far_weight": candidate.far_weight,\n            "far_threshold": candidate.far_threshold,\n            "gap_weight_power": candidate.gap_weight_power,\n            # 기존 필드명은 저장 artifact/SeatServiceModel 계약이다. 이 값은\n            # OOF 선택 점수가 아니라 미래 prediction에만 적용한다.\n            "bias_correction": deployment_bias,\n            "bias_correction_source": DEPLOYMENT_BIAS_SOURCE,\n            "oof_bias_correction_rule": STRICT_FORWARD_BIAS_RULE,\n            "oof_bias_corrections": json.dumps(\n                oof_biases, ensure_ascii=False, sort_keys=True\n            ),\n            "mean_daily_mae": float(daily_mae.mean()),\n            "std_daily_mae": float(daily_mae.std(ddof=0)),\n            "robust_score": float(daily_mae.mean() + 0.5 * daily_mae.std(ddof=0)),\n            "service_score": float(\n                daily_mae.mean()\n                + 0.5 * daily_mae.std(ddof=0)\n                + 0.05 * aggregate["low_0_5_mae"]\n            ),\n            "mean_train_mae": (\n                float(np.mean(train_maes)) if train_maes else np.nan\n            ),\n            "overfit_gap": (\n                float(aggregate["event_balanced_mae"] - np.mean(train_maes))\n                if train_maes\n                else np.nan\n            ),\n            "mean_fit_seconds": (\n                float(np.mean(fit_seconds)) if fit_seconds else np.nan\n            ),\n            "prediction_ms_per_1000": (\n                float(np.mean(prediction_ms_per_1000))\n                if prediction_ms_per_1000\n                else np.nan\n            ),\n            "mean_fold_tree_nodes": (\n                float(np.mean(fold_tree_nodes)) if fold_tree_nodes else 0.0\n            ),\n            "why": candidate.why,\n            "if_works": candidate.if_works,\n            "if_fails": candidate.if_fails,\n            "params": json.dumps(candidate.params, ensure_ascii=False, sort_keys=True),\n        }\n    )\n    return aggregate, oof, fold_rows\n\n\ndef best_name(results: pd.DataFrame, names: Iterable[str]) -> str:\n    subset = results.loc[results["candidate"].isin(list(names))]\n    if subset.empty:\n        raise ValueError("선택할 candidate 결과가 없습니다.")\n    return str(subset.sort_values(["service_score", "robust_score"]).iloc[0]["candidate"])\n\n\ndef combine_oof(\n    name: str,\n    left: pd.DataFrame,\n    right: pd.DataFrame,\n    *,\n    left_weight: np.ndarray | float,\n) -> pd.DataFrame:\n    keys = ["event_id", "snapshot_time"]\n    output = left.copy()\n    target_index = pd.MultiIndex.from_frame(output[keys])\n    left_prediction = output["prediction"].to_numpy(dtype=float)\n    left_deployment = output.get(\n        "deployment_prediction", output["prediction"]\n    ).to_numpy(dtype=float)\n    aligned_right = (\n        right.set_index(keys)["prediction"]\n        .reindex(target_index)\n        .to_numpy(dtype=float)\n    )\n    right_deployment = right.get(\n        "deployment_prediction", right["prediction"]\n    ).to_numpy(dtype=float)\n    aligned_right_deployment = (\n        right.assign(_deployment_prediction=right_deployment)\n        .set_index(keys)["_deployment_prediction"]\n        .reindex(target_index)\n        .to_numpy(dtype=float)\n    )\n    weights = np.asarray(left_weight, dtype=float)\n    output["prediction"] = (\n        weights * left_prediction + (1 - weights) * aligned_right\n    )\n    output["deployment_prediction"] = (\n        weights * left_deployment + (1 - weights) * aligned_right_deployment\n    )\n    output["candidate"] = name\n    return output\n\n\ndef combine_weighted_oof(\n    name: str,\n    components: dict[str, pd.DataFrame],\n    weights: dict[str, float],\n) -> pd.DataFrame:\n    first_name = next(iter(components))\n    output = components[first_name].copy()\n    keys = ["event_id", "snapshot_time"]\n    target_index = pd.MultiIndex.from_frame(output[keys])\n    prediction = np.zeros(len(output), dtype=float)\n    deployment_prediction = np.zeros(len(output), dtype=float)\n    for component, frame in components.items():\n        aligned = frame.set_index(keys)["prediction"].reindex(target_index)\n        prediction += float(weights[component]) * aligned.to_numpy(dtype=float)\n        deployment_column = frame.get("deployment_prediction", frame["prediction"])\n        aligned_deployment = (\n            frame.assign(\n                _deployment_prediction=deployment_column.to_numpy(dtype=float)\n            )\n            .set_index(keys)["_deployment_prediction"]\n            .reindex(target_index)\n        )\n        deployment_prediction += (\n            float(weights[component])\n            * aligned_deployment.to_numpy(dtype=float)\n        )\n    output["prediction"] = prediction\n    output["deployment_prediction"] = deployment_prediction\n    output["candidate"] = name\n    return output\n\n\ndef score_ensemble(\n    spec: EnsembleSpec,\n    output: pd.DataFrame,\n) -> tuple[dict[str, Any], pd.DataFrame]:\n    output = output.copy()\n    selection_base = output["prediction"].to_numpy(dtype=float)\n    strict_prediction, oof_biases = strict_forward_bias_predictions(\n        output, selection_base\n    )\n    deployment_base = output.get(\n        "deployment_prediction", output["prediction"]\n    ).to_numpy(dtype=float)\n    deployment_bias = event_bias(output, deployment_base)\n    spec.bias = deployment_bias\n    spec.oof_biases = oof_biases\n    output["raw_prediction"] = selection_base\n    output["prediction"] = postprocess_ensemble_prediction(\n        spec, strict_prediction, output\n    )\n    output["deployment_prediction"] = postprocess_ensemble_prediction(\n        spec, deployment_base + deployment_bias, output\n    )\n    aggregate = event_summary(output, output["prediction"].to_numpy(dtype=float))\n    daily = []\n    for _, group in output.groupby("date", sort=True):\n        daily.append(event_summary(group, group["prediction"].to_numpy(dtype=float)))\n    daily_mae = np.asarray([row["event_balanced_mae"] for row in daily])\n    aggregate.update(\n        {\n            "candidate": spec.name,\n            "stage": "H7_ensemble",\n            "model_kind": "ensemble",\n            "target_kind": "blend",\n            "feature_variant": "mixed",\n            "low_weight": np.nan,\n            "weighting_kind": "mixed",\n            "far_weight": np.nan,\n            "far_threshold": np.nan,\n            "bias_correction": deployment_bias,\n            "bias_correction_source": DEPLOYMENT_BIAS_SOURCE,\n            "oof_bias_correction_rule": STRICT_FORWARD_BIAS_RULE,\n            "oof_bias_corrections": json.dumps(\n                oof_biases, ensure_ascii=False, sort_keys=True\n            ),\n            "mean_daily_mae": float(daily_mae.mean()),\n            "std_daily_mae": float(daily_mae.std(ddof=0)),\n            "robust_score": float(daily_mae.mean() + 0.5 * daily_mae.std(ddof=0)),\n            "service_score": float(\n                daily_mae.mean()\n                + 0.5 * daily_mae.std(ddof=0)\n                + 0.05 * aggregate["low_0_5_mae"]\n            ),\n            "mean_train_mae": np.nan,\n            "overfit_gap": np.nan,\n            "mean_fit_seconds": np.nan,\n            "prediction_ms_per_1000": np.nan,\n            "mean_fold_tree_nodes": np.nan,\n            "why": (\n                "서로 다른 귀납 편향 또는 근거리 persistence를 결합하면 "\n                "날짜별 오차 분산을 줄일 수 있다."\n            ),\n            "if_works": "개별 최선보다 robust score와 날짜별 분산이 함께 감소한다.",\n            "if_fails": "기저 모델 오차가 강하게 상관되거나 거리별 게이트가 불필요하다.",\n            "params": json.dumps(spec.params, ensure_ascii=False, sort_keys=True),\n        }\n    )\n    return aggregate, output\n\n\ndef select_with_final_node_budget(\n    all_selection: pd.DataFrame,\n    ensemble_specs: dict[str, EnsembleSpec],\n    *,\n    ensure_final_fit: Callable[[str], None],\n    final_component_tree_nodes: dict[str, int],\n    maximum_final_fit_tree_nodes: int,\n) -> str:\n    """Select by score while enforcing the budget on final fitted models.\n\n    Candidate folds are smaller than the full development fit, so their mean\n    node counts are only diagnostics.  Components are fitted lazily in score\n    order and cached by the caller; selection stops at the first bundle whose\n    measured final node count satisfies the hard deployment constraint.\n    """\n    if maximum_final_fit_tree_nodes < 0:\n        raise ValueError("최종 fit tree node 상한은 0 이상이어야 합니다.")\n    required = {"candidate", "service_score", "robust_score"}\n    missing = sorted(required - set(all_selection.columns))\n    if missing:\n        raise ValueError(f"배포 후보 선택 열이 누락되었습니다: {missing}")\n\n    all_selection["final_fit_tree_nodes"] = np.nan\n    all_selection["deployment_eligible"] = pd.Series(\n        pd.NA, index=all_selection.index, dtype="boolean"\n    )\n    all_selection["deployment_check"] = "not_evaluated_after_selection"\n    for row_index in all_selection.sort_values(\n        ["service_score", "robust_score"]\n    ).index:\n        name = str(all_selection.at[row_index, "candidate"])\n        components = (\n            ensemble_specs[name].components\n            if name in ensemble_specs\n            else (name,)\n        )\n        for component in components:\n            ensure_final_fit(component)\n            if component not in final_component_tree_nodes:\n                raise ValueError(\n                    f"최종 fit node 수가 기록되지 않았습니다: {component}"\n                )\n        bundle_nodes = int(\n            sum(final_component_tree_nodes[component] for component in components)\n        )\n        eligible = bundle_nodes <= maximum_final_fit_tree_nodes\n        all_selection.at[row_index, "final_fit_tree_nodes"] = bundle_nodes\n        all_selection.at[row_index, "deployment_eligible"] = eligible\n        all_selection.at[row_index, "deployment_check"] = (\n            "passed" if eligible else "failed"\n        )\n        if eligible:\n            return name\n    raise ValueError("최종 fit tree node 제약을 만족하는 candidate가 없습니다.")\n\n\ndef make_candidates() -> tuple[list[Candidate], dict[str, dict[str, Any]]]:\n    baseline_hgb = {\n        "loss": "absolute_error",\n        "learning_rate": 0.05,\n        "max_iter": 300,\n        "max_leaf_nodes": 15,\n        "min_samples_leaf": 20,\n        "l2_regularization": 1.0,\n        "early_stopping": False,\n    }\n    formulas = [\n        Candidate(\n            name="persistence",\n            stage="H0_physical",\n            why="정류장 사이 좌석 변화 중앙값이 0이므로 현재 좌석 유지가 강한 기준선이다.",\n            if_works="근거리에서 학습 모델과 비슷하거나 더 정확하다.",\n            if_fails="도착 전 누적 승하차 변화가 무시할 수 없다는 뜻이다.",\n            model_kind="formula",\n        ),\n        Candidate(\n            name="route_stop_profile",\n            stage="H0_physical",\n            why=(\n                "선행 연구의 2단계 구조처럼 정류장별 직접 도착→출발 순좌석변화를 "\n                "shrinkage한 뒤 목표 직전까지 누적한다."\n            ),\n            if_works="특히 6정류장 이상과 신규 저잔여에서 persistence를 안정적으로 이긴다.",\n            if_fails="승차 변화만으로는 하차와 차량별 수요 차이를 설명하지 못한다.",\n            model_kind="route_formula",\n            feature_variant="route_profile",\n        ),\n    ]\n    for shrinkage in (0.25, 0.5, 1.0):\n        formulas.append(\n            Candidate(\n                name=f"trajectory_shrink_{int(shrinkage * 100):03d}",\n                stage="H0_physical",\n                why="최근 좌석 변화 추세에는 신호가 있지만 그대로 외삽하면 잡음이 증폭될 수 있다.",\n                if_works="0과 1 사이 shrinkage가 persistence와 완전 외삽을 모두 이긴다.",\n                if_fails="짧은 관측 궤적의 변화율이 반복 가능한 수요 신호가 아니다.",\n                model_kind="formula",\n                params={"shrinkage": shrinkage},\n            )\n        )\n    transformations = [\n        Candidate(\n            name="hgb_direct_reference",\n            stage="H1_target",\n            why="절대 도착 좌석을 직접 학습하면 정류장·시간대의 전역 수준을 활용할 수 있다.",\n            if_works="현재 좌석 기준 변화량보다 반복 시간대 패턴이 더 강하다.",\n            if_fails="현재 좌석을 기준점으로 두지 않아 차량별 상태 차이가 커진다.",\n            model_kind="hgb",\n            target_kind="direct",\n            params=baseline_hgb,\n        ),\n        Candidate(\n            name="hgb_delta_reference",\n            stage="H1_target",\n            why="실시간 현재 좌석이 강한 기준점이므로 남은 변화량만 학습하면 문제가 단순해진다.",\n            if_works="direct 및 persistence보다 날짜 외 MAE가 낮다.",\n            if_fails="변화량 분산이 절대 좌석보다 크거나 시간대 패턴을 잃는다.",\n            model_kind="hgb",\n            target_kind="delta",\n            params=baseline_hgb,\n        ),\n        Candidate(\n            name="hgb_delta_per_stop",\n            stage="H1_target",\n            why="예측 거리가 제각각이므로 정류장당 변화율을 학습하면 먼 거리 외삽이 쉬워질 수 있다.",\n            if_works="특히 6정류장 이상 MAE가 감소한다.",\n            if_fails="승하차 변화가 정류장 수에 선형 비례하지 않고 특정 정류장에 집중된다.",\n            model_kind="hgb",\n            target_kind="delta_per_stop",\n            params=baseline_hgb,\n        ),\n    ]\n    return formulas + transformations, {"baseline_hgb": baseline_hgb}\n\n\ndef candidate_from(\n    parent: Candidate,\n    *,\n    name: str,\n    stage: str,\n    why: str,\n    if_works: str,\n    if_fails: str,\n    **changes: Any,\n) -> Candidate:\n    return replace(\n        parent,\n        name=name,\n        stage=stage,\n        why=why,\n        if_works=if_works,\n        if_fails=if_fails,\n        **changes,\n    )\n\n\ndef fit_final_candidate(\n    candidate: Candidate,\n    train: pd.DataFrame,\n    targets: dict[str, pd.DataFrame],\n    flows: pd.DataFrame,\n    *,\n    bias: float,\n    seed: int,\n) -> tuple[dict[str, np.ndarray], Pipeline | None, dict[str, Any]]:\n    if candidate.model_kind == "formula":\n        return {\n            key: clip_seats(\n                formula_prediction(candidate, target) + bias, target["capacity"]\n            )\n            for key, target in targets.items()\n        }, None, {"feature_set": "formula"}\n    if candidate.model_kind == "route_formula":\n        return {\n            key: clip_seats(\n                route_profile_values(flows, target)[\n                    "route_profile_seats"\n                ].to_numpy(dtype=float)\n                + bias,\n                target["capacity"],\n            )\n            for key, target in targets.items()\n        }, None, {"feature_set": ROUTE_PROFILE_ALL_PREARRIVAL.name}\n\n    combined_target = pd.concat(\n        [target.assign(_target_split=key) for key, target in targets.items()],\n        ignore_index=True,\n    )\n    prepared_train, prepared_target, feature_set = feature_frames(\n        train,\n        combined_target,\n        candidate.feature_variant,\n        flows=flows,\n    )\n    model = make_regressor(candidate, seed)\n    model.fit(\n        prepared_train[feature_set.columns],\n        encode_target(prepared_train, candidate.target_kind),\n        regressor__sample_weight=candidate_weights(\n            prepared_train,\n            candidate.low_weight,\n            weighting_kind=candidate.weighting_kind,\n            far_weight=candidate.far_weight,\n            far_threshold=candidate.far_threshold,\n            gap_weight_power=candidate.gap_weight_power,\n        ),\n    )\n    predictions: dict[str, np.ndarray] = {}\n    for key in targets:\n        mask = prepared_target["_target_split"].eq(key)\n        split = prepared_target.loc[mask]\n        decoded = decode_target(\n            model.predict(split[feature_set.columns]), split, candidate.target_kind\n        )\n        predictions[key] = clip_seats(decoded + bias, split["capacity"])\n    return predictions, model, {\n        "feature_set": feature_set.name,\n        "feature_columns": feature_set.columns,\n        "target_kind": candidate.target_kind,\n        "feature_variant": candidate.feature_variant,\n    }\n\n\ndef ensemble_prediction(\n    spec: EnsembleSpec,\n    component_predictions: dict[str, np.ndarray],\n    target: pd.DataFrame,\n) -> np.ndarray:\n    if spec.kind == "linear":\n        left, right = spec.components\n        weight = float(spec.params["left_weight"])\n        prediction = (\n            weight * component_predictions[left]\n            + (1 - weight) * component_predictions[right]\n        )\n    elif spec.kind == "distance_gate":\n        left, right = spec.components\n        near = target["target_stop_gap"].le(2).to_numpy()\n        weight = np.where(\n            near,\n            float(spec.params["near_left_weight"]),\n            float(spec.params["far_left_weight"]),\n        )\n        prediction = (\n            weight * component_predictions[left]\n            + (1 - weight) * component_predictions[right]\n        )\n    elif spec.kind == "stop_gap_gate":\n        left, right = spec.components\n        threshold = float(spec.params["threshold"])\n        left_weight = np.where(\n            target["target_stop_gap"].lt(threshold).to_numpy(),\n            float(spec.params["below_left_weight"]),\n            float(spec.params["above_left_weight"]),\n        )\n        prediction = (\n            left_weight * component_predictions[left]\n            + (1 - left_weight) * component_predictions[right]\n        )\n    elif spec.kind == "weighted":\n        prediction = np.zeros(len(target), dtype=float)\n        for component in spec.components:\n            prediction += (\n                float(spec.params[component]) * component_predictions[component]\n            )\n    else:\n        raise ValueError(f"알 수 없는 ensemble: {spec.kind}")\n    return postprocess_ensemble_prediction(spec, prediction + spec.bias, target)\n\n\ndef stop_band_metrics(\n    data: pd.DataFrame,\n    predictions: np.ndarray,\n    model: str,\n) -> pd.DataFrame:\n    bands = pd.cut(\n        data["target_stop_gap"],\n        bins=[0, 2, 5, 10, np.inf],\n        labels=["1-2", "3-5", "6-10", "11+"],\n    )\n    rows: list[dict[str, Any]] = []\n    for band in ["1-2", "3-5", "6-10", "11+"]:\n        mask = bands.eq(band).to_numpy()\n        if not mask.any():\n            continue\n        result = event_summary(data.loc[mask], predictions[mask])\n        result.update({"model": model, "stop_band": band})\n        rows.append(result)\n    return pd.DataFrame(rows)\n\n\ndef cluster_bootstrap_mae_delta(\n    data: pd.DataFrame,\n    selected: np.ndarray,\n    baseline: np.ndarray,\n    *,\n    seed: int,\n    repeats: int = 2_000,\n) -> dict[str, Any]:\n    scored = data[["event_id", "trip_id", "label_seats"]].copy()\n    scored["selected_error"] = np.abs(\n        data["label_seats"].to_numpy(dtype=float) - selected\n    )\n    scored["baseline_error"] = np.abs(\n        data["label_seats"].to_numpy(dtype=float) - baseline\n    )\n    per_event = scored.groupby("event_id", sort=False).agg(\n        trip_id=("trip_id", "first"),\n        selected=("selected_error", "mean"),\n        baseline=("baseline_error", "mean"),\n    )\n    per_event["delta"] = per_event["baseline"] - per_event["selected"]\n    trip_groups = [\n        group["delta"].to_numpy(dtype=float)\n        for _, group in per_event.groupby("trip_id", sort=False)\n    ]\n    if not trip_groups:\n        raise ValueError("bootstrap할 trip이 없습니다.")\n    rng = np.random.default_rng(seed)\n    boot = np.empty(repeats, dtype=float)\n    for index in range(repeats):\n        selected_indices = rng.integers(0, len(trip_groups), size=len(trip_groups))\n        sampled = np.concatenate([trip_groups[position] for position in selected_indices])\n        boot[index] = sampled.mean()\n    lower, median, upper = np.quantile(boot, [0.025, 0.5, 0.975])\n    return {\n        "metric": "baseline MAE minus selected MAE; positive favors selected",\n        "unit": "trip_id",\n        "trips": int(len(trip_groups)),\n        "events": int(len(per_event)),\n        "observed_delta": float(per_event["delta"].mean()),\n        "lower_95": float(lower),\n        "median": float(median),\n        "upper_95": float(upper),\n        "probability_selected_better": float((boot > 0).mean()),\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="가설 기반 도착 잔여좌석 모델·파라미터·앙상블 탐색"\n    )\n    parser.add_argument(\n        "--db",\n        type=Path,\n        default=Path("data/gbis_api_cache.sqlite3"),\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    parser.add_argument("--test-date", default=DEFAULT_TEST_DATE)\n    parser.add_argument("--rebuild-cache", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    validate_test_date(args.test_date)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    experiment_source_sha256 = hashlib.sha256(\n        Path(__file__).read_bytes()\n    ).hexdigest()\n    inference_source_sha256 = {\n        filename: cache_payload_sha256(Path(__file__).resolve().parent / filename)\n        for filename in (\n            "seat_service_model.py",\n            "all_prearrival_seat_regression.py",\n        )\n    }\n\n    snapshots, stop_flows, cache_info = load_or_build_snapshots(\n        args.db, args.snapshot_cache, rebuild=args.rebuild_cache\n    )\n    output_support_audit = audit_route_output_support(args.db, ROUTE_ID)\n    development_dates = tuple(\n        date for date in DEFAULT_DEVELOPMENT_DATES if date in set(snapshots["date"])\n    )\n    if development_dates != DEFAULT_DEVELOPMENT_DATES:\n        raise ValueError(\n            f"개발 날짜가 부족합니다: {development_dates}"\n        )\n    development = snapshots.loc[\n        snapshots["date"].isin(development_dates)\n    ].copy()\n    test = snapshots.loc[snapshots["date"].eq(args.test_date)].copy()\n    stress = snapshots.loc[\n        snapshots["date"].isin(DEFAULT_STRESS_DATES)\n    ].copy()\n    if test.empty:\n        raise ValueError(f"최종 테스트 날짜 {args.test_date}가 비어 있습니다.")\n    folds = development_folds(development, development_dates)\n\n    candidates, settings = make_candidates()\n    registry = {candidate.name: candidate for candidate in candidates}\n    result_rows: list[dict[str, Any]] = []\n    fold_rows: list[dict[str, Any]] = []\n    oof_predictions: dict[str, pd.DataFrame] = {}\n\n    def evaluate(items: Iterable[Candidate]) -> None:\n        for candidate in items:\n            if candidate.name in oof_predictions:\n                continue\n            result, oof, candidate_folds = run_candidate(\n                candidate, folds, stop_flows, seed=args.seed\n            )\n            result_rows.append(result)\n            fold_rows.extend(candidate_folds)\n            oof_predictions[candidate.name] = oof\n            registry[candidate.name] = candidate\n\n    evaluate(candidates)\n    results = pd.DataFrame(result_rows)\n    target_names = [\n        "hgb_direct_reference",\n        "hgb_delta_reference",\n        "hgb_delta_per_stop",\n    ]\n    target_winner_name = best_name(results, target_names)\n    target_winner = registry[target_winner_name]\n\n    capacity_candidates = [\n        candidate_from(\n            target_winner,\n            name="hgb_flexible",\n            stage="H2_capacity",\n            why="기준 HGB의 15개 leaf가 정류장·거리별 비선형성을 과소적합할 수 있다.",\n            if_works="train 오차는 낮아지면서 날짜 외 robust score도 개선된다.",\n            if_fails="현재 데이터량에서 추가 분기는 날짜 고유 패턴에 과적합된다.",\n            params={\n                "loss": "absolute_error",\n                "learning_rate": 0.035,\n                "max_iter": 450,\n                "max_leaf_nodes": 31,\n                "min_samples_leaf": 10,\n                "l2_regularization": 0.5,\n                "early_stopping": False,\n            },\n        ),\n        candidate_from(\n            target_winner,\n            name="hgb_overfit_probe",\n            stage="H2_capacity",\n            why="먼저 충분히 큰 모델로 학습 가능한 신호의 상한과 과적합 간격을 측정한다.",\n            if_works="검증도 개선되어 기존 모델이 명백히 과소적합이었다.",\n            if_fails="train 개선 대비 검증 악화로 데이터 반복성이 부족함을 확인한다.",\n            params={\n                "loss": "absolute_error",\n                "learning_rate": 0.04,\n                "max_iter": 500,\n                "max_leaf_nodes": 63,\n                "min_samples_leaf": 5,\n                "l2_regularization": 0.0,\n                "early_stopping": False,\n            },\n        ),\n        candidate_from(\n            target_winner,\n            name="hgb_regularized",\n            stage="H2_capacity",\n            why="날짜가 네 개뿐이므로 더 큰 leaf와 L2가 일별 변동을 줄일 수 있다.",\n            if_works="평균 MAE가 비슷해도 날짜별 표준편차와 robust score가 감소한다.",\n            if_fails="이미 기준 모델의 규제가 충분해 추가 규제가 수요 신호까지 지운다.",\n            params={\n                "loss": "absolute_error",\n                "learning_rate": 0.04,\n                "max_iter": 350,\n                "max_leaf_nodes": 15,\n                "min_samples_leaf": 50,\n                "l2_regularization": 5.0,\n                "early_stopping": False,\n            },\n        ),\n        candidate_from(\n            target_winner,\n            name="hgb_squared_tail",\n            stage="H2_capacity",\n            why="제곱손실은 큰 오차를 더 벌주므로 저잔여와 먼 거리 꼬리오차를 줄일 수 있다.",\n            if_works="p90·저잔여 MAE가 크게 감소하고 평균 MAE 손실이 작다.",\n            if_fails="일반 사례를 평균 쪽으로 끌어 MAE와 보정이 악화된다.",\n            params={\n                "loss": "squared_error",\n                "learning_rate": 0.035,\n                "max_iter": 450,\n                "max_leaf_nodes": 31,\n                "min_samples_leaf": 15,\n                "l2_regularization": 1.0,\n                "early_stopping": False,\n            },\n        ),\n    ]\n    evaluate(capacity_candidates)\n    results = pd.DataFrame(result_rows)\n    hgb_pool = target_names + [candidate.name for candidate in capacity_candidates]\n    capacity_winner_name = best_name(results, hgb_pool)\n    capacity_winner = registry[capacity_winner_name]\n\n    weighting_candidates = [\n        candidate_from(\n            capacity_winner,\n            name="hgb_low_weight_2",\n            stage="H3_rare_weight",\n            why="0~5석 사건이 약 2%라 전체 MAE 학습에서 기울기가 묻힐 수 있다.",\n            if_works="전체 robust score를 거의 유지하며 저잔여·신규 저잔여 MAE가 감소한다.",\n            if_fails="현재 좌석 자체가 이미 위험을 설명하거나 양성 표본이 너무 적다.",\n            low_weight=2.0,\n        ),\n        candidate_from(\n            capacity_winner,\n            name="hgb_low_weight_4",\n            stage="H3_rare_weight",\n            why="더 강한 비용 민감 학습으로 서비스 핵심 꼬리구간의 상한을 확인한다.",\n            if_works="저잔여 성능 개선이 가중치 2보다 커서 별도 전문가 모델 가치가 있다.",\n            if_fails="전체 오차와 확률 보정만 악화되어 회귀 가중치보다 별도 위험 헤드가 낫다.",\n            low_weight=4.0,\n        ),\n        candidate_from(\n            capacity_winner,\n            name="hgb_low_weight_8",\n            stage="H3_rare_weight",\n            why="가중치 4에서 저잔여 개선이 계속됐으므로 비용 민감도의 포화점을 확인한다.",\n            if_works="저잔여·신규 저잔여 MAE가 더 줄고 전체 robust score 손실이 제한적이다.",\n            if_fails="희귀 사건을 과도하게 반복해 전체 오차와 보정편향이 급격히 커진다.",\n            low_weight=8.0,\n        ),\n    ]\n    evaluate(weighting_candidates)\n    results = pd.DataFrame(result_rows)\n    weighting_pool = [capacity_winner_name] + [\n        candidate.name for candidate in weighting_candidates\n    ]\n    weighting_winner_name = best_name(results, weighting_pool)\n    weighting_winner = registry[weighting_winner_name]\n\n    feature_candidates = [\n        candidate_from(\n            weighting_winner,\n            name="hgb_capacity_44_70",\n            stage="H4_features",\n            why=(\n                "일반차 37대의 관측 빈자리 상한은 모두 44석인데 기존 load ratio는 "\n                "45석을 사용해 1석 편향이 있다."\n            ),\n            if_works="44/70 관측 상한 적재율이 용량 regime과 저잔여 예측을 개선한다.",\n            if_fails="1석 비율 차이는 트리 분할에 미미하거나 45석 명목 정원이 더 적절하다.",\n            feature_variant="capacity_44_70",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_route_profile",\n            stage="H4_features",\n            why=(\n                "정류장×2시간대 직접 도착→출발 순좌석변화 누적값을 HGB가 "\n                "현재 차량 궤적과 함께 보정하면 구조와 residual을 분리할 수 있다."\n            ),\n            if_works="route formula보다 전체·장거리 MAE가 더 낮고 날짜별 방향도 안정적이다.",\n            if_fails="짧은 기간의 flow profile 오차를 모델이 다시 과적합한다.",\n            feature_variant="route_profile",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_source_target_pair",\n            stage="H4_features",\n            why="선행 연구에서 현재 정류장과 목표 정류장의 명시적 쌍이 재귀식보다 하류 예측에 강했다.",\n            if_works="별도 pair 범주가 6정류장 이상 MAE를 낮춰 상호작용 학습 부족을 보완한다.",\n            if_fails="기존 HGB가 두 정류장 피처의 상호작용을 이미 학습했거나 pair 표본이 희소하다.",\n            feature_variant="source_target_pair",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_previous_bus_seat",\n            stage="H4_features",\n            why=(\n                "같은 목표 정류장의 직전 버스 출발 잔여좌석 자체가 국소 수요의 "\n                "가장 직접적인 시계열 신호인지 다른 통계와 분리해 확인한다."\n            ),\n            if_works="단일 좌석 신호만으로 전체 또는 저잔여 MAE가 안정적으로 감소한다.",\n            if_fails="현재 버스의 실시간 좌석 궤적에 비해 직전 버스 값의 추가 정보가 작다.",\n            feature_variant="previous_bus_seat",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_previous_bus",\n            stage="H4_features",\n            why="같은 정류장의 직전 버스 출발 좌석은 국소 수요와 이월 혼잡의 대리변수다.",\n            if_works="특히 만차·저잔여 MAE가 개선되고 일별 효과가 같은 방향이다.",\n            if_fails="직전 버스와 현재 버스 사이 수요 변화가 커 일반 좌석에는 잡음이다.",\n            feature_variant="previous_bus",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_previous_bus_seat_capacity_44_70",\n            stage="H4_features",\n            why=(\n                "직전 버스 출발 잔여좌석의 효과를 검증된 44/70석 차량 용량 "\n                "보정과 결합해 순수한 증분 기여를 비교한다."\n            ),\n            if_works="용량 보정 단독보다 날짜별·저잔여 MAE가 함께 감소한다.",\n            if_fails="직전 버스 좌석은 용량 regime을 보정한 뒤에도 잡음이다.",\n            feature_variant="previous_bus_seat_capacity_44_70",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_previous_bus_normalized_capacity_44_70",\n            stage="H4_features",\n            why=(\n                "절대 출발좌석은 45/70석 정원 불일치와 오래된 참조에서 실패했다. "\n                "적재율·정원 일치·freshness·최근 3대 적재율만 분리하면 국소 수요 "\n                "신호를 좌석 단위 잡음 없이 사용할 수 있다."\n            ),\n            if_works=(\n                "용량 보정 단독보다 전체·저잔여·장거리 MAE가 네 날짜에서 "\n                "일관되게 감소한다."\n            ),\n            if_fails=(\n                "이전 차량 혼잡은 현재 차량 도착 좌석의 점 추정보다 위험 시간대 "\n                "분류에만 유효하거나 참조 차량 교체가 여전히 너무 잦다."\n            ),\n            feature_variant="previous_bus_normalized_capacity_44_70",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_previous_bus_capacity_44_70",\n            stage="H4_features",\n            why=(\n                "직전 버스의 좌석·나이·배차·최근 3대 통계를 44/70석 용량 "\n                "보정과 함께 사용해 정보 결합 효과를 확인한다."\n            ),\n            if_works="직전 버스 단독 및 용량 보정 단독보다 안정적으로 개선된다.",\n            if_fails="짧은 수집 기간의 배차·추세 통계가 과적합을 늘린다.",\n            feature_variant="previous_bus_capacity_44_70",\n        ),\n        candidate_from(\n            weighting_winner,\n            name="hgb_historical_profiles",\n            stage="H4_features",\n            why="선행 연구처럼 정류장쌍·시간대별 평균 변화량을 shrinkage하면 반복 수요를 포착할 수 있다.",\n            if_works="먼 거리와 날짜 외 MAE가 개선되며 충분한 count 그룹에서 효과가 크다.",\n            if_fails="수집 기간이 짧아 동일 정류장쌍·시간대 반복이 부족하거나 주간 변동이 크다.",\n            feature_variant="historical",\n        ),\n    ]\n    evaluate(feature_candidates)\n    results = pd.DataFrame(result_rows)\n    feature_pool = [weighting_winner_name] + [\n        candidate.name for candidate in feature_candidates\n    ]\n    feature_winner_name = best_name(results, feature_pool)\n    feature_winner = registry[feature_winner_name]\n\n    generalization_candidates = [\n        candidate_from(\n            feature_winner,\n            name="hgb_trip_balanced",\n            stage="H5_generalization",\n            why=(\n                "한 운행에서 많은 목표 사건과 스냅샷이 만들어지는 trip이 학습을 "\n                "지배하면 특정 혼잡 운행을 반복 암기할 수 있다."\n            ),\n            if_works="trip당 총 가중치를 같게 했을 때 일별 분산과 과적합 간격이 감소한다.",\n            if_fails="서비스 목표가 사건 평균이므로 trip 균형이 정보량 많은 운행을 과도하게 줄인다.",\n            weighting_kind="trip",\n        ),\n        candidate_from(\n            feature_winner,\n            name="hgb_event_trip_hybrid",\n            stage="H5_generalization",\n            why="사건 균형과 trip 균형의 기하평균은 두 반복 구조 사이의 완만한 절충이다.",\n            if_works="순수 trip 균형보다 전체 MAE를 유지하면서 날짜별 분산을 줄인다.",\n            if_fails="기존 사건 균형이 이미 충분해 가중치 변경이 유효 신호만 약화한다.",\n            weighting_kind="event_trip_hybrid",\n        ),\n        candidate_from(\n            feature_winner,\n            name="hgb_far_weight_2",\n            stage="H5_generalization",\n            why="6정류장 이상 MAE가 근거리의 3배 이상이라 평균 목적함수에서 장거리 기울기가 부족할 수 있다.",\n            if_works="근거리 손실을 제한하면서 6정류장 이상과 신규 저잔여 MAE가 감소한다.",\n            if_fails="장거리 오차는 표본 가중 부족이 아니라 본질적 수요 불확실성 때문이다.",\n            far_weight=2.0,\n            far_threshold=6,\n        ),\n        candidate_from(\n            feature_winner,\n            name="hgb_far_weight_4",\n            stage="H5_generalization",\n            why="장거리 가중치 2의 개선이 부족할 때 전문가 모델의 상한을 확인한다.",\n            if_works="장거리 꼬리오차가 추가로 줄어 거리 gate의 구성요소가 된다.",\n            if_fails="근거리와 전체 MAE 손실이 커져 장거리 데이터 자체가 부족하다는 뜻이다.",\n            far_weight=4.0,\n            far_threshold=6,\n        ),\n        candidate_from(\n            feature_winner,\n            name="hgb_route_residual",\n            stage="H5_generalization",\n            why=(\n                "정류장 flow 누적식은 persistence를 이겼지만 단순 피처 추가는 실패했다. "\n                "물리 예측을 기준점으로 두고 잔차만 학습하면 구조와 보정을 분리할 수 있다."\n            ),\n            if_works="route feature 모델보다 특히 6정류장 이상 MAE가 낮아진다.",\n            if_fails="짧은 기간의 stop-flow baseline 편향이 현재 좌석 기준점보다 불안정하다.",\n            feature_variant="route_profile",\n            target_kind="route_residual",\n        ),\n        candidate_from(\n            feature_winner,\n            name="hgb_route_residual_per_stop",\n            stage="H5_generalization",\n            why="물리 baseline의 남은 오차도 거리와 함께 누적되므로 정류장당 residual로 정규화한다.",\n            if_works="절대 residual보다 장거리 외삽과 일별 안정성이 개선된다.",\n            if_fails="baseline 오차가 특정 정류장에 집중돼 선형 거리 정규화가 맞지 않는다.",\n            feature_variant="route_profile",\n            target_kind="route_residual_per_stop",\n        ),\n    ]\n    evaluate(generalization_candidates)\n    results = pd.DataFrame(result_rows)\n    generalization_pool = [feature_winner_name] + [\n        candidate.name for candidate in generalization_candidates\n    ]\n    generalization_winner_name = best_name(results, generalization_pool)\n    generalization_winner = registry[generalization_winner_name]\n\n    tree_candidates = [\n        candidate_from(\n            generalization_winner,\n            name="extra_trees_diversity",\n            stage="H6_family",\n            why="무작위 분할 ExtraTrees는 boosting과 다른 오차를 내므로 단독 또는 앙상블에 유용할 수 있다.",\n            if_works="단독 MAE가 경쟁력 있거나 HGB와 블렌드할 때 분산이 감소한다.",\n            if_fails="희소 one-hot 공간에서 외삽이 약하고 날짜 변화에 불안정하다.",\n            model_kind="extra_trees",\n            params={\n                "n_estimators": 350,\n                "max_depth": None,\n                "min_samples_leaf": 3,\n                "max_features": 0.7,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="random_forest_diversity",\n            stage="H6_family",\n            why="bootstrap 평균화는 데이터가 짧을 때 HGB보다 날짜별 분산을 줄일 수 있다.",\n            if_works="robust score와 p90 오차가 HGB보다 안정적이다.",\n            if_fails="평균화 편향 때문에 좌석 변화량의 극단을 과소추정한다.",\n            model_kind="random_forest",\n            params={\n                "n_estimators": 300,\n                "max_depth": 22,\n                "min_samples_leaf": 3,\n                "max_features": 0.7,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="extra_trees_compact",\n            stage="H6_family",\n            why=(\n                "무제한 ExtraTrees의 앙상블 이득이 깊은 말단보다 모델 다양성에서 온다면 "\n                "깊이·트리 수를 줄여도 성능을 유지할 수 있다."\n            ),\n            if_works="OOF service score가 1% 이내이고 node 수와 추론시간이 크게 감소한다.",\n            if_fails="희귀 저잔여 패턴을 포착하는 데 깊은 개별 트리가 실제로 필요하다.",\n            model_kind="extra_trees",\n            params={\n                "n_estimators": 160,\n                "max_depth": 18,\n                "min_samples_leaf": 5,\n                "max_features": 0.7,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="extra_trees_tiny",\n            stage="H6_family",\n            why="서비스 가능한 크기의 하한을 확인해 정확도-용량 Pareto 경계를 만든다.",\n            if_works="100개 얕은 트리도 HGB 블렌드의 오차 다양성을 충분히 제공한다.",\n            if_fails="규제가 강해지며 HGB와 다른 유용한 잔차까지 사라진다.",\n            model_kind="extra_trees",\n            params={\n                "n_estimators": 100,\n                "max_depth": 14,\n                "min_samples_leaf": 8,\n                "max_features": 0.7,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="extra_trees_compact_70",\n            stage="H6_family",\n            why=(\n                "compact ExtraTrees의 앙상블 이득은 확인됐지만 160개 트리가 "\n                "100만-node 배포 상한을 넘었다. 깊이 18·leaf 5를 유지하고 "\n                "트리 수만 70으로 줄여 유용한 상호작용과 용량 비용을 분리한다."\n            ),\n            if_works=(\n                "HGB 50% 혼합이 tiny stack보다 낮은 service score를 내면서 "\n                "총 tree node가 100만 이하다."\n            ),\n            if_fails=(\n                "compact의 이득이 많은 트리의 분산 감소에서 왔거나 깊은 개별 "\n                "트리가 날짜 고유 패턴에 과적합된다."\n            ),\n            model_kind="extra_trees",\n            params={\n                "n_estimators": 70,\n                "max_depth": 18,\n                "min_samples_leaf": 5,\n                "max_features": 0.7,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="extra_trees_compact_48",\n            stage="H6_family",\n            why=(\n                "70-tree 후보는 fold 평균이 아니라 최종 재학습 artifact에서 "\n                "100만-node 상한을 넘었다. 같은 깊이 18·leaf 5를 유지하고 "\n                "트리 수만 48로 줄여 실제 최종 번들 제약을 검증한다."\n            ),\n            if_works=(\n                "최종 fit HGB·ExtraTrees·LightGBM node 합이 100만 이하이면서 "\n                "70-tree stack과 OOF service score 차이가 실질적으로 작다."\n            ),\n            if_fails=(\n                "트리 감소로 ExtraTrees의 분산이 커져 앙상블 이득이 사라지거나 "\n                "전체 데이터의 tree 성장이 예상보다 커 상한을 다시 넘는다."\n            ),\n            model_kind="extra_trees",\n            params={\n                "n_estimators": 48,\n                "max_depth": 18,\n                "min_samples_leaf": 5,\n                "max_features": 0.7,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="lightgbm_leafwise_l1",\n            stage="H6_family",\n            why=(\n                "HGB의 63-leaf 과적합 probe가 검증도 개선했으므로 leaf-wise boosting이 "\n                "같은 비선형 신호를 더 효율적으로 포착할 수 있다."\n            ),\n            if_works="HGB보다 낮은 robust score를 작은 artifact와 빠른 추론으로 달성한다.",\n            if_fails="날짜 수가 적어 leaf-wise 성장이 날짜 고유 패턴을 더 과적합한다.",\n            model_kind="lightgbm",\n            params={\n                "objective": "regression_l1",\n                "n_estimators": 600,\n                "learning_rate": 0.025,\n                "num_leaves": 63,\n                "min_child_samples": 10,\n                "subsample": 0.85,\n                "subsample_freq": 1,\n                "colsample_bytree": 0.8,\n                "reg_alpha": 0.1,\n                "reg_lambda": 1.0,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="lightgbm_sqrt_gap_l1",\n            stage="H6_family",\n            why=(\n                "개발 데이터의 좌석 변화량 분산이 gap의 약 0.5제곱으로 "\n                "증가한다. delta/sqrt(gap)으로 이분산을 줄이고 sample weight에 "\n                "sqrt(gap)을 곱해 최종 좌석 MAE 목적을 보존한다."\n            ),\n            if_works=(\n                "기존 delta/gap LightGBM보다 전체·장거리 MAE가 감소하고, "\n                "ExtraTrees/HGB와 혼합할 때 service score도 낮아진다."\n            ),\n            if_fails=(\n                "변화가 특정 정류장에 집중돼 단일 sqrt 거리 스케일이 맞지 "\n                "않거나 장거리 가중이 저잔여 표본을 희석한다."\n            ),\n            model_kind="lightgbm",\n            target_kind="delta_per_sqrt_stop",\n            gap_weight_power=0.5,\n            params={\n                "objective": "regression_l1",\n                "n_estimators": 600,\n                "learning_rate": 0.025,\n                "num_leaves": 63,\n                "min_child_samples": 10,\n                "subsample": 0.85,\n                "subsample_freq": 1,\n                "colsample_bytree": 0.8,\n                "reg_alpha": 0.1,\n                "reg_lambda": 1.0,\n            },\n        ),\n        candidate_from(\n            generalization_winner,\n            name="lightgbm_regularized_l1",\n            stage="H6_family",\n            why="더 규제된 LightGBM으로 leaf-wise 모델의 날짜 변동을 줄일 수 있는지 분리 검증한다.",\n            if_works="깊은 LightGBM보다 train 오차는 높아도 날짜별 분산과 service score가 낮다.",\n            if_fails="추가 규제가 희귀 저잔여 분기까지 지운다.",\n            model_kind="lightgbm",\n            params={\n                "objective": "regression_l1",\n                "n_estimators": 500,\n                "learning_rate": 0.03,\n                "num_leaves": 31,\n                "min_child_samples": 30,\n                "subsample": 0.9,\n                "subsample_freq": 1,\n                "colsample_bytree": 0.8,\n                "reg_alpha": 0.2,\n                "reg_lambda": 3.0,\n            },\n        ),\n    ]\n    evaluate(tree_candidates)\n    results = pd.DataFrame(result_rows)\n\n    best_individual_name = best_name(\n        results,\n        [candidate.name for candidate in registry.values()],\n    )\n    hgb_names = [\n        candidate.name\n        for candidate in registry.values()\n        if candidate.model_kind == "hgb"\n    ]\n    best_hgb_name = best_name(results, hgb_names)\n    tree_names = [candidate.name for candidate in tree_candidates]\n    best_tree_name = best_name(results, tree_names)\n\n    ensemble_specs: dict[str, EnsembleSpec] = {}\n    ensemble_rows: list[dict[str, Any]] = []\n    ensemble_oof: dict[str, pd.DataFrame] = {}\n    for tree_name in tree_names:\n        tree_result = results.loc[results["candidate"].eq(tree_name)].iloc[0]\n        for weight in np.linspace(0, 1, 11):\n            name = (\n                f"blend_{tree_name}_hgb_{int(weight * 100):03d}"\n            )\n            spec = EnsembleSpec(\n                name=name,\n                kind="linear",\n                components=(best_hgb_name, tree_name),\n                params={"left_weight": float(weight)},\n            )\n            output = combine_oof(\n                name,\n                oof_predictions[best_hgb_name],\n                oof_predictions[tree_name],\n                left_weight=float(weight),\n            )\n            row, output = score_ensemble(spec, output)\n            row["mean_fold_tree_nodes"] = float(\n                tree_result["mean_fold_tree_nodes"]\n            )\n            row["prediction_ms_per_1000"] = float(\n                results.loc[\n                    results["candidate"].isin([best_hgb_name, tree_name]),\n                    "prediction_ms_per_1000",\n                ].sum()\n            )\n            ensemble_specs[name] = spec\n            ensemble_rows.append(row)\n            ensemble_oof[name] = output\n\n    route_oof = oof_predictions["route_stop_profile"]\n    for weight in np.linspace(0, 1, 11):\n        name = f"blend_hgb_route_profile_{int(weight * 100):03d}"\n        spec = EnsembleSpec(\n            name=name,\n            kind="linear",\n            components=(best_hgb_name, "route_stop_profile"),\n            params={"left_weight": float(weight)},\n        )\n        output = combine_oof(\n            name,\n            oof_predictions[best_hgb_name],\n            route_oof,\n            left_weight=float(weight),\n        )\n        row, output = score_ensemble(spec, output)\n        row["mean_fold_tree_nodes"] = float(\n            results.loc[\n                results["candidate"].eq(best_hgb_name),\n                "mean_fold_tree_nodes",\n            ].iloc[0]\n        )\n        row["prediction_ms_per_1000"] = float(\n            results.loc[\n                results["candidate"].eq(best_hgb_name),\n                "prediction_ms_per_1000",\n            ].iloc[0]\n        )\n        ensemble_specs[name] = spec\n        ensemble_rows.append(row)\n        ensemble_oof[name] = output\n\n    model_oof = oof_predictions[best_individual_name]\n    persistence_oof = oof_predictions["persistence"]\n    for near_weight in (0.0, 0.25, 0.5, 0.75, 1.0):\n        for far_weight in (0.5, 0.75, 1.0):\n            name = (\n                f"distance_gate_near_{int(near_weight * 100):03d}"\n                f"_far_{int(far_weight * 100):03d}"\n            )\n            weights = np.where(\n                model_oof["target_stop_gap"].le(2).to_numpy(),\n                near_weight,\n                far_weight,\n            )\n            spec = EnsembleSpec(\n                name=name,\n                kind="distance_gate",\n                components=(best_individual_name, "persistence"),\n                params={\n                    "near_left_weight": near_weight,\n                    "far_left_weight": far_weight,\n                },\n            )\n            output = combine_oof(\n                name,\n                model_oof,\n                persistence_oof,\n                left_weight=weights,\n            )\n            row, output = score_ensemble(spec, output)\n            ensemble_specs[name] = spec\n            ensemble_rows.append(row)\n            ensemble_oof[name] = output\n\n    for specialist_name in (\n        "hgb_far_weight_2",\n        "hgb_far_weight_4",\n        "hgb_route_residual",\n        "hgb_route_residual_per_stop",\n    ):\n        for threshold in (6, 11):\n            for specialist_weight in (0.25, 0.5, 0.75, 1.0):\n                name = (\n                    f"stop_gate_{specialist_name}_gap_{threshold}_"\n                    f"weight_{int(specialist_weight * 100):03d}"\n                )\n                base_weight = np.where(\n                    oof_predictions[feature_winner_name]["target_stop_gap"]\n                    .lt(threshold)\n                    .to_numpy(),\n                    1.0,\n                    1.0 - specialist_weight,\n                )\n                spec = EnsembleSpec(\n                    name=name,\n                    kind="stop_gap_gate",\n                    components=(feature_winner_name, specialist_name),\n                    params={\n                        "threshold": float(threshold),\n                        "below_left_weight": 1.0,\n                        "above_left_weight": float(1.0 - specialist_weight),\n                    },\n                )\n                output = combine_oof(\n                    name,\n                    oof_predictions[feature_winner_name],\n                    oof_predictions[specialist_name],\n                    left_weight=base_weight,\n                )\n                row, output = score_ensemble(spec, output)\n                row["mean_fold_tree_nodes"] = float(\n                    results.loc[\n                        results["candidate"].isin(spec.components),\n                        "mean_fold_tree_nodes",\n                    ].sum()\n                )\n                row["prediction_ms_per_1000"] = float(\n                    results.loc[\n                        results["candidate"].isin(spec.components),\n                        "prediction_ms_per_1000",\n                    ].sum()\n                )\n                ensemble_specs[name] = spec\n                ensemble_rows.append(row)\n                ensemble_oof[name] = output\n\n    stack_components = (\n        best_hgb_name,\n        "extra_trees_tiny",\n        "lightgbm_leafwise_l1",\n    )\n    for hgb_units in range(11):\n        for extra_units in range(11 - hgb_units):\n            lightgbm_units = 10 - hgb_units - extra_units\n            raw_weights = {\n                stack_components[0]: hgb_units / 10,\n                stack_components[1]: extra_units / 10,\n                stack_components[2]: lightgbm_units / 10,\n            }\n            weights = {\n                component: weight\n                for component, weight in raw_weights.items()\n                if weight > 0\n            }\n            components = tuple(weights)\n            name = (\n                f"stack3_hgb_{hgb_units * 10:03d}_"\n                f"extra_{extra_units * 10:03d}_"\n                f"lgb_{lightgbm_units * 10:03d}"\n            )\n            spec = EnsembleSpec(\n                name=name,\n                kind="weighted",\n                components=components,\n                params=weights,\n            )\n            output = combine_weighted_oof(\n                name,\n                {component: oof_predictions[component] for component in components},\n                weights,\n            )\n            row, output = score_ensemble(spec, output)\n            row["mean_fold_tree_nodes"] = float(\n                results.loc[\n                    results["candidate"].isin(components),\n                    "mean_fold_tree_nodes",\n                ].sum()\n            )\n            row["prediction_ms_per_1000"] = float(\n                results.loc[\n                    results["candidate"].isin(components),\n                    "prediction_ms_per_1000",\n                ].sum()\n            )\n            ensemble_specs[name] = spec\n            ensemble_rows.append(row)\n            ensemble_oof[name] = output\n\n    # 압축 ExtraTrees와 sqrt-gap LightGBM은 각각 별도의 가설에서 나온\n    # 후보다. 전 조합 grid를 다시 훑지 않고, 기존 최적점 주변에서 두\n    # 구성요소의 증분 기여를 분리하는 세 조합만 검증한다.\n    focused_stack_components = (\n        best_hgb_name,\n        "extra_trees_compact_70",\n        "lightgbm_sqrt_gap_l1",\n    )\n    focused_stack_weights = (\n        (0.5, 0.4, 0.1),\n        (0.4, 0.5, 0.1),\n        (0.4, 0.4, 0.2),\n    )\n    for hgb_weight, extra_weight, lightgbm_weight in focused_stack_weights:\n        weights = dict(\n            zip(\n                focused_stack_components,\n                (hgb_weight, extra_weight, lightgbm_weight),\n                strict=True,\n            )\n        )\n        name = (\n            f"stack_sqrt_hgb_{int(hgb_weight * 100):03d}_"\n            f"extra_{int(extra_weight * 100):03d}_"\n            f"lgb_{int(lightgbm_weight * 100):03d}"\n        )\n        spec = EnsembleSpec(\n            name=name,\n            kind="weighted",\n            components=focused_stack_components,\n            params=weights,\n        )\n        output = combine_weighted_oof(\n            name,\n            {\n                component: oof_predictions[component]\n                for component in focused_stack_components\n            },\n            weights,\n        )\n        row, output = score_ensemble(spec, output)\n        row["mean_fold_tree_nodes"] = float(\n            results.loc[\n                results["candidate"].isin(focused_stack_components),\n                "mean_fold_tree_nodes",\n            ].sum()\n        )\n        row["prediction_ms_per_1000"] = float(\n            results.loc[\n                results["candidate"].isin(focused_stack_components),\n                "prediction_ms_per_1000",\n            ].sum()\n        )\n        ensemble_specs[name] = spec\n        ensemble_rows.append(row)\n        ensemble_oof[name] = output\n\n    # 최종 재학습 artifact에서 70-tree stack이 100만-node를 넘는 것을\n    # 확인한 뒤, 가중치를 다시 탐색하지 않고 트리 수만 48로 줄인 동일\n    # 40:40:20 조합을 배포 제약 대안으로 검증한다.\n    compact_48_components = (\n        best_hgb_name,\n        "extra_trees_compact_48",\n        "lightgbm_sqrt_gap_l1",\n    )\n    compact_48_weights = dict(\n        zip(compact_48_components, (0.4, 0.4, 0.2), strict=True)\n    )\n    compact_48_name = "stack_sqrt48_hgb_040_extra_040_lgb_020"\n    compact_48_spec = EnsembleSpec(\n        name=compact_48_name,\n        kind="weighted",\n        components=compact_48_components,\n        params=compact_48_weights,\n    )\n    compact_48_output = combine_weighted_oof(\n        compact_48_name,\n        {\n            component: oof_predictions[component]\n            for component in compact_48_components\n        },\n        compact_48_weights,\n    )\n    compact_48_row, compact_48_output = score_ensemble(\n        compact_48_spec, compact_48_output\n    )\n    compact_48_row["mean_fold_tree_nodes"] = float(\n        results.loc[\n            results["candidate"].isin(compact_48_components),\n            "mean_fold_tree_nodes",\n        ].sum()\n    )\n    compact_48_row["prediction_ms_per_1000"] = float(\n        results.loc[\n            results["candidate"].isin(compact_48_components),\n            "prediction_ms_per_1000",\n        ].sum()\n    )\n    ensemble_specs[compact_48_name] = compact_48_spec\n    ensemble_rows.append(compact_48_row)\n    ensemble_oof[compact_48_name] = compact_48_output\n\n    # H8: Route 1000\'s current fleet repeatedly exposes 0..44/0..70 point\n    # support. This is not a GBIS-wide capacity contract, so the raw-feed audit\n    # must pass before the no-parameter output projection is even registered.\n    if output_support_audit["selection_candidate_eligible"]:\n        observed_cap_name = f"{compact_48_name}_observed_cap44_70"\n        observed_cap_params = {\n            **compact_48_weights,\n            OBSERVED_SEAT_CEILING_PARAM: 1.0,\n        }\n        observed_cap_spec = EnsembleSpec(\n            name=observed_cap_name,\n            kind="weighted",\n            components=compact_48_components,\n            params=observed_cap_params,\n        )\n        observed_cap_output = combine_weighted_oof(\n            observed_cap_name,\n            {\n                component: oof_predictions[component]\n                for component in compact_48_components\n            },\n            compact_48_weights,\n        )\n        observed_cap_row, observed_cap_output = score_ensemble(\n            observed_cap_spec, observed_cap_output\n        )\n        observed_cap_row.update(\n            {\n                "stage": "H8_output_support",\n                "why": (\n                    "1000번 현행 일반차/2층차의 경험적 점 지지집합은 개발 선택 "\n                    "전인 2026-08-04까지 이미 0~44/0~70석으로 확인됐으므로 "\n                    "명목 45석에서 생기는 범위 밖 점 예측을 제거한다."\n                ),\n                "if_works": (\n                    "44석 초과 일반차 예측만 줄고 모든 날짜와 스트레스 "\n                    "구간에서 오차가 비악화한다."\n                ),\n                "if_fails": (\n                    "현행 fleet 지지집합이 바뀌었거나 상한 근처 예측이 이미 "\n                    "충분히 보정되어 있다."\n                ),\n                "mean_fold_tree_nodes": compact_48_row[\n                    "mean_fold_tree_nodes"\n                ],\n                "prediction_ms_per_1000": compact_48_row[\n                    "prediction_ms_per_1000"\n                ],\n                "output_support_scope": "route_219000013_current_fleet",\n                "output_support_evidence_cutoff": (\n                    OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF\n                ),\n            }\n        )\n        ensemble_specs[observed_cap_name] = observed_cap_spec\n        ensemble_rows.append(observed_cap_row)\n        ensemble_oof[observed_cap_name] = observed_cap_output\n\n    ensemble_results = pd.DataFrame(ensemble_rows)\n    all_selection = pd.concat([results, ensemble_results], ignore_index=True)\n    research_best_name = str(\n        all_selection.sort_values(["service_score", "robust_score"]).iloc[0][\n            "candidate"\n        ]\n    )\n    maximum_final_fit_tree_nodes = 1_000_000\n    final_targets = {"test": test, "stress": stress}\n    development_flows = stop_flows.loc[\n        stop_flows["date"].lt(args.test_date)\n    ].copy()\n    final_predictions: dict[str, dict[str, np.ndarray]] = {}\n    fitted_models: dict[str, Pipeline] = {}\n    final_metadata: dict[str, dict[str, Any]] = {}\n    final_component_tree_nodes: dict[str, int] = {}\n\n    def ensure_final_fit(name: str) -> None:\n        if name in final_predictions:\n            return\n        candidate = registry[name]\n        result = results.loc[results["candidate"].eq(name)].iloc[0]\n        predictions, model, model_metadata = fit_final_candidate(\n            candidate,\n            development,\n            final_targets,\n            development_flows,\n            bias=float(result["bias_correction"]),\n            seed=args.seed,\n        )\n        final_predictions[name] = predictions\n        final_component_tree_nodes[name] = tree_node_count(model)\n        if model is not None:\n            fitted_models[name] = model\n        model_metadata.update(\n            {\n                "candidate": asdict(candidate),\n                "bias_correction": float(result["bias_correction"]),\n                "bias_correction_source": str(\n                    result["bias_correction_source"]\n                ),\n                "oof_bias_correction_rule": str(\n                    result["oof_bias_correction_rule"]\n                ),\n                "final_fit_tree_nodes": final_component_tree_nodes[name],\n            }\n        )\n        final_metadata[name] = model_metadata\n\n    selected_name = select_with_final_node_budget(\n        all_selection,\n        ensemble_specs,\n        ensure_final_fit=ensure_final_fit,\n        final_component_tree_nodes=final_component_tree_nodes,\n        maximum_final_fit_tree_nodes=maximum_final_fit_tree_nodes,\n    )\n\n    component_names = {"persistence", best_individual_name}\n    if selected_name in ensemble_specs:\n        component_names.update(ensemble_specs[selected_name].components)\n    else:\n        component_names.add(selected_name)\n    for name in component_names:\n        ensure_final_fit(name)\n\n    for old_artifact in args.output_dir.glob("*.joblib"):\n        old_artifact.unlink()\n    for old_metadata in args.output_dir.glob("*.metadata.json"):\n        old_metadata.unlink()\n    artifact_sizes: dict[str, int] = {}\n    requires_route_profile = False\n    for name in sorted(component_names):\n        candidate = registry[name]\n        requires_route_profile = (\n            requires_route_profile or candidate.feature_variant == "route_profile"\n        )\n        model = fitted_models.get(name)\n        if model is not None:\n            artifact_path = args.output_dir / f"{name}.joblib"\n            joblib.dump(model, artifact_path, compress=3)\n            artifact_sizes[name] = int(artifact_path.stat().st_size)\n        (args.output_dir / f"{name}.metadata.json").write_text(\n            json.dumps(\n                json_ready(final_metadata[name]), ensure_ascii=False, indent=2\n            ),\n            encoding="utf-8",\n        )\n    if requires_route_profile:\n        flow_artifact = args.output_dir / "route_profile_flows.joblib"\n        joblib.dump(development_flows, flow_artifact, compress=3)\n        artifact_sizes["route_profile_flows"] = int(flow_artifact.stat().st_size)\n\n    saved_artifact_sha256 = {\n        path.name: cache_payload_sha256(path)\n        for path in sorted(\n            [\n                *args.output_dir.glob("*.joblib"),\n                *args.output_dir.glob("*.metadata.json"),\n            ],\n            key=lambda value: value.name,\n        )\n    }\n\n    if selected_name in ensemble_specs:\n        selected_spec = ensemble_specs[selected_name]\n        selected_test = ensemble_prediction(\n            selected_spec,\n            {name: final_predictions[name]["test"] for name in selected_spec.components},\n            test,\n        )\n        selected_stress = ensemble_prediction(\n            selected_spec,\n            {name: final_predictions[name]["stress"] for name in selected_spec.components},\n            stress,\n        ) if not stress.empty else np.asarray([], dtype=float)\n    else:\n        selected_spec = None\n        selected_test = final_predictions[selected_name]["test"]\n        selected_stress = final_predictions[selected_name]["stress"]\n\n    final_rows: list[dict[str, Any]] = []\n    final_output_names = ["persistence", best_individual_name, selected_name]\n    for name in dict.fromkeys(final_output_names):\n        if name == selected_name:\n            prediction = selected_test\n        else:\n            prediction = final_predictions[name]["test"]\n        row = event_summary(test, prediction)\n        row.update({"model": name, "split": args.test_date})\n        final_rows.append(row)\n    final_metrics = pd.DataFrame(final_rows)\n\n    stress_metrics = pd.DataFrame()\n    if not stress.empty:\n        stress_row = event_summary(stress, selected_stress)\n        stress_row.update(\n            {\n                "model": selected_name,\n                "split": "weekend_2026-08-08_09",\n            }\n        )\n        stress_metrics = pd.DataFrame([stress_row])\n\n    by_stop = stop_band_metrics(test, selected_test, selected_name)\n    persistence_test = final_predictions["persistence"]["test"]\n    bootstrap = cluster_bootstrap_mae_delta(\n        test,\n        selected_test,\n        persistence_test,\n        seed=args.seed,\n    )\n    bootstrap_best_individual = cluster_bootstrap_mae_delta(\n        test,\n        selected_test,\n        final_predictions[best_individual_name]["test"],\n        seed=args.seed,\n    )\n    prediction_output = test[\n        [\n            "event_id",\n            "date",\n            "event_time",\n            "snapshot_time",\n            "trip_id",\n            "station_seq_cat",\n            "snapshot_station_seq",\n            "target_stop_gap",\n            "minutes_to_arrival",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "previous_bus_departure_seats",\n        ]\n    ].copy()\n    prediction_output["persistence"] = persistence_test\n    prediction_output["selected_prediction"] = selected_test\n    prediction_output["absolute_error"] = np.abs(\n        prediction_output["label_seats"] - selected_test\n    )\n\n    selected_oof = (\n        ensemble_oof[selected_name]\n        if selected_name in ensemble_oof\n        else oof_predictions[selected_name]\n    ).copy()\n    persistence_oof_aligned = (\n        oof_predictions["persistence"]\n        .set_index(["event_id", "snapshot_time"])["prediction"]\n        .reindex(\n            pd.MultiIndex.from_frame(\n                selected_oof[["event_id", "snapshot_time"]]\n            )\n        )\n        .to_numpy(dtype=float)\n    )\n    selected_oof["persistence_prediction"] = persistence_oof_aligned\n    selected_oof["absolute_error"] = np.abs(\n        selected_oof["label_seats"] - selected_oof["prediction"]\n    )\n\n    deployment_columns = all_selection[\n        [\n            "candidate",\n            "final_fit_tree_nodes",\n            "deployment_eligible",\n            "deployment_check",\n        ]\n    ]\n    results = results.merge(deployment_columns, on="candidate", how="left")\n    ensemble_results = ensemble_results.merge(\n        deployment_columns, on="candidate", how="left"\n    )\n    results["stage_rank"] = results.groupby("stage")["service_score"].rank(\n        method="min"\n    )\n    results["verdict"] = np.where(\n        results["stage_rank"].eq(1),\n        "stage_best",\n        "not_selected",\n    )\n    fold_metrics = pd.DataFrame(fold_rows)\n    results.to_csv(args.output_dir / "hypothesis_results.csv", index=False)\n    fold_metrics.to_csv(args.output_dir / "fold_metrics.csv", index=False)\n    ensemble_results.to_csv(args.output_dir / "ensemble_results.csv", index=False)\n    final_metrics.to_csv(args.output_dir / "final_test_metrics.csv", index=False)\n    stress_metrics.to_csv(args.output_dir / "weekend_stress_metrics.csv", index=False)\n    by_stop.to_csv(args.output_dir / "final_metrics_by_stop_band.csv", index=False)\n    prediction_output.to_csv(\n        args.output_dir / "final_test_predictions.csv", index=False\n    )\n    selected_oof.to_csv(\n        args.output_dir / "selected_oof_predictions.csv", index=False\n    )\n\n    selected_development = all_selection.loc[\n        all_selection["candidate"].eq(selected_name)\n    ].iloc[0]\n    selected_components = (\n        selected_spec.components\n        if selected_spec is not None\n        else (selected_name,)\n    )\n    selected_artifact_sizes = {\n        name: artifact_sizes[name]\n        for name in selected_components\n        if name in artifact_sizes\n    }\n    deployment_evaluations = all_selection.loc[\n        all_selection["deployment_check"].ne(\n            "not_evaluated_after_selection"\n        ),\n        [\n            "candidate",\n            "final_fit_tree_nodes",\n            "deployment_eligible",\n            "deployment_check",\n        ],\n    ]\n    summary = {\n        "route_id": ROUTE_ID,\n        "route_name": ROUTE_NAME,\n        "target": "arrival_seats before boarding",\n        "label_quality": ["A"],\n        "data_cache": cache_info,\n        "protocol": {\n            "experiment_source_sha256": experiment_source_sha256,\n            "inference_source_sha256": inference_source_sha256,\n            "hypothesis_and_parameter_selection": {\n                "rolling_origin_validation_dates": list(development_dates[1:]),\n                "training_rule": "all earlier development dates",\n                "oof_bias_rule": STRICT_FORWARD_BIAS_RULE,\n                "deployment_bias_source": DEPLOYMENT_BIAS_SOURCE,\n                "selection_score": (\n                    "mean daily event-balanced MAE + 0.5 * day std + "\n                    "0.05 * low-seat MAE"\n                ),\n            },\n            "locked_confirmation_test": {\n                "date": args.test_date,\n                "selection_use": "none",\n                "caveat": (\n                    "partial-day data; label counts were inspected before scoring, "\n                    "so this is not a pristine holdout"\n                ),\n            },\n            "weekend_distribution_shift_stress": list(DEFAULT_STRESS_DATES),\n            "leakage_rules": [\n                "actual minutes_to_arrival is evaluation-only",\n                "historical target profiles are train-only and leave-one-row-out in training",\n                "previous-bus departures must be strictly earlier than snapshot_time",\n                "OOF bias for each validation date uses only earlier OOF residuals",\n                "the final test date is not used for candidate, parameter, or blend selection",\n                (\n                    "the Route 1000 44/70 output support was evidenced no later "\n                    f"than {OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF}"\n                ),\n            ],\n        },\n        "output_support_constraint": {\n            "selected_model_enabled": bool(\n                selected_spec is not None\n                and float(\n                    selected_spec.params.get(OBSERVED_SEAT_CEILING_PARAM, 0.0)\n                )\n                != 0.0\n            ),\n            "scope": "route_219000013_current_fleet_only",\n            "mapping": {"snapshot_low_plate_cat_0": 44, "category_2": 70},\n            "evidence_cutoff": OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF,\n            "unknown_category_rule": "fail_closed",\n            "not_a_gbis_wide_capacity_contract": True,\n            "raw_source_audit": output_support_audit,\n            "source_fingerprint": cache_info.get("fingerprint"),\n            "point_projection_only": (\n                "prediction intervals retain the existing nominal-capacity support"\n            ),\n        },\n        "data": {\n            "development_rows": int(len(development)),\n            "development_events": int(development["event_id"].nunique()),\n            "test_rows": int(len(test)),\n            "test_events": int(test["event_id"].nunique()),\n            "test_low_0_5_events": int(\n                test.loc[test["label_seats"].le(5), "event_id"].nunique()\n            ),\n            "stress_rows": int(len(stress)),\n            "stress_events": int(stress["event_id"].nunique()),\n        },\n        "stage_winners": {\n            "target_representation": target_winner_name,\n            "capacity_and_loss": capacity_winner_name,\n            "rare_weighting": weighting_winner_name,\n            "feature_engineering": feature_winner_name,\n            "generalization": generalization_winner_name,\n            "best_hgb": best_hgb_name,\n            "best_tree": best_tree_name,\n            "best_individual": best_individual_name,\n            "research_best_without_deployment_constraint": research_best_name,\n            "selected_final": selected_name,\n        },\n        "deployment_constraint": {\n            "rule": (\n                "lazy score-ordered selection using actual full-development "\n                "fitted component node counts"\n            ),\n            "maximum_final_fit_tree_nodes": maximum_final_fit_tree_nodes,\n            "selected_final_fit_tree_nodes": int(\n                selected_development["final_fit_tree_nodes"]\n            ),\n            "selected_component_tree_nodes": {\n                name: final_component_tree_nodes[name]\n                for name in selected_components\n            },\n            "selected_artifact_sizes_bytes": selected_artifact_sizes,\n            "all_saved_artifact_sizes_bytes": artifact_sizes,\n            "saved_artifact_sha256": saved_artifact_sha256,\n            "saved_artifact_manifest_rule": (\n                "all joblib and component metadata files present after stale "\n                "artifacts are removed and before summary publication"\n            ),\n            "evaluated_candidates": _as_records(deployment_evaluations),\n        },\n        "selected_development_metrics": json_ready(selected_development.to_dict()),\n        "selected_ensemble": (\n            asdict(selected_spec) if selected_spec is not None else None\n        ),\n        "final_test_metrics": _as_records(final_metrics),\n        "weekend_stress_metrics": _as_records(stress_metrics),\n        "final_metrics_by_stop_band": _as_records(by_stop),\n        "bootstrap_improvement_vs_persistence": bootstrap,\n        "bootstrap_improvement_vs_best_individual": bootstrap_best_individual,\n        "limitations": [\n            "잠금 확인셋은 2026-08-11 오전 일부이며 평가 전 라벨 건수는 확인했다.",\n            "2026-08-12 이후의 완전히 미개봉 평일 데이터가 아직 없다.",\n            "학기·날씨·승하차·대기열 데이터는 아직 없다.",\n            "저잔여 사건 수가 작아 서비스 경고 확률은 별도 장기 보정이 필요하다.",\n            "모델 선택용 rolling 검증일은 4개라 날짜 분산 추정이 아직 불안정하다.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(json.dumps(json_ready(summary), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/rebuild_active_route_feature_cache.py': 'from __future__ import annotations\n\nimport argparse\nimport sqlite3\nfrom pathlib import Path\n\nfrom route_specific_feature_experiment import DEFAULT_ROUTES, load_or_build_route_cache\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="전체 활성 노선의 snapshot/flow feature cache를 하나의 source cutoff로 재생성"\n    )\n    parser.add_argument("--database", type=Path, default=Path("data/gbis_api_cache.sqlite3"))\n    parser.add_argument(\n        "--cache-dir", type=Path, default=Path("data/analysis_cache/route_specific_features")\n    )\n    parser.add_argument("--source-cutoff")\n    args = parser.parse_args()\n    if args.source_cutoff:\n        cutoff = args.source_cutoff\n    else:\n        with sqlite3.connect(args.database) as connection:\n            cutoff = connection.execute(\n                "SELECT MAX(observed_at) FROM location_history"\n            ).fetchone()[0]\n    if not cutoff:\n        raise ValueError("location_history에 관측값이 없습니다.")\n    print(f"source cutoff: {cutoff}", flush=True)\n    for route_id, route_name in DEFAULT_ROUTES.items():\n        _, _, metadata = load_or_build_route_cache(\n            args.database,\n            args.cache_dir,\n            route_id,\n            source_cutoff=cutoff,\n            rebuild=True,\n        )\n        print(\n            f"[{route_name}] snapshots={metadata[\'snapshot_rows\']:,} flows={metadata[\'flow_rows\']:,}",\n            flush=True,\n        )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/transition_weight_threshold_search.py': '"""Search transition-case sample weights and full-occupancy thresholds.\n\nThe regression model is always trained on all rows.  Only rows with current\nseats 11--25 and arrival seats <=10 receive the searched extra weight.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sqlite3\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\n\nfrom all_prearrival_seat_regression import event_weights\nfrom emerging_low_correction_experiment import (\n    CURRENT_SEAT_MAX,\n    CURRENT_SEAT_MIN,\n    blend_by_fold,\n    calibrate,\n    complete_folds,\n    metric_tables,\n    pooled_features,\n    transition_label,\n)\nfrom latest_main_model_feature_recheck import LATEST_COMPLETE_DATES\nfrom linear_feature_experiment import json_ready\nfrom main_model_feature_augmentation import required_metrics\nfrom route_specific_feature_experiment import DEFAULT_ROUTES\n\n\nDEFAULT_WEIGHTS = (1.5, 2.0, 3.0, 4.0)\nDEFAULT_THRESHOLDS = tuple(np.arange(-2.0, 3.01, 0.25))\n\n\ndef weight_name(weight: float) -> str:\n    return f"transition_weight_{weight:g}x"\n\n\ndef scopes() -> dict[str, Any]:\n    return {\n        "all_complete": lambda frame: pd.Series(True, index=frame.index),\n        "current_seats_le25": lambda frame: frame["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX),\n        "emerging_low_11_25_to_low10": transition_label,\n    }\n\n\ndef threshold_metrics(frame: pd.DataFrame, threshold: float) -> dict[str, float]:\n    truth = frame["label_seats"].to_numpy(float)\n    predicted_full = frame["prediction"].to_numpy(float) <= threshold\n    full_true = truth == 0\n    weights = event_weights(frame)\n    return {\n        "full_accuracy": float(accuracy_score(full_true, predicted_full, sample_weight=weights)),\n        "full_recall": float(recall_score(full_true, predicted_full, sample_weight=weights, zero_division=0)),\n        "full_precision": float(precision_score(full_true, predicted_full, sample_weight=weights, zero_division=0)),\n        "full_f1": float(f1_score(full_true, predicted_full, sample_weight=weights, zero_division=0)),\n    }\n\n\ndef threshold_tables(predictions: pd.DataFrame, thresholds: tuple[float, ...]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pooled_rows: list[dict[str, Any]] = []\n    route_rows: list[dict[str, Any]] = []\n    expanded = predictions.copy()\n    expanded["route_id"] = expanded["event_id"].str.split("::", n=1).str[0]\n    for variant, frame in expanded.groupby("variant", sort=False):\n        for scope, rule in scopes().items():\n            selected = frame.loc[rule(frame)]\n            if selected.empty:\n                continue\n            regression = required_metrics(selected)\n            for threshold in thresholds:\n                row = {\n                    "variant": variant,\n                    "scope": scope,\n                    **regression,\n                    "full_threshold_seats": threshold,\n                    **threshold_metrics(selected, threshold),\n                }\n                pooled_rows.append(row)\n                for route_id, route in selected.groupby("route_id", sort=True):\n                    route_rows.append(\n                        {\n                            "variant": variant,\n                            "scope": scope,\n                            "route_id": route_id,\n                            "route_name": DEFAULT_ROUTES[route_id],\n                            "full_threshold_seats": threshold,\n                            **threshold_metrics(route, threshold),\n                        }\n                    )\n    pooled = pd.DataFrame(pooled_rows)\n    by_route = pd.DataFrame(route_rows)\n    metric_columns = ["full_accuracy", "full_recall", "full_precision", "full_f1"]\n    macro = by_route.groupby(["variant", "scope", "full_threshold_seats"], sort=False)[metric_columns].mean().reset_index()\n    macro.insert(3, "routes", len(DEFAULT_ROUTES))\n    return pooled, by_route, macro\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="전체 학습 전환 가중치·만석 threshold 탐색")\n    parser.add_argument("--featured-cache", type=Path, default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"))\n    parser.add_argument("--database", type=Path, default=Path("data/gbis_api_cache.sqlite3"))\n    parser.add_argument("--output-dir", type=Path, default=Path("analysis/transition_weight_threshold_results"))\n    parser.add_argument("--weights", type=float, nargs="*", default=list(DEFAULT_WEIGHTS))\n    parser.add_argument("--thresholds", type=float, nargs="*")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.featured_cache)\n    source_cutoff = data.attrs.get("source_cutoff")\n    if not source_cutoff:\n        raise ValueError("featured cache lacks source_cutoff metadata")\n    with sqlite3.connect(f"file:{args.database.resolve()}?mode=ro", uri=True) as connection:\n        source_newest = connection.execute("SELECT max(observed_at) FROM location_history").fetchone()[0]\n    folds = complete_folds(data)\n    features = pooled_features()\n\n    outputs: list[pd.DataFrame] = []\n    baseline, _, _ = blend_by_fold("whole_data", folds, features, seed=args.seed)\n    outputs.append(calibrate(baseline, "baseline"))\n    for weight in args.weights:\n        if weight <= 1:\n            raise ValueError("searched transition weights must be greater than 1")\n        prediction, _, _ = blend_by_fold(\n            "transition_weighted",\n            folds,\n            features,\n            seed=args.seed,\n            transition_weight=weight,\n        )\n        outputs.append(calibrate(prediction, weight_name(weight)))\n    predictions = pd.concat(outputs, ignore_index=True)\n    standard_pooled, standard_by_route, standard_macro = metric_tables(predictions)\n    thresholds = tuple(args.thresholds) if args.thresholds else DEFAULT_THRESHOLDS\n    threshold_pooled, threshold_by_route, threshold_macro = threshold_tables(predictions, thresholds)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")\n    standard_pooled.to_csv(args.output_dir / "metrics_at_default_threshold_pooled.csv", index=False)\n    standard_by_route.to_csv(args.output_dir / "metrics_at_default_threshold_by_route.csv", index=False)\n    standard_macro.to_csv(args.output_dir / "metrics_at_default_threshold_route_macro.csv", index=False)\n    threshold_pooled.to_csv(args.output_dir / "threshold_sweep_pooled.csv", index=False)\n    threshold_by_route.to_csv(args.output_dir / "threshold_sweep_by_route.csv", index=False)\n    threshold_macro.to_csv(args.output_dir / "threshold_sweep_route_macro.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": source_cutoff,\n            "source_newest_observation_at_refresh": source_newest,\n            "included_routes": DEFAULT_ROUTES,\n            "excluded_routes": "historical-only routes not actively collected at source cutoff",\n            "selection_dates": list(LATEST_COMPLETE_DATES),\n            "partial_dates_excluded": ["2026-08-13", "2026-08-14"],\n            "evaluation": "all pre-arrival snapshots; event-balanced metrics",\n            "baseline": "all training rows; deployed-family weighted ensemble",\n            "candidate": f"all training rows, with {CURRENT_SEAT_MIN}-{CURRENT_SEAT_MAX} current-seat and arrival <=10 rows multiplied by the searched transition weight",\n            "searched_weights": list(args.weights),\n            "searched_full_threshold_seats": list(thresholds),\n            "warning": "threshold sweep is retrospective OOF analysis; choose a production threshold on a separate calibration period.",\n        },\n        "default_threshold_pooled_metrics": standard_pooled.to_dict(orient="records"),\n        "default_threshold_route_macro_metrics": standard_macro.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8")\n    print("\\nDefault-threshold pooled metrics\\n", standard_pooled.to_string(index=False))\n    print("\\nThreshold sweep (all complete, sorted by F1)\\n", threshold_pooled.loc[threshold_pooled["scope"].eq("all_complete")].sort_values("full_f1", ascending=False).head(12).to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/plot_station_time_low_rate.py': 'from __future__ import annotations\n\nimport argparse\nimport sqlite3\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\n\nLOW_SEAT_THRESHOLD = 5\n\n\ndef load_station_names(\n    database: Path, data: pd.DataFrame\n) -> tuple[dict[int, str], str | None]:\n    if not database.exists():\n        return {}, None\n    targets = (\n        data[["station_seq_cat", "x", "y"]]\n        .dropna()\n        .drop_duplicates()\n        .assign(station_seq=lambda frame: pd.to_numeric(frame["station_seq_cat"]).astype(int))\n    )\n    with sqlite3.connect(database) as connection:\n        metadata = pd.read_sql_query(\n            "SELECT route_id, station_seq, station_name, x, y FROM route_stations",\n            connection,\n        )\n        candidates = targets.merge(metadata, on="station_seq", suffixes=("_target", "_meta"))\n        candidates["coordinate_match"] = (\n            candidates["x_target"].sub(candidates["x_meta"]).abs().le(1e-5)\n            & candidates["y_target"].sub(candidates["y_meta"]).abs().le(1e-5)\n        )\n        scores = candidates.groupby("route_id")["coordinate_match"].sum().sort_values(ascending=False)\n        if scores.empty or int(scores.iloc[0]) == 0:\n            return {}, None\n        route_id = str(scores.index[0])\n        rows = connection.execute(\n            """\n            SELECT station_seq, station_name\n            FROM route_stations\n            WHERE route_id = ?\n            ORDER BY station_seq\n            """,\n            (route_id,),\n        ).fetchall()\n    return {int(seq): str(name or "") for seq, name in rows}, route_id\n\n\ndef station_time_table(data: pd.DataFrame, minimum_count: int) -> pd.DataFrame:\n    events = data.drop_duplicates("event_id").copy()\n    events["target_station_seq"] = pd.to_numeric(events["station_seq_cat"]).astype(int)\n    events["arrival_hour"] = events["event_time"].dt.hour.astype(int)\n    events["is_low_0_5"] = events["label_seats"].le(LOW_SEAT_THRESHOLD)\n    result = (\n        events.groupby(["target_station_seq", "arrival_hour"], observed=True)\n        .agg(event_count=("event_id", "size"), low_event_count=("is_low_0_5", "sum"))\n        .reset_index()\n    )\n    result["low_rate_pct"] = (\n        result["low_event_count"] / result["event_count"] * 100\n    )\n    result["reliable"] = result["event_count"].ge(minimum_count)\n    return result\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="목표 정류장×도착 시간대 저잔여율")\n    parser.add_argument(\n        "--cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--database", type=Path, default=Path("data/gbis.sqlite3")\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/feature_distribution_results"),\n    )\n    parser.add_argument("--minimum-count", type=int, default=10)\n    parser.add_argument("--color-ceiling", type=float, default=None)\n    args = parser.parse_args()\n    if args.minimum_count <= 0:\n        raise ValueError("minimum-count는 1 이상이어야 합니다.")\n    if args.color_ceiling is not None and args.color_ceiling <= 0:\n        raise ValueError("color-ceiling은 0보다 커야 합니다.")\n\n    data = pd.read_pickle(args.cache)\n    required = {"event_id", "date", "event_time", "label_seats", "station_seq_cat"}\n    missing = sorted(required - set(data.columns))\n    if missing:\n        raise ValueError(f"필수 열이 없습니다: {missing}")\n\n    table = station_time_table(data, args.minimum_count)\n    station_names, route_id = load_station_names(args.database, data)\n    table["station_name"] = table["target_station_seq"].map(station_names).fillna("")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    csv_path = args.output_dir / "station_time_low_rate.csv"\n    table.to_csv(csv_path, index=False)\n\n    stations = sorted(table["target_station_seq"].unique())\n    hours = list(range(int(table["arrival_hour"].min()), int(table["arrival_hour"].max()) + 1))\n    rate = table.pivot(\n        index="target_station_seq", columns="arrival_hour", values="low_rate_pct"\n    ).reindex(index=stations, columns=hours)\n    count = table.pivot(\n        index="target_station_seq", columns="arrival_hour", values="event_count"\n    ).reindex(index=stations, columns=hours)\n    reliable_rate = rate.where(count.ge(args.minimum_count))\n\n    valid_values = reliable_rate.to_numpy(dtype=float)\n    valid_values = valid_values[np.isfinite(valid_values)]\n    color_ceiling = (\n        float(args.color_ceiling)\n        if args.color_ceiling is not None\n        else max(float(np.quantile(valid_values, 0.95)), 1.0)\n    )\n\n    plt.rcParams["font.family"] = "Noto Sans CJK KR"\n    plt.rcParams["axes.unicode_minus"] = False\n    figure, ax = plt.subplots(figsize=(20, 16))\n    figure.subplots_adjust(left=0.20, right=0.90, bottom=0.09, top=0.92)\n    color_map = plt.get_cmap("YlOrRd").copy()\n    color_map.set_bad("#E0E0E0")\n    image = ax.imshow(\n        reliable_rate.to_numpy(dtype=float),\n        aspect="auto",\n        cmap=color_map,\n        vmin=0,\n        vmax=color_ceiling,\n    )\n\n    ax.set_xticks(np.arange(len(hours)), labels=[f"{hour:02d}:00" for hour in hours])\n    station_labels = []\n    for seq in stations:\n        name = station_names.get(seq, "")\n        short_name = name if len(name) <= 20 else f"{name[:19]}…"\n        station_labels.append(f"{seq} · {short_name}" if short_name else str(seq))\n    ax.set_yticks(np.arange(len(stations)), labels=station_labels)\n    ax.tick_params(axis="y", labelsize=8)\n    ax.set_xlabel("실제 도착 시간대")\n    ax.set_ylabel("목표 정류장")\n\n    rate_values = rate.to_numpy(dtype=float)\n    count_values = count.to_numpy(dtype=float)\n    for row in range(len(stations)):\n        for column in range(len(hours)):\n            value = rate_values[row, column]\n            sample_count = count_values[row, column]\n            if not np.isfinite(value) or not np.isfinite(sample_count):\n                continue\n            if sample_count < args.minimum_count:\n                label = "·"\n                color = "#777777"\n            else:\n                label = f"{value:.0f}"\n                color = "white" if value >= color_ceiling * 0.55 else "#222222"\n            ax.text(column, row, label, ha="center", va="center", fontsize=7, color=color)\n\n    colorbar = figure.colorbar(image, ax=ax, fraction=0.025, pad=0.02)\n    colorbar.set_label("저잔여율(%)")\n    first_date = str(data["date"].astype(str).min())\n    last_date = str(data["date"].astype(str).max())\n    ax.set_title(\n        f"목표 정류장 × 실제 도착 시간대 저잔여율 (0~{LOW_SEAT_THRESHOLD}석)\\n"\n        f"{first_date} ~ {last_date} · 노선 {route_id or \'미확인\'} · 셀 숫자는 저잔여율(%)",\n        fontsize=16,\n        fontweight="bold",\n    )\n    figure.text(\n        0.5,\n        0.02,\n        f"회색·점 표시는 표본 {args.minimum_count}건 미만 또는 관측 없음 · 색상 상한 {color_ceiling:.1f}%",\n        ha="center",\n        fontsize=10,\n        color="#555555",\n    )\n\n    output_path = args.output_dir / "station_time_low_rate.png"\n    figure.savefig(output_path, dpi=180, bbox_inches="tight")\n    plt.close(figure)\n    print(output_path)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/route_distribution_search.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize_scalar\nfrom sklearn.metrics import average_precision_score, log_loss, roc_auc_score\n\nfrom all_prearrival_seat_regression import event_weights\nfrom route_distribution import (\n    LOWER_TAIL_PREDICTION_LABELS,\n    LOWER_TAIL_REFINEMENT_KIND,\n    POLICY_PROVENANCE_SCHEMA_VERSION,\n    ROUTE_DISTRIBUTION_FEATURES,\n    apply_interval_policy as apply_common_interval_policy,\n    apply_probability_calibration,\n    distribution_flow_fingerprint,\n    file_sha256,\n    json_ready,\n    lower_tail_prediction_band,\n    lower_tail_scale_values,\n    point_model_input_provenance_sha256,\n    route_distribution_values,\n    validate_interval_policy,\n)\n\n\nVALIDATION_DATES = (\n    "2026-08-05",\n    "2026-08-06",\n    "2026-08-07",\n    "2026-08-10",\n)\nSTOP_BINS = (0, 2, 5, 10, np.inf)\nSTOP_LABELS = ("1-2", "3-5", "6-10", "11+")\nTARGET_COVERAGE = 0.9\nFROZEN_DISTRIBUTION_FLOW_FILENAME = "frozen_distribution_flows.pkl"\nLOWER_TAIL_COVERAGE = 0.95\nLOWER_TAIL_SCALE_FLOOR = 0.5\nLOWER_TAIL_SHRINKAGE_EVENT_COUNT = 200.0\n\n\ndef _verify_frozen_distribution_flows(\n    path: Path,\n    flows: pd.DataFrame,\n    expected_fingerprint: str,\n) -> None:\n    restored = pd.read_pickle(path)\n    if not isinstance(restored, pd.DataFrame) or not restored.equals(flows):\n        raise ValueError("저장한 frozen distribution flow DataFrame이 원본과 다릅니다.")\n    if restored.attrs != flows.attrs:\n        raise ValueError("저장한 frozen distribution flow attrs가 원본과 다릅니다.")\n    if distribution_flow_fingerprint(restored) != expected_fingerprint:\n        raise ValueError("저장한 frozen distribution flow fingerprint가 다릅니다.")\n\n\ndef _atomic_text(path: Path, contents: str) -> None:\n    """Replace one derived text artifact only after a complete durable write."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary_path: Path | None = None\n    try:\n        with tempfile.NamedTemporaryFile(\n            mode="w",\n            encoding="utf-8",\n            dir=path.parent,\n            prefix=f".{path.name}.",\n            suffix=".tmp",\n            delete=False,\n        ) as temporary:\n            temporary_path = Path(temporary.name)\n            temporary.write(contents)\n            temporary.flush()\n            os.fsync(temporary.fileno())\n        os.chmod(temporary_path, 0o644)\n        os.replace(temporary_path, path)\n        temporary_path = None\n    finally:\n        if temporary_path is not None:\n            temporary_path.unlink(missing_ok=True)\n\n\ndef _atomic_csv(frame: pd.DataFrame, path: Path) -> None:\n    """Replace one derived CSV only after serialization succeeds."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary_path: Path | None = None\n    try:\n        with tempfile.NamedTemporaryFile(\n            mode="w",\n            encoding="utf-8",\n            newline="",\n            dir=path.parent,\n            prefix=f".{path.name}.",\n            suffix=".tmp",\n            delete=False,\n        ) as temporary:\n            temporary_path = Path(temporary.name)\n            frame.to_csv(temporary, index=False)\n            temporary.flush()\n            os.fsync(temporary.fileno())\n        os.chmod(temporary_path, 0o644)\n        os.replace(temporary_path, path)\n        temporary_path = None\n    finally:\n        if temporary_path is not None:\n            temporary_path.unlink(missing_ok=True)\n\n\ndef freeze_distribution_flows(\n    flows: pd.DataFrame,\n    output_dir: Path,\n) -> tuple[Path, dict[str, Any]]:\n    """Persist the exact policy input beside the policy, independent of caches."""\n\n    if not isinstance(flows, pd.DataFrame):\n        raise TypeError("frozen distribution flow는 pandas DataFrame이어야 합니다.")\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    artifact_path = output_dir / FROZEN_DISTRIBUTION_FLOW_FILENAME\n    expected_fingerprint = distribution_flow_fingerprint(flows)\n    if artifact_path.exists():\n        try:\n            _verify_frozen_distribution_flows(\n                artifact_path, flows, expected_fingerprint\n            )\n        except Exception as error:\n            raise FileExistsError(\n                "기존 frozen distribution flow를 덮어쓰지 않습니다. 다른 flow로 "\n                "정책을 생성하려면 새 --output-dir을 사용하세요: "\n                f"{artifact_path}"\n            ) from error\n    else:\n        temporary_path: Path | None = None\n        try:\n            with tempfile.NamedTemporaryFile(\n                mode="w+b",\n                dir=output_dir,\n                prefix=".frozen-distribution-flows.",\n                suffix=".tmp",\n                delete=False,\n            ) as temporary:\n                temporary_path = Path(temporary.name)\n                flows.to_pickle(temporary)\n                temporary.flush()\n                os.fsync(temporary.fileno())\n            os.chmod(temporary_path, 0o644)\n            _verify_frozen_distribution_flows(\n                temporary_path, flows, expected_fingerprint\n            )\n            try:\n                # A hard-link publish is atomic and never replaces an existing\n                # policy-bound artifact. Concurrent equal writers are harmless.\n                os.link(temporary_path, artifact_path)\n            except FileExistsError:\n                try:\n                    _verify_frozen_distribution_flows(\n                        artifact_path, flows, expected_fingerprint\n                    )\n                except Exception as error:\n                    raise FileExistsError(\n                        "동시에 생성된 frozen distribution flow가 현재 입력과 "\n                        f"다릅니다: {artifact_path}"\n                    ) from error\n        finally:\n            if temporary_path is not None:\n                temporary_path.unlink(missing_ok=True)\n    _verify_frozen_distribution_flows(artifact_path, flows, expected_fingerprint)\n    artifact = {\n        "path": artifact_path.name,\n        "file_sha256": file_sha256(artifact_path),\n        "format": "pandas_pickle",\n    }\n    return artifact_path, artifact\n\n\ndef weighted_quantile(values: np.ndarray, weights: np.ndarray, q: float) -> float:\n    quantile = float(q)\n    if not np.isfinite(quantile) or not 0 <= quantile <= 1:\n        raise ValueError("weighted quantile q는 0 이상 1 이하여야 합니다.")\n    values = np.asarray(values, dtype=float)\n    weights = np.asarray(weights, dtype=float)\n    if values.ndim != 1 or weights.ndim != 1 or len(values) != len(weights):\n        raise ValueError("weighted quantile values/weights는 같은 길이여야 합니다.")\n    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)\n    if not valid.any():\n        return float("nan")\n    order = np.argsort(values[valid])\n    sorted_values = values[valid][order]\n    sorted_weights = weights[valid][order]\n    cumulative = np.cumsum(sorted_weights)\n    cutoff = quantile * cumulative[-1]\n    index = min(int(np.searchsorted(cumulative, cutoff, side="left")), len(order) - 1)\n    return float(sorted_values[index])\n\n\ndef stop_band(data: pd.DataFrame) -> pd.Series:\n    return pd.cut(\n        data["target_stop_gap"],\n        bins=STOP_BINS,\n        labels=STOP_LABELS,\n        include_lowest=True,\n    ).astype(str)\n\n\ndef scale_values(data: pd.DataFrame, kind: str) -> np.ndarray:\n    if kind == "stop_gap":\n        return np.ones(len(data), dtype=float)\n    route_scale = data["route_dist_scale"].to_numpy(dtype=float)\n    if kind == "route_scale":\n        return np.maximum(route_scale, 0.5)\n    if kind == "route_scale_sqrt":\n        return np.sqrt(np.maximum(route_scale, 0.5))\n    raise ValueError(f"알 수 없는 interval scale: {kind}")\n\n\ndef fit_interval_policy(\n    data: pd.DataFrame,\n    kind: str,\n    *,\n    calibration_quantile: float = TARGET_COVERAGE,\n) -> dict[str, Any]:\n    absolute_error = np.abs(\n        data["label_seats"].to_numpy(dtype=float)\n        - data["prediction"].to_numpy(dtype=float)\n    )\n    scale = scale_values(data, kind)\n    normalized_error = absolute_error / np.maximum(scale, 1e-6)\n    weights = event_weights(data)\n    bands = stop_band(data)\n    global_factor = weighted_quantile(\n        normalized_error, weights, calibration_quantile\n    )\n    factors: dict[str, float] = {}\n    for label in STOP_LABELS:\n        mask = bands.eq(label).to_numpy()\n        if data.loc[mask, "event_id"].nunique() < 100:\n            factors[label] = global_factor\n            continue\n        factors[label] = weighted_quantile(\n            normalized_error[mask], weights[mask], calibration_quantile\n        )\n    return {\n        "kind": kind,\n        "target_coverage": TARGET_COVERAGE,\n        "calibration_quantile": calibration_quantile,\n        "global_factor": global_factor,\n        "band_factors": factors,\n    }\n\n\ndef fit_lower_tail_refinement(\n    data: pd.DataFrame,\n    *,\n    calibration_quantile: float = LOWER_TAIL_COVERAGE,\n    shrinkage_event_count: float = LOWER_TAIL_SHRINKAGE_EVENT_COUNT,\n    scale_floor: float = LOWER_TAIL_SCALE_FLOOR,\n    status: str = "active",\n) -> dict[str, Any]:\n    """Fit a one-sided q95 lower bound from strict prior OOF residuals.\n\n    Predicted-seat bands are deliberately based only on information available\n    at inference time.  Sparse band quantiles are shrunk toward the global q95\n    by their unique-event count, preventing a handful of repeated snapshots\n    from dominating a safety-critical lower bound.\n    """\n    if data.empty:\n        raise ValueError("lower-tail refinement를 fit할 calibration 행이 없습니다.")\n    quantile = float(calibration_quantile)\n    if not np.isclose(quantile, LOWER_TAIL_COVERAGE):\n        raise ValueError("lower-tail refinement는 q95만 지원합니다.")\n    shrinkage = float(shrinkage_event_count)\n    if not np.isfinite(shrinkage) or shrinkage < 0:\n        raise ValueError("lower-tail shrinkage_event_count가 올바르지 않습니다.")\n    if status not in {"shadow", "active"}:\n        raise ValueError("lower-tail status는 shadow 또는 active여야 합니다.")\n\n    prediction = data["prediction"].to_numpy(dtype=float)\n    label = data["label_seats"].to_numpy(dtype=float)\n    if not np.isfinite(prediction).all() or not np.isfinite(label).all():\n        raise ValueError("lower-tail calibration prediction/label은 유한해야 합니다.")\n    scale = lower_tail_scale_values(data, scale_floor=scale_floor)\n    one_sided_score = (prediction - label) / scale\n    weights = event_weights(data)\n    global_factor = max(\n        weighted_quantile(one_sided_score, weights, quantile),\n        0.0,\n    )\n    if not np.isfinite(global_factor):\n        raise ValueError("lower-tail global q95를 계산할 수 없습니다.")\n    bands = lower_tail_prediction_band(prediction, index=data.index)\n    band_factors: dict[str, float] = {}\n    band_event_counts: dict[str, int] = {}\n    for label_name in LOWER_TAIL_PREDICTION_LABELS:\n        mask = bands.eq(label_name).to_numpy()\n        event_count = int(data.loc[mask, "event_id"].nunique())\n        band_event_counts[label_name] = event_count\n        if event_count == 0:\n            band_factors[label_name] = global_factor\n            continue\n        raw_factor = max(\n            weighted_quantile(\n                one_sided_score[mask],\n                weights[mask],\n                quantile,\n            ),\n            0.0,\n        )\n        weight = event_count / (event_count + shrinkage)\n        band_factors[label_name] = float(\n            weight * raw_factor + (1.0 - weight) * global_factor\n        )\n\n    calibration_dates = (\n        sorted(data["date"].dropna().astype(str).unique().tolist())\n        if "date" in data.columns\n        else []\n    )\n    return {\n        "kind": LOWER_TAIL_REFINEMENT_KIND,\n        "status": status,\n        "target_coverage": LOWER_TAIL_COVERAGE,\n        "calibration_quantile": quantile,\n        "scale_kind": "route_scale",\n        "scale_floor": float(scale_floor),\n        "shrinkage_event_count": shrinkage,\n        "global_factor": global_factor,\n        "band_factors": band_factors,\n        "band_event_counts": band_event_counts,\n        "fit_provenance": {\n            "calibration_dates": calibration_dates,\n            "calibration_rows": int(len(data)),\n            "calibration_events": int(data["event_id"].nunique()),\n            "score": (\n                "(prediction-label_seats)/max(route_dist_scale,"\n                f"{float(scale_floor):g})"\n            ),\n            "calibration_protocol": "caller_supplied_strict_prior_oof",\n        },\n    }\n\n\ndef apply_interval_policy(\n    data: pd.DataFrame,\n    policy: dict[str, Any],\n) -> pd.DataFrame:\n    return apply_common_interval_policy(\n        data,\n        data["prediction"].to_numpy(dtype=float),\n        policy,\n    )\n\n\ndef interval_metrics(\n    data: pd.DataFrame,\n    interval: pd.DataFrame,\n    *,\n    policy_name: str,\n    split: str,\n) -> dict[str, Any]:\n    y = data["label_seats"].to_numpy(dtype=float)\n    lower = interval["interval_lower"].to_numpy(dtype=float)\n    upper = interval["interval_upper"].to_numpy(dtype=float)\n    weights = event_weights(data)\n    covered = (y >= lower) & (y <= upper)\n    width = upper - lower\n    alpha = 1 - TARGET_COVERAGE\n    interval_score = width.copy()\n    below = y < lower\n    above = y > upper\n    interval_score[below] += 2 / alpha * (lower[below] - y[below])\n    interval_score[above] += 2 / alpha * (y[above] - upper[above])\n    low = y <= 5\n    result = {\n        "policy": policy_name,\n        "split": split,\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n        "coverage_90": float(np.average(covered, weights=weights)),\n        "lower_miss_rate": float(np.average(below, weights=weights)),\n        "upper_miss_rate": float(np.average(above, weights=weights)),\n        "mean_width": float(np.average(width, weights=weights)),\n        "weighted_interval_score": float(\n            np.average(interval_score, weights=weights)\n        ),\n    }\n    result.update(\n        {\n            "low_5_rows": int(low.sum()),\n            "low_5_events": int(data.loc[low, "event_id"].nunique()),\n            "low_5_coverage_90": (\n                float(np.average(covered[low], weights=weights[low]))\n                if low.any()\n                else float("nan")\n            ),\n            "low_5_mean_width": (\n                float(np.average(width[low], weights=weights[low]))\n                if low.any()\n                else float("nan")\n            ),\n            "low_5_weighted_interval_score": (\n                float(np.average(interval_score[low], weights=weights[low]))\n                if low.any()\n                else float("nan")\n            ),\n        }\n    )\n    return result\n\n\ndef probability_metrics(\n    data: pd.DataFrame,\n    probability: np.ndarray,\n    *,\n    model: str,\n    split: str,\n) -> dict[str, Any]:\n    y = data["label_seats"].le(5).astype(int).to_numpy()\n    weights = event_weights(data)\n    probability = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)\n    return {\n        "model": model,\n        "split": split,\n        "positive_events": int(\n            data.loc[data["label_seats"].le(5), "event_id"].nunique()\n        ),\n        "weighted_prevalence": float(np.average(y, weights=weights)),\n        "mean_probability": float(np.average(probability, weights=weights)),\n        "brier": float(np.average((y - probability) ** 2, weights=weights)),\n        "log_loss": float(\n            log_loss(y, probability, sample_weight=weights, labels=[0, 1])\n        ),\n        "average_precision": float(\n            average_precision_score(y, probability, sample_weight=weights)\n        ),\n        "roc_auc": float(roc_auc_score(y, probability, sample_weight=weights)),\n    }\n\n\ndef fit_brier_logit_slope(\n    data: pd.DataFrame,\n    raw_probability: np.ndarray,\n    *,\n    status: str = "shadow",\n) -> dict[str, Any]:\n    """Fit one positive logit slope using event-balanced Brier loss."""\n    if status not in {"shadow", "active"}:\n        raise ValueError("probability calibration status가 올바르지 않습니다.")\n    y = data["label_seats"].le(5).astype(float).to_numpy()\n    weights = event_weights(data)\n    raw = np.clip(\n        np.asarray(raw_probability, dtype=float), 1e-6, 1 - 1e-6\n    )\n\n    def objective(slope: float) -> float:\n        calibrated = apply_probability_calibration(\n            raw,\n            {"kind": "logit_slope", "slope": slope},\n        )\n        return float(np.average((y - calibrated) ** 2, weights=weights))\n\n    result = minimize_scalar(\n        objective,\n        method="bounded",\n        bounds=(0.1, 5.0),\n        options={"xatol": 1e-10},\n    )\n    if not result.success or not np.isfinite(result.x):\n        raise ValueError("Brier logit-slope calibration 최적화에 실패했습니다.")\n    return {\n        "kind": "logit_slope",\n        "slope": float(result.x),\n        "temperature": float(1.0 / result.x),\n        "fit_objective": "event_balanced_brier",\n        "target": "arrival_seats_le_5",\n        "status": status,\n    }\n\n\ndef trip_bootstrap_probability_improvement(\n    data: pd.DataFrame,\n    calibrated_probability: np.ndarray,\n    raw_probability: np.ndarray,\n    *,\n    metric: str,\n    seed: int,\n    repeats: int = 5_000,\n) -> dict[str, Any]:\n    """Paired trip-cluster bootstrap; positive delta favors calibration."""\n    y = data["label_seats"].le(5).astype(float).to_numpy()\n    calibrated = np.clip(\n        np.asarray(calibrated_probability, dtype=float), 1e-6, 1 - 1e-6\n    )\n    raw = np.clip(\n        np.asarray(raw_probability, dtype=float), 1e-6, 1 - 1e-6\n    )\n    if metric == "brier":\n        calibrated_loss = (y - calibrated) ** 2\n        raw_loss = (y - raw) ** 2\n    elif metric == "log_loss":\n        calibrated_loss = -(y * np.log(calibrated) + (1 - y) * np.log1p(-calibrated))\n        raw_loss = -(y * np.log(raw) + (1 - y) * np.log1p(-raw))\n    else:\n        raise ValueError(f"알 수 없는 probability bootstrap metric: {metric}")\n\n    scored = data[["event_id", "trip_id"]].copy()\n    scored["delta"] = raw_loss - calibrated_loss\n    per_event = scored.groupby("event_id", sort=False).agg(\n        trip_id=("trip_id", "first"),\n        delta=("delta", "mean"),\n    )\n    trip_groups = [\n        group["delta"].to_numpy(dtype=float)\n        for _, group in per_event.groupby("trip_id", sort=False)\n    ]\n    if not trip_groups:\n        raise ValueError("probability bootstrap할 trip이 없습니다.")\n    rng = np.random.default_rng(seed)\n    boot = np.empty(repeats, dtype=float)\n    for index in range(repeats):\n        positions = rng.integers(0, len(trip_groups), size=len(trip_groups))\n        boot[index] = np.concatenate(\n            [trip_groups[position] for position in positions]\n        ).mean()\n    lower, median, upper = np.quantile(boot, [0.025, 0.5, 0.975])\n    return {\n        "metric": f"raw {metric} minus calibrated {metric}",\n        "unit": "trip_id",\n        "trips": int(len(trip_groups)),\n        "events": int(len(per_event)),\n        "observed_delta": float(per_event["delta"].mean()),\n        "lower_95": float(lower),\n        "median": float(median),\n        "upper_95": float(upper),\n        "probability_calibrated_better": float((boot > 0).mean()),\n    }\n\n\ndef trip_bootstrap_lower_tail_wis_improvement(\n    data: pd.DataFrame,\n    refined_interval: pd.DataFrame,\n    *,\n    seed: int,\n    repeats: int = 5_000,\n) -> dict[str, Any]:\n    """Paired trip bootstrap of low-seat WIS; positive favors refinement."""\n    y = data["label_seats"].to_numpy(dtype=float)\n    low = y <= 5\n    if not low.any():\n        raise ValueError("lower-tail bootstrap할 label_seats<=5 행이 없습니다.")\n    upper = refined_interval["interval_upper"].to_numpy(dtype=float)\n    refined_lower = refined_interval["interval_lower"].to_numpy(dtype=float)\n    base_lower = refined_interval["interval_lower_unrefined"].to_numpy(\n        dtype=float\n    )\n    alpha = 1 - TARGET_COVERAGE\n\n    def score(lower: np.ndarray) -> np.ndarray:\n        result = upper - lower\n        below = y < lower\n        above = y > upper\n        result[below] += 2 / alpha * (lower[below] - y[below])\n        result[above] += 2 / alpha * (y[above] - upper[above])\n        return result\n\n    scored = data.loc[low, ["event_id", "trip_id"]].copy()\n    scored["delta"] = (score(base_lower) - score(refined_lower))[low]\n    per_event = scored.groupby("event_id", sort=False).agg(\n        trip_id=("trip_id", "first"),\n        delta=("delta", "mean"),\n    )\n    trip_groups = [\n        group["delta"].to_numpy(dtype=float)\n        for _, group in per_event.groupby("trip_id", sort=False)\n    ]\n    if not trip_groups:\n        raise ValueError("lower-tail bootstrap할 trip이 없습니다.")\n    rng = np.random.default_rng(seed)\n    boot = np.empty(repeats, dtype=float)\n    for index in range(repeats):\n        positions = rng.integers(0, len(trip_groups), size=len(trip_groups))\n        boot[index] = np.concatenate(\n            [trip_groups[position] for position in positions]\n        ).mean()\n    lower, median, upper_bound = np.quantile(boot, [0.025, 0.5, 0.975])\n    return {\n        "metric": "base low-seat WIS minus refined low-seat WIS",\n        "unit": "trip_id",\n        "trips": int(len(trip_groups)),\n        "events": int(len(per_event)),\n        "observed_delta": float(per_event["delta"].mean()),\n        "lower_95": float(lower),\n        "median": float(median),\n        "upper_95": float(upper_bound),\n        "refinement_better": float((boot > 0).mean()),\n    }\n\n\ndef route_input_frame(\n    predictions: pd.DataFrame,\n    snapshots: pd.DataFrame,\n) -> pd.DataFrame:\n    route_columns = [\n        "event_id",\n        "snapshot_time",\n        "snapshot_station_seq",\n        "station_seq_cat",\n        "target_state_cat",\n        "direction",\n        "rolling_minutes_per_stop_3",\n        "date",\n        "capacity",\n    ]\n    route_source = snapshots[route_columns].copy()\n    output = predictions.copy()\n    output["snapshot_time"] = pd.to_datetime(output["snapshot_time"])\n    output = output.merge(\n        route_source,\n        on=["event_id", "snapshot_time"],\n        how="left",\n        validate="one_to_one",\n        suffixes=("", "_cache"),\n    )\n    if output["snapshot_station_seq"].isna().any():\n        raise ValueError("OOF 예측과 snapshot cache key가 완전히 일치하지 않습니다.")\n    if "date_cache" in output.columns:\n        if not output["date"].astype(str).eq(output["date_cache"].astype(str)).all():\n            raise ValueError("OOF와 cache의 날짜가 일치하지 않습니다.")\n        output = output.drop(columns="date_cache")\n    if "capacity_cache" in output.columns:\n        if "capacity" in output.columns:\n            if not np.allclose(\n                output["capacity"].to_numpy(dtype=float),\n                output["capacity_cache"].to_numpy(dtype=float),\n            ):\n                raise ValueError("OOF와 cache의 정원이 일치하지 않습니다.")\n        else:\n            output["capacity"] = output["capacity_cache"]\n        output = output.drop(columns="capacity_cache")\n    return output\n\n\ndef add_route_features(data: pd.DataFrame, flows: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    values = route_distribution_values(flows, output)\n    for column in ROUTE_DISTRIBUTION_FEATURES:\n        output[column] = values[column].to_numpy(dtype=float)\n    return output\n\n\ndef sequential_policy_predictions(\n    oof: pd.DataFrame,\n    kind: str,\n    calibration_quantile: float,\n    *,\n    lower_tail_refinement: bool = False,\n) -> tuple[pd.DataFrame, list[dict[str, Any]], list[dict[str, Any]]]:\n    base_policy_name = f"{kind}_q{int(calibration_quantile * 1000):03d}"\n    policy_name = (\n        f"{base_policy_name}__lower_tail_q950_k200"\n        if lower_tail_refinement\n        else base_policy_name\n    )\n    outputs: list[pd.DataFrame] = []\n    lower_tail_calibration_pool: list[pd.DataFrame] = []\n    interval_rows: list[dict[str, Any]] = []\n    probability_rows: list[dict[str, Any]] = []\n    for position, date in enumerate(VALIDATION_DATES[1:], start=1):\n        calibration_dates = VALIDATION_DATES[:position]\n        calibration = oof.loc[oof["date"].isin(calibration_dates)].copy()\n        validation = oof.loc[oof["date"].eq(date)].copy()\n        base_policy = fit_interval_policy(\n            calibration,\n            kind,\n            calibration_quantile=calibration_quantile,\n        )\n        calibration_interval = apply_interval_policy(calibration, base_policy)\n        probability_calibration = fit_brier_logit_slope(\n            calibration,\n            calibration_interval["p_low_5_raw"].to_numpy(dtype=float),\n            status="shadow",\n        )\n        policy = {\n            **base_policy,\n            "probability_calibration": probability_calibration,\n        }\n        if lower_tail_refinement and lower_tail_calibration_pool:\n            policy["lower_tail_refinement"] = fit_lower_tail_refinement(\n                pd.concat(lower_tail_calibration_pool, ignore_index=True),\n                status="active",\n            )\n        interval = apply_interval_policy(validation, policy)\n        scored = validation.copy()\n        for column in interval.columns:\n            scored[column] = interval[column].to_numpy()\n        scored["policy"] = policy_name\n        scored["probability_calibration_slope"] = probability_calibration[\n            "slope"\n        ]\n        outputs.append(scored)\n        if lower_tail_refinement:\n            # Only already-scored sequential residuals may calibrate the next\n            # date. Aug-05 seeds the symmetric interval but is intentionally not\n            # a lower-tail calibration residual; therefore Aug-06 stays base.\n            lower_tail_calibration_pool.append(validation)\n        interval_rows.append(\n            interval_metrics(\n                validation, interval, policy_name=policy_name, split=date\n            )\n        )\n        probability_rows.append(\n            probability_metrics(\n                validation,\n                interval["p_low_5"].to_numpy(dtype=float),\n                model=policy_name,\n                split=date,\n            )\n        )\n        probability_rows.append(\n            probability_metrics(\n                validation,\n                interval["p_low_5_calibrated"].to_numpy(dtype=float),\n                model=f"{policy_name}__temp_brier_shadow",\n                split=date,\n            )\n        )\n    combined = pd.concat(outputs, ignore_index=True)\n    interval_rows.append(\n        interval_metrics(\n            combined,\n            combined[\n                ["interval_half_width", "interval_lower", "interval_upper"]\n            ],\n            policy_name=policy_name,\n            split="sequential_oof_2026-08-06_10",\n        )\n    )\n    probability_rows.append(\n        probability_metrics(\n            combined,\n            combined["p_low_5"].to_numpy(dtype=float),\n            model=policy_name,\n            split="sequential_oof_2026-08-06_10",\n        )\n    )\n    probability_rows.append(\n        probability_metrics(\n            combined,\n            combined["p_low_5_calibrated"].to_numpy(dtype=float),\n            model=f"{policy_name}__temp_brier_shadow",\n            split="sequential_oof_2026-08-06_10",\n        )\n    )\n    return combined, interval_rows, probability_rows\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="Strict-forward route distribution and interval search"\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),\n    )\n    parser.add_argument(\n        "--model-results",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/route_distribution_results"),\n    )\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    point_model_summary_path = args.model_results / "summary.json"\n    point_model_summary = json.loads(\n        point_model_summary_path.read_text(encoding="utf-8")\n    )\n    selected_point_model = str(\n        point_model_summary["stage_winners"]["selected_final"]\n    )\n    point_model_source_sha256 = str(\n        point_model_summary["protocol"]["experiment_source_sha256"]\n    )\n    snapshots = pd.read_pickle(args.snapshot_cache)\n    flows = pd.read_pickle(args.flow_cache)\n    frozen_flow_path, frozen_flow_artifact = freeze_distribution_flows(\n        flows, args.output_dir\n    )\n    selected_oof_path = args.model_results / "selected_oof_predictions.csv"\n    final_test_predictions_path = (\n        args.model_results / "final_test_predictions.csv"\n    )\n    selected_oof = pd.read_csv(selected_oof_path)\n    oof_model_names = set(selected_oof["candidate"].astype(str).unique())\n    if oof_model_names != {selected_point_model}:\n        raise ValueError(\n            "selected OOF prediction과 point-model summary의 모델명이 "\n            f"다릅니다: oof={sorted(oof_model_names)}, "\n            f"summary={selected_point_model}"\n        )\n    policy_provenance = {\n        "schema_version": POLICY_PROVENANCE_SCHEMA_VERSION,\n        "point_model": {\n            "selected_name": selected_point_model,\n            "summary_sha256": file_sha256(point_model_summary_path),\n            "source_sha256": point_model_source_sha256,\n            "input_provenance_sha256": point_model_input_provenance_sha256(\n                point_model_summary\n            ),\n        },\n        "distribution": {\n            "flow_data_sha256": distribution_flow_fingerprint(flows),\n            "flow_rows": int(len(flows)),\n            "flow_artifact": frozen_flow_artifact,\n            "source_hashes": {\n                "route_distribution_search.py": file_sha256(Path(__file__)),\n                "route_distribution.py": file_sha256(\n                    Path(__file__).with_name("route_distribution.py")\n                ),\n            },\n            "input_hashes": {\n                "snapshot_cache": file_sha256(args.snapshot_cache),\n                "flow_cache": file_sha256(args.flow_cache),\n                "selected_oof_predictions": file_sha256(selected_oof_path),\n                "final_test_predictions": file_sha256(\n                    final_test_predictions_path\n                ),\n            },\n        },\n    }\n    oof = route_input_frame(selected_oof, snapshots)\n    oof = add_route_features(oof, flows)\n\n    raw_probability_rows = [\n        probability_metrics(\n            oof,\n            oof["route_dist_p_low_5"].to_numpy(dtype=float),\n            model="raw_route_distribution",\n            split="all_oof_2026-08-05_10",\n        )\n    ]\n    interval_rows: list[dict[str, Any]] = []\n    probability_rows: list[dict[str, Any]] = raw_probability_rows.copy()\n    sequential_outputs: dict[str, pd.DataFrame] = {}\n    policy_specs = [\n        (kind, quantile)\n        for kind in ("stop_gap", "route_scale_sqrt", "route_scale")\n        for quantile in (0.9, 0.925, 0.95)\n    ]\n    for kind, quantile in policy_specs:\n        policy_name = f"{kind}_q{int(quantile * 1000):03d}"\n        output, kind_intervals, kind_probabilities = sequential_policy_predictions(\n            oof, kind, quantile\n        )\n        sequential_outputs[policy_name] = output\n        interval_rows.extend(kind_intervals)\n        probability_rows.extend(kind_probabilities)\n\n    aggregate_intervals = pd.DataFrame(interval_rows)\n    daily_stability = (\n        aggregate_intervals.loc[\n            aggregate_intervals["split"].isin(VALIDATION_DATES[1:])\n        ]\n        .groupby("policy", as_index=False)["coverage_90"]\n        .agg(min_daily_coverage="min", max_daily_coverage="max")\n    )\n    aggregate_selection = aggregate_intervals.loc[\n        aggregate_intervals["split"].eq("sequential_oof_2026-08-06_10")\n    ].merge(daily_stability, on="policy", how="left")\n    selection = aggregate_selection.loc[\n        aggregate_selection["coverage_90"].between(0.88, 0.94)\n        & aggregate_selection["min_daily_coverage"].ge(0.85)\n        & aggregate_selection["max_daily_coverage"].le(0.96)\n    ].sort_values(["weighted_interval_score", "mean_width"])\n    if selection.empty:\n        raise ValueError("88~94% coverage를 만족하는 interval policy가 없습니다.")\n    selected_name = str(selection.iloc[0]["policy"])\n    selected_spec = next(\n        (kind, quantile)\n        for kind, quantile in policy_specs\n        if f"{kind}_q{int(quantile * 1000):03d}" == selected_name\n    )\n    selected_kind, selected_quantile = selected_spec\n\n    # Evaluate the safety refinement only after the symmetric base policy is\n    # selected.  Each validation date receives factors fit from strictly prior\n    # OOF dates; the locked Aug-11 confirmation cannot influence activation.\n    (\n        sequential_refined,\n        refined_interval_rows,\n        refined_probability_rows,\n    ) = sequential_policy_predictions(\n        oof,\n        selected_kind,\n        selected_quantile,\n        lower_tail_refinement=True,\n    )\n    sequential_base = sequential_outputs[selected_name]\n    if not sequential_base["event_id"].astype(str).equals(\n        sequential_refined["event_id"].astype(str)\n    ):\n        raise ValueError("base/refined sequential OOF 행 정렬이 다릅니다.")\n    invariant_columns = (\n        "prediction",\n        "interval_half_width",\n        "interval_upper",\n        "p_low_5",\n        "p_low_5_raw",\n        "p_low_5_calibrated",\n        "p_full",\n    )\n    lower_tail_invariance = {\n        column: bool(\n            np.array_equal(\n                sequential_base[column].to_numpy(),\n                sequential_refined[column].to_numpy(),\n                equal_nan=True,\n            )\n        )\n        for column in invariant_columns\n    }\n    if not all(lower_tail_invariance.values()):\n        changed = [\n            column for column, unchanged in lower_tail_invariance.items()\n            if not unchanged\n        ]\n        raise ValueError(\n            "lower-tail refinement가 보호된 출력을 변경했습니다: "\n            f"{changed}"\n        )\n    base_development_metrics = interval_metrics(\n        sequential_base,\n        sequential_base,\n        policy_name=selected_name,\n        split="sequential_oof_2026-08-06_10",\n    )\n    refined_name = f"{selected_name}__lower_tail_q950_k200"\n    refined_development_metrics = interval_metrics(\n        sequential_refined,\n        sequential_refined,\n        policy_name=refined_name,\n        split="sequential_oof_2026-08-06_10",\n    )\n    lower_tail_bootstrap = trip_bootstrap_lower_tail_wis_improvement(\n        sequential_refined,\n        sequential_refined,\n        seed=921,\n    )\n    lower_tail_activation_checks = {\n        "low_5_wis_improved": bool(\n            refined_development_metrics["low_5_weighted_interval_score"]\n            < base_development_metrics["low_5_weighted_interval_score"]\n        ),\n        "overall_wis_not_worse": bool(\n            refined_development_metrics["weighted_interval_score"]\n            <= base_development_metrics["weighted_interval_score"]\n        ),\n        "trip_bootstrap_low_5_wis_lower_95_positive": bool(\n            lower_tail_bootstrap["lower_95"] > 0\n        ),\n        "protected_outputs_unchanged": all(lower_tail_invariance.values()),\n    }\n    lower_tail_status = (\n        "active"\n        if all(lower_tail_activation_checks.values())\n        else "shadow"\n    )\n    interval_rows.extend(refined_interval_rows)\n    probability_rows.extend(refined_probability_rows)\n    aggregate_intervals = pd.DataFrame(interval_rows)\n\n    selected_policy = fit_interval_policy(\n        oof,\n        selected_kind,\n        calibration_quantile=selected_quantile,\n    )\n    selected_calibration_interval = apply_interval_policy(oof, selected_policy)\n    selected_probability_calibration = fit_brier_logit_slope(\n        oof,\n        selected_calibration_interval["p_low_5_raw"].to_numpy(dtype=float),\n        status="shadow",\n    )\n    selected_policy["probability_calibration"] = (\n        selected_probability_calibration\n    )\n    selected_lower_tail_refinement = fit_lower_tail_refinement(\n        sequential_base,\n        status=lower_tail_status,\n    )\n    selected_policy["lower_tail_refinement"] = (\n        selected_lower_tail_refinement\n    )\n    policy_provenance["lower_tail_refinement"] = {\n        "selection_protocol": (\n            "nested_strict_prior_sequential_oof_by_validation_date"\n        ),\n        "symmetric_interval_seed_date_excluded": VALIDATION_DATES[0],\n        "initial_validation_date": VALIDATION_DATES[1],\n        "initial_date_behavior": (\n            "base_interval_unchanged_no_prior_sequential_residual"\n        ),\n        "refined_evaluation_dates": list(VALIDATION_DATES[2:]),\n        "future_policy_calibration_dates": list(VALIDATION_DATES[1:]),\n        "locked_confirmation_excluded_from_selection": "2026-08-11",\n        "activation_checks": lower_tail_activation_checks,\n    }\n    selected_policy["provenance"] = policy_provenance\n    selected_policy = validate_interval_policy(selected_policy)\n\n    confirmation_raw = pd.read_csv(\n        final_test_predictions_path\n    ).rename(columns={"selected_prediction": "prediction"})\n    confirmation = route_input_frame(confirmation_raw, snapshots)\n    confirmation = add_route_features(confirmation, flows)\n    confirmation_rows: list[dict[str, Any]] = []\n    confirmation_probability_rows: list[dict[str, Any]] = []\n    for kind, quantile in policy_specs:\n        name = f"{kind}_q{int(quantile * 1000):03d}"\n        policy = fit_interval_policy(\n            oof, kind, calibration_quantile=quantile\n        )\n        oof_candidate_interval = apply_interval_policy(oof, policy)\n        candidate_probability_calibration = fit_brier_logit_slope(\n            oof,\n            oof_candidate_interval["p_low_5_raw"].to_numpy(dtype=float),\n            status="shadow",\n        )\n        policy["probability_calibration"] = candidate_probability_calibration\n        candidate_interval = apply_interval_policy(confirmation, policy)\n        confirmation_rows.append(\n            interval_metrics(\n                confirmation,\n                candidate_interval,\n                policy_name=name,\n                split="locked_confirmation_2026-08-11_partial",\n            )\n        )\n        confirmation_probability_rows.append(\n            probability_metrics(\n                confirmation,\n                candidate_interval["p_low_5_calibrated"].to_numpy(dtype=float),\n                model=f"{name}__temp_brier_shadow",\n                split="locked_confirmation_2026-08-11_partial",\n            )\n        )\n        confirmation_probability_rows.append(\n            probability_metrics(\n                confirmation,\n                candidate_interval["p_low_5"].to_numpy(dtype=float),\n                model=name,\n                split="locked_confirmation_2026-08-11_partial",\n            )\n        )\n\n    confirmation_base_policy = dict(selected_policy)\n    confirmation_base_policy.pop("lower_tail_refinement", None)\n    confirmation_base_interval = apply_interval_policy(\n        confirmation,\n        confirmation_base_policy,\n    )\n    confirmation_interval = apply_interval_policy(confirmation, selected_policy)\n    confirmation_interval_invariant_columns = tuple(\n        column for column in invariant_columns if column != "prediction"\n    )\n    confirmation_lower_tail_invariance = {\n        column: bool(\n            np.array_equal(\n                confirmation_base_interval[column].to_numpy(),\n                confirmation_interval[column].to_numpy(),\n                equal_nan=True,\n            )\n        )\n        for column in confirmation_interval_invariant_columns\n    }\n    confirmation_lower_tail_invariance["prediction"] = True\n    if not all(confirmation_lower_tail_invariance.values()):\n        raise ValueError(\n            "confirmation lower-tail refinement가 보호된 출력을 변경했습니다."\n        )\n    for column in confirmation_interval.columns:\n        confirmation[column] = confirmation_interval[column].to_numpy()\n    confirmation["policy"] = refined_name\n\n    confirmation_base_interval_metrics = interval_metrics(\n        confirmation,\n        confirmation_base_interval,\n        policy_name=selected_name,\n        split="locked_confirmation_2026-08-11_partial",\n    )\n    confirmation_interval_metrics = interval_metrics(\n        confirmation,\n        confirmation_interval,\n        policy_name=refined_name,\n        split="locked_confirmation_2026-08-11_partial",\n    )\n    confirmation_rows.append(confirmation_interval_metrics)\n    confirmation_probability_metrics = probability_metrics(\n        confirmation,\n        confirmation_interval["p_low_5"].to_numpy(dtype=float),\n        model=refined_name,\n        split="locked_confirmation_2026-08-11_partial",\n    )\n    confirmation_probability_calibrated_shadow_metrics = probability_metrics(\n        confirmation,\n        confirmation_interval["p_low_5_calibrated"].to_numpy(dtype=float),\n        model=f"{selected_name}__temp_brier_shadow",\n        split="locked_confirmation_2026-08-11_partial",\n    )\n\n    sequential_selected = (\n        sequential_refined\n        if lower_tail_status == "active"\n        else sequential_base\n    )\n    sequential_probability_calibration_bootstrap = {\n        metric: trip_bootstrap_probability_improvement(\n            sequential_selected,\n            sequential_selected["p_low_5_calibrated"].to_numpy(dtype=float),\n            sequential_selected["p_low_5_raw"].to_numpy(dtype=float),\n            metric=metric,\n            seed=901 + position,\n        )\n        for position, metric in enumerate(("brier", "log_loss"))\n    }\n    confirmation_probability_calibration_bootstrap = {\n        metric: trip_bootstrap_probability_improvement(\n            confirmation,\n            confirmation["p_low_5_calibrated"].to_numpy(dtype=float),\n            confirmation["p_low_5_raw"].to_numpy(dtype=float),\n            metric=metric,\n            seed=911 + position,\n        )\n        for position, metric in enumerate(("brier", "log_loss"))\n    }\n    keep = [\n        "event_id",\n        "date",\n        "trip_id",\n        "snapshot_time",\n        "label_seats",\n        "prediction",\n        "target_stop_gap",\n        "route_dist_scale",\n        "route_dist_cell_prior_share",\n        "route_dist_cross_bin_share",\n        "interval_lower_unrefined",\n        "interval_lower",\n        "interval_upper",\n        "lower_tail_refinement_factor",\n        "lower_tail_refinement_candidate",\n        "lower_tail_refinement_applied",\n        "p_low_5",\n        "p_low_5_raw",\n        "p_low_5_calibrated",\n        "probability_calibration_slope",\n        "policy",\n    ]\n    confirmation["probability_calibration_slope"] = (\n        selected_probability_calibration["slope"]\n    )\n    sequential_selected["lower_tail_refinement_status"] = lower_tail_status\n    confirmation["lower_tail_refinement_status"] = lower_tail_status\n    keep.append("lower_tail_refinement_status")\n    summary = {\n        "hypothesis": (\n            "route-flow variance predicts error magnitude, so it should scale a "\n            "past-only empirically calibrated interval around the selected point "\n            "model rather than move the point prediction"\n        ),\n        "selection_dates": list(VALIDATION_DATES),\n        "sequential_evaluation_dates": list(VALIDATION_DATES[1:]),\n        "selection_rule": (\n            "among policies with 88-94% aggregate coverage, 85-96% coverage on "\n            "every sequential date, minimize weighted interval score then width"\n        ),\n        "lower_tail_refinement": {\n            "hypothesis": (\n                "symmetric route-scale intervals miss predominantly below their "\n                "lower bound when predicted seats are low; a strict-prior, "\n                "predicted-seat-band one-sided q95 extension should improve "\n                "low-seat safety without changing point predictions, upper "\n                "bounds, or probabilities"\n            ),\n            "selection_data": "strict sequential OOF 2026-08-06..10",\n            "locked_confirmation_used_for_selection": False,\n            "status": lower_tail_status,\n            "activation_checks": lower_tail_activation_checks,\n            "base_development_metrics": base_development_metrics,\n            "refined_development_metrics": refined_development_metrics,\n            "trip_bootstrap_low_5_wis": lower_tail_bootstrap,\n            "development_invariance": lower_tail_invariance,\n            "confirmation_base_metrics": confirmation_base_interval_metrics,\n            "confirmation_refined_metrics": confirmation_interval_metrics,\n            "confirmation_invariance": confirmation_lower_tail_invariance,\n        },\n        "selected_policy": selected_policy,\n        "policy_provenance": policy_provenance,\n        "frozen_distribution_flow_artifact": {\n            **frozen_flow_artifact,\n            "resolved_path": str(frozen_flow_path.resolve()),\n            "rows": int(len(flows)),\n            "content_fingerprint": distribution_flow_fingerprint(flows),\n        },\n        "selected_probability_calibration": selected_probability_calibration,\n        "interval_metrics": interval_rows,\n        "probability_metrics": probability_rows,\n        "confirmation_interval_metrics": confirmation_interval_metrics,\n        "confirmation_probability_metrics": confirmation_probability_metrics,\n        "confirmation_probability_calibrated_shadow_metrics": (\n            confirmation_probability_calibrated_shadow_metrics\n        ),\n        "sequential_probability_calibration_bootstrap": (\n            sequential_probability_calibration_bootstrap\n        ),\n        "confirmation_probability_calibration_bootstrap": (\n            confirmation_probability_calibration_bootstrap\n        ),\n        "all_confirmation_interval_metrics": confirmation_rows,\n        "all_confirmation_probability_metrics": confirmation_probability_rows,\n        "confirmation_caveat": (\n            "2026-08-11 is partial-day and label counts were previously inspected; "\n            "it was not used for policy selection but is not pristine"\n        ),\n    }\n    _atomic_csv(\n        sequential_selected[keep],\n        args.output_dir / "sequential_oof_intervals.csv",\n    )\n    _atomic_csv(\n        confirmation[keep],\n        args.output_dir / "confirmation_intervals.csv",\n    )\n    _atomic_csv(\n        aggregate_intervals,\n        args.output_dir / "interval_policy_metrics.csv",\n    )\n    _atomic_csv(\n        pd.DataFrame(probability_rows),\n        args.output_dir / "low_5_probability_metrics.csv",\n    )\n    _atomic_csv(\n        pd.DataFrame(confirmation_rows),\n        args.output_dir / "confirmation_policy_metrics.csv",\n    )\n    _atomic_csv(\n        pd.DataFrame(confirmation_probability_rows),\n        args.output_dir / "confirmation_low_5_probability_metrics.csv",\n    )\n    _atomic_text(\n        args.output_dir / "summary.json",\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n    )\n    # The service policy is the bundle commit marker and is published last.\n    _atomic_text(\n        args.output_dir / "interval_policy.json",\n        json.dumps(json_ready(selected_policy), ensure_ascii=False, indent=2),\n    )\n    print(json.dumps(json_ready(summary), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/feature_importance_analysis.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom collections.abc import Iterable\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nfrom all_prearrival_seat_regression import clip_seats\nfrom hypothesis_model_search import decode_target, ensemble_prediction, event_summary\nfrom route_distribution import file_sha256\nfrom seat_service_model import SeatServiceModel, prepare_service_features\n\n\n# Every deployed point-model input belongs to exactly one semantic group.  A\n# shared donor permutation is used for all columns in a group so, for example,\n# sin/cos time or the three current-seat threshold columns stay internally\n# coherent.  Cross-group correlation is deliberately broken: that is the\n# dependence being measured.\nFEATURE_GROUPS: dict[str, tuple[str, ...]] = {\n    "current_seat_state": (\n        "snapshot_remaining_seats",\n        "target_load_ratio",\n        "currently_low_5",\n        "currently_low_10",\n        "observed_ceiling_load_ratio",\n    ),\n    "recent_seat_trajectory": (\n        "seat_delta_previous_stop",\n        "seat_change_per_stop",\n        "rolling_seat_change_per_stop_3",\n        "projected_arrival_seats",\n    ),\n    "distance_and_eta": (\n        "target_stop_gap",\n        "minutes_since_previous_stop",\n        "rolling_minutes_per_stop_3",\n        "estimated_minutes_to_arrival",\n        "seats_per_remaining_stop",\n        "load_gap_interaction",\n        "observed_ceiling_load_gap",\n    ),\n    "target_location": (\n        "station_seq_cat",\n        "route_progress",\n        "x",\n        "y",\n    ),\n    "snapshot_location": (\n        "snapshot_station_seq_cat",\n        "snapshot_route_progress",\n    ),\n    "calendar_time": (\n        "snapshot_time_sin",\n        "snapshot_time_cos",\n        "snapshot_time_bin_30",\n        "snapshot_day_of_week",\n    ),\n    "vehicle_capacity": (\n        "snapshot_capacity",\n        "observed_ceiling_capacity",\n        "snapshot_low_plate_cat",\n    ),\n    "direction": ("direction",),\n    "snapshot_state_code": ("target_state_cat",),\n}\n\nCONDITIONAL_STRATA = (\n    "date",\n    "direction",\n    "snapshot_low_plate_cat",\n    "target_stop_gap",\n)\n\n\ndef sha256_bytes(payload: bytes) -> str:\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef deployed_feature_columns(service: SeatServiceModel) -> list[str]:\n    columns_by_component = {\n        name: tuple(component.metadata["feature_columns"])\n        for name, component in service.components.items()\n    }\n    unique = set(columns_by_component.values())\n    if len(unique) != 1:\n        raise ValueError(\n            "선택 ensemble component의 feature schema가 서로 다릅니다: "\n            f"{columns_by_component}"\n        )\n    return list(next(iter(unique)))\n\n\ndef validate_feature_partition(feature_columns: Iterable[str]) -> None:\n    deployed = list(feature_columns)\n    flattened = [column for columns in FEATURE_GROUPS.values() for column in columns]\n    duplicates = sorted(\n        {column for column in flattened if flattened.count(column) > 1}\n    )\n    if duplicates:\n        raise ValueError(f"feature group에 중복 열이 있습니다: {duplicates}")\n    missing = sorted(set(deployed) - set(flattened))\n    unexpected = sorted(set(flattened) - set(deployed))\n    if missing or unexpected:\n        raise ValueError(\n            "feature group이 deployed schema를 정확히 분할하지 않습니다: "\n            f"missing={missing}, unexpected={unexpected}"\n        )\n\n\ndef donor_indices(\n    data: pd.DataFrame,\n    *,\n    rng: np.random.Generator,\n    scheme: str,\n) -> np.ndarray:\n    """Return one donor row per target row for a permutation scheme."""\n\n    if scheme == "global":\n        return rng.permutation(len(data))\n    if scheme != "conditional":\n        raise ValueError(f"알 수 없는 permutation scheme입니다: {scheme}")\n    missing = sorted(set(CONDITIONAL_STRATA) - set(data.columns))\n    if missing:\n        raise ValueError(f"conditional permutation strata가 누락되었습니다: {missing}")\n\n    donors = np.arange(len(data), dtype=int)\n    # observed=True avoids materializing unused categorical combinations.\n    groups = data.groupby(list(CONDITIONAL_STRATA), sort=False, observed=True).indices\n    for indices in groups.values():\n        indices = np.asarray(indices, dtype=int)\n        if len(indices) > 1:\n            donors[indices] = rng.permutation(indices)\n    return donors\n\n\ndef permuted_frame(\n    data: pd.DataFrame,\n    columns: Iterable[str],\n    donors: np.ndarray,\n) -> pd.DataFrame:\n    output = data.copy()\n    for column in columns:\n        output[column] = data[column].iloc[donors].array\n    return output\n\n\ndef predict_prepared_features(\n    service: SeatServiceModel,\n    prepared: pd.DataFrame,\n    *,\n    decode_frame: pd.DataFrame | None = None,\n) -> np.ndarray:\n    """Predict from an already prepared frame so derived inputs are perturbable."""\n\n    decode_frame = prepared if decode_frame is None else decode_frame\n    if len(decode_frame) != len(prepared):\n        raise ValueError("decode frame과 prepared feature frame의 행 수가 다릅니다.")\n    component_predictions: dict[str, np.ndarray] = {}\n    for name, component in service.components.items():\n        if component.model is None:\n            raise ValueError(f"permutation importance는 fitted component만 지원합니다: {name}")\n        metadata = component.metadata\n        candidate = metadata["candidate"]\n        columns = list(metadata["feature_columns"])\n        raw = component.model.predict(prepared[columns])\n        decoded = decode_target(raw, decode_frame, str(candidate["target_kind"]))\n        bias = float(metadata.get("bias_correction", 0.0))\n        component_predictions[name] = clip_seats(\n            decoded + bias, decode_frame["capacity"]\n        )\n    if service.ensemble is None:\n        return component_predictions[service.selected_name]\n    return ensemble_prediction(service.ensemble, component_predictions, decode_frame)\n\n\ndef importance_metrics(\n    data: pd.DataFrame,\n    baseline_prediction: np.ndarray,\n    permuted_prediction: np.ndarray,\n) -> dict[str, float]:\n    baseline = event_summary(data, baseline_prediction)\n    permuted = event_summary(data, permuted_prediction)\n    result: dict[str, float] = {\n        "overall_mae_increase": (\n            permuted["event_balanced_mae"] - baseline["event_balanced_mae"]\n        ),\n        "trip_mae_increase": (\n            permuted["trip_balanced_mae"] - baseline["trip_balanced_mae"]\n        ),\n        "within_3_decrease": (\n            baseline["event_balanced_within_3"]\n            - permuted["event_balanced_within_3"]\n        ),\n        "prediction_mean_abs_change": float(\n            np.mean(np.abs(permuted_prediction - baseline_prediction))\n        ),\n    }\n    for output_name, metric_name in (\n        ("low_0_5_mae_increase", "low_0_5_mae"),\n        ("emerging_low_mae_increase", "emerging_low_mae"),\n        ("near_1_2_mae_increase", "near_1_2_stop_mae"),\n        ("far_6_plus_mae_increase", "far_6_plus_stop_mae"),\n    ):\n        result[output_name] = float(permuted[metric_name] - baseline[metric_name])\n    return result\n\n\ndef run_permutations(\n    data: pd.DataFrame,\n    service: SeatServiceModel,\n    baseline_prediction: np.ndarray,\n    feature_sets: dict[str, tuple[str, ...]],\n    *,\n    schemes: tuple[str, ...],\n    repeats: int,\n    seed: int,\n    level: str,\n    scope: str,\n) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for scheme_index, scheme in enumerate(schemes):\n        for feature_index, (name, columns) in enumerate(feature_sets.items()):\n            for repeat in range(repeats):\n                rng = np.random.default_rng(\n                    seed\n                    + scheme_index * 1_000_000\n                    + feature_index * 10_000\n                    + repeat\n                )\n                donors = donor_indices(data, rng=rng, scheme=scheme)\n                shuffled = permuted_frame(data, columns, donors)\n                if scope == "end_to_end":\n                    prediction = predict_prepared_features(service, shuffled)\n                elif scope == "model_input_only":\n                    prediction = predict_prepared_features(\n                        service, shuffled, decode_frame=data\n                    )\n                else:\n                    raise ValueError(f"알 수 없는 importance scope입니다: {scope}")\n                row = {\n                    "level": level,\n                    "scope": scope,\n                    "scheme": scheme,\n                    "feature": name,\n                    "columns": json.dumps(list(columns), ensure_ascii=False),\n                    "repeat": repeat,\n                    **importance_metrics(data, baseline_prediction, prediction),\n                }\n                rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef summarize_importance(raw: pd.DataFrame) -> pd.DataFrame:\n    metric_columns = [\n        column\n        for column in raw.columns\n        if column.endswith(("_increase", "_decrease", "_change"))\n    ]\n    rows: list[dict[str, Any]] = []\n    for keys, group in raw.groupby(\n        ["level", "scope", "scheme", "feature", "columns"], sort=False\n    ):\n        row = dict(\n            zip(("level", "scope", "scheme", "feature", "columns"), keys)\n        )\n        row["repeats"] = int(len(group))\n        for metric in metric_columns:\n            values = group[metric].to_numpy(dtype=float)\n            row[f"{metric}_mean"] = float(np.mean(values))\n            row[f"{metric}_p05"] = float(np.quantile(values, 0.05))\n            row[f"{metric}_p95"] = float(np.quantile(values, 0.95))\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values(\n        ["level", "scope", "scheme", "overall_mae_increase_mean"],\n        ascending=[True, True, True, False],\n    )\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="공식 잔여좌석 ensemble의 grouped/conditional permutation importance"\n    )\n    parser.add_argument(\n        "--results-dir", type=Path, default=Path("analysis/model_search_results")\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/feature_importance_results"),\n    )\n    parser.add_argument("--date", default=None)\n    parser.add_argument("--group-repeats", type=int, default=20)\n    parser.add_argument("--feature-repeats", type=int, default=10)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.group_repeats <= 0 or args.feature_repeats <= 0:\n        raise ValueError("permutation repeats는 1 이상이어야 합니다.")\n\n    summary_path = args.results_dir / "summary.json"\n    summary_payload = summary_path.read_bytes()\n    model_summary = json.loads(summary_payload.decode("utf-8"))\n    locked_date = str(\n        model_summary["protocol"]["locked_confirmation_test"]["date"]\n    )\n    evaluation_date = str(args.date or locked_date)\n    snapshots = pd.read_pickle(args.snapshot_cache)\n    target = snapshots.loc[snapshots["date"].astype(str).eq(evaluation_date)].copy()\n    if target.empty:\n        raise ValueError(f"importance 평가 날짜가 cache에 없습니다: {evaluation_date}")\n\n    service = SeatServiceModel.load(args.results_dir)\n    feature_columns = deployed_feature_columns(service)\n    validate_feature_partition(feature_columns)\n    prepared, prepared_columns = prepare_service_features(\n        target, "capacity_44_70"\n    )\n    if list(prepared_columns) != feature_columns:\n        raise ValueError("prepared feature schema가 deployment metadata와 다릅니다.")\n    baseline_prediction = service.predict(target)\n    prepared_baseline_prediction = predict_prepared_features(service, prepared)\n    reconstruction_max_abs_diff = float(\n        np.max(np.abs(prepared_baseline_prediction - baseline_prediction))\n    )\n    if reconstruction_max_abs_diff > 1e-12:\n        raise ValueError(\n            "prepared feature prediction이 service prediction을 재현하지 못합니다: "\n            f"max_abs_diff={reconstruction_max_abs_diff}"\n        )\n    baseline_metrics = event_summary(target, baseline_prediction)\n\n    group_frames = []\n    for scope_index, scope in enumerate(("end_to_end", "model_input_only")):\n        group_frames.append(\n            run_permutations(\n                prepared,\n                service,\n                baseline_prediction,\n                FEATURE_GROUPS,\n                schemes=("global", "conditional"),\n                repeats=args.group_repeats,\n                seed=args.seed + scope_index * 100_000_000,\n                level="group",\n                scope=scope,\n            )\n        )\n    group_raw = pd.concat(group_frames, ignore_index=True)\n    individual_sets = {column: (column,) for column in feature_columns}\n    individual_raw = run_permutations(\n        prepared,\n        service,\n        baseline_prediction,\n        individual_sets,\n        schemes=("global", "conditional"),\n        repeats=args.feature_repeats,\n        seed=args.seed + 10_000_000,\n        level="feature",\n        scope="model_input_only",\n    )\n    raw = pd.concat([group_raw, individual_raw], ignore_index=True)\n    importance = summarize_importance(raw)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    raw_path = args.output_dir / "permutation_importance_repeats.csv"\n    importance_path = args.output_dir / "permutation_importance_summary.csv"\n    raw.to_csv(raw_path, index=False)\n    importance.to_csv(importance_path, index=False)\n\n    group_summary = importance.loc[importance["level"].eq("group")]\n    top_global = group_summary.loc[\n        group_summary["scheme"].eq("global")\n        & group_summary["scope"].eq("model_input_only")\n    ].head(9)\n    top_conditional = group_summary.loc[\n        group_summary["scheme"].eq("conditional")\n        & group_summary["scope"].eq("model_input_only")\n    ].head(9)\n    payload = {\n        "analysis_kind": "grouped and individual permutation importance",\n        "interpretation": {\n            "positive_mae_increase": (\n                "permuting the feature worsens MAE, so the frozen model relies on it"\n            ),\n            "negative_mae_increase": (\n                "permuting the feature improves MAE; this can indicate harmful reliance, "\n                "correlated-feature substitution, or permutation distribution shift"\n            ),\n            "global": "total model dependence, including proxy and interaction effects",\n            "conditional": (\n                "incremental dependence after preserving date, direction, vehicle type, "\n                "and exact target-stop gap"\n            ),\n            "end_to_end": (\n                "the permuted frame is also used by delta decoding and output support; "\n                "this measures full service-pipeline dependence"\n            ),\n            "model_input_only": (\n                "only estimator inputs are permuted while delta decoding and output "\n                "support use the original frame; this isolates learned-model dependence"\n            ),\n            "caveat": (\n                "2026-08-11 is a partial, previously inspected confirmation date; "\n                "importance is diagnostic and must not select a new model"\n            ),\n        },\n        "model": {\n            "selected_name": service.selected_name,\n            "summary_sha256": sha256_bytes(summary_payload),\n            "source_sha256": model_summary["protocol"]["experiment_source_sha256"],\n        },\n        "input": {\n            "snapshot_cache": str(args.snapshot_cache),\n            "snapshot_cache_sha256": file_sha256(args.snapshot_cache),\n            "date": evaluation_date,\n            "rows": int(len(target)),\n            "events": int(target["event_id"].nunique()),\n        },\n        "protocol": {\n            "seed": args.seed,\n            "group_repeats": args.group_repeats,\n            "feature_repeats": args.feature_repeats,\n            "conditional_strata": list(CONDITIONAL_STRATA),\n            "shared_donor_within_group": True,\n            "service_reconstruction_max_abs_diff": reconstruction_max_abs_diff,\n            "feature_partition": {\n                name: list(columns) for name, columns in FEATURE_GROUPS.items()\n            },\n        },\n        "baseline_metrics": baseline_metrics,\n        "top_group_global": top_global.to_dict("records"),\n        "top_group_conditional": top_conditional.to_dict("records"),\n        "artifacts": {\n            "raw": str(raw_path),\n            "summary": str(importance_path),\n        },\n    }\n    summary_output_path = args.output_dir / "summary.json"\n    summary_output_path.write_text(\n        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(\n        group_summary[\n            [\n                "scheme",\n                "scope",\n                "feature",\n                "overall_mae_increase_mean",\n                "low_0_5_mae_increase_mean",\n                "far_6_plus_mae_increase_mean",\n                "prediction_mean_abs_change_mean",\n            ]\n        ].to_string(index=False)\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/route_distribution.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\n\nROUTE_DISTRIBUTION_FEATURES = (\n    "route_dist_mean_delta",\n    "route_dist_mean_seats",\n    "route_dist_scale",\n    "route_dist_q10_seats",\n    "route_dist_q90_seats",\n    "route_dist_p_full",\n    "route_dist_p_low_5",\n    "route_dist_p_low_10",\n    "route_dist_expected_shortfall_5",\n    "route_dist_cell_prior_share",\n    "route_dist_cross_bin_share",\n    "route_dist_nonpass_stops",\n    "route_dist_global_fallback_share",\n)\n\nINTERVAL_STOP_BINS = (0, 2, 5, 10, np.inf)\nINTERVAL_STOP_LABELS = ("1-2", "3-5", "6-10", "11+")\nLOWER_TAIL_PREDICTION_BINS = (-np.inf, 5, 10, 20, np.inf)\nLOWER_TAIL_PREDICTION_LABELS = ("0-5", "6-10", "11-20", "21+")\nLOWER_TAIL_REFINEMENT_KIND = "predicted_seat_band_empirical_tail"\nNORMAL_90_Z = 1.6448536269514722\nSUPPORTED_INTERVAL_SCALES = frozenset(\n    {"stop_gap", "route_scale", "route_scale_sqrt"}\n)\nSUPPORTED_PROBABILITY_CALIBRATIONS = frozenset({"logit_slope"})\nPOLICY_PROVENANCE_SCHEMA_VERSION = 1\nDISTRIBUTION_FLOW_COLUMNS = (\n    "date",\n    "station_seq",\n    "direction",\n    "time_bin_2h",\n    "stop_net",\n)\n\n\ndef file_sha256(path: Path) -> str:\n    """Hash an artifact without loading the whole file into memory."""\n    digest = hashlib.sha256()\n    with path.open("rb") as artifact:\n        for chunk in iter(lambda: artifact.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef canonical_json_sha256(value: Any) -> str:\n    """Hash JSON-compatible provenance independent of mapping insertion order."""\n    serialized = json.dumps(\n        json_ready(value),\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n        allow_nan=False,\n    ).encode("utf-8")\n    return hashlib.sha256(serialized).hexdigest()\n\n\ndef point_model_input_provenance_sha256(summary: dict[str, Any]) -> str:\n    """Fingerprint the training-input identity declared by a model summary."""\n    if "data_cache" not in summary or "data" not in summary:\n        raise ValueError(\n            "point-model summary에 data_cache/data 입력 provenance가 없습니다."\n        )\n    protocol = summary.get("protocol", {})\n    return canonical_json_sha256(\n        {\n            "data_cache": summary["data_cache"],\n            "data": summary["data"],\n            "validation_dates": protocol.get(\n                "hypothesis_and_parameter_selection", {}\n            ).get("rolling_origin_validation_dates"),\n            "locked_confirmation_test": protocol.get(\n                "locked_confirmation_test"\n            ),\n            "weekend_distribution_shift_stress": protocol.get(\n                "weekend_distribution_shift_stress"\n            ),\n        }\n    )\n\n\ndef distribution_flow_fingerprint(flows: pd.DataFrame) -> str:\n    """Hash the exact semantic inputs used by route-distribution inference.\n\n    The canonical representation is insensitive to row ordering and to ordering\n    or duplication in ``pass_node_sequences`` because neither changes the\n    distribution calculation. Duplicate flow rows themselves remain significant.\n    """\n    if not isinstance(flows, pd.DataFrame):\n        raise TypeError("distribution flows는 pandas DataFrame이어야 합니다.")\n    missing = sorted(set(DISTRIBUTION_FLOW_COLUMNS) - set(flows.columns))\n    if missing:\n        raise ValueError(f"distribution flow 열이 누락되었습니다: {missing}")\n\n    canonical_rows: list[tuple[str, int, str, int, str]] = []\n    for row in flows.loc[:, DISTRIBUTION_FLOW_COLUMNS].itertuples(\n        index=False, name=None\n    ):\n        raw_date, raw_station, raw_direction, raw_time_bin, raw_stop_net = row\n        if pd.isna(raw_date) or pd.isna(raw_direction):\n            raise ValueError("distribution flow date/direction에 결측값이 있습니다.")\n        try:\n            station = int(raw_station)\n            time_bin = int(raw_time_bin)\n            stop_net = float(raw_stop_net)\n        except (TypeError, ValueError, OverflowError) as error:\n            raise ValueError(\n                "distribution flow 숫자 열을 정규화할 수 없습니다."\n            ) from error\n        if (\n            not np.isfinite(stop_net)\n            or not np.isfinite(float(raw_station))\n            or not np.isfinite(float(raw_time_bin))\n            or float(raw_station) != station\n            or float(raw_time_bin) != time_bin\n        ):\n            raise ValueError(\n                "distribution flow station/time_bin은 정수, stop_net은 "\n                "유한한 수여야 합니다."\n            )\n        if stop_net == 0:\n            stop_net = 0.0\n        canonical_rows.append(\n            (\n                str(raw_date),\n                station,\n                str(raw_direction),\n                time_bin,\n                stop_net.hex(),\n            )\n        )\n    canonical_rows.sort()\n\n    raw_pass_nodes = flows.attrs.get("pass_node_sequences", [])\n    try:\n        pass_nodes = sorted({int(value) for value in raw_pass_nodes})\n    except (TypeError, ValueError, OverflowError) as error:\n        raise ValueError(\n            "distribution flow pass_node_sequences를 정규화할 수 없습니다."\n        ) from error\n    digest = hashlib.sha256()\n    digest.update(b"route_distribution_flow_v1\\n")\n    digest.update(\n        json.dumps(\n            {\n                "columns": DISTRIBUTION_FLOW_COLUMNS,\n                "pass_node_sequences": pass_nodes,\n            },\n            ensure_ascii=False,\n            sort_keys=True,\n            separators=(",", ":"),\n        ).encode("utf-8")\n    )\n    digest.update(b"\\n")\n    for row in canonical_rows:\n        digest.update(\n            json.dumps(\n                row,\n                ensure_ascii=False,\n                separators=(",", ":"),\n            ).encode("utf-8")\n        )\n        digest.update(b"\\n")\n    return digest.hexdigest()\n\n\ndef _validated_sha256(value: Any, *, field: str) -> str:\n    normalized = str(value).lower()\n    if len(normalized) != 64 or any(\n        character not in "0123456789abcdef" for character in normalized\n    ):\n        raise ValueError(f"{field}는 64자리 SHA-256이어야 합니다.")\n    return normalized\n\n\ndef validate_policy_provenance(provenance: Any) -> dict[str, Any]:\n    """Validate the versioned point-model/distribution provenance contract."""\n    if not isinstance(provenance, dict):\n        raise ValueError("interval policy provenance는 객체여야 합니다.")\n    schema_version = int(provenance.get("schema_version", -1))\n    if schema_version != POLICY_PROVENANCE_SCHEMA_VERSION:\n        raise ValueError(\n            "지원하지 않는 interval policy provenance schema입니다: "\n            f"{schema_version}"\n        )\n    point_model = provenance.get("point_model")\n    distribution = provenance.get("distribution")\n    if not isinstance(point_model, dict) or not isinstance(distribution, dict):\n        raise ValueError(\n            "interval policy provenance에 point_model/distribution 객체가 "\n            "필요합니다."\n        )\n    selected_name = str(point_model.get("selected_name", ""))\n    if not selected_name:\n        raise ValueError("point_model selected_name이 비어 있습니다.")\n    normalized_point_model = dict(point_model)\n    normalized_point_model.update(\n        {\n            "selected_name": selected_name,\n            "summary_sha256": _validated_sha256(\n                point_model.get("summary_sha256"),\n                field="point_model summary_sha256",\n            ),\n            "source_sha256": _validated_sha256(\n                point_model.get("source_sha256"),\n                field="point_model source_sha256",\n            ),\n            "input_provenance_sha256": _validated_sha256(\n                point_model.get("input_provenance_sha256"),\n                field="point_model input_provenance_sha256",\n            ),\n        }\n    )\n\n    flow_rows = int(distribution.get("flow_rows", -1))\n    if flow_rows < 0:\n        raise ValueError("distribution provenance flow_rows는 0 이상이어야 합니다.")\n    source_hashes = distribution.get("source_hashes")\n    input_hashes = distribution.get("input_hashes")\n    if not isinstance(source_hashes, dict) or not source_hashes:\n        raise ValueError("distribution provenance source_hashes가 필요합니다.")\n    if not isinstance(input_hashes, dict) or not input_hashes:\n        raise ValueError("distribution provenance input_hashes가 필요합니다.")\n    flow_artifact = distribution.get("flow_artifact")\n    normalized_flow_artifact: dict[str, Any] | None = None\n    if flow_artifact is not None:\n        if not isinstance(flow_artifact, dict):\n            raise ValueError("distribution provenance flow_artifact는 객체여야 합니다.")\n        artifact_path = str(flow_artifact.get("path", "")).strip()\n        if not artifact_path:\n            raise ValueError("distribution flow_artifact path가 비어 있습니다.")\n        normalized_flow_artifact = dict(flow_artifact)\n        normalized_flow_artifact.update(\n            {\n                "path": artifact_path,\n                "file_sha256": _validated_sha256(\n                    flow_artifact.get("file_sha256"),\n                    field="distribution flow_artifact file_sha256",\n                ),\n                "format": str(flow_artifact.get("format", "pandas_pickle")),\n            }\n        )\n    normalized_distribution = dict(distribution)\n    normalized_distribution.update(\n        {\n            "flow_data_sha256": _validated_sha256(\n                distribution.get("flow_data_sha256"),\n                field="distribution flow_data_sha256",\n            ),\n            "flow_rows": flow_rows,\n            "source_hashes": {\n                str(name): _validated_sha256(\n                    digest,\n                    field=f"distribution source_hashes[{name}]",\n                )\n                for name, digest in source_hashes.items()\n            },\n            "input_hashes": {\n                str(name): _validated_sha256(\n                    digest,\n                    field=f"distribution input_hashes[{name}]",\n                )\n                for name, digest in input_hashes.items()\n            },\n        }\n    )\n    if normalized_flow_artifact is not None:\n        normalized_distribution["flow_artifact"] = normalized_flow_artifact\n    normalized = dict(provenance)\n    normalized.update(\n        {\n            "schema_version": schema_version,\n            "point_model": normalized_point_model,\n            "distribution": normalized_distribution,\n        }\n    )\n    return normalized\n\n\ndef _posterior_moments(\n    raw_mean: float,\n    raw_variance: float,\n    count: float,\n    prior_mean: float,\n    prior_variance: float,\n    strength: float,\n) -> tuple[float, float, float]:\n    """Moment-match an empirical distribution with a prior distribution."""\n    weight = count / (count + strength)\n    mean = weight * raw_mean + (1 - weight) * prior_mean\n    variance = weight * (raw_variance + (raw_mean - mean) ** 2) + (\n        1 - weight\n    ) * (prior_variance + (prior_mean - mean) ** 2)\n    return float(mean), float(max(variance, 0.0)), float(weight)\n\n\ndef _normal_cdf(values: np.ndarray) -> np.ndarray:\n    scaled = np.asarray(values, dtype=float) / math.sqrt(2.0)\n    return 0.5 * (\n        1.0\n        + np.fromiter(\n            (math.erf(float(value)) for value in scaled),\n            dtype=float,\n            count=len(scaled),\n        )\n    )\n\n\ndef interval_stop_band(data: pd.DataFrame) -> pd.Series:\n    """Map stop gaps to the exact calibration bands used during search."""\n    return pd.cut(\n        data["target_stop_gap"],\n        bins=INTERVAL_STOP_BINS,\n        labels=INTERVAL_STOP_LABELS,\n        include_lowest=True,\n    ).astype(str)\n\n\ndef interval_scale_values(data: pd.DataFrame, kind: str) -> np.ndarray:\n    """Return the exact uncertainty scale used by an interval policy."""\n    if kind == "stop_gap":\n        return np.ones(len(data), dtype=float)\n    route_scale = data["route_dist_scale"].to_numpy(dtype=float)\n    if kind == "route_scale":\n        return np.maximum(route_scale, 0.5)\n    if kind == "route_scale_sqrt":\n        return np.sqrt(np.maximum(route_scale, 0.5))\n    raise ValueError(f"알 수 없는 interval scale: {kind}")\n\n\ndef lower_tail_prediction_band(\n    prediction: np.ndarray,\n    *,\n    index: pd.Index | None = None,\n) -> pd.Series:\n    """Map a point prediction to the one-sided calibration bands."""\n    values = np.asarray(prediction, dtype=float)\n    if values.ndim != 1 or not np.isfinite(values).all():\n        raise ValueError("lower-tail prediction은 유한한 1차원 배열이어야 합니다.")\n    if index is not None and len(index) != len(values):\n        raise ValueError("lower-tail prediction index 길이가 다릅니다.")\n    return pd.cut(\n        pd.Series(values, index=index),\n        bins=LOWER_TAIL_PREDICTION_BINS,\n        labels=LOWER_TAIL_PREDICTION_LABELS,\n        include_lowest=True,\n    ).astype(str)\n\n\ndef lower_tail_scale_values(\n    data: pd.DataFrame,\n    *,\n    scale_floor: float,\n) -> np.ndarray:\n    """Return the route scale used by the empirical lower-tail guardrail."""\n    floor = float(scale_floor)\n    if not np.isfinite(floor) or floor <= 0:\n        raise ValueError("lower-tail scale_floor는 유한한 양수여야 합니다.")\n    scale = data["route_dist_scale"].to_numpy(dtype=float)\n    if not np.isfinite(scale).all():\n        raise ValueError("route_dist_scale에는 유한한 값만 허용됩니다.")\n    return np.maximum(scale, floor)\n\n\ndef apply_probability_calibration(\n    probability: np.ndarray,\n    calibration: dict[str, Any] | None,\n) -> np.ndarray:\n    """Apply a monotone probability-only calibration layer.\n\n    A slope below one softens overconfident log-odds without changing the\n    within-date ranking.  The caller decides whether the calibrated value is\n    active or shadow-only; this function only performs the transformation.\n    """\n    raw = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)\n    if calibration is None:\n        return raw\n    kind = str(calibration.get("kind", ""))\n    if kind not in SUPPORTED_PROBABILITY_CALIBRATIONS:\n        raise ValueError(f"알 수 없는 probability calibration: {kind}")\n    slope = float(calibration.get("slope", np.nan))\n    if not np.isfinite(slope) or slope <= 0:\n        raise ValueError("probability calibration slope는 유한한 양수여야 합니다.")\n    logit = np.log(raw) - np.log1p(-raw)\n    scaled = np.clip(slope * logit, -40, 40)\n    return 1.0 / (1.0 + np.exp(-scaled))\n\n\ndef validate_lower_tail_refinement(value: Any) -> dict[str, Any]:\n    """Validate the optional predicted-seat-band empirical tail guardrail."""\n    if not isinstance(value, dict):\n        raise ValueError("lower_tail_refinement은 객체여야 합니다.")\n    kind = str(value.get("kind", ""))\n    if kind != LOWER_TAIL_REFINEMENT_KIND:\n        raise ValueError(f"알 수 없는 lower-tail refinement: {kind}")\n    if "status" not in value:\n        raise ValueError("lower-tail refinement status를 명시해야 합니다.")\n    status = str(value["status"])\n    if status not in {"shadow", "active"}:\n        raise ValueError(\n            "lower-tail refinement status는 shadow 또는 active여야 합니다."\n        )\n    target_coverage = float(value.get("target_coverage", np.nan))\n    calibration_quantile = float(value.get("calibration_quantile", np.nan))\n    if not np.isclose(target_coverage, 0.95) or not np.isclose(\n        calibration_quantile, 0.95\n    ):\n        raise ValueError("lower-tail refinement는 one-sided q95만 지원합니다.")\n    scale_kind = str(value.get("scale_kind", ""))\n    if scale_kind != "route_scale":\n        raise ValueError(\n            "lower-tail refinement scale_kind는 route_scale이어야 합니다."\n        )\n    scale_floor = float(value.get("scale_floor", np.nan))\n    if not np.isfinite(scale_floor) or scale_floor <= 0:\n        raise ValueError("lower-tail refinement scale_floor는 유한한 양수여야 합니다.")\n    shrinkage_event_count = float(\n        value.get("shrinkage_event_count", np.nan)\n    )\n    if not np.isfinite(shrinkage_event_count) or shrinkage_event_count < 0:\n        raise ValueError(\n            "lower-tail shrinkage_event_count는 유한한 0 이상이어야 합니다."\n        )\n    global_factor = float(value.get("global_factor", np.nan))\n    if not np.isfinite(global_factor) or global_factor < 0:\n        raise ValueError("lower-tail global_factor는 유한한 0 이상이어야 합니다.")\n\n    raw_factors = value.get("band_factors")\n    raw_counts = value.get("band_event_counts")\n    if not isinstance(raw_factors, dict) or not isinstance(raw_counts, dict):\n        raise ValueError(\n            "lower-tail band_factors와 band_event_counts는 객체여야 합니다."\n        )\n    if set(map(str, raw_factors)) != set(LOWER_TAIL_PREDICTION_LABELS):\n        raise ValueError("lower-tail band_factors 구간이 올바르지 않습니다.")\n    if set(map(str, raw_counts)) != set(LOWER_TAIL_PREDICTION_LABELS):\n        raise ValueError("lower-tail band_event_counts 구간이 올바르지 않습니다.")\n    band_factors: dict[str, float] = {}\n    band_event_counts: dict[str, int] = {}\n    for label in LOWER_TAIL_PREDICTION_LABELS:\n        factor = float(raw_factors[label])\n        if not np.isfinite(factor) or factor < 0:\n            raise ValueError(\n                f"lower-tail factor는 유한한 0 이상이어야 합니다: {label}"\n            )\n        count = int(raw_counts[label])\n        if count < 0 or float(raw_counts[label]) != count:\n            raise ValueError(\n                f"lower-tail event count는 0 이상 정수여야 합니다: {label}"\n            )\n        band_factors[label] = factor\n        band_event_counts[label] = count\n\n    raw_fit_provenance = value.get("fit_provenance")\n    if not isinstance(raw_fit_provenance, dict):\n        raise ValueError("lower-tail fit_provenance는 객체여야 합니다.")\n    calibration_rows = int(raw_fit_provenance.get("calibration_rows", -1))\n    calibration_events = int(raw_fit_provenance.get("calibration_events", -1))\n    raw_dates = raw_fit_provenance.get("calibration_dates")\n    if calibration_rows < 0 or calibration_events < 0:\n        raise ValueError("lower-tail calibration rows/events는 0 이상이어야 합니다.")\n    if not isinstance(raw_dates, list):\n        raise ValueError("lower-tail calibration_dates는 배열이어야 합니다.")\n    calibration_dates = [str(date) for date in raw_dates]\n    if calibration_dates != sorted(set(calibration_dates)):\n        raise ValueError(\n            "lower-tail calibration_dates는 중복 없는 오름차순이어야 합니다."\n        )\n    fit_provenance = dict(raw_fit_provenance)\n    fit_provenance.update(\n        {\n            "calibration_rows": calibration_rows,\n            "calibration_events": calibration_events,\n            "calibration_dates": calibration_dates,\n        }\n    )\n\n    normalized = dict(value)\n    normalized.update(\n        {\n            "kind": kind,\n            "status": status,\n            "target_coverage": target_coverage,\n            "calibration_quantile": calibration_quantile,\n            "scale_kind": scale_kind,\n            "scale_floor": scale_floor,\n            "shrinkage_event_count": shrinkage_event_count,\n            "global_factor": global_factor,\n            "band_factors": band_factors,\n            "band_event_counts": band_event_counts,\n            "fit_provenance": fit_provenance,\n        }\n    )\n    return normalized\n\n\ndef validate_interval_policy(policy: dict[str, Any]) -> dict[str, Any]:\n    """Validate and normalize a serialized 90% interval policy."""\n    required = {"kind", "target_coverage", "global_factor", "band_factors"}\n    missing = sorted(required - set(policy))\n    if missing:\n        raise ValueError(f"interval policy 필드가 누락되었습니다: {missing}")\n\n    kind = str(policy["kind"])\n    if kind not in SUPPORTED_INTERVAL_SCALES:\n        raise ValueError(f"알 수 없는 interval scale: {kind}")\n    target_coverage = float(policy["target_coverage"])\n    if not np.isclose(target_coverage, 0.9):\n        raise ValueError(\n            "서비스 출력은 90% 구간만 지원합니다: "\n            f"target_coverage={target_coverage}"\n        )\n    global_factor = float(policy["global_factor"])\n    if not np.isfinite(global_factor) or global_factor < 0:\n        raise ValueError("interval policy global_factor는 유한한 0 이상이어야 합니다.")\n\n    raw_band_factors = policy["band_factors"]\n    if not isinstance(raw_band_factors, dict):\n        raise ValueError("interval policy band_factors는 객체여야 합니다.")\n    band_factors: dict[str, float] = {}\n    for label, raw_factor in raw_band_factors.items():\n        factor = float(raw_factor)\n        if not np.isfinite(factor) or factor < 0:\n            raise ValueError(\n                f"interval policy band factor는 유한한 0 이상이어야 합니다: {label}"\n            )\n        band_factors[str(label)] = factor\n\n    normalized = dict(policy)\n    normalized.update(\n        {\n            "kind": kind,\n            "target_coverage": target_coverage,\n            "global_factor": global_factor,\n            "band_factors": band_factors,\n        }\n    )\n    if "calibration_quantile" in normalized:\n        calibration_quantile = float(normalized["calibration_quantile"])\n        if not np.isfinite(calibration_quantile) or not (\n            0 < calibration_quantile <= 1\n        ):\n            raise ValueError(\n                "interval policy calibration_quantile은 0 초과 1 이하여야 합니다."\n            )\n        normalized["calibration_quantile"] = calibration_quantile\n    raw_probability_calibration = normalized.get("probability_calibration")\n    if raw_probability_calibration is not None:\n        if not isinstance(raw_probability_calibration, dict):\n            raise ValueError("probability_calibration은 객체여야 합니다.")\n        kind = str(raw_probability_calibration.get("kind", ""))\n        if kind not in SUPPORTED_PROBABILITY_CALIBRATIONS:\n            raise ValueError(f"알 수 없는 probability calibration: {kind}")\n        slope = float(raw_probability_calibration.get("slope", np.nan))\n        if not np.isfinite(slope) or slope <= 0:\n            raise ValueError(\n                "probability calibration slope는 유한한 양수여야 합니다."\n            )\n        status = str(raw_probability_calibration.get("status", "shadow"))\n        if status not in {"shadow", "active"}:\n            raise ValueError(\n                "probability calibration status는 shadow 또는 active여야 합니다."\n            )\n        calibration = dict(raw_probability_calibration)\n        calibration.update(\n            {\n                "kind": kind,\n                "slope": slope,\n                "status": status,\n            }\n        )\n        normalized["probability_calibration"] = calibration\n    raw_lower_tail_refinement = normalized.get("lower_tail_refinement")\n    if raw_lower_tail_refinement is not None:\n        normalized["lower_tail_refinement"] = validate_lower_tail_refinement(\n            raw_lower_tail_refinement\n        )\n    raw_provenance = normalized.get("provenance")\n    if raw_provenance is not None:\n        normalized["provenance"] = validate_policy_provenance(raw_provenance)\n    return normalized\n\n\ndef apply_interval_policy(\n    data: pd.DataFrame,\n    prediction: np.ndarray,\n    policy: dict[str, Any],\n    *,\n    full_threshold_seats: float = 0.0,\n) -> pd.DataFrame:\n    """Apply a calibrated interval around point predictions.\n\n    The interval and ``P(arrival <= 5)`` calculations intentionally mirror\n    ``route_distribution_search.apply_interval_policy``.  ``p_full`` uses the\n    same inferred Normal scale with a caller-defined seat threshold; zero seats\n    reproduces the search-time full probability exactly.\n    """\n    normalized_policy = validate_interval_policy(policy)\n    point = np.asarray(prediction, dtype=float)\n    if point.ndim != 1 or len(point) != len(data):\n        raise ValueError("prediction은 입력 행 수와 같은 1차원 배열이어야 합니다.")\n    if not np.isfinite(point).all():\n        raise ValueError("prediction에는 유한한 값만 허용됩니다.")\n    full_threshold = float(full_threshold_seats)\n    if not np.isfinite(full_threshold):\n        raise ValueError("full_threshold_seats는 유한한 값이어야 합니다.")\n\n    bands = interval_stop_band(data)\n    factor = bands.map(normalized_policy["band_factors"]).fillna(\n        normalized_policy["global_factor"]\n    ).to_numpy(dtype=float)\n    half_width = factor * interval_scale_values(\n        data, normalized_policy["kind"]\n    )\n    capacity = data["capacity"].to_numpy(dtype=float)\n    output = pd.DataFrame(index=data.index)\n    output["interval_half_width"] = half_width\n    unrefined_lower = np.clip(point - half_width, 0, capacity)\n    output["interval_lower_unrefined"] = unrefined_lower\n    refinement = normalized_policy.get("lower_tail_refinement")\n    if refinement is None:\n        refinement_factor = np.full(len(data), np.nan, dtype=float)\n        refinement_candidate = unrefined_lower.copy()\n        refined_lower = unrefined_lower.copy()\n        refinement_applied = np.zeros(len(data), dtype=bool)\n    else:\n        refinement_bands = lower_tail_prediction_band(point, index=data.index)\n        refinement_factor = refinement_bands.map(\n            refinement["band_factors"]\n        ).fillna(refinement["global_factor"]).to_numpy(dtype=float)\n        refinement_scale = lower_tail_scale_values(\n            data,\n            scale_floor=refinement["scale_floor"],\n        )\n        refinement_candidate = np.clip(\n            point - refinement_factor * refinement_scale,\n            0,\n            capacity,\n        )\n        if refinement["status"] == "active":\n            refined_lower = np.minimum(unrefined_lower, refinement_candidate)\n            refinement_applied = refined_lower < unrefined_lower\n        else:\n            refined_lower = unrefined_lower.copy()\n            refinement_applied = np.zeros(len(data), dtype=bool)\n    output["interval_lower"] = refined_lower\n    output["interval_upper"] = np.clip(point + half_width, 0, capacity)\n    output["lower_tail_refinement_factor"] = refinement_factor\n    output["lower_tail_refinement_candidate"] = refinement_candidate\n    output["lower_tail_refinement_applied"] = refinement_applied\n    sigma = np.maximum(half_width / NORMAL_90_Z, 0.25)\n    raw_low_5 = _normal_cdf((5.0 - point) / sigma)\n    probability_calibration = normalized_policy.get("probability_calibration")\n    calibrated_low_5 = apply_probability_calibration(\n        raw_low_5,\n        probability_calibration,\n    )\n    output["p_low_5_raw"] = raw_low_5\n    output["p_low_5_calibrated"] = calibrated_low_5\n    if (\n        probability_calibration is not None\n        and probability_calibration["status"] == "active"\n    ):\n        output["p_low_5"] = calibrated_low_5\n    else:\n        output["p_low_5"] = raw_low_5\n    output["p_full"] = _normal_cdf((full_threshold - point) / sigma)\n    return output\n\n\ndef _historical_distribution_values(\n    flows: pd.DataFrame,\n    target: pd.DataFrame,\n    *,\n    station_strength: float,\n    cell_strength: float,\n    observation_noise_seats: float,\n) -> pd.DataFrame:\n    output = pd.DataFrame(index=target.index)\n    if target.empty:\n        for column in ROUTE_DISTRIBUTION_FEATURES:\n            output[column] = np.asarray([], dtype=float)\n        return output\n\n    weekday = flows.loc[\n        pd.to_datetime(flows["date"]).dt.dayofweek.lt(5)\n    ].copy()\n    current = target["snapshot_remaining_seats"].to_numpy(dtype=float)\n    capacity = target["capacity"].to_numpy(dtype=float)\n    if weekday.empty:\n        scale = np.full(len(target), observation_noise_seats, dtype=float)\n        return _finish_distribution(\n            output,\n            current=current,\n            capacity=capacity,\n            delta=np.zeros(len(target), dtype=float),\n            scale=scale,\n            prior_share=np.ones(len(target), dtype=float),\n            cross_bin_share=np.zeros(len(target), dtype=float),\n            nonpass_stops=np.zeros(len(target), dtype=float),\n            global_fallback_share=np.ones(len(target), dtype=float),\n        )\n\n    global_mean = float(weekday["stop_net"].mean())\n    global_variance = float(weekday["stop_net"].var(ddof=0))\n    if not np.isfinite(global_variance) or global_variance <= 0:\n        global_variance = observation_noise_seats**2\n\n    station_stats = weekday.groupby(\n        ["station_seq", "direction"], sort=False\n    )["stop_net"].agg(["mean", "var", "count"])\n    station_moments: dict[tuple[int, str], tuple[float, float, float]] = {}\n    for key, row in station_stats.iterrows():\n        raw_variance = float(row["var"]) if pd.notna(row["var"]) else 0.0\n        station_moments[(int(key[0]), str(key[1]))] = _posterior_moments(\n            float(row["mean"]),\n            raw_variance,\n            float(row["count"]),\n            global_mean,\n            global_variance,\n            station_strength,\n        )\n\n    cell_stats = weekday.groupby(\n        ["station_seq", "direction", "time_bin_2h"], sort=False\n    )["stop_net"].agg(["mean", "var", "count"])\n    cell_moments: dict[tuple[int, str, int], tuple[float, float, float]] = {}\n    for key, row in cell_stats.iterrows():\n        station_key = (int(key[0]), str(key[1]))\n        station_mean, station_variance, _ = station_moments.get(\n            station_key, (global_mean, global_variance, 0.0)\n        )\n        raw_variance = float(row["var"]) if pd.notna(row["var"]) else 0.0\n        cell_moments[(int(key[0]), str(key[1]), int(key[2]))] = (\n            _posterior_moments(\n                float(row["mean"]),\n                raw_variance,\n                float(row["count"]),\n                station_mean,\n                station_variance,\n                cell_strength,\n            )\n        )\n\n    pass_nodes = {\n        int(value) for value in flows.attrs.get("pass_node_sequences", [])\n    }\n    deltas = np.zeros(len(target), dtype=float)\n    scales = np.full(len(target), observation_noise_seats, dtype=float)\n    prior_shares = np.ones(len(target), dtype=float)\n    cross_bin_shares = np.zeros(len(target), dtype=float)\n    nonpass_counts = np.zeros(len(target), dtype=float)\n    global_fallback_shares = np.ones(len(target), dtype=float)\n\n    for position, row in enumerate(target.itertuples(index=False)):\n        start = int(row.snapshot_station_seq)\n        if str(row.target_state_cat) != "1":\n            start += 1\n        stop = int(row.station_seq_cat)\n        if start >= stop:\n            continue\n\n        snapshot_time = pd.Timestamp(row.snapshot_time)\n        snapshot_bin = int(snapshot_time.hour // 2 * 2)\n        minutes_per_stop = float(row.rolling_minutes_per_stop_3)\n        if not np.isfinite(minutes_per_stop) or minutes_per_stop < 0:\n            minutes_per_stop = 0.0\n\n        means: list[float] = []\n        variances: list[float] = []\n        priors: list[float] = []\n        crossed = 0\n        global_fallbacks = 0\n        for station_seq in range(start, stop):\n            if station_seq in pass_nodes:\n                continue\n            offset = max(station_seq - int(row.snapshot_station_seq), 0)\n            stop_time = snapshot_time + pd.Timedelta(\n                minutes=offset * minutes_per_stop\n            )\n            time_bin = int(stop_time.hour // 2 * 2)\n            crossed += int(time_bin != snapshot_bin)\n            cell_key = (station_seq, str(row.direction), time_bin)\n            station_key = (station_seq, str(row.direction))\n            if cell_key in cell_moments:\n                mean, variance, empirical_weight = cell_moments[cell_key]\n                prior_share = 1.0 - empirical_weight\n            elif station_key in station_moments:\n                mean, variance, _ = station_moments[station_key]\n                prior_share = 1.0\n            else:\n                mean, variance = global_mean, global_variance\n                prior_share = 1.0\n                global_fallbacks += 1\n            means.append(mean)\n            variances.append(variance)\n            priors.append(prior_share)\n\n        count = len(means)\n        if count == 0:\n            continue\n        deltas[position] = float(np.sum(means))\n        scales[position] = math.sqrt(\n            observation_noise_seats**2 + max(float(np.sum(variances)), 0.0)\n        )\n        prior_shares[position] = float(np.mean(priors))\n        cross_bin_shares[position] = crossed / count\n        nonpass_counts[position] = count\n        global_fallback_shares[position] = global_fallbacks / count\n\n    return _finish_distribution(\n        output,\n        current=current,\n        capacity=capacity,\n        delta=deltas,\n        scale=scales,\n        prior_share=prior_shares,\n        cross_bin_share=cross_bin_shares,\n        nonpass_stops=nonpass_counts,\n        global_fallback_share=global_fallback_shares,\n    )\n\n\ndef _finish_distribution(\n    output: pd.DataFrame,\n    *,\n    current: np.ndarray,\n    capacity: np.ndarray,\n    delta: np.ndarray,\n    scale: np.ndarray,\n    prior_share: np.ndarray,\n    cross_bin_share: np.ndarray,\n    nonpass_stops: np.ndarray,\n    global_fallback_share: np.ndarray,\n) -> pd.DataFrame:\n    raw_mean = current + delta\n    safe_scale = np.maximum(scale, 0.5)\n    mean_seats = np.clip(raw_mean, 0, capacity)\n    q10 = np.clip(raw_mean - 1.2815515655446004 * safe_scale, 0, capacity)\n    q90 = np.clip(raw_mean + 1.2815515655446004 * safe_scale, 0, capacity)\n    z_full = (0.0 - raw_mean) / safe_scale\n    z_low_5 = (5.0 - raw_mean) / safe_scale\n    z_low_10 = (10.0 - raw_mean) / safe_scale\n    p_full = _normal_cdf(z_full)\n    p_low_5 = _normal_cdf(z_low_5)\n    p_low_10 = _normal_cdf(z_low_10)\n    phi_low_5 = np.exp(-0.5 * z_low_5**2) / math.sqrt(2 * math.pi)\n    expected_shortfall = (5.0 - raw_mean) * p_low_5 + safe_scale * phi_low_5\n\n    output["route_dist_mean_delta"] = delta\n    output["route_dist_mean_seats"] = mean_seats\n    output["route_dist_scale"] = safe_scale\n    output["route_dist_q10_seats"] = q10\n    output["route_dist_q90_seats"] = q90\n    output["route_dist_p_full"] = p_full\n    output["route_dist_p_low_5"] = p_low_5\n    output["route_dist_p_low_10"] = p_low_10\n    output["route_dist_expected_shortfall_5"] = np.maximum(\n        expected_shortfall, 0.0\n    )\n    output["route_dist_cell_prior_share"] = np.clip(prior_share, 0, 1)\n    output["route_dist_cross_bin_share"] = np.clip(cross_bin_share, 0, 1)\n    output["route_dist_nonpass_stops"] = nonpass_stops\n    output["route_dist_global_fallback_share"] = np.clip(\n        global_fallback_share, 0, 1\n    )\n    return output\n\n\ndef route_distribution_values(\n    flows: pd.DataFrame,\n    target: pd.DataFrame,\n    *,\n    station_strength: float = 20.0,\n    cell_strength: float = 10.0,\n    observation_noise_seats: float = 1.0,\n) -> pd.DataFrame:\n    """Build strict past-only route moments and Normal distribution features."""\n    if station_strength <= 0 or cell_strength <= 0:\n        raise ValueError("station_strength와 cell_strength는 0보다 커야 합니다.")\n    if observation_noise_seats <= 0:\n        raise ValueError("observation_noise_seats는 0보다 커야 합니다.")\n    output = pd.DataFrame(index=target.index)\n    if target.empty:\n        for column in ROUTE_DISTRIBUTION_FEATURES:\n            output[column] = np.asarray([], dtype=float)\n        return output\n\n    def calendar_dates(values: pd.Series, *, field: str) -> pd.Series:\n        normalized: list[str] = []\n        for value in values:\n            try:\n                timestamp = pd.Timestamp(value)\n            except (TypeError, ValueError) as error:\n                raise ValueError(f"{field} 날짜를 해석할 수 없습니다: {value!r}") from error\n            if pd.isna(timestamp):\n                raise ValueError(f"{field} 날짜가 비어 있습니다.")\n            normalized.append(timestamp.date().isoformat())\n        return pd.Series(normalized, index=values.index, dtype="string")\n\n    dates = calendar_dates(\n        target["date"] if "date" in target.columns else target["snapshot_time"],\n        field="target",\n    )\n    flow_dates = calendar_dates(flows["date"], field="distribution flow")\n    for date, indices in dates.groupby(dates, sort=False).groups.items():\n        historical = flows.loc[flow_dates.lt(str(date))].copy()\n        historical.attrs.update(flows.attrs)\n        values = _historical_distribution_values(\n            historical,\n            target.loc[indices],\n            station_strength=station_strength,\n            cell_strength=cell_strength,\n            observation_noise_seats=observation_noise_seats,\n        )\n        output.loc[indices, list(ROUTE_DISTRIBUTION_FEATURES)] = values[\n            list(ROUTE_DISTRIBUTION_FEATURES)\n        ].to_numpy()\n    return output.astype(float)\n\n\ndef json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): json_ready(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(item) for item in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return float(value)\n    return value\n', 'analysis/pooled_main_model_overfit_ablation.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport sqlite3\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nfrom hypothesis_model_search import add_observed_capacity_features\nfrom latest_main_model_feature_recheck import (\n    LATEST_COMPLETE_DATES,\n    LATEST_PARTIAL_DATE,\n)\nfrom latest_main_model_overfit_ablation import (\n    COORDINATE_FEATURES,\n    OBSERVED_CEILING_FEATURES,\n    apply_candidate,\n    train_schema,\n)\nfrom linear_feature_experiment import (\n    add_strict_prior_flow_features,\n    add_strict_prior_low_rate_features,\n    json_ready,\n)\nfrom main_model_feature_augmentation import feature_sets, required_metrics\nfrom model_feasibility import FeatureSet\nfrom route_specific_feature_experiment import DEFAULT_ROUTES, cache_paths\n\n\nSOURCE_CUTOFF = "2026-08-13 20:26:03+09:00"\nPRIMARY_NAME = "pooled_40_legacy_ceiling_features"\n\n\ndef pooled_candidates() -> dict[str, tuple[FeatureSet, bool]]:\n    primary = feature_sets()["importance_pruned"]\n\n    def schema(name: str, removed: tuple[str, ...]) -> FeatureSet:\n        return FeatureSet(\n            name=name,\n            numeric=tuple(\n                column for column in primary.numeric if column not in removed\n            ),\n            categorical=(*primary.categorical, "route_code"),\n        )\n\n    return {\n        # The Route-1000-only 44/70 output clip cannot be applied project-wide:\n        # active pooled routes contain vehicle category 5. Keep its three\n        # engineered inputs only as the legacy baseline and use nominal support.\n        PRIMARY_NAME: (schema("pooled_40", ()), False),\n        "pooled_37_no_ceiling": (\n            schema("pooled_37", OBSERVED_CEILING_FEATURES),\n            False,\n        ),\n        "pooled_38_no_coordinates": (\n            schema("pooled_38", COORDINATE_FEATURES),\n            False,\n        ),\n        "pooled_35_no_ceiling_coordinates": (\n            schema(\n                "pooled_35",\n                (*OBSERVED_CEILING_FEATURES, *COORDINATE_FEATURES),\n            ),\n            False,\n        ),\n    }\n\n\ndef active_route_audit(database: Path, cutoff: str) -> pd.DataFrame:\n    with sqlite3.connect(f"file:{database.resolve()}?mode=ro", uri=True) as connection:\n        routes = pd.read_sql_query(\n            """\n            SELECT route_id, station_count, observation_count,\n                   first_collected_at, last_collected_at\n            FROM routes\n            ORDER BY route_id\n            """,\n            connection,\n        )\n    active_ids = set(DEFAULT_ROUTES)\n    routes["route_name"] = routes["route_id"].map(DEFAULT_ROUTES)\n    routes["included"] = routes["route_id"].isin(active_ids)\n    routes["reason"] = np.select(\n        [routes["included"], routes["station_count"].eq(0)],\n        [\n            "actively collected through source cutoff",\n            "no cached station metadata; historical collection only",\n        ],\n        default="historical short collection only; not active at source cutoff",\n    )\n    routes["experiment_source_cutoff"] = cutoff\n    return routes\n\n\ndef prepare_pooled_data(\n    cache_dir: Path,\n    *,\n    source_cutoff: str,\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    frames: list[pd.DataFrame] = []\n    metadata: dict[str, Any] = {}\n    for route_id, route_name in DEFAULT_ROUTES.items():\n        snapshot_path, flow_path, metadata_path = cache_paths(cache_dir, route_id)\n        route_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if route_metadata.get("source_cutoff") != source_cutoff:\n            raise ValueError(\n                f"노선 {route_name} cache cutoff가 다릅니다: "\n                f"{route_metadata.get(\'source_cutoff\')}"\n            )\n        snapshots = pd.read_pickle(snapshot_path)\n        flows = pd.read_pickle(flow_path)\n        print(f"[{route_name}] strict-prior 피처 생성", flush=True)\n        featured = add_strict_prior_low_rate_features(snapshots)\n        featured = add_strict_prior_flow_features(featured, flows)\n        featured = add_observed_capacity_features(featured)\n        featured["event_id"] = route_id + "::" + featured["event_id"].astype(str)\n        featured["trip_id"] = route_id + "::" + featured["trip_id"].astype(str)\n        featured["route_id"] = route_id\n        featured["route_name"] = route_name\n        featured["route_code"] = route_id\n        frames.append(featured)\n        metadata[route_id] = {\n            "route_name": route_name,\n            **route_metadata,\n        }\n    pooled = pd.concat(frames, ignore_index=True)\n    pooled["route_code"] = pooled["route_code"].astype("string")\n    return pooled, metadata\n\n\ndef rolling_folds(data: pd.DataFrame) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []\n    for validation_date in (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE):\n        train = data.loc[\n            data["date"].lt(validation_date)\n            & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)\n        ].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        missing_routes = sorted(set(DEFAULT_ROUTES) - set(validation["route_id"]))\n        if train.empty or validation.empty or missing_routes:\n            raise ValueError(\n                f"통합 rolling fold가 불완전합니다: {validation_date}, "\n                f"missing_routes={missing_routes}"\n            )\n        folds.append((validation_date, train, validation))\n    return folds\n\n\ndef scoped_metrics(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    partitions = {\n        "latest_complete_08_11_12": predictions["date"].isin(\n            LATEST_COMPLETE_DATES\n        ),\n        "latest_partial_08_13": predictions["date"].eq(LATEST_PARTIAL_DATE),\n        "latest_all_08_11_13": predictions["date"].isin(\n            (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)\n        ),\n    }\n    pooled_rows: list[dict[str, Any]] = []\n    route_rows: list[dict[str, Any]] = []\n    for partition, mask in partitions.items():\n        selected = predictions.loc[mask]\n        for candidate, frame in selected.groupby("candidate", sort=False):\n            pooled_rows.append(\n                {\n                    "partition": partition,\n                    "candidate": candidate,\n                    "core_feature_count": int(frame["core_feature_count"].iloc[0]),\n                    "total_feature_count": int(frame["total_feature_count"].iloc[0]),\n                    **required_metrics(frame),\n                }\n            )\n        for (route_id, route_name, candidate), frame in selected.groupby(\n            ["route_id", "route_name", "candidate"], sort=False\n        ):\n            route_rows.append(\n                {\n                    "partition": partition,\n                    "route_id": route_id,\n                    "route_name": route_name,\n                    "candidate": candidate,\n                    **required_metrics(frame),\n                }\n            )\n    pooled = pd.DataFrame(pooled_rows)\n    by_route = pd.DataFrame(route_rows)\n    metric_columns = [\n        "event_balanced_mae",\n        "low_0_10_mae",\n        "full_accuracy",\n        "full_recall",\n        "full_precision",\n        "full_f1",\n    ]\n    macro = (\n        by_route.groupby(["partition", "candidate"], sort=False)[metric_columns]\n        .mean()\n        .reset_index()\n    )\n    macro.insert(2, "routes", len(DEFAULT_ROUTES))\n    return pooled, by_route, macro\n\n\ndef route_stratified_bootstrap(\n    predictions: pd.DataFrame,\n    candidate: str,\n    *,\n    low_only: bool,\n    repeats: int = 2_000,\n    seed: int = 42,\n) -> dict[str, Any]:\n    selected = predictions.loc[\n        predictions["candidate"].isin([PRIMARY_NAME, candidate])\n        & predictions["date"].isin(LATEST_COMPLETE_DATES)\n    ].copy()\n    if low_only:\n        selected = selected.loc[selected["label_seats"].le(10)].copy()\n    selected["absolute_error"] = (\n        selected["label_seats"] - selected["prediction"]\n    ).abs()\n    event_errors = (\n        selected.groupby(\n            ["route_id", "trip_id", "event_id", "candidate"], observed=True\n        )["absolute_error"]\n        .mean()\n        .unstack("candidate")\n        .dropna(subset=[PRIMARY_NAME, candidate])\n        .reset_index()\n    )\n    event_errors["improvement"] = (\n        event_errors[PRIMARY_NAME] - event_errors[candidate]\n    )\n    route_trip_values: dict[str, list[np.ndarray]] = {}\n    for route_id, route in event_errors.groupby("route_id", sort=False):\n        route_trip_values[str(route_id)] = [\n            trip["improvement"].to_numpy(dtype=float)\n            for _, trip in route.groupby("trip_id", sort=False)\n        ]\n    rng = np.random.default_rng(seed)\n    draws = np.empty(repeats, dtype=float)\n    for index in range(repeats):\n        route_means: list[float] = []\n        for trips in route_trip_values.values():\n            sampled = rng.integers(0, len(trips), size=len(trips))\n            route_means.append(\n                float(np.concatenate([trips[item] for item in sampled]).mean())\n            )\n        draws[index] = float(np.mean(route_means))\n    observed = float(\n        event_errors.groupby("route_id")["improvement"].mean().mean()\n    )\n    return {\n        "baseline": PRIMARY_NAME,\n        "candidate": candidate,\n        "metric": "macro_low_0_10_mae" if low_only else "macro_event_balanced_mae",\n        "candidate_improvement": observed,\n        "ci_95_lower": float(np.quantile(draws, 0.025)),\n        "ci_95_upper": float(np.quantile(draws, 0.975)),\n        "probability_candidate_better": float((draws > 0).mean()),\n        "routes": len(route_trip_values),\n        "trips": int(event_errors["trip_id"].nunique()),\n        "events": int(len(event_errors)),\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="전체 활성 노선 통합 40/37/38/35피처 앙상블 ablation"\n    )\n    parser.add_argument(\n        "--database", type=Path, default=Path("data/gbis_api_cache.sqlite3")\n    )\n    parser.add_argument(\n        "--cache-dir",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features"),\n    )\n    parser.add_argument("--source-cutoff", default=SOURCE_CUTOFF)\n    parser.add_argument(\n        "--featured-cache",\n        type=Path,\n        default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/pooled_main_model_overfit_ablation_results"),\n    )\n    parser.add_argument("--rebuild-features", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    if args.featured_cache.is_file() and not args.rebuild_features:\n        pooled = pd.read_pickle(args.featured_cache)\n        if pooled.attrs.get("source_cutoff") != args.source_cutoff:\n            raise ValueError("통합 피처 cache cutoff가 요청 cutoff와 다릅니다.")\n        route_metadata = pooled.attrs["route_metadata"]\n    else:\n        pooled, route_metadata = prepare_pooled_data(\n            args.cache_dir, source_cutoff=args.source_cutoff\n        )\n        pooled.attrs["source_cutoff"] = args.source_cutoff\n        pooled.attrs["route_metadata"] = route_metadata\n        args.featured_cache.parent.mkdir(parents=True, exist_ok=True)\n        pooled.to_pickle(args.featured_cache)\n\n    folds = rolling_folds(pooled)\n    candidates = pooled_candidates()\n    predictions: list[pd.DataFrame] = []\n    schema_predictions: dict[tuple[str, ...], pd.DataFrame] = {}\n    for name, (features, empirical_cap) in candidates.items():\n        key = tuple(features.columns)\n        if key not in schema_predictions:\n            schema_predictions[key] = train_schema(\n                features.name, features, folds, seed=args.seed\n            )\n        prediction, _ = apply_candidate(\n            name, features, empirical_cap, schema_predictions[key]\n        )\n        prediction["route_id"] = prediction["event_id"].str.split(\n            "::", n=1\n        ).str[0]\n        prediction["route_name"] = prediction["route_id"].map(DEFAULT_ROUTES)\n        prediction["core_feature_count"] = len(features.columns) - 1\n        prediction["total_feature_count"] = len(features.columns)\n        predictions.append(prediction)\n    prediction_table = pd.concat(predictions, ignore_index=True)\n    pooled_metrics, route_metrics, macro_metrics = scoped_metrics(prediction_table)\n    bootstrap = pd.DataFrame(\n        [\n            route_stratified_bootstrap(\n                prediction_table, candidate, low_only=low_only, seed=args.seed\n            )\n            for candidate in candidates\n            if candidate != PRIMARY_NAME\n            for low_only in (False, True)\n        ]\n    )\n    route_audit = active_route_audit(args.database, args.source_cutoff)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    prediction_table.to_pickle(args.output_dir / "oof_predictions.pkl")\n    pooled_metrics.to_csv(args.output_dir / "metrics_pooled.csv", index=False)\n    route_metrics.to_csv(args.output_dir / "metrics_by_route.csv", index=False)\n    macro_metrics.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)\n    bootstrap.to_csv(args.output_dir / "route_stratified_bootstrap.csv", index=False)\n    route_audit.to_csv(args.output_dir / "route_inclusion_audit.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": args.source_cutoff,\n            "included_routes": DEFAULT_ROUTES,\n            "validation_dates": [*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE],\n            "complete_selection_dates": LATEST_COMPLETE_DATES,\n            "partial_check_date": LATEST_PARTIAL_DATE,\n            "training_rule": "pooled route-aware model; each fold uses all routes\' earlier weekdays only",\n            "feature_history_rule": "strictly earlier calendar dates within each route",\n            "route_identity_rule": "route_code categorical feature is added to every pooled candidate",\n            "metric_rule": "report pooled, per-route, and unweighted route-macro metrics",\n        },\n        "route_metadata": route_metadata,\n        "candidates": {\n            name: {\n                "core_feature_count": len(features.columns) - 1,\n                "total_feature_count_with_route_code": len(features.columns),\n                "empirical_cap": empirical_cap,\n                "numeric": features.numeric,\n                "categorical": features.categorical,\n            }\n            for name, (features, empirical_cap) in candidates.items()\n        },\n        "pooled_metrics": pooled_metrics.to_dict(orient="records"),\n        "route_macro_metrics": macro_metrics.to_dict(orient="records"),\n        "bootstrap": bootstrap.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print("\\npooled")\n    print(pooled_metrics.to_string(index=False))\n    print("\\nroute macro")\n    print(macro_metrics.to_string(index=False))\n    print("\\nbootstrap")\n    print(bootstrap.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/lightgbm_feature_experiment.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\n\nfrom linear_feature_experiment import (\n    DEVELOPMENT_DATES,\n    FEATURE_BLOCKS,\n    VALIDATION_DATES,\n    add_preceding_bus_segment_features,\n    add_strict_prior_flow_features,\n    add_strict_prior_low_rate_features,\n    cumulative_feature_sets,\n    event_weights,\n    json_ready,\n    metric_row,\n)\n\n\nREGRESSOR_PARAMS: dict[str, Any] = {\n    "objective": "regression_l1",\n    "n_estimators": 500,\n    "learning_rate": 0.03,\n    "num_leaves": 31,\n    "min_child_samples": 30,\n    "reg_alpha": 0.2,\n    "reg_lambda": 3.0,\n    "subsample": 0.9,\n    "subsample_freq": 1,\n    "colsample_bytree": 0.8,\n    "random_state": 42,\n    "n_jobs": -1,\n    "verbosity": -1,\n}\n\nCLASSIFIER_PARAMS: dict[str, Any] = {\n    "objective": "binary",\n    "n_estimators": 300,\n    "learning_rate": 0.03,\n    "num_leaves": 15,\n    "min_child_samples": 30,\n    "reg_alpha": 0.2,\n    "reg_lambda": 3.0,\n    "subsample": 0.9,\n    "subsample_freq": 1,\n    "colsample_bytree": 0.8,\n    "random_state": 42,\n    "n_jobs": -1,\n    "verbosity": -1,\n}\n\n\ndef run_rolling_origin(\n    data: pd.DataFrame,\n    *,\n    feature_sets: dict[str, tuple[str, ...]] | None = None,\n    regressor_params: dict[str, Any] | None = None,\n    classifier_params: dict[str, Any] | None = None,\n    development_dates: tuple[str, ...] = DEVELOPMENT_DATES,\n    validation_dates: tuple[str, ...] = VALIDATION_DATES,\n    return_predictions: bool = False,\n) -> (\n    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]\n    | tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]\n):\n    """Run the same causal feature ablation with fixed LightGBM models."""\n    selected_feature_sets = feature_sets or cumulative_feature_sets()\n    regression_config = {**REGRESSOR_PARAMS, **(regressor_params or {})}\n    classification_config = {**CLASSIFIER_PARAMS, **(classifier_params or {})}\n    metrics: list[dict[str, Any]] = []\n    predictions: list[pd.DataFrame] = []\n    importances: list[dict[str, Any]] = []\n\n    for validation_date in validation_dates:\n        train_dates = [date for date in development_dates if date < validation_date]\n        train = data.loc[data["date"].isin(train_dates)].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        if train.empty or validation.empty:\n            raise ValueError(f"rolling fold가 비었습니다: {validation_date}")\n        train_weight = event_weights(train)\n        train_delta = (\n            train["label_seats"].to_numpy(float)\n            - train["snapshot_remaining_seats"].to_numpy(float)\n        )\n        train_full = train["label_seats"].eq(0).astype(int).to_numpy()\n        unique_full_classes = np.unique(train_full)\n        single_full_class = (\n            int(unique_full_classes[0]) if len(unique_full_classes) == 1 else None\n        )\n\n        persistence = validation[\n            [\n                "event_id",\n                "trip_id",\n                "date",\n                "snapshot_time",\n                "label_seats",\n                "snapshot_remaining_seats",\n                "capacity",\n                "target_stop_gap",\n            ]\n        ].copy()\n        persistence["candidate"] = "persistence"\n        persistence["prediction"] = persistence["snapshot_remaining_seats"]\n        persistence["full_probability"] = persistence[\n            "snapshot_remaining_seats"\n        ].eq(0).astype(float)\n        metrics.append(\n            {\n                "candidate": "persistence",\n                "validation_date": validation_date,\n                **metric_row(persistence),\n            }\n        )\n        predictions.append(persistence)\n\n        weighted_positive = float(train_weight[train_full == 1].sum())\n        weighted_negative = float(train_weight[train_full == 0].sum())\n        scale_pos_weight = weighted_negative / max(weighted_positive, 1e-12)\n\n        for candidate, columns in selected_feature_sets.items():\n            x_train = train.loc[:, list(columns)]\n            x_validation = validation.loc[:, list(columns)]\n            regressor = lgb.LGBMRegressor(**regression_config)\n            regressor.fit(x_train, train_delta, sample_weight=train_weight)\n            raw_delta = regressor.predict(x_validation)\n            prediction = np.clip(\n                validation["snapshot_remaining_seats"].to_numpy(float) + raw_delta,\n                0,\n                validation["capacity"].to_numpy(float),\n            )\n\n            classifier: lgb.LGBMClassifier | None = None\n            if single_full_class is None:\n                classifier = lgb.LGBMClassifier(\n                    **classification_config,\n                    scale_pos_weight=scale_pos_weight,\n                )\n                classifier.fit(x_train, train_full, sample_weight=train_weight)\n                full_probability = classifier.predict_proba(x_validation)[:, 1]\n                classifier_training_mode = "lightgbm"\n            else:\n                # Some route-local folds contain no historical full events. A\n                # binary tree cannot be fitted without both classes, so preserve\n                # the information available at that service date with a constant\n                # probability rather than leaking a later positive label.\n                full_probability = np.full(\n                    len(validation), float(single_full_class), dtype=float\n                )\n                classifier_training_mode = f"constant_{single_full_class}"\n\n            scored = validation[\n                [\n                    "event_id",\n                    "trip_id",\n                    "date",\n                    "snapshot_time",\n                    "label_seats",\n                    "snapshot_remaining_seats",\n                    "capacity",\n                    "target_stop_gap",\n                ]\n            ].copy()\n            scored["candidate"] = candidate\n            scored["prediction"] = prediction\n            scored["full_probability"] = full_probability\n            metrics.append(\n                {\n                    "candidate": candidate,\n                    "validation_date": validation_date,\n                    "classifier_training_mode": classifier_training_mode,\n                    "train_full_events": int(\n                        train.loc[train["label_seats"].eq(0), "event_id"].nunique()\n                    ),\n                    **metric_row(scored),\n                }\n            )\n            predictions.append(scored)\n\n            models: list[tuple[str, Any]] = [("regressor", regressor)]\n            if classifier is not None:\n                models.append(("classifier", classifier))\n            for model_name, model in models:\n                gains = model.booster_.feature_importance(importance_type="gain")\n                for feature, gain in zip(columns, gains):\n                    importances.append(\n                        {\n                            "candidate": candidate,\n                            "validation_date": validation_date,\n                            "model": model_name,\n                            "feature": feature,\n                            "gain": float(gain),\n                        }\n                    )\n\n    prediction_table = pd.concat(predictions, ignore_index=True)\n    aggregate = pd.DataFrame(\n        [\n            {"candidate": candidate, **metric_row(frame)}\n            for candidate, frame in prediction_table.groupby(\n                "candidate", sort=False\n            )\n        ]\n    )\n    result = (aggregate, pd.DataFrame(metrics), pd.DataFrame(importances))\n    if return_predictions:\n        return (*result, prediction_table)\n    return result\n\n\ndef comparison_with_linear(\n    lightgbm_metrics: pd.DataFrame,\n    linear_metrics_path: Path,\n) -> pd.DataFrame:\n    if not linear_metrics_path.is_file():\n        return pd.DataFrame()\n    linear = pd.read_csv(linear_metrics_path)\n    metric_columns = [\n        "event_balanced_mae",\n        "low_0_10_mae",\n        "low_0_5_mae",\n        "full_accuracy",\n        "full_recall",\n        "full_precision",\n        "full_f1",\n    ]\n    comparison = linear[["candidate", *metric_columns]].merge(\n        lightgbm_metrics[["candidate", *metric_columns]],\n        on="candidate",\n        suffixes=("_linear", "_lightgbm"),\n        validate="one_to_one",\n    )\n    for metric in metric_columns:\n        comparison[f"{metric}_lightgbm_minus_linear"] = (\n            comparison[f"{metric}_lightgbm"] - comparison[f"{metric}_linear"]\n        )\n    return comparison\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="사용자 제안 피처의 strict-prior LightGBM ablation"\n    )\n    parser.add_argument(\n        "--cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),\n    )\n    parser.add_argument(\n        "--linear-metrics",\n        type=Path,\n        default=Path("analysis/linear_feature_results/metrics.csv"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/lightgbm_feature_results"),\n    )\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.cache)\n    flows = pd.read_pickle(args.flow_cache)\n    data = add_strict_prior_low_rate_features(data)\n    data = add_strict_prior_flow_features(data, flows)\n    data, preceding_audit = add_preceding_bus_segment_features(data, flows)\n    development = data.loc[data["date"].isin(DEVELOPMENT_DATES)].copy()\n    metrics, daily_metrics, importances = run_rolling_origin(development)\n    comparison = comparison_with_linear(metrics, args.linear_metrics)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    metrics.to_csv(args.output_dir / "metrics.csv", index=False)\n    daily_metrics.to_csv(args.output_dir / "daily_metrics.csv", index=False)\n    importances.to_csv(args.output_dir / "feature_importance.csv", index=False)\n    if not comparison.empty:\n        comparison.to_csv(args.output_dir / "linear_comparison.csv", index=False)\n    coverage = {\n        column: float(development[column].notna().mean())\n        for _, columns in FEATURE_BLOCKS\n        for column in columns\n    }\n    summary = {\n        "protocol": {\n            "development_dates": DEVELOPMENT_DATES,\n            "validation_dates": VALIDATION_DATES,\n            "historical_feature_rule": "strictly earlier calendar dates only",\n            "regression_target": "arrival_seats - snapshot_remaining_seats",\n            "regression_params": REGRESSOR_PARAMS,\n            "classification_params": CLASSIFIER_PARAMS,\n            "classification_balance": "per-fold event-weighted scale_pos_weight",\n            "classification_threshold": 0.5,\n            "evaluation_weighting": "equal total weight per arrival event",\n        },\n        "feature_sets": cumulative_feature_sets(),\n        "preceding_bus_segment_audit": preceding_audit,\n        "feature_coverage": coverage,\n        "metrics": metrics.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(metrics.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/main_model_registry.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom all_prearrival_seat_regression import DYNAMIC_ALL_PREARRIVAL\nfrom hypothesis_model_search import (\n    OBSERVED_SEAT_CEILING_PARAM,\n    EnsembleSpec,\n    add_observed_capacity_features,\n    decode_target,\n    postprocess_ensemble_prediction,\n)\nfrom linear_feature_experiment import _binary_rate_lookup, _flow_lookup\n\n\nMODEL_FAMILY = "arrival-seat-1000"\nMODEL_VERSION = "v1.0.0"\nMODEL_ID = f"{MODEL_FAMILY}/{MODEL_VERSION}"\nMODEL_CONTRACT_VERSION = 1\nVERSION_PATTERN = re.compile(r"^v(0|[1-9]\\d*)\\.(0|[1-9]\\d*)\\.(0|[1-9]\\d*)$")\nPRIMARY_FEATURES = (\n    "target_low_10_rate",\n    "target_low_rate_log_count",\n    "path_low_10_mean",\n    "path_low_10_sum",\n    "path_flow_mean",\n    "path_flow_sum",\n    "path_flow_std",\n    "path_flow_fallback_share",\n    "previous_bus_departure_age_minutes",\n)\nCOMPONENTS = ("hgb", "extra_trees", "lightgbm")\n\n\ndef validate_version(version: str) -> tuple[int, int, int]:\n    match = VERSION_PATTERN.fullmatch(version)\n    if match is None:\n        raise ValueError(f"모델 버전은 vMAJOR.MINOR.PATCH 형식이어야 합니다: {version}")\n    return tuple(int(value) for value in match.groups())\n\n\ndef file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as source:\n        for chunk in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef build_feature_profile(\n    data: pd.DataFrame,\n    flows: pd.DataFrame,\n    *,\n    training_cutoff_date: str,\n) -> dict[str, Any]:\n    cutoff = str(training_cutoff_date)\n    events = data.loc[data["date"].le(cutoff)].drop_duplicates("event_id").copy()\n    events["target_station_seq"] = pd.to_numeric(\n        events["station_seq_cat"], errors="raise"\n    ).astype(int)\n    events["arrival_hour"] = pd.to_datetime(events["event_time"]).dt.hour.astype(int)\n    events["is_low_10"] = events["label_seats"].le(10).astype(float)\n    event_history = events[\n        ["target_station_seq", "direction", "arrival_hour", "is_low_10"]\n    ].copy()\n    flow_history = flows.loc[flows["date"].le(cutoff), [\n        "date",\n        "station_seq",\n        "direction",\n        "time_bin_2h",\n        "stop_net",\n    ]].copy()\n    return {\n        "contract_version": MODEL_CONTRACT_VERSION,\n        "training_cutoff_date": cutoff,\n        "low_rate_alpha": 20.0,\n        "flow_alpha": 10.0,\n        "event_history": event_history,\n        "flow_history": flow_history,\n        "pass_node_sequences": sorted(\n            int(value) for value in flows.attrs.get("pass_node_sequences", [])\n        ),\n    }\n\n\ndef apply_feature_profile(\n    data: pd.DataFrame,\n    profile: Mapping[str, Any],\n    *,\n    enforce_after_cutoff: bool = True,\n) -> pd.DataFrame:\n    if int(profile.get("contract_version", -1)) != MODEL_CONTRACT_VERSION:\n        raise ValueError("지원하지 않는 주 모델 피처 프로파일 계약입니다.")\n    output = data.copy()\n    if "snapshot_time" not in output:\n        raise ValueError("snapshot_time 열이 필요합니다.")\n    output["snapshot_time"] = pd.to_datetime(output["snapshot_time"], errors="raise")\n    cutoff = str(profile["training_cutoff_date"])\n    if enforce_after_cutoff:\n        prediction_dates = output["snapshot_time"].dt.strftime("%Y-%m-%d")\n        if prediction_dates.le(cutoff).any():\n            raise ValueError(\n                "고정 프로파일은 학습 종료일 이후 추론에만 사용할 수 있습니다: "\n                f"cutoff={cutoff}"\n            )\n\n    events = profile["event_history"]\n    flows = profile["flow_history"]\n    if not isinstance(events, pd.DataFrame) or not isinstance(flows, pd.DataFrame):\n        raise TypeError("피처 프로파일의 history는 pandas DataFrame이어야 합니다.")\n    low_10 = _binary_rate_lookup(\n        events,\n        "is_low_10",\n        alpha=float(profile["low_rate_alpha"]),\n    )\n    flow = _flow_lookup(flows, alpha=float(profile["flow_alpha"]))\n    pass_nodes = {int(value) for value in profile.get("pass_node_sequences", [])}\n\n    records: list[dict[str, float]] = []\n    for row in output.itertuples(index=False):\n        current = int(row.snapshot_station_seq)\n        target = int(row.station_seq_cat)\n        direction = str(row.direction)\n        hour = int(pd.Timestamp(row.snapshot_time).hour)\n        target_rate, target_count = low_10(target, direction, hour)\n        path_stations = list(range(current + 1, target + 1))\n        path_rates = np.asarray(\n            [low_10(station, direction, hour)[0] for station in path_stations],\n            dtype=float,\n        )\n\n        flow_start = current if str(row.target_state_cat) == "1" else current + 1\n        flow_stations = [\n            station for station in range(flow_start, target) if station not in pass_nodes\n        ]\n        time_bin = hour // 2 * 2\n        looked_up = [flow(station, direction, time_bin) for station in flow_stations]\n        flow_values = np.asarray([value for value, _ in looked_up], dtype=float)\n        fallbacks = np.asarray([fallback for _, fallback in looked_up], dtype=float)\n        records.append(\n            {\n                "target_low_10_rate": float(target_rate),\n                "target_low_rate_log_count": float(np.log1p(target_count)),\n                "path_low_10_mean": (\n                    float(path_rates.mean()) if len(path_rates) else 0.0\n                ),\n                "path_low_10_sum": float(path_rates.sum()),\n                "path_flow_mean": (\n                    float(flow_values.mean()) if len(flow_values) else 0.0\n                ),\n                "path_flow_sum": float(flow_values.sum()),\n                "path_flow_std": (\n                    float(flow_values.std(ddof=0)) if len(flow_values) else 0.0\n                ),\n                "path_flow_fallback_share": (\n                    float(fallbacks.mean()) if len(fallbacks) else 1.0\n                ),\n            }\n        )\n    engineered = pd.DataFrame(records, index=output.index)\n    for column in engineered:\n        output[column] = engineered[column]\n    return output\n\n\ndef read_registry(registry_path: Path) -> dict[str, Any]:\n    registry = json.loads(registry_path.read_text(encoding="utf-8"))\n    if registry.get("model_family") != MODEL_FAMILY:\n        raise ValueError("다른 model family의 registry입니다.")\n    versions = registry.get("versions")\n    if not isinstance(versions, dict) or not versions:\n        raise ValueError("registry에 모델 버전이 없습니다.")\n    for version in versions:\n        validate_version(version)\n    primary = str(registry.get("primary_version", ""))\n    if primary not in versions:\n        raise ValueError("registry의 primary_version이 versions에 없습니다.")\n    return registry\n\n\ndef resolve_model_dir(registry_path: Path, version: str | None = None) -> Path:\n    registry = read_registry(registry_path)\n    selected = version or str(registry["primary_version"])\n    validate_version(selected)\n    if selected not in registry["versions"]:\n        raise ValueError(f"등록되지 않은 모델 버전입니다: {selected}")\n    relative = Path(str(registry["versions"][selected]["artifact_dir"]))\n    if relative.is_absolute() or ".." in relative.parts:\n        raise ValueError("artifact_dir은 registry 하위 상대경로여야 합니다.")\n    resolved = (registry_path.parent / relative).resolve()\n    if not resolved.is_dir():\n        raise FileNotFoundError(f"모델 artifact 디렉터리가 없습니다: {resolved}")\n    return resolved\n\n\n@dataclass\nclass VersionedMainModel:\n    model_id: str\n    manifest: dict[str, Any]\n    models: dict[str, Any]\n    feature_profile: dict[str, Any]\n\n    @classmethod\n    def load(cls, model_dir: Path) -> "VersionedMainModel":\n        manifest_path = model_dir / "manifest.json"\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        validate_version(str(manifest["version"]))\n        if manifest.get("model_family") != MODEL_FAMILY:\n            raise ValueError("모델 family가 다릅니다.")\n        if int(manifest.get("contract_version", -1)) != MODEL_CONTRACT_VERSION:\n            raise ValueError("지원하지 않는 모델 계약 버전입니다.")\n        declared = manifest.get("artifact_sha256")\n        if not isinstance(declared, dict) or not declared:\n            raise ValueError("artifact SHA manifest가 없습니다.")\n        expected_files = {*[f"{name}.joblib" for name in COMPONENTS], "feature_profile.joblib"}\n        if set(declared) != expected_files:\n            raise ValueError("artifact 파일 집합이 모델 계약과 다릅니다.")\n        for name, expected in declared.items():\n            path = model_dir / name\n            if not path.is_file() or file_sha256(path) != str(expected):\n                raise ValueError(f"모델 artifact SHA가 다릅니다: {name}")\n        models = {name: joblib.load(model_dir / f"{name}.joblib") for name in COMPONENTS}\n        profile = joblib.load(model_dir / "feature_profile.joblib")\n        return cls(\n            model_id=str(manifest["model_id"]),\n            manifest=manifest,\n            models=models,\n            feature_profile=profile,\n        )\n\n    @classmethod\n    def load_primary(cls, registry_path: Path) -> "VersionedMainModel":\n        return cls.load(resolve_model_dir(registry_path))\n\n    def prepare_features(self, data: pd.DataFrame) -> pd.DataFrame:\n        prepared = data.copy()\n        prepared["snapshot_time"] = pd.to_datetime(\n            prepared["snapshot_time"], errors="raise"\n        )\n        for column in DYNAMIC_ALL_PREARRIVAL.categorical:\n            if column in prepared:\n                prepared[column] = prepared[column].astype("string")\n        prepared = add_observed_capacity_features(prepared)\n        prepared = apply_feature_profile(prepared, self.feature_profile)\n        missing = sorted(set(self.manifest["feature_columns"]) - set(prepared.columns))\n        if missing:\n            raise ValueError(f"주 모델 입력 피처가 누락되었습니다: {missing}")\n        return prepared\n\n    def predict(self, data: pd.DataFrame) -> np.ndarray:\n        prepared = self.prepare_features(data)\n        columns = list(self.manifest["feature_columns"])\n        target_kinds = self.manifest["component_target_kinds"]\n        weights = self.manifest["ensemble_weights"]\n        prediction = np.zeros(len(prepared), dtype=float)\n        for name in COMPONENTS:\n            raw = self.models[name].predict(prepared[columns])\n            decoded = decode_target(raw, prepared, str(target_kinds[name]))\n            prediction += float(weights[name]) * decoded\n        prediction += float(self.manifest["deployment_bias"])\n        spec = EnsembleSpec(\n            name=str(self.manifest["model_id"]),\n            kind="weighted",\n            components=COMPONENTS,\n            params={**weights, OBSERVED_SEAT_CEILING_PARAM: 1.0},\n        )\n        return postprocess_ensemble_prediction(spec, prediction, prepared)\n\n    def predict_frame(self, data: pd.DataFrame) -> pd.DataFrame:\n        output = data.copy()\n        output["predicted_arrival_seats"] = self.predict(data)\n        output["predicted_arrival_seats_rounded"] = np.rint(\n            output["predicted_arrival_seats"]\n        ).astype(int)\n        output["prediction_model"] = self.model_id\n        output["prediction_model_version"] = str(self.manifest["version"])\n        return output\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="버전된 주 모델로 잔여좌석을 예측")\n    parser.add_argument("--registry", type=Path, required=True)\n    parser.add_argument("--input-csv", type=Path, required=True)\n    parser.add_argument("--output-csv", type=Path, required=True)\n    args = parser.parse_args()\n    model = VersionedMainModel.load_primary(args.registry)\n    data = pd.read_csv(args.input_csv)\n    output = model.predict_frame(data)\n    args.output_csv.parent.mkdir(parents=True, exist_ok=True)\n    output.to_csv(args.output_csv, index=False)\n    print(f"{model.model_id}: {len(output)} predictions")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/route_local_model.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport re\nimport sqlite3\nimport sys\nfrom dataclasses import asdict\nfrom datetime import timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom all_prearrival_seat_regression import (\n    build_all_prearrival_table,\n)\nfrom hypothesis_model_search import (\n    Candidate,\n    candidate_weights,\n    development_folds,\n    encode_target,\n    feature_frames,\n    load_analysis_data,\n    make_regressor,\n    run_candidate,\n)\nfrom model_feasibility import build_model_table, build_visits\nfrom tminus_feasibility import prepare_raw_locations\n\n\nDEFAULT_DB = Path("data/gbis_api_cache.sqlite3")\nDEFAULT_CACHE_DIR = Path("data/analysis_cache/route_local")\nDEFAULT_OUTPUT_DIR = Path("analysis/route_local_model_results")\nCACHE_VERSION = 1\nMIN_DATES = 3\nMIN_EVENTS_PER_DATE = 20\nMIN_SNAPSHOTS_PER_DATE = 100\nPARTIAL_FIRST_DAY_CUTOFF_HOUR = 7\n\n\ndef json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): json_ready(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(item) for item in value]\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return None if np.isnan(value) else float(value)\n    if isinstance(value, pd.Timestamp):\n        return value.isoformat()\n    if pd.isna(value):\n        return None\n    return value\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as source:\n        for chunk in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef load_route_names(path: Path) -> dict[str, str]:\n    if not path.exists():\n        return {}\n    names: dict[str, str] = {}\n    pattern = re.compile(r"^(\\d+)\\s*(?:#\\s*([^/\\s]+))?")\n    for raw_line in path.read_text(encoding="utf-8").splitlines():\n        match = pattern.match(raw_line.strip())\n        if match:\n            names[match.group(1)] = match.group(2) or match.group(1)\n    return names\n\n\ndef discover_recent_routes(\n    db_path: Path,\n    *,\n    recency_hours: float = 24.0,\n) -> list[str]:\n    uri = f"file:{db_path.resolve()}?mode=ro"\n    with sqlite3.connect(uri, uri=True) as connection:\n        rows = pd.read_sql_query(\n            """\n            SELECT route_id, station_count, last_collected_at\n            FROM routes\n            WHERE station_count > 0 AND last_collected_at IS NOT NULL\n            ORDER BY route_id\n            """,\n            connection,\n        )\n    if rows.empty:\n        return []\n    rows["last_collected_at"] = pd.to_datetime(rows["last_collected_at"])\n    latest = rows["last_collected_at"].max()\n    cutoff = latest - pd.Timedelta(hours=recency_hours)\n    return rows.loc[rows["last_collected_at"].ge(cutoff), "route_id"].tolist()\n\n\ndef route_fingerprint(db_path: Path, route_id: str) -> dict[str, Any]:\n    uri = f"file:{db_path.resolve()}?mode=ro"\n    with sqlite3.connect(uri, uri=True) as connection:\n        history = connection.execute(\n            """\n            SELECT COUNT(*), MIN(observed_at), MAX(observed_at)\n            FROM location_history\n            WHERE route_id = ?\n            """,\n            (route_id,),\n        ).fetchone()\n        stations = connection.execute(\n            "SELECT COUNT(*) FROM route_stations WHERE route_id = ?",\n            (route_id,),\n        ).fetchone()\n    return {\n        "cache_version": CACHE_VERSION,\n        "route_id": route_id,\n        "history_rows": int(history[0]),\n        "history_min": history[1],\n        "history_max": history[2],\n        "station_rows": int(stations[0]),\n    }\n\n\ndef route_cache_paths(cache_dir: Path, route_id: str) -> tuple[Path, Path]:\n    return (\n        cache_dir / f"{route_id}_arrival_A.pkl",\n        cache_dir / f"{route_id}_arrival_A.metadata.json",\n    )\n\n\ndef build_route_snapshots(\n    db_path: Path,\n    route_id: str,\n    cache_dir: Path,\n    *,\n    rebuild: bool,\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    cache_path, metadata_path = route_cache_paths(cache_dir, route_id)\n    fingerprint = route_fingerprint(db_path, route_id)\n    if not rebuild and cache_path.exists() and metadata_path.exists():\n        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if metadata.get("fingerprint") == fingerprint:\n            snapshots = pd.read_pickle(cache_path)\n            return snapshots, {**metadata, "cache_hit": True}\n\n    locations, stations = load_analysis_data(db_path, route_id)\n    visits = build_visits(locations, stations)\n    table, turnaround_seq = build_model_table(\n        visits,\n        stations,\n        label_target="arrival",\n    )\n    source = table.loc[\n        table["is_peak"]\n        & table["label_quality"].eq("A")\n        & table["label_seats"].notna()\n    ].copy()\n    if source.empty:\n        raise ValueError("출퇴근 시간대 직접 도착 라벨(A)이 없습니다.")\n\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n    snapshots = build_all_prearrival_table(\n        source,\n        visits,\n        by_vehicle,\n        turnaround_seq=turnaround_seq,\n        max_seq=int(stations["station_seq"].max()),\n    )\n    if snapshots.empty:\n        raise ValueError("도착 전 스냅샷을 만들 수 없습니다.")\n    snapshots["route_id"] = route_id\n    snapshots = snapshots.sort_values(\n        ["snapshot_time", "event_id"]\n    ).reset_index(drop=True)\n\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    snapshots.to_pickle(cache_path)\n    metadata = {\n        "fingerprint": fingerprint,\n        "cache_path": str(cache_path),\n        "snapshot_rows": int(len(snapshots)),\n        "events": int(snapshots["event_id"].nunique()),\n        "dates": sorted(snapshots["date"].astype(str).unique().tolist()),\n        "source_first_seen": locations["observed_at"].min().isoformat(),\n        "source_last_seen": locations["observed_at"].max().isoformat(),\n    }\n    metadata_path.write_text(\n        json.dumps(json_ready(metadata), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    return snapshots, {**metadata, "cache_hit": False}\n\n\ndef select_usable_dates(\n    snapshots: pd.DataFrame,\n    *,\n    source_first_seen: pd.Timestamp | str,\n    min_events_per_date: int = MIN_EVENTS_PER_DATE,\n    min_snapshots_per_date: int = MIN_SNAPSHOTS_PER_DATE,\n) -> tuple[list[str], pd.DataFrame]:\n    coverage = (\n        snapshots.groupby("date", sort=True)\n        .agg(\n            snapshots=("event_id", "size"),\n            events=("event_id", "nunique"),\n            low_0_5_events=("label_seats", lambda values: int(\n                values.groupby(snapshots.loc[values.index, "event_id"])\n                .first()\n                .le(5)\n                .sum()\n            )),\n        )\n        .reset_index()\n    )\n    coverage["date"] = coverage["date"].astype(str)\n    coverage["eligible"] = (\n        coverage["events"].ge(min_events_per_date)\n        & coverage["snapshots"].ge(min_snapshots_per_date)\n    )\n    coverage["exclusion_reason"] = np.where(\n        coverage["eligible"],\n        "",\n        "insufficient_events_or_snapshots",\n    )\n\n    first_seen = pd.Timestamp(source_first_seen)\n    first_date = first_seen.date().isoformat()\n    if first_seen.hour >= PARTIAL_FIRST_DAY_CUTOFF_HOUR:\n        partial = coverage["date"].eq(first_date)\n        coverage.loc[partial, "eligible"] = False\n        coverage.loc[partial, "exclusion_reason"] = "partial_first_collection_day"\n\n    usable = coverage.loc[coverage["eligible"], "date"].tolist()\n    return usable, coverage\n\n\ndef local_candidate() -> Candidate:\n    return Candidate(\n        name="route_local_hgb",\n        stage="route_local_v1",\n        why=(\n            "각 노선의 정류장 ID와 좌석 궤적을 다른 노선과 섞지 않고 학습해 "\n            "노선 고유 수요 패턴을 포착한다."\n        ),\n        if_works="현재 좌석 유지 기준선보다 날짜 순서 OOF MAE가 낮아진다.",\n        if_fails=(\n            "노선별 데이터 기간이 짧거나 현재 좌석이 이미 대부분의 정보를 담고 있다."\n        ),\n        model_kind="hgb",\n        target_kind="delta_per_stop",\n        feature_variant="dynamic",\n        low_weight=1.0,\n        weighting_kind="event",\n        params={\n            "loss": "absolute_error",\n            "learning_rate": 0.05,\n            "max_iter": 300,\n            "max_leaf_nodes": 15,\n            "min_samples_leaf": 20,\n            "l2_regularization": 1.0,\n            "early_stopping": False,\n        },\n    )\n\n\ndef persistence_candidate() -> Candidate:\n    return Candidate(\n        name="persistence",\n        stage="route_local_baseline",\n        why="도착 전 현재 잔여좌석을 그대로 유지하는 강한 물리 기준선이다.",\n        if_works="좌석 변화가 작아 학습 모델과 비슷하거나 더 정확하다.",\n        if_fails="남은 정류장에서 반복 가능한 승하차 패턴이 존재한다.",\n        model_kind="formula",\n    )\n\n\ndef fit_final_local_model(\n    candidate: Candidate,\n    data: pd.DataFrame,\n    *,\n    bias: float,\n    seed: int,\n) -> tuple[Any, dict[str, Any]]:\n    empty_target = data.iloc[:0].copy()\n    prepared_train, _, feature_set = feature_frames(\n        data,\n        empty_target,\n        candidate.feature_variant,\n        flows=pd.DataFrame({"date": pd.Series(dtype=str)}),\n    )\n    model = make_regressor(candidate, seed)\n    model.fit(\n        prepared_train[feature_set.columns],\n        encode_target(prepared_train, candidate.target_kind),\n        regressor__sample_weight=candidate_weights(\n            prepared_train,\n            candidate.low_weight,\n            weighting_kind=candidate.weighting_kind,\n            far_weight=candidate.far_weight,\n            far_threshold=candidate.far_threshold,\n            gap_weight_power=candidate.gap_weight_power,\n        ),\n    )\n    return model, {\n        "feature_set": feature_set.name,\n        "feature_columns": list(feature_set.columns),\n        "target_kind": candidate.target_kind,\n        "feature_variant": candidate.feature_variant,\n        "bias_correction": float(bias),\n    }\n\n\ndef evaluate_route(\n    snapshots: pd.DataFrame,\n    route_id: str,\n    route_name: str,\n    usable_dates: list[str],\n    output_dir: Path,\n    *,\n    seed: int,\n) -> tuple[list[dict[str, Any]], list[dict[str, Any]], pd.DataFrame, dict[str, Any]]:\n    data = snapshots.loc[snapshots["date"].isin(usable_dates)].copy()\n    folds = development_folds(data, tuple(usable_dates))\n    empty_flows = pd.DataFrame({"date": pd.Series(dtype=str)})\n    candidates = (persistence_candidate(), local_candidate())\n    metrics: list[dict[str, Any]] = []\n    daily: list[dict[str, Any]] = []\n    oof_frames: list[pd.DataFrame] = []\n    aggregate_by_name: dict[str, dict[str, Any]] = {}\n\n    for candidate in candidates:\n        aggregate, oof, fold_rows = run_candidate(\n            candidate,\n            folds,\n            empty_flows,\n            seed=seed,\n        )\n        aggregate.update({"route_id": route_id, "route_name": route_name})\n        metrics.append(aggregate)\n        aggregate_by_name[candidate.name] = aggregate\n        for row in fold_rows:\n            daily.append({**row, "route_id": route_id, "route_name": route_name})\n        oof["route_id"] = route_id\n        oof["route_name"] = route_name\n        oof_frames.append(oof)\n\n    baseline = aggregate_by_name["persistence"]\n    learned = aggregate_by_name["route_local_hgb"]\n    improvement = float(\n        baseline["event_balanced_mae"] - learned["event_balanced_mae"]\n    )\n    model, model_metadata = fit_final_local_model(\n        local_candidate(),\n        data,\n        bias=float(learned["bias_correction"]),\n        seed=seed,\n    )\n    model_path = output_dir / "models" / f"{route_id}_route_local_hgb.joblib"\n    metadata_path = model_path.with_suffix(".metadata.json")\n    model_path.parent.mkdir(parents=True, exist_ok=True)\n    joblib.dump(model, model_path)\n    artifact_metadata = {\n        "route_id": route_id,\n        "route_name": route_name,\n        "candidate": asdict(local_candidate()),\n        "training_dates": usable_dates,\n        "training_rows": int(len(data)),\n        "training_events": int(data["event_id"].nunique()),\n        "rolling_validation_dates": usable_dates[1:],\n        "oof_event_balanced_mae": float(learned["event_balanced_mae"]),\n        "persistence_event_balanced_mae": float(baseline["event_balanced_mae"]),\n        "persistence_minus_model_mae": improvement,\n        "beats_persistence": improvement > 0,\n        "model": model_metadata,\n        "artifact": str(model_path),\n        "artifact_sha256": sha256(model_path),\n    }\n    metadata_path.write_text(\n        json.dumps(json_ready(artifact_metadata), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    return metrics, daily, pd.concat(oof_frames, ignore_index=True), artifact_metadata\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="노선별 데이터만 사용하는 도착 잔여좌석 HGB 실험"\n    )\n    parser.add_argument("--db", type=Path, default=DEFAULT_DB)\n    parser.add_argument("--route-ids", nargs="*")\n    parser.add_argument(\n        "--routes-file",\n        type=Path,\n        default=Path("collector/config/routes.txt"),\n    )\n    parser.add_argument("--cache-dir", type=Path, default=DEFAULT_CACHE_DIR)\n    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)\n    parser.add_argument("--rebuild-cache", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    route_ids = args.route_ids or discover_recent_routes(args.db)\n    if not route_ids:\n        raise ValueError("최근 수집 중이며 정류장 메타데이터가 있는 노선이 없습니다.")\n    route_names = load_route_names(args.routes_file)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    all_metrics: list[dict[str, Any]] = []\n    all_daily: list[dict[str, Any]] = []\n    all_oof: list[pd.DataFrame] = []\n    all_coverage: list[pd.DataFrame] = []\n    route_summaries: list[dict[str, Any]] = []\n    skipped: list[dict[str, str]] = []\n\n    for route_id in route_ids:\n        route_name = route_names.get(route_id, route_id)\n        print(f"[{route_name}/{route_id}] 스냅샷 준비", flush=True)\n        try:\n            snapshots, cache_metadata = build_route_snapshots(\n                args.db,\n                route_id,\n                args.cache_dir,\n                rebuild=args.rebuild_cache,\n            )\n            usable_dates, coverage = select_usable_dates(\n                snapshots,\n                source_first_seen=cache_metadata["source_first_seen"],\n            )\n            coverage["route_id"] = route_id\n            coverage["route_name"] = route_name\n            all_coverage.append(coverage)\n            if len(usable_dates) < MIN_DATES:\n                raise ValueError(\n                    f"완전 수집일이 {len(usable_dates)}일뿐이라 최소 {MIN_DATES}일에 미달합니다."\n                )\n            print(\n                f"[{route_name}/{route_id}] {len(usable_dates)}일 rolling 평가",\n                flush=True,\n            )\n            metrics, daily, oof, route_summary = evaluate_route(\n                snapshots,\n                route_id,\n                route_name,\n                usable_dates,\n                args.output_dir,\n                seed=args.seed,\n            )\n            all_metrics.extend(metrics)\n            all_daily.extend(daily)\n            all_oof.append(oof)\n            route_summaries.append(route_summary)\n        except Exception as error:\n            skipped.append(\n                {\n                    "route_id": route_id,\n                    "route_name": route_name,\n                    "reason": f"{type(error).__name__}: {error}",\n                }\n            )\n            print(f"[{route_name}/{route_id}] 제외: {error}", file=sys.stderr)\n\n    if not route_summaries:\n        raise RuntimeError("평가와 학습을 완료한 노선이 없습니다.")\n\n    metrics_frame = pd.DataFrame(all_metrics)\n    daily_frame = pd.DataFrame(all_daily)\n    coverage_frame = pd.concat(all_coverage, ignore_index=True)\n    oof_frame = pd.concat(all_oof, ignore_index=True)\n    metrics_frame.to_csv(args.output_dir / "metrics.csv", index=False)\n    daily_frame.to_csv(args.output_dir / "daily_metrics.csv", index=False)\n    coverage_frame.to_csv(args.output_dir / "data_coverage.csv", index=False)\n    oof_frame.to_csv(\n        args.output_dir / "oof_predictions.csv.gz",\n        index=False,\n        compression="gzip",\n    )\n\n    completed = pd.DataFrame(route_summaries)\n    summary = {\n        "hypothesis": (\n            "A model trained only within one route can learn route/station-specific "\n            "seat-change patterns that beat current-seat persistence."\n        ),\n        "protocol": {\n            "route_isolation": "one independent model per route; no cross-route rows",\n            "label": "direct arrival seat observation (quality A)",\n            "scope": "peak-hour pre-arrival snapshots",\n            "partial_first_day_rule": (\n                f"exclude first collection date when collection starts at or after "\n                f"{PARTIAL_FIRST_DAY_CUTOFF_HOUR:02d}:00"\n            ),\n            "date_eligibility": {\n                "minimum_events": MIN_EVENTS_PER_DATE,\n                "minimum_snapshots": MIN_SNAPSHOTS_PER_DATE,\n                "minimum_dates": MIN_DATES,\n            },\n            "validation": "rolling origin; each validation date uses earlier usable dates only",\n            "baseline": "current remaining seats persistence",\n            "model": asdict(local_candidate()),\n        },\n        "routes_requested": route_ids,\n        "routes_completed": completed["route_id"].tolist(),\n        "routes_skipped": skipped,\n        "route_results": route_summaries,\n        "routes_beating_persistence": completed.loc[\n            completed["beats_persistence"], "route_id"\n        ].tolist(),\n        "artifacts": {\n            "metrics": str(args.output_dir / "metrics.csv"),\n            "daily_metrics": str(args.output_dir / "daily_metrics.csv"),\n            "data_coverage": str(args.output_dir / "data_coverage.csv"),\n            "oof_predictions": str(args.output_dir / "oof_predictions.csv.gz"),\n            "models": str(args.output_dir / "models"),\n        },\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(\n        f"완료: {len(route_summaries)}개 노선, 제외 {len(skipped)}개",\n        flush=True,\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/pooled_main_model_registry.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom all_prearrival_seat_regression import DYNAMIC_ALL_PREARRIVAL\nfrom hypothesis_model_search import clip_seats, decode_target\nfrom linear_feature_experiment import _binary_rate_lookup, _flow_lookup\nfrom main_model_registry import COMPONENTS, file_sha256, validate_version\n\n\nMODEL_FAMILY = "arrival-seat-pooled"\nMODEL_VERSION = "v1.0.0"\nMODEL_ID = f"{MODEL_FAMILY}/{MODEL_VERSION}"\nMODEL_CONTRACT_VERSION = 1\n\n\ndef build_pooled_feature_profile(\n    data: pd.DataFrame,\n    route_flows: Mapping[str, pd.DataFrame],\n    *,\n    training_cutoff_date: str,\n) -> dict[str, Any]:\n    cutoff = str(training_cutoff_date)\n    profiles: dict[str, dict[str, Any]] = {}\n    for route_code, flow_history in route_flows.items():\n        route = data.loc[\n            data["route_code"].astype("string").eq(str(route_code))\n            & data["date"].le(cutoff)\n        ]\n        events = route.drop_duplicates("event_id").copy()\n        if events.empty:\n            raise ValueError(f"노선 {route_code}의 profile 학습 사건이 없습니다.")\n        events["target_station_seq"] = pd.to_numeric(\n            events["station_seq_cat"], errors="raise"\n        ).astype(int)\n        events["arrival_hour"] = pd.to_datetime(events["event_time"]).dt.hour.astype(int)\n        events["is_low_10"] = events["label_seats"].le(10).astype(float)\n        flows = flow_history.loc[\n            flow_history["date"].le(cutoff),\n            ["date", "station_seq", "direction", "time_bin_2h", "stop_net"],\n        ].copy()\n        profiles[str(route_code)] = {\n            "event_history": events[\n                ["target_station_seq", "direction", "arrival_hour", "is_low_10"]\n            ].copy(),\n            "flow_history": flows,\n            "pass_node_sequences": sorted(\n                int(value)\n                for value in flow_history.attrs.get("pass_node_sequences", [])\n            ),\n        }\n    return {\n        "contract_version": MODEL_CONTRACT_VERSION,\n        "training_cutoff_date": cutoff,\n        "low_rate_alpha": 20.0,\n        "flow_alpha": 10.0,\n        "routes": profiles,\n    }\n\n\ndef apply_pooled_feature_profile(\n    data: pd.DataFrame,\n    profile: Mapping[str, Any],\n    *,\n    enforce_after_cutoff: bool = True,\n) -> pd.DataFrame:\n    if int(profile.get("contract_version", -1)) != MODEL_CONTRACT_VERSION:\n        raise ValueError("지원하지 않는 통합 주 모델 피처 프로파일 계약입니다.")\n    output = data.copy()\n    if "route_code" not in output:\n        if "route_id" not in output:\n            raise ValueError("route_code 또는 route_id 열이 필요합니다.")\n        output["route_code"] = output["route_id"]\n    output["route_code"] = output["route_code"].astype("string")\n    output["snapshot_time"] = pd.to_datetime(output["snapshot_time"], errors="raise")\n    cutoff = str(profile["training_cutoff_date"])\n    if enforce_after_cutoff:\n        prediction_dates = output["snapshot_time"].dt.strftime("%Y-%m-%d")\n        if prediction_dates.le(cutoff).any():\n            raise ValueError(\n                "고정 통합 프로파일은 학습 종료일 이후 추론에만 사용할 수 있습니다: "\n                f"cutoff={cutoff}"\n            )\n\n    route_profiles = profile.get("routes")\n    if not isinstance(route_profiles, Mapping) or not route_profiles:\n        raise ValueError("통합 프로파일에 노선별 이력이 없습니다.")\n    unknown = sorted(set(output["route_code"].dropna()) - set(route_profiles))\n    if output["route_code"].isna().any() or unknown:\n        raise ValueError(f"통합 주 모델이 지원하지 않는 노선입니다: {unknown}")\n\n    engineered = pd.DataFrame(index=output.index)\n    for route_code, index in output.groupby("route_code", sort=False).groups.items():\n        route_profile = route_profiles[str(route_code)]\n        events = route_profile["event_history"]\n        flows = route_profile["flow_history"]\n        if not isinstance(events, pd.DataFrame) or not isinstance(flows, pd.DataFrame):\n            raise TypeError("노선별 history는 pandas DataFrame이어야 합니다.")\n        low_10 = _binary_rate_lookup(\n            events,\n            "is_low_10",\n            alpha=float(profile["low_rate_alpha"]),\n        )\n        flow = _flow_lookup(flows, alpha=float(profile["flow_alpha"]))\n        pass_nodes = {\n            int(value) for value in route_profile.get("pass_node_sequences", [])\n        }\n        records: list[dict[str, float]] = []\n        selected = output.loc[index]\n        for row in selected.itertuples(index=False):\n            current = int(row.snapshot_station_seq)\n            target = int(row.station_seq_cat)\n            direction = str(row.direction)\n            hour = int(pd.Timestamp(row.snapshot_time).hour)\n            target_rate, target_count = low_10(target, direction, hour)\n            path_stations = list(range(current + 1, target + 1))\n            path_rates = np.asarray(\n                [low_10(station, direction, hour)[0] for station in path_stations],\n                dtype=float,\n            )\n            flow_start = current if str(row.target_state_cat) == "1" else current + 1\n            flow_stations = [\n                station\n                for station in range(flow_start, target)\n                if station not in pass_nodes\n            ]\n            time_bin = hour // 2 * 2\n            looked_up = [flow(station, direction, time_bin) for station in flow_stations]\n            flow_values = np.asarray([value for value, _ in looked_up], dtype=float)\n            fallbacks = np.asarray([fallback for _, fallback in looked_up], dtype=float)\n            records.append(\n                {\n                    "target_low_10_rate": float(target_rate),\n                    "target_low_rate_log_count": float(np.log1p(target_count)),\n                    "path_low_10_mean": float(path_rates.mean()) if len(path_rates) else 0.0,\n                    "path_low_10_sum": float(path_rates.sum()),\n                    "path_flow_mean": float(flow_values.mean()) if len(flow_values) else 0.0,\n                    "path_flow_sum": float(flow_values.sum()),\n                    "path_flow_std": float(flow_values.std(ddof=0)) if len(flow_values) else 0.0,\n                    "path_flow_fallback_share": float(fallbacks.mean()) if len(fallbacks) else 1.0,\n                }\n            )\n        engineered.loc[index, list(records[0])] = pd.DataFrame(\n            records, index=index\n        ).to_numpy()\n    for column in engineered:\n        output[column] = engineered[column]\n    return output\n\n\ndef read_registry(registry_path: Path) -> dict[str, Any]:\n    registry = json.loads(registry_path.read_text(encoding="utf-8"))\n    if registry.get("model_family") != MODEL_FAMILY:\n        raise ValueError("다른 model family의 registry입니다.")\n    versions = registry.get("versions")\n    if not isinstance(versions, dict) or not versions:\n        raise ValueError("registry에 모델 버전이 없습니다.")\n    for version in versions:\n        validate_version(version)\n    primary = str(registry.get("primary_version", ""))\n    if primary not in versions:\n        raise ValueError("registry의 primary_version이 versions에 없습니다.")\n    return registry\n\n\ndef resolve_model_dir(registry_path: Path, version: str | None = None) -> Path:\n    registry = read_registry(registry_path)\n    selected = version or str(registry["primary_version"])\n    validate_version(selected)\n    if selected not in registry["versions"]:\n        raise ValueError(f"등록되지 않은 모델 버전입니다: {selected}")\n    relative = Path(str(registry["versions"][selected]["artifact_dir"]))\n    if relative.is_absolute() or ".." in relative.parts:\n        raise ValueError("artifact_dir은 registry 하위 상대경로여야 합니다.")\n    model_dir = (registry_path.parent / relative).resolve()\n    if not model_dir.is_dir():\n        raise FileNotFoundError(f"모델 artifact 디렉터리가 없습니다: {model_dir}")\n    return model_dir\n\n\n@dataclass\nclass PooledVersionedMainModel:\n    model_id: str\n    manifest: dict[str, Any]\n    models: dict[str, Any]\n    feature_profile: dict[str, Any]\n\n    @classmethod\n    def load(cls, model_dir: Path) -> "PooledVersionedMainModel":\n        manifest = json.loads((model_dir / "manifest.json").read_text(encoding="utf-8"))\n        validate_version(str(manifest["version"]))\n        if manifest.get("model_family") != MODEL_FAMILY:\n            raise ValueError("모델 family가 다릅니다.")\n        if int(manifest.get("contract_version", -1)) != MODEL_CONTRACT_VERSION:\n            raise ValueError("지원하지 않는 통합 모델 계약입니다.")\n        declared = manifest.get("artifact_sha256")\n        expected = {*[f"{name}.joblib" for name in COMPONENTS], "feature_profile.joblib"}\n        if not isinstance(declared, dict) or set(declared) != expected:\n            raise ValueError("artifact 파일 집합이 통합 모델 계약과 다릅니다.")\n        for name, digest in declared.items():\n            path = model_dir / name\n            if not path.is_file() or file_sha256(path) != str(digest):\n                raise ValueError(f"모델 artifact SHA가 다릅니다: {name}")\n        return cls(\n            model_id=str(manifest["model_id"]),\n            manifest=manifest,\n            models={name: joblib.load(model_dir / f"{name}.joblib") for name in COMPONENTS},\n            feature_profile=joblib.load(model_dir / "feature_profile.joblib"),\n        )\n\n    @classmethod\n    def load_primary(cls, registry_path: Path) -> "PooledVersionedMainModel":\n        return cls.load(resolve_model_dir(registry_path))\n\n    def prepare_features(self, data: pd.DataFrame) -> pd.DataFrame:\n        prepared = data.copy()\n        prepared["snapshot_time"] = pd.to_datetime(prepared["snapshot_time"], errors="raise")\n        for column in DYNAMIC_ALL_PREARRIVAL.categorical:\n            if column in prepared:\n                prepared[column] = prepared[column].astype("string")\n        prepared = apply_pooled_feature_profile(prepared, self.feature_profile)\n        missing = sorted(set(self.manifest["feature_columns"]) - set(prepared.columns))\n        if missing:\n            raise ValueError(f"통합 주 모델 입력 피처가 누락되었습니다: {missing}")\n        return prepared\n\n    def predict(self, data: pd.DataFrame) -> np.ndarray:\n        prepared = self.prepare_features(data)\n        columns = list(self.manifest["feature_columns"])\n        target_kinds = self.manifest["component_target_kinds"]\n        weights = self.manifest["ensemble_weights"]\n        prediction = np.zeros(len(prepared), dtype=float)\n        for name in COMPONENTS:\n            raw = self.models[name].predict(prepared[columns])\n            prediction += float(weights[name]) * decode_target(\n                raw, prepared, str(target_kinds[name])\n            )\n        # No Route-1000 44/70 empirical correction and no fitted output offset.\n        # Nominal capacity bounds are part of the target support, not calibration.\n        return clip_seats(prediction, prepared["capacity"])\n\n    def predict_frame(self, data: pd.DataFrame) -> pd.DataFrame:\n        output = data.copy()\n        output["predicted_arrival_seats"] = self.predict(data)\n        output["predicted_arrival_seats_rounded"] = np.rint(\n            output["predicted_arrival_seats"]\n        ).astype(int)\n        output["prediction_model"] = self.model_id\n        output["prediction_model_version"] = str(self.manifest["version"])\n        return output\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="버전된 전체 노선 통합 주 모델로 예측")\n    parser.add_argument("--registry", type=Path, required=True)\n    parser.add_argument("--input-csv", type=Path, required=True)\n    parser.add_argument("--output-csv", type=Path, required=True)\n    args = parser.parse_args()\n    model = PooledVersionedMainModel.load_primary(args.registry)\n    output = model.predict_frame(pd.read_csv(args.input_csv))\n    args.output_csv.parent.mkdir(parents=True, exist_ok=True)\n    output.to_csv(args.output_csv, index=False)\n    print(f"{model.model_id}: {len(output)} predictions")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/build_insight_report.py': '"""Build the three reproducible matplotlib figures used in the insight report.\n\nThe script deliberately reads frozen experiment artifacts rather than retraining\nmodels, so the report remains a faithful record of the completed OOF analyses.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\n\nROUTE_NAMES = {\n    "200000104": "3000", "218000010": "1500", "219000013": "1000",\n    "219000016": "1200", "222000074": "1100", "222000075": "2000",\n}\n\n\ndef setup_style() -> None:\n    plt.rcParams.update({"font.family": "Noto Sans CJK KR", "axes.unicode_minus": False})\n\n\ndef complete_route_events(cache_dir: Path) -> pd.DataFrame:\n    frames = []\n    for route_id in ROUTE_NAMES:\n        data = pd.read_pickle(cache_dir / f"{route_id}_snapshots.pkl")\n        data["route_id"] = route_id\n        frames.append(data)\n    output = pd.concat(frames, ignore_index=True)\n    # The newest date is still in progress. Keep it separate from the completed\n    # days used for distribution interpretation.\n    complete_last_date = sorted(output["date"].astype(str).unique())[-2]\n    output = output.loc[output["date"].astype(str).le(complete_last_date)].copy()\n    output["arrival_hour"] = pd.to_datetime(output["event_time"]).dt.hour\n    return output.drop_duplicates(["route_id", "event_id"])\n\n\ndef feature_distribution_figure(data: pd.DataFrame, output: Path) -> None:\n    fig, axes = plt.subplots(2, 3, figsize=(17, 11), constrained_layout=True)\n    tables = []\n    for route_id, route_name in ROUTE_NAMES.items():\n        events = data.loc[data["route_id"].eq(route_id)].copy()\n        events["station_seq"] = pd.to_numeric(events["station_seq_cat"], errors="raise")\n        events["is_low"] = events["label_seats"].le(5)\n        table = events.groupby(["station_seq", "arrival_hour"], observed=True).agg(\n            count=("event_id", "size"), low=("is_low", "sum")\n        )\n        table["rate"] = table["low"] / table["count"] * 100\n        tables.append(table.loc[table["count"].ge(10), "rate"])\n    valid = pd.concat(tables).to_numpy(float)\n    color_ceiling = max(float(np.nanquantile(valid, 0.95)), 1.0)\n    image = None\n    for axis, (route_id, route_name) in zip(axes.flat, ROUTE_NAMES.items(), strict=True):\n        events = data.loc[data["route_id"].eq(route_id)].copy()\n        events["station_seq"] = pd.to_numeric(events["station_seq_cat"], errors="raise")\n        events["is_low"] = events["label_seats"].le(5)\n        table = events.groupby(["station_seq", "arrival_hour"], observed=True).agg(\n            count=("event_id", "size"), low=("is_low", "sum")\n        )\n        table["rate"] = (table["low"] / table["count"] * 100).where(table["count"].ge(10))\n        matrix = table["rate"].unstack("arrival_hour")\n        image = axis.imshow(matrix, aspect="auto", cmap="YlOrRd", vmin=0, vmax=color_ceiling)\n        axis.set_title(f"{route_name}번", fontweight="bold")\n        axis.set_xlabel("실제 도착 시간")\n        axis.set_ylabel("목표 정류장 순번")\n        axis.set_xticks(np.arange(len(matrix.columns)), [f"{int(hour):02d}" for hour in matrix.columns])\n        tick_count = min(8, len(matrix.index))\n        ticks = np.linspace(0, len(matrix.index) - 1, tick_count, dtype=int)\n        axis.set_yticks(ticks, matrix.index.to_numpy()[ticks])\n    assert image is not None\n    colorbar = fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.75, pad=0.02)\n    colorbar.set_label("도착 저잔여율 (0–5석, %) · 표본 10건 미만 제외")\n    fig.suptitle("노선별 목표 정류장 × 도착 시간대 저잔여율", fontsize=17, fontweight="bold")\n    fig.savefig(output, dpi=200, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef spatial_map_figure(data: pd.DataFrame, output: Path) -> None:\n    stations = data.groupby(["x", "y"], as_index=False).agg(\n        mean_arrival_seats=("label_seats", "mean"),\n        events=("event_id", "size"),\n    )\n    fig, ax = plt.subplots(figsize=(10.5, 9), constrained_layout=True)\n    scatter = ax.scatter(\n        stations["x"], stations["y"], c=stations["mean_arrival_seats"],\n        s=np.clip(np.sqrt(stations["events"]) * 3.5, 20, 115),\n        cmap="RdYlBu", vmin=0, vmax=70, edgecolors="white", linewidths=0.25,\n    )\n    colorbar = fig.colorbar(scatter, ax=ax, shrink=0.82)\n    colorbar.set_label("좌표별 평균 도착 잔여좌석 (석)")\n    ax.set_xlabel("x (경도)")\n    ax.set_ylabel("y (위도)")\n    ax.set_title("x, y 좌표별 평균 도착 잔여좌석\\n점 크기 = 도착 사건 수", fontweight="bold")\n    ax.grid(alpha=0.2)\n    fig.savefig(output, dpi=200, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="인사이트 보고서용 pyplot figure 생성")\n    parser.add_argument("--output-dir", type=Path, default=Path("reports/insight_report_assets"))\n    args = parser.parse_args()\n    setup_style()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    data = complete_route_events(Path("data/analysis_cache/route_specific_features"))\n    feature_distribution_figure(data, args.output_dir / "01_station_time_heatmaps.png")\n    spatial_map_figure(data, args.output_dir / "02_spatial_mean_arrival_seats.png")\n    print(args.output_dir)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/high_risk_feasibility.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\nfrom model_feasibility import (\n    PLANNING,\n    REALTIME,\n    build_model_table,\n    build_visits,\n    cluster_bootstrap_summary,\n    evaluate_final_split,\n    json_ready,\n    load_data,\n    prepare_subset,\n    trip_level_diagnostics,\n)\n\n\nROUTE_ID = "219000013"\nROUTE_NAME = "1000"\n\n# 이 범위는 최종 테스트일(8월 7일)을 보기 전에 8월 4~5일 만차 분포로 고정했다.\n# 오전은 고양→서울의 만차 집중 구간, 오후는 서울 회차 후 고양 방향 구간이다.\nGATE_CONFIG = {\n    "weekday_only": True,\n    "morning": {"hours": [6, 7, 8], "station_seq_min": 11, "station_seq_max": 20},\n    "evening": {"hours": [17, 18, 19], "station_seq_min": 29, "station_seq_max": 40},\n}\n\n\ndef high_risk_mask(data: pd.DataFrame) -> pd.Series:\n    weekday = data["event_time"].dt.dayofweek.lt(5)\n    morning = data["hour"].isin(GATE_CONFIG["morning"]["hours"]) & data[\n        "station_seq"\n    ].between(\n        GATE_CONFIG["morning"]["station_seq_min"],\n        GATE_CONFIG["morning"]["station_seq_max"],\n    )\n    evening = data["hour"].isin(GATE_CONFIG["evening"]["hours"]) & data[\n        "station_seq"\n    ].between(\n        GATE_CONFIG["evening"]["station_seq_min"],\n        GATE_CONFIG["evening"]["station_seq_max"],\n    )\n    return weekday & (morning | evening)\n\n\ndef gate_by_date(data: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    mask = high_risk_mask(data)\n    for date, day in data.groupby("date", sort=True):\n        gated = day.loc[mask.loc[day.index]]\n        positives = int(day["is_full"].sum())\n        gated_positives = int(gated["is_full"].sum())\n        rows.append(\n            {\n                "date": date,\n                "all_rows": int(len(day)),\n                "all_positives": positives,\n                "all_prevalence": float(day["is_full"].mean()),\n                "gated_rows": int(len(gated)),\n                "gated_positives": gated_positives,\n                "gate_coverage": float(len(gated) / len(day)),\n                "gated_prevalence": float(gated["is_full"].mean()) if len(gated) else 0.0,\n                "gate_positive_recall": (\n                    float(gated_positives / positives) if positives else None\n                ),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef selected_row(metrics: pd.DataFrame, feature_set: str) -> pd.Series:\n    candidates = metrics.loc[\n        metrics["feature_set"].eq(feature_set)\n        & metrics["model"].ne("historical_rate")\n    ]\n    return candidates.sort_values(\n        ["selection_average_precision", "brier"], ascending=[False, True]\n    ).iloc[0]\n\n\ndef comparison_table(\n    full_metrics: pd.DataFrame,\n    gated_metrics: pd.DataFrame,\n) -> pd.DataFrame:\n    rows = []\n    for feature_set in [PLANNING.name, REALTIME.name]:\n        for scope, metrics in [("full_route", full_metrics), ("high_risk_gate", gated_metrics)]:\n            selected = selected_row(metrics, feature_set)\n            rows.append(\n                {\n                    "scope": scope,\n                    "feature_set": feature_set,\n                    "selected_model": selected["model"],\n                    "test_rows": int(selected["rows"]),\n                    "test_positives": int(selected["positives"]),\n                    "prevalence": float(selected["prevalence"]),\n                    "selection_average_precision": float(\n                        selected["selection_average_precision"]\n                    ),\n                    "test_average_precision": float(selected["average_precision"]),\n                    "test_brier": float(selected["brier"]),\n                    "precision": float(selected["precision_at_threshold"]),\n                    "recall": float(selected["recall_at_threshold"]),\n                    "alert_rate": float(selected["alert_rate"]),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef make_comparison_plot(comparison: pd.DataFrame, output: Path) -> None:\n    labels = [\n        f"{row.feature_set}\\n{row.scope}"\n        for row in comparison.itertuples(index=False)\n    ]\n    x = range(len(comparison))\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    axes[0].bar(x, comparison["precision"] * 100, color=["#94a3b8", "#2563eb"] * 2)\n    axes[0].set_xticks(list(x), labels, rotation=15, ha="right")\n    axes[0].set_ylabel("Precision (%)")\n    axes[0].set_title("Precision at calibration-selected threshold")\n    axes[0].axhline(30, color="#dc2626", linestyle="--", linewidth=1, label="30% target")\n    axes[0].legend()\n\n    axes[1].bar(x, comparison["recall"] * 100, color=["#94a3b8", "#2563eb"] * 2)\n    axes[1].set_xticks(list(x), labels, rotation=15, ha="right")\n    axes[1].set_ylabel("Recall (%)")\n    axes[1].set_title("Recall at calibration-selected threshold")\n    axes[1].axhline(50, color="#dc2626", linestyle="--", linewidth=1, label="50% target")\n    axes[1].legend()\n    fig.tight_layout()\n    fig.savefig(output, dpi=160)\n    plt.close(fig)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="1000번 고위험 구간 전용 모델 검증")\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument(\n        "--output-dir", type=Path, default=Path("analysis/high_risk_results")\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    locations, stations = load_data(args.db, ROUTE_ID)\n    visits = build_visits(locations, stations)\n    table, _ = build_model_table(visits, stations)\n    data = prepare_subset(table)\n    gated = data.loc[high_risk_mask(data)].copy().reset_index(drop=True)\n\n    full_metrics, _, _ = evaluate_final_split(\n        data, [PLANNING, REALTIME], args.seed\n    )\n    gated_metrics, gated_fitted, gated_predictions = evaluate_final_split(\n        gated, [PLANNING, REALTIME], args.seed\n    )\n    strict_gated = gated.loc[gated["label_quality"].eq("A")].copy()\n    strict_metrics, _, _ = evaluate_final_split(\n        strict_gated, [PLANNING, REALTIME], args.seed\n    )\n    gate_daily = gate_by_date(data)\n    comparison = comparison_table(full_metrics, gated_metrics)\n\n    test = gated.loc[gated["date"].eq("2026-08-07")].copy()\n    selected_details: dict[str, Any] = {}\n    baseline_predictions = gated_predictions[("planning", "historical_rate")]\n    for feature_set in [PLANNING.name, REALTIME.name]:\n        selected = selected_row(gated_metrics, feature_set)\n        key = (feature_set, str(selected["model"]))\n        selected_details[feature_set] = {\n            "metrics": json_ready(selected.to_dict()),\n            "trip_diagnostics": trip_level_diagnostics(\n                test,\n                gated_predictions[key],\n                float(selected["threshold"]),\n            ),\n            "trip_cluster_bootstrap": cluster_bootstrap_summary(\n                test,\n                gated_predictions[key],\n                baseline_predictions,\n                args.seed,\n            ),\n        }\n\n    gate_daily.to_csv(args.output_dir / "gate_by_date.csv", index=False)\n    gated_metrics.to_csv(args.output_dir / "gated_final_metrics.csv", index=False)\n    strict_metrics.to_csv(args.output_dir / "gated_strict_A_metrics.csv", index=False)\n    comparison.to_csv(args.output_dir / "scope_comparison.csv", index=False)\n    make_comparison_plot(comparison, args.output_dir / "precision_recall_comparison.png")\n\n    test_gate = gate_daily.loc[gate_daily["date"].eq("2026-08-07")].iloc[0]\n    planning_selected = selected_row(gated_metrics, PLANNING.name)\n    realtime_selected = selected_row(gated_metrics, REALTIME.name)\n    result = {\n        "route_id": ROUTE_ID,\n        "route_name": ROUTE_NAME,\n        "gate_config": GATE_CONFIG,\n        "gate_selection_period": ["2026-08-04", "2026-08-05"],\n        "calibration_date": "2026-08-06",\n        "untouched_test_date": "2026-08-07",\n        "test_gate": json_ready(test_gate.to_dict()),\n        "selected_models": selected_details,\n        "strict_A_selected": {\n            feature_set: json_ready(selected_row(strict_metrics, feature_set).to_dict())\n            for feature_set in [PLANNING.name, REALTIME.name]\n        },\n        "product_kpi": {"minimum_precision": 0.30, "minimum_recall": 0.50},\n        "verdict": {\n            "planning_passes_kpi": bool(\n                planning_selected["precision_at_threshold"] >= 0.30\n                and planning_selected["recall_at_threshold"] >= 0.50\n            ),\n            "realtime_passes_kpi": bool(\n                realtime_selected["precision_at_threshold"] >= 0.30\n                and realtime_selected["recall_at_threshold"] >= 0.50\n            ),\n            "planning": (\n                "게이트는 양성률을 높였지만 개별 차량을 구분할 동적 신호가 없어 "\n                "계획형 정밀도는 개선되지 않았다."\n            ),\n            "realtime": (\n                "상류 좌석을 포함하면 정밀도가 약 두 배 개선되지만 현재 제품 KPI에는 미달한다."\n            ),\n        },\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/promote_pooled_main_model.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport tempfile\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom hypothesis_model_search import candidate_weights, encode_target, tree_node_count\nfrom main_model_feature_augmentation import component_candidates, make_model, required_metrics\nfrom main_model_registry import COMPONENTS, file_sha256, validate_version\nfrom pooled_main_model_overfit_ablation import pooled_candidates, prepare_pooled_data\nfrom pooled_main_model_registry import (\n    MODEL_CONTRACT_VERSION,\n    MODEL_FAMILY,\n    MODEL_ID,\n    MODEL_VERSION,\n    PooledVersionedMainModel,\n    build_pooled_feature_profile,\n)\nfrom route_specific_feature_experiment import DEFAULT_ROUTES, cache_paths\n\n\nTRAINING_CUTOFF_DATE = "2026-08-12"\nSOURCE_CUTOFF = "2026-08-13 20:26:03+09:00"\nSELECTED_CANDIDATE = "pooled_37_no_ceiling"\nNODE_BUDGET = 1_000_000\n\n\ndef json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): json_ready(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(item) for item in value]\n    if isinstance(value, (np.integer, np.floating)):\n        return json_ready(value.item())\n    if isinstance(value, float) and not np.isfinite(value):\n        return None\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, pd.Timestamp):\n        return value.isoformat()\n    return value\n\n\ndef selected_metrics(results_dir: Path) -> dict[str, Any]:\n    pooled = pd.read_csv(results_dir / "metrics_pooled.csv")\n    macro = pd.read_csv(results_dir / "metrics_route_macro.csv")\n    by_route = pd.read_csv(results_dir / "metrics_by_route.csv")\n    partition = "latest_complete_08_11_12"\n    return {\n        "partition": partition,\n        "pooled": pooled.loc[\n            pooled["partition"].eq(partition)\n            & pooled["candidate"].eq(SELECTED_CANDIDATE)\n        ].iloc[0].to_dict(),\n        "route_macro": macro.loc[\n            macro["partition"].eq(partition)\n            & macro["candidate"].eq(SELECTED_CANDIDATE)\n        ].iloc[0].to_dict(),\n        "by_route": by_route.loc[\n            by_route["partition"].eq(partition)\n            & by_route["candidate"].eq(SELECTED_CANDIDATE)\n        ].to_dict(orient="records"),\n    }\n\n\ndef train_components(\n    training: pd.DataFrame,\n    feature_columns: tuple[str, ...],\n    feature_set: Any,\n    output_dir: Path,\n    *,\n    seed: int,\n) -> tuple[dict[str, int], dict[str, str], list[dict[str, Any]]]:\n    nodes: dict[str, int] = {}\n    targets: dict[str, str] = {}\n    candidates: list[dict[str, Any]] = []\n    for candidate in component_candidates():\n        print(f"[final train] {candidate.name}", flush=True)\n        model = make_model(candidate, feature_set, seed)\n        model.fit(\n            training[list(feature_columns)],\n            encode_target(training, candidate.target_kind),\n            regressor__sample_weight=candidate_weights(\n                training,\n                candidate.low_weight,\n                weighting_kind=candidate.weighting_kind,\n                gap_weight_power=candidate.gap_weight_power,\n            ),\n        )\n        joblib.dump(model, output_dir / f"{candidate.name}.joblib", compress=3)\n        nodes[candidate.name] = int(tree_node_count(model))\n        targets[candidate.name] = candidate.target_kind\n        candidates.append(asdict(candidate))\n    if set(nodes) != set(COMPONENTS):\n        raise ValueError("통합 주 모델 component 집합이 계약과 다릅니다.")\n    return nodes, targets, candidates\n\n\ndef publish_registry(path: Path, manifest: dict[str, Any]) -> None:\n    registry = {\n        "schema_version": 1,\n        "model_family": MODEL_FAMILY,\n        "primary_version": MODEL_VERSION,\n        "naming_policy": {\n            "format": f"{MODEL_FAMILY}/vMAJOR.MINOR.PATCH",\n            "major": "prediction target or serving contract change",\n            "minor": "feature schema, algorithm, or ensemble change",\n            "patch": "same schema and algorithm retrained on newer data",\n        },\n        "versions": {\n            MODEL_VERSION: {\n                "model_id": MODEL_ID,\n                "status": "primary",\n                "artifact_dir": MODEL_VERSION,\n                "training_cutoff_date": TRAINING_CUTOFF_DATE,\n                "deployment_status": manifest["deployment_status"],\n            }\n        },\n    }\n    temporary = path.with_suffix(".json.tmp")\n    temporary.write_text(\n        json.dumps(json_ready(registry), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    os.replace(temporary, path)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="전체 활성 노선 통합 모델을 주 모델로 승격")\n    parser.add_argument("--cache-dir", type=Path, default=Path("data/analysis_cache/route_specific_features"))\n    parser.add_argument("--featured-cache", type=Path, default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"))\n    parser.add_argument("--evaluation-dir", type=Path, default=Path("analysis/pooled_main_model_overfit_ablation_results"))\n    parser.add_argument("--registry-root", type=Path, default=Path("analysis/model_registry") / MODEL_FAMILY)\n    parser.add_argument("--source-cutoff", default=SOURCE_CUTOFF)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    validate_version(MODEL_VERSION)\n    target_dir = args.registry_root / MODEL_VERSION\n    registry_path = args.registry_root / "registry.json"\n    if target_dir.exists():\n        raise FileExistsError(f"immutable 모델 버전이 이미 존재합니다: {target_dir}")\n    args.registry_root.mkdir(parents=True, exist_ok=True)\n\n    if args.featured_cache.is_file():\n        pooled = pd.read_pickle(args.featured_cache)\n        if pooled.attrs.get("source_cutoff") != args.source_cutoff:\n            pooled, route_metadata = prepare_pooled_data(\n                args.cache_dir, source_cutoff=args.source_cutoff\n            )\n        else:\n            route_metadata = pooled.attrs["route_metadata"]\n    else:\n        pooled, route_metadata = prepare_pooled_data(\n            args.cache_dir, source_cutoff=args.source_cutoff\n        )\n    pooled.attrs["source_cutoff"] = args.source_cutoff\n    pooled.attrs["route_metadata"] = route_metadata\n    pooled.to_pickle(args.featured_cache)\n\n    feature_set, empirical_cap = pooled_candidates()[SELECTED_CANDIDATE]\n    if empirical_cap:\n        raise ValueError("통합 주 모델에는 경험적 출력 보정을 사용할 수 없습니다.")\n    if len(feature_set.columns) != 38 or "route_code" not in feature_set.categorical:\n        raise ValueError("통합 37 core + route_code schema가 버전 계약과 다릅니다.")\n    training = pooled.loc[\n        pooled["date"].le(TRAINING_CUTOFF_DATE)\n        & pd.to_datetime(pooled["date"]).dt.dayofweek.lt(5)\n    ].copy()\n    if training.empty or training["date"].max() != TRAINING_CUTOFF_DATE:\n        raise ValueError("학습 종료일까지의 완전한 평일 통합 데이터가 없습니다.")\n    if set(training["route_code"].astype(str)) != set(DEFAULT_ROUTES):\n        raise ValueError("최종 학습 데이터의 활성 노선 집합이 불완전합니다.")\n\n    route_flows = {\n        route_id: pd.read_pickle(cache_paths(args.cache_dir, route_id)[1])\n        for route_id in DEFAULT_ROUTES\n    }\n    profile = build_pooled_feature_profile(\n        pooled, route_flows, training_cutoff_date=TRAINING_CUTOFF_DATE\n    )\n    metrics = selected_metrics(args.evaluation_dir)\n\n    with tempfile.TemporaryDirectory(prefix=f".{MODEL_VERSION}-", dir=args.registry_root) as temporary:\n        staging = Path(temporary)\n        nodes, target_kinds, candidates = train_components(\n            training, feature_set.columns, feature_set, staging, seed=args.seed\n        )\n        joblib.dump(profile, staging / "feature_profile.joblib", compress=3)\n        artifact_sha256 = {\n            path.name: file_sha256(path)\n            for path in sorted(staging.glob("*.joblib"), key=lambda value: value.name)\n        }\n        total_nodes = int(sum(nodes.values()))\n        node_ready = total_nodes <= NODE_BUDGET\n        manifest = {\n            "contract_version": MODEL_CONTRACT_VERSION,\n            "model_family": MODEL_FAMILY,\n            "version": MODEL_VERSION,\n            "model_id": MODEL_ID,\n            "status": "primary",\n            "supersedes": "arrival-seat-1000/v1.0.0 as project-wide main model",\n            "selection_reason": "pooled all-active-route model selected with low-seat priority and generalizable feature contract",\n            "routes": dict(DEFAULT_ROUTES),\n            "target": "arrival_seats before boarding",\n            "source_cutoff": args.source_cutoff,\n            "training_cutoff_date": TRAINING_CUTOFF_DATE,\n            "training_rows": int(len(training)),\n            "training_events": int(training["event_id"].nunique()),\n            "training_dates": sorted(training["date"].unique().tolist()),\n            "feature_set": SELECTED_CANDIDATE,\n            "core_feature_count": 37,\n            "feature_columns": list(feature_set.columns),\n            "numeric_features": list(feature_set.numeric),\n            "categorical_features": list(feature_set.categorical),\n            "component_candidates": candidates,\n            "component_target_kinds": target_kinds,\n            "ensemble_weights": {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2},\n            "postprocessing": {\n                "empirical_44_70_output_correction": False,\n                "observed_ceiling_features": False,\n                "fitted_output_offset": False,\n                "nominal_capacity_support_clip": True,\n            },\n            "component_tree_nodes": nodes,\n            "total_tree_nodes": total_nodes,\n            "node_budget": NODE_BUDGET,\n            "deployment_status": (\n                "point_model_ready_distribution_recalibration_pending"\n                if node_ready\n                else "blocked_node_budget_and_distribution_recalibration"\n            ),\n            "evaluation": metrics,\n            "feature_profile": {\n                "training_cutoff_date": TRAINING_CUTOFF_DATE,\n                "routes": sorted(profile["routes"]),\n                "inference_rule": "route-local profile may only be used after training cutoff",\n            },\n            "route_cache_metadata": route_metadata,\n            "source_sha256": {\n                path.name: file_sha256(path)\n                for path in [\n                    Path(__file__),\n                    Path(__file__).with_name("pooled_main_model_registry.py"),\n                    Path(__file__).with_name("pooled_main_model_overfit_ablation.py"),\n                    Path(__file__).with_name("main_model_feature_augmentation.py"),\n                ]\n            },\n            "input_sha256": {\n                "featured_cache": file_sha256(args.featured_cache),\n                **{\n                    f"{route_id}_snapshots": file_sha256(cache_paths(args.cache_dir, route_id)[0])\n                    for route_id in DEFAULT_ROUTES\n                },\n                **{\n                    f"{route_id}_flows": file_sha256(cache_paths(args.cache_dir, route_id)[1])\n                    for route_id in DEFAULT_ROUTES\n                },\n            },\n            "artifact_sha256": artifact_sha256,\n        }\n        (staging / "manifest.json").write_text(\n            json.dumps(json_ready(manifest), ensure_ascii=False, indent=2), encoding="utf-8"\n        )\n        smoke = pooled.loc[pooled["date"].gt(TRAINING_CUTOFF_DATE)].copy()\n        if set(smoke["route_code"].astype(str)) != set(DEFAULT_ROUTES):\n            raise ValueError("부분일 artifact 검증의 활성 노선 집합이 불완전합니다.")\n        loaded = PooledVersionedMainModel.load(staging)\n        prediction = loaded.predict(smoke)\n        if not np.isfinite(prediction).all():\n            raise ValueError("통합 주 모델 artifact roundtrip 예측에 비유한 값이 있습니다.")\n        if (prediction < 0).any() or (prediction > smoke["capacity"].to_numpy(float)).any():\n            raise ValueError("통합 주 모델 예측이 명목 정원 범위를 벗어났습니다.")\n        manifest["artifact_roundtrip"] = {\n            "rows": int(len(smoke)),\n            "routes": int(smoke["route_code"].nunique()),\n            "finite": True,\n            "nominal_support_valid": True,\n            "partial_day_metrics": required_metrics(\n                smoke.assign(prediction=prediction)\n            ),\n            "partial_day_metrics_by_route": {\n                str(route_code): required_metrics(\n                    frame.assign(prediction=prediction[frame.index.to_numpy()])\n                )\n                for route_code, frame in smoke.reset_index(drop=True).groupby(\n                    "route_code", sort=False\n                )\n            },\n        }\n        (staging / "manifest.json").write_text(\n            json.dumps(json_ready(manifest), ensure_ascii=False, indent=2), encoding="utf-8"\n        )\n        os.replace(staging, target_dir)\n\n    publish_registry(registry_path, manifest)\n    validation_path = args.registry_root / f"promotion_validation_{MODEL_VERSION}.json"\n    validation_path.write_text(\n        json.dumps(\n            json_ready({\n                "model_id": MODEL_ID,\n                "source_cutoff": args.source_cutoff,\n                "artifact_roundtrip": manifest["artifact_roundtrip"],\n                "total_tree_nodes": manifest["total_tree_nodes"],\n                "node_budget": NODE_BUDGET,\n                "deployment_status": manifest["deployment_status"],\n            }),\n            ensure_ascii=False,\n            indent=2,\n        ),\n        encoding="utf-8",\n    )\n    print(json.dumps(json_ready({\n        "model_id": MODEL_ID,\n        "registry": str(registry_path),\n        "artifact_dir": str(target_dir),\n        "training_rows": len(training),\n        "training_events": training["event_id"].nunique(),\n        "total_tree_nodes": manifest["total_tree_nodes"],\n        "deployment_status": manifest["deployment_status"],\n    }), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/refresh_prospective_cache.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\nfrom hypothesis_model_search import (\n    cache_payload_hashes_match,\n    load_or_build_snapshots,\n)\nfrom route_distribution import file_sha256\n\n\nANALYSIS_DIR = Path(__file__).resolve().parent\nCANONICAL_MODEL_RESULTS = ANALYSIS_DIR / "model_search_results"\nCANONICAL_DISTRIBUTION_RESULTS = ANALYSIS_DIR / "route_distribution_results"\nCANONICAL_CLAIM_REGISTRY = ANALYSIS_DIR / "prospective_claim_registry"\n\n\ndef cache_bundle_paths(snapshot_cache: Path) -> tuple[Path, Path, Path]:\n    snapshot_cache = Path(snapshot_cache)\n    flow_cache = snapshot_cache.with_name(\n        snapshot_cache.stem + "_stop_flows" + snapshot_cache.suffix\n    )\n    metadata = snapshot_cache.with_suffix(snapshot_cache.suffix + ".json")\n    return snapshot_cache, flow_cache, metadata\n\n\ndef _assert_mutable_cache_target(\n    db_path: Path,\n    snapshot_cache: Path,\n    *,\n    model_results: Path,\n    distribution_results: Path,\n) -> None:\n    # CLI path overrides must not turn off protection for the repository\'s\n    # canonical frozen artifacts.  Protect both the canonical locations and\n    # any caller-supplied artifact roots.\n    protected_roots = {\n        "canonical model results": CANONICAL_MODEL_RESULTS.resolve(),\n        "canonical distribution policy results": (\n            CANONICAL_DISTRIBUTION_RESULTS.resolve()\n        ),\n        "canonical prospective claim registry": CANONICAL_CLAIM_REGISTRY.resolve(),\n        "configured model results": Path(model_results).resolve(),\n        "configured distribution policy results": Path(\n            distribution_results\n        ).resolve(),\n    }\n    db_resolved = Path(db_path).resolve()\n    for name, protected in protected_roots.items():\n        if db_resolved == protected or db_resolved.is_relative_to(protected):\n            raise ValueError(\n                "분석 DB는 model/policy artifact 디렉터리 안에 둘 수 없습니다: "\n                f"db={db_resolved}, protected={name}({protected})"\n            )\n    for target in cache_bundle_paths(snapshot_cache):\n        resolved = target.resolve()\n        if resolved == db_resolved:\n            raise ValueError(\n                "prospective cache bundle이 원본 DB를 덮어쓸 수 없습니다: "\n                f"target={resolved}, db={db_resolved}"\n            )\n        for name, protected in protected_roots.items():\n            if resolved == protected or resolved.is_relative_to(protected):\n                raise ValueError(\n                    "prospective cache refresh는 model/policy artifact를 쓸 수 "\n                    f"없습니다: target={resolved}, protected={name}({protected})"\n                )\n\n\ndef _protected_artifact_hashes(\n    model_results: Path,\n    distribution_results: Path,\n) -> dict[str, str]:\n    paths = {\n        "point_model_summary": Path(model_results) / "summary.json",\n        "interval_policy": Path(distribution_results) / "interval_policy.json",\n        "distribution_summary": Path(distribution_results) / "summary.json",\n        "frozen_distribution_flows": (\n            Path(distribution_results) / "frozen_distribution_flows.pkl"\n        ),\n    }\n    return {\n        name: file_sha256(path)\n        for name, path in paths.items()\n        if path.is_file()\n    }\n\n\ndef refresh_prospective_cache(\n    db_path: Path,\n    snapshot_cache: Path,\n    *,\n    model_results: Path = Path("analysis/model_search_results"),\n    distribution_results: Path = Path("analysis/route_distribution_results"),\n) -> dict[str, Any]:\n    """Refresh only mutable analysis caches, never model/policy artifacts."""\n\n    db_path = Path(db_path)\n    snapshot_cache = Path(snapshot_cache)\n    if not db_path.is_file():\n        raise ValueError(f"분석 DB가 없습니다: {db_path}")\n    if snapshot_cache.suffix.lower() not in {".pkl", ".pickle"}:\n        raise ValueError("snapshot cache 확장자는 .pkl 또는 .pickle이어야 합니다.")\n    _assert_mutable_cache_target(\n        db_path,\n        snapshot_cache,\n        model_results=model_results,\n        distribution_results=distribution_results,\n    )\n    protected_before = _protected_artifact_hashes(\n        model_results, distribution_results\n    )\n\n    snapshot_cache.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(\n        dir=snapshot_cache.parent,\n        prefix=".prospective-cache-refresh-",\n    ) as temporary:\n        staged_snapshot = Path(temporary) / snapshot_cache.name\n        snapshots, flows, cache_info = load_or_build_snapshots(\n            db_path, staged_snapshot, rebuild=True\n        )\n        staged_bundle = cache_bundle_paths(staged_snapshot)\n        missing = [str(path) for path in staged_bundle if not path.is_file()]\n        if missing:\n            raise ValueError(f"cache refresh staging artifact가 없습니다: {missing}")\n        staged_metadata = json.loads(staged_bundle[2].read_text(encoding="utf-8"))\n        if not cache_payload_hashes_match(\n            staged_metadata, staged_bundle[0], staged_bundle[1]\n        ):\n            raise ValueError(\n                "cache refresh staging metadata의 snapshot/flow SHA가 payload와 "\n                "일치하지 않습니다."\n            )\n        target_bundle = cache_bundle_paths(snapshot_cache)\n        # The metadata is the commit marker and is published last.  All staged\n        # files live on the same filesystem, so each replacement is atomic.\n        for staged, target in zip(\n            (staged_bundle[1], staged_bundle[0], staged_bundle[2]),\n            (target_bundle[1], target_bundle[0], target_bundle[2]),\n            strict=True,\n        ):\n            os.replace(staged, target)\n\n    protected_after = _protected_artifact_hashes(model_results, distribution_results)\n    if protected_after != protected_before:\n        raise RuntimeError(\n            "cache refresh 중 frozen model/policy artifact hash가 변경되었습니다."\n        )\n    snapshot_target, flow_target, metadata_target = cache_bundle_paths(snapshot_cache)\n    published_metadata = json.loads(metadata_target.read_text(encoding="utf-8"))\n    if not cache_payload_hashes_match(\n        published_metadata, snapshot_target, flow_target\n    ):\n        raise RuntimeError(\n            "게시된 cache metadata의 snapshot/flow SHA가 payload와 일치하지 "\n            "않습니다."\n        )\n    return {\n        "operation": "prospective_cache_refresh_only",\n        "model_or_policy_training_performed": False,\n        "db": str(db_path.resolve()),\n        "snapshot_cache": str(snapshot_target.resolve()),\n        "snapshot_cache_sha256": file_sha256(snapshot_target),\n        "stop_flow_cache": str(flow_target.resolve()),\n        "stop_flow_cache_sha256": file_sha256(flow_target),\n        "metadata": str(metadata_target.resolve()),\n        "metadata_sha256": file_sha256(metadata_target),\n        "rows": int(len(snapshots)),\n        "events": int(snapshots["event_id"].nunique()),\n        "stop_flow_rows": int(len(flows)),\n        "dates": sorted(snapshots["date"].astype(str).unique().tolist()),\n        "cache_info": {\n            **cache_info,\n            "cache_path": str(snapshot_target.resolve()),\n        },\n        "protected_artifact_hashes_unchanged": protected_after,\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description=(\n            "새 prospective 날짜를 위해 mutable snapshot cache만 갱신; "\n            "frozen 모델/interval policy/flow artifact는 변경하지 않음"\n        )\n    )\n    parser.add_argument(\n        "--db",\n        type=Path,\n        default=Path("data/gbis_api_cache.sqlite3"),\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--model-results",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    parser.add_argument(\n        "--distribution-results",\n        type=Path,\n        default=Path("analysis/route_distribution_results"),\n    )\n    args = parser.parse_args()\n    result = refresh_prospective_cache(\n        args.db,\n        args.snapshot_cache,\n        model_results=args.model_results,\n        distribution_results=args.distribution_results,\n    )\n    print(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/model_feasibility.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport sqlite3\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.inspection import permutation_importance\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    brier_score_loss,\n    log_loss,\n    precision_recall_curve,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\n\nDEFAULT_ROUTE_ID = "219000013"\nDEFAULT_ROUTE_NAME = "1000"\nPEAK_HOURS = set(range(6, 11)) | set(range(16, 22))\nQUALITY_ORDER = {"A": 0, "B": 1, "C": 2}\n\n\n@dataclass(frozen=True)\nclass FeatureSet:\n    name: str\n    numeric: tuple[str, ...]\n    categorical: tuple[str, ...]\n\n    @property\n    def columns(self) -> list[str]:\n        return [*self.numeric, *self.categorical]\n\n\nPLANNING = FeatureSet(\n    name="planning",\n    numeric=(\n        "time_sin",\n        "time_cos",\n        "route_progress",\n        "x",\n        "y",\n    ),\n    categorical=(\n        "station_seq_cat",\n        "direction",\n        "day_of_week",\n        "center_yn",\n    ),\n)\n\nREALTIME = FeatureSet(\n    name="realtime_3stops",\n    numeric=(\n        *PLANNING.numeric,\n        "capacity",\n        "upstream_load_ratio_3",\n        "upstream_load_ratio_5",\n        "upstream_stop_distance_3",\n        "upstream_age_minutes_3",\n        "load_change_5_to_3",\n        "previous_bus_load_ratio",\n        "headway_minutes",\n    ),\n    categorical=(*PLANNING.categorical, "low_plate_cat"),\n)\n\n\ndef _as_records(frame: pd.DataFrame) -> list[dict[str, Any]]:\n    result: list[dict[str, Any]] = []\n    for record in frame.to_dict(orient="records"):\n        cleaned: dict[str, Any] = {}\n        for key, value in record.items():\n            if pd.isna(value):\n                cleaned[key] = None\n            elif isinstance(value, (np.integer,)):\n                cleaned[key] = int(value)\n            elif isinstance(value, (np.floating,)):\n                cleaned[key] = float(value)\n            else:\n                cleaned[key] = value\n        result.append(cleaned)\n    return result\n\n\ndef load_data(db_path: Path, route_id: str) -> tuple[pd.DataFrame, pd.DataFrame]:\n    uri = f"file:{db_path.resolve()}?mode=ro"\n    with sqlite3.connect(uri, uri=True) as connection:\n        locations = pd.read_sql_query(\n            """\n            SELECT run_id, observed_at_kst, query_time, route_id, vehicle_id,\n                   plate_no, station_id, station_seq, remaining_seats,\n                   low_plate, state_code\n            FROM bus_locations\n            WHERE route_id = ?\n            ORDER BY vehicle_id, observed_at_kst, run_id\n            """,\n            connection,\n            params=(route_id,),\n        )\n        stations = pd.read_sql_query(\n            """\n            SELECT route_id, station_id, station_seq, station_name, mobile_no,\n                   region_name, x, y, center_yn\n            FROM route_stations\n            WHERE route_id = ?\n            ORDER BY station_seq\n            """,\n            connection,\n            params=(route_id,),\n        )\n    if locations.empty:\n        raise ValueError(f"노선 {route_id}의 차량 위치 데이터가 없습니다.")\n    if stations.empty:\n        raise ValueError(f"노선 {route_id}의 정류장 메타데이터가 없습니다.")\n    locations["observed_at"] = pd.to_datetime(locations["observed_at_kst"])\n    return locations, stations\n\n\ndef infer_turnaround_seq(stations: pd.DataFrame) -> int:\n    ordered = stations.sort_values("station_seq")\n    valid = ordered.dropna(subset=["x", "y"])\n    if len(valid) < 3:\n        return int(round(float(ordered["station_seq"].max()) / 2))\n    origin = valid.iloc[0]\n    # GBIS x/y는 경도/위도다. 한 노선 안에서 최대 직선거리 정류장은 회차점의 좋은 근사다.\n    lat0 = math.radians(float(origin["y"]))\n    lon0 = math.radians(float(origin["x"]))\n    lat = np.radians(valid["y"].astype(float).to_numpy())\n    lon = np.radians(valid["x"].astype(float).to_numpy())\n    dlat = lat - lat0\n    dlon = lon - lon0\n    hav = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat) * np.sin(dlon / 2) ** 2\n    return int(valid.iloc[int(np.argmax(hav))]["station_seq"])\n\n\ndef build_visits(locations: pd.DataFrame, stations: pd.DataFrame) -> pd.DataFrame:\n    rows = locations.sort_values(["vehicle_id", "observed_at", "run_id"]).copy()\n    grouped = rows.groupby("vehicle_id", sort=False)\n    rows["previous_station_seq"] = grouped["station_seq"].shift()\n    rows["previous_observed_at"] = grouped["observed_at"].shift()\n    gap_minutes = (rows["observed_at"] - rows["previous_observed_at"]).dt.total_seconds() / 60\n    rows["new_visit"] = (\n        rows["previous_station_seq"].isna()\n        | rows["station_seq"].ne(rows["previous_station_seq"])\n        | gap_minutes.gt(20)\n    )\n    rows["visit_no"] = rows.groupby("vehicle_id", sort=False)["new_visit"].cumsum()\n    keys = ["vehicle_id", "visit_no"]\n\n    summary = rows.groupby(keys, sort=False).agg(\n        route_id=("route_id", "first"),\n        plate_no=("plate_no", "first"),\n        station_id=("station_id", "first"),\n        station_seq=("station_seq", "first"),\n        first_seen=("observed_at", "first"),\n        last_seen=("observed_at", "last"),\n        first_state=("state_code", "first"),\n        first_seats=("remaining_seats", "first"),\n        last_state=("state_code", "last"),\n        last_seats=("remaining_seats", "last"),\n        low_plate=("low_plate", "first"),\n        samples=("run_id", "size"),\n    ).reset_index()\n\n    departure = (\n        rows.loc[(rows["state_code"] == 2) & (rows["remaining_seats"] >= 0)]\n        .groupby(keys, sort=False)\n        .tail(1)[[*keys, "remaining_seats", "observed_at"]]\n        .rename(\n            columns={\n                "remaining_seats": "departure_seats",\n                "observed_at": "departure_seen",\n            }\n        )\n    )\n    arrival = (\n        rows.loc[(rows["state_code"] == 1) & (rows["remaining_seats"] >= 0)]\n        .groupby(keys, sort=False)\n        .head(1)[[*keys, "remaining_seats", "observed_at"]]\n        .rename(\n            columns={\n                "remaining_seats": "observed_arrival_seats",\n                "observed_at": "arrival_seen",\n            }\n        )\n    )\n    visits = summary.merge(departure, on=keys, how="left").merge(\n        arrival, on=keys, how="left"\n    )\n    visits = visits.sort_values(["vehicle_id", "visit_no"]).reset_index(drop=True)\n\n    by_vehicle = visits.groupby("vehicle_id", sort=False)\n    visits["next_station_seq"] = by_vehicle["station_seq"].shift(-1)\n    visits["next_first_seen"] = by_vehicle["first_seen"].shift(-1)\n    visits["next_first_state"] = by_vehicle["first_state"].shift(-1)\n    visits["next_first_seats"] = by_vehicle["first_seats"].shift(-1)\n    next_gap = (visits["next_first_seen"] - visits["last_seen"]).dt.total_seconds() / 60\n\n    visits["label_quality"] = pd.Series(pd.NA, index=visits.index, dtype="string")\n    visits["label_seats"] = np.nan\n    is_a = visits["departure_seats"].notna()\n    is_b = (~is_a) & visits["last_state"].eq(0) & visits["last_seats"].ge(0)\n    is_c = (\n        (~is_a)\n        & (~is_b)\n        & visits["next_station_seq"].eq(visits["station_seq"] + 1)\n        & next_gap.le(10)\n        & visits["next_first_state"].isin([0, 1])\n        & visits["next_first_seats"].ge(0)\n    )\n    # 문자 등급과 수치 좌석을 한 번에 대입하면 NumPy가 전체를 문자열로\n    # 승격할 수 있으므로 열별로 대입해 label_seats의 수치형을 보존한다.\n    visits.loc[is_a, "label_quality"] = "A"\n    visits.loc[is_a, "label_seats"] = visits.loc[is_a, "departure_seats"]\n    visits.loc[is_b, "label_quality"] = "B"\n    visits.loc[is_b, "label_seats"] = visits.loc[is_b, "last_seats"]\n    # C는 좌석 수 회귀에는 부적합하지만 만차/비만차 분류 라벨에는 검증상 안정적이다.\n    visits.loc[is_c, "label_quality"] = "C"\n    visits.loc[is_c, "label_seats"] = visits.loc[is_c, "next_first_seats"]\n    visits["is_full"] = np.where(visits["label_quality"].notna(), visits["label_seats"].eq(0), np.nan)\n\n    previous_seq = by_vehicle["station_seq"].shift()\n    previous_time = by_vehicle["last_seen"].shift()\n    trip_gap = (visits["first_seen"] - previous_time).dt.total_seconds() / 60\n    new_trip = previous_seq.isna() | visits["station_seq"].le(previous_seq) | trip_gap.gt(30)\n    visits["trip_no"] = new_trip.groupby(visits["vehicle_id"], sort=False).cumsum().astype(int)\n    visits["trip_id"] = visits["vehicle_id"].astype(str) + "-" + visits["trip_no"].astype(str)\n\n    # 사용자가 정류장에서 실제로 마주치는 좌석은 승객을 태운 뒤의 출발 좌석이\n    # 아니라 태우기 전 도착 좌석이다. stateCd=1 직접 관측을 우선하고, 이것이\n    # 빠졌을 때만 같은 운행의 바로 전 정류장 출발 좌석으로 보완한다.\n    by_vehicle = visits.groupby("vehicle_id", sort=False)\n    visits["previous_visit_station_seq"] = by_vehicle["station_seq"].shift()\n    visits["previous_visit_trip_id"] = by_vehicle["trip_id"].shift()\n    visits["previous_departure_seats"] = by_vehicle["departure_seats"].shift()\n    visits["previous_departure_seen"] = by_vehicle["departure_seen"].shift()\n    previous_gap = (\n        visits["first_seen"] - visits["previous_departure_seen"]\n    ).dt.total_seconds() / 60\n    direct_arrival = visits["observed_arrival_seats"].notna()\n    inferred_arrival = (\n        (~direct_arrival)\n        & visits["previous_visit_trip_id"].eq(visits["trip_id"])\n        & visits["station_seq"].eq(visits["previous_visit_station_seq"] + 1)\n        & visits["previous_departure_seats"].ge(0)\n        & previous_gap.between(0, 10, inclusive="both")\n    )\n    visits["arrival_label_quality"] = pd.Series(\n        pd.NA, index=visits.index, dtype="string"\n    )\n    visits["arrival_seats"] = np.nan\n    visits["arrival_event_time"] = visits["arrival_seen"].where(direct_arrival)\n    visits.loc[direct_arrival, "arrival_label_quality"] = "A"\n    visits.loc[direct_arrival, "arrival_seats"] = visits.loc[\n        direct_arrival, "observed_arrival_seats"\n    ]\n    visits.loc[inferred_arrival, "arrival_label_quality"] = "B"\n    visits.loc[inferred_arrival, "arrival_seats"] = visits.loc[\n        inferred_arrival, "previous_departure_seats"\n    ]\n    # B등급은 실제 도착 상태가 누락됐으므로 목표 정류장의 최초 관측시각을\n    # 도착시각 근사로 사용한다.\n    visits.loc[inferred_arrival, "arrival_event_time"] = visits.loc[\n        inferred_arrival, "first_seen"\n    ]\n    visits["arrival_is_full"] = np.where(\n        visits["arrival_label_quality"].notna(), visits["arrival_seats"].eq(0), np.nan\n    )\n\n    visits = visits.merge(stations, on=["route_id", "station_id", "station_seq"], how="left")\n    visits["is_pass_node"] = visits["station_name"].str.contains(r"\\(경유\\)", na=False)\n    return visits\n\n\ndef add_upstream_features(visits: pd.DataFrame, horizon: int) -> pd.DataFrame:\n    output = visits.copy()\n    seat_signal = np.where(\n        output["departure_seats"].notna(),\n        output["departure_seats"],\n        np.where(\n            output["last_state"].eq(0) & output["last_seats"].ge(0),\n            output["last_seats"],\n            np.nan,\n        ),\n    )\n    output["seat_signal"] = seat_signal.astype(float)\n    result_seats = np.full(len(output), np.nan)\n    result_seq = np.full(len(output), np.nan)\n    result_time = np.full(len(output), np.datetime64("NaT", "ns"), dtype="datetime64[ns]")\n\n    for _, indices in output.groupby("trip_id", sort=False).groups.items():\n        idx = np.asarray(list(indices), dtype=int)\n        group = output.loc[idx]\n        candidates = group.loc[group["seat_signal"].notna()]\n        if candidates.empty:\n            continue\n        candidate_seq = candidates["station_seq"].to_numpy(dtype=int)\n        candidate_seats = candidates["seat_signal"].to_numpy(dtype=float)\n        candidate_time = candidates["last_seen"].dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")\n        target_seq = group["station_seq"].to_numpy(dtype=int) - horizon\n        positions = np.searchsorted(candidate_seq, target_seq, side="right") - 1\n        valid = positions >= 0\n        result_seats[idx[valid]] = candidate_seats[positions[valid]]\n        result_seq[idx[valid]] = candidate_seq[positions[valid]]\n        result_time[idx[valid]] = candidate_time[positions[valid]]\n\n    output[f"upstream_seats_{horizon}"] = result_seats\n    output[f"upstream_seq_{horizon}"] = result_seq\n    output[f"upstream_time_{horizon}"] = pd.to_datetime(result_time).tz_localize("Asia/Seoul")\n    output[f"upstream_stop_distance_{horizon}"] = output["station_seq"] - output[f"upstream_seq_{horizon}"]\n    output[f"upstream_age_minutes_{horizon}"] = (\n        output["first_seen"] - output[f"upstream_time_{horizon}"]\n    ).dt.total_seconds() / 60\n    return output\n\n\ndef nominal_capacity(low_plate: pd.Series) -> pd.Series:\n    # 현재 표본의 정상 차량 최대 빈자리 45, 2층버스 최대 빈자리 70과 일치한다.\n    return pd.Series(np.where(low_plate.eq(2), 70.0, 45.0), index=low_plate.index)\n\n\ndef build_model_table(\n    visits: pd.DataFrame,\n    stations: pd.DataFrame,\n    *,\n    label_target: str = "departure",\n) -> tuple[pd.DataFrame, int]:\n    if label_target not in {"departure", "arrival"}:\n        raise ValueError("label_target은 departure 또는 arrival이어야 합니다.")\n    featured = add_upstream_features(visits, 3)\n    featured = add_upstream_features(featured, 5)\n    turnaround_seq = infer_turnaround_seq(stations)\n    max_seq = int(stations["station_seq"].max())\n\n    featured["direction"] = np.where(featured["station_seq"] <= turnaround_seq, "to_city", "return")\n    to_city_progress = featured["station_seq"] / max(turnaround_seq, 1)\n    return_denominator = max(max_seq - turnaround_seq, 1)\n    return_progress = (max_seq - featured["station_seq"]) / return_denominator\n    featured["route_progress"] = np.where(\n        featured["direction"].eq("to_city"), to_city_progress, return_progress\n    ).clip(0, 1)\n\n    if label_target == "arrival":\n        featured["label_quality"] = featured["arrival_label_quality"]\n        featured["label_seats"] = featured["arrival_seats"]\n        featured["is_full"] = featured["arrival_is_full"]\n        featured["event_time"] = featured["arrival_event_time"].fillna(\n            featured["first_seen"]\n        )\n    else:\n        featured["event_time"] = featured["first_seen"]\n    featured["date"] = featured["event_time"].dt.date.astype(str)\n    featured["hour"] = featured["event_time"].dt.hour\n    featured["minute_of_day"] = featured["hour"] * 60 + featured["event_time"].dt.minute\n    angle = 2 * np.pi * featured["minute_of_day"] / (24 * 60)\n    featured["time_sin"] = np.sin(angle)\n    featured["time_cos"] = np.cos(angle)\n    featured["day_of_week"] = featured["event_time"].dt.dayofweek.astype(str)\n    featured["time_bin_30"] = (featured["minute_of_day"] // 30).astype(int)\n    featured["is_peak"] = featured["hour"].isin(PEAK_HOURS)\n    featured["station_seq_cat"] = featured["station_seq"].astype(str)\n    featured["low_plate_cat"] = featured["low_plate"].astype("Int64").astype(str)\n    featured["center_yn"] = featured["center_yn"].fillna("unknown").astype(str)\n    featured["capacity"] = nominal_capacity(featured["low_plate"])\n    featured["upstream_load_ratio_3"] = 1 - featured["upstream_seats_3"] / featured["capacity"]\n    featured["upstream_load_ratio_5"] = 1 - featured["upstream_seats_5"] / featured["capacity"]\n    featured["load_change_5_to_3"] = (\n        featured["upstream_load_ratio_3"] - featured["upstream_load_ratio_5"]\n    )\n\n    actual_stops = featured.loc[~featured["is_pass_node"]].copy()\n    actual_stops = actual_stops.sort_values(["station_seq", "event_time", "vehicle_id"])\n    at_stop = actual_stops.groupby("station_seq", sort=False)\n    # 이전 버스 피처에는 검증된 정류장 출발/교차로 관측(A/B)만 쓴다.\n    # C는 만차 여부에는 안정적이지만 정확한 좌석 수에는 오차가 있어 제외한다.\n    actual_stops["exact_label_seats"] = actual_stops["label_seats"].where(\n        actual_stops["label_quality"].isin(["A", "B"])\n    )\n    actual_stops["previous_bus_seats"] = at_stop["exact_label_seats"].shift()\n    actual_stops["previous_bus_time"] = at_stop["event_time"].shift()\n    actual_stops["headway_minutes"] = (\n        actual_stops["event_time"] - actual_stops["previous_bus_time"]\n    ).dt.total_seconds() / 60\n    actual_stops.loc[~actual_stops["headway_minutes"].between(1, 120), "headway_minutes"] = np.nan\n    actual_stops["previous_bus_load_ratio"] = 1 - actual_stops["previous_bus_seats"] / actual_stops["capacity"]\n    actual_stops = actual_stops.sort_values("event_time").reset_index(drop=True)\n    return actual_stops, turnaround_seq\n\n\ndef make_preprocessor(feature_set: FeatureSet) -> ColumnTransformer:\n    numeric = Pipeline(\n        [\n            ("impute", SimpleImputer(strategy="median", add_indicator=True)),\n            ("scale", StandardScaler()),\n        ]\n    )\n    categorical = Pipeline(\n        [\n            ("impute", SimpleImputer(strategy="most_frequent")),\n            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),\n        ]\n    )\n    return ColumnTransformer(\n        [("numeric", numeric, list(feature_set.numeric)), ("categorical", categorical, list(feature_set.categorical))],\n        remainder="drop",\n    )\n\n\ndef model_factories(feature_set: FeatureSet, seed: int) -> dict[str, Pipeline]:\n    preprocessor = make_preprocessor(feature_set)\n    return {\n        "logistic": Pipeline(\n            [\n                ("features", clone(preprocessor)),\n                (\n                    "classifier",\n                    LogisticRegression(\n                        class_weight="balanced",\n                        C=0.5,\n                        max_iter=2_000,\n                        random_state=seed,\n                    ),\n                ),\n            ]\n        ),\n        "random_forest": Pipeline(\n            [\n                ("features", clone(preprocessor)),\n                (\n                    "classifier",\n                    RandomForestClassifier(\n                        n_estimators=300,\n                        max_depth=8,\n                        min_samples_leaf=15,\n                        max_features="sqrt",\n                        class_weight="balanced_subsample",\n                        n_jobs=-1,\n                        random_state=seed,\n                    ),\n                ),\n            ]\n        ),\n        "hist_gradient_boosting": Pipeline(\n            [\n                ("features", clone(preprocessor)),\n                (\n                    "classifier",\n                    HistGradientBoostingClassifier(\n                        learning_rate=0.05,\n                        max_iter=200,\n                        max_leaf_nodes=15,\n                        min_samples_leaf=30,\n                        l2_regularization=1.0,\n                        random_state=seed,\n                    ),\n                ),\n            ]\n        ),\n    }\n\n\ndef balanced_weights(y: np.ndarray) -> np.ndarray:\n    positives = max(int(y.sum()), 1)\n    negatives = max(int((1 - y).sum()), 1)\n    return np.where(y == 1, len(y) / (2 * positives), len(y) / (2 * negatives))\n\n\ndef fit_model(model: Pipeline, x: pd.DataFrame, y: np.ndarray, name: str) -> Pipeline:\n    if name == "hist_gradient_boosting":\n        model.fit(x, y, classifier__sample_weight=balanced_weights(y))\n    else:\n        model.fit(x, y)\n    return model\n\n\ndef platt_fit(probabilities: np.ndarray, y: np.ndarray, seed: int) -> LogisticRegression | None:\n    if len(np.unique(y)) < 2:\n        return None\n    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)\n    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)\n    calibrator = LogisticRegression(C=1.0, max_iter=1_000, random_state=seed)\n    calibrator.fit(logits, y)\n    return calibrator\n\n\ndef platt_predict(calibrator: LogisticRegression | None, probabilities: np.ndarray) -> np.ndarray:\n    if calibrator is None:\n        return probabilities\n    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)\n    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)\n    return calibrator.predict_proba(logits)[:, 1]\n\n\ndef expected_calibration_error(y: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:\n    if len(y) == 0:\n        return float("nan")\n    frame = pd.DataFrame({"y": y, "p": probabilities})\n    try:\n        frame["bin"] = pd.qcut(frame["p"], q=min(bins, frame["p"].nunique()), duplicates="drop")\n    except ValueError:\n        return float(abs(frame["y"].mean() - frame["p"].mean()))\n    grouped = frame.groupby("bin", observed=True)\n    total = len(frame)\n    return float(\n        sum(len(group) / total * abs(group["y"].mean() - group["p"].mean()) for _, group in grouped)\n    )\n\n\ndef choose_threshold(y: np.ndarray, probabilities: np.ndarray, target_recall: float = 0.8) -> float:\n    if y.sum() == 0:\n        return 0.5\n    precision, recall, thresholds = precision_recall_curve(y, probabilities)\n    candidates = np.flatnonzero(recall[:-1] >= target_recall)\n    if len(candidates) == 0:\n        return 0.5\n    best = candidates[int(np.argmax(precision[candidates]))]\n    return float(thresholds[best])\n\n\ndef metric_row(\n    y: np.ndarray,\n    probabilities: np.ndarray,\n    *,\n    threshold: float,\n    model: str,\n    feature_set: str,\n    split: str,\n) -> dict[str, Any]:\n    probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)\n    predicted = probabilities >= threshold\n    positives = int(y.sum())\n    top_n = max(1, int(math.ceil(len(y) * 0.05)))\n    top_indices = np.argsort(probabilities)[-top_n:]\n    top_recall = float(y[top_indices].sum() / positives) if positives else float("nan")\n    prevalence = float(y.mean()) if len(y) else float("nan")\n    top_precision = float(y[top_indices].mean())\n    return {\n        "split": split,\n        "feature_set": feature_set,\n        "model": model,\n        "rows": int(len(y)),\n        "positives": positives,\n        "prevalence": prevalence,\n        "average_precision": float(average_precision_score(y, probabilities)) if positives else float("nan"),\n        "roc_auc": float(roc_auc_score(y, probabilities)) if positives and positives < len(y) else float("nan"),\n        "brier": float(brier_score_loss(y, probabilities)),\n        "log_loss": float(log_loss(y, probabilities, labels=[0, 1])),\n        "ece": expected_calibration_error(y, probabilities),\n        "threshold": float(threshold),\n        "precision_at_threshold": float(y[predicted].mean()) if predicted.any() else 0.0,\n        "recall_at_threshold": float(y[predicted].sum() / positives) if positives else float("nan"),\n        "alert_rate": float(predicted.mean()),\n        "top_5pct_precision": top_precision,\n        "top_5pct_recall": top_recall,\n        "top_5pct_lift": float(top_precision / prevalence) if prevalence > 0 else float("nan"),\n    }\n\n\ndef historical_rate_predict(train: pd.DataFrame, target: pd.DataFrame, alpha: float = 20.0) -> np.ndarray:\n    global_rate = float(train["is_full"].mean())\n    exact = train.groupby(["station_seq", "direction", "time_bin_30"])["is_full"].agg(["sum", "count"])\n    station = train.groupby(["station_seq", "direction"])["is_full"].agg(["sum", "count"])\n    predictions = []\n    for row in target.itertuples(index=False):\n        key = (row.station_seq, row.direction, row.time_bin_30)\n        fallback_key = (row.station_seq, row.direction)\n        if key in exact.index:\n            stats = exact.loc[key]\n        elif fallback_key in station.index:\n            stats = station.loc[fallback_key]\n        else:\n            predictions.append(global_rate)\n            continue\n        predictions.append((float(stats["sum"]) + alpha * global_rate) / (float(stats["count"]) + alpha))\n    return np.asarray(predictions, dtype=float)\n\n\ndef prepare_subset(table: pd.DataFrame) -> pd.DataFrame:\n    subset = table.loc[\n        table["is_peak"]\n        & table["label_quality"].notna()\n        & table["date"].between("2026-08-04", "2026-08-09")\n    ].copy()\n    subset["is_full"] = subset["is_full"].astype(int)\n    return subset.sort_values("event_time").reset_index(drop=True)\n\n\ndef evaluate_final_split(\n    data: pd.DataFrame,\n    feature_sets: Iterable[FeatureSet],\n    seed: int,\n    threshold_target_recall: float = 0.8,\n) -> tuple[pd.DataFrame, dict[tuple[str, str], Pipeline], dict[tuple[str, str], np.ndarray]]:\n    train = data.loc[data["date"].isin(["2026-08-04", "2026-08-05"])]\n    calibration = data.loc[data["date"].eq("2026-08-06")]\n    test = data.loc[data["date"].eq("2026-08-07")]\n    if min(train["is_full"].sum(), calibration["is_full"].sum(), test["is_full"].sum()) == 0:\n        raise ValueError("최종 시간분할 중 하나에 만차 사례가 없습니다.")\n\n    rows: list[dict[str, Any]] = []\n    fitted: dict[tuple[str, str], Pipeline] = {}\n    test_predictions: dict[tuple[str, str], np.ndarray] = {}\n\n    baseline_cal = historical_rate_predict(train, calibration)\n    baseline_test = historical_rate_predict(train, test)\n    baseline_calibrator = platt_fit(baseline_cal, calibration["is_full"].to_numpy(), seed)\n    baseline_test_calibrated = platt_predict(baseline_calibrator, baseline_test)\n    baseline_threshold = choose_threshold(\n        calibration["is_full"].to_numpy(),\n        platt_predict(baseline_calibrator, baseline_cal),\n        target_recall=threshold_target_recall,\n    )\n    baseline_row = metric_row(\n            test["is_full"].to_numpy(),\n            baseline_test_calibrated,\n            threshold=baseline_threshold,\n            model="historical_rate",\n            feature_set="planning",\n            split="test_2026-08-07",\n        )\n    baseline_row["selection_average_precision"] = float(\n        average_precision_score(calibration["is_full"].to_numpy(), baseline_cal)\n    )\n    rows.append(baseline_row)\n    test_predictions[("planning", "historical_rate")] = baseline_test_calibrated\n\n    for feature_set in feature_sets:\n        x_train = train[feature_set.columns]\n        x_cal = calibration[feature_set.columns]\n        x_test = test[feature_set.columns]\n        y_train = train["is_full"].to_numpy()\n        y_cal = calibration["is_full"].to_numpy()\n        y_test = test["is_full"].to_numpy()\n        for model_name, model in model_factories(feature_set, seed).items():\n            fitted_model = fit_model(model, x_train, y_train, model_name)\n            raw_cal = fitted_model.predict_proba(x_cal)[:, 1]\n            raw_test = fitted_model.predict_proba(x_test)[:, 1]\n            calibrator = platt_fit(raw_cal, y_cal, seed)\n            calibrated_cal = platt_predict(calibrator, raw_cal)\n            calibrated_test = platt_predict(calibrator, raw_test)\n            threshold = choose_threshold(\n                y_cal, calibrated_cal, target_recall=threshold_target_recall\n            )\n            result_row = metric_row(\n                    y_test,\n                    calibrated_test,\n                    threshold=threshold,\n                    model=model_name,\n                    feature_set=feature_set.name,\n                    split="test_2026-08-07",\n                )\n            # 최종 테스트를 보지 않고 보정일의 순위화 성능으로 후보를 선택한다.\n            result_row["selection_average_precision"] = float(\n                average_precision_score(y_cal, raw_cal)\n            )\n            rows.append(result_row)\n            key = (feature_set.name, model_name)\n            fitted[key] = fitted_model\n            test_predictions[key] = calibrated_test\n    return pd.DataFrame(rows), fitted, test_predictions\n\n\ndef cluster_bootstrap_summary(\n    test: pd.DataFrame,\n    probabilities: np.ndarray,\n    baseline_probabilities: np.ndarray,\n    seed: int,\n    repeats: int = 2_000,\n) -> dict[str, Any]:\n    """운행(trip) 단위 재표집으로 연속 정류장 관측의 상관을 보존한다."""\n    y = test["is_full"].to_numpy()\n    groups = test.groupby("trip_id", sort=False).indices\n    trip_ids = np.asarray(list(groups), dtype=object)\n    rng = np.random.default_rng(seed)\n    values: list[tuple[float, float, float]] = []\n    for _ in range(repeats):\n        selected = rng.choice(trip_ids, size=len(trip_ids), replace=True)\n        positions = np.concatenate([groups[trip_id] for trip_id in selected])\n        sampled_y = y[positions]\n        if sampled_y.sum() == 0:\n            continue\n        model_ap = average_precision_score(sampled_y, probabilities[positions])\n        baseline_ap = average_precision_score(sampled_y, baseline_probabilities[positions])\n        values.append(\n            (\n                float(model_ap),\n                float(brier_score_loss(sampled_y, probabilities[positions])),\n                float(model_ap - baseline_ap),\n            )\n        )\n    array = np.asarray(values)\n\n    def interval(column: int) -> dict[str, float]:\n        lower, median, upper = np.quantile(array[:, column], [0.025, 0.5, 0.975])\n        return {"lower_95": float(lower), "median": float(median), "upper_95": float(upper)}\n\n    return {\n        "unit": "trip_id",\n        "test_trips": int(len(trip_ids)),\n        "valid_repeats": int(len(values)),\n        "average_precision": interval(0),\n        "brier": interval(1),\n        "average_precision_delta_vs_historical_rate": interval(2),\n    }\n\n\ndef trip_level_diagnostics(\n    test: pd.DataFrame,\n    probabilities: np.ndarray,\n    threshold: float,\n) -> dict[str, Any]:\n    scored = test.copy()\n    scored["probability"] = probabilities\n    full = scored.loc[scored["is_full"].eq(1)].sort_values("event_time")\n    first_full = full.groupby("trip_id", sort=False).head(1)\n    positive_trips = int(first_full["trip_id"].nunique())\n    any_alert = full.groupby("trip_id", sort=False)["probability"].max().ge(threshold)\n    return {\n        "positive_trips": positive_trips,\n        "first_full_stop_alerted": int(first_full["probability"].ge(threshold).sum()),\n        "first_full_stop_recall": float(first_full["probability"].ge(threshold).mean()),\n        "any_full_stop_alerted_trips": int(any_alert.sum()),\n        "any_full_stop_trip_recall": float(any_alert.mean()),\n        "observed_full_stop_rows": int(len(full)),\n    }\n\n\ndef evaluate_forward_chaining(\n    data: pd.DataFrame,\n    feature_sets: Iterable[FeatureSet],\n    seed: int,\n) -> pd.DataFrame:\n    dates = ["2026-08-05", "2026-08-06", "2026-08-07"]\n    rows: list[dict[str, Any]] = []\n    for test_date in dates:\n        train = data.loc[data["date"] < test_date]\n        test = data.loc[data["date"] == test_date]\n        if train["is_full"].sum() == 0 or test["is_full"].sum() == 0:\n            continue\n        baseline = historical_rate_predict(train, test)\n        rows.append(\n            metric_row(\n                test["is_full"].to_numpy(),\n                baseline,\n                threshold=0.5,\n                model="historical_rate",\n                feature_set="planning",\n                split=f"forward_{test_date}",\n            )\n        )\n        for feature_set in feature_sets:\n            y_train = train["is_full"].to_numpy()\n            y_test = test["is_full"].to_numpy()\n            for model_name, model in model_factories(feature_set, seed).items():\n                fitted_model = fit_model(model, train[feature_set.columns], y_train, model_name)\n                probabilities = fitted_model.predict_proba(test[feature_set.columns])[:, 1]\n                rows.append(\n                    metric_row(\n                        y_test,\n                        probabilities,\n                        threshold=0.5,\n                        model=model_name,\n                        feature_set=feature_set.name,\n                        split=f"forward_{test_date}",\n                    )\n                )\n    return pd.DataFrame(rows)\n\n\ndef summarize_eda(data: pd.DataFrame) -> dict[str, Any]:\n    labeled = data.loc[data["label_quality"].notna()].copy()\n    by_date = (\n        labeled.groupby("date")["is_full"]\n        .agg(rows="size", full="sum", full_rate="mean")\n        .reset_index()\n    )\n    by_quality = (\n        labeled.groupby("label_quality")["is_full"]\n        .agg(rows="size", full="sum", full_rate="mean")\n        .reset_index()\n    )\n    by_direction = (\n        labeled.groupby("direction")["is_full"]\n        .agg(rows="size", full="sum", full_rate="mean")\n        .reset_index()\n    )\n    top_stops = (\n        labeled.groupby(["station_seq", "station_name", "direction"])["is_full"]\n        .agg(rows="size", full="sum", full_rate="mean")\n        .reset_index()\n        .sort_values(["full", "full_rate"], ascending=False)\n        .head(12)\n    )\n    full_rows = labeled.loc[labeled["is_full"].eq(1)]\n    return {\n        "rows": int(len(labeled)),\n        "full_events": int(full_rows.shape[0]),\n        "full_rate": float(labeled["is_full"].mean()),\n        "full_trips": int(full_rows["trip_id"].nunique()),\n        "full_vehicles": int(full_rows["vehicle_id"].nunique()),\n        "date_range": [str(labeled["date"].min()), str(labeled["date"].max())],\n        "upstream_3_coverage": float(labeled["upstream_load_ratio_3"].notna().mean()),\n        "previous_bus_coverage": float(labeled["previous_bus_load_ratio"].notna().mean()),\n        "by_date": _as_records(by_date),\n        "by_quality": _as_records(by_quality),\n        "by_direction": _as_records(by_direction),\n        "top_stops": _as_records(top_stops),\n    }\n\n\ndef make_eda_plot(data: pd.DataFrame, output_path: Path) -> None:\n    labeled = data.loc[data["label_quality"].notna()].copy()\n    hourly = labeled.groupby("hour")["is_full"].agg(rate="mean", full="sum", rows="size").reset_index()\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    axes[0].bar(hourly["hour"], hourly["rate"] * 100, color="#2563eb")\n    axes[0].set_title("Full-bus rate by hour")\n    axes[0].set_xlabel("Hour (KST)")\n    axes[0].set_ylabel("Full rate (%)")\n    axes[0].set_xticks(range(0, 24, 2))\n\n    stop = (\n        labeled.groupby(["station_seq", "direction"])["is_full"]\n        .agg(rate="mean", full="sum", rows="size")\n        .reset_index()\n    )\n    for direction, group in stop.groupby("direction"):\n        axes[1].plot(group["station_seq"], group["rate"] * 100, marker="o", ms=3, label=direction)\n    axes[1].set_title("Full-bus rate along route")\n    axes[1].set_xlabel("Station sequence")\n    axes[1].set_ylabel("Full rate (%)")\n    axes[1].legend()\n    fig.tight_layout()\n    fig.savefig(output_path, dpi=160)\n    plt.close(fig)\n\n\ndef permutation_table(\n    model: Pipeline,\n    test: pd.DataFrame,\n    feature_set: FeatureSet,\n    seed: int,\n) -> pd.DataFrame:\n    result = permutation_importance(\n        model,\n        test[feature_set.columns],\n        test["is_full"].to_numpy(),\n        scoring="average_precision",\n        n_repeats=12,\n        random_state=seed,\n        n_jobs=-1,\n    )\n    return (\n        pd.DataFrame(\n            {\n                "feature": feature_set.columns,\n                "importance_mean": result.importances_mean,\n                "importance_std": result.importances_std,\n            }\n        )\n        .sort_values("importance_mean", ascending=False)\n        .reset_index(drop=True)\n    )\n\n\ndef json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): json_ready(item) for key, item in value.items()}\n    if isinstance(value, list):\n        return [json_ready(item) for item in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating, float)):\n        return None if not math.isfinite(float(value)) else float(value)\n    return value\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="광역버스 만차 예측 가능성 분석")\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument("--route-id", default=DEFAULT_ROUTE_ID)\n    parser.add_argument("--route-name", default=DEFAULT_ROUTE_NAME)\n    parser.add_argument("--output-dir", type=Path, default=Path("analysis/results"))\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    locations, stations = load_data(args.db, args.route_id)\n    visits = build_visits(locations, stations)\n    table, turnaround_seq = build_model_table(visits, stations)\n    data = prepare_subset(table)\n\n    final_metrics, fitted, predictions = evaluate_final_split(data, [PLANNING, REALTIME], args.seed)\n    forward_metrics = evaluate_forward_chaining(data, [PLANNING, REALTIME], args.seed)\n    test = data.loc[data["date"].eq("2026-08-07")].copy()\n\n    candidates = final_metrics.loc[final_metrics["model"] != "historical_rate"].copy()\n    best_row = candidates.sort_values(\n        ["selection_average_precision", "brier"], ascending=[False, True]\n    ).iloc[0]\n    best_key = (str(best_row["feature_set"]), str(best_row["model"]))\n    best_feature_set = REALTIME if best_key[0] == REALTIME.name else PLANNING\n    importance = permutation_table(fitted[best_key], test, best_feature_set, args.seed)\n\n    strict_mask = test["label_quality"].eq("A")\n    strict_metrics = metric_row(\n        test.loc[strict_mask, "is_full"].to_numpy(),\n        predictions[best_key][strict_mask.to_numpy()],\n        threshold=float(best_row["threshold"]),\n        model=best_key[1],\n        feature_set=best_key[0],\n        split="test_2026-08-07_quality_A_only",\n    )\n    strict_data = data.loc[data["label_quality"].eq("A")].copy()\n    strict_retrained_metrics, _, _ = evaluate_final_split(\n        strict_data, [PLANNING, REALTIME], args.seed\n    )\n    strict_selected = (\n        strict_retrained_metrics.loc[strict_retrained_metrics["model"].ne("historical_rate")]\n        .sort_values(["selection_average_precision", "brier"], ascending=[False, True])\n        .iloc[0]\n    )\n\n    baseline_key = ("planning", "historical_rate")\n    bootstrap = cluster_bootstrap_summary(\n        test,\n        predictions[best_key],\n        predictions[baseline_key],\n        args.seed,\n    )\n    trip_diagnostics = trip_level_diagnostics(\n        test, predictions[best_key], float(best_row["threshold"])\n    )\n    calibration_diagnostics = {\n        "observed_rate": float(test["is_full"].mean()),\n        "mean_predicted_probability": float(predictions[best_key].mean()),\n        "ratio_predicted_to_observed": float(\n            predictions[best_key].mean() / test["is_full"].mean()\n        ),\n    }\n\n    weekend = data.loc[data["date"].isin(["2026-08-08", "2026-08-09"])]\n    weekend_summary = {\n        "rows": int(len(weekend)),\n        "positives": int(weekend["is_full"].sum()),\n    }\n\n    final_metrics.to_csv(args.output_dir / "final_metrics.csv", index=False)\n    forward_metrics.to_csv(args.output_dir / "forward_metrics.csv", index=False)\n    strict_retrained_metrics.to_csv(\n        args.output_dir / "strict_A_retrained_metrics.csv", index=False\n    )\n    importance.to_csv(args.output_dir / "permutation_importance.csv", index=False)\n    make_eda_plot(data, args.output_dir / "eda_full_patterns.png")\n\n    result = {\n        "route_id": args.route_id,\n        "route_name": args.route_name,\n        "turnaround_seq": turnaround_seq,\n        "source_rows": int(len(locations)),\n        "visit_rows": int(len(visits)),\n        "model_rows_peak_labeled": int(len(data)),\n        "eda": summarize_eda(data),\n        "split": {\n            "train": ["2026-08-04", "2026-08-05"],\n            "calibration": ["2026-08-06"],\n            "test": ["2026-08-07"],\n            "weekend_sanity": ["2026-08-08", "2026-08-09"],\n        },\n        "best_model": {\n            "feature_set": best_key[0],\n            "model": best_key[1],\n            "test_metrics": json_ready(best_row.to_dict()),\n            "strict_label_sensitivity": json_ready(strict_metrics),\n            "strict_A_retrained_same_selected_model": json_ready(\n                strict_retrained_metrics.loc[\n                    (strict_retrained_metrics["feature_set"].eq(best_key[0]))\n                    & (strict_retrained_metrics["model"].eq(best_key[1]))\n                ].iloc[0].to_dict()\n            ),\n            "strict_A_retrained_calibration_selected_model": json_ready(\n                strict_selected.to_dict()\n            ),\n            "trip_level_diagnostics": trip_diagnostics,\n            "calibration_diagnostics": calibration_diagnostics,\n            "cluster_bootstrap": bootstrap,\n        },\n        "weekend_sanity": weekend_summary,\n        "feature_importance": json_ready(_as_records(importance.head(12))),\n        "limitations": [\n            "수집 기간이 6일이고 최종 테스트가 1개 평일이라 운영 성능의 확정치가 아니다.",\n            "노선 1000만 만차 양성 사례가 있어 노선 간 일반화는 검증하지 못했다.",\n            "날씨·교통·행사·공휴일 피처는 현재 DB에 없어 효과를 검증하지 못했다.",\n            "C 등급 라벨은 정확한 잔여좌석 수가 아니라 만차/비만차 분류에만 사용해야 한다.",\n            "실시간 배차간격은 실제 목표 정류장 관측시각으로 계산한 근사치라 운영 시 ETA로 대체해야 한다.",\n            "만차는 승차 실패와 같지 않으며 대기열 데이터가 없어서 보내야 할 차량 수는 아직 검증할 수 없다.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/export_exact_weighted_colab_model.py': '"""Export the exact 3-model low25 4x-transition specialist for Colab sharing.\n\nThis intentionally has no sampling or model-family fallback.  Use an external\nexecution environment when the local runtime cannot fit the full dataset.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport tempfile\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom emerging_low_correction_experiment import (\n    CURRENT_SEAT_MAX,\n    TRANSITION_WEIGHT,\n    experiment_components,\n    pooled_features,\n    transition_label,\n)\nfrom hypothesis_model_search import candidate_weights, encode_target\nfrom main_model_feature_augmentation import make_model\nfrom pooled_main_model_registry import build_pooled_feature_profile\nfrom route_specific_feature_experiment import DEFAULT_ROUTES, cache_paths\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="정확한 3모델 4x 전환 가중 Colab artifact export")\n    parser.add_argument(\n        "--featured-cache",\n        type=Path,\n        default=Path("data/analysis_cache/pooled_main_model_features_colab_release.pkl"),\n    )\n    parser.add_argument(\n        "--cache-dir",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features"),\n    )\n    parser.add_argument(\n        "--output",\n        type=Path,\n        default=Path("colab/model_artifacts/arrival_seat_low25_weighted_v1.pkl"),\n    )\n    parser.add_argument("--training-cutoff-date", default="2026-08-12")\n    parser.add_argument("--source-cutoff", required=True)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.featured_cache)\n    if data.attrs.get("source_cutoff") != args.source_cutoff:\n        raise ValueError(\n            "featured cache source cutoff 불일치: "\n            f"{data.attrs.get(\'source_cutoff\')} != {args.source_cutoff}"\n        )\n    metadata = data.attrs.get("route_metadata")\n    if not isinstance(metadata, dict):\n        raise ValueError("featured cache route metadata가 없습니다.")\n    for route_id in DEFAULT_ROUTES:\n        if metadata.get(route_id, {}).get("source_cutoff") != args.source_cutoff:\n            raise ValueError(f"{route_id} feature cache cutoff 불일치")\n\n    training = data.loc[\n        data["date"].le(args.training_cutoff_date)\n        & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)\n    ].copy()\n    fit_train = training.loc[\n        training["snapshot_remaining_seats"].le(CURRENT_SEAT_MAX)\n    ].copy()\n    if fit_train.empty:\n        raise ValueError("현재 ≤25석 학습 행이 없습니다.")\n\n    features = pooled_features()\n    models = {}\n    target_kinds = {}\n    for index, candidate in enumerate(experiment_components("l1")):\n        print(f"[exact weighted export] fitting {candidate.name}", flush=True)\n        model = make_model(candidate, features, seed=args.seed + index)\n        weights = candidate_weights(\n            fit_train,\n            candidate.low_weight,\n            weighting_kind=candidate.weighting_kind,\n            gap_weight_power=candidate.gap_weight_power,\n        )\n        weights *= np.where(transition_label(fit_train), TRANSITION_WEIGHT, 1.0)\n        model.fit(\n            fit_train[list(features.columns)],\n            encode_target(fit_train, candidate.target_kind),\n            regressor__sample_weight=weights,\n        )\n        models[candidate.name] = model\n        target_kinds[candidate.name] = candidate.target_kind\n\n    route_flows = {\n        route_id: pd.read_pickle(cache_paths(args.cache_dir, route_id)[1])\n        for route_id in DEFAULT_ROUTES\n    }\n    profile = build_pooled_feature_profile(\n        data, route_flows, training_cutoff_date=args.training_cutoff_date\n    )\n    payload = {\n        "format_version": 1,\n        "model_variant": "low25_transition_weighted_specialist",\n        "model_id": "arrival-seat-pooled-low25-transition/v1.0.0-exact",\n        "training_cutoff_date": args.training_cutoff_date,\n        "source_cutoff": args.source_cutoff,\n        "routes": dict(DEFAULT_ROUTES),\n        "feature_columns": list(features.columns),\n        "component_target_kinds": target_kinds,\n        "ensemble_weights": {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2},\n        "feature_profile": profile,\n        "models": models,\n        "gate": {"snapshot_remaining_seats_lte": CURRENT_SEAT_MAX},\n        "training_policy": {\n            "fit_population": "all training rows with current seats <= 25",\n            "transition_definition": "current seats 11-25 and arrival seats <= 10",\n            "transition_sample_weight_multiplier": TRANSITION_WEIGHT,\n            "sampling": "none",\n        },\n        "training_rows": int(len(fit_train)),\n        "training_events": int(fit_train["event_id"].nunique()),\n        "training_dates": sorted(training["date"].unique().tolist()),\n        "route_cache_metadata": metadata,\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(\n        dir=args.output.parent, prefix=".weighted-model-", suffix=".pkl", delete=False\n    ) as temporary:\n        temporary_path = Path(temporary.name)\n    try:\n        joblib.dump(payload, temporary_path, compress=3)\n        os.replace(temporary_path, args.output)\n    finally:\n        temporary_path.unlink(missing_ok=True)\n    print(args.output, flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/spatial_semantic_feature_experiment.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans\n\nfrom latest_main_model_overfit_ablation import apply_candidate, train_schema\nfrom main_model_feature_augmentation import component_candidates, run_component\nfrom model_feasibility import FeatureSet\nfrom pooled_main_model_overfit_ablation import (\n    LATEST_PARTIAL_DATE,\n    pooled_candidates,\n    prepare_pooled_data,\n    rolling_folds,\n    scoped_metrics,\n)\nfrom linear_feature_experiment import json_ready\nfrom route_specific_feature_experiment import DEFAULT_ROUTES\n\n\nCLUSTER_COUNT = 12\n\n\ndef infer_common_cache_cutoff(cache_dir: Path) -> str:\n    """Return the cutoff shared by every actively collected route cache."""\n    cutoffs: dict[str, str | None] = {}\n    for route_id, route_name in DEFAULT_ROUTES.items():\n        metadata_path = cache_dir / f"{route_id}_metadata.json"\n        if not metadata_path.is_file():\n            raise ValueError(\n                f"노선 {route_name} feature cache metadata가 없습니다: {metadata_path}"\n            )\n        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        cutoffs[route_name] = metadata.get("source_cutoff")\n    unique = set(cutoffs.values())\n    if len(unique) != 1 or None in unique:\n        raise ValueError(f"활성 노선 feature cache cutoff가 일치하지 않습니다: {cutoffs}")\n    return unique.pop()  # type: ignore[return-value]\n\n\ndef train_screening_schema(\n    name: str,\n    features: FeatureSet,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    *,\n    device: str,\n    gpu_platform_id: int,\n    gpu_device_id: int,\n    max_train_rows: int,\n    n_estimators: int,\n) -> pd.DataFrame:\n    """Fast gate: reproduce only the deployed LightGBM component across folds."""\n    candidate = next(item for item in component_candidates() if item.name == "lightgbm")\n    params = {**candidate.params, "n_estimators": n_estimators}\n    if device == "gpu":\n        # LightGBM\'s OpenCL GPU learner uses the installed NVIDIA driver.  Keep\n        # every other parameter identical to the primary-model component.\n        candidate = replace(\n            candidate,\n            params={\n                **params,\n                "device_type": "gpu",\n                "gpu_platform_id": gpu_platform_id,\n                "gpu_device_id": gpu_device_id,\n            },\n        )\n    else:\n        candidate = replace(candidate, params=params)\n    sampled_folds = []\n    for fold_number, (date, train, validation) in enumerate(folds):\n        sampled_train = balanced_training_sample(\n            train, max_rows=max_train_rows, seed=42 + fold_number\n        )\n        print(\n            f"[{name}] lightgbm screening ({device}, {len(sampled_train):,} train rows, "\n            f"{n_estimators} trees, validation={date})",\n            flush=True,\n        )\n        sampled_folds.append((date, sampled_train, validation))\n    output, _, _ = run_component(\n        candidate, features, sampled_folds, feature_set_name=name, seed=42\n    )\n    return output\n\n\ndef balanced_training_sample(\n    data: pd.DataFrame, *, max_rows: int, seed: int\n) -> pd.DataFrame:\n    """Route-balanced deterministic sample used only for the cheap screen gate."""\n    if max_rows <= 0 or len(data) <= max_rows:\n        return data\n    groups = list(data.groupby("route_id", sort=True))\n    quota, remainder = divmod(max_rows, len(groups))\n    takes = [min(len(group), quota + int(index < remainder)) for index, (_, group) in enumerate(groups)]\n    remaining = max_rows - sum(takes)\n    while remaining:\n        progressed = False\n        for index, (_, group) in enumerate(groups):\n            if takes[index] < len(group):\n                takes[index] += 1\n                remaining -= 1\n                progressed = True\n                if not remaining:\n                    break\n        if not progressed:\n            break\n    frames = []\n    for index, (_, group) in enumerate(groups):\n        frames.append(group.sample(n=takes[index], random_state=seed + index))\n    return pd.concat(frames, ignore_index=False).sort_index()\n\n\ndef select_screening_folds(\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]], mode: str\n) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:\n    """Keep screening cheap; full mode remains the decision-grade evaluation."""\n    complete = [fold for fold in folds if fold[0] != LATEST_PARTIAL_DATE]\n    if mode == "latest_complete":\n        return [complete[-1]]\n    if mode == "complete":\n        return complete\n    return folds\n\n\ndef add_spatial_semantic_features(data: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    points = output[["x", "y"]].drop_duplicates().dropna().to_numpy(float)\n    clusterer = KMeans(n_clusters=CLUSTER_COUNT, n_init=20, random_state=42).fit(points)\n    coords = output[["x", "y"]].to_numpy(float)\n    labels = clusterer.predict(coords)\n    centers = clusterer.cluster_centers_[labels]\n    output["spatial_cluster"] = pd.Series(labels, index=output.index).astype("string")\n    # Equirectangular approximation is accurate enough at this local scale.\n    dx = (coords[:, 0] - centers[:, 0]) * 88.8\n    dy = (coords[:, 1] - centers[:, 1]) * 111.2\n    output["spatial_cluster_distance_km"] = np.hypot(dx, dy)\n\n    unique = output[["x", "y"]].drop_duplicates()\n    unique_coords = unique[["x", "y"]].to_numpy(float)\n    dx = (unique_coords[:, None, 0] - unique_coords[None, :, 0]) * 88.8\n    dy = (unique_coords[:, None, 1] - unique_coords[None, :, 1]) * 111.2\n    nearby = np.hypot(dx, dy) <= 1.0\n    density = nearby.sum(axis=1)\n    route_lookup = output[["x", "y", "route_id"]].drop_duplicates().groupby(\n        ["x", "y"]\n    )["route_id"].nunique()\n    route_density = np.asarray([\n        route_lookup.reindex(pd.MultiIndex.from_frame(unique.iloc[np.flatnonzero(mask)][["x", "y"]])).sum()\n        for mask in nearby\n    ])\n    lookup = unique.assign(\n        spatial_stop_density_1km=density,\n        spatial_route_density_1km=route_density,\n    ).set_index(["x", "y"])\n    index = pd.MultiIndex.from_frame(output[["x", "y"]])\n    output["spatial_stop_density_1km"] = lookup["spatial_stop_density_1km"].reindex(index).to_numpy(float)\n    output["spatial_route_density_1km"] = lookup["spatial_route_density_1km"].reindex(index).to_numpy(float)\n\n    output["spatial_cluster_hour_low10_rate"] = 0.0\n    output["spatial_cluster_hour_low10_log_count"] = 0.0\n    events = output.drop_duplicates("event_id").copy()\n    events["hour"] = pd.to_datetime(events["event_time"]).dt.hour\n    events["low10"] = events["label_seats"].le(10).astype(float)\n    for date in sorted(output["date"].unique()):\n        history = events.loc[events["date"].lt(date)]\n        target = output.loc[output["date"].eq(date)]\n        global_rate = float(history["low10"].mean()) if len(history) else 0.0\n        stats = history.groupby(["spatial_cluster", "hour"])["low10"].agg(["sum", "count"])\n        keys = pd.MultiIndex.from_arrays([\n            target["spatial_cluster"].astype(str),\n            pd.to_datetime(target["snapshot_time"]).dt.hour,\n        ])\n        matched = stats.reindex(keys)\n        count = matched["count"].fillna(0).to_numpy(float)\n        rate = (matched["sum"].fillna(0).to_numpy(float) + 20 * global_rate) / (count + 20)\n        output.loc[target.index, "spatial_cluster_hour_low10_rate"] = rate\n        output.loc[target.index, "spatial_cluster_hour_low10_log_count"] = np.log1p(count)\n    return output\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="원시 좌표 대비 공간 의미 피처 통합 ablation")\n    parser.add_argument("--cache-dir", type=Path, default=Path("data/analysis_cache/route_specific_features"))\n    parser.add_argument("--featured-cache", type=Path, default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"))\n    parser.add_argument("--output-dir", type=Path, default=Path("analysis/spatial_semantic_feature_results"))\n    parser.add_argument(\n        "--source-cutoff",\n        default=None,\n        help="생략하면 모든 활성 노선 cache metadata의 공통 cutoff를 자동 사용",\n    )\n    parser.add_argument(\n        "--max-train-rows",\n        type=int,\n        default=25_000,\n        help="screen 모드에서만 쓸 노선 균형 학습 행 수 (기본: 25,000, 0=전체)",\n    )\n    parser.add_argument(\n        "--screen-estimators",\n        type=int,\n        default=150,\n        help="screen 모드 LightGBM 트리 수 (기본: 150, full은 생산용 600)",\n    )\n    parser.add_argument(\n        "--mode", choices=("screen", "full"), default="screen",\n        help="screen=LightGBM 단독 빠른 선별, full=기존 3모델 앙상블 확정 검증",\n    )\n    parser.add_argument(\n        "--device", choices=("gpu", "cpu"), default="cpu",\n        help="screen 모드의 LightGBM 장치 (기본: CPU, 현 OpenCL GPU보다 빠름)",\n    )\n    parser.add_argument("--gpu-platform-id", type=int, default=0)\n    parser.add_argument("--gpu-device-id", type=int, default=0)\n    parser.add_argument(\n        "--screen-folds",\n        choices=("latest_complete", "complete", "all"),\n        default="latest_complete",\n        help="screen 모드 검증 범위 (기본: 최신 완료일 1개, full은 항상 전체 3폴드)",\n    )\n    parser.add_argument(\n        "--variant",\n        choices=("semantic_no_xy", "rates_only_with_xy"),\n        default="rates_only_with_xy",\n        help="비교 후보: 좌표 대체 전체 공간 피처 또는 좌표 유지+군집 시간대 피처 2개",\n    )\n    args = parser.parse_args()\n    args.source_cutoff = args.source_cutoff or infer_common_cache_cutoff(args.cache_dir)\n    print(f"source cutoff: {args.source_cutoff}", flush=True)\n    pooled = pd.read_pickle(args.featured_cache)\n    if pooled.attrs.get("source_cutoff") != args.source_cutoff:\n        pooled, metadata = prepare_pooled_data(args.cache_dir, source_cutoff=args.source_cutoff)\n        pooled.attrs["source_cutoff"] = args.source_cutoff\n        pooled.attrs["route_metadata"] = metadata\n    enriched = add_spatial_semantic_features(pooled)\n    baseline, _ = pooled_candidates()["pooled_37_no_ceiling"]\n    semantic = FeatureSet(\n        "spatial_semantic_no_xy",\n        tuple(column for column in baseline.numeric if column not in {"x", "y"}) + (\n            "spatial_cluster_distance_km", "spatial_stop_density_1km", "spatial_route_density_1km",\n            "spatial_cluster_hour_low10_rate", "spatial_cluster_hour_low10_log_count",\n        ),\n        (*baseline.categorical, "spatial_cluster"),\n    )\n    rates_only = FeatureSet(\n        "rates_only_with_xy",\n        (*baseline.numeric, "spatial_cluster_hour_low10_rate", "spatial_cluster_hour_low10_log_count"),\n        baseline.categorical,\n    )\n    candidates = (\n        {"raw_xy": baseline, "spatial_semantic_no_xy": semantic}\n        if args.variant == "semantic_no_xy"\n        else {"raw_xy": baseline, "rates_only_with_xy": rates_only}\n    )\n    predictions = []\n    for name, features in candidates.items():\n        folds = rolling_folds(enriched)\n        evaluation_folds = (\n            select_screening_folds(folds, args.screen_folds)\n            if args.mode == "screen"\n            else folds\n        )\n        schema = (\n            train_screening_schema(\n                name,\n                features,\n                evaluation_folds,\n                device=args.device,\n                gpu_platform_id=args.gpu_platform_id,\n                gpu_device_id=args.gpu_device_id,\n                max_train_rows=args.max_train_rows,\n                n_estimators=args.screen_estimators,\n            )\n            if args.mode == "screen"\n            else train_schema(name, features, evaluation_folds, seed=42)\n        )\n        scored, _ = apply_candidate(name, features, False, schema)\n        scored["route_id"] = scored["event_id"].str.split("::", n=1).str[0]\n        scored["route_name"] = scored["route_id"].map(DEFAULT_ROUTES)\n        scored["feature_count"] = len(features.columns)\n        scored["core_feature_count"] = len(features.columns) - 1\n        scored["total_feature_count"] = len(features.columns)\n        predictions.append(scored)\n    table = pd.concat(predictions, ignore_index=True)\n    pooled_metrics, route_metrics, macro = scoped_metrics(table)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    pooled_metrics.to_csv(args.output_dir / "metrics_pooled.csv", index=False)\n    route_metrics.to_csv(args.output_dir / "metrics_by_route.csv", index=False)\n    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)\n    selected = semantic if args.variant == "semantic_no_xy" else rates_only\n    (args.output_dir / "summary.json").write_text(json.dumps(json_ready({"source_cutoff": args.source_cutoff, "mode": args.mode, "variant": args.variant, "device": args.device if args.mode == "screen" else "mixed_cpu_gpu", "screen_folds": args.screen_folds if args.mode == "screen" else "all", "screen_max_train_rows": args.max_train_rows if args.mode == "screen" else None, "screen_estimators": args.screen_estimators if args.mode == "screen" else None, "spatial_features": list(selected.columns), "pooled": pooled_metrics.to_dict(orient="records"), "macro": macro.to_dict(orient="records")}), ensure_ascii=False, indent=2), encoding="utf-8")\n    print(pooled_metrics.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/promote_main_model.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport tempfile\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom all_prearrival_seat_regression import event_bias\nfrom hypothesis_model_search import (\n    add_observed_capacity_features,\n    candidate_weights,\n    encode_target,\n    tree_node_count,\n)\nfrom latest_main_model_feature_recheck import rolling_folds\nfrom main_model_feature_augmentation import (\n    component_candidates,\n    feature_sets,\n    make_model,\n    prepare_augmented,\n)\nfrom main_model_registry import (\n    COMPONENTS,\n    MODEL_CONTRACT_VERSION,\n    MODEL_FAMILY,\n    MODEL_ID,\n    MODEL_VERSION,\n    PRIMARY_FEATURES,\n    build_feature_profile,\n    file_sha256,\n    validate_version,\n)\n\n\nTRAINING_CUTOFF_DATE = "2026-08-12"\nNODE_BUDGET = 1_000_000\n\n\ndef json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): json_ready(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(item) for item in value]\n    if isinstance(value, (np.integer, np.floating)):\n        return value.item()\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, pd.Timestamp):\n        return value.isoformat()\n    return value\n\n\ndef selected_metrics(results_dir: Path) -> dict[str, Any]:\n    metrics = pd.read_csv(results_dir / "metrics_by_partition.csv")\n    latest = metrics.loc[metrics["partition"].eq("latest_complete_08_11_12")]\n    rows = latest.loc[\n        latest["feature_set"].isin(["baseline_official", "importance_pruned"])\n    ]\n    if set(rows["feature_set"]) != {"baseline_official", "importance_pruned"}:\n        raise ValueError("최신 완전일 baseline/candidate 평가가 없습니다.")\n    bootstrap = pd.read_csv(results_dir / "paired_trip_bootstrap.csv")\n    bootstrap = bootstrap.loc[\n        bootstrap["partition"].eq("latest_complete_08_11_12")\n        & bootstrap["candidate"].eq("importance_pruned")\n    ]\n    return {\n        "partition": "latest_complete_08_11_12",\n        "baseline": rows.loc[rows["feature_set"].eq("baseline_official")]\n        .iloc[0]\n        .to_dict(),\n        "selected": rows.loc[rows["feature_set"].eq("importance_pruned")]\n        .iloc[0]\n        .to_dict(),\n        "paired_trip_bootstrap": bootstrap.to_dict(orient="records"),\n        "selection_priority": [\n            "low_0_10_mae",\n            "full_f1",\n            "event_balanced_mae_guardrail",\n        ],\n    }\n\n\ndef deployment_bias_from_oof(oof_path: Path) -> float:\n    oof = pd.read_pickle(oof_path)\n    selected = oof.loc[\n        oof["candidate"].eq("importance_pruned")\n        & oof["date"].le(TRAINING_CUTOFF_DATE)\n    ].copy()\n    if selected.empty:\n        raise ValueError("importance_pruned OOF 예측이 없습니다.")\n    return float(\n        event_bias(selected, selected["raw_prediction"].to_numpy(dtype=float))\n    )\n\n\ndef train_components(\n    training: pd.DataFrame,\n    feature_columns: tuple[str, ...],\n    output_dir: Path,\n    *,\n    seed: int,\n) -> tuple[dict[str, int], dict[str, str], list[dict[str, Any]]]:\n    nodes: dict[str, int] = {}\n    target_kinds: dict[str, str] = {}\n    candidates: list[dict[str, Any]] = []\n    for candidate in component_candidates():\n        model = make_model(candidate, feature_sets()["importance_pruned"], seed)\n        model.fit(\n            training[list(feature_columns)],\n            encode_target(training, candidate.target_kind),\n            regressor__sample_weight=candidate_weights(\n                training,\n                candidate.low_weight,\n                weighting_kind=candidate.weighting_kind,\n                gap_weight_power=candidate.gap_weight_power,\n            ),\n        )\n        joblib.dump(model, output_dir / f"{candidate.name}.joblib", compress=3)\n        nodes[candidate.name] = int(tree_node_count(model))\n        target_kinds[candidate.name] = candidate.target_kind\n        candidates.append(asdict(candidate))\n    if set(nodes) != set(COMPONENTS):\n        raise ValueError("주 모델 component 집합이 계약과 다릅니다.")\n    return nodes, target_kinds, candidates\n\n\ndef publish_registry(registry_path: Path, manifest: dict[str, Any]) -> None:\n    registry = {\n        "schema_version": 1,\n        "model_family": MODEL_FAMILY,\n        "primary_version": MODEL_VERSION,\n        "naming_policy": {\n            "format": f"{MODEL_FAMILY}/vMAJOR.MINOR.PATCH",\n            "major": "prediction target or serving contract change",\n            "minor": "feature schema, algorithm, or ensemble change",\n            "patch": "same schema and algorithm retrained on newer data",\n        },\n        "versions": {\n            MODEL_VERSION: {\n                "model_id": MODEL_ID,\n                "status": "primary",\n                "artifact_dir": MODEL_VERSION,\n                "training_cutoff_date": TRAINING_CUTOFF_DATE,\n                "deployment_status": manifest["deployment_status"],\n            }\n        },\n    }\n    temporary = registry_path.with_suffix(".json.tmp")\n    temporary.write_text(\n        json.dumps(json_ready(registry), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    os.replace(temporary, registry_path)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="40피처 후보를 버전된 주 모델로 승격")\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path(\n            "data/analysis_cache/route_specific_features/219000013_snapshots.pkl"\n        ),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features/219000013_flows.pkl"),\n    )\n    parser.add_argument(\n        "--augmented-cache",\n        type=Path,\n        default=Path("data/analysis_cache/main_model_augmented_features_latest.pkl"),\n    )\n    parser.add_argument(\n        "--evaluation-dir",\n        type=Path,\n        default=Path("analysis/latest_main_model_importance_pruned_results"),\n    )\n    parser.add_argument(\n        "--registry-root",\n        type=Path,\n        default=Path("analysis/model_registry") / MODEL_FAMILY,\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    validate_version(MODEL_VERSION)\n    target_dir = args.registry_root / MODEL_VERSION\n    registry_path = args.registry_root / "registry.json"\n    if target_dir.exists():\n        raise FileExistsError(\n            f"immutable 모델 버전이 이미 존재합니다: {target_dir}"\n        )\n    args.registry_root.mkdir(parents=True, exist_ok=True)\n\n    data = prepare_augmented(\n        args.snapshot_cache, args.flow_cache, args.augmented_cache\n    )\n    data = add_observed_capacity_features(data)\n    flows = pd.read_pickle(args.flow_cache)\n    training = data.loc[\n        data["date"].le(TRAINING_CUTOFF_DATE)\n        & pd.to_datetime(data["snapshot_time"]).dt.dayofweek.lt(5)\n    ].copy()\n    if training.empty or training["date"].max() != TRAINING_CUTOFF_DATE:\n        raise ValueError("학습 종료일까지의 완전한 평일 데이터가 없습니다.")\n    selected_features = feature_sets()["importance_pruned"]\n    if tuple(selected_features.numeric[-len(PRIMARY_FEATURES) :]) != PRIMARY_FEATURES:\n        raise ValueError("40피처 주 모델 schema가 버전 계약과 다릅니다.")\n\n    metrics = selected_metrics(args.evaluation_dir)\n    deployment_bias = deployment_bias_from_oof(\n        args.evaluation_dir / "oof_predictions.pkl"\n    )\n    profile = build_feature_profile(\n        data, flows, training_cutoff_date=TRAINING_CUTOFF_DATE\n    )\n\n    with tempfile.TemporaryDirectory(\n        prefix=f".{MODEL_VERSION}-", dir=args.registry_root\n    ) as temporary:\n        staging = Path(temporary)\n        nodes, target_kinds, candidates = train_components(\n            training,\n            selected_features.columns,\n            staging,\n            seed=args.seed,\n        )\n        joblib.dump(profile, staging / "feature_profile.joblib", compress=3)\n        artifact_sha256 = {\n            path.name: file_sha256(path)\n            for path in sorted(staging.glob("*.joblib"), key=lambda value: value.name)\n        }\n        total_nodes = int(sum(nodes.values()))\n        manifest = {\n            "contract_version": MODEL_CONTRACT_VERSION,\n            "model_family": MODEL_FAMILY,\n            "version": MODEL_VERSION,\n            "model_id": MODEL_ID,\n            "status": "primary",\n            "supersedes": "stack_sqrt48_hgb_040_extra_040_lgb_020_observed_cap44_70",\n            "selection_reason": (\n                "latest complete-day low<=10 MAE and full-occupancy F1 improvement; "\n                "overall MAE retained as a guardrail"\n            ),\n            "route_id": "219000013",\n            "route_name": "1000",\n            "target": "arrival_seats before boarding",\n            "training_cutoff_date": TRAINING_CUTOFF_DATE,\n            "training_rows": int(len(training)),\n            "training_events": int(training["event_id"].nunique()),\n            "training_dates": sorted(training["date"].unique().tolist()),\n            "feature_set": "importance_pruned",\n            "feature_columns": list(selected_features.columns),\n            "numeric_features": list(selected_features.numeric),\n            "categorical_features": list(selected_features.categorical),\n            "added_primary_features": list(PRIMARY_FEATURES),\n            "component_candidates": candidates,\n            "component_target_kinds": target_kinds,\n            "ensemble_weights": {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2},\n            "deployment_bias": deployment_bias,\n            "component_tree_nodes": nodes,\n            "total_tree_nodes": total_nodes,\n            "node_budget": NODE_BUDGET,\n            "deployment_status": (\n                "ready" if total_nodes <= NODE_BUDGET else "blocked_node_budget"\n            ),\n            "evaluation": metrics,\n            "feature_profile": {\n                "training_cutoff_date": TRAINING_CUTOFF_DATE,\n                "event_history_rows": int(len(profile["event_history"])),\n                "flow_history_rows": int(len(profile["flow_history"])),\n                "inference_rule": "profile may only be used after training cutoff",\n            },\n            "source_sha256": {\n                path.name: file_sha256(path)\n                for path in [\n                    Path(__file__),\n                    Path(__file__).with_name("main_model_registry.py"),\n                    Path(__file__).with_name("main_model_feature_augmentation.py"),\n                    Path(__file__).with_name("linear_feature_experiment.py"),\n                ]\n            },\n            "input_sha256": {\n                "snapshot_cache": file_sha256(args.snapshot_cache),\n                "flow_cache": file_sha256(args.flow_cache),\n                "augmented_cache": file_sha256(args.augmented_cache),\n            },\n            "artifact_sha256": artifact_sha256,\n        }\n        (staging / "manifest.json").write_text(\n            json.dumps(json_ready(manifest), ensure_ascii=False, indent=2),\n            encoding="utf-8",\n        )\n        os.replace(staging, target_dir)\n\n    publish_registry(registry_path, manifest)\n    print(\n        json.dumps(\n            {\n                "model_id": MODEL_ID,\n                "registry": str(registry_path),\n                "artifact_dir": str(target_dir),\n                "training_rows": len(training),\n                "training_events": training["event_id"].nunique(),\n                "total_tree_nodes": manifest["total_tree_nodes"],\n                "deployment_status": manifest["deployment_status"],\n            },\n            ensure_ascii=False,\n            indent=2,\n        )\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/route_progress_curve_analysis.py': 'from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import (\n    accuracy_score,\n    f1_score,\n    precision_score,\n    recall_score,\n)\nfrom sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler\n\nfrom all_prearrival_seat_regression import clip_seats, event_bias, event_weights\nfrom hypothesis_model_search import encode_target\nfrom route_local_feature_importance import filter_to_oof_manifest\nfrom route_local_model import DEFAULT_CACHE_DIR, json_ready, route_cache_paths\n\n\nDEFAULT_MODEL_RESULTS = Path("analysis/route_local_model_results")\nDEFAULT_OUTPUT_DIR = Path("analysis/route_progress_curve_results")\nDEFAULT_ALPHAS = (1.0, 10.0, 100.0, 1000.0)\nPRIMARY_ALPHA = 100.0\n\n# Route/station identifiers and progress are deliberately absent. Both compared\n# models receive exactly these same context controls.\nCONTROL_NUMERIC = (\n    "snapshot_time_sin",\n    "snapshot_time_cos",\n    "snapshot_capacity",\n    "target_load_ratio",\n    "target_stop_gap",\n    "snapshot_remaining_seats",\n    "seat_delta_previous_stop",\n    "seat_change_per_stop",\n    "rolling_seat_change_per_stop_3",\n    "minutes_since_previous_stop",\n    "rolling_minutes_per_stop_3",\n    "estimated_minutes_to_arrival",\n    "projected_arrival_seats",\n    "seats_per_remaining_stop",\n    "load_gap_interaction",\n    "currently_low_5",\n    "currently_low_10",\n)\nCONTROL_CATEGORICAL = (\n    "snapshot_day_of_week",\n    "snapshot_low_plate_cat",\n    "target_state_cat",\n    "snapshot_time_bin_30",\n)\nDIRECTIONS = ("to_city", "return")\n\n\ndef add_leg_progress(data: pd.DataFrame) -> pd.DataFrame:\n    """Normalize both route legs to travel direction 0 -> 1.\n\n    The source route_progress increases on the outbound leg but decreases after\n    the turnaround. The midpoint represents the portion of the route over which\n    the per-stop change target is predicted.\n    """\n    output = data.copy()\n    returning = output["direction"].astype(str).eq("return")\n    output["snapshot_leg_progress"] = output["snapshot_route_progress"].where(\n        ~returning, 1.0 - output["snapshot_route_progress"]\n    )\n    output["target_leg_progress"] = output["route_progress"].where(\n        ~returning, 1.0 - output["route_progress"]\n    )\n    output["leg_progress"] = (\n        output["snapshot_leg_progress"] + output["target_leg_progress"]\n    ) / 2.0\n    tolerance = 1e-10\n    if (\n        output["leg_progress"].lt(-tolerance).any()\n        or output["leg_progress"].gt(1 + tolerance).any()\n    ):\n        raise ValueError("정규화된 진행률이 [0, 1] 범위를 벗어났습니다.")\n    if (\n        output["target_leg_progress"]\n        .add(tolerance)\n        .lt(output["snapshot_leg_progress"])\n        .any()\n    ):\n        raise ValueError("목표 정류장이 현재 정류장보다 뒤에 있습니다.")\n    return output\n\n\ndef globalize_ids(data: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    prefix = output["route_id"].astype(str) + "|"\n    output["event_id"] = prefix + output["event_id"].astype(str)\n    output["trip_id"] = prefix + output["trip_id"].astype(str)\n    return output\n\n\ndef common_validation_dates(route_results: Iterable[dict[str, Any]]) -> list[str]:\n    date_sets = [\n        set(str(value) for value in route["rolling_validation_dates"])\n        for route in route_results\n    ]\n    if not date_sets:\n        return []\n    return sorted(set.intersection(*date_sets))\n\n\ndef rowwise_category_interaction(\n    category_codes: np.ndarray,\n    basis: np.ndarray,\n    category_count: int,\n) -> sparse.csr_matrix:\n    codes = np.asarray(category_codes, dtype=int)\n    values = np.asarray(basis, dtype=np.float32)\n    if len(codes) != len(values):\n        raise ValueError("category code와 spline basis의 행 수가 다릅니다.")\n    if codes.min(initial=0) < 0 or codes.max(initial=0) >= category_count:\n        raise ValueError("알 수 없는 interaction category가 있습니다.")\n    rows = np.repeat(np.arange(len(values)), values.shape[1])\n    columns = (\n        codes[:, None] * values.shape[1]\n        + np.arange(values.shape[1], dtype=int)[None, :]\n    ).ravel()\n    return sparse.csr_matrix(\n        (values.ravel(), (rows, columns)),\n        shape=(len(values), category_count * values.shape[1]),\n        dtype=np.float32,\n    )\n\n\nclass ProgressDesign:\n    """Fold-local sparse design for shared or route-specific progress curves."""\n\n    def __init__(self, *, route_specific: bool) -> None:\n        self.route_specific = route_specific\n        self.numeric_imputer = SimpleImputer(strategy="median", add_indicator=True)\n        self.numeric_scaler = StandardScaler()\n        self.context_encoder = OneHotEncoder(\n            handle_unknown="ignore", sparse_output=True, dtype=np.float32\n        )\n        self.route_direction_encoder = OneHotEncoder(\n            handle_unknown="error", sparse_output=True, dtype=np.float32\n        )\n        self.progress_spline = SplineTransformer(\n            n_knots=5,\n            degree=3,\n            include_bias=False,\n            knots="quantile",\n            extrapolation="linear",\n        )\n        self.route_direction_categories: list[str] = []\n\n    @staticmethod\n    def _route_direction(data: pd.DataFrame) -> pd.DataFrame:\n        values = data["route_id"].astype(str) + "|" + data["direction"].astype(str)\n        return values.to_frame("route_direction")\n\n    def fit(self, data: pd.DataFrame) -> "ProgressDesign":\n        numeric = self.numeric_imputer.fit_transform(data[list(CONTROL_NUMERIC)])\n        self.numeric_scaler.fit(numeric)\n        self.context_encoder.fit(\n            data[list(CONTROL_CATEGORICAL)].astype("string").fillna("missing")\n        )\n        route_direction = self._route_direction(data)\n        self.route_direction_encoder.fit(route_direction)\n        self.route_direction_categories = list(\n            self.route_direction_encoder.categories_[0].astype(str)\n        )\n        self.progress_spline.fit(data[["leg_progress"]])\n        return self\n\n    def transform(self, data: pd.DataFrame) -> sparse.csr_matrix:\n        numeric = self.numeric_imputer.transform(data[list(CONTROL_NUMERIC)])\n        numeric = sparse.csr_matrix(\n            self.numeric_scaler.transform(numeric).astype(np.float32)\n        )\n        context = self.context_encoder.transform(\n            data[list(CONTROL_CATEGORICAL)].astype("string").fillna("missing")\n        ).tocsr()\n        route_direction_frame = self._route_direction(data)\n        route_direction = self.route_direction_encoder.transform(\n            route_direction_frame\n        ).tocsr()\n        basis = self.progress_spline.transform(\n            data[["leg_progress"]]\n        ).astype(np.float32)\n\n        if self.route_specific:\n            lookup = {\n                value: index\n                for index, value in enumerate(self.route_direction_categories)\n            }\n            values = route_direction_frame["route_direction"].astype(str)\n            unknown = sorted(set(values) - set(lookup))\n            if unknown:\n                raise ValueError(f"학습에서 보지 못한 노선/방향입니다: {unknown}")\n            codes = values.map(lookup).to_numpy(dtype=int)\n            curve = rowwise_category_interaction(\n                codes, basis, len(self.route_direction_categories)\n            )\n        else:\n            direction_lookup = {value: index for index, value in enumerate(DIRECTIONS)}\n            values = data["direction"].astype(str)\n            unknown = sorted(set(values) - set(direction_lookup))\n            if unknown:\n                raise ValueError(f"알 수 없는 방향입니다: {unknown}")\n            codes = values.map(direction_lookup).to_numpy(dtype=int)\n            curve = rowwise_category_interaction(codes, basis, len(DIRECTIONS))\n\n        # The route-direction intercept appears in both designs. The only\n        # difference is whether the spline shape is shared by routes.\n        return sparse.hstack(\n            [numeric, context, route_direction, curve],\n            format="csr",\n            dtype=np.float32,\n        )\n\n    def fit_transform(self, data: pd.DataFrame) -> sparse.csr_matrix:\n        return self.fit(data).transform(data)\n\n\ndef extended_event_metrics(\n    data: pd.DataFrame, predictions: np.ndarray\n) -> dict[str, float | int]:\n    prediction = clip_seats(np.asarray(predictions, dtype=float), data["capacity"])\n    label = data["label_seats"].to_numpy(dtype=float)\n    weights = event_weights(data)\n    absolute_error = np.abs(label - prediction)\n    low = label <= 10\n    actual_full = label == 0\n    predicted_full = prediction < 0.5\n\n    def weighted_mean(values: np.ndarray, mask: np.ndarray | None = None) -> float:\n        current_weights = weights if mask is None else weights[mask]\n        current_values = values if mask is None else values[mask]\n        if len(current_values) == 0 or current_weights.sum() == 0:\n            return np.nan\n        return float(np.average(current_values, weights=current_weights))\n\n    return {\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n        "event_balanced_mae": weighted_mean(absolute_error),\n        "low_0_10_mae": weighted_mean(absolute_error, low),\n        "full_accuracy": float(\n            accuracy_score(actual_full, predicted_full, sample_weight=weights)\n        ),\n        "full_recall": float(\n            recall_score(\n                actual_full,\n                predicted_full,\n                sample_weight=weights,\n                zero_division=0,\n            )\n        ),\n        "full_precision": float(\n            precision_score(\n                actual_full,\n                predicted_full,\n                sample_weight=weights,\n                zero_division=0,\n            )\n        ),\n        "full_f1": float(\n            f1_score(\n                actual_full,\n                predicted_full,\n                sample_weight=weights,\n                zero_division=0,\n            )\n        ),\n    }\n\n\ndef fit_ridge(\n    design: sparse.csr_matrix,\n    target: np.ndarray,\n    weights: np.ndarray,\n    *,\n    alpha: float,\n) -> Ridge:\n    model = Ridge(alpha=alpha, solver="lsqr", tol=1e-5, max_iter=5_000)\n    model.fit(design, target, sample_weight=weights)\n    return model\n\n\ndef route_manifest(\n    manifest: pd.DataFrame, route_id: str, validation_date: str\n) -> pd.DataFrame:\n    return manifest.loc[\n        manifest["route_id"].astype(str).eq(str(route_id))\n        & manifest["date"].astype(str).eq(str(validation_date))\n        & manifest["candidate"].astype(str).eq("route_local_hgb")\n    ].copy()\n\n\ndef attach_hgb_predictions(\n    validation: pd.DataFrame, manifest: pd.DataFrame\n) -> pd.DataFrame:\n    left = validation.copy()\n    right = manifest[["event_id", "snapshot_time", "prediction"]].copy()\n    left["_snapshot_ns"] = pd.to_datetime(left["snapshot_time"], utc=True).astype(\n        "int64"\n    )\n    right["_snapshot_ns"] = pd.to_datetime(\n        right["snapshot_time"], utc=True\n    ).astype("int64")\n    right = right.drop(columns="snapshot_time").rename(\n        columns={"prediction": "hgb_oof_prediction"}\n    )\n    output = left.merge(\n        right,\n        on=["event_id", "_snapshot_ns"],\n        how="left",\n        validate="one_to_one",\n    ).drop(columns="_snapshot_ns")\n    if output["hgb_oof_prediction"].isna().any():\n        raise ValueError("기존 HGB OOF 예측을 validation row에 연결하지 못했습니다.")\n    return output\n\n\ndef load_route_frames(\n    summary: dict[str, Any],\n    manifest: pd.DataFrame,\n    cache_dir: Path,\n    validation_dates: list[str],\n) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:\n    training: dict[str, pd.DataFrame] = {}\n    validations: dict[str, list[pd.DataFrame]] = {\n        date: [] for date in validation_dates\n    }\n    for route in summary["route_results"]:\n        route_id = str(route["route_id"])\n        cache_path, _ = route_cache_paths(cache_dir, route_id)\n        data = pd.read_pickle(cache_path)\n        data = data.loc[\n            data["date"].astype(str).isin(route["training_dates"])\n        ].copy()\n        data["route_id"] = route_id\n        data["route_name"] = str(route["route_name"])\n        data = add_leg_progress(data)\n        training[route_id] = globalize_ids(data)\n\n        for validation_date in validation_dates:\n            local_manifest = route_manifest(manifest, route_id, validation_date)\n            validation = filter_to_oof_manifest(\n                data.loc[data["date"].astype(str).eq(validation_date)].copy(),\n                local_manifest,\n                validation_date=validation_date,\n            )\n            validation = attach_hgb_predictions(validation, local_manifest)\n            validations[validation_date].append(globalize_ids(validation))\n    pooled_validation = {\n        date: pd.concat(frames, ignore_index=True)\n        for date, frames in validations.items()\n    }\n    return training, pooled_validation\n\n\ndef pooled_training_before(\n    route_frames: dict[str, pd.DataFrame], validation_date: str\n) -> pd.DataFrame:\n    frames = [\n        data.loc[data["date"].astype(str).lt(validation_date)]\n        for data in route_frames.values()\n    ]\n    return pd.concat(frames, ignore_index=True)\n\n\ndef evaluate_shared_vs_route_curves(\n    route_frames: dict[str, pd.DataFrame],\n    validations: dict[str, pd.DataFrame],\n    *,\n    alphas: tuple[float, ...],\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    histories: dict[tuple[str, float], list[pd.DataFrame]] = {\n        (kind, alpha): []\n        for kind in ("shared_curve", "route_specific_curve")\n        for alpha in alphas\n    }\n    daily_rows: list[dict[str, Any]] = []\n\n    for validation_date in sorted(validations):\n        train = pooled_training_before(route_frames, validation_date)\n        validation = validations[validation_date]\n        target = encode_target(train, "delta_per_stop")\n        weights = event_weights(train)\n\n        for kind, route_specific in (\n            ("shared_curve", False),\n            ("route_specific_curve", True),\n        ):\n            transformer = ProgressDesign(route_specific=route_specific)\n            train_design = transformer.fit_transform(train)\n            validation_design = transformer.transform(validation)\n            for alpha in alphas:\n                model = fit_ridge(\n                    train_design, target, weights, alpha=alpha\n                )\n                raw_delta = model.predict(validation_design)\n                raw_seats = clip_seats(\n                    validation["snapshot_remaining_seats"].to_numpy(dtype=float)\n                    + raw_delta\n                    * validation["target_stop_gap"].to_numpy(dtype=float),\n                    validation["capacity"],\n                )\n                previous = histories[(kind, alpha)]\n                bias = (\n                    event_bias(\n                        pd.concat(previous, ignore_index=True),\n                        pd.concat(previous, ignore_index=True)[\n                            "raw_prediction"\n                        ].to_numpy(dtype=float),\n                    )\n                    if previous\n                    else 0.0\n                )\n                prediction = clip_seats(\n                    raw_seats + bias, validation["capacity"]\n                )\n                scored = validation[\n                    [\n                        "event_id",\n                        "trip_id",\n                        "date",\n                        "route_id",\n                        "route_name",\n                        "label_seats",\n                        "capacity",\n                    ]\n                ].copy()\n                scored["raw_prediction"] = raw_seats\n                scored["prediction"] = prediction\n                previous.append(scored)\n\n                for route_id, route_data in scored.groupby("route_id", sort=True):\n                    metrics = extended_event_metrics(\n                        route_data,\n                        route_data["prediction"].to_numpy(dtype=float),\n                    )\n                    daily_rows.append(\n                        {\n                            "model": kind,\n                            "alpha": alpha,\n                            "validation_date": validation_date,\n                            "route_id": str(route_id),\n                            "route_name": str(route_data["route_name"].iloc[0]),\n                            "strict_forward_bias": bias,\n                            **metrics,\n                        }\n                    )\n\n    aggregate_rows: list[dict[str, Any]] = []\n    for (kind, alpha), frames in histories.items():\n        scored = pd.concat(frames, ignore_index=True)\n        aggregate_rows.append(\n            {\n                "model": kind,\n                "alpha": alpha,\n                **extended_event_metrics(\n                    scored, scored["prediction"].to_numpy(dtype=float)\n                ),\n            }\n        )\n    aggregate = pd.DataFrame(aggregate_rows).sort_values(["alpha", "model"])\n    daily = pd.DataFrame(daily_rows).sort_values(\n        ["alpha", "validation_date", "route_name", "model"]\n    )\n\n    comparison_rows: list[dict[str, Any]] = []\n    for alpha in alphas:\n        shared = aggregate.loc[\n            aggregate["alpha"].eq(alpha)\n            & aggregate["model"].eq("shared_curve")\n        ].iloc[0]\n        route = aggregate.loc[\n            aggregate["alpha"].eq(alpha)\n            & aggregate["model"].eq("route_specific_curve")\n        ].iloc[0]\n        route_daily = daily.loc[\n            daily["alpha"].eq(alpha)\n            & daily["model"].eq("route_specific_curve")\n        ].set_index(["validation_date", "route_id"])\n        shared_daily = daily.loc[\n            daily["alpha"].eq(alpha) & daily["model"].eq("shared_curve")\n        ].set_index(["validation_date", "route_id"])\n        daily_improvement = (\n            shared_daily["event_balanced_mae"]\n            - route_daily["event_balanced_mae"]\n        )\n        comparison_rows.append(\n            {\n                "alpha": alpha,\n                "shared_mae": shared["event_balanced_mae"],\n                "route_specific_mae": route["event_balanced_mae"],\n                "mae_improvement": (\n                    shared["event_balanced_mae"]\n                    - route["event_balanced_mae"]\n                ),\n                "shared_low_0_10_mae": shared["low_0_10_mae"],\n                "route_specific_low_0_10_mae": route["low_0_10_mae"],\n                "low_0_10_mae_improvement": (\n                    shared["low_0_10_mae"] - route["low_0_10_mae"]\n                ),\n                "shared_full_accuracy": shared["full_accuracy"],\n                "route_specific_full_accuracy": route["full_accuracy"],\n                "shared_full_recall": shared["full_recall"],\n                "route_specific_full_recall": route["full_recall"],\n                "shared_full_precision": shared["full_precision"],\n                "route_specific_full_precision": route["full_precision"],\n                "shared_full_f1": shared["full_f1"],\n                "route_specific_full_f1": route["full_f1"],\n                "route_specific_better_date_route_fraction": float(\n                    daily_improvement.gt(0).mean()\n                ),\n                "date_route_comparisons": int(len(daily_improvement)),\n            }\n        )\n    return aggregate, daily, pd.DataFrame(comparison_rows)\n\n\ndef reference_rows(\n    data: pd.DataFrame,\n    route_names: dict[str, str],\n    progress_grid: np.ndarray,\n) -> pd.DataFrame:\n    numeric_reference = {\n        column: float(data[column].median()) for column in CONTROL_NUMERIC\n    }\n    categorical_reference = {\n        column: str(data[column].astype("string").mode(dropna=True).iloc[0])\n        for column in CONTROL_CATEGORICAL\n    }\n    rows: list[dict[str, Any]] = []\n    for route_id, direction, progress in itertools.product(\n        sorted(route_names), DIRECTIONS, progress_grid\n    ):\n        rows.append(\n            {\n                "route_id": route_id,\n                "route_name": route_names[route_id],\n                "direction": direction,\n                "leg_progress": float(progress),\n                **numeric_reference,\n                **categorical_reference,\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef hgb_adjusted_curves(\n    validations: dict[str, pd.DataFrame],\n    route_names: dict[str, str],\n    *,\n    alpha: float,\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\n    data = pd.concat(validations.values(), ignore_index=True)\n    target = (\n        data["hgb_oof_prediction"].to_numpy(dtype=float)\n        - data["snapshot_remaining_seats"].to_numpy(dtype=float)\n    ) / np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)\n    transformer = ProgressDesign(route_specific=True)\n    design = transformer.fit_transform(data)\n    model = fit_ridge(design, target, event_weights(data), alpha=alpha)\n    fitted = model.predict(design)\n    weights = event_weights(data)\n    weighted_mean = np.average(target, weights=weights)\n    weighted_sse = np.sum(weights * np.square(target - fitted))\n    weighted_sst = np.sum(weights * np.square(target - weighted_mean))\n\n    grid = np.linspace(0.05, 0.95, 19)\n    curves = reference_rows(data, route_names, grid)\n    curves["predicted_delta_per_stop"] = model.predict(\n        transformer.transform(curves)\n    )\n    center = curves.loc[np.isclose(curves["leg_progress"], 0.5)].set_index(\n        ["route_id", "direction"]\n    )["predicted_delta_per_stop"]\n    curves["shape_effect"] = [\n        value - center.loc[(route_id, direction)]\n        for value, route_id, direction in curves[\n            ["predicted_delta_per_stop", "route_id", "direction"]\n        ].itertuples(index=False, name=None)\n    ]\n\n    similarity_rows: list[dict[str, Any]] = []\n    for direction in DIRECTIONS:\n        directional = curves.loc[curves["direction"].eq(direction)]\n        series = {\n            route_id: group.sort_values("leg_progress")["shape_effect"].to_numpy()\n            for route_id, group in directional.groupby("route_id")\n        }\n        for left, right in itertools.combinations(sorted(series), 2):\n            left_values = series[left]\n            right_values = series[right]\n            similarity_rows.append(\n                {\n                    "direction": direction,\n                    "left_route_id": left,\n                    "left_route_name": route_names[left],\n                    "right_route_id": right,\n                    "right_route_name": route_names[right],\n                    "shape_correlation": float(\n                        np.corrcoef(left_values, right_values)[0, 1]\n                    ),\n                    "shape_rmse": float(\n                        np.sqrt(np.mean(np.square(left_values - right_values)))\n                    ),\n                    "shape_max_abs_difference": float(\n                        np.max(np.abs(left_values - right_values))\n                    ),\n                }\n            )\n    similarity = pd.DataFrame(similarity_rows)\n    diagnostics = {\n        "surrogate_weighted_rmse_delta_per_stop": float(\n            np.sqrt(weighted_sse / weights.sum())\n        ),\n        "surrogate_weighted_r2": float(1 - weighted_sse / weighted_sst),\n        "mean_pairwise_shape_correlation": float(\n            similarity["shape_correlation"].mean()\n        ),\n        "mean_pairwise_shape_rmse": float(similarity["shape_rmse"].mean()),\n        "max_pairwise_shape_difference": float(\n            similarity["shape_max_abs_difference"].max()\n        ),\n    }\n    for direction in DIRECTIONS:\n        directional = similarity.loc[similarity["direction"].eq(direction)]\n        diagnostics[f"{direction}_mean_shape_correlation"] = float(\n            directional["shape_correlation"].mean()\n        )\n        diagnostics[f"{direction}_mean_shape_rmse"] = float(\n            directional["shape_rmse"].mean()\n        )\n    most_different = similarity.loc[similarity["shape_rmse"].idxmax()]\n    diagnostics["most_different_pair"] = {\n        "direction": str(most_different["direction"]),\n        "routes": [\n            str(most_different["left_route_name"]),\n            str(most_different["right_route_name"]),\n        ],\n        "shape_correlation": float(most_different["shape_correlation"]),\n        "shape_rmse": float(most_different["shape_rmse"]),\n        "shape_max_abs_difference": float(\n            most_different["shape_max_abs_difference"]\n        ),\n    }\n    return curves, similarity, diagnostics\n\n\ndef plot_curves(curves: pd.DataFrame, output_path: Path) -> None:\n    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)\n    for column, row, ylabel in (\n        ("predicted_delta_per_stop", 0, "Adjusted predicted seat change / stop"),\n        ("shape_effect", 1, "Shape effect vs progress=0.50"),\n    ):\n        for direction_index, direction in enumerate(DIRECTIONS):\n            axis = axes[row, direction_index]\n            current = curves.loc[curves["direction"].eq(direction)]\n            for route_name, group in current.groupby("route_name", sort=True):\n                group = group.sort_values("leg_progress")\n                axis.plot(\n                    group["leg_progress"],\n                    group[column],\n                    marker="o",\n                    markersize=2.5,\n                    linewidth=1.5,\n                    label=str(route_name),\n                )\n            axis.axhline(0, color="#888888", linewidth=0.8)\n            axis.set_title("Outbound" if direction == "to_city" else "Return")\n            axis.set_ylabel(ylabel)\n            axis.grid(alpha=0.2)\n            if row == 1:\n                axis.set_xlabel("Normalized leg progress")\n            if row == 0 and direction_index == 1:\n                axis.legend(title="Route", ncol=2, fontsize=8)\n    fig.suptitle("Route-local HGB OOF predictions: context-adjusted progress curves")\n    fig.tight_layout()\n    fig.savefig(output_path, dpi=170, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_primary_improvement(\n    daily: pd.DataFrame, output_path: Path, *, alpha: float\n) -> None:\n    current = daily.loc[daily["alpha"].eq(alpha)]\n    pivot = current.pivot_table(\n        index="route_name",\n        columns="model",\n        values="event_balanced_mae",\n        aggfunc="mean",\n    )\n    improvement = (\n        pivot["shared_curve"] - pivot["route_specific_curve"]\n    ).sort_index()\n    colors = np.where(improvement.ge(0), "#2878B5", "#D1495B")\n    fig, axis = plt.subplots(figsize=(9, 4.8))\n    axis.bar(improvement.index.astype(str), improvement, color=colors)\n    axis.axhline(0, color="#333333", linewidth=0.9)\n    axis.set_ylabel("MAE improvement (shared - route-specific)")\n    axis.set_xlabel("Route")\n    axis.set_title(f"Route-specific progress curve gain (alpha={alpha:g})")\n    axis.grid(axis="y", alpha=0.2)\n    fig.tight_layout()\n    fig.savefig(output_path, dpi=170, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef write_report(\n    output_path: Path,\n    *,\n    validation_dates: list[str],\n    comparison: pd.DataFrame,\n    daily: pd.DataFrame,\n    curve_diagnostics: dict[str, Any],\n) -> None:\n    primary = comparison.loc[comparison["alpha"].eq(PRIMARY_ALPHA)].iloc[0]\n    primary_daily = daily.loc[daily["alpha"].eq(PRIMARY_ALPHA)]\n    pivot = primary_daily.pivot_table(\n        index="route_name",\n        columns="model",\n        values="event_balanced_mae",\n        aggfunc="mean",\n    )\n    pivot["improvement"] = (\n        pivot["shared_curve"] - pivot["route_specific_curve"]\n    )\n    all_alpha_positive = comparison["mae_improvement"].gt(0).all()\n    direction = "지지한다" if primary["mae_improvement"] > 0 else "지지하지 않는다"\n    low_delta = float(primary["low_0_10_mae_improvement"])\n    low_word = "개선" if low_delta > 0 else "악화"\n    most_different = curve_diagnostics["most_different_pair"]\n\n    lines = [\n        "# 노선별 진행률 곡선 비교",\n        "",\n        "## 결론",\n        "",\n        (\n            f"공통 진행률 곡선보다 노선별 진행률 곡선의 OOF MAE가 "\n            f"{primary[\'mae_improvement\']:.3f}석 "\n            f"({\'개선\' if primary[\'mae_improvement\'] > 0 else \'악화\'})되어, "\n            f"현재 표본은 노선 고유 진행 구조 가설을 **{direction}**."\n        ),\n        (\n            f"정규화 강도 4개 모두에서 개선 방향이 같은가: "\n            f"**{\'예\' if all_alpha_positive else \'아니오\'}**."\n        ),\n        (\n            f"주 설정에서는 6개 노선×2개 날짜의 "\n            f"{int(primary[\'date_route_comparisons\'])}개 비교 중 "\n            f"{primary[\'route_specific_better_date_route_fraction\'] * 100:.0f}%가 개선됐다."\n        ),\n        (\n            "다만 모든 6개 노선이 동시에 엄격한 날짜 순서 OOF가 되는 날은 "\n            f"{\', \'.join(validation_dates)} 두 날뿐이므로 확정 결론이 아니라 "\n            "추가 수집으로 재검증할 탐색 결과다."\n        ),\n        "",\n        "## 비교 설계",\n        "",\n        "- 두 모델 모두 노선×방향별 절편과 좌석 상태·거리·시간·차량 통제를 가진다.",\n        "- 공통 모델은 방향별 진행률 spline을 모든 노선이 공유한다.",\n        "- 노선별 모델만 노선×방향별 진행률 spline을 허용한다.",\n        "- 회차 후 1→0인 원 진행률은 뒤집어 두 방향 모두 운행 방향 0→1로 맞췄다.",\n        "- 평가는 각 검증일보다 앞선 데이터만 학습하며, 원 실험 OOF 행 manifest를 그대로 사용했다.",\n        "",\n        "## 주 분석 (alpha=100)",\n        "",\n        "| 지표 | 공통 곡선 | 노선별 곡선 | 차이 |",\n        "|---|---:|---:|---:|",\n        (\n            f"| 전체 MAE | {primary[\'shared_mae\']:.3f} | "\n            f"{primary[\'route_specific_mae\']:.3f} | "\n            f"{primary[\'mae_improvement\']:+.3f} 개선 |"\n        ),\n        (\n            f"| 실제 잔여좌석 ≤10 MAE | {primary[\'shared_low_0_10_mae\']:.3f} | "\n            f"{primary[\'route_specific_low_0_10_mae\']:.3f} | "\n            f"{abs(low_delta):.3f} {low_word} |"\n        ),\n        (\n            f"| 만석 Accuracy | {primary[\'shared_full_accuracy\']:.3f} | "\n            f"{primary[\'route_specific_full_accuracy\']:.3f} | "\n            f"{primary[\'route_specific_full_accuracy\'] - primary[\'shared_full_accuracy\']:+.3f} |"\n        ),\n        (\n            f"| 만석 Recall | {primary[\'shared_full_recall\']:.3f} | "\n            f"{primary[\'route_specific_full_recall\']:.3f} | "\n            f"{primary[\'route_specific_full_recall\'] - primary[\'shared_full_recall\']:+.3f} |"\n        ),\n        (\n            f"| 만석 Precision | {primary[\'shared_full_precision\']:.3f} | "\n            f"{primary[\'route_specific_full_precision\']:.3f} | "\n            f"{primary[\'route_specific_full_precision\'] - primary[\'shared_full_precision\']:+.3f} |"\n        ),\n        (\n            f"| 만석 F1 | {primary[\'shared_full_f1\']:.3f} | "\n            f"{primary[\'route_specific_full_f1\']:.3f} | "\n            f"{primary[\'route_specific_full_f1\'] - primary[\'shared_full_f1\']:+.3f} |"\n        ),\n        "",\n        "### 노선별 평균 MAE",\n        "",\n        "| 노선 | 공통 곡선 | 노선별 곡선 | 개선 |",\n        "|---|---:|---:|---:|",\n    ]\n    for route_name, row in pivot.sort_index().iterrows():\n        lines.append(\n            f"| {route_name} | {row[\'shared_curve\']:.3f} | "\n            f"{row[\'route_specific_curve\']:.3f} | {row[\'improvement\']:+.3f} |"\n        )\n    lines.extend(\n        [\n            "",\n            "## 기존 HGB 예측 곡선 진단",\n            "",\n            (\n                "기존 노선별 HGB의 공통 날짜 OOF 예측을 좌석 상태·거리·시간·차량이 "\n                "같은 기준 조건이 되도록 spline surrogate로 보정했다."\n            ),\n            (\n                f"- surrogate 가중 R²: {curve_diagnostics[\'surrogate_weighted_r2\']:.3f}"\n            ),\n            (\n                f"- 중심화한 곡선의 노선 쌍 평균 상관: "\n                f"{curve_diagnostics[\'mean_pairwise_shape_correlation\']:.3f}"\n            ),\n            (\n                f"- 중심화한 곡선의 노선 쌍 평균 RMSE: "\n                f"{curve_diagnostics[\'mean_pairwise_shape_rmse\']:.3f} 좌석/정류장"\n            ),\n            (\n                f"- 도심행 평균 곡선 상관: "\n                f"{curve_diagnostics[\'to_city_mean_shape_correlation\']:.3f}; "\n                f"회차 후 평균 곡선 상관: "\n                f"{curve_diagnostics[\'return_mean_shape_correlation\']:.3f}"\n            ),\n            (\n                f"- 최대 노선 쌍 곡선 차이: "\n                f"{curve_diagnostics[\'max_pairwise_shape_difference\']:.3f} 좌석/정류장"\n            ),\n            (\n                f"- 가장 다른 쌍: {most_different[\'routes\'][0]}–"\n                f"{most_different[\'routes\'][1]} ({most_different[\'direction\']}, "\n                f"상관 {most_different[\'shape_correlation\']:.3f})"\n            ),\n            "",\n            "`route_progress_prediction_curves.png`의 아래 행은 진행률 0.5에서 각 곡선을 "\n            "0으로 맞춘 형태 비교다. 위 행의 높이 차이는 노선 평균 효과까지 포함하고, "\n            "아래 행의 차이가 우리가 확인하려는 노선 고유 진행 구조다.",\n            "",\n            "## 해석 시 주의",\n            "",\n            "- 이 결과는 인과효과가 아니라 관측 데이터의 조건부 예측 관계다.",\n            "- 저잔여좌석 MAE는 개선되지 않았고 만석 Recall도 매우 낮아, 혼잡 예측 개선의 증거로 해석하면 안 된다.",\n            "- 1200·1500·2000·3000은 공통 OOF 이전 학습일이 짧아 곡선 불확실성이 크다.",\n            "- 수집일이 늘어나면 같은 프로토콜을 다시 실행해 날짜별 개선 부호와 곡선 안정성을 확인해야 한다.",\n        ]\n    )\n    output_path.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="노선별 진행률-예측 곡선과 공통/노선별 곡선 OOF 비교"\n    )\n    parser.add_argument("--model-results", type=Path, default=DEFAULT_MODEL_RESULTS)\n    parser.add_argument("--cache-dir", type=Path, default=DEFAULT_CACHE_DIR)\n    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)\n    parser.add_argument("--alphas", nargs="*", type=float, default=DEFAULT_ALPHAS)\n    args = parser.parse_args()\n    alphas = tuple(float(value) for value in args.alphas)\n    if PRIMARY_ALPHA not in alphas:\n        raise ValueError(f"주 분석 alpha={PRIMARY_ALPHA:g}가 --alphas에 필요합니다.")\n\n    summary = json.loads(\n        (args.model_results / "summary.json").read_text(encoding="utf-8")\n    )\n    manifest = pd.read_csv(args.model_results / "oof_predictions.csv.gz")\n    validation_dates = common_validation_dates(summary["route_results"])\n    if len(validation_dates) < 2:\n        raise ValueError("모든 노선이 공유하는 OOF 검증일이 2일 미만입니다.")\n    route_names = {\n        str(route["route_id"]): str(route["route_name"])\n        for route in summary["route_results"]\n    }\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    route_frames, validations = load_route_frames(\n        summary,\n        manifest,\n        args.cache_dir,\n        validation_dates,\n    )\n    aggregate, daily, comparison = evaluate_shared_vs_route_curves(\n        route_frames, validations, alphas=alphas\n    )\n    curves, similarity, curve_diagnostics = hgb_adjusted_curves(\n        validations, route_names, alpha=PRIMARY_ALPHA\n    )\n\n    aggregate.to_csv(args.output_dir / "model_metrics.csv", index=False)\n    daily.to_csv(args.output_dir / "daily_route_metrics.csv", index=False)\n    comparison.to_csv(args.output_dir / "model_comparison.csv", index=False)\n    curves.to_csv(args.output_dir / "adjusted_hgb_progress_curves.csv", index=False)\n    similarity.to_csv(args.output_dir / "curve_pairwise_similarity.csv", index=False)\n    plot_curves(curves, args.output_dir / "route_progress_prediction_curves.png")\n    plot_primary_improvement(\n        daily,\n        args.output_dir / "route_specific_curve_mae_gain.png",\n        alpha=PRIMARY_ALPHA,\n    )\n    write_report(\n        args.output_dir / "report.md",\n        validation_dates=validation_dates,\n        comparison=comparison,\n        daily=daily,\n        curve_diagnostics=curve_diagnostics,\n    )\n    result_summary = {\n        "question": (\n            "Are route-local HGB predictions shaped differently by normalized "\n            "route progress, and do route-specific progress curves improve OOF "\n            "prediction over a shared bus-route curve?"\n        ),\n        "validation_dates": validation_dates,\n        "routes": route_names,\n        "progress_definition": (\n            "midpoint between snapshot and target stops; each direction normalized "\n            "to travel direction 0 -> 1"\n        ),\n        "comparison_controls": {\n            "both_models_include": (\n                "route-by-direction intercepts, current seats, gap/ETA, recent seat "\n                "trajectory, calendar time, and vehicle/observation state"\n            ),\n            "only_difference": (\n                "direction-shared progress spline versus route-by-direction progress spline"\n            ),\n            "alphas": alphas,\n            "primary_alpha": PRIMARY_ALPHA,\n        },\n        "primary_comparison": comparison.loc[\n            comparison["alpha"].eq(PRIMARY_ALPHA)\n        ].iloc[0].to_dict(),\n        "curve_diagnostics": curve_diagnostics,\n        "artifacts": {\n            "report": str(args.output_dir / "report.md"),\n            "model_comparison": str(args.output_dir / "model_comparison.csv"),\n            "daily_route_metrics": str(args.output_dir / "daily_route_metrics.csv"),\n            "adjusted_curves": str(\n                args.output_dir / "adjusted_hgb_progress_curves.csv"\n            ),\n            "curve_similarity": str(\n                args.output_dir / "curve_pairwise_similarity.csv"\n            ),\n            "curve_plot": str(\n                args.output_dir / "route_progress_prediction_curves.png"\n            ),\n            "mae_gain_plot": str(\n                args.output_dir / "route_specific_curve_mae_gain.png"\n            ),\n        },\n        "limitation": (\n            "Only two validation dates are shared by all six routes; treat this as "\n            "exploratory evidence and rerun after more collection days."\n        ),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result_summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(\n        f"완료: {len(route_names)}개 노선, 공통 OOF {validation_dates}, "\n        f"결과 {args.output_dir}",\n        flush=True,\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/probabilistic_seat_regression.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, log_loss, mean_pinball_loss\nfrom sklearn.pipeline import Pipeline\n\nfrom all_prearrival_seat_regression import (\n    ALL_PREARRIVAL,\n    DYNAMIC_ALL_PREARRIVAL,\n    ENGINEERED_ALL_PREARRIVAL,\n    build_all_prearrival_table,\n    clip_seats,\n    event_balanced_metrics,\n    event_bias,\n    event_weights,\n)\nfrom model_feasibility import (\n    FeatureSet,\n    _as_records,\n    build_model_table,\n    build_visits,\n    json_ready,\n    load_data,\n    make_preprocessor,\n    prepare_subset,\n)\nfrom hypothesis_model_search import PREVIOUS_BUS_SEAT_ALL_PREARRIVAL\nfrom tminus_feasibility import ROUTE_ID, ROUTE_NAME, prepare_raw_locations\n\n\nQUANTILES = (0.1, 0.5, 0.9)\nCDF_THRESHOLDS = (0, 5, 10, 20)\n\n\ndef quantile_model(feature_set: FeatureSet, quantile: float, seed: int) -> Pipeline:\n    return Pipeline(\n        [\n            ("features", clone(make_preprocessor(feature_set))),\n            (\n                "regressor",\n                HistGradientBoostingRegressor(\n                    loss="quantile",\n                    quantile=quantile,\n                    learning_rate=0.05,\n                    max_iter=250,\n                    max_leaf_nodes=15,\n                    min_samples_leaf=20,\n                    l2_regularization=1.0,\n                    random_state=seed,\n                ),\n            ),\n        ]\n    )\n\n\ndef cumulative_model(feature_set: FeatureSet, seed: int) -> Pipeline:\n    return Pipeline(\n        [\n            ("features", clone(make_preprocessor(feature_set))),\n            (\n                "classifier",\n                HistGradientBoostingClassifier(\n                    loss="log_loss",\n                    learning_rate=0.05,\n                    max_iter=250,\n                    max_leaf_nodes=15,\n                    min_samples_leaf=20,\n                    l2_regularization=1.0,\n                    random_state=seed,\n                ),\n            ),\n        ]\n    )\n\n\ndef weighted_quantile(values: np.ndarray, weights: np.ndarray, q: float) -> float:\n    order = np.argsort(values)\n    sorted_values = values[order]\n    sorted_weights = weights[order]\n    cumulative = np.cumsum(sorted_weights)\n    cutoff = q * cumulative[-1]\n    return float(sorted_values[min(np.searchsorted(cumulative, cutoff), len(values) - 1)])\n\n\ndef conformal_group(data: pd.DataFrame) -> pd.Series:\n    seat_band = pd.cut(\n        data["snapshot_remaining_seats"],\n        bins=[-np.inf, 5, 10, 20, np.inf],\n        labels=["seats_0_5", "seats_6_10", "seats_11_20", "seats_21_plus"],\n    )\n    stop_band = pd.cut(\n        data["target_stop_gap"],\n        bins=[0, 2, 5, 10, np.inf],\n        labels=["stops_1_2", "stops_3_5", "stops_6_10", "stops_11_plus"],\n    )\n    return seat_band.astype(str) + "|" + stop_band.astype(str)\n\n\ndef conditional_conformal_adjustments(\n    calibration: pd.DataFrame,\n    conformity: np.ndarray,\n) -> tuple[dict[str, float], float]:\n    weights = event_weights(calibration)\n    global_adjustment = weighted_quantile(conformity, weights, 0.8)\n    groups = conformal_group(calibration)\n    adjustments: dict[str, float] = {}\n    for group_name in groups.unique():\n        mask = groups.eq(group_name).to_numpy()\n        event_count = calibration.loc[mask, "event_id"].nunique()\n        if event_count < 30:\n            continue\n        adjustments[str(group_name)] = weighted_quantile(\n            conformity[mask], weights[mask], 0.8\n        )\n    return adjustments, global_adjustment\n\n\ndef fit_quantiles(\n    train: pd.DataFrame,\n    calibration: pd.DataFrame,\n    test: pd.DataFrame,\n    feature_set: FeatureSet,\n    seed: int,\n) -> tuple[\n    dict[float, np.ndarray],\n    dict[float, np.ndarray],\n    float,\n    dict[str, float],\n]:\n    train_delta = (\n        train["label_seats"].to_numpy(dtype=float)\n        - train["snapshot_remaining_seats"].to_numpy(dtype=float)\n    )\n    cal_predictions: dict[float, np.ndarray] = {}\n    test_predictions: dict[float, np.ndarray] = {}\n    for quantile in QUANTILES:\n        model = quantile_model(feature_set, quantile, seed)\n        model.fit(\n            train[feature_set.columns],\n            train_delta,\n            regressor__sample_weight=event_weights(train),\n        )\n        cal_predictions[quantile] = clip_seats(\n            calibration["snapshot_remaining_seats"].to_numpy(dtype=float)\n            + model.predict(calibration[feature_set.columns]),\n            calibration["capacity"],\n        )\n        test_predictions[quantile] = clip_seats(\n            test["snapshot_remaining_seats"].to_numpy(dtype=float)\n            + model.predict(test[feature_set.columns]),\n            test["capacity"],\n        )\n\n    median_bias = event_bias(calibration, cal_predictions[0.5])\n    for quantile in QUANTILES:\n        cal_predictions[quantile] = clip_seats(\n            cal_predictions[quantile] + median_bias, calibration["capacity"]\n        )\n        test_predictions[quantile] = clip_seats(\n            test_predictions[quantile] + median_bias, test["capacity"]\n        )\n\n    cal_matrix = np.sort(\n        np.column_stack([cal_predictions[q] for q in QUANTILES]), axis=1\n    )\n    test_matrix = np.sort(\n        np.column_stack([test_predictions[q] for q in QUANTILES]), axis=1\n    )\n    cal_y = calibration["label_seats"].to_numpy(dtype=float)\n    conformity = np.maximum.reduce(\n        [\n            cal_matrix[:, 0] - cal_y,\n            cal_y - cal_matrix[:, 2],\n            np.zeros(len(calibration), dtype=float),\n        ]\n    )\n    group_adjustments, global_adjustment = conditional_conformal_adjustments(\n        calibration, conformity\n    )\n    calibration_adjustment = conformal_group(calibration).map(\n        group_adjustments\n    ).fillna(global_adjustment).to_numpy(dtype=float)\n    test_adjustment = conformal_group(test).map(group_adjustments).fillna(\n        global_adjustment\n    ).to_numpy(dtype=float)\n    cal_matrix[:, 0] -= calibration_adjustment\n    cal_matrix[:, 2] += calibration_adjustment\n    test_matrix[:, 0] -= test_adjustment\n    test_matrix[:, 2] += test_adjustment\n    cal_matrix[:, 0] = np.maximum(cal_matrix[:, 0], 0)\n    test_matrix[:, 0] = np.maximum(test_matrix[:, 0], 0)\n    cal_matrix[:, 2] = np.minimum(\n        cal_matrix[:, 2], calibration["capacity"].to_numpy(dtype=float)\n    )\n    test_matrix[:, 2] = np.minimum(\n        test_matrix[:, 2], test["capacity"].to_numpy(dtype=float)\n    )\n    return (\n        {q: cal_matrix[:, index] for index, q in enumerate(QUANTILES)},\n        {q: test_matrix[:, index] for index, q in enumerate(QUANTILES)},\n        median_bias,\n        {"global": global_adjustment, **group_adjustments},\n    )\n\n\ndef weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:\n    return float(np.average(values, weights=weights))\n\n\ndef quantile_metrics(\n    data: pd.DataFrame, predictions: dict[float, np.ndarray]\n) -> dict[str, Any]:\n    y = data["label_seats"].to_numpy(dtype=float)\n    weights = event_weights(data)\n    lower = predictions[0.1]\n    median = predictions[0.5]\n    upper = predictions[0.9]\n    covered = (y >= lower) & (y <= upper)\n    width = upper - lower\n    low = y <= 5\n    result: dict[str, Any] = {\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n        "median": event_balanced_metrics(\n            data, median, model="quantile_median", split="test"\n        ),\n        "interval_80_event_balanced_coverage": weighted_mean(covered, weights),\n        "interval_80_event_balanced_mean_width": weighted_mean(width, weights),\n        "low_0_5_interval_coverage": (\n            weighted_mean(covered[low], weights[low]) if low.any() else np.nan\n        ),\n        "low_0_5_interval_mean_width": (\n            weighted_mean(width[low], weights[low]) if low.any() else np.nan\n        ),\n    }\n    for quantile in QUANTILES:\n        result[f"pinball_q{int(quantile * 100):02d}"] = float(\n            mean_pinball_loss(\n                y,\n                predictions[quantile],\n                alpha=quantile,\n                sample_weight=weights,\n            )\n        )\n    return result\n\n\ndef platt_calibrate(\n    calibration_probability: np.ndarray,\n    calibration_y: np.ndarray,\n    test_probability: np.ndarray,\n    weights: np.ndarray,\n) -> np.ndarray:\n    epsilon = 1e-6\n    cal_logit = np.log(\n        np.clip(calibration_probability, epsilon, 1 - epsilon)\n        / np.clip(1 - calibration_probability, epsilon, 1 - epsilon)\n    )\n    test_logit = np.log(\n        np.clip(test_probability, epsilon, 1 - epsilon)\n        / np.clip(1 - test_probability, epsilon, 1 - epsilon)\n    )\n    calibrator = LogisticRegression(C=1000.0, max_iter=1000, random_state=42)\n    calibrator.fit(cal_logit.reshape(-1, 1), calibration_y, sample_weight=weights)\n    return calibrator.predict_proba(test_logit.reshape(-1, 1))[:, 1]\n\n\ndef fit_cdf(\n    train: pd.DataFrame,\n    calibration: pd.DataFrame,\n    test: pd.DataFrame,\n    feature_set: FeatureSet,\n    seed: int,\n) -> np.ndarray:\n    calibrated: list[np.ndarray] = []\n    for threshold in CDF_THRESHOLDS:\n        train_y = train["label_seats"].le(threshold).astype(int).to_numpy()\n        cal_y = calibration["label_seats"].le(threshold).astype(int).to_numpy()\n        model = cumulative_model(feature_set, seed)\n        model.fit(\n            train[feature_set.columns],\n            train_y,\n            classifier__sample_weight=event_weights(train),\n        )\n        raw_cal = model.predict_proba(calibration[feature_set.columns])[:, 1]\n        raw_test = model.predict_proba(test[feature_set.columns])[:, 1]\n        calibrated.append(\n            platt_calibrate(\n                raw_cal,\n                cal_y,\n                raw_test,\n                event_weights(calibration),\n            )\n        )\n    # 개별 누적 분류기의 표본 오차로 P(≤0)>P(≤5)가 되는 경우를 보정한다.\n    return np.maximum.accumulate(np.column_stack(calibrated), axis=1)\n\n\ndef cdf_metrics(data: pd.DataFrame, cdf: np.ndarray) -> pd.DataFrame:\n    weights = event_weights(data)\n    rows: list[dict[str, Any]] = []\n    for index, threshold in enumerate(CDF_THRESHOLDS):\n        y = data["label_seats"].le(threshold).astype(int).to_numpy()\n        probability = np.clip(cdf[:, index], 1e-6, 1 - 1e-6)\n        rows.append(\n            {\n                "threshold_seats": threshold,\n                "events_positive": int(\n                    data.loc[data["label_seats"].le(threshold), "event_id"].nunique()\n                ),\n                "weighted_prevalence": weighted_mean(y, weights),\n                "weighted_mean_probability": weighted_mean(probability, weights),\n                "weighted_brier": weighted_mean((y - probability) ** 2, weights),\n                "weighted_log_loss": float(\n                    log_loss(y, probability, sample_weight=weights, labels=[0, 1])\n                ),\n                "weighted_average_precision": float(\n                    average_precision_score(y, probability, sample_weight=weights)\n                ),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef trip_cluster_bootstrap_cdf_delta(\n    data: pd.DataFrame,\n    candidate: np.ndarray,\n    baseline: np.ndarray,\n    *,\n    threshold_index: int,\n    seed: int,\n    repeats: int = 500,\n) -> dict[str, Any]:\n    """trip을 재표집해 previous-bus 위험 헤드의 증분 불확실성을 잰다."""\n    threshold = CDF_THRESHOLDS[threshold_index]\n    y = data["label_seats"].le(threshold).astype(int).to_numpy()\n    weights = 1.0 / data.groupby("event_id")["event_id"].transform("size").to_numpy()\n    candidate_probability = candidate[:, threshold_index]\n    baseline_probability = baseline[:, threshold_index]\n    groups = [\n        np.asarray(indices, dtype=int)\n        for indices in data.groupby("trip_id", sort=False).indices.values()\n    ]\n\n    def score(indices: np.ndarray) -> tuple[float, float]:\n        sample_y = y[indices]\n        sample_weights = weights[indices]\n        if np.unique(sample_y).size < 2:\n            return np.nan, np.nan\n        candidate_ap = average_precision_score(\n            sample_y,\n            candidate_probability[indices],\n            sample_weight=sample_weights,\n        )\n        baseline_ap = average_precision_score(\n            sample_y,\n            baseline_probability[indices],\n            sample_weight=sample_weights,\n        )\n        candidate_brier = np.average(\n            (sample_y - candidate_probability[indices]) ** 2,\n            weights=sample_weights,\n        )\n        baseline_brier = np.average(\n            (sample_y - baseline_probability[indices]) ** 2,\n            weights=sample_weights,\n        )\n        return candidate_ap - baseline_ap, baseline_brier - candidate_brier\n\n    all_indices = np.arange(len(data))\n    observed_ap, observed_brier = score(all_indices)\n    rng = np.random.default_rng(seed + threshold_index)\n    bootstrap_ap = np.empty(repeats, dtype=float)\n    bootstrap_brier = np.empty(repeats, dtype=float)\n    for repeat in range(repeats):\n        sampled_groups = rng.integers(0, len(groups), size=len(groups))\n        sampled_indices = np.concatenate([groups[index] for index in sampled_groups])\n        bootstrap_ap[repeat], bootstrap_brier[repeat] = score(sampled_indices)\n    valid_ap = bootstrap_ap[np.isfinite(bootstrap_ap)]\n    valid_brier = bootstrap_brier[np.isfinite(bootstrap_brier)]\n    return {\n        "threshold_seats": int(threshold),\n        "unit": "trip_id",\n        "trips": int(len(groups)),\n        "repeats": int(repeats),\n        "average_precision_delta_candidate_minus_dynamic": float(observed_ap),\n        "average_precision_delta_95": [\n            float(value) for value in np.quantile(valid_ap, [0.025, 0.975])\n        ],\n        "probability_ap_improves": float((valid_ap > 0).mean()),\n        "brier_improvement_dynamic_minus_candidate": float(observed_brier),\n        "brier_improvement_95": [\n            float(value) for value in np.quantile(valid_brier, [0.025, 0.975])\n        ],\n        "probability_brier_improves": float((valid_brier > 0).mean()),\n    }\n\n\ndef bucket_distribution(cdf: np.ndarray) -> np.ndarray:\n    buckets = np.column_stack(\n        [\n            cdf[:, 0],\n            cdf[:, 1] - cdf[:, 0],\n            cdf[:, 2] - cdf[:, 1],\n            cdf[:, 3] - cdf[:, 2],\n            1 - cdf[:, 3],\n        ]\n    )\n    buckets = np.clip(buckets, 1e-8, 1)\n    return buckets / buckets.sum(axis=1, keepdims=True)\n\n\ndef distribution_log_loss(data: pd.DataFrame, buckets: np.ndarray) -> float:\n    y = data["label_seats"].to_numpy(dtype=float)\n    true_bucket = np.select(\n        [y == 0, y <= 5, y <= 10, y <= 20],\n        [0, 1, 2, 3],\n        default=4,\n    ).astype(int)\n    probability = buckets[np.arange(len(y)), true_bucket]\n    return weighted_mean(-np.log(np.clip(probability, 1e-8, 1)), event_weights(data))\n\n\ndef reliability_table(data: pd.DataFrame, probability: np.ndarray) -> pd.DataFrame:\n    bins = [0, 0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]\n    labels = ["0-2%", "2-5%", "5-10%", "10-20%", "20-40%", "40-60%", "60-80%", "80-100%"]\n    frame = data[["event_id", "label_seats"]].copy()\n    frame["probability"] = probability\n    frame["actual"] = frame["label_seats"].le(5).astype(float)\n    frame["weight"] = event_weights(data)\n    frame["bin"] = pd.cut(probability, bins=bins, labels=labels, include_lowest=True)\n    rows: list[dict[str, Any]] = []\n    for label in labels:\n        group = frame.loc[frame["bin"].eq(label)]\n        if group.empty:\n            continue\n        rows.append(\n            {\n                "probability_bin": label,\n                "rows": int(len(group)),\n                "events": int(group["event_id"].nunique()),\n                "mean_predicted": weighted_mean(\n                    group["probability"].to_numpy(), group["weight"].to_numpy()\n                ),\n                "observed_rate": weighted_mean(\n                    group["actual"].to_numpy(), group["weight"].to_numpy()\n                ),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef engineered_feature_profile(train: pd.DataFrame) -> pd.DataFrame:\n    delta = train["label_seats"] - train["snapshot_remaining_seats"]\n    added_numeric = [\n        column\n        for column in ENGINEERED_ALL_PREARRIVAL.numeric\n        if column not in ALL_PREARRIVAL.numeric\n    ]\n    rows: list[dict[str, Any]] = []\n    for column in added_numeric:\n        values = train[column]\n        valid = values.notna() & delta.notna()\n        rows.append(\n            {\n                "feature": column,\n                "coverage": float(values.notna().mean()),\n                "unique_values": int(values.nunique(dropna=True)),\n                "median": float(values.median()) if values.notna().any() else np.nan,\n                "spearman_with_arrival_delta": (\n                    float(values.loc[valid].corr(delta.loc[valid], method="spearman"))\n                    if valid.sum() >= 2\n                    else np.nan\n                ),\n            }\n        )\n    return pd.DataFrame(rows).sort_values(\n        "spearman_with_arrival_delta", key=lambda column: column.abs(), ascending=False\n    )\n\n\ndef marginal_bucket_probabilities(train: pd.DataFrame) -> np.ndarray:\n    events = train.drop_duplicates("event_id")\n    y = events["label_seats"].to_numpy(dtype=float)\n    bucket = np.select(\n        [y == 0, y <= 5, y <= 10, y <= 20],\n        [0, 1, 2, 3],\n        default=4,\n    ).astype(int)\n    counts = np.bincount(bucket, minlength=5).astype(float) + 1.0\n    return counts / counts.sum()\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="도착 잔여좌석 확률분포 모델")\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/probabilistic_seat_results"),\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    locations, stations = load_data(args.db, ROUTE_ID)\n    visits = build_visits(locations, stations)\n    table, turnaround_seq = build_model_table(\n        visits, stations, label_target="arrival"\n    )\n    source = prepare_subset(table)\n    source = source.loc[source["label_quality"].eq("A")].copy()\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n    snapshots = build_all_prearrival_table(\n        source,\n        visits,\n        by_vehicle,\n        turnaround_seq=turnaround_seq,\n        max_seq=int(stations["station_seq"].max()),\n    )\n    train = snapshots.loc[snapshots["date"].isin(["2026-08-04", "2026-08-05"])]\n    calibration = snapshots.loc[snapshots["date"].eq("2026-08-06")]\n    test = snapshots.loc[snapshots["date"].eq("2026-08-07")]\n\n    ablation_rows: list[dict[str, Any]] = []\n    prediction_sets: dict[str, dict[float, np.ndarray]] = {}\n    calibration_details: dict[str, dict[str, Any]] = {}\n    for feature_set in [\n        ALL_PREARRIVAL,\n        DYNAMIC_ALL_PREARRIVAL,\n        PREVIOUS_BUS_SEAT_ALL_PREARRIVAL,\n        ENGINEERED_ALL_PREARRIVAL,\n    ]:\n        calibration_quantiles, test_quantiles, bias, conformal = fit_quantiles(\n            train, calibration, test, feature_set, args.seed\n        )\n        prediction_sets[feature_set.name] = test_quantiles\n        median_metrics = event_balanced_metrics(\n            test,\n            test_quantiles[0.5],\n            model=feature_set.name,\n            split="test_2026-08-07",\n        )\n        median_metrics["feature_set"] = feature_set.name\n        median_metrics["selection_calibration_mae"] = event_balanced_metrics(\n            calibration,\n            calibration_quantiles[0.5],\n            model=feature_set.name,\n            split="calibration_2026-08-06",\n        )["event_balanced_mae_seats"]\n        ablation_rows.append(median_metrics)\n        calibration_details[feature_set.name] = {\n            "median_bias_seats": bias,\n            "conformal_interval_adjustment_seats": conformal,\n        }\n\n    ablation = pd.DataFrame(ablation_rows)\n    selected_feature = (\n        ablation.sort_values("selection_calibration_mae").iloc[0]["feature_set"]\n    )\n    selected_quantiles = prediction_sets[str(selected_feature)]\n    quantiles = quantile_metrics(test, selected_quantiles)\n    feature_profile = engineered_feature_profile(train)\n    marginal_probability = marginal_bucket_probabilities(train)\n    marginal_buckets = np.tile(marginal_probability, (len(test), 1))\n    marginal_bucket_log_loss = distribution_log_loss(test, marginal_buckets)\n\n    cdf_frames: list[pd.DataFrame] = []\n    cdf_sets: dict[str, np.ndarray] = {}\n    distribution_rows: list[dict[str, Any]] = []\n    for feature_set in [\n        DYNAMIC_ALL_PREARRIVAL,\n        PREVIOUS_BUS_SEAT_ALL_PREARRIVAL,\n        ENGINEERED_ALL_PREARRIVAL,\n    ]:\n        feature_cdf = fit_cdf(\n            train, calibration, test, feature_set, args.seed\n        )\n        cdf_sets[feature_set.name] = feature_cdf\n        feature_metrics = cdf_metrics(test, feature_cdf)\n        feature_metrics["feature_set"] = feature_set.name\n        cdf_frames.append(feature_metrics)\n        feature_bucket_log_loss = distribution_log_loss(\n            test, bucket_distribution(feature_cdf)\n        )\n        distribution_rows.append(\n            {\n                "feature_set": feature_set.name,\n                "seat_bucket_log_loss": feature_bucket_log_loss,\n                "skill_vs_marginal": (\n                    1 - feature_bucket_log_loss / marginal_bucket_log_loss\n                ),\n            }\n        )\n    cdf_table = pd.concat(cdf_frames, ignore_index=True)\n    distribution_ablation = pd.DataFrame(distribution_rows)\n    distribution_feature_set = DYNAMIC_ALL_PREARRIVAL.name\n    cdf = cdf_sets[distribution_feature_set]\n    buckets = bucket_distribution(cdf)\n    reliability = reliability_table(test, cdf[:, 1])\n    model_bucket_log_loss = distribution_log_loss(test, buckets)\n\n    prediction_output = test[\n        [\n            "event_id",\n            "date",\n            "event_time",\n            "snapshot_time",\n            "trip_id",\n            "minutes_to_arrival",\n            "target_stop_gap",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "previous_bus_departure_seats",\n            "previous_bus_departure_age_minutes",\n            "previous_bus_headway_minutes",\n            "previous_3_bus_departure_mean",\n        ]\n    ].copy()\n    prediction_output["q10"] = selected_quantiles[0.1]\n    prediction_output["q50"] = selected_quantiles[0.5]\n    prediction_output["q90"] = selected_quantiles[0.9]\n    for index, threshold in enumerate(CDF_THRESHOLDS):\n        prediction_output[f"p_le_{threshold}"] = cdf[:, index]\n    previous_bus_cdf = cdf_sets[ENGINEERED_ALL_PREARRIVAL.name]\n    previous_bus_seat_cdf = cdf_sets[PREVIOUS_BUS_SEAT_ALL_PREARRIVAL.name]\n    prediction_output["experimental_previous_bus_seat_p_le_0"] = (\n        previous_bus_seat_cdf[:, 0]\n    )\n    prediction_output["experimental_previous_bus_seat_p_le_5"] = (\n        previous_bus_seat_cdf[:, 1]\n    )\n    prediction_output["experimental_previous_bus_p_le_0"] = previous_bus_cdf[:, 0]\n    prediction_output["experimental_previous_bus_p_le_5"] = previous_bus_cdf[:, 1]\n    for index, name in enumerate(["p_0", "p_1_5", "p_6_10", "p_11_20", "p_21_plus"]):\n        prediction_output[name] = buckets[:, index]\n\n    ablation.to_csv(args.output_dir / "feature_ablation.csv", index=False)\n    cdf_table.to_csv(args.output_dir / "cdf_metrics.csv", index=False)\n    distribution_ablation.to_csv(\n        args.output_dir / "distribution_feature_ablation.csv", index=False\n    )\n    reliability.to_csv(args.output_dir / "low_0_5_reliability.csv", index=False)\n    feature_profile.to_csv(args.output_dir / "engineered_feature_profile.csv", index=False)\n    prediction_output.to_csv(args.output_dir / "test_distributions.csv", index=False)\n    result = {\n        "route_id": ROUTE_ID,\n        "route_name": ROUTE_NAME,\n        "target": "probability distribution of arrival_seats before boarding",\n        "label_quality": ["A"],\n        "quantiles": list(QUANTILES),\n        "cdf_thresholds": list(CDF_THRESHOLDS),\n        "split": {\n            "train": ["2026-08-04", "2026-08-05"],\n            "calibration": ["2026-08-06"],\n            "evaluation_test": ["2026-08-07"],\n        },\n        "selected_feature_set": str(selected_feature),\n        "quantile_selection_rule": "lowest event-balanced MAE on 2026-08-06 calibration day",\n        "feature_ablation": _as_records(ablation),\n        "calibration": calibration_details,\n        "quantile_metrics": json_ready(quantiles),\n        "cdf_metrics": _as_records(cdf_table),\n        "distribution_feature_set": distribution_feature_set,\n        "distribution_feature_ablation": _as_records(distribution_ablation),\n        "previous_bus_seat_risk_bootstrap_vs_dynamic": [\n            trip_cluster_bootstrap_cdf_delta(\n                test,\n                previous_bus_seat_cdf,\n                cdf_sets[DYNAMIC_ALL_PREARRIVAL.name],\n                threshold_index=index,\n                seed=args.seed,\n            )\n            for index in (0, 1)\n        ],\n        "previous_bus_asof_rule": (\n            "latest direct stateCd=2 departure at the same target stop and direction "\n            "strictly before snapshot_time; maximum age 180 minutes"\n        ),\n        "recommended_feature_routing": {\n            "arrival_seat_quantiles_and_general_buckets": DYNAMIC_ALL_PREARRIVAL.name,\n            "experimental_full_risk_head": ENGINEERED_ALL_PREARRIVAL.name,\n            "experimental_low_0_5_risk_head": PREVIOUS_BUS_SEAT_ALL_PREARRIVAL.name,\n        },\n        "seat_bucket_log_loss": model_bucket_log_loss,\n        "marginal_bucket_baseline_log_loss": marginal_bucket_log_loss,\n        "bucket_log_loss_skill_vs_marginal": (\n            1 - model_bucket_log_loss / marginal_bucket_log_loss\n        ),\n        "marginal_bucket_probabilities": {\n            name: float(marginal_probability[index])\n            for index, name in enumerate(["0", "1-5", "6-10", "11-20", "21+"])\n        },\n        "low_0_5_reliability": _as_records(reliability),\n        "engineered_feature_profile": _as_records(feature_profile),\n        "engineered_features": [\n            column\n            for column in ENGINEERED_ALL_PREARRIVAL.columns\n            if column not in ALL_PREARRIVAL.columns\n        ],\n        "previous_bus_features": [\n            column\n            for column in ENGINEERED_ALL_PREARRIVAL.columns\n            if column not in DYNAMIC_ALL_PREARRIVAL.columns\n        ],\n        "limitations": [\n            "평가일은 이전 피처 실험에서도 사용되어 더 이상 untouched test가 아니다.",\n            "평가가 하루이며 저잔여 도착 사건은 41개뿐이다.",\n            "분위수 구간은 보정일의 사건 균형 conformal 보정 한 번만 적용했다.",\n            "누적확률 모델은 임계값별 이진 모델을 보정한 뒤 단조성을 강제했다.",\n            "날씨, 학기 여부, 실제 승하차 인원과 대기열은 아직 피처에 없다.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/latest_main_model_overfit_ablation.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nfrom hypothesis_model_search import (\n    OBSERVED_SEAT_CEILING_PARAM,\n    EnsembleSpec,\n    add_observed_capacity_features,\n    combine_weighted_oof,\n    postprocess_ensemble_prediction,\n    strict_forward_bias_predictions,\n)\nfrom latest_main_model_feature_recheck import (\n    LATEST_COMPLETE_DATES,\n    LATEST_PARTIAL_DATE,\n    rolling_folds,\n)\nfrom main_model_feature_augmentation import (\n    component_candidates,\n    feature_sets,\n    prepare_augmented,\n    required_metrics,\n    run_component,\n)\nfrom linear_feature_experiment import json_ready\nfrom model_feasibility import FeatureSet\n\n\nOBSERVED_CEILING_FEATURES = (\n    "observed_ceiling_capacity",\n    "observed_ceiling_load_ratio",\n    "observed_ceiling_load_gap",\n)\nCOORDINATE_FEATURES = ("x", "y")\nPRIMARY_NAME = "primary_40_empirical_cap"\n\n\ndef ablation_candidates() -> dict[str, tuple[FeatureSet, bool]]:\n    primary = feature_sets()["importance_pruned"]\n\n    def without(name: str, removed: tuple[str, ...]) -> FeatureSet:\n        numeric = tuple(column for column in primary.numeric if column not in removed)\n        return FeatureSet(name=name, numeric=numeric, categorical=primary.categorical)\n\n    no_observed = without("no_observed_features_37", OBSERVED_CEILING_FEATURES)\n    no_coordinates = without("no_coordinates_38", COORDINATE_FEATURES)\n    clean = without(\n        "clean_35",\n        (*OBSERVED_CEILING_FEATURES, *COORDINATE_FEATURES),\n    )\n    return {\n        PRIMARY_NAME: (primary, True),\n        "primary_40_nominal_cap": (primary, False),\n        "no_observed_features_37_empirical_cap": (no_observed, True),\n        "no_observed_features_37_nominal_cap": (no_observed, False),\n        "no_coordinates_38_empirical_cap": (no_coordinates, True),\n        "clean_35_empirical_cap": (clean, True),\n        "clean_35_nominal_cap": (clean, False),\n    }\n\n\ndef train_schema(\n    schema_name: str,\n    features: FeatureSet,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    *,\n    seed: int,\n) -> pd.DataFrame:\n    components: dict[str, pd.DataFrame] = {}\n    # Keep the exact v1.0.0 model families and parameters.\n    for candidate in component_candidates():\n        print(f"[{schema_name}] {candidate.name}", flush=True)\n        output, _, _ = run_component(\n            candidate,\n            features,\n            folds,\n            feature_set_name=schema_name,\n            seed=seed,\n        )\n        components[candidate.name] = output\n\n    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}\n    blended = combine_weighted_oof(schema_name, components, weights)\n    strict, _ = strict_forward_bias_predictions(blended, blended["prediction"])\n    blended["prediction"] = strict\n    return blended\n\n\ndef apply_candidate(\n    name: str,\n    features: FeatureSet,\n    empirical_cap: bool,\n    schema_prediction: pd.DataFrame,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    blended = schema_prediction.copy()\n    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}\n    params: dict[str, float] = dict(weights)\n    if empirical_cap:\n        params[OBSERVED_SEAT_CEILING_PARAM] = 1.0\n    spec = EnsembleSpec(\n        name=name,\n        kind="weighted",\n        components=tuple(weights),\n        params=params,\n    )\n    blended["prediction"] = postprocess_ensemble_prediction(\n        spec, blended["prediction"].to_numpy(dtype=float), blended\n    )\n    blended["candidate"] = name\n    blended["empirical_cap"] = empirical_cap\n    blended["feature_count"] = len(features.columns)\n    daily = pd.DataFrame(\n        [\n            {\n                "candidate": name,\n                "validation_date": date,\n                "empirical_cap": empirical_cap,\n                "feature_count": len(features.columns),\n                **required_metrics(frame),\n            }\n            for date, frame in blended.groupby("date", sort=True)\n        ]\n    )\n    return blended, daily\n\n\ndef partition_metrics(predictions: pd.DataFrame) -> pd.DataFrame:\n    partitions = {\n        "latest_complete_08_11_12": predictions["date"].isin(LATEST_COMPLETE_DATES),\n        "latest_partial_08_13": predictions["date"].eq(LATEST_PARTIAL_DATE),\n        "latest_all_08_11_13": predictions["date"].isin(\n            (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)\n        ),\n    }\n    rows: list[dict[str, Any]] = []\n    for partition, mask in partitions.items():\n        for candidate, frame in predictions.loc[mask].groupby("candidate", sort=False):\n            rows.append(\n                {\n                    "partition": partition,\n                    "candidate": candidate,\n                    "feature_count": int(frame["feature_count"].iloc[0]),\n                    "empirical_cap": bool(frame["empirical_cap"].iloc[0]),\n                    **required_metrics(frame),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef paired_trip_bootstrap(\n    predictions: pd.DataFrame,\n    candidate: str,\n    *,\n    low_only: bool,\n    repeats: int = 2_000,\n    seed: int = 42,\n) -> dict[str, Any]:\n    selected = predictions.loc[\n        predictions["candidate"].isin([PRIMARY_NAME, candidate])\n    ].copy()\n    if low_only:\n        selected = selected.loc[selected["label_seats"].le(10)].copy()\n    selected["absolute_error"] = (\n        selected["label_seats"] - selected["prediction"]\n    ).abs()\n    per_event = (\n        selected.groupby(["trip_id", "event_id", "candidate"], observed=True)[\n            "absolute_error"\n        ]\n        .mean()\n        .unstack("candidate")\n        .dropna(subset=[PRIMARY_NAME, candidate])\n        .reset_index()\n    )\n    per_event["improvement"] = per_event[PRIMARY_NAME] - per_event[candidate]\n    trip_groups = [\n        group["improvement"].to_numpy(dtype=float)\n        for _, group in per_event.groupby("trip_id", sort=False)\n    ]\n    rng = np.random.default_rng(seed)\n    draws = np.empty(repeats, dtype=float)\n    for index in range(repeats):\n        sampled = rng.integers(0, len(trip_groups), size=len(trip_groups))\n        draws[index] = np.concatenate([trip_groups[item] for item in sampled]).mean()\n    return {\n        "baseline": PRIMARY_NAME,\n        "candidate": candidate,\n        "metric": "low_0_10_mae" if low_only else "event_balanced_mae",\n        "candidate_improvement": float(per_event["improvement"].mean()),\n        "ci_95_lower": float(np.quantile(draws, 0.025)),\n        "ci_95_upper": float(np.quantile(draws, 0.975)),\n        "probability_candidate_better": float((draws > 0).mean()),\n        "trips": int(per_event["trip_id"].nunique()),\n        "events": int(len(per_event)),\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="1000번 경험적 상한 피처·출력 보정과 좌표 피처 제거 ablation"\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path(\n            "data/analysis_cache/route_specific_features/219000013_snapshots.pkl"\n        ),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features/219000013_flows.pkl"),\n    )\n    parser.add_argument(\n        "--augmented-cache",\n        type=Path,\n        default=Path("data/analysis_cache/main_model_augmented_features_latest.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/latest_main_model_overfit_ablation_results"),\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    data = prepare_augmented(\n        args.snapshot_cache, args.flow_cache, args.augmented_cache\n    )\n    data = add_observed_capacity_features(data)\n    folds = rolling_folds(data)\n    candidates = ablation_candidates()\n\n    prediction_frames: list[pd.DataFrame] = []\n    daily_frames: list[pd.DataFrame] = []\n    schema_predictions: dict[tuple[str, ...], pd.DataFrame] = {}\n    for name, (features, empirical_cap) in candidates.items():\n        schema_key = tuple(features.columns)\n        if schema_key not in schema_predictions:\n            schema_predictions[schema_key] = train_schema(\n                features.name,\n                features,\n                folds,\n                seed=args.seed,\n            )\n        prediction, daily = apply_candidate(\n            name,\n            features,\n            empirical_cap,\n            schema_predictions[schema_key],\n        )\n        prediction_frames.append(prediction)\n        daily_frames.append(daily)\n    predictions = pd.concat(prediction_frames, ignore_index=True)\n    daily = pd.concat(daily_frames, ignore_index=True)\n    metrics = partition_metrics(predictions)\n\n    latest_complete = predictions.loc[\n        predictions["date"].isin(LATEST_COMPLETE_DATES)\n    ]\n    bootstrap = pd.DataFrame(\n        [\n            paired_trip_bootstrap(\n                latest_complete,\n                candidate,\n                low_only=low_only,\n                seed=args.seed,\n            )\n            for candidate in candidates\n            if candidate != PRIMARY_NAME\n            for low_only in (False, True)\n        ]\n    )\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")\n    daily.to_csv(args.output_dir / "daily_metrics.csv", index=False)\n    metrics.to_csv(args.output_dir / "metrics_by_partition.csv", index=False)\n    bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)\n    summary = {\n        "protocol": {\n            "rolling_folds": [fold[0] for fold in folds],\n            "latest_complete_dates": LATEST_COMPLETE_DATES,\n            "latest_partial_date": LATEST_PARTIAL_DATE,\n            "training_rule": "each fold uses only earlier listed weekdays",\n            "feature_history_rule": "same-route strictly earlier calendar dates",\n            "model_rule": "fixed v1.0.0 HGB/ExtraTrees/LightGBM parameters and 0.4/0.4/0.2 weights",\n        },\n        "removed_feature_blocks": {\n            "observed_ceiling": OBSERVED_CEILING_FEATURES,\n            "coordinates": COORDINATE_FEATURES,\n        },\n        "candidates": {\n            name: {\n                "feature_count": len(features.columns),\n                "empirical_cap": empirical_cap,\n                "numeric": features.numeric,\n                "categorical": features.categorical,\n            }\n            for name, (features, empirical_cap) in candidates.items()\n        },\n        "metrics": metrics.to_dict(orient="records"),\n        "bootstrap": bootstrap.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(metrics.to_string(index=False))\n    print("\\npaired bootstrap, latest complete")\n    print(bootstrap.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/pooled_vs_route_specific_experiment.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nfrom lightgbm_feature_experiment import run_rolling_origin\nfrom linear_feature_experiment import (\n    add_preceding_bus_segment_features,\n    add_strict_prior_flow_features,\n    add_strict_prior_low_rate_features,\n    json_ready,\n    metric_row,\n)\nfrom route_specific_feature_experiment import (\n    DEFAULT_ROUTES,\n    DEFAULT_SOURCE_CUTOFF,\n    VALIDATION_DATES,\n    feature_sets,\n    load_or_build_route_cache,\n)\n\n\ndef comparison_feature_sets() -> dict[str, tuple[str, ...]]:\n    current = feature_sets()["all_features"]\n    return {\n        "pooled_shared": current,\n        "pooled_route_aware": (*current, "route_code"),\n    }\n\n\ndef prepare_featured_routes(\n    database: Path,\n    cache_dir: Path,\n    routes: list[str],\n    *,\n    source_cutoff: str,\n) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:\n    frames: dict[str, pd.DataFrame] = {}\n    metadata: dict[str, Any] = {}\n    for route_id in routes:\n        route_name = DEFAULT_ROUTES[route_id]\n        print(f"[{route_name}] strict-prior 피처 준비", flush=True)\n        snapshots, flows, route_metadata = load_or_build_route_cache(\n            database,\n            cache_dir,\n            route_id,\n            source_cutoff=source_cutoff,\n            rebuild=False,\n        )\n        featured = add_strict_prior_low_rate_features(snapshots)\n        featured = add_strict_prior_flow_features(featured, flows)\n        featured, preceding_audit = add_preceding_bus_segment_features(\n            featured, flows\n        )\n        featured["event_id"] = (\n            route_id + "::" + featured["event_id"].astype(str)\n        )\n        featured["trip_id"] = route_id + "::" + featured["trip_id"].astype(str)\n        featured["route_id"] = route_id\n        featured["route_name"] = route_name\n        frames[route_id] = featured\n        metadata[route_id] = {\n            **route_metadata,\n            "preceding_bus_segment_audit": preceding_audit,\n        }\n    return frames, metadata\n\n\ndef _attach_route(predictions: pd.DataFrame) -> pd.DataFrame:\n    result = predictions.copy()\n    result["route_id"] = result["event_id"].str.split("::", n=1).str[0]\n    result["route_name"] = result["route_id"].map(DEFAULT_ROUTES)\n    return result\n\n\ndef metrics_by_route(predictions: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for (route_id, route_name, candidate), frame in predictions.groupby(\n        ["route_id", "route_name", "candidate"], sort=False\n    ):\n        rows.append(\n            {\n                "route_id": route_id,\n                "route_name": route_name,\n                "candidate": candidate,\n                **metric_row(frame),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef metrics_overall(predictions: pd.DataFrame) -> pd.DataFrame:\n    return pd.DataFrame(\n        [\n            {"candidate": candidate, **metric_row(frame)}\n            for candidate, frame in predictions.groupby("candidate", sort=False)\n        ]\n    )\n\n\ndef metrics_by_day(predictions: pd.DataFrame) -> pd.DataFrame:\n    return pd.DataFrame(\n        [\n            {\n                "candidate": candidate,\n                "validation_date": validation_date,\n                **metric_row(frame),\n            }\n            for (candidate, validation_date), frame in predictions.groupby(\n                ["candidate", "date"], sort=False\n            )\n        ]\n    )\n\n\ndef comparison_rows(metrics: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for (route_id, route_name), route_metrics in metrics.groupby(\n        ["route_id", "route_name"], sort=False\n    ):\n        indexed = route_metrics.set_index("candidate")\n        separate = indexed.loc["route_specific"]\n        for pooled_name in ("pooled_shared", "pooled_route_aware"):\n            pooled = indexed.loc[pooled_name]\n            rows.append(\n                {\n                    "route_id": route_id,\n                    "route_name": route_name,\n                    "pooled_candidate": pooled_name,\n                    "events": int(separate["events"]),\n                    "full_events": int(separate["full_events"]),\n                    "low_0_10_events": int(separate["low_0_10_events"]),\n                    "route_specific_mae": float(separate["event_balanced_mae"]),\n                    "pooled_mae": float(pooled["event_balanced_mae"]),\n                    "route_specific_mae_improvement": float(\n                        pooled["event_balanced_mae"]\n                        - separate["event_balanced_mae"]\n                    ),\n                    "route_specific_low_0_10_mae": float(\n                        separate["low_0_10_mae"]\n                    ),\n                    "pooled_low_0_10_mae": float(pooled["low_0_10_mae"]),\n                    "route_specific_low_0_10_mae_improvement": float(\n                        pooled["low_0_10_mae"] - separate["low_0_10_mae"]\n                    ),\n                    "route_specific_full_accuracy": float(\n                        separate["full_accuracy"]\n                    ),\n                    "pooled_full_accuracy": float(pooled["full_accuracy"]),\n                    "route_specific_full_recall": float(separate["full_recall"]),\n                    "pooled_full_recall": float(pooled["full_recall"]),\n                    "route_specific_full_precision": float(\n                        separate["full_precision"]\n                    ),\n                    "pooled_full_precision": float(pooled["full_precision"]),\n                    "route_specific_full_f1": float(separate["full_f1"]),\n                    "pooled_full_f1": float(pooled["full_f1"]),\n                    "route_specific_full_f1_improvement": float(\n                        separate["full_f1"] - pooled["full_f1"]\n                    ),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef paired_trip_bootstrap(\n    predictions: pd.DataFrame,\n    *,\n    pooled_candidate: str,\n    low_only: bool,\n    iterations: int = 2_000,\n    seed: int = 42,\n) -> pd.DataFrame:\n    selected = predictions.loc[\n        predictions["candidate"].isin(["route_specific", pooled_candidate])\n    ].copy()\n    selected["absolute_error"] = (\n        selected["label_seats"] - selected["prediction"]\n    ).abs()\n    if low_only:\n        selected = selected.loc[selected["label_seats"].le(10)].copy()\n\n    event_errors = (\n        selected.groupby(\n            ["route_id", "route_name", "trip_id", "event_id", "candidate"],\n            observed=True,\n        )["absolute_error"]\n        .mean()\n        .unstack("candidate")\n        .dropna(subset=["route_specific", pooled_candidate])\n        .reset_index()\n    )\n    rng = np.random.default_rng(seed)\n    rows: list[dict[str, Any]] = []\n    for (route_id, route_name), route_errors in event_errors.groupby(\n        ["route_id", "route_name"], sort=False\n    ):\n        route_errors = route_errors.copy()\n        route_errors["improvement"] = (\n            route_errors[pooled_candidate] - route_errors["route_specific"]\n        )\n        trip_values = [\n            group["improvement"].to_numpy(float)\n            for _, group in route_errors.groupby("trip_id", sort=False)\n        ]\n        draws = np.empty(iterations, dtype=float)\n        for index in range(iterations):\n            sampled = rng.integers(0, len(trip_values), size=len(trip_values))\n            draws[index] = np.concatenate(\n                [trip_values[item] for item in sampled]\n            ).mean()\n        rows.append(\n            {\n                "route_id": route_id,\n                "route_name": route_name,\n                "pooled_candidate": pooled_candidate,\n                "metric": "low_0_10_mae" if low_only else "event_balanced_mae",\n                "route_specific_improvement": float(\n                    route_errors["improvement"].mean()\n                ),\n                "ci_95_lower": float(np.quantile(draws, 0.025)),\n                "ci_95_upper": float(np.quantile(draws, 0.975)),\n                "trips": int(route_errors["trip_id"].nunique()),\n                "events": int(len(route_errors)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="동일 피처의 통합 LightGBM과 노선별 독립 LightGBM 비교"\n    )\n    parser.add_argument(\n        "--database", type=Path, default=Path("data/gbis_api_cache.sqlite3")\n    )\n    parser.add_argument("--routes", nargs="*", default=list(DEFAULT_ROUTES))\n    parser.add_argument("--source-cutoff", default=DEFAULT_SOURCE_CUTOFF)\n    parser.add_argument(\n        "--cache-dir",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/pooled_vs_route_specific_results"),\n    )\n    args = parser.parse_args()\n\n    frames, route_metadata = prepare_featured_routes(\n        args.database,\n        args.cache_dir,\n        args.routes,\n        source_cutoff=args.source_cutoff,\n    )\n    all_dates = tuple(\n        sorted({date for frame in frames.values() for date in frame["date"].unique()})\n    )\n    all_feature_columns = feature_sets()["all_features"]\n\n    route_predictions: list[pd.DataFrame] = []\n    for route_id, frame in frames.items():\n        print(f"[{DEFAULT_ROUTES[route_id]}] 독립 모델 재학습", flush=True)\n        result = run_rolling_origin(\n            frame,\n            feature_sets={"route_specific": all_feature_columns},\n            development_dates=all_dates,\n            validation_dates=VALIDATION_DATES,\n            return_predictions=True,\n        )\n        route_predictions.append(\n            result[3].loc[result[3]["candidate"].eq("route_specific")].copy()\n        )\n\n    print("[pooled] 통합 모델 학습", flush=True)\n    pooled = pd.concat(frames.values(), ignore_index=True)\n    pooled["route_code"] = pd.Categorical(\n        pooled["route_id"], categories=args.routes\n    )\n    pooled_result = run_rolling_origin(\n        pooled,\n        feature_sets=comparison_feature_sets(),\n        development_dates=all_dates,\n        validation_dates=VALIDATION_DATES,\n        return_predictions=True,\n    )\n    pooled_predictions = pooled_result[3].loc[\n        pooled_result[3]["candidate"].isin(comparison_feature_sets())\n    ].copy()\n    predictions = _attach_route(\n        pd.concat([*route_predictions, pooled_predictions], ignore_index=True)\n    )\n\n    by_route = metrics_by_route(predictions)\n    overall = metrics_overall(predictions)\n    by_day = metrics_by_day(predictions)\n    comparison = comparison_rows(by_route)\n    bootstrap = pd.concat(\n        [\n            paired_trip_bootstrap(\n                predictions,\n                pooled_candidate=pooled_candidate,\n                low_only=low_only,\n            )\n            for pooled_candidate in comparison_feature_sets()\n            for low_only in (False, True)\n        ],\n        ignore_index=True,\n    )\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)\n    overall.to_csv(args.output_dir / "metrics_overall.csv", index=False)\n    by_day.to_csv(args.output_dir / "metrics_by_day.csv", index=False)\n    comparison.to_csv(args.output_dir / "comparison_by_route.csv", index=False)\n    bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": args.source_cutoff,\n            "validation_dates": VALIDATION_DATES,\n            "training_rule": "rolling origin; each validation fold uses only earlier dates",\n            "strict_prior_rule": "historical feature statistics are computed route-locally from strictly earlier calendar dates",\n            "route_specific_features": all_feature_columns,\n            "pooled_feature_sets": comparison_feature_sets(),\n        },\n        "route_metadata": route_metadata,\n        "overall_metrics": overall.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(comparison.to_string(index=False))\n    print("\\n전체 통합 지표")\n    print(overall.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/linear_feature_experiment.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any, Callable\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression, Ridge\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\n\nDEVELOPMENT_DATES = (\n    "2026-08-04",\n    "2026-08-05",\n    "2026-08-06",\n    "2026-08-07",\n    "2026-08-10",\n)\nVALIDATION_DATES = DEVELOPMENT_DATES[1:]\n\nCURRENT_FEATURES = ("snapshot_remaining_seats",)\nTARGET_LOW_RATE_FEATURES = (\n    "target_low_5_rate",\n    "target_low_10_rate",\n    "target_low_rate_log_count",\n)\nPATH_LOW_RATE_FEATURES = (\n    "path_low_5_mean",\n    "path_low_5_median",\n    "path_low_5_sum",\n    "path_low_10_mean",\n    "path_low_10_median",\n    "path_low_10_sum",\n    "path_low_rate_stop_count",\n)\nPATH_FLOW_FEATURES = (\n    "path_flow_mean",\n    "path_flow_median",\n    "path_flow_sum",\n    "path_flow_std",\n    "path_flow_fallback_share",\n)\nPRECEDING_BUS_FEATURES = (\n    "preceding_bus_segment_delta",\n    "preceding_bus_segment_delta_per_stop",\n    "preceding_bus_segment_start_seats",\n    "preceding_bus_target_arrival_seats",\n    "previous_bus_departure_age_minutes",\n    "preceding_bus_segment_missing",\n)\n\nFEATURE_BLOCKS = (\n    ("current_only", CURRENT_FEATURES),\n    ("target_low_rate", TARGET_LOW_RATE_FEATURES),\n    ("path_low_rate", PATH_LOW_RATE_FEATURES),\n    ("path_flow", PATH_FLOW_FEATURES),\n    ("preceding_bus", PRECEDING_BUS_FEATURES),\n)\n\n\ndef event_weights(data: pd.DataFrame) -> np.ndarray:\n    counts = data.groupby("event_id")["event_id"].transform("size").to_numpy(float)\n    weights = 1.0 / counts\n    return weights / weights.mean()\n\n\ndef _binary_rate_lookup(\n    events: pd.DataFrame,\n    outcome: str,\n    *,\n    alpha: float,\n) -> Callable[[int, str, int], tuple[float, float]]:\n    if events.empty:\n        return lambda station, direction, hour: (0.0, 0.0)\n\n    global_rate = float(events[outcome].mean())\n    direction_stats = events.groupby("direction", observed=True)[outcome].agg(\n        ["sum", "count"]\n    )\n    direction_rate = {\n        str(key): (float(row["sum"]) + alpha * global_rate)\n        / (float(row["count"]) + alpha)\n        for key, row in direction_stats.iterrows()\n    }\n\n    station_stats = events.groupby(\n        ["target_station_seq", "direction"], observed=True\n    )[outcome].agg(["sum", "count"])\n    station_rate: dict[tuple[int, str], float] = {}\n    for key, row in station_stats.iterrows():\n        normalized = (int(key[0]), str(key[1]))\n        prior = direction_rate.get(normalized[1], global_rate)\n        station_rate[normalized] = (\n            float(row["sum"]) + alpha * prior\n        ) / (float(row["count"]) + alpha)\n\n    cell_stats = events.groupby(\n        ["target_station_seq", "direction", "arrival_hour"], observed=True\n    )[outcome].agg(["sum", "count"])\n    cell_rate: dict[tuple[int, str, int], tuple[float, float]] = {}\n    for key, row in cell_stats.iterrows():\n        normalized = (int(key[0]), str(key[1]), int(key[2]))\n        prior = station_rate.get(normalized[:2], direction_rate.get(normalized[1], global_rate))\n        cell_rate[normalized] = (\n            (float(row["sum"]) + alpha * prior)\n            / (float(row["count"]) + alpha),\n            float(row["count"]),\n        )\n\n    def lookup(station: int, direction: str, hour: int) -> tuple[float, float]:\n        cell = cell_rate.get((int(station), str(direction), int(hour)))\n        if cell is not None:\n            return cell\n        station_value = station_rate.get(\n            (int(station), str(direction)),\n            direction_rate.get(str(direction), global_rate),\n        )\n        return float(station_value), 0.0\n\n    return lookup\n\n\ndef _event_history(data: pd.DataFrame) -> pd.DataFrame:\n    events = data.drop_duplicates("event_id").copy()\n    events["target_station_seq"] = pd.to_numeric(\n        events["station_seq_cat"], errors="raise"\n    ).astype(int)\n    events["arrival_hour"] = events["event_time"].dt.hour.astype(int)\n    events["is_low_5"] = events["label_seats"].le(5).astype(float)\n    events["is_low_10"] = events["label_seats"].le(10).astype(float)\n    return events\n\n\ndef add_strict_prior_low_rate_features(\n    data: pd.DataFrame,\n    *,\n    alpha: float = 20.0,\n) -> pd.DataFrame:\n    """Attach station/time low-seat rates using strictly earlier dates only."""\n    output = data.copy()\n    columns = (*TARGET_LOW_RATE_FEATURES, *PATH_LOW_RATE_FEATURES)\n    for column in columns:\n        output[column] = np.nan\n    events = _event_history(data)\n\n    for date, indices in output.groupby("date", sort=True).groups.items():\n        history = events.loc[events["date"].lt(str(date))]\n        low_5 = _binary_rate_lookup(history, "is_low_5", alpha=alpha)\n        low_10 = _binary_rate_lookup(history, "is_low_10", alpha=alpha)\n        records: list[dict[str, float]] = []\n        for row in output.loc[indices].itertuples(index=False):\n            target = int(row.station_seq_cat)\n            current = int(row.snapshot_station_seq)\n            hour = int(pd.Timestamp(row.snapshot_time).hour)\n            direction = str(row.direction)\n            target_low_5, target_count = low_5(target, direction, hour)\n            target_low_10, _ = low_10(target, direction, hour)\n            path_stations = list(range(current + 1, target + 1))\n            path_5 = np.asarray(\n                [low_5(station, direction, hour)[0] for station in path_stations],\n                dtype=float,\n            )\n            path_10 = np.asarray(\n                [low_10(station, direction, hour)[0] for station in path_stations],\n                dtype=float,\n            )\n            records.append(\n                {\n                    "target_low_5_rate": target_low_5,\n                    "target_low_10_rate": target_low_10,\n                    "target_low_rate_log_count": np.log1p(target_count),\n                    "path_low_5_mean": float(path_5.mean()) if len(path_5) else 0.0,\n                    "path_low_5_median": float(np.median(path_5)) if len(path_5) else 0.0,\n                    "path_low_5_sum": float(path_5.sum()),\n                    "path_low_10_mean": float(path_10.mean()) if len(path_10) else 0.0,\n                    "path_low_10_median": float(np.median(path_10)) if len(path_10) else 0.0,\n                    "path_low_10_sum": float(path_10.sum()),\n                    "path_low_rate_stop_count": float(len(path_stations)),\n                }\n            )\n        output.loc[indices, list(columns)] = pd.DataFrame(\n            records, index=indices\n        )[list(columns)].to_numpy()\n    return output\n\n\ndef _flow_lookup(\n    flows: pd.DataFrame,\n    *,\n    alpha: float,\n) -> Callable[[int, str, int], tuple[float, bool]]:\n    weekday = flows.loc[pd.to_datetime(flows["date"]).dt.dayofweek.lt(5)]\n    if weekday.empty:\n        return lambda station, direction, time_bin: (0.0, True)\n    global_mean = float(weekday["stop_net"].mean())\n    station_stats = weekday.groupby(\n        ["station_seq", "direction"], observed=True\n    )["stop_net"].agg(["mean", "count"])\n    station_mean = {\n        (int(key[0]), str(key[1])): float(row["mean"])\n        for key, row in station_stats.iterrows()\n    }\n    cell_stats = weekday.groupby(\n        ["station_seq", "direction", "time_bin_2h"], observed=True\n    )["stop_net"].agg(["sum", "count"])\n    cell_mean: dict[tuple[int, str, int], float] = {}\n    for key, row in cell_stats.iterrows():\n        normalized = (int(key[0]), str(key[1]), int(key[2]))\n        prior = station_mean.get(normalized[:2], global_mean)\n        cell_mean[normalized] = (\n            float(row["sum"]) + alpha * prior\n        ) / (float(row["count"]) + alpha)\n\n    def lookup(station: int, direction: str, time_bin: int) -> tuple[float, bool]:\n        key = (int(station), str(direction), int(time_bin))\n        if key in cell_mean:\n            return cell_mean[key], False\n        station_key = key[:2]\n        if station_key in station_mean:\n            return station_mean[station_key], True\n        return global_mean, True\n\n    return lookup\n\n\ndef add_strict_prior_flow_features(\n    data: pd.DataFrame,\n    flows: pd.DataFrame,\n    *,\n    alpha: float = 10.0,\n) -> pd.DataFrame:\n    """Attach path summaries of past station-level net seat changes."""\n    output = data.copy()\n    for column in PATH_FLOW_FEATURES:\n        output[column] = np.nan\n    pass_nodes = {int(value) for value in flows.attrs.get("pass_node_sequences", [])}\n    for date, indices in output.groupby("date", sort=True).groups.items():\n        history = flows.loc[flows["date"].lt(str(date))].copy()\n        lookup = _flow_lookup(history, alpha=alpha)\n        records: list[dict[str, float]] = []\n        for row in output.loc[indices].itertuples(index=False):\n            current = int(row.snapshot_station_seq)\n            target = int(row.station_seq_cat)\n            start = current if str(row.target_state_cat) == "1" else current + 1\n            stations = [station for station in range(start, target) if station not in pass_nodes]\n            time_bin = int(pd.Timestamp(row.snapshot_time).hour // 2 * 2)\n            looked_up = [lookup(station, str(row.direction), time_bin) for station in stations]\n            values = np.asarray([item[0] for item in looked_up], dtype=float)\n            fallback = np.asarray([item[1] for item in looked_up], dtype=float)\n            records.append(\n                {\n                    "path_flow_mean": float(values.mean()) if len(values) else 0.0,\n                    "path_flow_median": float(np.median(values)) if len(values) else 0.0,\n                    "path_flow_sum": float(values.sum()),\n                    "path_flow_std": float(values.std(ddof=0)) if len(values) else 0.0,\n                    "path_flow_fallback_share": float(fallback.mean()) if len(fallback) else 1.0,\n                }\n            )\n        output.loc[indices, list(PATH_FLOW_FEATURES)] = pd.DataFrame(\n            records, index=indices\n        )[list(PATH_FLOW_FEATURES)].to_numpy()\n    return output\n\n\ndef add_preceding_bus_segment_features(\n    data: pd.DataFrame,\n    flows: pd.DataFrame,\n) -> tuple[pd.DataFrame, dict[str, int | float]]:\n    """Join the prior bus\'s directly observed source-departure to target-arrival delta."""\n    output = data.copy()\n    for column in PRECEDING_BUS_FEATURES:\n        if column not in output.columns:\n            output[column] = np.nan\n    output["preceding_bus_segment_missing"] = 1.0\n\n    ordered = flows.sort_values(["departure_seen", "arrival_seen"]).drop_duplicates(\n        ["trip_id", "station_seq"], keep="last"\n    )\n    departure = {\n        (str(row.trip_id), int(row.station_seq)): (\n            float(row.departure_seats),\n            pd.Timestamp(row.departure_seen),\n        )\n        for row in ordered.itertuples(index=False)\n    }\n    arrival = {\n        (str(row.trip_id), int(row.station_seq)): (\n            float(row.observed_arrival_seats),\n            pd.Timestamp(row.arrival_seen),\n        )\n        for row in ordered.itertuples(index=False)\n    }\n\n    matched = 0\n    future_reference = 0\n    invalid_order = 0\n    for index, row in output.iterrows():\n        if pd.isna(row.get("previous_bus_trip_id")):\n            continue\n        trip_id = str(row["previous_bus_trip_id"])\n        current = int(row["snapshot_station_seq"])\n        target = int(row["station_seq_cat"])\n        source_value = departure.get((trip_id, current))\n        target_value = arrival.get((trip_id, target))\n        if source_value is None or target_value is None or target <= current:\n            continue\n        source_seats, source_time = source_value\n        target_seats, target_time = target_value\n        snapshot_time = pd.Timestamp(row["snapshot_time"])\n        if target_time >= snapshot_time:\n            future_reference += 1\n            continue\n        if source_time > target_time:\n            invalid_order += 1\n            continue\n        delta = target_seats - source_seats\n        output.at[index, "preceding_bus_segment_delta"] = delta\n        output.at[index, "preceding_bus_segment_delta_per_stop"] = delta / (\n            target - current\n        )\n        output.at[index, "preceding_bus_segment_start_seats"] = source_seats\n        output.at[index, "preceding_bus_target_arrival_seats"] = target_seats\n        output.at[index, "preceding_bus_segment_missing"] = 0.0\n        matched += 1\n    return output, {\n        "rows": int(len(output)),\n        "matched_rows": matched,\n        "coverage": matched / max(len(output), 1),\n        "future_reference_rows": future_reference,\n        "invalid_order_rows": invalid_order,\n    }\n\n\ndef cumulative_feature_sets() -> dict[str, tuple[str, ...]]:\n    output: dict[str, tuple[str, ...]] = {}\n    selected: list[str] = []\n    for name, columns in FEATURE_BLOCKS:\n        selected.extend(columns)\n        output[name] = tuple(selected)\n    return output\n\n\ndef regression_pipeline() -> Pipeline:\n    return Pipeline(\n        [\n            ("impute", SimpleImputer(strategy="median", add_indicator=True)),\n            ("scale", StandardScaler()),\n            ("model", Ridge(alpha=10.0)),\n        ]\n    )\n\n\ndef classification_pipeline() -> Pipeline:\n    return Pipeline(\n        [\n            ("impute", SimpleImputer(strategy="median", add_indicator=True)),\n            ("scale", StandardScaler()),\n            (\n                "model",\n                LogisticRegression(\n                    C=0.5,\n                    class_weight="balanced",\n                    max_iter=2_000,\n                    random_state=42,\n                ),\n            ),\n        ]\n    )\n\n\ndef _weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:\n    return float(np.average(values, weights=weights))\n\n\ndef metric_row(data: pd.DataFrame) -> dict[str, Any]:\n    weights = event_weights(data)\n    true = data["label_seats"].to_numpy(float)\n    prediction = data["prediction"].to_numpy(float)\n    errors = np.abs(true - prediction)\n    low_10 = true <= 10\n    low_5 = true <= 5\n    full_true = true == 0\n    full_pred = data["full_probability"].to_numpy(float) >= 0.5\n    return {\n        "rows": int(len(data)),\n        "events": int(data["event_id"].nunique()),\n        "full_events": int(data.loc[full_true, "event_id"].nunique()),\n        "low_0_10_events": int(data.loc[low_10, "event_id"].nunique()),\n        "event_balanced_mae": _weighted_mean(errors, weights),\n        "low_0_10_mae": (\n            _weighted_mean(errors[low_10], weights[low_10]) if low_10.any() else np.nan\n        ),\n        "low_0_5_mae": (\n            _weighted_mean(errors[low_5], weights[low_5]) if low_5.any() else np.nan\n        ),\n        "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),\n        "full_recall": float(\n            recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)\n        ),\n        "full_precision": float(\n            precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)\n        ),\n        "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),\n    }\n\n\ndef run_rolling_origin(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    metrics: list[dict[str, Any]] = []\n    predictions: list[pd.DataFrame] = []\n    coefficients: list[dict[str, Any]] = []\n    feature_sets = cumulative_feature_sets()\n    for validation_date in VALIDATION_DATES:\n        train_dates = [date for date in DEVELOPMENT_DATES if date < validation_date]\n        train = data.loc[data["date"].isin(train_dates)].copy()\n        validation = data.loc[data["date"].eq(validation_date)].copy()\n        if train.empty or validation.empty:\n            raise ValueError(f"rolling fold가 비었습니다: {validation_date}")\n        train_weight = event_weights(train)\n        train_delta = (\n            train["label_seats"].to_numpy(float)\n            - train["snapshot_remaining_seats"].to_numpy(float)\n        )\n        train_full = train["label_seats"].eq(0).astype(int).to_numpy()\n        if len(np.unique(train_full)) < 2:\n            raise ValueError(f"만차 분류 학습에 두 클래스가 없습니다: {validation_date}")\n\n        persistence = validation[\n            [\n                "event_id",\n                "trip_id",\n                "date",\n                "snapshot_time",\n                "label_seats",\n                "snapshot_remaining_seats",\n                "capacity",\n                "target_stop_gap",\n            ]\n        ].copy()\n        persistence["candidate"] = "persistence"\n        persistence["prediction"] = persistence["snapshot_remaining_seats"]\n        persistence["full_probability"] = persistence[\n            "snapshot_remaining_seats"\n        ].eq(0).astype(float)\n        metrics.append(\n            {\n                "candidate": "persistence",\n                "validation_date": validation_date,\n                **metric_row(persistence),\n            }\n        )\n        predictions.append(persistence)\n\n        for candidate, columns in feature_sets.items():\n            regressor = regression_pipeline()\n            regressor.fit(\n                train[list(columns)],\n                train_delta,\n                model__sample_weight=train_weight,\n            )\n            raw_delta = regressor.predict(validation[list(columns)])\n            prediction = np.clip(\n                validation["snapshot_remaining_seats"].to_numpy(float) + raw_delta,\n                0,\n                validation["capacity"].to_numpy(float),\n            )\n            classifier = classification_pipeline()\n            classifier.fit(\n                train[list(columns)],\n                train_full,\n                model__sample_weight=train_weight,\n            )\n            full_probability = classifier.predict_proba(validation[list(columns)])[:, 1]\n            scored = validation[\n                [\n                    "event_id",\n                    "trip_id",\n                    "date",\n                    "snapshot_time",\n                    "label_seats",\n                    "snapshot_remaining_seats",\n                    "capacity",\n                    "target_stop_gap",\n                ]\n            ].copy()\n            scored["candidate"] = candidate\n            scored["prediction"] = prediction\n            scored["full_probability"] = full_probability\n            row = metric_row(scored)\n            metrics.append({"candidate": candidate, "validation_date": validation_date, **row})\n            predictions.append(scored)\n\n            transformed_names = regressor.named_steps["impute"].get_feature_names_out(columns)\n            ridge_coefficients = regressor.named_steps["model"].coef_\n            for feature, coefficient in zip(transformed_names, ridge_coefficients):\n                coefficients.append(\n                    {\n                        "candidate": candidate,\n                        "validation_date": validation_date,\n                        "feature": str(feature),\n                        "standardized_coefficient": float(coefficient),\n                    }\n                )\n\n    prediction_table = pd.concat(predictions, ignore_index=True)\n    aggregate: list[dict[str, Any]] = []\n    for candidate, frame in prediction_table.groupby("candidate", sort=False):\n        aggregate.append({"candidate": candidate, **metric_row(frame)})\n    return pd.DataFrame(aggregate), pd.DataFrame(metrics), pd.DataFrame(coefficients)\n\n\ndef json_ready(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): json_ready(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(item) for item in value]\n    if isinstance(value, np.integer):\n        return int(value)\n    if isinstance(value, np.floating):\n        return None if not np.isfinite(value) else float(value)\n    return value\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="사용자 제안 피처의 strict-prior 선형 ablation"\n    )\n    parser.add_argument(\n        "--cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/linear_feature_results"),\n    )\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.cache)\n    flows = pd.read_pickle(args.flow_cache)\n    data = add_strict_prior_low_rate_features(data)\n    data = add_strict_prior_flow_features(data, flows)\n    data, preceding_audit = add_preceding_bus_segment_features(data, flows)\n    development = data.loc[data["date"].isin(DEVELOPMENT_DATES)].copy()\n    metrics, daily_metrics, coefficients = run_rolling_origin(development)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    metrics.to_csv(args.output_dir / "metrics.csv", index=False)\n    daily_metrics.to_csv(args.output_dir / "daily_metrics.csv", index=False)\n    coefficients.to_csv(args.output_dir / "coefficients.csv", index=False)\n    coverage = {\n        column: float(development[column].notna().mean())\n        for _, columns in FEATURE_BLOCKS\n        for column in columns\n    }\n    summary = {\n        "protocol": {\n            "development_dates": DEVELOPMENT_DATES,\n            "validation_dates": VALIDATION_DATES,\n            "historical_feature_rule": "strictly earlier calendar dates only",\n            "regression": "Ridge(alpha=10) on arrival delta, clipped to [0, capacity]",\n            "classification": "balanced LogisticRegression(C=0.5), threshold=0.5",\n            "evaluation_weighting": "equal total weight per arrival event",\n        },\n        "feature_sets": cumulative_feature_sets(),\n        "preceding_bus_segment_audit": preceding_audit,\n        "feature_coverage": coverage,\n        "metrics": metrics.to_dict(orient="records"),\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(metrics.to_string(index=False))\n    print(json.dumps(preceding_audit, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/route_specific_feature_experiment.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\nfrom all_prearrival_seat_regression import build_all_prearrival_table\nfrom hypothesis_model_search import load_analysis_data\nfrom lightgbm_feature_experiment import run_rolling_origin\nfrom linear_feature_experiment import (\n    add_preceding_bus_segment_features,\n    add_strict_prior_flow_features,\n    add_strict_prior_low_rate_features,\n    cumulative_feature_sets,\n    json_ready,\n)\nfrom model_feasibility import build_model_table, build_visits\nfrom tminus_feasibility import prepare_raw_locations\n\n\nDEFAULT_ROUTES = {\n    "219000013": "1000",\n    "222000074": "1100",\n    "219000016": "1200",\n    "218000010": "1500",\n    "222000075": "2000",\n    "200000104": "3000",\n}\nVALIDATION_DATES = ("2026-08-11", "2026-08-12", "2026-08-13")\nDEFAULT_SOURCE_CUTOFF = "2026-08-13 20:26:03+09:00"\n\n\ndef cache_paths(cache_dir: Path, route_id: str) -> tuple[Path, Path, Path]:\n    return (\n        cache_dir / f"{route_id}_snapshots.pkl",\n        cache_dir / f"{route_id}_flows.pkl",\n        cache_dir / f"{route_id}_metadata.json",\n    )\n\n\ndef build_route_snapshots(\n    database: Path,\n    route_id: str,\n    *,\n    source_cutoff: str,\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\n    locations, stations = load_analysis_data(database, route_id)\n    cutoff = pd.Timestamp(source_cutoff)\n    locations = locations.loc[locations["observed_at"].le(cutoff)].copy()\n    if locations.empty:\n        raise ValueError(f"노선 {route_id}에 cutoff 이전 관측이 없습니다.")\n    visits = build_visits(locations, stations)\n    table, turnaround_seq = build_model_table(\n        visits, stations, label_target="arrival"\n    )\n    source = table.loc[\n        table["is_peak"]\n        & table["label_quality"].eq("A")\n        & table["label_seats"].notna()\n    ].copy()\n    if source.empty:\n        raise ValueError(f"노선 {route_id}에 peak A급 도착 라벨이 없습니다.")\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n    snapshots = build_all_prearrival_table(\n        source,\n        visits,\n        by_vehicle,\n        turnaround_seq=turnaround_seq,\n        max_seq=int(stations["station_seq"].max()),\n    )\n\n    flows = visits.loc[\n        (~visits["is_pass_node"])\n        & visits["observed_arrival_seats"].ge(0)\n        & visits["departure_seats"].ge(0)\n        & visits["arrival_seen"].notna()\n        & visits["departure_seen"].notna()\n    ].copy()\n    flows["direction"] = "return"\n    flows.loc[flows["station_seq"].le(turnaround_seq), "direction"] = "to_city"\n    flows["date"] = flows["arrival_seen"].dt.date.astype(str)\n    flows["time_bin_2h"] = (flows["arrival_seen"].dt.hour // 2 * 2).astype(int)\n    flows["stop_net"] = (\n        flows["departure_seats"] - flows["observed_arrival_seats"]\n    ).astype(float)\n    flows = flows[\n        [\n            "vehicle_id",\n            "trip_id",\n            "station_seq",\n            "direction",\n            "date",\n            "arrival_seen",\n            "departure_seen",\n            "time_bin_2h",\n            "observed_arrival_seats",\n            "departure_seats",\n            "stop_net",\n        ]\n    ].sort_values(["arrival_seen", "station_seq", "vehicle_id"])\n    flows.attrs["pass_node_sequences"] = sorted(\n        visits.loc[visits["is_pass_node"], "station_seq"]\n        .dropna()\n        .astype(int)\n        .unique()\n        .tolist()\n    )\n    metadata = {\n        "route_id": route_id,\n        "source_cutoff": source_cutoff,\n        "source_rows": int(len(locations)),\n        "source_observed_at_min": locations["observed_at"].min().isoformat(),\n        "source_observed_at_max": locations["observed_at"].max().isoformat(),\n        "turnaround_seq": int(turnaround_seq),\n        "snapshot_rows": int(len(snapshots)),\n        "events": int(snapshots["event_id"].nunique()),\n        "flow_rows": int(len(flows)),\n        "dates": sorted(snapshots["date"].unique().tolist()),\n    }\n    return snapshots, flows, metadata\n\n\ndef load_or_build_route_cache(\n    database: Path,\n    cache_dir: Path,\n    route_id: str,\n    *,\n    source_cutoff: str,\n    rebuild: bool,\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\n    snapshot_path, flow_path, metadata_path = cache_paths(cache_dir, route_id)\n    if not rebuild and snapshot_path.is_file() and flow_path.is_file() and metadata_path.is_file():\n        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if metadata.get("source_cutoff") == source_cutoff:\n            return pd.read_pickle(snapshot_path), pd.read_pickle(flow_path), {\n                **metadata,\n                "cache_hit": True,\n            }\n\n    snapshots, flows, metadata = build_route_snapshots(\n        database, route_id, source_cutoff=source_cutoff\n    )\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    snapshots.to_pickle(snapshot_path)\n    flows.to_pickle(flow_path)\n    metadata_path.write_text(\n        json.dumps(json_ready(metadata), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    return snapshots, flows, {**metadata, "cache_hit": False}\n\n\ndef feature_sets() -> dict[str, tuple[str, ...]]:\n    cumulative = cumulative_feature_sets()\n    return {\n        "current_only": cumulative["current_only"],\n        "path_features": cumulative["path_flow"],\n        "all_features": cumulative["preceding_bus"],\n    }\n\n\ndef comparison_rows(\n    route_id: str,\n    route_name: str,\n    metrics: pd.DataFrame,\n    metadata: dict[str, Any],\n    preceding_audit: dict[str, Any],\n) -> list[dict[str, Any]]:\n    indexed = metrics.set_index("candidate")\n    baseline = indexed.loc["current_only"]\n    rows: list[dict[str, Any]] = []\n    for candidate in ("path_features", "all_features"):\n        candidate_row = indexed.loc[candidate]\n        rows.append(\n            {\n                "route_id": route_id,\n                "route_name": route_name,\n                "candidate": candidate,\n                "snapshot_rows": metadata["snapshot_rows"],\n                "events": metadata["events"],\n                "validation_events": int(candidate_row["events"]),\n                "full_events": int(candidate_row["full_events"]),\n                "low_0_10_events": int(candidate_row["low_0_10_events"]),\n                "preceding_bus_coverage": float(preceding_audit["coverage"]),\n                "baseline_mae": float(baseline["event_balanced_mae"]),\n                "candidate_mae": float(candidate_row["event_balanced_mae"]),\n                "mae_improvement": float(\n                    baseline["event_balanced_mae"]\n                    - candidate_row["event_balanced_mae"]\n                ),\n                "baseline_low_0_10_mae": float(baseline["low_0_10_mae"]),\n                "candidate_low_0_10_mae": float(candidate_row["low_0_10_mae"]),\n                "low_0_10_mae_improvement": float(\n                    baseline["low_0_10_mae"] - candidate_row["low_0_10_mae"]\n                ),\n                "baseline_full_accuracy": float(baseline["full_accuracy"]),\n                "candidate_full_accuracy": float(candidate_row["full_accuracy"]),\n                "baseline_full_recall": float(baseline["full_recall"]),\n                "candidate_full_recall": float(candidate_row["full_recall"]),\n                "baseline_full_precision": float(baseline["full_precision"]),\n                "candidate_full_precision": float(candidate_row["full_precision"]),\n                "baseline_full_f1": float(baseline["full_f1"]),\n                "candidate_full_f1": float(candidate_row["full_f1"]),\n                "full_f1_improvement": float(\n                    candidate_row["full_f1"] - baseline["full_f1"]\n                ),\n            }\n        )\n    return rows\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="사용자 제안 피처의 노선별 독립 LightGBM 실험"\n    )\n    parser.add_argument(\n        "--database",\n        type=Path,\n        default=Path("data/gbis_api_cache.sqlite3"),\n    )\n    parser.add_argument(\n        "--routes",\n        nargs="*",\n        default=list(DEFAULT_ROUTES),\n    )\n    parser.add_argument("--source-cutoff", default=DEFAULT_SOURCE_CUTOFF)\n    parser.add_argument(\n        "--cache-dir",\n        type=Path,\n        default=Path("data/analysis_cache/route_specific_features"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/route_specific_feature_results"),\n    )\n    parser.add_argument("--rebuild", action="store_true")\n    args = parser.parse_args()\n\n    unknown = sorted(set(args.routes) - set(DEFAULT_ROUTES))\n    if unknown:\n        raise ValueError(f"노선 이름 매핑이 없는 route_id입니다: {unknown}")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    all_metrics: list[pd.DataFrame] = []\n    all_daily: list[pd.DataFrame] = []\n    all_comparisons: list[dict[str, Any]] = []\n    route_summaries: dict[str, Any] = {}\n\n    for route_id in args.routes:\n        route_name = DEFAULT_ROUTES[route_id]\n        print(f"[{route_name}] snapshot/flow 준비", flush=True)\n        snapshots, flows, metadata = load_or_build_route_cache(\n            args.database,\n            args.cache_dir,\n            route_id,\n            source_cutoff=args.source_cutoff,\n            rebuild=args.rebuild,\n        )\n        missing_validation = sorted(set(VALIDATION_DATES) - set(snapshots["date"]))\n        if missing_validation:\n            raise ValueError(\n                f"노선 {route_name}에 검증일이 없습니다: {missing_validation}"\n            )\n        print(f"[{route_name}] strict-prior 피처 생성", flush=True)\n        featured = add_strict_prior_low_rate_features(snapshots)\n        featured = add_strict_prior_flow_features(featured, flows)\n        featured, preceding_audit = add_preceding_bus_segment_features(\n            featured, flows\n        )\n        development_dates = tuple(\n            date\n            for date in sorted(featured["date"].unique())\n            if date <= max(VALIDATION_DATES)\n        )\n        print(f"[{route_name}] rolling-origin LightGBM", flush=True)\n        metrics, daily, _ = run_rolling_origin(\n            featured,\n            feature_sets=feature_sets(),\n            development_dates=development_dates,\n            validation_dates=VALIDATION_DATES,\n        )\n        metrics.insert(0, "route_name", route_name)\n        metrics.insert(0, "route_id", route_id)\n        daily.insert(0, "route_name", route_name)\n        daily.insert(0, "route_id", route_id)\n        all_metrics.append(metrics)\n        all_daily.append(daily)\n        all_comparisons.extend(\n            comparison_rows(\n                route_id,\n                route_name,\n                metrics,\n                metadata,\n                preceding_audit,\n            )\n        )\n        route_summaries[route_id] = {\n            "route_name": route_name,\n            "cache": metadata,\n            "development_dates": development_dates,\n            "validation_dates": VALIDATION_DATES,\n            "preceding_bus_segment_audit": preceding_audit,\n            "metrics": metrics.to_dict(orient="records"),\n        }\n\n    metrics_table = pd.concat(all_metrics, ignore_index=True)\n    daily_table = pd.concat(all_daily, ignore_index=True)\n    comparison_table = pd.DataFrame(all_comparisons)\n    metrics_table.to_csv(args.output_dir / "metrics_by_route.csv", index=False)\n    daily_table.to_csv(args.output_dir / "daily_metrics_by_route.csv", index=False)\n    comparison_table.to_csv(args.output_dir / "improvement_by_route.csv", index=False)\n    summary = {\n        "protocol": {\n            "source_cutoff": args.source_cutoff,\n            "validation_dates": VALIDATION_DATES,\n            "training_rule": "each route separately; all route-local dates strictly before each validation date",\n            "feature_history_rule": "strictly earlier calendar dates within the same route",\n            "feature_sets": feature_sets(),\n        },\n        "routes": route_summaries,\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(comparison_table.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/strict_meta_stack_experiment.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingRegressor\n\nfrom hypothesis_model_search import (\n    Candidate,\n    EnsembleSpec,\n    combine_weighted_oof,\n    development_folds,\n    event_summary,\n    json_ready,\n    postprocess_ensemble_prediction,\n    run_candidate,\n    score_ensemble,\n    tree_node_count,\n)\nfrom seat_service_model import SeatServiceModel\n\n\nDEVELOPMENT_DATES = (\n    "2026-08-04",\n    "2026-08-05",\n    "2026-08-06",\n    "2026-08-07",\n    "2026-08-10",\n)\nVALIDATION_DATES = DEVELOPMENT_DATES[1:]\nCOMPONENTS = (\n    "hgb_capacity_44_70",\n    "extra_trees_compact_48",\n    "lightgbm_sqrt_gap_l1",\n)\nFIXED_WEIGHTS = dict(zip(COMPONENTS, (0.4, 0.4, 0.2), strict=True))\nKEYS = ["event_id", "snapshot_time"]\n\n\n@dataclass(frozen=True)\nclass MetaVariant:\n    name: str\n    why: str\n    if_works: str\n    if_fails: str\n    feature_kind: str\n    params: dict[str, Any]\n\n\nMETA_VARIANTS = (\n    MetaVariant(\n        name="meta_hgb_overfit_full_residual",\n        why=(\n            "The fixed blend may underfit nonlinear regions in component disagreement; "\n            "a deliberately flexible residual learner measures the learnable ceiling."\n        ),\n        if_works=(\n            "Training error and strict-forward validation error both fall, especially "\n            "where the three components disagree."\n        ),\n        if_fails=(\n            "Disagreement identifies uncertainty but not a stable correction direction, "\n            "or the mapping changes as component training windows grow."\n        ),\n        feature_kind="full",\n        params={\n            "loss": "absolute_error",\n            "learning_rate": 0.05,\n            "max_iter": 300,\n            "max_leaf_nodes": 31,\n            "min_samples_leaf": 10,\n            "l2_regularization": 0.0,\n            "early_stopping": False,\n        },\n    ),\n    MetaVariant(\n        name="meta_hgb_regularized_full_residual",\n        why=(\n            "If the flexible learner finds real but noisy interactions, fewer leaves, "\n            "larger leaves, and L2 should retain their stable part."\n        ),\n        if_works=(\n            "It gives up in-sample gain but improves daily stability and service score "\n            "relative to both the flexible learner and the fixed blend."\n        ),\n        if_fails=(\n            "The useful relationship is too weak/nonstationary, or strong regularization "\n            "removes the rare low-seat correction."\n        ),\n        feature_kind="full",\n        params={\n            "loss": "absolute_error",\n            "learning_rate": 0.04,\n            "max_iter": 180,\n            "max_leaf_nodes": 7,\n            "min_samples_leaf": 100,\n            "l2_regularization": 5.0,\n            "early_stopping": False,\n        },\n    ),\n    MetaVariant(\n        name="meta_hgb_regularized_disagreement_residual",\n        why=(\n            "A smaller feature set tests the stated hypothesis directly and prevents "\n            "the meta learner from becoming a second unconstrained seat model."\n        ),\n        if_works=(\n            "Centered component differences plus gap/load context generalize while the "\n            "absolute-prediction meta learner overfits."\n        ),\n        if_fails=(\n            "The magnitude/order of disagreement contains uncertainty information only, "\n            "not enough information to choose a correction sign."\n        ),\n        feature_kind="disagreement",\n        params={\n            "loss": "absolute_error",\n            "learning_rate": 0.03,\n            "max_iter": 150,\n            "max_leaf_nodes": 7,\n            "min_samples_leaf": 200,\n            "l2_regularization": 10.0,\n            "early_stopping": False,\n        },\n    ),\n)\n\n\ndef sha256(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef fast_trip_bootstrap_mae_delta(\n    data: pd.DataFrame,\n    candidate_prediction: np.ndarray,\n    baseline_prediction: np.ndarray,\n    *,\n    seed: int,\n    repeats: int,\n    confidence: float = 0.95,\n) -> dict[str, Any]:\n    """Vectorized equivalent of the project\'s trip-cluster MAE bootstrap."""\n    if not 0.0 < confidence < 1.0:\n        raise ValueError("confidence must be between zero and one")\n    scored = data[["event_id", "trip_id", "label_seats"]].copy()\n    label = data["label_seats"].to_numpy(dtype=float)\n    scored["candidate_error"] = np.abs(label - candidate_prediction)\n    scored["baseline_error"] = np.abs(label - baseline_prediction)\n    per_event = scored.groupby("event_id", sort=False).agg(\n        trip_id=("trip_id", "first"),\n        candidate=("candidate_error", "mean"),\n        baseline=("baseline_error", "mean"),\n    )\n    per_event["delta"] = per_event["baseline"] - per_event["candidate"]\n    per_trip = per_event.groupby("trip_id", sort=False)["delta"].agg(\n        ["sum", "count"]\n    )\n    trip_sums = per_trip["sum"].to_numpy(dtype=float)\n    trip_counts = per_trip["count"].to_numpy(dtype=float)\n    number_of_trips = len(per_trip)\n    if number_of_trips == 0:\n        raise ValueError("no trips to bootstrap")\n    rng = np.random.default_rng(seed)\n    boot = np.empty(repeats, dtype=float)\n    batch_size = 1_000\n    for start in range(0, repeats, batch_size):\n        stop = min(start + batch_size, repeats)\n        indices = rng.integers(\n            0, number_of_trips, size=(stop - start, number_of_trips)\n        )\n        boot[start:stop] = (\n            trip_sums[indices].sum(axis=1) / trip_counts[indices].sum(axis=1)\n        )\n    tail = (1.0 - confidence) / 2.0\n    lower, median, upper = np.quantile(boot, [tail, 0.5, 1.0 - tail])\n    return {\n        "metric": "baseline MAE minus candidate MAE; positive favors candidate",\n        "unit": "trip_id",\n        "trips": int(number_of_trips),\n        "events": int(len(per_event)),\n        "repeats": int(repeats),\n        "confidence": float(confidence),\n        "observed_delta": float(per_event["delta"].mean()),\n        "lower": float(lower),\n        "median": float(median),\n        "upper": float(upper),\n        "probability_candidate_better": float((boot > 0).mean()),\n    }\n\n\ndef load_component_candidate(metadata_path: Path) -> Candidate:\n    payload = json.loads(metadata_path.read_text(encoding="utf-8"))\n    candidate = Candidate(**payload["candidate"])\n    if candidate.name != metadata_path.name.removesuffix(".metadata.json"):\n        raise ValueError(f"candidate name mismatch: {metadata_path}")\n    return candidate\n\n\ndef align_prediction(reference: pd.DataFrame, other: pd.DataFrame) -> np.ndarray:\n    reference_index = pd.MultiIndex.from_frame(reference[KEYS])\n    other_indexed = other.set_index(KEYS)\n    if not other_indexed.index.is_unique:\n        raise ValueError("component OOF keys are not unique")\n    aligned = other_indexed["prediction"].reindex(reference_index)\n    if aligned.isna().any():\n        raise ValueError("component OOF alignment produced missing predictions")\n    return aligned.to_numpy(dtype=float)\n\n\ndef build_meta_table(\n    component_oof: dict[str, pd.DataFrame], fixed_oof: pd.DataFrame\n) -> pd.DataFrame:\n    table = fixed_oof.copy()\n    table = table.rename(columns={"prediction": "fixed_prediction"})\n    for component in COMPONENTS:\n        table[f"prediction_{component}"] = align_prediction(\n            table, component_oof[component]\n        )\n    prediction_columns = [f"prediction_{name}" for name in COMPONENTS]\n    component_values = table[prediction_columns].to_numpy(dtype=float)\n    table["component_mean"] = component_values.mean(axis=1)\n    table["component_std"] = component_values.std(axis=1, ddof=0)\n    table["component_range"] = component_values.max(axis=1) - component_values.min(\n        axis=1\n    )\n    for component in COMPONENTS:\n        table[f"centered_{component}"] = (\n            table[f"prediction_{component}"] - table["fixed_prediction"]\n        )\n    table["current_minus_fixed"] = (\n        table["snapshot_remaining_seats"] - table["fixed_prediction"]\n    )\n    safe_capacity = table["capacity"].clip(lower=1.0)\n    table["snapshot_load_ratio"] = table["snapshot_remaining_seats"] / safe_capacity\n    table["fixed_load_ratio"] = table["fixed_prediction"] / safe_capacity\n    return table\n\n\ndef build_target_meta_table(\n    target: pd.DataFrame,\n    component_predictions: dict[str, np.ndarray],\n    fixed_prediction: np.ndarray,\n) -> pd.DataFrame:\n    columns = ["event_id", "date", "trip_id", "snapshot_time"]\n    if "label_seats" in target.columns:\n        columns.append("label_seats")\n    columns.extend(\n        [\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "capacity",\n            "snapshot_low_plate_cat",\n        ]\n    )\n    table = target[columns].copy()\n    table["fixed_prediction"] = np.asarray(fixed_prediction, dtype=float)\n    for component in COMPONENTS:\n        table[f"prediction_{component}"] = np.asarray(\n            component_predictions[component], dtype=float\n        )\n    prediction_columns = [f"prediction_{name}" for name in COMPONENTS]\n    component_values = table[prediction_columns].to_numpy(dtype=float)\n    table["component_mean"] = component_values.mean(axis=1)\n    table["component_std"] = component_values.std(axis=1, ddof=0)\n    table["component_range"] = component_values.max(axis=1) - component_values.min(\n        axis=1\n    )\n    for component in COMPONENTS:\n        table[f"centered_{component}"] = (\n            table[f"prediction_{component}"] - table["fixed_prediction"]\n        )\n    table["current_minus_fixed"] = (\n        table["snapshot_remaining_seats"] - table["fixed_prediction"]\n    )\n    safe_capacity = table["capacity"].clip(lower=1.0)\n    table["snapshot_load_ratio"] = table["snapshot_remaining_seats"] / safe_capacity\n    table["fixed_load_ratio"] = table["fixed_prediction"] / safe_capacity\n    return table\n\n\ndef meta_feature_columns(kind: str) -> list[str]:\n    disagreement = [\n        *(f"centered_{component}" for component in COMPONENTS),\n        "component_std",\n        "component_range",\n        "current_minus_fixed",\n        "target_stop_gap",\n        "snapshot_load_ratio",\n        "fixed_load_ratio",\n    ]\n    if kind == "disagreement":\n        return disagreement\n    if kind == "full":\n        return [\n            *(f"prediction_{component}" for component in COMPONENTS),\n            "fixed_prediction",\n            "component_mean",\n            *disagreement,\n            "snapshot_remaining_seats",\n            "capacity",\n        ]\n    raise ValueError(f"unknown feature kind: {kind}")\n\n\ndef day_event_weights(data: pd.DataFrame) -> np.ndarray:\n    """Equal weight per date, then per event, then per snapshot within event."""\n    rows_per_event = data.groupby("event_id")["event_id"].transform("size")\n    events_per_date = data.groupby("date")["event_id"].transform("nunique")\n    weights = 1.0 / (\n        rows_per_event.to_numpy(dtype=float)\n        * events_per_date.to_numpy(dtype=float)\n    )\n    return weights / weights.mean()\n\n\ndef strict_forward_meta_prediction(\n    table: pd.DataFrame,\n    variant: MetaVariant,\n    fixed_spec: EnsembleSpec,\n    *,\n    seed: int,\n) -> tuple[np.ndarray, list[dict[str, Any]]]:\n    features = meta_feature_columns(variant.feature_kind)\n    prediction = table["fixed_prediction"].to_numpy(dtype=float).copy()\n    fold_rows: list[dict[str, Any]] = []\n    residual = (\n        table["label_seats"].to_numpy(dtype=float)\n        - table["fixed_prediction"].to_numpy(dtype=float)\n    )\n    for fold_number, validation_date in enumerate(VALIDATION_DATES):\n        current = table["date"].eq(validation_date).to_numpy()\n        if not current.any():\n            raise ValueError(f"missing validation date: {validation_date}")\n        history_dates = VALIDATION_DATES[:fold_number]\n        if not history_dates:\n            fold_rows.append(\n                {\n                    "candidate": variant.name,\n                    "validation_date": validation_date,\n                    "history_dates": [],\n                    "history_rows": 0,\n                    "history_events": 0,\n                    "fallback": "fixed_40_40_20_no_prior_meta_labels",\n                    "tree_nodes": 0,\n                    "train_fixed_mae": np.nan,\n                    "train_meta_mae": np.nan,\n                    "mean_absolute_correction": 0.0,\n                }\n            )\n            continue\n        history = table["date"].isin(history_dates).to_numpy()\n        model = HistGradientBoostingRegressor(\n            random_state=seed + fold_number, **variant.params\n        )\n        model.fit(\n            table.loc[history, features],\n            residual[history],\n            sample_weight=day_event_weights(table.loc[history]),\n        )\n        correction = model.predict(table.loc[current, features])\n        prediction[current] = postprocess_ensemble_prediction(\n            fixed_spec,\n            table.loc[current, "fixed_prediction"].to_numpy(dtype=float)\n            + correction,\n            table.loc[current],\n        )\n        train_prediction = postprocess_ensemble_prediction(\n            fixed_spec,\n            table.loc[history, "fixed_prediction"].to_numpy(dtype=float)\n            + model.predict(table.loc[history, features]),\n            table.loc[history],\n        )\n        fold_rows.append(\n            {\n                "candidate": variant.name,\n                "validation_date": validation_date,\n                "history_dates": list(history_dates),\n                "history_rows": int(history.sum()),\n                "history_events": int(table.loc[history, "event_id"].nunique()),\n                "fallback": None,\n                "tree_nodes": tree_node_count(model),\n                "train_fixed_mae": event_summary(\n                    table.loc[history],\n                    table.loc[history, "fixed_prediction"].to_numpy(dtype=float),\n                )["event_balanced_mae"],\n                "train_meta_mae": event_summary(\n                    table.loc[history], train_prediction\n                )["event_balanced_mae"],\n                "mean_absolute_correction": float(np.mean(np.abs(correction))),\n            }\n        )\n    return prediction, fold_rows\n\n\ndef fit_deployment_meta(\n    history: pd.DataFrame, variant: MetaVariant, *, seed: int\n) -> HistGradientBoostingRegressor:\n    features = meta_feature_columns(variant.feature_kind)\n    model = HistGradientBoostingRegressor(random_state=seed, **variant.params)\n    residual = (\n        history["label_seats"].to_numpy(dtype=float)\n        - history["fixed_prediction"].to_numpy(dtype=float)\n    )\n    model.fit(\n        history[features],\n        residual,\n        sample_weight=day_event_weights(history),\n    )\n    return model\n\n\ndef deployment_meta_predict(\n    model: HistGradientBoostingRegressor,\n    target: pd.DataFrame,\n    variant: MetaVariant,\n    fixed_spec: EnsembleSpec,\n) -> tuple[np.ndarray, dict[str, Any]]:\n    features = meta_feature_columns(variant.feature_kind)\n    correction = model.predict(target[features])\n    prediction = postprocess_ensemble_prediction(\n        fixed_spec,\n        target["fixed_prediction"].to_numpy(dtype=float) + correction,\n        target,\n    )\n    return prediction, {\n        "tree_nodes": tree_node_count(model),\n        "mean_absolute_correction": float(np.mean(np.abs(correction))),\n        "correction_quantiles": {\n            str(quantile): float(value)\n            for quantile, value in zip(\n                (0.0, 0.1, 0.5, 0.9, 1.0),\n                np.quantile(correction, [0.0, 0.1, 0.5, 0.9, 1.0]),\n                strict=True,\n            )\n        },\n    }\n\n\ndef metrics_with_daily(\n    table: pd.DataFrame, prediction: np.ndarray, candidate: str\n) -> tuple[dict[str, Any], list[dict[str, Any]]]:\n    aggregate = event_summary(table, prediction)\n    daily: list[dict[str, Any]] = []\n    for validation_date in VALIDATION_DATES:\n        mask = table["date"].eq(validation_date).to_numpy()\n        row = event_summary(table.loc[mask], prediction[mask])\n        row.update({"candidate": candidate, "validation_date": validation_date})\n        daily.append(row)\n    daily_mae = np.asarray(\n        [row["event_balanced_mae"] for row in daily], dtype=float\n    )\n    aggregate.update(\n        {\n            "candidate": candidate,\n            "mean_daily_mae": float(daily_mae.mean()),\n            "std_daily_mae": float(daily_mae.std(ddof=0)),\n            "robust_score": float(daily_mae.mean() + 0.5 * daily_mae.std(ddof=0)),\n            "service_score": float(\n                daily_mae.mean()\n                + 0.5 * daily_mae.std(ddof=0)\n                + 0.05 * aggregate["low_0_5_mae"]\n            ),\n        }\n    )\n    return aggregate, daily\n\n\ndef disagreement_diagnostics(table: pd.DataFrame) -> dict[str, Any]:\n    diagnostic = table.copy()\n    diagnostic["fixed_absolute_error"] = np.abs(\n        diagnostic["label_seats"] - diagnostic["fixed_prediction"]\n    )\n    per_event = diagnostic.groupby("event_id", sort=False).agg(\n        date=("date", "first"),\n        component_std=("component_std", "mean"),\n        component_range=("component_range", "mean"),\n        fixed_absolute_error=("fixed_absolute_error", "mean"),\n    )\n    component_values = diagnostic[\n        [f"prediction_{component}" for component in COMPONENTS]\n    ].to_numpy(dtype=float)\n    lower = component_values.min(axis=1)\n    upper = component_values.max(axis=1)\n    labels = diagnostic["label_seats"].to_numpy(dtype=float)\n    oracle_prediction = np.minimum(np.maximum(labels, lower), upper)\n    fixed_metrics = event_summary(\n        diagnostic, diagnostic["fixed_prediction"].to_numpy(dtype=float)\n    )\n    oracle_metrics = event_summary(diagnostic, oracle_prediction)\n    quantiles = per_event["component_std"].quantile([0.25, 0.5, 0.75]).to_dict()\n    return {\n        "unit": "event",\n        "spread_error_spearman": float(\n            per_event["component_std"].corr(\n                per_event["fixed_absolute_error"], method="spearman"\n            )\n        ),\n        "spread_error_pearson": float(\n            per_event["component_std"].corr(per_event["fixed_absolute_error"])\n        ),\n        "component_std_quantiles": {\n            str(key): float(value) for key, value in quantiles.items()\n        },\n        "fixed_event_balanced_mae": fixed_metrics["event_balanced_mae"],\n        "component_interval_oracle_event_balanced_mae": oracle_metrics[\n            "event_balanced_mae"\n        ],\n        "component_interval_oracle_note": (\n            "diagnostic-only lower bound; it uses the label to choose a point inside "\n            "the component prediction interval"\n        ),\n    }\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="Bounded strict-forward residual meta-stack experiment"\n    )\n    parser.add_argument(\n        "--snapshot-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),\n    )\n    parser.add_argument(\n        "--component-dir",\n        type=Path,\n        default=Path("analysis/model_search_results"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/strict_meta_stack_results"),\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--bootstrap-repeats", type=int, default=10_000)\n    parser.add_argument("--confirmation-date", default="2026-08-11")\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    snapshots = pd.read_pickle(args.snapshot_cache)\n    flows = pd.read_pickle(args.flow_cache)\n    point_summary_path = args.component_dir / "summary.json"\n    point_summary = json.loads(point_summary_path.read_text(encoding="utf-8"))\n    raw_fixed_spec = point_summary.get("selected_ensemble")\n    if not isinstance(raw_fixed_spec, dict):\n        raise ValueError("selected point model is not an ensemble")\n    fixed_spec = EnsembleSpec(\n        name=str(raw_fixed_spec["name"]),\n        kind=str(raw_fixed_spec["kind"]),\n        components=tuple(raw_fixed_spec["components"]),\n        params={\n            str(key): float(value)\n            for key, value in raw_fixed_spec["params"].items()\n        },\n        bias=float(raw_fixed_spec.get("bias", 0.0)),\n    )\n    if fixed_spec.kind != "weighted" or fixed_spec.components != COMPONENTS:\n        raise ValueError("selected point ensemble structure changed")\n    if any(\n        not np.isclose(fixed_spec.params[name], FIXED_WEIGHTS[name])\n        for name in COMPONENTS\n    ):\n        raise ValueError("selected point ensemble weights changed")\n    development = snapshots.loc[snapshots["date"].isin(DEVELOPMENT_DATES)].copy()\n    observed_dates = tuple(\n        date for date in DEVELOPMENT_DATES if date in set(development["date"])\n    )\n    if observed_dates != DEVELOPMENT_DATES:\n        raise ValueError(f"development dates are incomplete: {observed_dates}")\n    folds = development_folds(development, DEVELOPMENT_DATES)\n\n    component_oof: dict[str, pd.DataFrame] = {}\n    component_metrics: dict[str, dict[str, Any]] = {}\n    component_metadata_hashes: dict[str, str] = {}\n    for component in COMPONENTS:\n        metadata_path = args.component_dir / f"{component}.metadata.json"\n        candidate = load_component_candidate(metadata_path)\n        metrics, oof, _ = run_candidate(candidate, folds, flows, seed=args.seed)\n        component_oof[component] = oof\n        component_metrics[component] = metrics\n        component_metadata_hashes[component] = sha256(metadata_path)\n\n    fixed_unscored = combine_weighted_oof(\n        fixed_spec.name, component_oof, FIXED_WEIGHTS\n    )\n    fixed_metrics_from_shared, fixed_oof = score_ensemble(\n        fixed_spec, fixed_unscored\n    )\n    selected_path = args.component_dir / "selected_oof_predictions.csv"\n    selected = pd.read_csv(selected_path, parse_dates=["snapshot_time"])\n    selected_aligned = align_prediction(fixed_oof, selected)\n    reconstruction_max_abs_diff = float(\n        np.max(np.abs(selected_aligned - fixed_oof["prediction"].to_numpy(dtype=float)))\n    )\n    if reconstruction_max_abs_diff > 1e-9:\n        raise ValueError(\n            "fixed-stack reconstruction does not match the preserved selected OOF: "\n            f"max abs diff={reconstruction_max_abs_diff}"\n        )\n\n    table = build_meta_table(component_oof, fixed_oof)\n    predictions: dict[str, np.ndarray] = {\n        fixed_spec.name: table["fixed_prediction"].to_numpy(dtype=float)\n    }\n    metrics_rows: list[dict[str, Any]] = []\n    daily_rows: list[dict[str, Any]] = []\n    fixed_metrics, fixed_daily = metrics_with_daily(\n        table, predictions[fixed_spec.name], fixed_spec.name\n    )\n    metrics_rows.append(fixed_metrics)\n    daily_rows.extend(fixed_daily)\n\n    meta_fold_rows: list[dict[str, Any]] = []\n    bootstraps: dict[str, dict[str, Any]] = {}\n    active_bootstraps: dict[str, dict[str, Any]] = {}\n    simultaneous_bootstraps: dict[str, dict[str, Any]] = {}\n    simultaneous_confidence = 1.0 - 0.05 / len(META_VARIANTS)\n    for variant in META_VARIANTS:\n        prediction, fit_rows = strict_forward_meta_prediction(\n            table, variant, fixed_spec, seed=args.seed + 10_000\n        )\n        predictions[variant.name] = prediction\n        metrics, daily = metrics_with_daily(table, prediction, variant.name)\n        metrics.update(\n            {\n                "why": variant.why,\n                "if_works": variant.if_works,\n                "if_fails": variant.if_fails,\n                "feature_kind": variant.feature_kind,\n                "params": variant.params,\n            }\n        )\n        metrics_rows.append(metrics)\n        daily_rows.extend(daily)\n        meta_fold_rows.extend(fit_rows)\n        bootstraps[variant.name] = fast_trip_bootstrap_mae_delta(\n            table,\n            prediction,\n            predictions[fixed_spec.name],\n            seed=args.seed + 20_000,\n            repeats=args.bootstrap_repeats,\n        )\n        active = table["date"].isin(VALIDATION_DATES[1:]).to_numpy()\n        active_bootstraps[variant.name] = fast_trip_bootstrap_mae_delta(\n            table.loc[active],\n            prediction[active],\n            predictions[fixed_spec.name][active],\n            seed=args.seed + 30_000,\n            repeats=args.bootstrap_repeats,\n        )\n        simultaneous_bootstraps[variant.name] = fast_trip_bootstrap_mae_delta(\n            table,\n            prediction,\n            predictions[fixed_spec.name],\n            seed=args.seed + 20_000,\n            repeats=args.bootstrap_repeats,\n            confidence=simultaneous_confidence,\n        )\n\n    metrics_frame = pd.DataFrame(metrics_rows)\n    fixed_service_score = float(fixed_metrics["service_score"])\n    fixed_mae = float(fixed_metrics["event_balanced_mae"])\n    for row in metrics_rows:\n        row["fixed_minus_candidate_event_balanced_mae"] = (\n            fixed_mae - float(row["event_balanced_mae"])\n        )\n        row["fixed_minus_candidate_service_score"] = (\n            fixed_service_score - float(row["service_score"])\n        )\n    metrics_frame = pd.DataFrame(metrics_rows)\n    meta_names = [variant.name for variant in META_VARIANTS]\n    best_meta_name = str(\n        metrics_frame.loc[metrics_frame["candidate"].isin(meta_names)]\n        .sort_values(["service_score", "robust_score"])\n        .iloc[0]["candidate"]\n    )\n    development_gate = bool(\n        metrics_frame.loc[metrics_frame["candidate"].eq(best_meta_name), "service_score"]\n        .iloc[0]\n        < fixed_service_score\n        and simultaneous_bootstraps[best_meta_name]["lower"] > 0.0\n    )\n\n    confirmation = snapshots.loc[\n        snapshots["date"].eq(args.confirmation_date)\n    ].copy()\n    if confirmation.empty:\n        raise ValueError(f"confirmation date is empty: {args.confirmation_date}")\n    if args.confirmation_date <= max(VALIDATION_DATES):\n        raise ValueError("confirmation date must be after every meta OOF date")\n    service = SeatServiceModel.load(args.component_dir)\n    if service.ensemble is None:\n        raise ValueError("selected service model is not the expected ensemble")\n    if tuple(service.ensemble.components) != COMPONENTS:\n        raise ValueError(\n            "service component order differs from the frozen experiment order"\n        )\n    if any(\n        not np.isclose(service.ensemble.params[name], FIXED_WEIGHTS[name])\n        for name in COMPONENTS\n    ):\n        raise ValueError("service weights differ from frozen 40/40/20 weights")\n    if service.ensemble.name != fixed_spec.name or service.ensemble.params != fixed_spec.params:\n        raise ValueError("service ensemble spec differs from the point summary")\n    confirmation_components = {\n        name: service._predict_component(service.components[name], confirmation)\n        for name in COMPONENTS\n    }\n    confirmation_fixed = service.predict(confirmation)\n    preserved_confirmation_path = args.component_dir / "final_test_predictions.csv"\n    preserved_confirmation = pd.read_csv(\n        preserved_confirmation_path, parse_dates=["snapshot_time"]\n    )\n    preserved_confirmation = preserved_confirmation.loc[\n        preserved_confirmation["date"].eq(args.confirmation_date)\n    ].copy()\n    if preserved_confirmation.empty:\n        raise ValueError("preserved confirmation predictions are empty")\n    confirmation_reference = confirmation[KEYS].copy()\n    confirmation_reference["prediction"] = confirmation_fixed\n    preserved_for_alignment = preserved_confirmation[KEYS].copy()\n    preserved_for_alignment["prediction"] = preserved_confirmation[\n        "selected_prediction"\n    ].to_numpy(dtype=float)\n    preserved_aligned = align_prediction(\n        confirmation_reference, preserved_for_alignment\n    )\n    confirmation_reconstruction_max_abs_diff = float(\n        np.max(np.abs(preserved_aligned - confirmation_fixed))\n    )\n    if confirmation_reconstruction_max_abs_diff > 1e-9:\n        raise ValueError(\n            "service confirmation prediction does not match preserved output: "\n            f"max abs diff={confirmation_reconstruction_max_abs_diff}"\n        )\n    confirmation_table = build_target_meta_table(\n        confirmation, confirmation_components, confirmation_fixed\n    )\n    selected_variant = next(\n        variant for variant in META_VARIANTS if variant.name == best_meta_name\n    )\n    deployment_features = meta_feature_columns(selected_variant.feature_kind)\n    forbidden_inference_features = {\n        "label_seats",\n        "minutes_to_arrival",\n        "actual_minutes_to_arrival",\n    }\n    if forbidden_inference_features.intersection(deployment_features):\n        raise ValueError("deployment meta features contain evaluation-only columns")\n    deployment_model = fit_deployment_meta(\n        table,\n        selected_variant,\n        seed=args.seed + 10_000 + len(VALIDATION_DATES),\n    )\n    confirmation_predictions: dict[str, np.ndarray] = {\n        fixed_spec.name: confirmation_fixed\n    }\n    confirmation_metrics: list[dict[str, Any]] = []\n    fixed_confirmation_metrics = event_summary(\n        confirmation_table, confirmation_fixed\n    )\n    fixed_confirmation_metrics["candidate"] = fixed_spec.name\n    confirmation_metrics.append(fixed_confirmation_metrics)\n    confirmation_prediction, confirmation_fit = deployment_meta_predict(\n        deployment_model, confirmation_table, selected_variant, fixed_spec\n    )\n    confirmation_predictions[best_meta_name] = confirmation_prediction\n    best_confirmation = event_summary(\n        confirmation_table, confirmation_prediction\n    )\n    best_confirmation["candidate"] = best_meta_name\n    best_confirmation["fixed_minus_candidate_event_balanced_mae"] = float(\n        fixed_confirmation_metrics["event_balanced_mae"]\n        - best_confirmation["event_balanced_mae"]\n    )\n    best_confirmation["fixed_minus_candidate_low_0_5_mae"] = float(\n        fixed_confirmation_metrics["low_0_5_mae"]\n        - best_confirmation["low_0_5_mae"]\n    )\n    best_confirmation["fixed_minus_candidate_emerging_low_mae"] = float(\n        fixed_confirmation_metrics["emerging_low_mae"]\n        - best_confirmation["emerging_low_mae"]\n    )\n    best_confirmation["fixed_minus_candidate_far_6_plus_stop_mae"] = float(\n        fixed_confirmation_metrics["far_6_plus_stop_mae"]\n        - best_confirmation["far_6_plus_stop_mae"]\n    )\n    confirmation_metrics.append(best_confirmation)\n    confirmation_bootstrap = fast_trip_bootstrap_mae_delta(\n        confirmation_table,\n        confirmation_prediction,\n        confirmation_fixed,\n        seed=args.seed + 40_000,\n        repeats=args.bootstrap_repeats,\n    )\n    confirmation_overall_gate = bool(\n        best_confirmation["event_balanced_mae"]\n        < fixed_confirmation_metrics["event_balanced_mae"]\n        and confirmation_bootstrap["lower"] > 0.0\n    )\n    protected_metrics_nonworse = bool(\n        best_confirmation["low_0_5_mae"]\n        <= fixed_confirmation_metrics["low_0_5_mae"]\n        and best_confirmation["emerging_low_mae"]\n        <= fixed_confirmation_metrics["emerging_low_mae"]\n        and best_confirmation["event_balanced_mae_p90"]\n        <= fixed_confirmation_metrics["event_balanced_mae_p90"]\n        and best_confirmation["event_balanced_within_3"]\n        >= fixed_confirmation_metrics["event_balanced_within_3"]\n    )\n\n    stress_dates = ("2026-08-08", "2026-08-09")\n    stress = snapshots.loc[snapshots["date"].isin(stress_dates)].copy()\n    if stress.empty:\n        raise ValueError("weekend stress rows are empty")\n    stress_components = {\n        name: service._predict_component(service.components[name], stress)\n        for name in COMPONENTS\n    }\n    stress_fixed = service.predict(stress)\n    stress_table = build_target_meta_table(stress, stress_components, stress_fixed)\n    stress_prediction, stress_fit = deployment_meta_predict(\n        deployment_model, stress_table, selected_variant, fixed_spec\n    )\n    fixed_stress_metrics = event_summary(stress_table, stress_fixed)\n    fixed_stress_metrics["candidate"] = fixed_spec.name\n    meta_stress_metrics = event_summary(stress_table, stress_prediction)\n    meta_stress_metrics["candidate"] = best_meta_name\n    meta_stress_metrics["fixed_minus_candidate_event_balanced_mae"] = float(\n        fixed_stress_metrics["event_balanced_mae"]\n        - meta_stress_metrics["event_balanced_mae"]\n    )\n    preserved_stress = next(\n        row\n        for row in point_summary["weekend_stress_metrics"]\n        if row["model"] == fixed_spec.name\n    )\n    stress_metric_reconstruction_abs_diff = abs(\n        float(preserved_stress["event_balanced_mae"])\n        - float(fixed_stress_metrics["event_balanced_mae"])\n    )\n    if stress_metric_reconstruction_abs_diff > 1e-12:\n        raise ValueError(\n            "service weekend metric does not reproduce the preserved result"\n        )\n    stress_bootstrap = fast_trip_bootstrap_mae_delta(\n        stress_table,\n        stress_prediction,\n        stress_fixed,\n        seed=args.seed + 50_000,\n        repeats=args.bootstrap_repeats,\n    )\n\n    deployment_model_path = args.output_dir / "selected_meta_deployment.joblib"\n    joblib.dump(deployment_model, deployment_model_path)\n    deployment_model_path.chmod(0o644)\n    roundtrip_model = joblib.load(deployment_model_path)\n    roundtrip_prediction, _ = deployment_meta_predict(\n        roundtrip_model, confirmation_table, selected_variant, fixed_spec\n    )\n    deployment_roundtrip_max_abs_diff = float(\n        np.max(np.abs(roundtrip_prediction - confirmation_prediction))\n    )\n    if deployment_roundtrip_max_abs_diff > 1e-12:\n        raise ValueError("deployment meta artifact failed prediction roundtrip")\n    deployment_model_size = deployment_model_path.stat().st_size\n    deployment_model_nodes = tree_node_count(deployment_model)\n    component_artifact_size = int(\n        sum(\n            (args.component_dir / f"{name}.joblib").stat().st_size\n            for name in COMPONENTS\n        )\n    )\n    selected_component_nodes = point_summary["deployment_constraint"][\n        "selected_component_tree_nodes"\n    ]\n    point_model_nodes = int(sum(selected_component_nodes.values()))\n    deployment_model.predict(confirmation_table[deployment_features])\n    timing_repeats = 20\n    timing = np.empty(timing_repeats, dtype=float)\n    for timing_index in range(timing_repeats):\n        started = time.perf_counter()\n        deployment_model.predict(confirmation_table[deployment_features])\n        timing[timing_index] = time.perf_counter() - started\n    additive_inference_ms_per_1000 = float(\n        np.median(timing) * 1_000_000 / len(confirmation_table)\n    )\n    deployment_feasibility = {\n        "selected_meta_candidate": best_meta_name,\n        "feature_columns": deployment_features,\n        "forbidden_feature_intersection": sorted(\n            forbidden_inference_features.intersection(deployment_features)\n        ),\n        "meta_tree_nodes": deployment_model_nodes,\n        "point_model_tree_nodes": point_model_nodes,\n        "combined_tree_nodes": point_model_nodes + deployment_model_nodes,\n        "one_million_node_budget_passed": bool(\n            point_model_nodes + deployment_model_nodes <= 1_000_000\n        ),\n        "meta_artifact_size_bytes": deployment_model_size,\n        "point_component_artifact_size_bytes": component_artifact_size,\n        "combined_artifact_size_bytes": (\n            component_artifact_size + deployment_model_size\n        ),\n        "additive_meta_inference_ms_per_1000_median": (\n            additive_inference_ms_per_1000\n        ),\n        "timing_repeats": timing_repeats,\n        "component_order": list(COMPONENTS),\n        "service_component_order_matches": True,\n        "service_weights_match": True,\n        "artifact_roundtrip_max_abs_prediction_diff": (\n            deployment_roundtrip_max_abs_diff\n        ),\n    }\n    deployment_metadata_path = args.output_dir / "selected_meta_deployment.metadata.json"\n    deployment_metadata = {\n        "candidate": best_meta_name,\n        "variant": {\n            "feature_kind": selected_variant.feature_kind,\n            "params": selected_variant.params,\n        },\n        "feature_columns": deployment_features,\n        "training_rule": (\n            "all strict-forward component OOF rows from 2026-08-05,06,07,10; "\n            "equal-date/event/snapshot weights; residual target"\n        ),\n        "training_rows": int(len(table)),\n        "training_events": int(table["event_id"].nunique()),\n        "source_script_sha256": sha256(Path(__file__)),\n        "snapshot_cache_sha256": sha256(args.snapshot_cache),\n        "component_metadata_sha256": component_metadata_hashes,\n        "point_model_summary_sha256": sha256(point_summary_path),\n        "fixed_point_ensemble": asdict(fixed_spec),\n        "model_artifact_sha256": sha256(deployment_model_path),\n        "feasibility": deployment_feasibility,\n    }\n    deployment_metadata_path.write_text(\n        json.dumps(json_ready(deployment_metadata), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    deployment_metadata_path.chmod(0o644)\n\n    fixed_daily_lookup = {\n        row["validation_date"]: row["event_balanced_mae"]\n        for row in daily_rows\n        if row["candidate"] == fixed_spec.name\n    }\n    best_daily_deltas = {\n        row["validation_date"]: float(\n            fixed_daily_lookup[row["validation_date"]]\n            - row["event_balanced_mae"]\n        )\n        for row in daily_rows\n        if row["candidate"] == best_meta_name\n    }\n    meta_active_daily_deltas = {\n        date: delta\n        for date, delta in best_daily_deltas.items()\n        if date != VALIDATION_DATES[0]\n    }\n\n    output_predictions = table[\n        [\n            "event_id",\n            "date",\n            "trip_id",\n            "snapshot_time",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "capacity",\n            *(f"prediction_{component}" for component in COMPONENTS),\n            "component_std",\n            "component_range",\n        ]\n    ].copy()\n    for candidate, prediction in predictions.items():\n        output_predictions[candidate] = prediction\n\n    metrics_frame.to_csv(args.output_dir / "metrics.csv", index=False)\n    pd.DataFrame(daily_rows).to_csv(\n        args.output_dir / "daily_metrics.csv", index=False\n    )\n    pd.DataFrame(meta_fold_rows).to_csv(\n        args.output_dir / "meta_fold_fit.csv", index=False\n    )\n    output_predictions.to_csv(\n        args.output_dir / "predictions.csv.gz", index=False, compression="gzip"\n    )\n    output_confirmation = confirmation_table[\n        [\n            "event_id",\n            "date",\n            "trip_id",\n            "snapshot_time",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "capacity",\n            *(f"prediction_{component}" for component in COMPONENTS),\n            "component_std",\n            "component_range",\n        ]\n    ].copy()\n    for candidate, prediction in confirmation_predictions.items():\n        output_confirmation[candidate] = prediction\n    output_confirmation.to_csv(\n        args.output_dir / "confirmation_predictions.csv.gz",\n        index=False,\n        compression="gzip",\n    )\n    output_stress = stress_table[\n        [\n            "event_id",\n            "date",\n            "trip_id",\n            "snapshot_time",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "capacity",\n            *(f"prediction_{component}" for component in COMPONENTS),\n            "component_std",\n            "component_range",\n        ]\n    ].copy()\n    output_stress[fixed_spec.name] = stress_fixed\n    output_stress[best_meta_name] = stress_prediction\n    output_stress.to_csv(\n        args.output_dir / "weekend_stress_predictions.csv.gz",\n        index=False,\n        compression="gzip",\n    )\n    summary = {\n        "hypothesis": (\n            "The fixed 40/40/20 blend misses a stable, conditional residual signal "\n            "in disagreement among HGB-capacity, ExtraTrees48, and sqrt-gap LightGBM."\n        ),\n        "protocol": {\n            "validation_dates": list(VALIDATION_DATES),\n            "component_rule": (\n                "Each component OOF for date D is trained only on development dates "\n                "strictly before D using the unchanged base-model protocol."\n            ),\n            "meta_rule": (\n                "For date D, fit the residual meta model only on component OOF rows "\n                "and labels from validation dates strictly before D."\n            ),\n            "first_date_rule": "Fixed 40/40/20 fallback because no prior meta labels exist.",\n            "weighting": "equal date -> equal event -> equal snapshot within event",\n            "target": "label_seats - strict-forward fixed-stack prediction",\n            "actual_minutes_to_arrival_used": False,\n            "variant_count": len(META_VARIANTS),\n            "search": "three predeclared hypothesis variants; no random/grid sweep",\n            "selection_score": (\n                "mean daily event-balanced MAE + 0.5 * daily MAE std + "\n                "0.05 * low-0-5 event-balanced MAE"\n            ),\n            "adoption_gate": (\n                "best meta service score < fixed service score and paired trip "\n                "Bonferroni-simultaneous 98.33% bootstrap MAE-improvement lower > 0"\n            ),\n            "bootstrap_repeats": args.bootstrap_repeats,\n            "bootstrap_seed": args.seed + 20_000,\n            "confirmation_rule": (\n                "Fit the meta model on all four earlier OOF dates, then predict the "\n                "later confirmation date without using its labels."\n            ),\n            "confirmation_date": args.confirmation_date,\n            "confirmation_status": (\n                "partial-day, previously inspected confirmation; not a pristine holdout"\n            ),\n            "weekend_stress_rule": (\n                "Apply the same already-selected deployment meta artifact once to "\n                "2026-08-08/09; no parameter or candidate reselection"\n            ),\n            "weekend_stress_status": (\n                "distribution-shift stress only, not strict-forward: the frozen point "\n                "components and meta training include later 2026-08-10 data"\n            ),\n        },\n        "provenance": {\n            "script_sha256": sha256(Path(__file__)),\n            "snapshot_cache": str(args.snapshot_cache),\n            "snapshot_cache_sha256": sha256(args.snapshot_cache),\n            "flow_cache": str(args.flow_cache),\n            "flow_cache_sha256": sha256(args.flow_cache),\n            "selected_oof": str(selected_path),\n            "selected_oof_sha256": sha256(selected_path),\n            "preserved_confirmation_predictions": str(\n                preserved_confirmation_path\n            ),\n            "preserved_confirmation_predictions_sha256": sha256(\n                preserved_confirmation_path\n            ),\n            "component_metadata_sha256": component_metadata_hashes,\n            "point_model_summary": str(point_summary_path),\n            "point_model_summary_sha256": sha256(point_summary_path),\n            "fixed_point_ensemble": asdict(fixed_spec),\n            "fixed_reconstruction_max_abs_diff": reconstruction_max_abs_diff,\n            "confirmation_fixed_reconstruction_max_abs_diff": (\n                confirmation_reconstruction_max_abs_diff\n            ),\n            "weekend_fixed_metric_reconstruction_abs_diff": (\n                stress_metric_reconstruction_abs_diff\n            ),\n            "deployment_meta_artifact": str(deployment_model_path),\n            "deployment_meta_artifact_sha256": sha256(deployment_model_path),\n            "deployment_meta_metadata": str(deployment_metadata_path),\n            "deployment_meta_metadata_sha256": sha256(deployment_metadata_path),\n        },\n        "component_metrics": component_metrics,\n        "fixed_metrics_from_shared_scorer": fixed_metrics_from_shared,\n        "metrics": metrics_rows,\n        "daily_metrics": daily_rows,\n        "trip_bootstrap_vs_fixed": bootstraps,\n        "multiplicity_adjusted_trip_bootstrap_vs_fixed": {\n            "method": (\n                "Bonferroni simultaneous intervals for three predeclared variants"\n            ),\n            "familywise_confidence": 0.95,\n            "per_interval_confidence": simultaneous_confidence,\n            "candidates": simultaneous_bootstraps,\n        },\n        "trip_bootstrap_vs_fixed_meta_active_dates": active_bootstraps,\n        "confirmation_metrics": confirmation_metrics,\n        "confirmation_fit": confirmation_fit,\n        "confirmation_trip_bootstrap_vs_fixed": confirmation_bootstrap,\n        "weekend_stress_metrics": [fixed_stress_metrics, meta_stress_metrics],\n        "weekend_stress_fit": stress_fit,\n        "weekend_stress_trip_bootstrap_vs_fixed": stress_bootstrap,\n        "deployment_feasibility": deployment_feasibility,\n        "disagreement_diagnostics": disagreement_diagnostics(table),\n        "selection": {\n            "best_meta_candidate": best_meta_name,\n            "development_gate_passed": development_gate,\n            "development_daily_mae_delta_fixed_minus_meta": best_daily_deltas,\n            "meta_active_dates_improved": int(\n                sum(delta > 0.0 for delta in meta_active_daily_deltas.values())\n            ),\n            "meta_active_dates_total": len(meta_active_daily_deltas),\n            "confirmation_overall_mae_gate_passed": confirmation_overall_gate,\n            "confirmation_protected_metrics_nonworse": protected_metrics_nonworse,\n            "service_adoption_recommended": False,\n            "decision": (\n                "keep_fixed_primary_promote_meta_to_pristine_shadow"\n                if development_gate\n                else "reject_meta_keep_fixed_primary"\n            ),\n            "decision_reason": (\n                (\n                    "The disagreement meta learner passes the predeclared development "\n                    "gate, but the non-pristine confirmation worsens low-seat, "\n                    "emerging-low, p90, and within-3 metrics. A future pristine date "\n                    "is required before adding service complexity."\n                )\n                if development_gate\n                else (\n                    "The disagreement meta learner does not pass the predeclared "\n                    "development service-score and simultaneous-bootstrap gate, so "\n                    "it is rejected without prospective shadow evaluation."\n                )\n            ),\n        },\n        "limitations": [\n            "Only four strict-forward validation dates are available.",\n            "The first validation date is necessarily identical to the fixed fallback.",\n            "Earlier OOF meta rows come from components fitted on smaller histories, so "\n            "the meta input distribution changes as the base training window grows.",\n            "No locked/pristine date is used because this is a development experiment.",\n            "The trip bootstrap clusters trips, not dates; with only three meta-active "\n            "development dates it cannot quantify date-level generalization uncertainty.",\n            "Weekend stress uses point/meta fits containing later 2026-08-10 data and "\n            "therefore measures robustness only, not prospective generalization.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(metrics_frame[[\n        "candidate",\n        "event_balanced_mae",\n        "low_0_5_mae",\n        "emerging_low_mae",\n        "far_6_plus_stop_mae",\n        "std_daily_mae",\n        "service_score",\n        "fixed_minus_candidate_event_balanced_mae",\n    ]].to_string(index=False))\n    print(json.dumps(summary["selection"], ensure_ascii=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/plot_spatial_feature_maps.py': 'from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans\n\nfrom gbis_client.cache import GBISApiCache\nfrom route_specific_feature_experiment import DEFAULT_ROUTES\n\n\ndef station_features(cache: GBISApiCache) -> pd.DataFrame:\n    stations = cache.stations_df()\n    history = cache.history_df()\n    stations = stations.loc[stations["route_id"].isin(DEFAULT_ROUTES)].copy()\n    stations = stations[["route_id", "station_id", "x", "y"]].dropna()\n    history = history.loc[\n        history["route_id"].isin(DEFAULT_ROUTES) & history["remaining_seats"].ge(0)\n    ].copy()\n    events = history.merge(stations, on=["route_id", "station_id"], how="inner")\n    events["observed_at"] = pd.to_datetime(events["observed_at"])\n    points = events[["x", "y"]].drop_duplicates().reset_index(drop=True)\n    coords = points[["x", "y"]].to_numpy(float)\n    clusterer = KMeans(n_clusters=12, n_init=20, random_state=42).fit(coords)\n    points["spatial_cluster"] = clusterer.labels_\n    centers = clusterer.cluster_centers_[clusterer.labels_]\n    points["spatial_cluster_distance_km"] = np.hypot(\n        (coords[:, 0] - centers[:, 0]) * 88.8,\n        (coords[:, 1] - centers[:, 1]) * 111.2,\n    )\n    dx = (coords[:, None, 0] - coords[None, :, 0]) * 88.8\n    dy = (coords[:, None, 1] - coords[None, :, 1]) * 111.2\n    nearby = np.hypot(dx, dy) <= 1.0\n    points["spatial_stop_density_1km"] = nearby.sum(axis=1)\n    station_routes = stations.groupby(["x", "y"])["route_id"].nunique()\n    points["spatial_route_density_1km"] = [\n        station_routes.reindex(pd.MultiIndex.from_frame(points.loc[mask, ["x", "y"]])).sum()\n        for mask in nearby\n    ]\n    events = events.merge(points, on=["x", "y"], how="left")\n    events["date"] = events["observed_at"].dt.date.astype(str)\n    events["hour"] = events["observed_at"].dt.hour\n    events["low10"] = events["remaining_seats"].le(10).astype(float)\n    events["spatial_cluster_hour_low10_rate"] = 0.0\n    events["spatial_cluster_hour_low10_log_count"] = 0.0\n    for date in sorted(events["date"].unique()):\n        historical = events.loc[events["date"].lt(date)]\n        target = events["date"].eq(date)\n        global_rate = float(historical["low10"].mean()) if len(historical) else 0.0\n        stats = historical.groupby(["spatial_cluster", "hour"])["low10"].agg(["sum", "count"])\n        keys = pd.MultiIndex.from_arrays([\n            events.loc[target, "spatial_cluster"], events.loc[target, "hour"]\n        ])\n        match = stats.reindex(keys)\n        count = match["count"].fillna(0).to_numpy(float)\n        events.loc[target, "spatial_cluster_hour_low10_rate"] = (\n            match["sum"].fillna(0).to_numpy(float) + 20 * global_rate\n        ) / (count + 20)\n        events.loc[target, "spatial_cluster_hour_low10_log_count"] = np.log1p(count)\n    means = events.groupby(["x", "y"], as_index=False).agg(\n        mean_remaining_seats=("remaining_seats", "mean"),\n        spatial_cluster=("spatial_cluster", "first"),\n        spatial_cluster_distance_km=("spatial_cluster_distance_km", "first"),\n        spatial_stop_density_1km=("spatial_stop_density_1km", "first"),\n        spatial_route_density_1km=("spatial_route_density_1km", "first"),\n        spatial_cluster_hour_low10_rate=("spatial_cluster_hour_low10_rate", "mean"),\n        spatial_cluster_hour_low10_log_count=("spatial_cluster_hour_low10_log_count", "mean"),\n    )\n    return means\n\n\ndef draw(data: pd.DataFrame, output: Path) -> None:\n    columns = [\n        ("mean_remaining_seats", "Mean remaining seats", "RdYlBu"),\n        ("spatial_cluster", "Spatial cluster (12)", "tab20"),\n        ("spatial_cluster_distance_km", "Distance to cluster center (km)", "viridis"),\n        ("spatial_stop_density_1km", "Stops within 1 km", "magma"),\n        ("spatial_route_density_1km", "Routes within 1 km", "magma"),\n        ("spatial_cluster_hour_low10_rate", "Mean cluster-hour low-seat rate", "YlOrRd"),\n        ("spatial_cluster_hour_low10_log_count", "Mean cluster-hour history (log1p)", "Blues"),\n    ]\n    fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=True)\n    for axis, (column, title, cmap) in zip(axes.flat, columns, strict=False):\n        image = axis.scatter(\n            data["x"], data["y"], c=data[column], s=16, cmap=cmap,\n            alpha=0.86, linewidths=0,\n        )\n        axis.set_title(title, fontsize=12, fontweight="bold")\n        axis.set_xlabel("Longitude (x)")\n        axis.set_ylabel("Latitude (y)")\n        fig.colorbar(image, ax=axis, shrink=0.78)\n    axes.flat[-1].axis("off")\n    fig.suptitle("All 6 active routes · station-level spatial feature values", fontsize=16, fontweight="bold")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(output, dpi=180, bbox_inches="tight")\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    with GBISApiCache.from_env() as cache:\n        data = station_features(cache)\n    draw(data, args.output)\n    print(f"points={len(data)} output={args.output}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/plot_low_seat_concentration.py': 'from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\n\nLOW_SEAT_THRESHOLD = 5\n\n\ndef concentration_table(data: pd.DataFrame, group: str) -> pd.DataFrame:\n    """Return P(group | low seats), with one event per displayed group value."""\n\n    frame = data.drop_duplicates(["event_id", group])\n    all_counts = frame[group].value_counts(dropna=False)\n    low_counts = frame.loc[\n        frame["label_seats"].le(LOW_SEAT_THRESHOLD), group\n    ].value_counts(dropna=False)\n    result = pd.DataFrame({"all_count": all_counts, "low_count": low_counts}).fillna(0)\n    result["all_count"] = result["all_count"].astype(int)\n    result["low_count"] = result["low_count"].astype(int)\n    result["all_share_pct"] = result["all_count"] / result["all_count"].sum() * 100\n    result["low_share_pct"] = result["low_count"] / result["low_count"].sum() * 100\n    result["low_rate_pct"] = result["low_count"] / result["all_count"] * 100\n    result["concentration_ratio"] = (\n        result["low_share_pct"] / result["all_share_pct"]\n    )\n    return result.rename_axis("value").reset_index()\n\n\ndef numeric_order(table: pd.DataFrame) -> pd.DataFrame:\n    output = table.copy()\n    output["numeric_value"] = pd.to_numeric(output["value"])\n    return output.sort_values("numeric_value")\n\n\ndef draw_concentration(\n    ax: plt.Axes,\n    table: pd.DataFrame,\n    *,\n    title: str,\n    xlabel: str,\n    tick_step: int | None = None,\n    categorical_labels: dict[str, str] | None = None,\n    time_axis: bool = False,\n) -> None:\n    if categorical_labels is None:\n        table = numeric_order(table)\n        positions = table["numeric_value"].to_numpy(dtype=float)\n        if tick_step is None:\n            raise ValueError("numeric axis에는 tick_step이 필요합니다.")\n        first, last = int(positions.min()), int(positions.max())\n        ticks = np.arange(int(np.ceil(first / tick_step) * tick_step), last + 1, tick_step)\n        labels = [f"{value // 2:02d}:00" if time_axis else str(value) for value in ticks]\n        ax.set_xticks(ticks, labels=labels)\n        ax.set_xlim(first - 1, last + 1)\n        bar_width = 0.75\n    else:\n        order = list(categorical_labels)\n        table = table.set_index(table["value"].astype(str)).reindex(order).reset_index(drop=True)\n        positions = np.arange(len(table), dtype=float)\n        ax.set_xticks(positions, labels=[categorical_labels[value] for value in order])\n        ax.set_xlim(-0.6, len(table) - 0.4)\n        bar_width = 0.55\n\n    ax.bar(\n        positions,\n        table["low_share_pct"],\n        width=bar_width,\n        color="#DD8452",\n        alpha=0.8,\n        label="저잔여 사례 점유율",\n    )\n    ax.plot(\n        positions,\n        table["all_share_pct"],\n        color="#4C72B0",\n        marker="o",\n        markersize=3,\n        linewidth=1.2,\n        label="전체 사례 점유율",\n    )\n\n    annotation_count = len(table) if len(table) <= 3 else 3\n    top = table.nlargest(annotation_count, "low_share_pct")\n    position_lookup = {\n        str(value): position for value, position in zip(table["value"], positions)\n    }\n    for _, row in top.iterrows():\n        x = position_lookup[str(row["value"])]\n        y = float(row["low_share_pct"])\n        ax.annotate(\n            f"{y:.1f}%",\n            (x, y),\n            xytext=(0, 4),\n            textcoords="offset points",\n            ha="center",\n            va="bottom",\n            fontsize=9,\n        )\n\n    ax.set_ylim(0, max(float(table["low_share_pct"].max()) * 1.2, 1))\n    ax.set_title(title)\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("사례 점유율(%)")\n    ax.grid(axis="y", alpha=0.25)\n    ax.legend(fontsize=9)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="저잔여 도착 사건의 피처값 집중도")\n    parser.add_argument(\n        "--cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/feature_distribution_results"),\n    )\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.cache)\n    feature_specs = {\n        "target_station": "station_seq_cat",\n        "direction": "direction",\n        "current_station": "snapshot_station_seq_cat",\n        "snapshot_time": "snapshot_time_bin_30",\n    }\n    required = {"event_id", "date", "label_seats", *feature_specs.values()}\n    missing = sorted(required - set(data.columns))\n    if missing:\n        raise ValueError(f"필수 열이 없습니다: {missing}")\n\n    tables = {\n        name: concentration_table(data, column)\n        for name, column in feature_specs.items()\n    }\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    combined = pd.concat(\n        [table.assign(feature=name) for name, table in tables.items()],\n        ignore_index=True,\n    )\n    combined.to_csv(args.output_dir / "low_seat_concentration.csv", index=False)\n\n    plt.rcParams["font.family"] = "Noto Sans CJK KR"\n    plt.rcParams["axes.unicode_minus"] = False\n    figure, axes = plt.subplots(2, 2, figsize=(18, 11), constrained_layout=True)\n\n    draw_concentration(\n        axes[0, 0],\n        tables["target_station"],\n        title="저잔여 사례의 목표 정류장 분포",\n        xlabel="목표 정류장 순번",\n        tick_step=5,\n    )\n    draw_concentration(\n        axes[0, 1],\n        tables["direction"],\n        title="저잔여 사례의 운행 방향 분포",\n        xlabel="운행 방향",\n        categorical_labels={"to_city": "도시 방향", "return": "회차 후 복귀"},\n    )\n    draw_concentration(\n        axes[1, 0],\n        tables["current_station"],\n        title="저잔여 사례의 현재 정류장 분포",\n        xlabel="현재 정류장 순번",\n        tick_step=5,\n    )\n    draw_concentration(\n        axes[1, 1],\n        tables["snapshot_time"],\n        title="저잔여 사례의 관측 시간대 분포",\n        xlabel="관측 시각",\n        tick_step=2,\n        time_axis=True,\n    )\n\n    first_date = str(data["date"].astype(str).min())\n    last_date = str(data["date"].astype(str).max())\n    low_events = data.loc[data["label_seats"].le(LOW_SEAT_THRESHOLD), "event_id"].nunique()\n    all_events = data["event_id"].nunique()\n    figure.suptitle(\n        f"도착 잔여좌석 0~{LOW_SEAT_THRESHOLD}석 사례의 특성값 집중도 "\n        f"({first_date} ~ {last_date}, {low_events:,}/{all_events:,} 사건)",\n        fontsize=17,\n        fontweight="bold",\n    )\n    figure.text(\n        0.5,\n        0.005,\n        "주황 막대=P(특성값 | 저잔여), 파란선=P(특성값) · 막대가 선보다 높을수록 저잔여 사례가 상대적으로 집중",\n        ha="center",\n        fontsize=10,\n        color="#555555",\n    )\n\n    output_path = args.output_dir / "low_seat_concentration.png"\n    figure.savefig(output_path, dpi=180, bbox_inches="tight")\n    plt.close(figure)\n    print(output_path)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/all_prearrival_seat_regression.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.pipeline import Pipeline\n\nfrom model_feasibility import (\n    FeatureSet,\n    _as_records,\n    build_model_table,\n    build_visits,\n    json_ready,\n    load_data,\n    make_preprocessor,\n    prepare_subset,\n)\nfrom tminus_feasibility import ROUTE_ID, ROUTE_NAME, prepare_raw_locations\n\n\nALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_target_vehicle",\n    numeric=(\n        "snapshot_time_sin",\n        "snapshot_time_cos",\n        "route_progress",\n        "snapshot_route_progress",\n        "x",\n        "y",\n        "snapshot_capacity",\n        "target_load_ratio",\n        "target_stop_gap",\n    ),\n    categorical=(\n        "station_seq_cat",\n        "direction",\n        "snapshot_day_of_week",\n        "snapshot_low_plate_cat",\n        "target_state_cat",\n    ),\n)\n\n\nDYNAMIC_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_dynamic",\n    numeric=(\n        *ALL_PREARRIVAL.numeric,\n        "snapshot_remaining_seats",\n        "seat_delta_previous_stop",\n        "seat_change_per_stop",\n        "rolling_seat_change_per_stop_3",\n        "minutes_since_previous_stop",\n        "rolling_minutes_per_stop_3",\n        "estimated_minutes_to_arrival",\n        "projected_arrival_seats",\n        "seats_per_remaining_stop",\n        "load_gap_interaction",\n        "currently_low_5",\n        "currently_low_10",\n    ),\n    categorical=(\n        *ALL_PREARRIVAL.categorical,\n        "snapshot_station_seq_cat",\n        "snapshot_time_bin_30",\n    ),\n)\n\n\nENGINEERED_ALL_PREARRIVAL = FeatureSet(\n    name="all_prearrival_previous_bus",\n    numeric=(\n        *DYNAMIC_ALL_PREARRIVAL.numeric,\n        "previous_bus_departure_seats",\n        "previous_bus_departure_capacity",\n        "previous_bus_departure_load_ratio",\n        "previous_bus_same_capacity",\n        "previous_bus_departure_age_minutes",\n        "previous_bus_projected_headway_minutes",\n        "previous_bus_freshness_exp",\n        "previous_bus_departure_missing",\n        "previous_bus_headway_minutes",\n        "previous_bus_headway_log1p",\n        "previous_bus_is_bunched",\n        "previous_bus_headway_missing",\n        "previous_3_bus_departure_mean",\n        "previous_3_bus_departure_std",\n        "previous_bus_departure_trend",\n        "previous_3_bus_departure_load_ratio_mean",\n        "previous_3_bus_departure_load_ratio_std",\n        "previous_3_bus_departure_load_ratio_trend",\n        "previous_bus_was_full",\n        "previous_bus_was_low_5",\n        "previous_bus_was_full_normalized",\n        "previous_bus_was_low_10pct",\n    ),\n    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,\n)\n\n\ndef departure_histories(\n    visits: pd.DataFrame,\n    *,\n    turnaround_seq: int,\n) -> dict[tuple[int, str], pd.DataFrame]:\n    departures = visits.loc[\n        visits["departure_seats"].ge(0)\n        & visits["departure_seen"].notna()\n        & visits["station_seq"].notna()\n    ].copy()\n    departures["station_seq"] = departures["station_seq"].astype(int)\n    departures["direction"] = np.where(\n        departures["station_seq"].le(turnaround_seq), "to_city", "return"\n    )\n    departures["departure_capacity"] = np.where(\n        departures["low_plate"].eq(2), 70.0, 45.0\n    )\n    departures["departure_load_ratio"] = (\n        1 - departures["departure_seats"] / departures["departure_capacity"]\n    ).clip(0, 1)\n    departures = departures.sort_values(\n        ["station_seq", "direction", "departure_seen", "vehicle_id"]\n    )\n    grouped = departures.groupby(["station_seq", "direction"], sort=False)\n    raw_headway = (\n        grouped["departure_seen"].diff().dt.total_seconds() / 60\n    )\n    departures["previous_bus_is_bunched"] = raw_headway.between(\n        0, 2, inclusive="both"\n    ).astype(float)\n    departures["previous_bus_headway_minutes"] = raw_headway.where(\n        raw_headway.between(0, 120, inclusive="both")\n    )\n    departures["previous_bus_headway_log1p"] = np.log1p(\n        departures["previous_bus_headway_minutes"]\n    )\n    departures["previous_bus_headway_missing"] = departures[\n        "previous_bus_headway_minutes"\n    ].isna().astype(float)\n    departures["previous_bus_departure_trend"] = grouped["departure_seats"].diff()\n    departures["previous_3_bus_departure_mean"] = grouped["departure_seats"].transform(\n        lambda values: values.rolling(3, min_periods=1).mean()\n    )\n    departures["previous_3_bus_departure_std"] = grouped["departure_seats"].transform(\n        lambda values: values.rolling(3, min_periods=2).std(ddof=0)\n    )\n    departures["previous_3_bus_departure_load_ratio_mean"] = grouped[\n        "departure_load_ratio"\n    ].transform(lambda values: values.rolling(3, min_periods=1).mean())\n    departures["previous_3_bus_departure_load_ratio_std"] = grouped[\n        "departure_load_ratio"\n    ].transform(lambda values: values.rolling(3, min_periods=2).std(ddof=0))\n    departures["previous_3_bus_departure_load_ratio_trend"] = grouped[\n        "departure_load_ratio"\n    ].transform(\n        lambda values: values.rolling(3, min_periods=2).apply(\n            lambda window: (window[-1] - window[0]) / (len(window) - 1),\n            raw=True,\n        )\n    )\n    return {\n        (int(station_seq), str(direction)): group.reset_index(drop=True)\n        for (station_seq, direction), group in departures.groupby(\n            ["station_seq", "direction"], sort=False\n        )\n    }\n\n\ndef add_previous_bus_features(\n    frame: pd.DataFrame,\n    history: pd.DataFrame | None,\n    *,\n    max_age_minutes: float = 180.0,\n    freshness_time_constant_minutes: float = 30.0,\n) -> pd.DataFrame:\n    if freshness_time_constant_minutes <= 0:\n        raise ValueError("freshness_time_constant_minutes는 0보다 커야 합니다.")\n    output = frame.copy()\n    feature_columns = [\n        "previous_bus_departure_seats",\n        "previous_bus_departure_capacity",\n        "previous_bus_departure_load_ratio",\n        "previous_bus_same_capacity",\n        "previous_bus_departure_age_minutes",\n        "previous_bus_projected_headway_minutes",\n        "previous_bus_freshness_exp",\n        "previous_bus_departure_missing",\n        "previous_bus_headway_minutes",\n        "previous_bus_headway_log1p",\n        "previous_bus_is_bunched",\n        "previous_bus_headway_missing",\n        "previous_3_bus_departure_mean",\n        "previous_3_bus_departure_std",\n        "previous_bus_departure_trend",\n        "previous_3_bus_departure_load_ratio_mean",\n        "previous_3_bus_departure_load_ratio_std",\n        "previous_3_bus_departure_load_ratio_trend",\n        "previous_bus_was_full",\n        "previous_bus_was_low_5",\n        "previous_bus_was_full_normalized",\n        "previous_bus_was_low_10pct",\n    ]\n    for column in feature_columns:\n        output[column] = np.nan\n    output["previous_bus_departure_missing"] = 1.0\n    output["previous_bus_vehicle_id"] = pd.Series(\n        pd.NA, index=output.index, dtype="string"\n    )\n    output["previous_bus_trip_id"] = pd.Series(\n        pd.NA, index=output.index, dtype="string"\n    )\n    if history is None or history.empty or output.empty:\n        return output\n\n    sort_columns = ["departure_seen"]\n    if "vehicle_id" in history.columns:\n        sort_columns.append("vehicle_id")\n    history = history.sort_values(sort_columns, kind="stable").reset_index(drop=True)\n\n    # side="left"를 사용해 스냅샷과 같은 시각의 출발도 제외한다. 실제 API\n    # 요청 시점에 이미 완료된 출발 관측만 피처로 허용하기 위해서다.\n    positions = np.asarray(\n        history["departure_seen"].searchsorted(\n            output["snapshot_time"], side="left"\n        ),\n        dtype=int,\n    ) - 1\n    # 정상 노선 순서에서는 현재 운행이 목표 정류장을 이미 출발했을 수 없지만,\n    # trip 분할 오류나 순서 이상에도 자기 라벨을 참조하지 않도록 명시적으로\n    # 같은 trip을 건너뛰고 그보다 앞선 적격 출발까지 역검색한다.\n    if "trip_id" in output.columns and "trip_id" in history.columns:\n        target_trip_ids = output["trip_id"].astype(str).to_numpy()\n        history_trip_ids = history["trip_id"].astype(str).to_numpy()\n        for index, position in enumerate(positions):\n            while (\n                position >= 0\n                and history_trip_ids[position] == target_trip_ids[index]\n            ):\n                position -= 1\n            positions[index] = position\n    valid = positions >= 0\n    if not valid.any():\n        return output\n    target_indices = np.flatnonzero(valid)\n    selected = history.iloc[positions[valid]].reset_index(drop=True)\n    age = (\n        output.iloc[target_indices]["snapshot_time"].reset_index(drop=True)\n        - selected["departure_seen"]\n    ).dt.total_seconds() / 60\n    fresh = age.between(0, max_age_minutes, inclusive="both").to_numpy()\n    target_indices = target_indices[fresh]\n    selected = selected.loc[fresh].reset_index(drop=True)\n    age = age.loc[fresh].reset_index(drop=True)\n    if len(target_indices) == 0:\n        return output\n\n    seats = selected["departure_seats"].to_numpy(dtype=float)\n    capacity = selected["departure_capacity"].to_numpy(dtype=float)\n    load_ratio = np.clip(1 - seats / capacity, 0, 1)\n    output.iloc[target_indices, output.columns.get_loc("previous_bus_departure_seats")] = seats\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_departure_capacity")\n    ] = capacity\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_departure_load_ratio")\n    ] = load_ratio\n    if "snapshot_capacity" in output.columns:\n        snapshot_capacity = pd.to_numeric(\n            output.iloc[target_indices]["snapshot_capacity"], errors="coerce"\n        ).to_numpy(dtype=float)\n        comparable = np.isfinite(capacity) & np.isfinite(snapshot_capacity)\n        same_capacity = np.full(len(target_indices), np.nan)\n        same_capacity[comparable] = np.isclose(\n            capacity[comparable], snapshot_capacity[comparable]\n        ).astype(float)\n        output.iloc[\n            target_indices, output.columns.get_loc("previous_bus_same_capacity")\n        ] = same_capacity\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_departure_age_minutes")\n    ] = age.to_numpy(dtype=float)\n    if "estimated_minutes_to_arrival" in output.columns:\n        estimated_eta = pd.to_numeric(\n            output.iloc[target_indices]["estimated_minutes_to_arrival"],\n            errors="coerce",\n        ).reset_index(drop=True)\n        estimated_eta = estimated_eta.where(estimated_eta.ge(0))\n        output.iloc[\n            target_indices,\n            output.columns.get_loc("previous_bus_projected_headway_minutes"),\n        ] = (age + estimated_eta).to_numpy(dtype=float)\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_freshness_exp")\n    ] = np.exp(-age.to_numpy(dtype=float) / freshness_time_constant_minutes)\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_departure_missing")\n    ] = 0.0\n    if "vehicle_id" in selected.columns:\n        output.iloc[\n            target_indices, output.columns.get_loc("previous_bus_vehicle_id")\n        ] = selected["vehicle_id"].astype(str).to_numpy()\n    if "trip_id" in selected.columns:\n        output.iloc[\n            target_indices, output.columns.get_loc("previous_bus_trip_id")\n        ] = selected["trip_id"].astype(str).to_numpy()\n    for target_column, source_column in [\n        ("previous_bus_headway_minutes", "previous_bus_headway_minutes"),\n        ("previous_bus_headway_log1p", "previous_bus_headway_log1p"),\n        ("previous_bus_is_bunched", "previous_bus_is_bunched"),\n        ("previous_bus_headway_missing", "previous_bus_headway_missing"),\n        ("previous_3_bus_departure_mean", "previous_3_bus_departure_mean"),\n        ("previous_3_bus_departure_std", "previous_3_bus_departure_std"),\n        ("previous_bus_departure_trend", "previous_bus_departure_trend"),\n        (\n            "previous_3_bus_departure_load_ratio_mean",\n            "previous_3_bus_departure_load_ratio_mean",\n        ),\n        (\n            "previous_3_bus_departure_load_ratio_std",\n            "previous_3_bus_departure_load_ratio_std",\n        ),\n        (\n            "previous_3_bus_departure_load_ratio_trend",\n            "previous_3_bus_departure_load_ratio_trend",\n        ),\n    ]:\n        if source_column not in selected.columns:\n            continue\n        output.iloc[target_indices, output.columns.get_loc(target_column)] = selected[\n            source_column\n        ].to_numpy(dtype=float)\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_was_full")\n    ] = (seats == 0).astype(float)\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_was_low_5")\n    ] = (seats <= 5).astype(float)\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_was_full_normalized")\n    ] = np.isclose(load_ratio, 1.0).astype(float)\n    output.iloc[\n        target_indices, output.columns.get_loc("previous_bus_was_low_10pct")\n    ] = ((1 - load_ratio) <= 0.10).astype(float)\n    return output\n\n\ndef route_progress(\n    station_seq: pd.Series,\n    direction: pd.Series,\n    turnaround_seq: int,\n    max_seq: int,\n) -> pd.Series:\n    to_city = station_seq / max(turnaround_seq, 1)\n    returning = (max_seq - station_seq) / max(max_seq - turnaround_seq, 1)\n    return pd.Series(\n        np.where(direction.eq("to_city"), to_city, returning),\n        index=station_seq.index,\n    ).clip(0, 1)\n\n\ndef build_all_prearrival_table(\n    events: pd.DataFrame,\n    visits: pd.DataFrame,\n    by_vehicle: dict[str, pd.DataFrame],\n    *,\n    turnaround_seq: int,\n    max_seq: int,\n) -> pd.DataFrame:\n    """각 도착 사건에 대해 상류 정류장마다 마지막 관측 하나를 만든다."""\n    trip_starts = visits.groupby("trip_id", sort=False)["first_seen"].min()\n    histories = departure_histories(visits, turnaround_seq=turnaround_seq)\n    frames: list[pd.DataFrame] = []\n\n    for event in events.itertuples(index=False):\n        vehicle_rows = by_vehicle.get(str(event.vehicle_id))\n        trip_start = trip_starts.get(event.trip_id)\n        if vehicle_rows is None or pd.isna(trip_start):\n            continue\n        candidates = vehicle_rows.loc[\n            vehicle_rows["observed_at"].ge(trip_start)\n            & vehicle_rows["observed_at"].lt(event.event_time)\n            & vehicle_rows["direction"].eq(event.direction)\n            & vehicle_rows["station_seq"].lt(event.station_seq)\n        ].copy()\n        if candidates.empty:\n            continue\n\n        # 같은 상류 정류장에 오래 머문 차량이 과대표집되지 않도록 정류장마다\n        # 목표 도착 전에 이용 가능한 마지막 스냅샷 하나만 남긴다.\n        candidates = (\n            candidates.sort_values(["station_seq", "observed_at", "run_id"])\n            .groupby("station_seq", sort=False)\n            .tail(1)\n            .sort_values("observed_at")\n        )\n        minute = candidates["observed_at"].dt.hour * 60 + candidates["observed_at"].dt.minute\n        angle = 2 * np.pi * minute / (24 * 60)\n        frame = pd.DataFrame(\n            {\n                "event_id": (\n                    str(event.trip_id)\n                    + ":"\n                    + str(int(event.station_seq))\n                    + ":"\n                    + str(event.event_time)\n                ),\n                "trip_id": str(event.trip_id),\n                "vehicle_id": str(event.vehicle_id),\n                "date": str(event.date),\n                "event_time": event.event_time,\n                "snapshot_time": candidates["observed_at"].to_numpy(),\n                "label_quality": str(event.label_quality),\n                "label_seats": float(event.label_seats),\n                "capacity": float(event.capacity),\n                "station_seq_cat": str(event.station_seq_cat),\n                "direction": str(event.direction),\n                "route_progress": float(event.route_progress),\n                "x": float(event.x) if pd.notna(event.x) else np.nan,\n                "y": float(event.y) if pd.notna(event.y) else np.nan,\n                "snapshot_station_seq": candidates["station_seq"].to_numpy(dtype=int),\n                "snapshot_remaining_seats": candidates["remaining_seats"].to_numpy(dtype=float),\n                "snapshot_capacity": candidates["capacity"].to_numpy(dtype=float),\n                "target_load_ratio": candidates["load_ratio"].to_numpy(dtype=float),\n                "target_stop_gap": float(event.station_seq) - candidates["station_seq"].to_numpy(dtype=float),\n                "snapshot_time_sin": np.sin(angle).to_numpy(dtype=float),\n                "snapshot_time_cos": np.cos(angle).to_numpy(dtype=float),\n                "snapshot_day_of_week": candidates["observed_at"].dt.dayofweek.astype(str).to_numpy(),\n                "snapshot_low_plate_cat": candidates["low_plate"].astype(int).astype(str).to_numpy(),\n                "target_state_cat": candidates["state_code"].astype(int).astype(str).to_numpy(),\n            }\n        )\n        frame["snapshot_route_progress"] = route_progress(\n            frame["snapshot_station_seq"],\n            frame["direction"],\n            turnaround_seq,\n            max_seq,\n        )\n        frame["minutes_to_arrival"] = (\n            frame["event_time"] - frame["snapshot_time"]\n        ).dt.total_seconds() / 60\n        stops_moved = frame["snapshot_station_seq"].diff()\n        frame["seat_delta_previous_stop"] = frame["snapshot_remaining_seats"].diff()\n        frame["seat_change_per_stop"] = (\n            frame["seat_delta_previous_stop"] / stops_moved.where(stops_moved.gt(0))\n        )\n        frame["rolling_seat_change_per_stop_3"] = (\n            frame["seat_change_per_stop"].rolling(3, min_periods=1).mean()\n        )\n        frame["minutes_since_previous_stop"] = (\n            frame["snapshot_time"].diff().dt.total_seconds() / 60\n        )\n        frame["rolling_minutes_per_stop_3"] = (\n            (frame["minutes_since_previous_stop"] / stops_moved.where(stops_moved.gt(0)))\n            .rolling(3, min_periods=1)\n            .median()\n        )\n        frame["estimated_minutes_to_arrival"] = (\n            frame["rolling_minutes_per_stop_3"] * frame["target_stop_gap"]\n        )\n        frame["projected_arrival_seats"] = (\n            frame["snapshot_remaining_seats"]\n            + frame["rolling_seat_change_per_stop_3"] * frame["target_stop_gap"]\n        ).clip(lower=0, upper=float(event.capacity))\n        frame["seats_per_remaining_stop"] = (\n            frame["snapshot_remaining_seats"] / frame["target_stop_gap"]\n        )\n        frame["load_gap_interaction"] = (\n            frame["target_load_ratio"] * frame["target_stop_gap"]\n        )\n        frame["currently_low_5"] = frame["snapshot_remaining_seats"].le(5).astype(float)\n        frame["currently_low_10"] = frame["snapshot_remaining_seats"].le(10).astype(float)\n        frame["snapshot_station_seq_cat"] = frame["snapshot_station_seq"].astype(str)\n        snapshot_minute = (\n            frame["snapshot_time"].dt.hour * 60 + frame["snapshot_time"].dt.minute\n        )\n        frame["snapshot_time_bin_30"] = (snapshot_minute // 30).astype(str)\n        frame = add_previous_bus_features(\n            frame,\n            histories.get((int(event.station_seq), str(event.direction))),\n        )\n        frame = frame.loc[\n            frame["minutes_to_arrival"].gt(0) & frame["target_stop_gap"].gt(0)\n        ]\n        frames.append(frame)\n\n    if not frames:\n        return pd.DataFrame()\n    return pd.concat(frames, ignore_index=True).sort_values(\n        ["event_time", "event_id", "snapshot_time"]\n    ).reset_index(drop=True)\n\n\ndef event_weights(data: pd.DataFrame) -> np.ndarray:\n    counts = data.groupby("event_id")["event_id"].transform("size").to_numpy(dtype=float)\n    weights = 1.0 / counts\n    return weights / weights.mean()\n\n\ndef event_bias(data: pd.DataFrame, predictions: np.ndarray) -> float:\n    residual = data[["event_id", "label_seats"]].copy()\n    residual["residual"] = residual["label_seats"].to_numpy(dtype=float) - predictions\n    per_event = residual.groupby("event_id", sort=False)["residual"].median()\n    return float(per_event.median())\n\n\ndef clip_seats(predictions: np.ndarray, capacity: pd.Series) -> np.ndarray:\n    return np.minimum(np.maximum(predictions, 0), capacity.to_numpy(dtype=float))\n\n\ndef event_balanced_metrics(\n    data: pd.DataFrame,\n    predictions: np.ndarray,\n    *,\n    model: str,\n    split: str,\n) -> dict[str, Any]:\n    scored = data[["event_id", "label_seats"]].copy()\n    scored["prediction"] = clip_seats(predictions, data["capacity"])\n    scored["absolute_error"] = (\n        scored["label_seats"] - scored["prediction"]\n    ).abs()\n    scored["within_3"] = scored["absolute_error"].le(3)\n    scored["within_5"] = scored["absolute_error"].le(5)\n    per_event = scored.groupby("event_id", sort=False).agg(\n        label_seats=("label_seats", "first"),\n        mae=("absolute_error", "mean"),\n        within_3=("within_3", "mean"),\n        within_5=("within_5", "mean"),\n    )\n    low = per_event["label_seats"].le(5)\n    y = scored["label_seats"].to_numpy(dtype=float)\n    pred = scored["prediction"].to_numpy(dtype=float)\n    return {\n        "split": split,\n        "model": model,\n        "rows": int(len(scored)),\n        "events": int(len(per_event)),\n        "low_0_5_events": int(low.sum()),\n        "row_mae_seats": float(mean_absolute_error(y, pred)),\n        "row_rmse_seats": float(math.sqrt(mean_squared_error(y, pred))),\n        "row_r2": float(r2_score(y, pred)),\n        "event_balanced_mae_seats": float(per_event["mae"].mean()),\n        "event_balanced_within_3": float(per_event["within_3"].mean()),\n        "event_balanced_within_5": float(per_event["within_5"].mean()),\n        "low_0_5_event_balanced_mae_seats": (\n            float(per_event.loc[low, "mae"].mean()) if low.any() else np.nan\n        ),\n        "low_0_5_event_balanced_within_3": (\n            float(per_event.loc[low, "within_3"].mean()) if low.any() else np.nan\n        ),\n    }\n\n\ndef make_model(seed: int) -> Pipeline:\n    return Pipeline(\n        [\n            ("features", clone(make_preprocessor(ALL_PREARRIVAL))),\n            (\n                "regressor",\n                HistGradientBoostingRegressor(\n                    loss="absolute_error",\n                    learning_rate=0.05,\n                    max_iter=300,\n                    max_leaf_nodes=15,\n                    min_samples_leaf=20,\n                    l2_regularization=1.0,\n                    random_state=seed,\n                ),\n            ),\n        ]\n    )\n\n\ndef fit_and_predict(\n    train: pd.DataFrame,\n    calibration: pd.DataFrame,\n    test: pd.DataFrame,\n    *,\n    delta_target: bool,\n    seed: int,\n) -> tuple[np.ndarray, float, float]:\n    model = make_model(seed)\n    train_y = train["label_seats"].to_numpy(dtype=float)\n    if delta_target:\n        train_y = train_y - train["snapshot_remaining_seats"].to_numpy(dtype=float)\n    model.fit(\n        train[ALL_PREARRIVAL.columns],\n        train_y,\n        regressor__sample_weight=event_weights(train),\n    )\n    cal_prediction = model.predict(calibration[ALL_PREARRIVAL.columns])\n    test_prediction = model.predict(test[ALL_PREARRIVAL.columns])\n    if delta_target:\n        cal_prediction += calibration["snapshot_remaining_seats"].to_numpy(dtype=float)\n        test_prediction += test["snapshot_remaining_seats"].to_numpy(dtype=float)\n    bias = event_bias(calibration, cal_prediction)\n    cal_prediction = clip_seats(cal_prediction + bias, calibration["capacity"])\n    calibration_mae = event_balanced_metrics(\n        calibration,\n        cal_prediction,\n        model="calibration",\n        split="calibration",\n    )["event_balanced_mae_seats"]\n    return test_prediction + bias, float(bias), float(calibration_mae)\n\n\ndef bucket_metrics(\n    test: pd.DataFrame,\n    predictions: np.ndarray,\n    column: str,\n    bins: list[float],\n    labels: list[str],\n) -> pd.DataFrame:\n    bucket = pd.cut(test[column], bins=bins, labels=labels, include_lowest=True)\n    rows: list[dict[str, Any]] = []\n    for label in labels:\n        mask = bucket.eq(label).to_numpy()\n        if not mask.any():\n            continue\n        result = event_balanced_metrics(\n            test.loc[mask], predictions[mask], model="selected", split=str(label)\n        )\n        result["bucket"] = str(label)\n        rows.append(result)\n    return pd.DataFrame(rows)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="모든 상류 정류장 스냅샷에서 도착 잔여좌석 회귀"\n    )\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/all_prearrival_seat_results"),\n    )\n    parser.add_argument("--include-inferred", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    locations, stations = load_data(args.db, ROUTE_ID)\n    visits = build_visits(locations, stations)\n    model_table, turnaround_seq = build_model_table(\n        visits, stations, label_target="arrival"\n    )\n    source = prepare_subset(model_table)\n    qualities = ["A", "B"] if args.include_inferred else ["A"]\n    source = source.loc[source["label_quality"].isin(qualities)].copy()\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n    snapshots = build_all_prearrival_table(\n        source,\n        visits,\n        by_vehicle,\n        turnaround_seq=turnaround_seq,\n        max_seq=int(stations["station_seq"].max()),\n    )\n    train = snapshots.loc[snapshots["date"].isin(["2026-08-04", "2026-08-05"])]\n    calibration = snapshots.loc[snapshots["date"].eq("2026-08-06")]\n    test = snapshots.loc[snapshots["date"].eq("2026-08-07")]\n    if min(train["event_id"].nunique(), calibration["event_id"].nunique(), test["event_id"].nunique()) == 0:\n        raise ValueError("시간 분할 중 하나에 도착 사건이 없습니다.")\n\n    rows: list[dict[str, Any]] = []\n    predictions: dict[str, np.ndarray] = {}\n    persistence = test["snapshot_remaining_seats"].to_numpy(dtype=float)\n    predictions["current_seats_persistence"] = persistence\n    baseline = event_balanced_metrics(\n        test,\n        persistence,\n        model="current_seats_persistence",\n        split="test_2026-08-07",\n    )\n    baseline["selection_calibration_mae"] = event_balanced_metrics(\n        calibration,\n        calibration["snapshot_remaining_seats"].to_numpy(dtype=float),\n        model="current_seats_persistence",\n        split="calibration",\n    )["event_balanced_mae_seats"]\n    baseline["bias_correction_seats"] = 0.0\n    rows.append(baseline)\n\n    for model_name, delta in [("direct_hgb", False), ("delta_hgb", True)]:\n        prediction, bias, calibration_mae = fit_and_predict(\n            train, calibration, test, delta_target=delta, seed=args.seed\n        )\n        predictions[model_name] = clip_seats(prediction, test["capacity"])\n        result = event_balanced_metrics(\n            test,\n            prediction,\n            model=model_name,\n            split="test_2026-08-07",\n        )\n        result["selection_calibration_mae"] = calibration_mae\n        result["bias_correction_seats"] = bias\n        rows.append(result)\n\n    metrics = pd.DataFrame(rows)\n    selected_row = metrics.loc[metrics["model"].ne("current_seats_persistence")].sort_values(\n        ["selection_calibration_mae", "event_balanced_mae_seats"]\n    ).iloc[0]\n    selected_name = str(selected_row["model"])\n    selected_prediction = predictions[selected_name]\n    by_minutes = bucket_metrics(\n        test,\n        selected_prediction,\n        "minutes_to_arrival",\n        [-np.inf, 5, 10, 20, 30, 60, np.inf],\n        ["0-5", "5-10", "10-20", "20-30", "30-60", "60+"],\n    )\n    by_stops = bucket_metrics(\n        test,\n        selected_prediction,\n        "target_stop_gap",\n        [0, 1, 2, 5, 10, 20, np.inf],\n        ["1", "2", "3-5", "6-10", "11-20", "21+"],\n    )\n\n    metrics.to_csv(args.output_dir / "model_metrics.csv", index=False)\n    by_minutes.to_csv(args.output_dir / "metrics_by_minutes.csv", index=False)\n    by_stops.to_csv(args.output_dir / "metrics_by_stops.csv", index=False)\n    result = {\n        "route_id": ROUTE_ID,\n        "route_name": ROUTE_NAME,\n        "target": "arrival_seats before boarding",\n        "label_quality": qualities,\n        "snapshot_unit": "last available observation at every upstream station",\n        "feature_leakage_rule": (\n            "actual minutes_to_arrival is evaluation-only; model uses current time, "\n            "current seats, current/target position, vehicle state, and target metadata"\n        ),\n        "split": {\n            "train": ["2026-08-04", "2026-08-05"],\n            "calibration_and_selection": ["2026-08-06"],\n            "untouched_test": ["2026-08-07"],\n        },\n        "rows": {\n            "train": int(len(train)),\n            "calibration": int(len(calibration)),\n            "test": int(len(test)),\n        },\n        "events": {\n            "train": int(train["event_id"].nunique()),\n            "calibration": int(calibration["event_id"].nunique()),\n            "test": int(test["event_id"].nunique()),\n        },\n        "models": _as_records(metrics),\n        "selected_on_calibration": json_ready(selected_row.to_dict()),\n        "performance_by_minutes": _as_records(by_minutes),\n        "performance_by_stops": _as_records(by_stops),\n        "limitations": [\n            "실제 도착까지 남은 시간은 평가 구간화에만 사용했으며 모델 피처에는 넣지 않았다.",\n            "한 도착 사건에 여러 상류 정류장 스냅샷이 있으므로 사건별 동일 가중 지표를 주 지표로 사용했다.",\n            "A등급 직접 도착 관측만 기본 평가에 사용한다.",\n            "최종 테스트가 하루라 장기 운영 성능 확정치가 아니다.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/main_model_feature_augmentation.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\nfrom sklearn.pipeline import Pipeline\n\nfrom all_prearrival_seat_regression import DYNAMIC_ALL_PREARRIVAL, event_weights\nfrom hypothesis_model_search import (\n    CAPACITY_44_ALL_PREARRIVAL,\n    DEFAULT_DEVELOPMENT_DATES,\n    DEFAULT_STRESS_DATES,\n    DEFAULT_TEST_DATE,\n    OBSERVED_SEAT_CEILING_PARAM,\n    Candidate,\n    EnsembleSpec,\n    add_observed_capacity_features,\n    candidate_weights,\n    combine_weighted_oof,\n    decode_target,\n    development_folds,\n    encode_target,\n    event_bias,\n    event_summary,\n    postprocess_ensemble_prediction,\n    scored_frame,\n    strict_forward_bias_predictions,\n    tree_node_count,\n)\nfrom linear_feature_experiment import (\n    PATH_FLOW_FEATURES,\n    PATH_LOW_RATE_FEATURES,\n    PRECEDING_BUS_FEATURES,\n    TARGET_LOW_RATE_FEATURES,\n    add_preceding_bus_segment_features,\n    add_strict_prior_flow_features,\n    add_strict_prior_low_rate_features,\n    json_ready,\n)\nfrom model_feasibility import FeatureSet, make_preprocessor\n\n\nBASE_NUMERIC = CAPACITY_44_ALL_PREARRIVAL.numeric\nBASE_CATEGORICAL = CAPACITY_44_ALL_PREARRIVAL.categorical\nALL_PROPOSED = (\n    *TARGET_LOW_RATE_FEATURES,\n    *PATH_LOW_RATE_FEATURES,\n    *PATH_FLOW_FEATURES,\n    *PRECEDING_BUS_FEATURES,\n)\n\n# This is not merely correlated: it is exactly target_stop_gap. Current seats\n# are also already deployed and therefore are not repeated in ALL_PROPOSED.\nEXACT_DUPLICATE_PROPOSED = ("path_low_rate_stop_count",)\nUNIQUE_PROPOSED = tuple(\n    column for column in ALL_PROPOSED if column not in EXACT_DUPLICATE_PROPOSED\n)\n\n# Qualitative meaning review plus the development-only Pearson/Spearman audit:\n# retain a target risk, normalized/cumulative path risk, normalized/cumulative\n# stop flow, dispersion/reliability, and one normalized preceding-bus signal.\nCORRELATION_PRUNED_PROPOSED = (\n    "target_low_10_rate",\n    "target_low_rate_log_count",\n    "path_low_10_mean",\n    "path_low_10_sum",\n    "path_flow_mean",\n    "path_flow_sum",\n    "path_flow_std",\n    "path_flow_fallback_share",\n    "preceding_bus_segment_delta_per_stop",\n    "previous_bus_departure_age_minutes",\n    "preceding_bus_segment_missing",\n)\n\n# Existing features removed only where their meaning is already represented and\n# both Pearson and Spearman are near deterministic on development data. Keep the\n# observed-ceiling variants because they encode the corrected 44/70-seat support.\nREDUNDANT_EXISTING = (\n    "snapshot_capacity",\n    "target_load_ratio",\n    "seat_change_per_stop",\n    "load_gap_interaction",\n    "currently_low_5",\n    "currently_low_10",\n    "x",\n    "y",\n)\n\n\ndef feature_sets() -> dict[str, FeatureSet]:\n    lean_numeric = tuple(\n        column for column in BASE_NUMERIC if column not in REDUNDANT_EXISTING\n    )\n    low_rate_block = (\n        "target_low_10_rate",\n        "target_low_rate_log_count",\n        "path_low_10_mean",\n        "path_low_10_sum",\n    )\n    path_flow_block = (\n        "path_flow_mean",\n        "path_flow_sum",\n        "path_flow_std",\n        "path_flow_fallback_share",\n    )\n    importance_pruned = tuple(\n        column\n        for column in CORRELATION_PRUNED_PROPOSED\n        if column\n        not in {"preceding_bus_segment_delta_per_stop", "preceding_bus_segment_missing"}\n    )\n    return {\n        "baseline_official": FeatureSet(\n            "baseline_official", BASE_NUMERIC, BASE_CATEGORICAL\n        ),\n        "augmented_unique": FeatureSet(\n            "augmented_unique", (*BASE_NUMERIC, *UNIQUE_PROPOSED), BASE_CATEGORICAL\n        ),\n        "correlation_pruned": FeatureSet(\n            "correlation_pruned",\n            (*BASE_NUMERIC, *CORRELATION_PRUNED_PROPOSED),\n            BASE_CATEGORICAL,\n        ),\n        "lean_merged": FeatureSet(\n            "lean_merged",\n            (*lean_numeric, *CORRELATION_PRUNED_PROPOSED),\n            BASE_CATEGORICAL,\n        ),\n        "baseline_no_redundant": FeatureSet(\n            "baseline_no_redundant", lean_numeric, BASE_CATEGORICAL\n        ),\n        "low_rate_block": FeatureSet(\n            "low_rate_block", (*BASE_NUMERIC, *low_rate_block), BASE_CATEGORICAL\n        ),\n        "path_flow_block": FeatureSet(\n            "path_flow_block", (*BASE_NUMERIC, *path_flow_block), BASE_CATEGORICAL\n        ),\n        "path_flow_sum_only": FeatureSet(\n            "path_flow_sum_only", (*BASE_NUMERIC, "path_flow_sum"), BASE_CATEGORICAL\n        ),\n        "low_rate_plus_flow_sum": FeatureSet(\n            "low_rate_plus_flow_sum",\n            (*BASE_NUMERIC, *low_rate_block, "path_flow_sum"),\n            BASE_CATEGORICAL,\n        ),\n        "importance_pruned": FeatureSet(\n            "importance_pruned",\n            (*BASE_NUMERIC, *importance_pruned),\n            BASE_CATEGORICAL,\n        ),\n        "lean_importance_pruned": FeatureSet(\n            "lean_importance_pruned",\n            (*lean_numeric, *importance_pruned),\n            BASE_CATEGORICAL,\n        ),\n        "target_low_10_only": FeatureSet(\n            "target_low_10_only",\n            (*BASE_NUMERIC, "target_low_10_rate"),\n            BASE_CATEGORICAL,\n        ),\n        "target_low_10_with_count": FeatureSet(\n            "target_low_10_with_count",\n            (*BASE_NUMERIC, "target_low_10_rate", "target_low_rate_log_count"),\n            BASE_CATEGORICAL,\n        ),\n        "path_low_10_mean_only": FeatureSet(\n            "path_low_10_mean_only",\n            (*BASE_NUMERIC, "path_low_10_mean"),\n            BASE_CATEGORICAL,\n        ),\n        "path_low_10_sum_only": FeatureSet(\n            "path_low_10_sum_only",\n            (*BASE_NUMERIC, "path_low_10_sum"),\n            BASE_CATEGORICAL,\n        ),\n        "low_rate_without_count": FeatureSet(\n            "low_rate_without_count",\n            (\n                *BASE_NUMERIC,\n                "target_low_10_rate",\n                "path_low_10_mean",\n                "path_low_10_sum",\n            ),\n            BASE_CATEGORICAL,\n        ),\n        "previous_bus_age_only": FeatureSet(\n            "previous_bus_age_only",\n            (*BASE_NUMERIC, "previous_bus_departure_age_minutes"),\n            BASE_CATEGORICAL,\n        ),\n    }\n\n\ndef component_candidates() -> tuple[Candidate, ...]:\n    common = {\n        "stage": "main_feature_augmentation",\n        "why": "fixed reproduction of the deployed component with a candidate schema",\n        "if_works": "strict OOF improves without changing model family or weights",\n        "if_fails": "the added schema is redundant or unstable",\n        "feature_variant": "custom",\n        "low_weight": 2.0,\n        "weighting_kind": "event",\n    }\n    return (\n        Candidate(\n            name="hgb",\n            model_kind="hgb",\n            target_kind="delta_per_stop",\n            params={\n                "loss": "absolute_error",\n                "learning_rate": 0.04,\n                "max_iter": 500,\n                "max_leaf_nodes": 63,\n                "min_samples_leaf": 5,\n                "l2_regularization": 0.0,\n                "early_stopping": False,\n            },\n            **common,\n        ),\n        Candidate(\n            name="extra_trees",\n            model_kind="extra_trees",\n            target_kind="delta_per_stop",\n            params={\n                "n_estimators": 48,\n                "max_depth": 18,\n                "min_samples_leaf": 5,\n                "max_features": 0.7,\n            },\n            **common,\n        ),\n        Candidate(\n            name="lightgbm",\n            model_kind="lightgbm",\n            target_kind="delta_per_sqrt_stop",\n            gap_weight_power=0.5,\n            params={\n                "objective": "regression_l1",\n                "n_estimators": 600,\n                "learning_rate": 0.025,\n                "num_leaves": 63,\n                "min_child_samples": 10,\n                "subsample": 0.85,\n                "subsample_freq": 1,\n                "colsample_bytree": 0.8,\n                "reg_alpha": 0.1,\n                "reg_lambda": 1.0,\n            },\n            **common,\n        ),\n    )\n\n\ndef make_model(candidate: Candidate, features: FeatureSet, seed: int) -> Pipeline:\n    if candidate.model_kind == "hgb":\n        estimator = HistGradientBoostingRegressor(\n            random_state=seed, **candidate.params\n        )\n    elif candidate.model_kind == "extra_trees":\n        estimator = ExtraTreesRegressor(\n            n_jobs=-1, random_state=seed, **candidate.params\n        )\n    elif candidate.model_kind == "lightgbm":\n        estimator = lgb.LGBMRegressor(\n            n_jobs=-1, random_state=seed, verbosity=-1, **candidate.params\n        )\n    else:\n        raise ValueError(candidate.model_kind)\n    return Pipeline(\n        [("features", make_preprocessor(features)), ("regressor", estimator)]\n    )\n\n\ndef prepare_augmented(cache: Path, flow_cache: Path, output_cache: Path) -> pd.DataFrame:\n    if output_cache.is_file():\n        return pd.read_pickle(output_cache)\n    data = pd.read_pickle(cache)\n    flows = pd.read_pickle(flow_cache)\n    data = add_strict_prior_low_rate_features(data)\n    data = add_strict_prior_flow_features(data, flows)\n    data, audit = add_preceding_bus_segment_features(data, flows)\n    data.attrs["preceding_bus_segment_audit"] = audit\n    output_cache.parent.mkdir(parents=True, exist_ok=True)\n    data.to_pickle(output_cache)\n    return data\n\n\ndef correlation_audit(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    development = data.loc[data["date"].isin(DEFAULT_DEVELOPMENT_DATES)].copy()\n    numeric = list(dict.fromkeys((*BASE_NUMERIC, *ALL_PROPOSED)))\n    rows: list[dict[str, Any]] = []\n    for method in ("pearson", "spearman"):\n        correlation = development[numeric].corr(method=method)\n        for position, left in enumerate(numeric):\n            for right in numeric[:position]:\n                value = float(correlation.at[left, right])\n                if np.isfinite(value) and abs(value) >= 0.85:\n                    rows.append(\n                        {\n                            "method": method,\n                            "left": left,\n                            "right": right,\n                            "correlation": value,\n                            "absolute_correlation": abs(value),\n                            "contains_proposed_feature": bool(\n                                left in UNIQUE_PROPOSED or right in UNIQUE_PROPOSED\n                            ),\n                        }\n                    )\n    missing = pd.DataFrame(\n        [\n            {\n                "feature": column,\n                "missing_rate": float(development[column].isna().mean()),\n                "non_missing_unique": int(development[column].nunique(dropna=True)),\n            }\n            for column in numeric\n        ]\n    )\n    return pd.DataFrame(rows), missing\n\n\ndef qualitative_catalog() -> pd.DataFrame:\n    decisions = {\n        "target_low_5_rate": ("remove_correlation", "same target risk as low10; lower support and Pearson/Spearman 0.924/0.952"),\n        "target_low_10_rate": ("keep", "directly aligned with the required low<=10 metric"),\n        "target_low_rate_log_count": ("keep", "history reliability, not a risk level duplicate"),\n        "path_low_5_mean": ("remove_correlation", "same path risk as low10 mean; Pearson/Spearman 0.942/0.975"),\n        "path_low_5_median": ("remove_correlation", "same distribution center as retained mean"),\n        "path_low_5_sum": ("remove_correlation", "same cumulative risk as low10 sum; Pearson/Spearman 0.972/0.988"),\n        "path_low_10_mean": ("keep", "distance-normalized path risk"),\n        "path_low_10_median": ("remove_correlation", "same path center as mean; Pearson/Spearman 0.900/0.936"),\n        "path_low_10_sum": ("keep", "cumulative path risk distinct from normalized mean"),\n        "path_low_rate_stop_count": ("remove_exact", "exactly equals target_stop_gap"),\n        "path_flow_mean": ("keep", "distance-normalized historical net seat flow"),\n        "path_flow_median": ("remove_correlation", "same flow center as mean; Pearson/Spearman 0.971/0.949"),\n        "path_flow_sum": ("keep", "expected cumulative net seat change"),\n        "path_flow_std": ("keep", "path-flow heterogeneity"),\n        "path_flow_fallback_share": ("keep", "historical lookup reliability"),\n        "preceding_bus_segment_delta": ("remove_correlation", "same segment signal as normalized delta; Spearman 0.968"),\n        "preceding_bus_segment_delta_per_stop": ("keep_pending_importance", "normalized preceding-bus segment signal"),\n        "preceding_bus_segment_start_seats": ("remove_semantic", "overlaps existing previous-bus departure seats"),\n        "preceding_bus_target_arrival_seats": ("remove_semantic", "arithmetic combination of segment start and delta"),\n        "previous_bus_departure_age_minutes": ("keep_pending_importance", "prior-bus freshness; available in raw data but not the deployed schema"),\n        "preceding_bus_segment_missing": ("keep_pending_importance", "required missingness signal for 95% sparse segment feature"),\n    }\n    rows = [\n        {\n            "source": "proposed",\n            "feature": feature,\n            "decision": decisions[feature][0],\n            "reason": decisions[feature][1],\n        }\n        for feature in ALL_PROPOSED\n    ]\n    existing = {\n        "snapshot_capacity": "same capacity regime as observed_ceiling_capacity; Pearson/Spearman 1.000",\n        "target_load_ratio": "legacy-capacity version of observed_ceiling_load_ratio; correlation above 0.998",\n        "seat_change_per_stop": "near-deterministic proxy of seat_delta_previous_stop; correlation above 0.993",\n        "load_gap_interaction": "legacy-capacity version of observed_ceiling_load_gap; correlation above 0.986",\n        "currently_low_5": "deterministic threshold of snapshot_remaining_seats and low prior permutation importance",\n        "currently_low_10": "deterministic threshold of snapshot_remaining_seats and low prior permutation importance",\n        "x": "fixed-route location proxy of route_progress; correlation above 0.988",\n        "y": "fixed-route location proxy of route_progress; absolute correlation above 0.972",\n    }\n    rows.extend(\n        {\n            "source": "existing",\n            "feature": feature,\n            "decision": "candidate_remove_rejected_by_ablation",\n            "reason": reason,\n        }\n        for feature, reason in existing.items()\n    )\n    return pd.DataFrame(rows)\n\n\ndef paired_trip_bootstrap(\n    predictions: pd.DataFrame,\n    candidate: str,\n    *,\n    low_only: bool,\n    repeats: int = 2_000,\n    seed: int = 42,\n) -> dict[str, Any]:\n    selected = predictions.loc[\n        predictions["candidate"].isin(["baseline_official", candidate])\n    ].copy()\n    selected["absolute_error"] = (\n        selected["label_seats"] - selected["prediction"]\n    ).abs()\n    if low_only:\n        selected = selected.loc[selected["label_seats"].le(10)].copy()\n    per_event = (\n        selected.groupby(["trip_id", "event_id", "candidate"], observed=True)[\n            "absolute_error"\n        ]\n        .mean()\n        .unstack("candidate")\n        .dropna(subset=["baseline_official", candidate])\n        .reset_index()\n    )\n    per_event["improvement"] = (\n        per_event["baseline_official"] - per_event[candidate]\n    )\n    trip_groups = [\n        group["improvement"].to_numpy(float)\n        for _, group in per_event.groupby("trip_id", sort=False)\n    ]\n    rng = np.random.default_rng(seed)\n    draws = np.empty(repeats, dtype=float)\n    for index in range(repeats):\n        sampled = rng.integers(0, len(trip_groups), size=len(trip_groups))\n        draws[index] = np.concatenate([trip_groups[item] for item in sampled]).mean()\n    return {\n        "candidate": candidate,\n        "metric": "low_0_10_mae" if low_only else "event_balanced_mae",\n        "candidate_improvement": float(per_event["improvement"].mean()),\n        "ci_95_lower": float(np.quantile(draws, 0.025)),\n        "ci_95_upper": float(np.quantile(draws, 0.975)),\n        "probability_candidate_better": float((draws > 0).mean()),\n        "trips": int(per_event["trip_id"].nunique()),\n        "events": int(len(per_event)),\n    }\n\n\ndef _feature_importance_rows(\n    model: Pipeline,\n    features: FeatureSet,\n    *,\n    feature_set_name: str,\n    component: str,\n    validation_date: str,\n) -> list[dict[str, Any]]:\n    estimator = model.named_steps["regressor"]\n    if hasattr(estimator, "booster_"):\n        raw = estimator.booster_.feature_importance(importance_type="gain")\n    elif hasattr(estimator, "feature_importances_"):\n        raw = estimator.feature_importances_\n    else:\n        return []\n    # Numeric values are emitted first by make_preprocessor. Missing indicators\n    # and one-hot columns follow; this audit deliberately reports the direct\n    # numeric contribution used for pruning proposed numeric features.\n    direct = np.asarray(raw, dtype=float)[: len(features.numeric)]\n    total = max(float(np.asarray(raw, dtype=float).sum()), 1e-12)\n    return [\n        {\n            "feature_set": feature_set_name,\n            "component": component,\n            "validation_date": validation_date,\n            "feature": feature,\n            "importance": float(value),\n            "normalized_importance": float(value / total),\n        }\n        for feature, value in zip(features.numeric, direct, strict=True)\n    ]\n\n\ndef run_component(\n    candidate: Candidate,\n    features: FeatureSet,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    *,\n    feature_set_name: str,\n    seed: int,\n) -> tuple[pd.DataFrame, list[dict[str, Any]], list[dict[str, Any]]]:\n    raw_frames: list[pd.DataFrame] = []\n    importance: list[dict[str, Any]] = []\n    diagnostics: list[dict[str, Any]] = []\n    for fold_number, (validation_date, train, validation) in enumerate(folds):\n        model = make_model(candidate, features, seed + fold_number)\n        started = time.perf_counter()\n        model.fit(\n            train[list(features.columns)],\n            encode_target(train, candidate.target_kind),\n            regressor__sample_weight=candidate_weights(\n                train,\n                candidate.low_weight,\n                weighting_kind=candidate.weighting_kind,\n                gap_weight_power=candidate.gap_weight_power,\n            ),\n        )\n        prediction = decode_target(\n            model.predict(validation[list(features.columns)]),\n            validation,\n            candidate.target_kind,\n        )\n        raw_frames.append(scored_frame(validation, prediction, candidate=candidate.name))\n        importance.extend(\n            _feature_importance_rows(\n                model,\n                features,\n                feature_set_name=feature_set_name,\n                component=candidate.name,\n                validation_date=validation_date,\n            )\n        )\n        diagnostics.append(\n            {\n                "feature_set": feature_set_name,\n                "component": candidate.name,\n                "validation_date": validation_date,\n                "fit_seconds": time.perf_counter() - started,\n                "tree_nodes": tree_node_count(model),\n            }\n        )\n    output = pd.concat(raw_frames, ignore_index=True)\n    raw = output["prediction"].to_numpy(float)\n    strict, biases = strict_forward_bias_predictions(\n        output, raw, validation_dates=[fold[0] for fold in folds]\n    )\n    output["raw_prediction"] = raw\n    output["prediction"] = strict\n    output["deployment_prediction"] = raw + event_bias(output, raw)\n    for row in diagnostics:\n        row["strict_oof_bias"] = biases[row["validation_date"]]\n    return output, importance, diagnostics\n\n\ndef required_metrics(data: pd.DataFrame) -> dict[str, Any]:\n    prediction = data["prediction"].to_numpy(float)\n    base = event_summary(data, prediction)\n    weights = event_weights(data)\n    truth = data["label_seats"].to_numpy(float)\n    error = np.abs(truth - prediction)\n    low10 = truth <= 10\n    full_true = truth == 0\n    full_pred = prediction <= 0.5\n    base.update(\n        {\n            "low_0_10_events": int(data.loc[low10, "event_id"].nunique()),\n            "low_0_10_mae": float(np.average(error[low10], weights=weights[low10])),\n            "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),\n            "full_recall": float(recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)),\n            "full_precision": float(precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)),\n            "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),\n            "full_threshold_seats": 0.5,\n        }\n    )\n    return base\n\n\ndef run_feature_set(\n    name: str,\n    features: FeatureSet,\n    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],\n    *,\n    seed: int,\n) -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame, pd.DataFrame]:\n    components: dict[str, pd.DataFrame] = {}\n    importance: list[dict[str, Any]] = []\n    diagnostics: list[dict[str, Any]] = []\n    for candidate in component_candidates():\n        print(f"[{name}] {candidate.name}", flush=True)\n        output, component_importance, component_diagnostics = run_component(\n            candidate,\n            features,\n            folds,\n            feature_set_name=name,\n            seed=seed,\n        )\n        components[candidate.name] = output\n        importance.extend(component_importance)\n        diagnostics.extend(component_diagnostics)\n    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}\n    blended = combine_weighted_oof(name, components, weights)\n    strict, biases = strict_forward_bias_predictions(blended, blended["prediction"])\n    spec = EnsembleSpec(\n        name=name,\n        kind="weighted",\n        components=tuple(weights),\n        params={**weights, OBSERVED_SEAT_CEILING_PARAM: 1.0},\n    )\n    blended["prediction"] = postprocess_ensemble_prediction(spec, strict, blended)\n    deployment_base = blended["deployment_prediction"].to_numpy(float)\n    deployment_bias = event_bias(blended, deployment_base)\n    blended["deployment_prediction"] = postprocess_ensemble_prediction(\n        spec, deployment_base + deployment_bias, blended\n    )\n    blended["ensemble_deployment_bias"] = deployment_bias\n    blended["candidate"] = name\n    metrics = required_metrics(blended)\n    daily = pd.DataFrame(\n        [\n            {"feature_set": name, "validation_date": date, **required_metrics(frame)}\n            for date, frame in blended.groupby("date", sort=True)\n        ]\n    )\n    metrics.update(\n        {\n            "feature_set": name,\n            "numeric_features": len(features.numeric),\n            "categorical_features": len(features.categorical),\n            "total_features": len(features.columns),\n            "strict_ensemble_biases": json.dumps(biases, sort_keys=True),\n            "deployment_bias": deployment_bias,\n        }\n    )\n    return blended, metrics, daily, pd.DataFrame(importance + diagnostics)\n\n\ndef fit_final_models(\n    name: str,\n    features: FeatureSet,\n    development: pd.DataFrame,\n    targets: dict[str, pd.DataFrame],\n    oof: pd.DataFrame,\n    output_dir: Path,\n    *,\n    seed: int,\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    component_predictions: dict[str, dict[str, np.ndarray]] = {}\n    component_nodes: dict[str, int] = {}\n    model_dir = output_dir / "candidate_models" / name\n    model_dir.mkdir(parents=True, exist_ok=True)\n    for candidate in component_candidates():\n        model = make_model(candidate, features, seed)\n        model.fit(\n            development[list(features.columns)],\n            encode_target(development, candidate.target_kind),\n            regressor__sample_weight=candidate_weights(\n                development,\n                candidate.low_weight,\n                weighting_kind=candidate.weighting_kind,\n                gap_weight_power=candidate.gap_weight_power,\n            ),\n        )\n        component_predictions[candidate.name] = {\n            key: decode_target(\n                model.predict(frame[list(features.columns)]),\n                frame,\n                candidate.target_kind,\n            )\n            for key, frame in targets.items()\n        }\n        component_nodes[candidate.name] = tree_node_count(model)\n        joblib.dump(model, model_dir / f"{candidate.name}.joblib")\n\n    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}\n    spec = EnsembleSpec(\n        name=name,\n        kind="weighted",\n        components=tuple(weights),\n        params={**weights, OBSERVED_SEAT_CEILING_PARAM: 1.0},\n    )\n    # This is the same all-development OOF deployment correction computed from\n    # component deployment predictions in run_feature_set. It never changes a\n    # development-fold selection prediction.\n    deployment_bias = float(oof["ensemble_deployment_bias"].iloc[0])\n    metric_rows: list[dict[str, Any]] = []\n    for target_name, target in targets.items():\n        prediction = sum(\n            weights[component] * component_predictions[component][target_name]\n            for component in weights\n        )\n        prediction = postprocess_ensemble_prediction(\n            spec, prediction + deployment_bias, target\n        )\n        scored = scored_frame(target, prediction, candidate=name)\n        metric_rows.append(\n            {"feature_set": name, "split": target_name, **required_metrics(scored)}\n        )\n    metadata = {\n        "feature_set": name,\n        "feature_columns": features.columns,\n        "numeric_features": features.numeric,\n        "categorical_features": features.categorical,\n        "component_candidates": [asdict(item) for item in component_candidates()],\n        "weights": weights,\n        "deployment_bias": deployment_bias,\n        "component_tree_nodes": component_nodes,\n        "total_tree_nodes": int(sum(component_nodes.values())),\n    }\n    (model_dir / "metadata.json").write_text(\n        json.dumps(json_ready(metadata), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    return pd.DataFrame(metric_rows), metadata\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="기존 주 모델에 strict-prior 제안 피처를 결합하고 중복/저효율 schema를 비교"\n    )\n    parser.add_argument(\n        "--cache", type=Path, default=Path("data/analysis_cache/all_prearrival_A.pkl")\n    )\n    parser.add_argument(\n        "--flow-cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),\n    )\n    parser.add_argument(\n        "--augmented-cache",\n        type=Path,\n        default=Path("data/analysis_cache/main_model_augmented_features.pkl"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/main_model_feature_augmentation_results"),\n    )\n    parser.add_argument("--feature-sets", nargs="*", default=list(feature_sets()))\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--skip-final-fit", action="store_true")\n    args = parser.parse_args()\n\n    data = prepare_augmented(args.cache, args.flow_cache, args.augmented_cache)\n    data = add_observed_capacity_features(data)\n    selected_sets = feature_sets()\n    unknown = sorted(set(args.feature_sets) - set(selected_sets))\n    if unknown:\n        raise ValueError(f"unknown feature sets: {unknown}")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    correlations, missing = correlation_audit(data)\n    correlations.to_csv(args.output_dir / "high_correlation_pairs.csv", index=False)\n    missing.to_csv(args.output_dir / "feature_missingness.csv", index=False)\n    qualitative_catalog().to_csv(\n        args.output_dir / "qualitative_overlap_catalog.csv", index=False\n    )\n\n    development = data.loc[data["date"].isin(DEFAULT_DEVELOPMENT_DATES)].copy()\n    folds = development_folds(development, DEFAULT_DEVELOPMENT_DATES)\n    all_oof: list[pd.DataFrame] = []\n    all_metrics: list[dict[str, Any]] = []\n    all_daily: list[pd.DataFrame] = []\n    all_diagnostics: list[pd.DataFrame] = []\n    for name in args.feature_sets:\n        output, metrics, daily, diagnostics = run_feature_set(\n            name, selected_sets[name], folds, seed=args.seed\n        )\n        all_oof.append(output)\n        all_metrics.append(metrics)\n        all_daily.append(daily)\n        all_diagnostics.append(diagnostics)\n    oof = pd.concat(all_oof, ignore_index=True)\n    metrics = pd.DataFrame(all_metrics)\n    daily = pd.concat(all_daily, ignore_index=True)\n    diagnostics = pd.concat(all_diagnostics, ignore_index=True)\n    oof.to_pickle(args.output_dir / "oof_predictions.pkl")\n    metrics.to_csv(args.output_dir / "development_metrics.csv", index=False)\n    daily.to_csv(args.output_dir / "daily_metrics.csv", index=False)\n    diagnostics.to_csv(args.output_dir / "importance_and_fit_diagnostics.csv", index=False)\n    bootstrap = pd.DataFrame()\n    if "baseline_official" in set(oof["candidate"]):\n        candidates = [\n            name for name in args.feature_sets if name != "baseline_official"\n        ]\n        bootstrap = pd.DataFrame(\n            [\n                paired_trip_bootstrap(oof, candidate, low_only=low_only, seed=args.seed)\n                for candidate in candidates\n                for low_only in (False, True)\n            ]\n        )\n        bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)\n\n    final_metrics = pd.DataFrame()\n    final_metadata: dict[str, Any] = {}\n    if not args.skip_final_fit:\n        targets = {\n            "locked_test_2026_08_11": data.loc[data["date"].eq(DEFAULT_TEST_DATE)].copy(),\n            "weekend_stress": data.loc[data["date"].isin(DEFAULT_STRESS_DATES)].copy(),\n        }\n        final_rows: list[pd.DataFrame] = []\n        for name in args.feature_sets:\n            print(f"[{name}] final fit", flush=True)\n            rows, metadata = fit_final_models(\n                name,\n                selected_sets[name],\n                development,\n                targets,\n                oof.loc[oof["candidate"].eq(name)].copy(),\n                args.output_dir,\n                seed=args.seed,\n            )\n            final_rows.append(rows)\n            final_metadata[name] = metadata\n        final_metrics = pd.concat(final_rows, ignore_index=True)\n        final_metrics.to_csv(args.output_dir / "confirmation_metrics.csv", index=False)\n\n    summary = {\n        "protocol": {\n            "development_dates": DEFAULT_DEVELOPMENT_DATES,\n            "locked_test_date": DEFAULT_TEST_DATE,\n            "stress_dates": DEFAULT_STRESS_DATES,\n            "feature_history_rule": "same-route strictly earlier calendar dates",\n            "selection_rule": "development rolling-origin only; locked test and stress are report-only",\n            "full_classification_rule": "regression point prediction <= 0.5 seats",\n            "correlation_rule": "development-only Pearson and Spearman; report abs(correlation)>=0.85",\n            "model_rule": "fixed official HGB/ExtraTrees/LightGBM families, parameters and 0.4/0.4/0.2 weights",\n        },\n        "feature_sets": {\n            name: {"numeric": value.numeric, "categorical": value.categorical}\n            for name, value in selected_sets.items()\n        },\n        "preceding_bus_segment_audit": data.attrs.get("preceding_bus_segment_audit"),\n        "development_metrics": metrics.to_dict(orient="records"),\n        "paired_trip_bootstrap": bootstrap.to_dict(orient="records"),\n        "confirmation_metrics": final_metrics.to_dict(orient="records"),\n        "final_metadata": final_metadata,\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(metrics.to_string(index=False))\n    if not final_metrics.empty:\n        print("\\nconfirmation")\n        print(final_metrics.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/route_local_feature_importance.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.pipeline import Pipeline\n\nfrom all_prearrival_seat_regression import clip_seats, event_bias\nfrom hypothesis_model_search import (\n    candidate_weights,\n    decode_target,\n    encode_target,\n    event_summary,\n    feature_frames,\n    make_preprocessor,\n)\nfrom model_feasibility import FeatureSet\nfrom route_local_model import (\n    DEFAULT_CACHE_DIR,\n    DEFAULT_OUTPUT_DIR as DEFAULT_MODEL_RESULTS_DIR,\n    json_ready,\n    local_candidate,\n    route_cache_paths,\n)\n\n\nDEFAULT_OUTPUT_DIR = Path("analysis/route_local_feature_importance_results")\n\n# These groups are an exact semantic partition of the route-local model input.\n# One donor row is shared by every column in a group so related values such as\n# station ID/coordinates and sin/cos time stay coherent after permutation.\nFEATURE_GROUPS: dict[str, tuple[str, ...]] = {\n    "route_specific_structure": (\n        "station_seq_cat",\n        "route_progress",\n        "x",\n        "y",\n        "snapshot_station_seq_cat",\n        "snapshot_route_progress",\n        "direction",\n    ),\n    "current_seat_state": (\n        "snapshot_remaining_seats",\n        "target_load_ratio",\n        "currently_low_5",\n        "currently_low_10",\n    ),\n    "recent_seat_trajectory": (\n        "seat_delta_previous_stop",\n        "seat_change_per_stop",\n        "rolling_seat_change_per_stop_3",\n        "projected_arrival_seats",\n    ),\n    "distance_and_eta": (\n        "target_stop_gap",\n        "minutes_since_previous_stop",\n        "rolling_minutes_per_stop_3",\n        "estimated_minutes_to_arrival",\n        "seats_per_remaining_stop",\n        "load_gap_interaction",\n    ),\n    "calendar_time": (\n        "snapshot_time_sin",\n        "snapshot_time_cos",\n        "snapshot_day_of_week",\n        "snapshot_time_bin_30",\n    ),\n    "vehicle_and_observation_state": (\n        "snapshot_capacity",\n        "snapshot_low_plate_cat",\n        "target_state_cat",\n    ),\n}\n\n# Overlapping diagnostics split the route-specific block into the two concepts\n# motivating the experiment. Their importance is not additive.\nROUTE_DETAIL_GROUPS: dict[str, tuple[str, ...]] = {\n    "station_identity": (\n        "station_seq_cat",\n        "x",\n        "y",\n        "snapshot_station_seq_cat",\n    ),\n    "normalized_progress_and_direction": (\n        "route_progress",\n        "snapshot_route_progress",\n        "direction",\n    ),\n    "target_stop_identity": ("station_seq_cat", "x", "y"),\n    "snapshot_stop_identity": ("snapshot_station_seq_cat",),\n}\n\nABLATIONS: dict[str, tuple[str, ...]] = {\n    "route_specific_structure": FEATURE_GROUPS["route_specific_structure"],\n    "station_identity": ROUTE_DETAIL_GROUPS["station_identity"],\n    "normalized_progress_and_direction": ROUTE_DETAIL_GROUPS[\n        "normalized_progress_and_direction"\n    ],\n}\n\nCONDITIONAL_STRATA = (\n    "direction",\n    "snapshot_low_plate_cat",\n    "target_stop_gap",\n    "_current_seat_band",\n    "_peak_period",\n)\n\n\ndef validate_feature_partition(feature_columns: Iterable[str]) -> None:\n    expected = list(feature_columns)\n    flattened = [column for columns in FEATURE_GROUPS.values() for column in columns]\n    duplicates = sorted(\n        {column for column in flattened if flattened.count(column) > 1}\n    )\n    missing = sorted(set(expected) - set(flattened))\n    unexpected = sorted(set(flattened) - set(expected))\n    if duplicates or missing or unexpected:\n        raise ValueError(\n            "feature group이 route-local schema를 정확히 분할하지 않습니다: "\n            f"duplicates={duplicates}, missing={missing}, unexpected={unexpected}"\n        )\n\n\ndef add_conditional_strata(data: pd.DataFrame) -> pd.DataFrame:\n    output = data.copy()\n    output["_current_seat_band"] = pd.cut(\n        output["snapshot_remaining_seats"],\n        bins=[-np.inf, 5, 10, 20, np.inf],\n        labels=["0-5", "6-10", "11-20", "21+"],\n    ).astype(str)\n    hour = pd.to_datetime(output["snapshot_time"]).dt.hour\n    output["_peak_period"] = np.where(hour.lt(12), "am", "pm")\n    return output\n\n\ndef donor_indices(\n    data: pd.DataFrame,\n    *,\n    rng: np.random.Generator,\n    scheme: str,\n) -> np.ndarray:\n    if scheme == "global":\n        return rng.permutation(len(data))\n    if scheme != "conditional":\n        raise ValueError(f"알 수 없는 permutation scheme입니다: {scheme}")\n    missing = sorted(set(CONDITIONAL_STRATA) - set(data.columns))\n    if missing:\n        raise ValueError(f"conditional strata가 누락되었습니다: {missing}")\n    donors = np.arange(len(data), dtype=int)\n    groups = data.groupby(\n        list(CONDITIONAL_STRATA), sort=False, observed=True\n    ).indices\n    for indices in groups.values():\n        positions = np.asarray(indices, dtype=int)\n        if len(positions) > 1:\n            donors[positions] = rng.permutation(positions)\n    return donors\n\n\ndef permuted_frame(\n    data: pd.DataFrame,\n    columns: Iterable[str],\n    donors: np.ndarray,\n) -> pd.DataFrame:\n    output = data.copy()\n    for column in columns:\n        output[column] = data[column].iloc[donors].array\n    return output\n\n\ndef reduced_feature_set(\n    feature_set: FeatureSet,\n    removed_columns: Iterable[str],\n    *,\n    name: str,\n) -> FeatureSet:\n    removed = set(removed_columns)\n    missing = sorted(removed - set(feature_set.columns))\n    if missing:\n        raise ValueError(f"ablation 대상이 feature schema에 없습니다: {missing}")\n    return FeatureSet(\n        name=name,\n        numeric=tuple(\n            column for column in feature_set.numeric if column not in removed\n        ),\n        categorical=tuple(\n            column for column in feature_set.categorical if column not in removed\n        ),\n    )\n\n\ndef make_model(feature_set: FeatureSet, *, seed: int) -> Pipeline:\n    candidate = local_candidate()\n    return Pipeline(\n        [\n            ("features", clone(make_preprocessor(feature_set))),\n            (\n                "regressor",\n                HistGradientBoostingRegressor(\n                    random_state=seed,\n                    **candidate.params,\n                ),\n            ),\n        ]\n    )\n\n\ndef prior_oof_bias(history: list[pd.DataFrame]) -> float:\n    if not history:\n        return 0.0\n    previous = pd.concat(history, ignore_index=True)\n    return event_bias(previous, previous["raw_prediction"].to_numpy(dtype=float))\n\n\ndef scored_frame(\n    data: pd.DataFrame,\n    raw_prediction: np.ndarray,\n    prediction: np.ndarray,\n) -> pd.DataFrame:\n    output = data[\n        [\n            "event_id",\n            "date",\n            "trip_id",\n            "snapshot_time",\n            "label_seats",\n            "snapshot_remaining_seats",\n            "target_stop_gap",\n            "minutes_to_arrival",\n            "capacity",\n        ]\n    ].copy()\n    output["raw_prediction"] = raw_prediction\n    output["prediction"] = prediction\n    return output\n\n\ndef filter_to_oof_manifest(\n    validation: pd.DataFrame,\n    manifest: pd.DataFrame,\n    *,\n    validation_date: str,\n) -> pd.DataFrame:\n    expected = manifest.loc[\n        manifest["date"].astype(str).eq(validation_date),\n        ["event_id", "snapshot_time"],\n    ].copy()\n    if expected.empty:\n        raise ValueError(f"OOF manifest 날짜가 비어 있습니다: {validation_date}")\n    expected["_snapshot_ns"] = pd.to_datetime(\n        expected["snapshot_time"], utc=True\n    ).astype("int64")\n    expected_index = pd.MultiIndex.from_frame(\n        expected[["event_id", "_snapshot_ns"]]\n    )\n    if not expected_index.is_unique:\n        raise ValueError(f"OOF manifest key가 중복됩니다: {validation_date}")\n\n    candidates = validation.copy()\n    candidates["_snapshot_ns"] = pd.to_datetime(\n        candidates["snapshot_time"], utc=True\n    ).astype("int64")\n    candidate_index = pd.MultiIndex.from_frame(\n        candidates[["event_id", "_snapshot_ns"]]\n    )\n    keep = candidate_index.isin(expected_index)\n    filtered = candidates.loc[keep].drop(columns="_snapshot_ns")\n    filtered_index = candidate_index[keep]\n    if len(filtered) != len(expected) or not expected_index.isin(filtered_index).all():\n        raise ValueError(\n            f"snapshot cache가 원 OOF 행을 재현하지 못합니다: date={validation_date}, "\n            f"expected={len(expected)}, found={len(filtered)}"\n        )\n    return filtered\n\n\ndef metric_deltas(\n    data: pd.DataFrame,\n    baseline_prediction: np.ndarray,\n    candidate_prediction: np.ndarray,\n) -> dict[str, float]:\n    baseline = event_summary(data, baseline_prediction)\n    candidate = event_summary(data, candidate_prediction)\n    result = {\n        "overall_mae_increase": float(\n            candidate["event_balanced_mae"] - baseline["event_balanced_mae"]\n        ),\n        "within_3_decrease": float(\n            baseline["event_balanced_within_3"]\n            - candidate["event_balanced_within_3"]\n        ),\n        "prediction_mean_abs_change": float(\n            np.mean(np.abs(candidate_prediction - baseline_prediction))\n        ),\n    }\n    for output_name, metric_name in (\n        ("low_0_5_mae_increase", "low_0_5_mae"),\n        ("far_6_plus_mae_increase", "far_6_plus_stop_mae"),\n    ):\n        left = baseline[metric_name]\n        right = candidate[metric_name]\n        result[output_name] = (\n            float(right - left)\n            if np.isfinite(left) and np.isfinite(right)\n            else np.nan\n        )\n    return result\n\n\ndef summarize_permutations(raw: pd.DataFrame) -> pd.DataFrame:\n    metric_columns = [\n        "overall_mae_increase",\n        "within_3_decrease",\n        "prediction_mean_abs_change",\n        "low_0_5_mae_increase",\n        "far_6_plus_mae_increase",\n        "donor_change_rate",\n    ]\n    rows: list[dict[str, Any]] = []\n    keys = ["route_id", "route_name", "level", "scheme", "feature", "columns"]\n    for values, group in raw.groupby(keys, sort=False):\n        row = dict(zip(keys, values))\n        row["folds"] = int(group["validation_date"].nunique())\n        row["repeats"] = int(group["repeat"].nunique())\n        per_date = group.groupby("validation_date")["overall_mae_increase"].mean()\n        row["positive_date_fraction"] = float(per_date.gt(0).mean())\n        for metric in metric_columns:\n            series = group[metric].dropna().to_numpy(dtype=float)\n            if len(series) == 0:\n                row[f"{metric}_mean"] = np.nan\n                row[f"{metric}_p05"] = np.nan\n                row[f"{metric}_p95"] = np.nan\n            else:\n                row[f"{metric}_mean"] = float(np.mean(series))\n                row[f"{metric}_p05"] = float(np.quantile(series, 0.05))\n                row[f"{metric}_p95"] = float(np.quantile(series, 0.95))\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values(\n        ["route_name", "level", "scheme", "overall_mae_increase_mean"],\n        ascending=[True, True, True, False],\n    )\n\n\ndef summarize_ablations(raw: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    keys = ["route_id", "route_name", "ablation", "removed_columns"]\n    for values, group in raw.groupby(keys, sort=False):\n        row = dict(zip(keys, values))\n        row["folds"] = int(group["validation_date"].nunique())\n        row["baseline_mean_daily_mae"] = float(\n            group["baseline_mae"].mean()\n        )\n        row["ablated_mean_daily_mae"] = float(group["ablated_mae"].mean())\n        delta = group["mae_increase"].to_numpy(dtype=float)\n        row["mae_increase_mean"] = float(np.mean(delta))\n        row["mae_increase_min"] = float(np.min(delta))\n        row["mae_increase_max"] = float(np.max(delta))\n        row["positive_date_fraction"] = float(np.mean(delta > 0))\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values(\n        ["route_name", "mae_increase_mean"], ascending=[True, False]\n    )\n\n\ndef analyze_route(\n    route_id: str,\n    route_name: str,\n    dates: list[str],\n    snapshots: pd.DataFrame,\n    oof_manifest: pd.DataFrame,\n    *,\n    repeats: int,\n    seed: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\n    candidate = local_candidate()\n    data = snapshots.loc[snapshots["date"].astype(str).isin(dates)].copy()\n    _, _, feature_set = feature_frames(\n        data.iloc[:0], data.iloc[:0], candidate.feature_variant\n    )\n    validate_feature_partition(feature_set.columns)\n\n    groups = {**FEATURE_GROUPS, **ROUTE_DETAIL_GROUPS}\n    group_levels = {\n        **{name: "semantic_group" for name in FEATURE_GROUPS},\n        **{name: "route_detail" for name in ROUTE_DETAIL_GROUPS},\n    }\n    baseline_history: list[pd.DataFrame] = []\n    ablation_history: dict[str, list[pd.DataFrame]] = defaultdict(list)\n    permutation_rows: list[dict[str, Any]] = []\n    ablation_rows: list[dict[str, Any]] = []\n    baseline_oof: list[pd.DataFrame] = []\n\n    for fold_number, validation_date in enumerate(dates[1:], start=1):\n        train = data.loc[data["date"].astype(str).isin(dates[:fold_number])].copy()\n        validation = data.loc[\n            data["date"].astype(str).eq(validation_date)\n        ].copy()\n        validation = filter_to_oof_manifest(\n            validation,\n            oof_manifest,\n            validation_date=validation_date,\n        )\n        prepared_train, prepared_validation, prepared_feature_set = feature_frames(\n            train, validation, candidate.feature_variant\n        )\n        # route_local_model.run_candidate numbers folds from zero. Keep the\n        # identical seed so this analysis reconstructs the original OOF model.\n        model = make_model(prepared_feature_set, seed=seed + fold_number - 1)\n        model.fit(\n            prepared_train[prepared_feature_set.columns],\n            encode_target(prepared_train, candidate.target_kind),\n            regressor__sample_weight=candidate_weights(\n                prepared_train,\n                candidate.low_weight,\n                weighting_kind=candidate.weighting_kind,\n            ),\n        )\n        raw = decode_target(\n            model.predict(prepared_validation[prepared_feature_set.columns]),\n            prepared_validation,\n            candidate.target_kind,\n        )\n        bias = prior_oof_bias(baseline_history)\n        baseline_prediction = clip_seats(raw + bias, validation["capacity"])\n        baseline_scored = scored_frame(validation, raw, baseline_prediction)\n        baseline_history.append(baseline_scored)\n        baseline_oof.append(baseline_scored)\n\n        permutation_data = add_conditional_strata(prepared_validation)\n        for scheme_index, scheme in enumerate(("global", "conditional")):\n            for group_index, (name, columns) in enumerate(groups.items()):\n                for repeat_index in range(repeats):\n                    rng = np.random.default_rng(\n                        seed\n                        + fold_number * 10_000_000\n                        + scheme_index * 1_000_000\n                        + group_index * 10_000\n                        + repeat_index\n                    )\n                    donors = donor_indices(\n                        permutation_data, rng=rng, scheme=scheme\n                    )\n                    shuffled = permuted_frame(\n                        permutation_data, columns, donors\n                    )\n                    permuted_raw = decode_target(\n                        model.predict(shuffled[prepared_feature_set.columns]),\n                        prepared_validation,\n                        candidate.target_kind,\n                    )\n                    prediction = clip_seats(\n                        permuted_raw + bias, validation["capacity"]\n                    )\n                    permutation_rows.append(\n                        {\n                            "route_id": route_id,\n                            "route_name": route_name,\n                            "validation_date": validation_date,\n                            "level": group_levels[name],\n                            "scheme": scheme,\n                            "feature": name,\n                            "columns": json.dumps(\n                                list(columns), ensure_ascii=False\n                            ),\n                            "repeat": repeat_index,\n                            "donor_change_rate": float(\n                                np.mean(donors != np.arange(len(donors)))\n                            ),\n                            **metric_deltas(\n                                validation,\n                                baseline_prediction,\n                                prediction,\n                            ),\n                        }\n                    )\n\n        for ablation_index, (name, removed_columns) in enumerate(ABLATIONS.items()):\n            reduced = reduced_feature_set(\n                prepared_feature_set,\n                removed_columns,\n                name=f"route_local_without_{name}",\n            )\n            ablated_model = make_model(\n                reduced,\n                seed=seed + 100_000 + ablation_index * 1_000 + fold_number,\n            )\n            ablated_model.fit(\n                prepared_train[reduced.columns],\n                encode_target(prepared_train, candidate.target_kind),\n                regressor__sample_weight=candidate_weights(\n                    prepared_train,\n                    candidate.low_weight,\n                    weighting_kind=candidate.weighting_kind,\n                ),\n            )\n            ablated_raw = decode_target(\n                ablated_model.predict(prepared_validation[reduced.columns]),\n                prepared_validation,\n                candidate.target_kind,\n            )\n            ablated_bias = prior_oof_bias(ablation_history[name])\n            ablated_prediction = clip_seats(\n                ablated_raw + ablated_bias, validation["capacity"]\n            )\n            ablated_scored = scored_frame(\n                validation, ablated_raw, ablated_prediction\n            )\n            ablation_history[name].append(ablated_scored)\n            baseline_metrics = event_summary(validation, baseline_prediction)\n            ablated_metrics = event_summary(validation, ablated_prediction)\n            ablation_rows.append(\n                {\n                    "route_id": route_id,\n                    "route_name": route_name,\n                    "validation_date": validation_date,\n                    "ablation": name,\n                    "removed_columns": json.dumps(\n                        list(removed_columns), ensure_ascii=False\n                    ),\n                    "baseline_mae": baseline_metrics["event_balanced_mae"],\n                    "ablated_mae": ablated_metrics["event_balanced_mae"],\n                    "mae_increase": (\n                        ablated_metrics["event_balanced_mae"]\n                        - baseline_metrics["event_balanced_mae"]\n                    ),\n                    "baseline_low_0_5_mae": baseline_metrics["low_0_5_mae"],\n                    "ablated_low_0_5_mae": ablated_metrics["low_0_5_mae"],\n                }\n            )\n\n    baseline_all = pd.concat(baseline_oof, ignore_index=True)\n    baseline_metrics = event_summary(\n        baseline_all, baseline_all["prediction"].to_numpy(dtype=float)\n    )\n    return (\n        pd.DataFrame(permutation_rows),\n        pd.DataFrame(ablation_rows),\n        {\n            "route_id": route_id,\n            "route_name": route_name,\n            "dates": dates,\n            "validation_dates": dates[1:],\n            "rows": int(len(baseline_all)),\n            "events": int(baseline_all["event_id"].nunique()),\n            "baseline_metrics": baseline_metrics,\n        },\n    )\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="노선별 HGB의 OOF grouped permutation importance와 재학습 ablation"\n    )\n    parser.add_argument(\n        "--model-results-dir", type=Path, default=DEFAULT_MODEL_RESULTS_DIR\n    )\n    parser.add_argument("--cache-dir", type=Path, default=DEFAULT_CACHE_DIR)\n    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)\n    parser.add_argument("--route-ids", nargs="*")\n    parser.add_argument("--repeats", type=int, default=8)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.repeats <= 0:\n        raise ValueError("repeats는 1 이상이어야 합니다.")\n\n    model_summary = json.loads(\n        (args.model_results_dir / "summary.json").read_text(encoding="utf-8")\n    )\n    oof_manifest_path = args.model_results_dir / "oof_predictions.csv.gz"\n    oof_manifest = pd.read_csv(\n        oof_manifest_path,\n        usecols=[\n            "route_id",\n            "date",\n            "event_id",\n            "snapshot_time",\n            "candidate",\n        ],\n    )\n    oof_manifest = oof_manifest.loc[\n        oof_manifest["candidate"].eq("route_local_hgb")\n    ].copy()\n    selected_routes = set(args.route_ids or model_summary["routes_completed"])\n    route_results = [\n        row\n        for row in model_summary["route_results"]\n        if str(row["route_id"]) in selected_routes\n    ]\n    if not route_results:\n        raise ValueError("분석할 완료 노선이 없습니다.")\n\n    raw_permutations: list[pd.DataFrame] = []\n    raw_ablations: list[pd.DataFrame] = []\n    routes: list[dict[str, Any]] = []\n    for row in route_results:\n        route_id = str(row["route_id"])\n        route_name = str(row["route_name"])\n        cache_path, _ = route_cache_paths(args.cache_dir, route_id)\n        if not cache_path.exists():\n            raise FileNotFoundError(f"노선 snapshot cache가 없습니다: {cache_path}")\n        print(f"[{route_name}/{route_id}] OOF importance", flush=True)\n        snapshots = pd.read_pickle(cache_path)\n        permutation, ablation, route_summary = analyze_route(\n            route_id,\n            route_name,\n            [str(value) for value in row["training_dates"]],\n            snapshots,\n            oof_manifest.loc[oof_manifest["route_id"].astype(str).eq(route_id)],\n            repeats=args.repeats,\n            seed=args.seed,\n        )\n        raw_permutations.append(permutation)\n        raw_ablations.append(ablation)\n        routes.append(route_summary)\n\n    permutation_raw = pd.concat(raw_permutations, ignore_index=True)\n    ablation_raw = pd.concat(raw_ablations, ignore_index=True)\n    permutation_summary = summarize_permutations(permutation_raw)\n    ablation_summary = summarize_ablations(ablation_raw)\n\n    original_daily = pd.read_csv(args.model_results_dir / "daily_metrics.csv")\n    original_daily = original_daily.loc[\n        original_daily["candidate"].eq("route_local_hgb")\n    ].copy()\n    reconstruction_differences: list[float] = []\n    for row in ablation_summary.loc[\n        ablation_summary["ablation"].eq("route_specific_structure")\n    ].itertuples(index=False):\n        expected = original_daily.loc[\n            original_daily["route_id"].astype(str).eq(str(row.route_id)),\n            "event_balanced_mae",\n        ].mean()\n        reconstruction_differences.append(\n            abs(float(row.baseline_mean_daily_mae) - float(expected))\n        )\n    reconstruction_max_abs_diff = max(reconstruction_differences, default=0.0)\n    if reconstruction_max_abs_diff > 1e-12:\n        raise ValueError(\n            "importance baseline이 원 route-local OOF를 재현하지 못합니다: "\n            f"max_abs_diff={reconstruction_max_abs_diff}"\n        )\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    permutation_raw.to_csv(\n        args.output_dir / "permutation_importance_repeats.csv", index=False\n    )\n    permutation_summary.to_csv(\n        args.output_dir / "permutation_importance_summary.csv", index=False\n    )\n    ablation_raw.to_csv(args.output_dir / "ablation_daily_metrics.csv", index=False)\n    ablation_summary.to_csv(args.output_dir / "ablation_summary.csv", index=False)\n\n    route_structure = permutation_summary.loc[\n        permutation_summary["feature"].eq("route_specific_structure")\n    ].copy()\n    route_ablation = ablation_summary.loc[\n        ablation_summary["ablation"].eq("route_specific_structure")\n    ].copy()\n    payload = {\n        "purpose": (\n            "Test whether each route-local model relies on and benefits from "\n            "route/station-specific structure rather than only current seats."\n        ),\n        "interpretation": {\n            "permutation_positive": (\n                "Shuffling a frozen model input worsens OOF MAE, indicating model reliance."\n            ),\n            "ablation_positive": (\n                "Retraining without the group worsens future-date OOF MAE, indicating "\n                "generalizable contribution rather than frozen-model reliance alone."\n            ),\n            "global": "Total dependence, including correlations and interactions.",\n            "conditional": (\n                "Dependence within direction, vehicle type, exact stop gap, current-seat "\n                "band, and AM/PM context."\n            ),\n            "route_specific_structure": list(\n                FEATURE_GROUPS["route_specific_structure"]\n            ),\n            "caveat": (\n                "Permutation quantiles describe random shuffles, not sampling confidence; "\n                "routes with only two validation dates remain preliminary."\n            ),\n        },\n        "protocol": {\n            "validation": (\n                "rolling origin; each fold model is trained only on earlier route-local dates"\n            ),\n            "permutation_repeats": args.repeats,\n            "permutation_schemes": ["global", "conditional"],\n            "conditional_strata": list(CONDITIONAL_STRATA),\n            "shared_donor_within_group": True,\n            "scope": (\n                "model-input-only; delta decoding keeps original current seats and stop gap"\n            ),\n            "ablation": "remove group, retrain the same HGB, and score the same future folds",\n            "evaluation_row_manifest": str(oof_manifest_path),\n            "baseline_mean_daily_mae_reconstruction_max_abs_diff": (\n                reconstruction_max_abs_diff\n            ),\n        },\n        "routes": routes,\n        "route_structure_permutation": route_structure.to_dict("records"),\n        "route_structure_ablation": route_ablation.to_dict("records"),\n        "artifacts": {\n            "permutation_raw": str(\n                args.output_dir / "permutation_importance_repeats.csv"\n            ),\n            "permutation_summary": str(\n                args.output_dir / "permutation_importance_summary.csv"\n            ),\n            "ablation_daily": str(args.output_dir / "ablation_daily_metrics.csv"),\n            "ablation_summary": str(args.output_dir / "ablation_summary.csv"),\n        },\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(payload), ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print(\n        route_ablation[\n            [\n                "route_name",\n                "folds",\n                "baseline_mean_daily_mae",\n                "ablated_mean_daily_mae",\n                "mae_increase_mean",\n                "positive_date_fraction",\n            ]\n        ].to_string(index=False),\n        flush=True,\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/plot_arrival_seat_distributions.py': 'from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\n\nTURNAROUND_SEQ = 43\n\n\ndef unique_for_group(data: pd.DataFrame, group: str) -> pd.DataFrame:\n    """Count each arrival event at most once within a displayed group."""\n\n    return data.drop_duplicates(["event_id", group])\n\n\ndef draw_boxplot(\n    ax: plt.Axes,\n    data: pd.DataFrame,\n    group: str,\n    *,\n    title: str,\n    xlabel: str,\n    tick_step: int,\n) -> None:\n    frame = unique_for_group(data, group).copy()\n    frame[group] = pd.to_numeric(frame[group])\n    groups = sorted(frame[group].dropna().astype(int).unique())\n    values = [\n        frame.loc[frame[group].eq(value), "label_seats"].to_numpy()\n        for value in groups\n    ]\n    ax.boxplot(\n        values,\n        positions=groups,\n        widths=0.65,\n        showfliers=False,\n        patch_artist=True,\n        medianprops={"color": "black", "linewidth": 1.0},\n        boxprops={"facecolor": "#72A0C1", "alpha": 0.7},\n        whiskerprops={"color": "#657786", "linewidth": 0.8},\n        capprops={"color": "#657786", "linewidth": 0.8},\n    )\n    ax.axhline(5, color="#C44E52", linestyle="--", linewidth=1.2, label="저잔여 기준(5석)")\n    minimum, maximum = min(groups), max(groups)\n    ticks = np.arange(\n        int(np.ceil(minimum / tick_step) * tick_step), maximum + 1, tick_step\n    )\n    ax.set_xticks(ticks, labels=[str(value) for value in ticks])\n    ax.set_xlim(minimum - 1, maximum + 1)\n    ax.set_ylim(-2, 72)\n    ax.set_title(title)\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("도착 잔여좌석(석)")\n    ax.grid(axis="y", alpha=0.25)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="도착 잔여좌석의 피처별 단순 분포 그래프")\n    parser.add_argument(\n        "--cache",\n        type=Path,\n        default=Path("data/analysis_cache/all_prearrival_A.pkl"),\n    )\n    parser.add_argument(\n        "--output",\n        type=Path,\n        default=Path("analysis/feature_distribution_results/arrival_seat_distributions.png"),\n    )\n    args = parser.parse_args()\n\n    data = pd.read_pickle(args.cache)\n    required = {\n        "event_id",\n        "date",\n        "label_seats",\n        "station_seq_cat",\n        "direction",\n        "snapshot_station_seq_cat",\n        "snapshot_time_bin_30",\n    }\n    missing = sorted(required - set(data.columns))\n    if missing:\n        raise ValueError(f"필수 열이 없습니다: {missing}")\n\n    plt.rcParams["font.family"] = "Noto Sans CJK KR"\n    plt.rcParams["axes.unicode_minus"] = False\n    figure, axes = plt.subplots(2, 2, figsize=(18, 11), constrained_layout=True)\n\n    draw_boxplot(\n        axes[0, 0],\n        data,\n        "station_seq_cat",\n        title="목표 정류장 위치별 도착 잔여좌석",\n        xlabel="목표 정류장 순번",\n        tick_step=5,\n    )\n    axes[0, 0].axvline(\n        TURNAROUND_SEQ + 0.5,\n        color="#888888",\n        linestyle=":",\n        linewidth=1,\n        label="회차점",\n    )\n    axes[0, 0].legend(loc="lower left", fontsize=9)\n\n    event_level = data.drop_duplicates("event_id")\n    direction_labels = {"to_city": "도시 방향", "return": "회차 후 복귀"}\n    colors = {"to_city": "#4C72B0", "return": "#DD8452"}\n    bins = np.arange(-0.5, 71.5, 2)\n    for direction in ("to_city", "return"):\n        subset = event_level.loc[event_level["direction"].eq(direction), "label_seats"]\n        axes[0, 1].hist(\n            subset,\n            bins=bins,\n            density=True,\n            alpha=0.55,\n            color=colors[direction],\n            label=f"{direction_labels[direction]} (n={len(subset):,})",\n        )\n    axes[0, 1].axvline(5, color="#C44E52", linestyle="--", linewidth=1.2)\n    axes[0, 1].set_xlim(-2, 72)\n    axes[0, 1].set_title("운행 방향별 도착 잔여좌석")\n    axes[0, 1].set_xlabel("도착 잔여좌석(석)")\n    axes[0, 1].set_ylabel("확률밀도")\n    axes[0, 1].grid(axis="y", alpha=0.25)\n    axes[0, 1].legend(fontsize=9)\n\n    draw_boxplot(\n        axes[1, 0],\n        data,\n        "snapshot_station_seq_cat",\n        title="현재 정류장 위치별 최종 도착 잔여좌석",\n        xlabel="현재 정류장 순번",\n        tick_step=5,\n    )\n    axes[1, 0].axvline(\n        TURNAROUND_SEQ + 0.5,\n        color="#888888",\n        linestyle=":",\n        linewidth=1,\n        label="회차점",\n    )\n    axes[1, 0].legend(loc="lower left", fontsize=9)\n\n    draw_boxplot(\n        axes[1, 1],\n        data,\n        "snapshot_time_bin_30",\n        title="관측 시간대별 최종 도착 잔여좌석",\n        xlabel="관측 시각",\n        tick_step=2,\n    )\n    time_ticks = axes[1, 1].get_xticks().astype(int)\n    axes[1, 1].set_xticklabels([f"{value // 2:02d}:00" for value in time_ticks])\n    axes[1, 1].legend(loc="lower left", fontsize=9)\n\n    first_date = str(data["date"].astype(str).min())\n    last_date = str(data["date"].astype(str).max())\n    figure.suptitle(\n        f"도착 전 잔여좌석 분포 ({first_date} ~ {last_date})",\n        fontsize=17,\n        fontweight="bold",\n    )\n    figure.text(\n        0.5,\n        0.005,\n        "박스=25~75%, 중앙선=중앙값, 수염=1.5×IQR · 목표/방향은 사건당 1회, 현재 위치/시간은 사건×그룹당 1회",\n        ha="center",\n        fontsize=10,\n        color="#555555",\n    )\n\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    figure.savefig(args.output, dpi=180, bbox_inches="tight")\n    plt.close(figure)\n    print(args.output)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'analysis/tminus_seat_regression.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.dummy import DummyRegressor\nfrom sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.pipeline import Pipeline\n\nfrom high_risk_feasibility import high_risk_mask\nfrom model_feasibility import (\n    PLANNING,\n    FeatureSet,\n    _as_records,\n    build_model_table,\n    build_visits,\n    json_ready,\n    load_data,\n    make_preprocessor,\n    prepare_subset,\n)\nfrom tminus_feasibility import (\n    DEFAULT_HORIZONS,\n    FIXED_3STOP,\n    ROUTE_ID,\n    ROUTE_NAME,\n    build_tminus_table,\n    prepare_raw_locations,\n    tminus_feature_sets,\n)\n\n\ndef regression_factories(feature_set: FeatureSet, seed: int) -> dict[str, Pipeline]:\n    preprocessor = make_preprocessor(feature_set)\n    return {\n        "ridge": Pipeline(\n            [\n                ("features", clone(preprocessor)),\n                ("regressor", Ridge(alpha=10.0)),\n            ]\n        ),\n        "random_forest": Pipeline(\n            [\n                ("features", clone(preprocessor)),\n                (\n                    "regressor",\n                    RandomForestRegressor(\n                        n_estimators=400,\n                        max_depth=10,\n                        min_samples_leaf=6,\n                        max_features="sqrt",\n                        n_jobs=-1,\n                        random_state=seed,\n                    ),\n                ),\n            ]\n        ),\n        "hist_gradient_boosting": Pipeline(\n            [\n                ("features", clone(preprocessor)),\n                (\n                    "regressor",\n                    HistGradientBoostingRegressor(\n                        loss="absolute_error",\n                        learning_rate=0.05,\n                        max_iter=300,\n                        max_leaf_nodes=15,\n                        min_samples_leaf=20,\n                        l2_regularization=1.0,\n                        random_state=seed,\n                    ),\n                ),\n            ]\n        ),\n    }\n\n\ndef clip_seats(predictions: np.ndarray, capacity: pd.Series) -> np.ndarray:\n    return np.minimum(np.maximum(predictions, 0), capacity.to_numpy(dtype=float))\n\n\ndef bias_correct(\n    calibration_predictions: np.ndarray,\n    calibration_true: np.ndarray,\n    target_predictions: np.ndarray,\n) -> tuple[np.ndarray, float]:\n    bias = float(np.median(calibration_true - calibration_predictions))\n    return target_predictions + bias, bias\n\n\ndef regression_metrics(\n    data: pd.DataFrame,\n    predictions: np.ndarray,\n    *,\n    horizon: int,\n    feature_set: str,\n    family: str,\n    model: str,\n    split: str,\n    calibration_mae: float,\n    bias_correction: float,\n) -> dict[str, Any]:\n    true = data["label_seats"].to_numpy(dtype=float)\n    predicted = clip_seats(predictions, data["capacity"])\n    errors = np.abs(true - predicted)\n    low = true <= 5\n    full = true == 0\n    predicted_full = predicted <= 1\n    true_full_count = int(full.sum())\n    return {\n        "horizon_minutes": horizon,\n        "split": split,\n        "feature_set": feature_set,\n        "family": family,\n        "model": model,\n        "rows": int(len(true)),\n        "true_mean_seats": float(true.mean()),\n        "predicted_mean_seats": float(predicted.mean()),\n        "mae_seats": float(mean_absolute_error(true, predicted)),\n        "rmse_seats": float(math.sqrt(mean_squared_error(true, predicted))),\n        "median_absolute_error_seats": float(np.median(errors)),\n        "r2": float(r2_score(true, predicted)),\n        "within_1_seat": float((errors <= 1).mean()),\n        "within_3_seats": float((errors <= 3).mean()),\n        "within_5_seats": float((errors <= 5).mean()),\n        "low_0_5_rows": int(low.sum()),\n        "low_0_5_mae_seats": float(errors[low].mean()) if low.any() else np.nan,\n        "low_0_5_within_3_seats": float((errors[low] <= 3).mean()) if low.any() else np.nan,\n        "full_rows": true_full_count,\n        "full_predicted_mean_seats": float(predicted[full].mean()) if full.any() else np.nan,\n        "derived_full_precision": (\n            float(full[predicted_full].mean()) if predicted_full.any() else 0.0\n        ),\n        "derived_full_recall": (\n            float(predicted_full[full].mean()) if true_full_count else np.nan\n        ),\n        "alert_at_most_5_rate": float((predicted <= 5).mean()),\n        "selection_calibration_mae": float(calibration_mae),\n        "bias_correction_seats": float(bias_correction),\n    }\n\n\ndef split_data(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    train = data.loc[data["date"].isin(["2026-08-04", "2026-08-05"])]\n    calibration = data.loc[data["date"].eq("2026-08-06")]\n    test = data.loc[data["date"].eq("2026-08-07")]\n    if min(len(train), len(calibration), len(test)) == 0:\n        raise ValueError("회귀 시간 분할 중 하나가 비어 있습니다.")\n    return train, calibration, test\n\n\ndef evaluate_learned_models(\n    data: pd.DataFrame,\n    feature_set: FeatureSet,\n    family: str,\n    horizon: int,\n    seed: int,\n) -> tuple[list[dict[str, Any]], dict[str, np.ndarray]]:\n    train, calibration, test = split_data(data)\n    y_train = train["label_seats"].to_numpy(dtype=float)\n    y_cal = calibration["label_seats"].to_numpy(dtype=float)\n    rows: list[dict[str, Any]] = []\n    predictions: dict[str, np.ndarray] = {}\n    for model_name, model in regression_factories(feature_set, seed).items():\n        model.fit(train[feature_set.columns], y_train)\n        raw_cal = model.predict(calibration[feature_set.columns])\n        raw_test = model.predict(test[feature_set.columns])\n        calibrated_test, bias = bias_correct(raw_cal, y_cal, raw_test)\n        calibrated_cal = raw_cal + bias\n        calibrated_cal = clip_seats(calibrated_cal, calibration["capacity"])\n        calibration_mae = mean_absolute_error(y_cal, calibrated_cal)\n        rows.append(\n            regression_metrics(\n                test,\n                calibrated_test,\n                horizon=horizon,\n                feature_set=feature_set.name,\n                family=family,\n                model=model_name,\n                split="test_2026-08-07",\n                calibration_mae=calibration_mae,\n                bias_correction=bias,\n            )\n        )\n        predictions[model_name] = clip_seats(calibrated_test, test["capacity"])\n    return rows, predictions\n\n\ndef evaluate_delta_models(\n    data: pd.DataFrame,\n    feature_set: FeatureSet,\n    family: str,\n    horizon: int,\n    seed: int,\n    source_column: str,\n) -> tuple[list[dict[str, Any]], dict[str, np.ndarray]]:\n    """현재 좌석에서 도착 시점까지의 변화량을 학습해 도착 좌석으로 복원한다."""\n    train, calibration, test = split_data(data)\n    y_train = (\n        train["label_seats"].to_numpy(dtype=float)\n        - train[source_column].to_numpy(dtype=float)\n    )\n    y_cal = calibration["label_seats"].to_numpy(dtype=float)\n    rows: list[dict[str, Any]] = []\n    predictions: dict[str, np.ndarray] = {}\n    for model_name, model in regression_factories(feature_set, seed).items():\n        model.fit(train[feature_set.columns], y_train)\n        raw_cal = (\n            calibration[source_column].to_numpy(dtype=float)\n            + model.predict(calibration[feature_set.columns])\n        )\n        raw_test = (\n            test[source_column].to_numpy(dtype=float)\n            + model.predict(test[feature_set.columns])\n        )\n        calibrated_test, bias = bias_correct(raw_cal, y_cal, raw_test)\n        calibrated_cal = clip_seats(raw_cal + bias, calibration["capacity"])\n        calibration_mae = mean_absolute_error(y_cal, calibrated_cal)\n        output_name = f"delta_{model_name}"\n        rows.append(\n            regression_metrics(\n                test,\n                calibrated_test,\n                horizon=horizon,\n                feature_set=feature_set.name,\n                family=family,\n                model=output_name,\n                split="test_2026-08-07",\n                calibration_mae=calibration_mae,\n                bias_correction=bias,\n            )\n        )\n        predictions[output_name] = clip_seats(\n            calibrated_test, test["capacity"]\n        )\n    return rows, predictions\n\n\ndef evaluate_naive_baseline(\n    data: pd.DataFrame,\n    *,\n    horizon: int,\n    feature_set: str,\n    family: str,\n    model: str,\n    source_column: str | None,\n) -> tuple[dict[str, Any], np.ndarray]:\n    train, calibration, test = split_data(data)\n    if source_column is None:\n        regressor = DummyRegressor(strategy="median")\n        regressor.fit(np.zeros((len(train), 1)), train["label_seats"].to_numpy(dtype=float))\n        raw_cal = regressor.predict(np.zeros((len(calibration), 1)))\n        raw_test = regressor.predict(np.zeros((len(test), 1)))\n    else:\n        raw_cal = calibration[source_column].to_numpy(dtype=float)\n        raw_test = test[source_column].to_numpy(dtype=float)\n    valid_cal = np.isfinite(raw_cal)\n    valid_test = np.isfinite(raw_test)\n    calibration_subset = calibration.loc[valid_cal]\n    test_subset = test.loc[valid_test]\n    calibrated_test, bias = bias_correct(\n        raw_cal[valid_cal],\n        calibration_subset["label_seats"].to_numpy(dtype=float),\n        raw_test[valid_test],\n    )\n    calibrated_cal = clip_seats(raw_cal[valid_cal] + bias, calibration_subset["capacity"])\n    calibration_mae = mean_absolute_error(\n        calibration_subset["label_seats"].to_numpy(dtype=float), calibrated_cal\n    )\n    predictions = clip_seats(calibrated_test, test_subset["capacity"])\n    row = regression_metrics(\n        test_subset,\n        predictions,\n        horizon=horizon,\n        feature_set=feature_set,\n        family=family,\n        model=model,\n        split="test_2026-08-07",\n        calibration_mae=calibration_mae,\n        bias_correction=bias,\n    )\n    return row, predictions\n\n\ndef selected_rows(metrics: pd.DataFrame) -> pd.DataFrame:\n    candidates = metrics.loc[~metrics["model"].str.contains("baseline|persistence")].copy()\n    return (\n        candidates.sort_values(\n            ["horizon_minutes", "family", "selection_calibration_mae", "mae_seats"]\n        )\n        .groupby(["horizon_minutes", "family"], sort=False)\n        .head(1)\n        .reset_index(drop=True)\n    )\n\n\ndef make_plot(selected: pd.DataFrame, output: Path) -> None:\n    visible = selected.loc[selected["family"].isin(["planning", "target", "route_context"])]\n    colors = {"planning": "#94a3b8", "target": "#2563eb", "route_context": "#f97316"}\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))\n    for family, group in visible.groupby("family", sort=False):\n        group = group.sort_values("horizon_minutes")\n        axes[0].plot(\n            group["horizon_minutes"], group["mae_seats"], marker="o",\n            label=family.replace("_", " "), color=colors[family],\n        )\n        axes[1].plot(\n            group["horizon_minutes"], group["within_3_seats"] * 100, marker="o",\n            label=family.replace("_", " "), color=colors[family],\n        )\n        axes[2].plot(\n            group["horizon_minutes"], group["low_0_5_mae_seats"], marker="o",\n            label=family.replace("_", " "), color=colors[family],\n        )\n    axes[0].set_title("Arrival-seat MAE")\n    axes[0].set_ylabel("Seats")\n    axes[1].set_title("Predictions within ±3 seats")\n    axes[1].set_ylabel("Percent")\n    axes[2].set_title("MAE when actual seats are 0–5")\n    axes[2].set_ylabel("Seats")\n    for axis in axes:\n        axis.set_xlabel("Minutes before arrival")\n        axis.set_xticks(sorted(visible["horizon_minutes"].unique()))\n        axis.grid(alpha=0.2)\n    axes[0].legend()\n    fig.tight_layout()\n    fig.savefig(output, dpi=160)\n    plt.close(fig)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="1000번 버스 도착 시 잔여좌석 회귀 실험")\n    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("analysis/arrival_seat_regression_results"),\n    )\n    parser.add_argument(\n        "--horizons", type=int, nargs="+", default=list(DEFAULT_HORIZONS)\n    )\n    parser.add_argument(\n        "--scope",\n        choices=("peak", "high-risk"),\n        default="peak",\n        help="출퇴근 전체(기본값) 또는 기존 만차 고위험 게이트만 평가",\n    )\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    locations, stations = load_data(args.db, ROUTE_ID)\n    visits = build_visits(locations, stations)\n    model_table, turnaround_seq = build_model_table(\n        visits, stations, label_target="arrival"\n    )\n    source = prepare_subset(model_table)\n    source = source.loc[source["label_quality"].isin(["A", "B"])].copy()\n    if args.scope == "high-risk":\n        source = source.loc[high_risk_mask(source)].copy()\n    source = source.reset_index(drop=True)\n    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)\n\n    all_rows: list[dict[str, Any]] = []\n    coverage_rows: list[dict[str, Any]] = []\n    for horizon in sorted(set(args.horizons)):\n        snapshot = build_tminus_table(source, visits, by_vehicle, horizon)\n        target_features, route_features = tminus_feature_sets(horizon)\n        snapshot["snapshot_remaining_seats"] = (\n            snapshot["snapshot_capacity"] * (1 - snapshot["target_load_ratio"])\n        )\n        test_source = source.loc[source["date"].eq("2026-08-07")]\n        test_snapshot = snapshot.loc[snapshot["date"].eq("2026-08-07")]\n        coverage_rows.append(\n            {\n                "horizon_minutes": horizon,\n                "source_test_rows": int(len(test_source)),\n                "snapshot_test_rows": int(len(test_snapshot)),\n                "row_coverage": float(len(test_snapshot) / len(test_source)),\n                "source_low_0_5": int(test_source["label_seats"].le(5).sum()),\n                "snapshot_low_0_5": int(test_snapshot["label_seats"].le(5).sum()),\n            }\n        )\n\n        for feature_set, family in [\n            (PLANNING, "planning"),\n            (FIXED_3STOP, "fixed_3stop"),\n            (target_features, "target"),\n            (route_features, "route_context"),\n        ]:\n            rows, _ = evaluate_learned_models(\n                snapshot, feature_set, family, horizon, args.seed\n            )\n            all_rows.extend(rows)\n\n        for feature_set, family in [\n            (target_features, "target"),\n            (route_features, "route_context"),\n        ]:\n            rows, _ = evaluate_delta_models(\n                snapshot,\n                feature_set,\n                family,\n                horizon,\n                args.seed,\n                source_column="snapshot_remaining_seats",\n            )\n            all_rows.extend(rows)\n\n        planning_baseline, _ = evaluate_naive_baseline(\n            snapshot,\n            horizon=horizon,\n            feature_set=PLANNING.name,\n            family="planning",\n            model="global_median_baseline",\n            source_column=None,\n        )\n        fixed_baseline, _ = evaluate_naive_baseline(\n            snapshot,\n            horizon=horizon,\n            feature_set=FIXED_3STOP.name,\n            family="fixed_3stop",\n            model="upstream_3stop_persistence",\n            source_column="upstream_seats_3",\n        )\n        target_baseline, _ = evaluate_naive_baseline(\n            snapshot,\n            horizon=horizon,\n            feature_set=target_features.name,\n            family="target",\n            model="current_seats_persistence",\n            source_column="snapshot_remaining_seats",\n        )\n        all_rows.extend([planning_baseline, fixed_baseline, target_baseline])\n\n    metrics = pd.DataFrame(all_rows)\n    selected = selected_rows(metrics)\n    coverage = pd.DataFrame(coverage_rows)\n    dynamic = selected.loc[selected["family"].isin(["target", "route_context"])]\n    best_dynamic = dynamic.sort_values(\n        ["selection_calibration_mae", "mae_seats"]\n    ).iloc[0]\n    best_planning = (\n        selected.loc[\n            selected["family"].eq("planning")\n            & selected["horizon_minutes"].eq(best_dynamic["horizon_minutes"])\n        ]\n        .sort_values(["selection_calibration_mae", "mae_seats"])\n        .iloc[0]\n    )\n\n    metrics.to_csv(args.output_dir / "all_metrics.csv", index=False)\n    selected.to_csv(args.output_dir / "selected_by_horizon.csv", index=False)\n    coverage.to_csv(args.output_dir / "snapshot_coverage.csv", index=False)\n    make_plot(selected, args.output_dir / "seat_regression_comparison.png")\n\n    result = {\n        "route_id": ROUTE_ID,\n        "route_name": ROUTE_NAME,\n        "target": "remaining seats before boarding at target-stop arrival",\n        "label_quality": ["A", "B"],\n        "scope": args.scope,\n        "split": {\n            "train": ["2026-08-04", "2026-08-05"],\n            "calibration_and_model_selection": ["2026-08-06"],\n            "untouched_test": ["2026-08-07"],\n        },\n        "coverage_test": _as_records(coverage),\n        "selected_by_horizon": _as_records(selected),\n        "best_dynamic_selected_on_calibration": json_ready(best_dynamic.to_dict()),\n        "best_planning_selected_on_calibration": json_ready(best_planning.to_dict()),\n        "mae_improvement_vs_planning": float(\n            best_planning["mae_seats"] - best_dynamic["mae_seats"]\n        ),\n        "limitations": [\n            "실제 도착시각으로 T-minus 기준점을 만들었으므로 운영 ETA 오차는 포함되지 않았다.",\n            "테스트가 하루이고 저잔여좌석 사례가 제한적이라 운영 성능 확정치가 아니다.",\n            "A등급은 stateCd=1 직접 관측, B등급은 직전 정류장 출발 좌석으로 보완한 값이다.",\n            "B등급의 도착시각은 목표 정류장 최초 관측시각 근사라 수 분의 오차가 있을 수 있다.",\n            "대기 승객 수가 없어 예측 좌석 수를 실제 탑승 가능 인원과 동일하게 볼 수 없다.",\n        ],\n    }\n    (args.output_dir / "summary.json").write_text(\n        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}
RUNTIME_ROOT = Path('/content/state_profile_runtime')
for relative, source in MODULE_SOURCES.items():
    target = RUNTIME_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
sys.path[:0] = [str(RUNTIME_ROOT / 'analysis'), str(RUNTIME_ROOT)]


In [ ]:
from gbis_client import GBISApiCache
from route_specific_feature_experiment import DEFAULT_ROUTES, build_route_snapshots, cache_paths
from pooled_main_model_overfit_ablation import prepare_pooled_data
import json, pandas as pd
cache = GBISApiCache(base_url=API_BASE_URL, api_key=API_KEY, cache_path=LOCAL_CACHE)
try:
    cache.refresh_routes()
    for i, route_id in enumerate(DEFAULT_ROUTES, 1):
        print(f'[sync {i}/6] {DEFAULT_ROUTES[route_id]}', flush=True)
        cache.refresh_stations(route_id); cache.refresh_full_history(route_id)
    cache.refresh_latest(); history_df = cache.history_df()
finally:
    cache.close()
    if LOCAL_CACHE.is_file(): shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
source_cutoff = str(history_df.observed_at.max())
feature_dir = RUNTIME_ROOT / 'feature_cache'
for route_id in DEFAULT_ROUTES:
    snapshots, flows, metadata = build_route_snapshots(LOCAL_CACHE, route_id, source_cutoff=source_cutoff)
    snapshot_path, flow_path, metadata_path = cache_paths(feature_dir, route_id)
    feature_dir.mkdir(parents=True, exist_ok=True)
    snapshots.to_pickle(snapshot_path); flows.to_pickle(flow_path)
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False), encoding='utf-8')
pooled_features, route_metadata = prepare_pooled_data(feature_dir, source_cutoff=source_cutoff)
print('source cutoff:', source_cutoff, 'rows:', len(pooled_features))


In [ ]:
# State Profile main model. 현재/미래 날짜를 절대 사용하지 않는다.
import joblib, numpy as np, pandas as pd
from model_feasibility import FeatureSet
from pooled_main_model_overfit_ablation import pooled_candidates
from main_model_feature_augmentation import component_candidates, make_model
from hypothesis_model_search import candidate_weights, encode_target
STATE = ('snapshot_state_expected_seats', 'snapshot_state_seat_deviation', 'snapshot_state_low10_rate', 'snapshot_state_log_count')
def add_state_profile(data):
    out = data.copy(); cols = list(STATE); out[cols] = np.nan
    keys = ['route_code','direction','snapshot_station_seq_cat','snapshot_time_bin_30']
    fallback_keys = ['route_code','direction','snapshot_time_bin_30']
    weekday = pd.to_datetime(out.date).dt.dayofweek.lt(5)
    global_mean = float(out.snapshot_remaining_seats.median())
    global_low = float(out.snapshot_remaining_seats.le(10).mean())
    for date in sorted(out.date.unique()):
        mask = out.date.eq(date); hist = out.loc[out.date.lt(date) & weekday]
        stats = hist.groupby(keys, observed=True).snapshot_remaining_seats.agg(['mean','count'])
        low = hist.assign(_low=hist.snapshot_remaining_seats.le(10).astype(float)).groupby(keys, observed=True)['_low'].mean()
        fallback = hist.groupby(fallback_keys, observed=True).snapshot_remaining_seats.agg(['mean','count'])
        fallback_low = hist.assign(_low=hist.snapshot_remaining_seats.le(10).astype(float)).groupby(fallback_keys, observed=True)['_low'].mean()
        target = out.loc[mask]
        mean = pd.Series(stats['mean'].reindex(target.set_index(keys).index).to_numpy(), index=target.index).fillna(pd.Series(fallback['mean'].reindex(target.set_index(fallback_keys).index).to_numpy(), index=target.index)).fillna(global_mean)
        count = pd.Series(stats['count'].reindex(target.set_index(keys).index).to_numpy(), index=target.index).fillna(pd.Series(fallback['count'].reindex(target.set_index(fallback_keys).index).to_numpy(), index=target.index)).fillna(0)
        rate = pd.Series(low.reindex(target.set_index(keys).index).to_numpy(), index=target.index).fillna(pd.Series(fallback_low.reindex(target.set_index(fallback_keys).index).to_numpy(), index=target.index)).fillna(global_low)
        out.loc[mask, cols] = np.column_stack([mean, target.snapshot_remaining_seats.to_numpy(float)-mean.to_numpy(float), rate, np.log1p(count)])
    return out
training_cutoff_date = '2026-08-14'
data = add_state_profile(pooled_features)
training = data.loc[data.date.le(training_cutoff_date) & pd.to_datetime(data.date).dt.dayofweek.lt(5)].copy()
base, _ = pooled_candidates()['pooled_40_legacy_ceiling_features']
features = FeatureSet('pooled_state_profile', (*base.numeric, *STATE), base.categorical)
models, target_kinds = {}, {}
for i, candidate in enumerate(component_candidates()):
    print('[fit]', candidate.name, flush=True)
    model = make_model(candidate, features, 42+i)
    weights = candidate_weights(training, candidate.low_weight, weighting_kind=candidate.weighting_kind, gap_weight_power=candidate.gap_weight_power)
    model.fit(training[list(features.columns)], encode_target(training, candidate.target_kind), regressor__sample_weight=weights)
    models[candidate.name] = model; target_kinds[candidate.name] = candidate.target_kind
lookup_keys = ['route_code','direction','snapshot_station_seq_cat','snapshot_time_bin_30']
lookup = training.groupby(lookup_keys, observed=True).snapshot_remaining_seats.agg(['mean','count'])
low = training.assign(_low=training.snapshot_remaining_seats.le(10).astype(float)).groupby(lookup_keys, observed=True)['_low'].mean()
payload = {'format_version':1, 'model_variant':'pooled_state_profile_main', 'model_id':'arrival-seat-pooled-state-profile/v1.0.0', 'training_cutoff_date':training_cutoff_date, 'source_cutoff':source_cutoff, 'routes':dict(DEFAULT_ROUTES), 'feature_columns':list(features.columns), 'component_target_kinds':target_kinds, 'ensemble_weights':{'hgb':.4,'extra_trees':.4,'lightgbm':.2}, 'models':models, 'state_profile':{'keys':lookup_keys, 'mean':lookup['mean'].to_dict(), 'count':lookup['count'].to_dict(), 'low10_rate':low.to_dict(), 'global_mean':float(training.snapshot_remaining_seats.mean()), 'global_low10_rate':float(training.snapshot_remaining_seats.le(10).mean())}, 'training_policy':{'sampling':'none','feature_history':'strictly earlier weekday dates'}, 'training_rows':int(len(training)), 'training_events':int(training.event_id.nunique()), 'training_dates':sorted(training.date.unique().tolist()), 'route_cache_metadata':route_metadata}
artifact = Path('/content/arrival_seat_state_profile_v1.pkl')
joblib.dump(payload, artifact, compress=3)
print(artifact, payload['training_rows'], payload['training_events'])


In [ ]:
from google.colab import files
files.download('/content/arrival_seat_state_profile_v1.pkl')
